# Philippine Rice Pest Surveillance
## From published insect annotations to an exploratory counting benchmark

**Scientific applications / composed-system capstone · E2E / WORKSHOP · Candidate**

Can a detector and crop classifier reproduce published rice-pest counts, and where do errors enter the system?

**Input:** full detection-image exports. **System:** DETR → insect crops → BioCLIP species head. **Output:** boxes, species predictions, counts and review flags. SigLIP and simple baselines help interpret the result.

This notebook uses **200 images**, split into **120 train / 40 validation / 40 test**, grouped by source-image family and duplicate checks. These groups are **not verified independent capture sessions or specimens**. The published annotations are unverified references, not independently checked ground truth. This is an exploratory benchmark, not a qualified field-surveillance system.

The source is a CC BY 4.0 [PhilRice-affiliated Zenodo dataset](https://doi.org/10.5281/zenodo.20066074). Its “raw” images are resized Roboflow exports; capture dates, trap IDs and locations are blank. Philippine institutional provenance is documented; exact capture location is not. Four source images exceed DETR's 100-query capacity and are excluded from this bounded study before model evaluation.

**Prerequisites:** basic Python/Colab familiarity. Choose a **fresh T4 GPU** and **Run all** with defaults. No token, repository clone, DIMER worker, private upload or manual restart is required. Initial budget: up to 45 minutes compute and 20 GiB free disk; these are targets pending hosted measurement.

**Predict before running:** will missed boxes or species mistakes contribute more to count error? Write one reason. Completion notes are optional, not required submissions.

**How to read this notebook:** each section says what goes in, what comes out and what to notice. Terms are defined in the glossary below; come back to it when a metric first appears.

**AI Assistance Disclosure:** Code and instructional text were developed with generative AI assistance under maintainer direction. The maintainer is responsible for review, validation and release decisions. AI assistance is not independent verification or provider endorsement.

### Glossary
| Term | Meaning in this notebook |
|---|---|
| **Reference box** | A box and species label published with the dataset. Treated as the answer key, but not independently verified. |
| **IoU** (intersection over union) | Overlap between two boxes, from 0 (none) to 1 (identical). A prediction matches a reference box when IoU ≥ 0.5. |
| **AP50** | Detection average precision at IoU 0.5: how well ranked boxes find reference insects, from 0 to 1. It ignores species. |
| **Detector score** | DETR's confidence that a box contains *a target pest*. It says nothing about which species. |
| **Species score / margin** | The classifier's probability for its chosen species, and the gap between the two species' probabilities. A small margin sends the detection to **review**. |
| **Raw / accepted / referred count** | Raw counts every detection above the detector threshold. Accepted excludes detections sent to review; referred counts those review items. Review never deletes insects. |
| **Count MAE** | Mean absolute error of per-image, per-species counts. 0.5 means counts are off by half an insect per image and species on average. |
| **Missed / spurious / wrong species** | A reference with no matching prediction; a prediction with no matching reference; a matched pair whose species differ. |
| **Macro-F1** | The average of per-species F1 scores, so the rarer species counts as much as the common one. |
| **Cross-entropy** | The training loss for the species head; lower on validation means better-calibrated species scores. |
| **RGB centroid** | A deliberately simple baseline that classifies crops by average colour. Beating it shows the model uses more than colour. |
| **Bootstrap interval** | A range from resampling source families with replacement. It shows sampling uncertainty over these families, not over independent field captures. |

In [ ]:
import csv
import hashlib
import io
import json
import os
from pathlib import Path
import platform
import shutil
import subprocess
import time
import urllib.request
import uuid
import zipfile
from IPython.display import Image, Markdown, FileLink, display

if platform.system() != 'Linux' or platform.machine() != 'x86_64':
    raise RuntimeError('Use a fresh Colab Linux x86-64 T4 runtime.')
try:
    GPU = subprocess.check_output(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True).strip()
except (FileNotFoundError, subprocess.CalledProcessError) as exc:
    raise RuntimeError('Select Runtime > Change runtime type > T4 GPU, then Run all.') from exc
if 'T4' not in GPU:
    raise RuntimeError('The canonical qualification target is a fresh T4 runtime.')
ROOT = Path.cwd() / 'outputs' / 'rice_pest' / uuid.uuid4().hex[:12]
ROOT.mkdir(parents=True)
if shutil.disk_usage(ROOT).free < 20 * 1024**3:
    raise RuntimeError('Allow at least 20 GiB free disk for the isolated environment, weights and outputs.')
print('GPU:', GPU, '\nRun directory:', ROOT)

## 1. Reconstruct the portable experiment
The notebook carries its own source, immutable data/model manifests and a hashed dependency lock. The separate Python environment avoids changing libraries already loaded by Colab. Hash failures stop execution rather than substituting a different experiment.

The next cell is **collapsed on purpose**: it holds about 1.3 million characters of embedded source and manifests, and you do not need to read it to follow the experiment. Readable copies are in the repository: the [runtime](https://github.com/kurtvalcorza/detr-detection-pipeline/blob/main/tools/rice_capstone.py), [metrics and matching](https://github.com/kurtvalcorza/detr-detection-pipeline/blob/main/tools/rice_core.py) and [figures](https://github.com/kurtvalcorza/detr-detection-pipeline/blob/main/tools/rice_figures.py). The same files are written into the run directory.

In [ ]:
# @title Carrier cell: write embedded source, manifests and lock, then build the isolated environment
FILES = {'capstone.py': '"""Standalone, qualification-gated Philippine rice-pest capstone runtime.\n\nModel stages require CUDA. Importing this module downloads nothing. Reference\nlabels are never used to select settings on the test split.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport contextlib\nimport csv\nimport gc\nimport importlib.metadata\nimport io\nimport json\nimport math\nimport os\nimport platform\nimport shutil\nimport time\nimport zipfile\nfrom pathlib import Path\n\nimport numpy as np\nimport rice_core as core\nfrom PIL import Image, ImageOps\n\nSPECIES = ("rice_black_bug", "white_stemborer")\nPINS = {\n    "detr": ("facebook/detr-resnet-50", "1d5f47bd3bdd2c4bbfa585418ffe6da5028b4c0b"),\n    "bioclip": ("imageomics/bioclip-2", "2957b322090f9cb17ae72c71981c7218a28d81e0"),\n    "siglip": ("google/siglip2-base-patch16-224", "5ffaac51d5e2f3367f7dab0cad4be4cb07c0caa2"),\n}\n# Model stages run on the T4; CPU tests override this to exercise the same code paths.\nDEVICE = "cuda"\nSTAGES = (\n    "prepare",\n    "features",\n    "classifier",\n    "detector",\n    "policy",\n    "evaluate",\n    "activity",\n    "reload",\n    "report",\n)\n\n\ndef read(path):\n    return json.loads(Path(path).read_text(encoding="utf-8"))\n\n\ndef write(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(value, indent=2, allow_nan=False) + "\\n", encoding="utf-8")\n\n\ndef csv_write(path, rows, fields=None):\n    rows = list(rows)\n    fields = fields or (list(rows[0]) if rows else [])\n    with Path(path).open("w", encoding="utf-8", newline="") as handle:\n        writer = csv.DictWriter(handle, fieldnames=fields)\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef context(root):\n    manifest, audit = read(root / "data_manifest.json"), read(root / "dataset_audit.json")\n    exploratory = manifest.get("scope") == "exploratory_published_annotations"\n    required = (\n        ("rights", "original_images", "crop_lineage", "capacity")\n        if exploratory\n        else (\n            "rights",\n            "philippine_provenance",\n            "original_images",\n            "reference_annotations",\n            "completeness",\n            "crop_lineage",\n            "independence",\n            "capacity",\n        )\n    )\n    if not audit.get("qualification", {}).get("passed") or any(\n        audit.get("gates", {}).get(key, {}).get("status") != "passed" for key in required\n    ):\n        raise RuntimeError(\n            "Dataset qualification is incomplete. Resolve the recorded audit gates before model execution."\n        )\n    if audit.get("manifest_sha256") != core.sha256(root / "data_manifest.json"):\n        raise ValueError("Dataset audit does not bind the exact data manifest")\n    if exploratory and audit.get("qualification", {}).get("scope") != "exploratory_published_annotations":\n        raise ValueError("Exploratory scope must be explicitly recorded in the qualification")\n    core.validate_manifest(manifest, require_capacity=not exploratory)\n    if any(obj.get("ignore", False) for record in manifest["records"] for obj in record["objects"]):\n        raise ValueError("This bounded count benchmark requires records without ignored target regions")\n    return manifest, manifest["records"]\n\n\ndef active_objects(record):\n    return [obj for obj in record["objects"] if not obj.get("ignore", False)]\n\n\ndef image(root, record):\n    path = core.safe_path(root / "data", record["relative_path"])\n    if path.stat().st_size != record["bytes"] or core.sha256(path) != record["sha256"]:\n        raise ValueError(f"Image digest mismatch: {record[\'image_id\']}")\n    with Image.open(path) as source:\n        if source.getexif().get(274, 1) != 1:\n            raise ValueError("Normalize EXIF orientation and box coordinates during dataset freeze")\n        result = source.convert("RGB")\n    if result.size != (record["width"], record["height"]):\n        raise ValueError("Decoded image dimensions disagree with annotation coordinates")\n    return result\n\n\ndef crops(root, records):\n    result, lineage = [], []\n    for record in records:\n        original = image(root, record)\n        for obj in active_objects(record):\n            coords = obj["bbox_xyxy"]\n            effective = [\n                math.floor(coords[0]),\n                math.floor(coords[1]),\n                math.ceil(coords[2]),\n                math.ceil(coords[3]),\n            ]\n            result.append(original.crop(tuple(effective)))\n            lineage.append(\n                {\n                    "crop_id": f"{record[\'image_id\']}:{obj[\'object_id\']}",\n                    "parent_image_id": record["image_id"],\n                    "parent_object_id": obj["object_id"],\n                    "split": record["split"],\n                    "capture_group_id": record["capture_group_id"],\n                    "species": obj["species"],\n                    "bbox_xyxy": coords,\n                    "effective_crop_xyxy": effective,\n                }\n            )\n    return result, lineage\n\n\ndef softmax(x):\n    x = np.asarray(x, dtype=float)\n    exp = np.exp(x - x.max(axis=1, keepdims=True))\n    return exp / exp.sum(axis=1, keepdims=True)\n\n\ndef classification(y, scores):\n    pred = np.asarray(scores).argmax(axis=1)\n    cm = np.zeros((2, 2), dtype=int)\n    np.add.at(cm, (np.asarray(y, dtype=int), pred), 1)\n    recall = np.divide(cm.diagonal(), cm.sum(axis=1), out=np.zeros(2), where=cm.sum(axis=1) > 0)\n    precision = np.divide(cm.diagonal(), cm.sum(axis=0), out=np.zeros(2), where=cm.sum(axis=0) > 0)\n    f1 = np.divide(2 * precision * recall, precision + recall, out=np.zeros(2), where=precision + recall > 0)\n    return {\n        "accuracy": float(np.mean(np.asarray(y) == pred)),\n        "macro_f1": float(f1.mean()),\n        "confusion": cm.tolist(),\n        "per_class": [\n            {\n                "species": s,\n                "precision": float(precision[k]),\n                "recall": float(recall[k]),\n                "f1": float(f1[k]),\n                "support": int(cm[k].sum()),\n            }\n            for k, s in enumerate(SPECIES)\n        ],\n    }\n\n\ndef torch_runtime():\n    import torch\n\n    if not torch.cuda.is_available():\n        raise RuntimeError("Select a fresh Colab T4 GPU runtime for model execution")\n    torch.manual_seed(42)\n    torch.cuda.manual_seed_all(42)\n    torch.backends.cuda.matmul.allow_tf32 = False\n    torch.backends.cudnn.allow_tf32 = False\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n    return torch\n\n\ndef stage_model(root, key):\n    from huggingface_hub import hf_hub_download\n\n    manifest = read(root / "model_manifest.json")[key]\n    if (manifest["modelId"], manifest["revision"]) != PINS[key]:\n        raise ValueError("Model identity mismatch")\n    folder = root / "weights" / key\n    folder.mkdir(parents=True, exist_ok=True)\n    for item in manifest["files"]:\n        path = core.safe_path(folder, item["path"])\n        if not path.exists():\n            hf_hub_download(\n                manifest["modelId"], item["path"], revision=manifest["revision"], local_dir=folder\n            )\n        if path.stat().st_size != item["bytes"] or core.sha256(path) != item["sha256"]:\n            raise ValueError(f"Snapshot digest mismatch: {key}/{item[\'path\']}")\n    return folder\n\n\ndef bioclip(root):\n    torch_runtime()\n    from open_clip import image_transform\n    from open_clip.model import CLIP\n    from open_clip.tokenizer import SimpleTokenizer\n    from safetensors.torch import load_file\n\n    folder = stage_model(root, "bioclip")\n    cfg = read(folder / "open_clip_config.json")\n    mc, pc = cfg["model_cfg"], cfg["preprocess_cfg"]\n    model = CLIP(\n        embed_dim=mc["embed_dim"],\n        vision_cfg=mc["vision_cfg"],\n        text_cfg=mc["text_cfg"],\n        quick_gelu=bool(mc.get("quick_gelu", False)),\n    )\n    model.load_state_dict(load_file(str(folder / "open_clip_model.safetensors")), strict=True)\n    preprocess = image_transform(\n        mc["vision_cfg"]["image_size"],\n        is_train=False,\n        mean=tuple(pc["mean"]),\n        std=tuple(pc["std"]),\n        resize_mode=pc.get("resize_mode", "shortest"),\n        interpolation=pc.get("interpolation", "bicubic"),\n    )\n    return model.cuda().eval(), preprocess, SimpleTokenizer(context_length=mc["text_cfg"]["context_length"])\n\n\ndef embed(model, preprocess, images):\n    import torch\n\n    vectors = []\n    with torch.inference_mode():\n        for start in range(0, len(images), 8):\n            pixels = torch.stack([preprocess(im) for im in images[start : start + 8]]).cuda()\n            vectors.append(model.encode_image(pixels, normalize=True).float().cpu().numpy())\n    return (\n        np.concatenate(vectors)\n        if vectors\n        else np.empty((0, model.text_projection.shape[1]), dtype=np.float32)\n    )\n\n\ndef prepare(root):\n    _, records = context(root)\n    _, lineage = crops(root, records)\n    csv_write(\n        root / "outputs" / "crop_lineage.csv",\n        [{**x, "bbox_xyxy": json.dumps(x["bbox_xyxy"])} for x in lineage],\n    )\n    csv_write(\n        root / "outputs" / "split_manifest.csv",\n        [{k: r[k] for k in ("image_id", "capture_group_id", "split", "sha256")} for r in records],\n    )\n    write(root / "outputs" / "lineage.json", lineage)\n    csv_write(\n        root / "outputs" / "sample_summary.csv",\n        [\n            {\n                "split": split,\n                "images": sum(r["split"] == split for r in records),\n                "source_families": len({r["capture_group_id"] for r in records if r["split"] == split}),\n                **{\n                    species: sum(x["split"] == split and x["species"] == species for x in lineage)\n                    for species in SPECIES\n                },\n                "max_objects_per_image": max(\n                    (len(active_objects(r)) for r in records if r["split"] == split), default=0\n                ),\n            }\n            for split in ("train", "validation", "test")\n        ],\n    )\n    audit = read(root / "dataset_audit.json")\n    csv_write(\n        root / "outputs" / "audit_gates.csv",\n        [\n            {"gate": name, "status": gate.get("status"), "evidence": gate.get("evidence", "see audit JSON")}\n            for name, gate in audit.get("gates", {}).items()\n        ],\n    )\n    write(\n        root / "outputs" / "environment.json",\n        {\n            "python": platform.python_version(),\n            "platform": platform.platform(),\n            "pid": os.getpid(),\n            "packages": {\n                name: importlib.metadata.version(name)\n                for name in (\n                    "torch",\n                    "torchvision",\n                    "transformers",\n                    "open_clip_torch",\n                    "numpy",\n                    "scipy",\n                    "Pillow",\n                    "pycocotools",\n                    "safetensors",\n                )\n            },\n            "status": "Candidate; hosted end-to-end verification required",\n        },\n    )\n\n\ndef features(root):\n    torch = torch_runtime()\n    _, records = context(root)\n    pictures, lineage = crops(root, records)\n    prompts = ["a photo of a rice black bug.", "a photo of a white stemborer."]\n    model, transform, tokenizer = bioclip(root)\n    vectors = embed(model, transform, pictures)\n    with torch.inference_mode():\n        text = model.encode_text(tokenizer(prompts).cuda(), normalize=True).cpu().numpy()\n    scale = float(model.logit_scale.exp().item())\n    np.savez(\n        root / "outputs" / "features.npz",\n        images=vectors,\n        text=text,\n        scale=scale,\n        zero_shot=softmax(scale * vectors @ text.T),\n    )\n    del model\n    gc.collect()\n    torch.cuda.empty_cache()\n    from transformers import AutoModel, AutoProcessor\n\n    folder = stage_model(root, "siglip")\n    model = (\n        AutoModel.from_pretrained(\n            folder, local_files_only=True, trust_remote_code=False, use_safetensors=True\n        )\n        .float()\n        .cuda()\n        .eval()\n    )\n    processor = AutoProcessor.from_pretrained(folder, local_files_only=True, trust_remote_code=False)\n    scores = []\n    with torch.inference_mode():\n        for start in range(0, len(pictures), 8):\n            inputs = processor(\n                text=prompts, images=pictures[start : start + 8], padding="max_length", return_tensors="pt"\n            ).to("cuda")\n            scores.extend(torch.sigmoid(model(**inputs).logits_per_image).cpu().numpy())\n    np.save(root / "outputs" / "siglip.npy", np.asarray(scores))\n    train = np.array([x["split"] == "train" for x in lineage])\n    y = np.array([SPECIES.index(x["species"]) for x in lineage])\n    rgb = np.array(\n        [\n            np.concatenate(\n                [\n                    np.asarray(ImageOps.fit(im, (64, 64)), dtype=float).mean(axis=(0, 1)),\n                    np.asarray(ImageOps.fit(im, (64, 64)), dtype=float).std(axis=(0, 1)),\n                ]\n            )\n            for im in pictures\n        ]\n    )\n    centroids = np.stack([rgb[train & (y == k)].mean(axis=0) for k in range(2)])\n    np.save(root / "outputs" / "rgb.npy", -np.linalg.norm(rgb[:, None] - centroids[None], axis=2))\n    val = np.array([x["split"] == "validation" for x in lineage])\n    majority = int(np.bincount(y[train], minlength=2).argmax())\n    majority_scores = np.tile(np.eye(2)[majority], (len(y), 1))\n    np.save(root / "outputs" / "majority.npy", majority_scores)\n    write(\n        root / "outputs" / "validation_crops.json",\n        {\n            key: classification(y[val], value[val])\n            for key, value in {\n                "bioclip": softmax(scale * vectors @ text.T),\n                "siglip": np.asarray(scores),\n                "rgb": -np.linalg.norm(rgb[:, None] - centroids[None], axis=2),\n                "majority": majority_scores,\n            }.items()\n        },\n    )\n\n\ndef fit_head(features, labels_array, text, scale, train, val, *, device="cpu", epochs=20):\n    """Pure bounded fitter; CPU fixtures verify selection without model downloads."""\n    import torch\n\n    torch.manual_seed(42)\n    train, val = np.asarray(train), np.asarray(val)\n    if len(train) > 1000:\n        train = np.sort(np.random.default_rng(42).choice(train, 1000, replace=False))\n    x = torch.tensor(features, device=device, dtype=torch.float32)\n    labels = torch.tensor(labels_array, device=device, dtype=torch.long)\n    head = torch.nn.Linear(x.shape[1], 2).to(device)\n    with torch.no_grad():\n        head.weight.copy_(torch.tensor(text * scale, device=device))\n        head.bias.zero_()\n    optimizer = torch.optim.AdamW(head.parameters(), lr=1e-4, weight_decay=0.01)\n    best_loss, best, history = float("inf"), None, []\n    rng = np.random.default_rng(42)\n    for epoch in range(epochs + 1):\n        if epoch:\n            head.train()\n            order = rng.permutation(train)\n            for start in range(0, len(order), 32):\n                ix = order[start : start + 32]\n                optimizer.zero_grad(set_to_none=True)\n                loss = torch.nn.functional.cross_entropy(head(x[ix]), labels[ix])\n                loss.backward()\n                optimizer.step()\n        head.eval()\n        with torch.no_grad():\n            loss_value = float(torch.nn.functional.cross_entropy(head(x[val]), labels[val]).item())\n            train_value = float(torch.nn.functional.cross_entropy(head(x[train]), labels[train]).item())\n        history.append(\n            {"epoch": epoch, "validation_cross_entropy": loss_value, "training_cross_entropy": train_value}\n        )\n        if loss_value < best_loss:\n            best_loss, selected = loss_value, epoch\n            best = {key: value.detach().cpu().clone() for key, value in head.state_dict().items()}\n    return best, selected, history\n\n\ndef classifier(root):\n    torch_runtime()\n    from safetensors.torch import save_file\n\n    lineage = read(root / "outputs" / "lineage.json")\n    data = np.load(root / "outputs" / "features.npz", allow_pickle=False)\n    y = np.array([SPECIES.index(x["species"]) for x in lineage])\n    train = np.flatnonzero([x["split"] == "train" for x in lineage])\n    val = np.flatnonzero([x["split"] == "validation" for x in lineage])\n    best, selected, history = fit_head(\n        data["images"], y, data["text"], data["scale"], train, val, device="cuda"\n    )\n    selected_train = (\n        np.sort(np.random.default_rng(42).choice(train, 1000, replace=False)) if len(train) > 1000 else train\n    )\n    write(\n        root / "outputs" / "classifier_training_subset.json",\n        {"seed": 42, "maximum_crops": 1000, "crop_ids": [lineage[int(i)]["crop_id"] for i in selected_train]},\n    )\n    folder = root / "outputs" / "classifier_adapter"\n    folder.mkdir(exist_ok=True)\n    save_file(best, str(folder / "head.safetensors"))\n    write(\n        folder / "manifest.json",\n        {\n            "format": "dimer-rice-frozen-head-v1",\n            "classes": list(SPECIES),\n            "base": list(PINS["bioclip"]),\n            "preprocessing": "pinned-bioclIP-config-normalized-features",\n            "sha256": core.sha256(folder / "head.safetensors"),\n            "epoch": selected,\n            "trainable_parameters": sum(v.numel() for v in best.values()),\n        },\n    )\n    np.save(\n        root / "outputs" / "adapted.npy",\n        softmax(data["images"] @ best["weight"].numpy().T + best["bias"].numpy()),\n    )\n    write(root / "outputs" / "classifier_history.json", history)\n\n\ndef load_head(root):\n    from safetensors.numpy import load_file\n\n    folder = root / "outputs" / "classifier_adapter"\n    meta = read(folder / "manifest.json")\n    if (\n        meta["format"] != "dimer-rice-frozen-head-v1"\n        or meta["classes"] != list(SPECIES)\n        or meta["base"] != list(PINS["bioclip"])\n        or meta["preprocessing"] != "pinned-bioclIP-config-normalized-features"\n    ):\n        raise ValueError("Classifier adapter identity mismatch")\n    if core.sha256(folder / "head.safetensors") != meta["sha256"]:\n        raise ValueError("Classifier tensor digest mismatch")\n    tensors = load_file(str(folder / "head.safetensors"))\n    if (\n        set(tensors) != {"weight", "bias"}\n        or tensors["weight"].shape != (2, 768)\n        or tensors["bias"].shape != (2,)\n        or not all(v.dtype == np.float32 and np.isfinite(v).all() for v in tensors.values())\n    ):\n        raise ValueError("Classifier tensor contract mismatch")\n    return tensors\n\n\ndef detr_model(root, adapted=False):\n    from transformers import DetrForObjectDetection, DetrImageProcessor\n\n    torch_runtime()\n    folder = stage_model(root, "detr")\n    processor = DetrImageProcessor.from_pretrained(folder, local_files_only=True, trust_remote_code=False)\n    if dict(processor.size) != {"shortest_edge": 800, "longest_edge": 1333}:\n        raise ValueError("Pinned DETR resizing differs from the frozen training recipe")\n    model = (\n        DetrForObjectDetection.from_pretrained(\n            folder,\n            num_labels=1,\n            id2label={0: "target_pest"},\n            label2id={"target_pest": 0},\n            ignore_mismatched_sizes=True,\n            use_pretrained_backbone=False,\n            local_files_only=True,\n            trust_remote_code=False,\n            use_safetensors=True,\n        )\n        .float()\n        .cuda()\n    )\n    if adapted:\n        from safetensors.torch import load_file\n\n        adapter = root / "outputs" / "detector_adapter"\n        meta = read(adapter / "manifest.json")\n        if (\n            meta.get("format") != "dimer-rice-detr-adapter-v1"\n            or meta.get("preprocessing") != dict(processor.size)\n            or meta.get("frozen_prefix") != "model.backbone."\n            or meta["base"] != list(PINS["detr"])\n            or meta["classes"] != ["target_pest"]\n            or meta["sha256"] != core.sha256(adapter / "adapter.safetensors")\n        ):\n            raise ValueError("Detector adapter identity mismatch")\n        tensors = load_file(str(adapter / "adapter.safetensors"))\n        import torch\n\n        if any(not torch.isfinite(tensor).all() for tensor in tensors.values()):\n            raise ValueError("Nonfinite detector adapter tensor")\n        expected = {key for key in model.state_dict() if not key.startswith("model.backbone.")}\n        if set(tensors) != expected:\n            raise ValueError("Detector adapter tensor inventory mismatch")\n        result = model.load_state_dict(tensors, strict=False)\n        if result.unexpected_keys or any(not k.startswith("model.backbone.") for k in result.missing_keys):\n            raise ValueError("Detector adapter shape mismatch")\n    return model.eval(), processor\n\n\ndef validate_raw_detections(logits, pred_boxes, *, queries, classes):\n    """Refuse malformed or nonfinite DETR tensors before any score threshold is applied.\n\n    The pinned postprocessor keeps entries whose score exceeds the threshold, so a NaN\n    score is silently dropped and an all-NaN image would look like a valid empty result.\n    Returns the fewest queries the threshold-0 postprocessor may keep.\n    """\n    logits = np.asarray(logits, dtype=np.float64)\n    pred_boxes = np.asarray(pred_boxes, dtype=np.float64)\n    if logits.shape != (1, queries, classes + 1) or pred_boxes.shape != (1, queries, 4):\n        raise ValueError(\n            f"Malformed DETR output: logits {logits.shape}, boxes {pred_boxes.shape}; "\n            f"expected (1, {queries}, {classes + 1}) and (1, {queries}, 4)"\n        )\n    if not (np.isfinite(logits).all() and np.isfinite(pred_boxes).all()):\n        raise ValueError("Nonfinite DETR logits or boxes; refusing to treat them as zero detections")\n    if (pred_boxes < 0).any() or (pred_boxes > 1).any():\n        raise ValueError("DETR normalised boxes fall outside [0, 1]")\n    # Explicit filtering policy: threshold 0 may drop only queries whose target-class\n    # probability underflows in float32; anything above 1e-30 must be retained.\n    return int((softmax(logits[0])[:, :-1].max(axis=1) > 1e-30).sum())\n\n\ndef validate_postprocessed(scores, boxes, labels, minimum, queries):\n    scores, boxes, labels = np.asarray(scores), np.asarray(boxes), np.asarray(labels)\n    kept = len(scores)\n    if (\n        not minimum <= kept <= queries\n        or scores.shape != (kept,)\n        or boxes.shape != (kept, 4)\n        or labels.shape != (kept,)\n    ):\n        raise ValueError(\n            f"Postprocessed detections {scores.shape}/{boxes.shape} disagree with "\n            f"{minimum}-{queries} valid queries"\n        )\n    if not (np.isfinite(scores).all() and np.isfinite(boxes).all()):\n        raise ValueError("Nonfinite postprocessed detections")\n    if ((scores < 0) | (scores > 1)).any() or (labels != 0).any():\n        raise ValueError("Postprocessed detection scores or labels outside the declared domain")\n\n\ndef validate_species_scores(probabilities, rows):\n    probabilities = np.asarray(probabilities, dtype=np.float64)\n    if probabilities.shape != (rows, len(SPECIES)):\n        raise ValueError(f"Species scores {probabilities.shape} disagree with {rows} detections")\n    if not np.isfinite(probabilities).all():\n        raise ValueError("Nonfinite species scores; refusing to count these detections")\n    if ((probabilities < 0) | (probabilities > 1)).any() or not np.allclose(\n        probabilities.sum(axis=1), 1, atol=1e-4\n    ):\n        raise ValueError("Species scores are not a probability distribution")\n\n\ndef detections(root, model, processor, records):\n    import torch\n\n    result = []\n    with torch.inference_mode():\n        for record in records:\n            original = image(root, record)\n            inputs = processor(images=original, return_tensors="pt").to(DEVICE)\n            output = model(**inputs)\n            minimum = validate_raw_detections(\n                output.logits.float().cpu().numpy(),\n                output.pred_boxes.float().cpu().numpy(),\n                queries=model.config.num_queries,\n                classes=model.config.num_labels,\n            )\n            post = processor.post_process_object_detection(\n                output,\n                threshold=0.0,\n                target_sizes=torch.tensor([[original.height, original.width]], device=DEVICE),\n            )[0]\n            scores, boxes = post["scores"].cpu().numpy(), post["boxes"].cpu().numpy()\n            validate_postprocessed(\n                scores, boxes, post["labels"].cpu().numpy(), minimum, model.config.num_queries\n            )\n            # Preserve all ranked queries; thresholds applied later, no NMS.\n            order = np.argsort(-scores, kind="stable")\n            result.append(\n                {\n                    "image_id": record["image_id"],\n                    "boxes": boxes[order].tolist(),\n                    "scores": scores[order].tolist(),\n                }\n            )\n    return result\n\n\ndef coco_metrics(records, predictions):\n    from pycocotools.coco import COCO\n    from pycocotools.cocoeval import COCOeval\n\n    if not sum(len(active_objects(r)) for r in records):\n        return {"ap50": None, "ap": None, "reason": "No reference targets"}\n    annotations, predicted, images = [], [], []\n    for index, (record, prediction) in enumerate(zip(records, predictions, strict=True)):\n        images.append({"id": index, "width": record["width"], "height": record["height"]})\n        for obj in record["objects"]:\n            x1, y1, x2, y2 = obj["bbox_xyxy"]\n            annotations.append(\n                {\n                    "id": len(annotations) + 1,\n                    "image_id": index,\n                    "category_id": 1,\n                    "bbox": [x1, y1, x2 - x1, y2 - y1],\n                    "area": (x2 - x1) * (y2 - y1),\n                    "iscrowd": int(obj.get("ignore", False)),\n                }\n            )\n        for box, score in zip(prediction["boxes"], prediction["scores"], strict=True):\n            x1, y1, x2, y2 = box\n            predicted.append(\n                {"image_id": index, "category_id": 1, "bbox": [x1, y1, x2 - x1, y2 - y1], "score": score}\n            )\n    with contextlib.redirect_stdout(io.StringIO()):\n        truth = COCO()\n        truth.dataset = {\n            "images": images,\n            "annotations": annotations,\n            "categories": [{"id": 1, "name": "target_pest"}],\n            "info": {},\n        }\n        truth.createIndex()\n        if predicted:\n            detected = truth.loadRes(predicted)\n        else:\n            detected = COCO()\n            detected.dataset = {**truth.dataset, "annotations": []}\n            detected.createIndex()\n        evaluator = COCOeval(truth, detected, "bbox")\n        evaluator.params.maxDets = [1, 10, 100]\n        evaluator.evaluate()\n        evaluator.accumulate()\n        evaluator.summarize()\n    return {\n        "ap": float(evaluator.stats[0]),\n        "ap50": float(evaluator.stats[1]),\n        "max_detections": 100,\n        "ignore_semantics": "Explicit ignored regions represented as COCO crowd regions",\n    }\n\n\ndef detector(root):\n    torch = torch_runtime()\n    from safetensors.torch import save_file\n\n    _, records = context(root)\n    train = [r for r in records if r["split"] == "train"]\n    val = [r for r in records if r["split"] == "validation"]\n    model, processor = detr_model(root)\n    for name, parameter in model.named_parameters():\n        parameter.requires_grad = not name.startswith("model.backbone.")\n    params = [p for p in model.parameters() if p.requires_grad]\n    optimizer = torch.optim.AdamW(params, lr=1e-4, weight_decay=1e-4)\n    folder = root / "outputs" / "detector_adapter"\n    folder.mkdir(exist_ok=True)\n    best, history = -1.0, []\n    rng = np.random.default_rng(42)\n    for epoch in range(16):\n        losses = []\n        if epoch:\n            model.train()\n            # Frozen convolutional statistics must not update during adaptation.\n            model.model.backbone.eval()\n            order = rng.permutation(len(train))\n            batches = [order[start : start + 2] for start in range(0, len(order), 2)]\n            for window_start in range(0, len(batches), 4):\n                window = batches[window_start : window_start + 4]\n                optimizer.zero_grad(set_to_none=True)\n                for batch_ix in window:\n                    batch = [train[int(i)] for i in batch_ix]\n                    annotations = []\n                    for k, record in enumerate(batch):\n                        objects = []\n                        for obj in active_objects(record):\n                            x1, y1, x2, y2 = obj["bbox_xyxy"]\n                            objects.append(\n                                {\n                                    "image_id": k,\n                                    "category_id": 0,\n                                    "bbox": [x1, y1, x2 - x1, y2 - y1],\n                                    "area": (x2 - x1) * (y2 - y1),\n                                    "iscrowd": 0,\n                                }\n                            )\n                        annotations.append({"image_id": k, "annotations": objects})\n                    encoded = processor(\n                        images=[image(root, r) for r in batch], annotations=annotations, return_tensors="pt"\n                    )\n                    labels = [{k: v.cuda() for k, v in target.items()} for target in encoded["labels"]]\n                    loss = model(\n                        pixel_values=encoded["pixel_values"].cuda(),\n                        pixel_mask=encoded["pixel_mask"].cuda(),\n                        labels=labels,\n                    ).loss\n                    (loss / len(window)).backward()\n                    losses.append(float(loss.detach().cpu()))\n                torch.nn.utils.clip_grad_norm_(params, 0.1)\n                optimizer.step()\n        model.eval()\n        measured = coco_metrics(val, detections(root, model, processor, val))\n        ap50 = measured["ap50"]\n        if ap50 is None:\n            raise ValueError("Validation detection AP undefined")\n        history.append(\n            {\n                "epoch": epoch,\n                "validation_ap50": ap50,\n                "training_loss": float(np.mean(losses)) if losses else None,\n            }\n        )\n        if ap50 > best:\n            best, selected = ap50, epoch\n            tensors = {\n                key: value.detach().cpu().contiguous().clone()\n                for key, value in model.state_dict().items()\n                if not key.startswith("model.backbone.")\n            }\n            save_file(tensors, str(folder / "adapter.safetensors"))\n    write(\n        folder / "manifest.json",\n        {\n            "format": "dimer-rice-detr-adapter-v1",\n            "base": list(PINS["detr"]),\n            "classes": ["target_pest"],\n            "sha256": core.sha256(folder / "adapter.safetensors"),\n            "epoch": selected,\n            "preprocessing": dict(processor.size),\n            "frozen_prefix": "model.backbone.",\n            "trainable_parameters": sum(p.numel() for p in params),\n        },\n    )\n    write(root / "outputs" / "detector_history.json", history)\n    del model\n    gc.collect()\n    torch.cuda.empty_cache()\n\n\ndef count_reference(records):\n    return np.array(\n        [[sum(o["species"] == s for o in active_objects(r)) for s in SPECIES] for r in records], dtype=int\n    )\n\n\ndef compose(root, records):\n    """Infer full images sequentially; reference annotations are not model inputs."""\n    torch = torch_runtime()\n    model, processor = detr_model(root, adapted=True)\n    predicted = detections(root, model, processor, records)\n    del model, processor\n    gc.collect()\n    torch.cuda.empty_cache()\n    model, transform, tokenizer = bioclip(root)\n    head = load_head(root)\n    with torch.inference_mode():\n        text = (\n            model.encode_text(\n                tokenizer(["a photo of a rice black bug.", "a photo of a white stemborer."]).to(DEVICE),\n                normalize=True,\n            )\n            .cpu()\n            .numpy()\n        )\n    scale = float(model.logit_scale.exp().item())\n    if not (np.isfinite(text).all() and math.isfinite(scale)):\n        raise ValueError("Nonfinite BioCLIP text embeddings or logit scale")\n    for record, output in zip(records, predicted, strict=True):\n        original = image(root, record)\n        # Clip only predicted crop extraction, retaining original predicted boxes\n        # for evaluation. Reference invalid boxes are refused at qualification.\n        crop_images = []\n        for box in output["boxes"]:\n            x1, y1, x2, y2 = box\n            bounded = (\n                math.floor(max(0, min(original.width - 1, x1))),\n                math.floor(max(0, min(original.height - 1, y1))),\n                math.ceil(max(1, min(original.width, x2))),\n                math.ceil(max(1, min(original.height, y2))),\n            )\n            if bounded[2] <= bounded[0] or bounded[3] <= bounded[1]:\n                raise ValueError("Invalid predicted crop extent")\n            crop_images.append(original.crop(bounded))\n        vectors = embed(model, transform, crop_images)\n        if vectors.shape != (len(crop_images), head["weight"].shape[1]) or not np.isfinite(vectors).all():\n            raise ValueError(f"Invalid BioCLIP crop embeddings for {record[\'image_id\']}")\n        adapted = softmax(vectors @ head["weight"].T + head["bias"])\n        zero_shot = softmax(scale * vectors @ text.T)\n        validate_species_scores(adapted, len(crop_images))\n        validate_species_scores(zero_shot, len(crop_images))\n        output["adapted"], output["zero_shot"] = adapted.tolist(), zero_shot.tolist()\n    return predicted\n\n\ndef count_predictions(predictions, threshold, field="adapted", majority=None):\n    counts = np.zeros((len(predictions), 2), dtype=int)\n    for index, row in enumerate(predictions):\n        for score, species_scores in zip(row["scores"], row[field], strict=True):\n            if score >= threshold:\n                label = int(np.argmax(species_scores)) if majority is None else majority\n                counts[index, label] += 1\n    return counts\n\n\ndef review_policy(y, probabilities):\n    probabilities = np.asarray(probabilities)\n    margin = np.abs(probabilities[:, 0] - probabilities[:, 1])\n    rows = []\n    for threshold in sorted(set([0.0, *margin.tolist()])):\n        accepted = margin >= threshold\n        coverage = float(accepted.mean())\n        accuracy = (\n            float(np.mean(probabilities.argmax(axis=1)[accepted] == np.asarray(y)[accepted]))\n            if accepted.any()\n            else None\n        )\n        rows.append({"threshold": threshold, "coverage": coverage, "accuracy": accuracy})\n    # Specification section 7, decision 5: target >=80% selective accuracy at >=50% coverage;\n    # if unattainable, maximise accuracy, then coverage, then margin over every nonempty set.\n    target = [r for r in rows if r["coverage"] >= 0.5 and r["accuracy"] >= 0.8]\n    candidates = target or [r for r in rows if r["accuracy"] is not None]\n    if not candidates:\n        raise ValueError("Review policy needs at least one validation crop")\n    selected = (\n        max(target, key=lambda r: (r["coverage"], r["threshold"]))\n        if target\n        else max(candidates, key=lambda r: (r["accuracy"], r["coverage"], r["threshold"]))\n    )\n    rule = (\n        "target met: highest coverage with >=80% selective accuracy and >=50% coverage"\n        if target\n        else "fallback: target unattainable; maximise accuracy, then coverage, then margin"\n    )\n    return {**selected, "target_met": bool(target), "selection_rule": rule, "curve": rows}\n\n\ndef policy(root):\n    manifest, records = context(root)\n    val = [r for r in records if r["split"] == "validation"]\n    predictions = compose(root, val)\n    write(root / "outputs" / "validation_detections.json", predictions)\n    rows = [\n        {\n            "threshold": i / 20,\n            **core.count_metrics(count_reference(val), count_predictions(predictions, i / 20)),\n        }\n        for i in range(1, 20)\n    ]\n    selected = core.select_count_threshold(rows)\n    lineage = read(root / "outputs" / "lineage.json")\n    ix = np.flatnonzero([r["split"] == "validation" for r in lineage])\n    y = np.array([SPECIES.index(r["species"]) for r in lineage])[ix]\n    review = review_policy(y, np.load(root / "outputs" / "adapted.npy", allow_pickle=False)[ix])\n    result = {\n        "scope": manifest.get("scope"),\n        "detector_threshold": selected["threshold"],\n        "review_margin": review["threshold"],\n        "review_target_met": review["target_met"],\n        "review_selection_rule": review["selection_rule"],\n        "review_validation_coverage": review["coverage"],\n        "review_validation_accuracy": review["accuracy"],\n        "selection_split": "validation",\n        "selected_at_unix": time.time(),\n        "classifier_sha256": core.sha256(root / "outputs" / "classifier_adapter" / "head.safetensors"),\n        "detector_sha256": core.sha256(root / "outputs" / "detector_adapter" / "adapter.safetensors"),\n        "data_sha256": core.sha256(root / "data_manifest.json"),\n        "validation_threshold_grid": rows,\n        "review_validation": review,\n    }\n    result["digest"] = core.digest_json(result)\n    write(root / "outputs" / "selected_policy.json", result)\n\n\ndef locked_policy(root):\n    policy = read(root / "outputs" / "selected_policy.json")\n    digest = policy.pop("digest")\n    if core.digest_json(policy) != digest or policy["selection_split"] != "validation":\n        raise ValueError("Policy lock changed")\n    for key, path in (\n        ("classifier_sha256", "outputs/classifier_adapter/head.safetensors"),\n        ("detector_sha256", "outputs/detector_adapter/adapter.safetensors"),\n        ("data_sha256", "data_manifest.json"),\n    ):\n        if policy[key] != core.sha256(root / path):\n            raise ValueError("Selected checkpoint/data changed after policy lock")\n    return {**policy, "digest": digest}\n\n\ndef oracle_counts(records, lineage, probabilities):\n    by_id = {row["image_id"]: i for i, row in enumerate(records)}\n    counts = np.zeros((len(records), 2), dtype=int)\n    for row, scores in zip(lineage, probabilities, strict=True):\n        if row["parent_image_id"] in by_id:\n            counts[by_id[row["parent_image_id"]], int(np.argmax(scores))] += 1\n    return counts\n\n\ndef chosen_objects(prediction, policy):\n    result = []\n    for k, (box, detection_score, scores) in enumerate(\n        zip(prediction["boxes"], prediction["scores"], prediction["adapted"], strict=True)\n    ):\n        if detection_score < policy["detector_threshold"]:\n            continue\n        margin = abs(scores[0] - scores[1])\n        result.append(\n            {\n                "query_rank": k,\n                "bbox_xyxy": box,\n                "detector_score": detection_score,\n                "species": SPECIES[int(np.argmax(scores))],\n                "species_scores": scores,\n                "margin": margin,\n                "review": margin < policy["review_margin"],\n            }\n        )\n    return result\n\n\nCHART_FIGURES = ("classifier.png", "detector.png", "evaluate.png", "activity.png")\nSHORT = {"rice_black_bug": "RBB", "white_stemborer": "WSB"}\nOUTCOMES = ("correct", "wrong_species", "missed", "spurious")\n\n\ndef object_outcomes(record, objects):\n    """One row per reference or retained prediction: correct, wrong_species, missed or spurious."""\n    references = active_objects(record)\n    err = core.error_decomposition(references, objects)\n    rows = []\n\n    def row(outcome, reference, prediction, iou):\n        scores = prediction["species_scores"] if prediction else None\n        return {\n            "image_id": record["image_id"],\n            "outcome": outcome,\n            "reference_species": reference["species"] if reference else "",\n            "predicted_species": prediction["species"] if prediction else "",\n            "detector_score": round(prediction["detector_score"], 4) if prediction else "",\n            "species_score": round(max(scores), 4) if prediction else "",\n            "species_margin": round(prediction["margin"], 4) if prediction else "",\n            "review": ("REVIEW" if prediction["review"] else "accepted") if prediction else "",\n            "iou": round(iou, 3) if iou is not None else "",\n            "reference_box": reference["bbox_xyxy"] if reference else None,\n            "predicted_box": prediction["bbox_xyxy"] if prediction else None,\n        }\n\n    for r, q, iou in err["matches"]:\n        same = references[r]["species"] == objects[q]["species"]\n        rows.append(row("correct" if same else "wrong_species", references[r], objects[q], iou))\n    rows += [row("missed", references[r], None, None) for r in err["misses"]]\n    rows += [row("spurious", None, objects[q], None) for q in err["spurious"]]\n    return rows\n\n\ndef species_reconciliation(image_id, rows, objects):\n    """Per-species identities: reference = correct + wrong_out + missed;\n    raw = correct + wrong_in + spurious."""\n    result = []\n    for species in SPECIES:\n        entry = {\n            "image_id": image_id,\n            "species": species,\n            "reference": sum(r["reference_species"] == species for r in rows),\n            "raw": sum(o["species"] == species for o in objects),\n            "accepted": sum(o["species"] == species and not o["review"] for o in objects),\n            "referred": sum(o["species"] == species and o["review"] for o in objects),\n            "correct": sum(r["outcome"] == "correct" and r["reference_species"] == species for r in rows),\n            "wrong_in": sum(\n                r["outcome"] == "wrong_species" and r["predicted_species"] == species for r in rows\n            ),\n            "wrong_out": sum(\n                r["outcome"] == "wrong_species" and r["reference_species"] == species for r in rows\n            ),\n            "missed": sum(r["outcome"] == "missed" and r["reference_species"] == species for r in rows),\n            "spurious": sum(r["outcome"] == "spurious" and r["predicted_species"] == species for r in rows),\n        }\n        if (\n            entry["reference"] != entry["correct"] + entry["wrong_out"] + entry["missed"]\n            or entry["raw"] != entry["correct"] + entry["wrong_in"] + entry["spurious"]\n        ):\n            raise ValueError("Error decomposition does not reconcile with counts")\n        result.append(entry)\n    return result\n\n\ndef error_summaries(root, records, predictions, policy_value):\n    """Test-set decomposition learners read beside the panels: per-species counts and confusion."""\n    totals = {\n        s: dict.fromkeys(\n            (\n                "reference",\n                "raw",\n                "accepted",\n                "referred",\n                "correct",\n                "wrong_in",\n                "wrong_out",\n                "missed",\n                "spurious",\n            ),\n            0,\n        )\n        for s in SPECIES\n    }\n    confusion = np.zeros((3, 3), dtype=int)  # rows: reference RBB/WSB/none; cols: predicted RBB/WSB/none\n    for record, prediction in zip(records, predictions, strict=True):\n        objects = chosen_objects(prediction, policy_value)\n        rows = object_outcomes(record, objects)\n        for entry in species_reconciliation(record["image_id"], rows, objects):\n            for key in totals[entry["species"]]:\n                totals[entry["species"]][key] += entry[key]\n        for r in rows:\n            ref = SPECIES.index(r["reference_species"]) if r["reference_species"] else 2\n            pred = SPECIES.index(r["predicted_species"]) if r["predicted_species"] else 2\n            confusion[ref, pred] += 1\n    summary = [{"species": s, **totals[s]} for s in SPECIES]\n    summary.append({"species": "all", **{k: sum(t[k] for t in totals.values()) for k in totals[SPECIES[0]]}})\n    csv_write(root / "outputs" / "error_summary.csv", summary)\n    labels = [*SPECIES, "no match"]\n    csv_write(\n        root / "outputs" / "matched_species_confusion.csv",\n        [\n            {\n                "reference": ("reference " + labels[i]) if i < 2 else "no reference (spurious)",\n                **{\n                    f"predicted {labels[j]}" if j < 2 else "not detected (missed)": int(confusion[i, j])\n                    for j in range(3)\n                },\n            }\n            for i in range(3)\n        ],\n    )\n\n\ndef evaluate(root):\n    manifest, records = context(root)\n    policy_value = locked_policy(root)\n    test = [r for r in records if r["split"] == "test"]\n    predictions = compose(root, test)\n    write(root / "outputs" / "test_detections.json", predictions)\n    lineage = read(root / "outputs" / "lineage.json")\n    features_data = np.load(root / "outputs" / "features.npz", allow_pickle=False)\n    adapted = np.load(root / "outputs" / "adapted.npy", allow_pickle=False)\n    train = [r for r in records if r["split"] == "train"]\n    majority = int(count_reference(train).sum(axis=0).argmax())\n    baseline = np.tile(count_reference(train).mean(axis=0), (len(test), 1))\n    systems = {\n        "training_mean": baseline,\n        "zero_count": np.zeros_like(baseline),\n        "oracle_bioclip": oracle_counts(test, lineage, features_data["zero_shot"]),\n        "oracle_adapted": oracle_counts(test, lineage, adapted),\n        "detector_majority": count_predictions(\n            predictions, policy_value["detector_threshold"], majority=majority\n        ),\n        "detector_bioclip": count_predictions(\n            predictions, policy_value["detector_threshold"], field="zero_shot"\n        ),\n        "detector_adapted": count_predictions(predictions, policy_value["detector_threshold"]),\n    }\n    truth = count_reference(test)\n    metrics = {name: core.count_metrics(truth, counts) for name, counts in systems.items()}\n    bootstrap = core.paired_group_bootstrap(\n        truth, systems["detector_adapted"], baseline, [r["capture_group_id"] for r in test]\n    )\n    bootstrap["group_interpretation"] = (\n        "source-image/duplicate groups; capture independence unverified"\n        if manifest.get("scope") == "exploratory_published_annotations"\n        else "verified capture groups"\n    )\n    write(root / "outputs" / "metrics.json", metrics)\n    write(root / "outputs" / "bootstrap.json", bootstrap)\n    csv_write(\n        root / "outputs" / "metrics.csv",\n        [\n            {"system": name, **{k: v for k, v in value.items() if not isinstance(v, list)}}\n            for name, value in metrics.items()\n        ],\n    )\n    crop_y = np.array([SPECIES.index(r["species"]) for r in lineage])\n    ix = np.flatnonzero([r["split"] == "test" for r in lineage])\n    crop_methods = {\n        "bioclip": features_data["zero_shot"],\n        "adapted": adapted,\n        "siglip": np.load(root / "outputs" / "siglip.npy", allow_pickle=False),\n        "rgb": np.load(root / "outputs" / "rgb.npy", allow_pickle=False),\n        "majority": np.load(root / "outputs" / "majority.npy", allow_pickle=False),\n    }\n    write(\n        root / "outputs" / "test_crop_metrics.json",\n        {key: classification(crop_y[ix], score[ix]) for key, score in crop_methods.items()},\n    )\n    csv_write(\n        root / "outputs" / "crop_predictions.csv",\n        [\n            {\n                "crop_id": lineage[i]["crop_id"],\n                "system": key,\n                "reference": SPECIES[crop_y[i]],\n                "predicted": SPECIES[int(score[i].argmax())],\n                "score_rbb": float(score[i, 0]),\n                "score_wsb": float(score[i, 1]),\n            }\n            for key, score in crop_methods.items()\n            for i in ix\n        ],\n    )\n    decomposition, counts_rows, detection_rows = [], [], []\n    for index, (record, prediction) in enumerate(zip(test, predictions, strict=True)):\n        objects = chosen_objects(prediction, policy_value)\n        errors = core.error_decomposition(record["objects"], objects)\n        decomposition.append(\n            {\n                "image_id": record["image_id"],\n                "missed": errors["missed"],\n                "spurious": errors["spurious_count"],\n                "wrong_species": errors["wrong_species"],\n                "matches": len(errors["matches"]),\n                "matched_confusion": json.dumps(errors["matched_confusion"]),\n            }\n        )\n        for s, species in enumerate(SPECIES):\n            counts_rows.append(\n                {\n                    "image_id": record["image_id"],\n                    "capture_group_id": record["capture_group_id"],\n                    "species": species,\n                    "reference": int(truth[index, s]),\n                    "raw": int(systems["detector_adapted"][index, s]),\n                    "accepted": sum(o["species"] == species and not o["review"] for o in objects),\n                    "unresolved_total": sum(o["review"] for o in objects),\n                    "training_mean": float(baseline[index, s]),\n                }\n            )\n        detection_rows.append({"image_id": record["image_id"], "detections": objects})\n    csv_write(root / "outputs" / "counts.csv", counts_rows)\n    csv_write(root / "outputs" / "error_decomposition.csv", decomposition)\n    with (root / "outputs" / "detections.jsonl").open("w", encoding="utf-8") as handle:\n        for row in detection_rows:\n            handle.write(json.dumps(row) + "\\n")\n    detection_metrics = coco_metrics(test, predictions)\n    matched = sum(x["matches"] for x in decomposition)\n    misses = sum(x["missed"] for x in decomposition)\n    spurious = sum(x["spurious"] for x in decomposition)\n    detection_metrics.update(\n        precision=matched / (matched + spurious) if matched + spurious else None,\n        recall=matched / (matched + misses) if matched + misses else None,\n    )\n    write(root / "outputs" / "detection_metrics.json", detection_metrics)\n    error_summaries(root, test, predictions, policy_value)\n    write(\n        root / "outputs" / "reload_expected.json",\n        {"pid": os.getpid(), "policy_digest": policy_value["digest"], "predictions": predictions[:2]},\n    )\n    panels(root, test, predictions, policy_value)\n\n\nPANEL_STYLE = {\n    "correct": ("#1a9850", "-"),\n    "wrong_species": ("#d01c8b", "-"),\n    "spurious": ("#ff7f00", "-"),\n    "missed": ("#00bfff", "--"),\n}\n\n\ndef panels(root, records, predictions, policy_value):\n    import matplotlib.pyplot as plt\n    from matplotlib.lines import Line2D\n    from matplotlib.patches import Rectangle\n\n    folder = root / "outputs" / "local_figures"\n    folder.mkdir(exist_ok=True)\n    selected, found = [], {}\n    for record, prediction in zip(records, predictions, strict=True):\n        err = core.error_decomposition(record["objects"], chosen_objects(prediction, policy_value))\n        categories = {\n            "misses": err["missed"] > 0,\n            "spurious": err["spurious_count"] > 0,\n            "wrong_species": err["wrong_species"] > 0,\n            "success": err["missed"] + err["spurious_count"] + err["wrong_species"] == 0,\n        }\n        for category, present in categories.items():\n            if present and category not in found:\n                found[category] = record["image_id"]\n                if record["image_id"] not in [r[0]["image_id"] for r in selected]:\n                    selected.append((record, prediction))\n    absent = sorted(set(("misses", "spurious", "wrong_species", "success")) - set(found))\n    write(\n        root / "outputs" / "error_panel_inventory.json",\n        {\n            "selected": found,\n            "absent_categories": absent,\n            "note": "An absent category was not found in the held-out split; none is manufactured.",\n        },\n    )\n    example_rows, example_counts = [], []\n    for record, prediction in selected:\n        objects = chosen_objects(prediction, policy_value)\n        rows = object_outcomes(record, objects)\n        reconciled = species_reconciliation(record["image_id"], rows, objects)\n        example_counts += reconciled\n        shown_for = [c for c, image_id in found.items() if image_id == record["image_id"]]\n        fig, ax = plt.subplots(figsize=(10, 8.5))\n        ax.imshow(image(root, record))\n        tag = 0\n        # Crowded trays get compact #tags; the table below carries the full labels and scores.\n        compact = sum(r["outcome"] != "correct" for r in rows) > 12\n        for row in sorted(rows, key=lambda r: OUTCOMES.index(r["outcome"])):\n            colour, style = PANEL_STYLE[row["outcome"]]\n            box = row["predicted_box"] or row["reference_box"]\n            x1, y1, x2, y2 = box\n            ax.add_patch(\n                Rectangle(\n                    (x1, y1),\n                    x2 - x1,\n                    y2 - y1,\n                    fill=False,\n                    edgecolor=colour,\n                    linestyle=style,\n                    linewidth=1.6 if row["outcome"] != "correct" else 0.9,\n                )\n            )\n            if row["outcome"] == "correct":\n                continue\n            tag += 1\n            row["tag"] = tag\n            if row["outcome"] == "wrong_species":\n                rx1, ry1, rx2, ry2 = row["reference_box"]\n                ax.add_patch(\n                    Rectangle(\n                        (rx1, ry1),\n                        rx2 - rx1,\n                        ry2 - ry1,\n                        fill=False,\n                        edgecolor="white",\n                        linestyle="--",\n                        linewidth=0.8,\n                    )\n                )\n                text = (\n                    f"#{tag} ref {SHORT[row[\'reference_species\']]} → pred {SHORT[row[\'predicted_species\']]}"\n                )\n            elif row["outcome"] == "missed":\n                text = f"#{tag} missed ref {SHORT[row[\'reference_species\']]}"\n            else:\n                text = f"#{tag} extra pred {SHORT[row[\'predicted_species\']]}"\n            if row["review"] == "REVIEW":\n                text += " REVIEW"\n            if compact:\n                ax.text(x1, y1, f"#{tag}", fontsize=5, color="black", backgroundcolor=colour, va="top")\n            else:\n                ax.text(x1, y1 - 2, text, fontsize=6, color="black", backgroundcolor=colour, va="bottom")\n        for row in rows:\n            if row["outcome"] != "correct":\n                example_rows.append({"panel_tag": row["tag"], **{k: v for k, v in row.items() if k != "tag"}})\n        totals = {k: sum(e[k] for e in reconciled) for k in ("reference", "raw", "accepted", "referred")}\n        counted = {o: sum(r["outcome"] == o for r in rows) for o in OUTCOMES}\n        ax.set_title(\n            f"Test {record[\'image_id\']} · example of: {\', \'.join(shown_for)}\\n"\n            f"reference {totals[\'reference\']} · raw predicted {totals[\'raw\']} · accepted {totals[\'accepted\']}"\n            f" · referred {totals[\'referred\']} | correct {counted[\'correct\']} · wrong species "\n            f"{counted[\'wrong_species\']} · missed {counted[\'missed\']} · spurious {counted[\'spurious\']}\\n"\n            + (\n                "Crowded image: #tags only; see the error-example table for species and scores\\n"\n                if compact\n                else ""\n            )\n            + record["attribution"],\n            fontsize=8,\n        )\n        handles = [\n            Line2D([], [], color=PANEL_STYLE["correct"][0], label="correct species (matched)"),\n            Line2D(\n                [],\n                [],\n                color=PANEL_STYLE["wrong_species"][0],\n                label="wrong species (matched; white = reference)",\n            ),\n            Line2D([], [], color=PANEL_STYLE["missed"][0], linestyle="--", label="missed reference box"),\n            Line2D([], [], color=PANEL_STYLE["spurious"][0], label="spurious prediction (no reference)"),\n        ]\n        ax.legend(\n            handles=handles,\n            loc="upper center",\n            bbox_to_anchor=(0.5, -0.01),\n            ncol=2,\n            fontsize=7,\n            title="RBB = rice black bug · WSB = white stemborer · scores per #tag in the table below",\n            title_fontsize=7,\n            frameon=False,\n        )\n        ax.axis("off")\n        fig.savefig(folder / f"{record[\'image_id\']}.png", dpi=120, bbox_inches="tight")\n        plt.close(fig)\n    fields = [\n        "panel_tag",\n        "image_id",\n        "outcome",\n        "reference_species",\n        "predicted_species",\n        "detector_score",\n        "species_score",\n        "species_margin",\n        "review",\n        "iou",\n    ]\n    csv_write(\n        root / "outputs" / "error_examples.csv", [{k: r[k] for k in fields} for r in example_rows], fields\n    )\n    csv_write(\n        root / "outputs" / "error_example_counts.csv",\n        example_counts,\n        list(species_reconciliation("", [], [])[0]),\n    )\n\n\ndef activity(root):\n    _, records = context(root)\n    test = [r for r in records if r["split"] == "test"]\n    canonical = core.sha256(root / "outputs" / "test_detections.json")\n    policy_value = locked_policy(root)\n    predictions = read(root / "outputs" / "test_detections.json")\n    locked = policy_value["detector_threshold"]\n    thresholds = sorted({max(0.0, locked - 0.1), locked, min(1.0, locked + 0.1)})\n    role = {t: "canonical" if t == locked else ("lower" if t < locked else "higher") for t in thresholds}\n    rows, per_image = [], {}\n    for threshold in thresholds:\n        count_result = core.count_metrics(count_reference(test), count_predictions(predictions, threshold))\n        errors = [\n            core.error_decomposition(\n                r["objects"], chosen_objects(p, {**policy_value, "detector_threshold": threshold})\n            )\n            for r, p in zip(test, predictions, strict=True)\n        ]\n        per_image[threshold] = count_predictions(predictions, threshold)\n        rows.append(\n            {\n                "display_threshold": threshold,\n                "role": role[threshold],\n                "mae": count_result["mae"],\n                "missed": sum(e["missed"] for e in errors),\n                "spurious": sum(e["spurious_count"] for e in errors),\n                "wrong_species": sum(e["wrong_species"] for e in errors),\n                "note": "Retrospective test illustration; canonical policy unchanged",\n            }\n        )\n    csv_write(root / "outputs" / "activity_thresholds.csv", rows)\n    # Paired view: the held-out image whose raw count moves most across the three\n    # settings (earliest on ties), so a learner can follow one photograph end to end.\n    spread = [\n        int(max(per_image[t][i].sum() for t in thresholds) - min(per_image[t][i].sum() for t in thresholds))\n        for i in range(len(test))\n    ]\n    chosen = int(np.argmax(spread)) if test else None\n    paired = []\n    if chosen is not None:\n        record, prediction = test[chosen], predictions[chosen]\n        for threshold in thresholds:\n            objects = chosen_objects(prediction, {**policy_value, "detector_threshold": threshold})\n            for entry in species_reconciliation(\n                record["image_id"], object_outcomes(record, objects), objects\n            ):\n                paired.append(\n                    {\n                        "image_id": record["image_id"],\n                        "display_threshold": threshold,\n                        "role": role[threshold],\n                        "species": entry["species"],\n                        "reference": entry["reference"],\n                        "raw": entry["raw"],\n                        "missed": entry["missed"],\n                        "spurious": entry["spurious"],\n                        "wrong_species": entry["wrong_out"],\n                    }\n                )\n    csv_write(\n        root / "outputs" / "activity_paired_counts.csv",\n        paired,\n        [\n            "image_id",\n            "display_threshold",\n            "role",\n            "species",\n            "reference",\n            "raw",\n            "missed",\n            "spurious",\n            "wrong_species",\n        ],\n    )\n    if (\n        core.sha256(root / "outputs" / "test_detections.json") != canonical\n        or locked_policy(root)["digest"] != policy_value["digest"]\n    ):\n        raise ValueError("Activity changed canonical predictions or policy")\n\n\ndef reload(root):\n    _, records = context(root)\n    policy_value = locked_policy(root)\n    expected = read(root / "outputs" / "reload_expected.json")\n    if expected["pid"] == os.getpid() or expected["policy_digest"] != policy_value["digest"]:\n        raise ValueError("Reload requires a fresh process and unchanged policy")\n    ids = [p["image_id"] for p in expected["predictions"]]\n    by_id = {r["image_id"]: r for r in records}\n    actual = compose(root, [by_id[key] for key in ids])\n    for left, right in zip(expected["predictions"], actual, strict=True):\n        for field, atol in (("boxes", 1e-3), ("scores", 1e-5), ("adapted", 1e-5), ("zero_shot", 1e-5)):\n            if not np.allclose(left[field], right[field], atol=atol, rtol=1e-5):\n                raise ValueError(f"Reload parity failed for {field}")\n        lobjects, robjects = chosen_objects(left, policy_value), chosen_objects(right, policy_value)\n        if [(o["query_rank"], o["species"], o["review"]) for o in lobjects] != [\n            (o["query_rank"], o["species"], o["review"]) for o in robjects\n        ]:\n            raise ValueError("Reload threshold/label/referral decisions changed")\n    write(\n        root / "outputs" / "verification.json",\n        {\n            "fresh_process": True,\n            "reference_pid": expected["pid"],\n            "reload_pid": os.getpid(),\n            "original_image_ids": ids,\n            "box_atol": 1e-3,\n            "score_atol": 1e-5,\n            "rtol": 1e-5,\n            "identical_labels_counts_referrals": True,\n        },\n    )\n\n\ndef report(root):\n    manifest, records = context(root)\n    policy_value = locked_policy(root)\n    out = root / "outputs"\n    verified = read(out / "verification.json")\n    if not verified.get("fresh_process"):\n        raise ValueError("Fresh-process verification missing")\n    with (out / "counts.csv").open(encoding="utf-8", newline="") as handle:\n        count_rows = list(csv.DictReader(handle))\n    if not count_rows or len(count_rows) % 2:\n        raise ValueError("Count CSV row inventory mismatch")\n    csv_reference = np.array([int(row["reference"]) for row in count_rows]).reshape(-1, 2)\n    csv_prediction = np.array([int(row["raw"]) for row in count_rows]).reshape(-1, 2)\n    recomputed = core.count_metrics(csv_reference, csv_prediction)\n    if recomputed != read(out / "metrics.json")["detector_adapted"]:\n        raise ValueError("Count CSV-to-metric parity failed")\n    test = [record for record in records if record["split"] == "test"]\n    expected_keys = [(record["image_id"], species) for record in test for species in SPECIES]\n    if [(row["image_id"], row["species"]) for row in count_rows] != expected_keys:\n        raise ValueError("CSV identifiers/class order differ from manifest")\n    if not np.array_equal(csv_reference, count_reference(test)):\n        raise ValueError("CSV reference counts differ from annotations")\n    predictions = read(out / "test_detections.json")\n    if [p["image_id"] for p in predictions] != [r["image_id"] for r in test]:\n        raise ValueError("Raw prediction identifiers differ from manifest")\n    if not np.array_equal(csv_prediction, count_predictions(predictions, policy_value["detector_threshold"])):\n        raise ValueError("Raw prediction-to-count CSV parity failed")\n    for index, row in enumerate(predictions):\n        objects = chosen_objects(row, policy_value)\n        for species_index, species in enumerate(SPECIES):\n            exported = count_rows[2 * index + species_index]\n            if int(exported["accepted"]) != sum(\n                o["species"] == species and not o["review"] for o in objects\n            ) or int(exported["unresolved_total"]) != sum(o["review"] for o in objects):\n                raise ValueError("Referral count parity failed")\n    verified["csv_metric_parity"] = True\n    # Chart PNGs carry no source photographs; previews and annotated panels stay excluded.\n    charts = [out / "figures" / name for name in CHART_FIGURES if (out / "figures" / name).is_file()]\n    write(\n        out / "run_summary.json",\n        {\n            "status": "Executed; maintainer review required for release",\n            "scope": manifest.get("scope"),\n            "policy_digest": policy_value["digest"],\n            "limitations": [\n                "Published annotations are not independently verified",\n                "Capture independence and completeness are unverified",\n                "Source-image/duplicate groups do not establish capture independence",\n                "Pretraining overlap is unresolved",\n                "Counts are image contents, not field abundance or intervention advice",\n            ]\n            if manifest.get("scope") == "exploratory_published_annotations"\n            else [\n                "Pretraining overlap unresolved",\n                "Small held-out sample does not establish field validity",\n            ],\n            "metrics": read(out / "metrics.json"),\n            "verification": verified,\n            "charts_included": [path.relative_to(out).as_posix() for path in charts],\n            "stage_resources": {\n                name: read(out / f"receipt_{name}.json")\n                for name in STAGES[:-1]\n                if (out / f"receipt_{name}.json").is_file()\n            },\n        },\n    )\n    for name in ("data_manifest.json", "dataset_audit.json", "model_manifest.json"):\n        shutil.copyfile(root / name, out / name)\n    (out / "DATA_LICENSE.txt").write_text(\n        "Dataset: https://doi.org/10.5281/zenodo.20066074\\n"\n        "Creative Commons Attribution 4.0 International\\n"\n        "https://creativecommons.org/licenses/by/4.0/\\n"\n        "Attributions for each selected source image are recorded in data_manifest.json.\\n"\n        "Source images, annotated image panels and photo previews are excluded from this archive;\\n"\n        "only the metric charts listed in run_summary.json (no photographs) are included.\\n",\n        encoding="utf-8",\n    )\n    files = sorted(\n        p\n        for p in out.rglob("*")\n        if p.is_file()\n        and "local_figures" not in p.parts\n        and p.name not in {"results.zip", "checksums.json"}\n        and p.suffix in {".json", ".jsonl", ".csv", ".safetensors", ".txt"}\n    )\n    files = sorted(files + charts)\n    if sum(p.stat().st_size for p in files) > 100 * 1024**2:\n        raise ValueError("Derived artifact archive exceeds the 100 MiB bound")\n    checksums = {p.relative_to(out).as_posix(): core.sha256(p) for p in files}\n    write(out / "checksums.json", checksums)\n    with zipfile.ZipFile(out / "results.zip", "w", zipfile.ZIP_DEFLATED) as archive:\n        for path in files + [out / "checksums.json"]:\n            archive.write(path, path.relative_to(out).as_posix())\n    import hashlib\n\n    with zipfile.ZipFile(out / "results.zip") as archive:\n        if archive.testzip() is not None:\n            raise ValueError("ZIP CRC verification failed")\n        for name, digest in checksums.items():\n            if hashlib.sha256(archive.read(name)).hexdigest() != digest:\n                raise ValueError("ZIP member digest mismatch")\n\n\ndef execution_identity(root):\n    paths = [\n        root / "data_manifest.json",\n        root / "dataset_audit.json",\n        root / "model_manifest.json",\n        Path(__file__),\n        Path(core.__file__),\n    ]\n    paths.extend(\n        p\n        for p in (\n            root / "requirements.txt",\n            root / "source.json",\n            root / "rice_assets.py",\n            root / "rice_figures.py",\n            Path(__file__).parent / "rice_assets.py",\n            Path(__file__).parent / "rice_figures.py",\n        )\n        if p.is_file()\n    )\n    return core.digest_json({str(p.name): core.sha256(p) for p in paths})\n\n\ndef verify_receipts(root, before_stage):\n    """Refuse stale source, artifacts or partially rerun dependency chains."""\n    identity = execution_identity(root)\n    previous = None\n    for stage in STAGES[: STAGES.index(before_stage)]:\n        receipt_path = root / "outputs" / f"receipt_{stage}.json"\n        receipt = read(receipt_path)\n        if (\n            not receipt.get("completed")\n            or receipt.get("identity") != identity\n            or receipt.get("previous") != previous\n        ):\n            raise ValueError(f"Missing/stale {stage} receipt; rerun from the first changed stage")\n        for relative, digest in receipt.get("outputs", {}).items():\n            if core.sha256(core.safe_path(root / "outputs", relative)) != digest:\n                raise ValueError(f"Changed output from {stage}: {relative}")\n        previous = core.sha256(receipt_path)\n    return previous\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--root", type=Path, required=True)\n    parser.add_argument("--stage", choices=STAGES, required=True)\n    args = parser.parse_args()\n    root = args.root.resolve()\n    (root / "outputs").mkdir(parents=True, exist_ok=True)\n    context(root)\n    previous = verify_receipts(root, args.stage)\n    out = root / "outputs"\n    before = {p.relative_to(out).as_posix(): core.sha256(p) for p in out.rglob("*") if p.is_file()}\n    prior_receipt = out / f"receipt_{args.stage}.json"\n    previously_owned = set(read(prior_receipt).get("outputs", {})) if prior_receipt.exists() else set()\n    # Remove downstream receipts only, preserving artifacts for diagnosis. Their\n    # absence prevents stale outputs from masquerading as a completed fresh run.\n    for stage in STAGES[STAGES.index(args.stage) :]:\n        (out / f"receipt_{stage}.json").unlink(missing_ok=True)\n    started = time.monotonic()\n    if args.stage in {"features", "classifier", "detector", "policy", "evaluate", "reload"}:\n        torch_runtime().cuda.reset_peak_memory_stats()\n    globals()[args.stage](root)\n    produced = {\n        p.relative_to(out).as_posix(): core.sha256(p)\n        for p in out.rglob("*")\n        if p.is_file() and not p.name.startswith("receipt_")\n    }\n    changed = {\n        name: digest\n        for name, digest in produced.items()\n        if before.get(name) != digest or name in previously_owned\n    }\n    write(\n        root / "outputs" / f"receipt_{args.stage}.json",\n        {\n            "stage": args.stage,\n            "seconds": time.monotonic() - started,\n            "pid": os.getpid(),\n            "data_sha256": core.sha256(root / "data_manifest.json"),\n            "runtime_sha256": core.sha256(Path(__file__)),\n            "identity": execution_identity(root),\n            "previous": previous,\n            "outputs": changed,\n            "completed": True,\n            "peak_allocated_gpu_bytes": int(__import__("torch").cuda.max_memory_allocated())\n            if args.stage in {"features", "classifier", "detector", "policy", "evaluate", "reload"}\n            else None,\n            "gpu_name": __import__("torch").cuda.get_device_name(0)\n            if args.stage in {"features", "classifier", "detector", "policy", "evaluate", "reload"}\n            else None,\n        },\n    )\n\n\nif __name__ == "__main__":\n    main()\n', 'rice_core.py': '"""Pure CPU contracts for the rice-pest capstone; these do not certify source data."""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport re\nfrom pathlib import Path, PurePosixPath\n\nimport numpy as np\nfrom scipy.optimize import linear_sum_assignment\n\nSPECIES = ("rice_black_bug", "white_stemborer")\nROLES = ("train", "validation", "test")\n\n\ndef sha256(path: Path) -> str:\n    """Hash a file without loading it into memory."""\n    with Path(path).open("rb") as stream:\n        return hashlib.file_digest(stream, "sha256").hexdigest()\n\n\ndef digest_json(value: object) -> str:\n    """Hash canonical JSON, refusing NaN and infinity."""\n    return hashlib.sha256(\n        json.dumps(value, sort_keys=True, separators=(",", ":"), ensure_ascii=False, allow_nan=False).encode()\n    ).hexdigest()\n\n\ndef safe_path(root: Path, name: str) -> Path:\n    """Refuse portable archive traversal and symlink escapes."""\n    if not isinstance(name, str) or not name or "\\\\" in name or ":" in name or "\\x00" in name:\n        raise ValueError("Unsafe relative path")\n    relative = PurePosixPath(name)\n    if relative.is_absolute() or any(part in ("", ".", "..") for part in name.split("/")):\n        raise ValueError("Unsafe relative path")\n    resolved = Path(root).resolve() / relative\n    resolved = resolved.resolve()\n    if not resolved.is_relative_to(Path(root).resolve()):\n        raise ValueError("Path escapes root")\n    return resolved\n\n\ndef _boxes(value: object) -> np.ndarray:\n    array = np.asarray(value, dtype=float)\n    if array.size == 0:\n        return np.empty((0, 4))\n    if array.ndim != 2 or array.shape[1] != 4 or not np.isfinite(array).all():\n        raise ValueError("Boxes must be finite N x 4 xyxy")\n    if np.any(array[:, 2:] <= array[:, :2]):\n        raise ValueError("Boxes require positive area")\n    return array\n\n\ndef validate_manifest(manifest: dict, require_capacity: bool = False) -> dict:\n    """Validate normalized references; provenance evidence still needs an independent audit.\n\n    Complete annotation assertions are required even for zero-target images.\n    Any declared specimen/session/duplicate lineage must have one split owner.\n    """\n    if manifest.get("classes") != list(SPECIES):\n        raise ValueError("Class order mismatch")\n    records = manifest.get("records", [])\n    exploratory = manifest.get("scope") == "exploratory_published_annotations"\n    if exploratory and require_capacity:\n        raise ValueError("Exploratory source families cannot certify independent capture-group capacity")\n    if not records:\n        raise ValueError("Empty image manifest")\n    ids, paths, object_ids, lineage = set(), set(), set(), {}\n    parents = {}\n    counts = {role: np.zeros(2, dtype=int) for role in ROLES}\n    groups = {role: set() for role in ROLES}\n    presence = {(role, species): set() for role in ROLES for species in SPECIES}\n    for row in records:\n        image_id, split, group = row["image_id"], row["split"], row["capture_group_id"]\n        if not image_id or image_id in ids or split not in ROLES or not group:\n            raise ValueError("Invalid image ID, role, or capture group")\n        ids.add(image_id)\n        safe_path(Path("."), row["relative_path"])\n        if row["relative_path"] in paths:\n            raise ValueError("Duplicate image path")\n        paths.add(row["relative_path"])\n        if not re.fullmatch(r"[0-9a-f]{64}", row["sha256"]):\n            raise ValueError("Invalid image hash")\n        if not 0 < row["bytes"] <= 25 * 1024**2:\n            raise ValueError("Image byte limit")\n        if any(type(row[key]) is not int or not 32 <= row[key] <= 4096 for key in ("width", "height")):\n            raise ValueError("Image dimension limit")\n        if not all(row.get(key) for key in ("source_record", "licence", "attribution")):\n            raise ValueError("Missing source attribution")\n        allowed_status = {"human_verified_complete", "human_created_complete"}\n        if exploratory:\n            allowed_status.add("published_unverified")\n        if row.get("annotation_status") not in allowed_status:\n            raise ValueError("Reference annotations must be human-verified and complete")\n        for key in (\n            "capture_group_id",\n            "sha256",\n            "decoded_sha256",\n            "duplicate_cluster_id",\n            "source_family_id",\n            "capture_session_id",\n            "specimen_id",\n            "tray_id",\n            "burst_id",\n        ):\n            value = row.get(key)\n            if value:\n                old = lineage.setdefault((key, value), split)\n                if old != split:\n                    raise ValueError(f"Cross-split lineage: {key}")\n        groups[split].add(group)\n        parents[image_id] = row\n        for obj in row["objects"]:\n            if not obj["object_id"] or obj["object_id"] in object_ids:\n                raise ValueError("Duplicate object ID")\n            object_ids.add(obj["object_id"])\n            box = _boxes([obj["bbox_xyxy"]])[0]\n            if np.any(box < 0) or box[2] > row["width"] or box[3] > row["height"]:\n                raise ValueError("Out-of-bounds reference box")\n            if obj["species"] not in SPECIES or not obj.get("original_label"):\n                raise ValueError("Unknown or missing reference label")\n            if type(obj.get("ignore")) is not bool or (obj["ignore"] and not obj.get("ignore_reason")):\n                raise ValueError("Explicit ignore flag and reason required")\n            if obj.get("sex") is not None and obj.get("sex_verified") is not True:\n                raise ValueError("Unverified sex label")\n            if not obj["ignore"]:\n                counts[split][SPECIES.index(obj["species"])] += 1\n                presence[(split, obj["species"])].add(group)\n    crop_ids = set()\n    for crop in manifest.get("crops", []):\n        if not crop["crop_id"] or crop["crop_id"] in crop_ids:\n            raise ValueError("Duplicate crop ID")\n        crop_ids.add(crop["crop_id"])\n        parent = parents.get(crop["parent_image_id"])\n        if parent is None:\n            raise ValueError("Unmapped crop")\n        obj = next((o for o in parent["objects"] if o["object_id"] == crop["parent_object_id"]), None)\n        if obj is None or obj["ignore"]:\n            raise ValueError("Invalid crop reference object")\n        if crop.get("split", parent["split"]) != parent["split"]:\n            raise ValueError("Crop split leakage")\n        box = _boxes([crop["bbox_xyxy"]])[0]\n        if np.any(box < 0) or box[2] > parent["width"] or box[3] > parent["height"]:\n            raise ValueError("Out-of-bounds crop")\n        if not crop.get("preprocessing"):\n            raise ValueError("Missing crop preprocessing")\n    if require_capacity:\n        for role, minimum_groups, minimum_instances in (\n            ("train", 6, 60),\n            ("validation", 3, 20),\n            ("test", 3, 20),\n        ):\n            if len(groups[role]) < minimum_groups or np.any(counts[role] < minimum_instances):\n                raise ValueError(f"Insufficient independent support: {role}")\n            if role != "train" and any(len(presence[(role, s)]) < 2 for s in SPECIES):\n                raise ValueError("Species requires two evaluation groups")\n    return {\n        "images": len(records),\n        "crops": len(crop_ids),\n        "objects": len(object_ids),\n        "groups": {k: len(v) for k, v in groups.items()},\n        "instances": {k: v.tolist() for k, v in counts.items()},\n        "capacity_checked": require_capacity,\n        "dataset_qualified": False,\n        "scope": manifest.get("scope", "strict_capture_group_study"),\n        "limitations": (\n            [\n                "Published annotations are unverified; completeness and label accuracy are unknown.",\n                "Source-family splits do not establish independent capture sessions or specimens.",\n            ]\n            if exploratory\n            else []\n        ),\n    }\n\n\ndef _counts(reference: object, predicted: object) -> tuple[np.ndarray, np.ndarray]:\n    truth, prediction = np.asarray(reference, dtype=float), np.asarray(predicted, dtype=float)\n    if truth.ndim != 2 or truth.shape[1] != 2 or not len(truth) or prediction.shape != truth.shape:\n        raise ValueError("Counts must be aligned nonempty N x 2 arrays")\n    if (\n        not np.isfinite(truth).all()\n        or not np.isfinite(prediction).all()\n        or np.any(truth < 0)\n        or np.any(prediction < 0)\n    ):\n        raise ValueError("Counts must be finite and nonnegative")\n    if np.any(truth != np.floor(truth)):\n        raise ValueError("Reference counts must be integers")\n    return truth, prediction\n\n\ndef count_metrics(reference: object, predicted: object) -> dict:\n    """Include every image/species pair, including zeros and fractional baselines."""\n    truth, prediction = _counts(reference, predicted)\n    error = prediction - truth\n    return {\n        "images": len(truth),\n        "image_species_pairs": truth.size,\n        "mae": float(np.abs(error).mean()),\n        "bias": float(error.mean()),\n        "exact_count_rate": float((error == 0).mean()),\n        "all_species_exact_image_rate": float(np.all(error == 0, axis=1).mean()),\n        "per_species_mae": np.abs(error).mean(axis=0).tolist(),\n        "per_species_bias": error.mean(axis=0).tolist(),\n        "per_species_exact_rate": (error == 0).mean(axis=0).tolist(),\n        "total_target_mae": float(np.abs(error.sum(axis=1)).mean()),\n    }\n\n\ndef pairwise_iou(reference: object, predicted: object) -> np.ndarray:\n    """Compute continuous-coordinate IoU (no pixel +1 convention)."""\n    a, b = _boxes(reference), _boxes(predicted)\n    intersection = np.maximum(\n        0, np.minimum(a[:, None, 2:], b[None, :, 2:]) - np.maximum(a[:, None, :2], b[None, :, :2])\n    ).prod(axis=2)\n    aa, bb = (a[:, 2:] - a[:, :2]).prod(axis=1), (b[:, 2:] - b[:, :2]).prod(axis=1)\n    return intersection / (aa[:, None] + bb[None, :] - intersection)\n\n\ndef match_boxes(reference: object, predicted: object, iou_threshold: float = 0.5) -> dict:\n    """Maximum cardinality, then IoU sum, with stable input-index tie resolution.\n\n    Fixed SciPy assignment is deterministic for fixed ordered inputs. The cardinality\n    bonus is greater than the maximum possible total IoU change. No perturbation\n    changes the secondary scientific objective.\n    """\n    if not 0 < iou_threshold <= 1:\n        raise ValueError("Invalid IoU threshold")\n    iou = pairwise_iou(reference, predicted)\n    n, m = iou.shape\n    eligible = iou >= iou_threshold\n    score = np.where(eligible, min(n, m) + 1 + iou, 0)\n    row, col = linear_sum_assignment(score, maximize=True)\n    pairs = [(int(r), int(c), float(iou[r, c])) for r, c in zip(row, col, strict=True) if eligible[r, c]]\n    return {\n        "matches": pairs,\n        "misses": sorted(set(range(n)) - {p[0] for p in pairs}),\n        "spurious": sorted(set(range(m)) - {p[1] for p in pairs}),\n    }\n\n\ndef error_decomposition(reference: list[dict], predicted: list[dict]) -> dict:\n    """Decompose only audited nonignored images; do not invent ignore semantics.\n\n    Images with ignored objects need an explicit evaluator-specific handling policy;\n    refusing them here prevents ignored insects being silently counted as spurious.\n    """\n    if any(obj.get("ignore", False) for obj in reference):\n        raise ValueError("Ignore regions require an explicit evaluation policy")\n    if any(obj.get("species") not in SPECIES for obj in reference + predicted):\n        raise ValueError("Unknown species")\n    result = match_boxes([o["bbox_xyxy"] for o in reference], [o["bbox_xyxy"] for o in predicted])\n    confusion = np.zeros((2, 2), dtype=int)\n    for r, p, _ in result["matches"]:\n        confusion[SPECIES.index(reference[r]["species"]), SPECIES.index(predicted[p]["species"])] += 1\n    result.update(\n        missed=len(result["misses"]),\n        spurious_count=len(result["spurious"]),\n        wrong_species=int(confusion.sum() - np.trace(confusion)),\n        matched_confusion=confusion.tolist(),\n    )\n    return result\n\n\ndef select_count_threshold(rows: list[dict], split: str = "validation") -> dict:\n    """Choose predeclared validation grid minimum MAE; exact ties go higher."""\n    if split != "validation" or len(rows) != 19:\n        raise ValueError("Selection requires the complete validation threshold grid")\n    thresholds = [float(row["threshold"]) for row in rows]\n    if sorted(round(t, 12) for t in thresholds) != [i / 20 for i in range(1, 20)]:\n        raise ValueError("Threshold grid mismatch")\n    if any(not np.isfinite(row["mae"]) or row["mae"] < 0 for row in rows):\n        raise ValueError("Invalid validation MAE")\n    return dict(min(rows, key=lambda row: (row["mae"], -row["threshold"])))\n\n\ndef paired_group_bootstrap(\n    reference: object,\n    prediction_a: object,\n    prediction_b: object,\n    groups: list[str],\n    n_boot: int = 2000,\n    seed: int = 42,\n) -> dict:\n    """Resample whole groups, preserving image weighting; difference is A minus B."""\n    truth, a = _counts(reference, prediction_a)\n    _, b = _counts(reference, prediction_b)\n    if len(groups) != len(truth) or any(not isinstance(g, str) or not g for g in groups) or n_boot < 1:\n        raise ValueError("Invalid bootstrap groups or replicate count")\n    unique = sorted(set(groups))\n    indices = [np.flatnonzero(np.asarray(groups) == group) for group in unique]\n    rng = np.random.default_rng(seed)\n    values = []\n    for _ in range(n_boot):\n        selected = np.concatenate([indices[i] for i in rng.integers(len(unique), size=len(unique))])\n        ma, mb = np.abs(a[selected] - truth[selected]).mean(), np.abs(b[selected] - truth[selected]).mean()\n        values.append((ma, mb, ma - mb))\n    intervals = np.quantile(values, [0.025, 0.975], axis=0)\n    return {\n        "groups": len(unique),\n        "images": len(truth),\n        "replicates": n_boot,\n        "seed": seed,\n        "difference_direction": "a_minus_b",\n        "unstable_few_groups": len(unique) < 10,\n        "mae_a_ci": intervals[:, 0].tolist(),\n        "mae_b_ci": intervals[:, 1].tolist(),\n        "difference_ci": intervals[:, 2].tolist(),\n        "difference": float(np.abs(a - truth).mean() - np.abs(b - truth).mean()),\n    }\n', 'rice_assets.py': '"""Fetch only frozen ZIP members, checking HTTP ranges, ZIP metadata and image digests."""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport io\nimport struct\nimport time\nimport urllib.error\nimport urllib.parse\nimport urllib.request\nimport zipfile\nimport zlib\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\n\nimport rice_core as core\nfrom PIL import Image\n\nATTEMPTS = 6\nRETRY_AFTER_CAP = 90.0\nEXTRA_FIELD_CAP = 1024\nMIRROR_HOSTS = {"github.com"}\nMIRROR_CEILING = 64 * 1024**2\n\n\ndef retry_delay(error: Exception, attempt: int) -> float:\n    """Honour a server\'s Retry-After on 429/503; otherwise back off exponentially."""\n    headers = getattr(error, "headers", None)\n    value = headers.get("Retry-After") if headers is not None else None\n    try:\n        delay = float(value) if value is not None else float(2**attempt)\n    except ValueError:\n        delay = float(2**attempt)\n    return max(1.0, min(delay, RETRY_AFTER_CAP))\n\n\ndef open_with_retry(request: urllib.request.Request, reader):\n    """Retry throttling (429), server errors (5xx) and dropped connections; raise other HTTP errors."""\n    for attempt in range(ATTEMPTS):\n        try:\n            with urllib.request.urlopen(request, timeout=60) as response:\n                return reader(response)\n        except urllib.error.HTTPError as error:\n            if (error.code != 429 and error.code < 500) or attempt == ATTEMPTS - 1:\n                raise\n            time.sleep(retry_delay(error, attempt))\n        except (urllib.error.URLError, TimeoutError, ConnectionError) as error:\n            if attempt == ATTEMPTS - 1:\n                raise\n            time.sleep(retry_delay(error, attempt))\n    raise RuntimeError("Unreachable retry state")\n\n\ndef read_range(url: str, start: int, length: int, archive_bytes: int) -> bytes:\n    """Bound every network read; never accept a server silently returning the whole archive."""\n    parsed = urllib.parse.urlsplit(url)\n    if parsed.scheme != "https" or parsed.hostname != "zenodo.org":\n        raise ValueError("Only the pinned Zenodo HTTPS archive is supported")\n    if start < 0 or length < 1 or length > 26 * 1024**2 or start + length > archive_bytes:\n        raise ValueError("Invalid archive byte range")\n    end = start + length - 1\n    request = urllib.request.Request(\n        url, headers={"Range": f"bytes={start}-{end}", "User-Agent": "DIMER-Rice-Capstone/1"}\n    )\n\n    def reader(response):\n        if response.status != 206:\n            raise ValueError("Archive server did not honour the bounded range")\n        if response.headers.get("Content-Range") != f"bytes {start}-{end}/{archive_bytes}":\n            raise ValueError("Archive Content-Range mismatch")\n        return response.read(length + 1)\n\n    payload = open_with_retry(request, reader)\n    if len(payload) != length:\n        raise ValueError("Archive range length mismatch")\n    return payload\n\n\ndef unpack_member(header: bytes, remainder: bytes, member: dict) -> bytes:\n    """Decode a pinned local ZIP member without extracting arbitrary paths."""\n    if len(header) != 30:\n        raise ValueError("Invalid local ZIP header length")\n    fields = struct.unpack("<4s5H3L2H", header)\n    signature, _, flags, method = fields[:4]\n    name_size, extra_size = fields[-2:]\n    if signature != b"PK\\x03\\x04" or flags & 1 or method != member["method"]:\n        raise ValueError("Local ZIP signature/encryption/compression mismatch")\n    name = remainder[:name_size].decode("utf-8" if flags & 0x800 else "cp437")\n    if name != member["name"]:\n        raise ValueError("ZIP member identity mismatch")\n    core.safe_path(Path("."), name)\n    expected = name_size + extra_size + member["compressed"]\n    if len(remainder) != expected or member["size"] > 25 * 1024**2:\n        raise ValueError("ZIP member byte ceiling or length mismatch")\n    payload = remainder[name_size + extra_size :]\n    if method == 8:\n        decoder = zlib.decompressobj(-15)\n        payload = decoder.decompress(payload, member["size"] + 1)\n        if not decoder.eof or decoder.unconsumed_tail or decoder.unused_data:\n            raise ValueError("ZIP deflate stream exceeds declared size or contains extra bytes")\n    elif method != 0:\n        raise ValueError("Unsupported ZIP compression")\n    if len(payload) != member["size"] or f"{zlib.crc32(payload):08x}" != member["crc32"]:\n        raise ValueError("ZIP member size/CRC mismatch")\n    return payload\n\n\ndef fetch_member(archive: dict, member: dict) -> tuple[bytes, bytes]:\n    """Read header and member in one bounded request; a second only for an unusually large extra field."""\n    offset = member["offset"]\n    guess = 30 + len(member["name"].encode("utf-8")) + EXTRA_FIELD_CAP + member["compressed"]\n    data = read_range(archive["url"], offset, min(guess, archive["bytes"] - offset), archive["bytes"])\n    header = data[:30]\n    fields = struct.unpack("<4s5H3L2H", header)\n    length = fields[-2] + fields[-1] + member["compressed"]\n    if len(data) >= 30 + length:\n        return header, data[30 : 30 + length]\n    return header, read_range(archive["url"], offset + 30, length, archive["bytes"])\n\n\ndef fetch_image(root: Path, record: dict, archive: dict) -> Path:\n    """Validate existing caches as strictly as freshly downloaded members."""\n    target = core.safe_path(root / "data", record["relative_path"])\n    if not target.exists():\n        member = record["zip_member"]\n        if member["size"] != record["bytes"] or member["compressed"] > 25 * 1024**2:\n            raise ValueError("Manifest ZIP member size mismatch")\n        header, remainder = fetch_member(archive, member)\n        payload = unpack_member(header, remainder, member)\n        if hashlib.sha256(payload).hexdigest() != record["sha256"]:\n            raise ValueError("Downloaded image SHA-256 mismatch")\n        target.parent.mkdir(parents=True, exist_ok=True)\n        target.write_bytes(payload)\n    if target.stat().st_size != record["bytes"] or core.sha256(target) != record["sha256"]:\n        raise ValueError("Image cache byte/hash mismatch")\n    with Image.open(target) as image:\n        if image.format not in {"JPEG", "PNG", "WEBP"}:\n            raise ValueError("Unsupported image format")\n        if image.size != (record["width"], record["height"]):\n            raise ValueError("Image dimensions mismatch")\n        image.load()\n    return target\n\n\ndef download_mirror(mirror: dict) -> bytes:\n    """One bounded HTTPS download of the sample bundle, accepted only at its pinned size and SHA-256."""\n    parsed = urllib.parse.urlsplit(mirror["url"])\n    if parsed.scheme != "https" or parsed.hostname not in MIRROR_HOSTS:\n        raise ValueError("Mirror URL is not an allowed HTTPS host")\n    size = mirror["bytes"]\n    if not 0 < size <= MIRROR_CEILING:\n        raise ValueError("Mirror bundle size ceiling exceeded")\n    request = urllib.request.Request(mirror["url"], headers={"User-Agent": "DIMER-Rice-Capstone/1"})\n\n    def reader(response):\n        if urllib.parse.urlsplit(response.geturl()).scheme != "https":\n            raise ValueError("Mirror redirected away from HTTPS")\n        return response.read(size + 1)\n\n    payload = open_with_retry(request, reader)\n    if len(payload) != size or hashlib.sha256(payload).hexdigest() != mirror["sha256"]:\n        raise ValueError("Mirror bundle size/SHA-256 mismatch")\n    return payload\n\n\ndef extract_mirror(root: Path, bundle: bytes, records: list[dict]) -> int:\n    """Write only the manifest\'s own paths, each re-checked against its record before it touches disk."""\n    wanted = {r["relative_path"]: r for r in records}\n    with zipfile.ZipFile(io.BytesIO(bundle)) as archive:\n        infos = archive.infolist()\n        for info in infos:\n            core.safe_path(root / "data", info.filename)\n        if len(infos) != len(wanted) or sorted(i.filename for i in infos) != sorted(wanted):\n            raise ValueError("Mirror bundle members do not match the manifest")\n        written = 0\n        for info in infos:\n            record = wanted[info.filename]\n            if info.file_size != record["bytes"]:\n                raise ValueError("Mirror member size mismatch")\n            payload = archive.read(info)\n            if len(payload) != record["bytes"] or hashlib.sha256(payload).hexdigest() != record["sha256"]:\n                raise ValueError("Mirror member SHA-256 mismatch")\n            target = core.safe_path(root / "data", info.filename)\n            if not target.exists():\n                target.parent.mkdir(parents=True, exist_ok=True)\n                target.write_bytes(payload)\n                written += 1\n    return written\n\n\ndef prepare_images(root: Path, manifest: dict, mirror: dict | None = None) -> dict:\n    """Try the hash-pinned mirror, fall back to rate-limit-aware Zenodo ranges, then verify every image."""\n    archive = manifest["archive"]\n    records = manifest["records"]\n    if len(records) > 1000 or sum(r["bytes"] for r in records) > 500 * 1024**2:\n        raise ValueError("Frozen sample ceiling exceeded")\n    source = "zenodo"\n    if mirror is not None:\n        try:\n            if mirror["members"] != len(records):\n                raise ValueError("Mirror member count does not match the manifest")\n            extract_mirror(root, download_mirror(mirror), records)\n            source = "mirror"\n        except Exception as error:  # noqa: BLE001 - any mirror failure falls back to the source of record\n            reason = f"{type(error).__name__}: {error}"\n            print(f"Mirror unavailable ({reason}); falling back to Zenodo.", flush=True)\n    with ThreadPoolExecutor(max_workers=2) as pool:\n        paths = list(pool.map(lambda row: fetch_image(root, row, archive), records))\n    return {\n        "images": len(paths),\n        "bytes": sum(p.stat().st_size for p in paths),\n        "sha256_verified": True,\n        "source": source,\n    }\n', 'rice_figures.py': '"""Read-only learner figures; never changes canonical policies or model outputs."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport json\nfrom pathlib import Path\n\nimport matplotlib\nimport numpy as np\n\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt  # noqa: E402\nfrom matplotlib.patches import Rectangle  # noqa: E402\nfrom PIL import Image  # noqa: E402\n\n\ndef read(path: Path):\n    return json.loads(path.read_text(encoding="utf-8"))\n\n\ndef render(root: Path, stage: str) -> None:\n    out = root / "outputs"\n    folder = out / "figures"\n    folder.mkdir(exist_ok=True)\n    if stage == "prepare":\n        manifest = read(root / "data_manifest.json")\n        train = [r for r in manifest["records"] if r["split"] == "train"]\n        fig, axes = plt.subplots(3, 4, figsize=(15, 12))\n        for ax, row in zip(axes.flat, train[:12], strict=True):\n            with Image.open(root / "data" / row["relative_path"]) as source:\n                ax.imshow(source.convert("RGB"))\n            for obj in row["objects"]:\n                x1, y1, x2, y2 = obj["bbox_xyxy"]\n                ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="red"))\n            ax.set_title(f"{row[\'capture_group_id\']}\\n{len(row[\'objects\'])} published boxes", fontsize=8)\n            ax.axis("off")\n        fig.suptitle("Training image exports and published reference boxes · not independently verified")\n        fig.text(\n            0.5,\n            0.01,\n            "Deleña et al. · Zenodo 10.5281/zenodo.20066074 · CC BY 4.0 · resized previews",\n            ha="center",\n        )\n        fig.tight_layout(rect=(0, 0.025, 1, 0.96))\n        fig.subplots_adjust(hspace=0.22)\n    elif stage == "crops":\n        # What the species classifier receives: crops cut from published reference boxes.\n        manifest = read(root / "data_manifest.json")\n        by_id = {r["image_id"]: r for r in manifest["records"]}\n        lineage = [x for x in read(out / "lineage.json") if x["split"] == "train"]\n        species = ("rice_black_bug", "white_stemborer")\n        fig, axes = plt.subplots(2, 6, figsize=(15, 6))\n        for row_axes, name in zip(axes, species, strict=True):\n            chosen, parents = [], set()\n            for crop in lineage:\n                if crop["species"] == name and crop["parent_image_id"] not in parents:\n                    chosen.append(crop)\n                    parents.add(crop["parent_image_id"])\n                if len(chosen) == len(row_axes):\n                    break\n            for ax in row_axes:\n                ax.axis("off")\n            for ax, crop in zip(row_axes, chosen, strict=False):\n                record = by_id[crop["parent_image_id"]]\n                with Image.open(root / "data" / record["relative_path"]) as source:\n                    ax.imshow(source.convert("RGB").crop(tuple(crop["effective_crop_xyxy"])))\n                x1, y1, x2, y2 = crop["effective_crop_xyxy"]\n                ax.set_title(f"{name.replace(\'_\', \' \')}\\n{x2 - x1}×{y2 - y1} px", fontsize=8)\n        fig.suptitle("Classifier input: training crops from published reference boxes · labels unverified")\n        fig.text(\n            0.5,\n            0.01,\n            "Deleña et al. · Zenodo 10.5281/zenodo.20066074 · CC BY 4.0 · crops of resized exports",\n            ha="center",\n        )\n        fig.tight_layout(rect=(0, 0.03, 1, 0.95))\n    elif stage in {"classifier", "detector"}:\n        history = read(out / f"{stage}_history.json")\n        keys = (\n            ["training_cross_entropy", "validation_cross_entropy"]\n            if stage == "classifier"\n            else ["training_loss", "validation_ap50"]\n        )\n        fig, axes = plt.subplots(1, len(keys), figsize=(12, 4), squeeze=False)\n        for ax, key in zip(axes.flat, keys, strict=True):\n            rows = [r for r in history if r.get(key) is not None]\n            if not rows:\n                raise ValueError(f"Missing learning evidence: {key}")\n            ax.plot([r["epoch"] for r in rows], [r[key] for r in rows], marker="o")\n            ax.set(xlabel="Epoch", ylabel=key.replace("_", " "), title=key.replace("_", " "))\n        fig.suptitle("Validation selects the checkpoint; test results do not select epochs")\n        fig.tight_layout()\n    elif stage == "evaluate":\n        values = read(out / "metrics.json")\n        names = list(values)\n        fig, axes = plt.subplots(1, 2, figsize=(14, 6))\n        axes[0].barh(names, [values[n]["mae"] for n in names])\n        axes[0].set(\n            xlabel="MAE per image/species (lower is better)", title="Held-out source-family benchmark"\n        )\n        with (out / "counts.csv").open(encoding="utf-8", newline="") as handle:\n            rows = list(csv.DictReader(handle))\n        reference = np.array([float(r["reference"]) for r in rows])\n        prediction = np.array([float(r["raw"]) for r in rows])\n        axes[1].scatter(reference, prediction, alpha=0.7)\n        ceiling = max(float(reference.max()), float(prediction.max()), 1)\n        axes[1].plot([0, ceiling], [0, ceiling], linestyle="--", color="black")\n        axes[1].set(\n            xlabel="Published reference count",\n            ylabel="Raw predicted count",\n            title=f"{len(rows)} image/species pairs",\n        )\n        fig.tight_layout()\n    elif stage == "activity":\n        with (out / "activity_thresholds.csv").open(encoding="utf-8", newline="") as handle:\n            rows = list(csv.DictReader(handle))\n        fig, axes = plt.subplots(1, 4, figsize=(16, 4))\n        for ax, key in zip(axes.flat, ("mae", "missed", "spurious", "wrong_species"), strict=True):\n            ax.plot([float(r["display_threshold"]) for r in rows], [float(r[key]) for r in rows], marker="o")\n            ax.set(xlabel="Display threshold", ylabel=key, title=key)\n        fig.suptitle("Retrospective illustration · locked policy unchanged")\n        fig.tight_layout()\n    else:\n        raise ValueError("Unknown figure stage")\n    fig.savefig(folder / f"{stage}.png", dpi=120, bbox_inches="tight")\n    plt.close(fig)\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--root", type=Path, required=True)\n    parser.add_argument("--stage", required=True)\n    args = parser.parse_args()\n    render(args.root, args.stage)\n', 'data_manifest.json': '{\n  "format_version": 1,\n  "scope": "exploratory_published_annotations",\n  "classes": [\n    "rice_black_bug",\n    "white_stemborer"\n  ],\n  "archive": {\n    "url": "https://zenodo.org/api/records/20066074/files/datasets.zip/content",\n    "bytes": 1005857218,\n    "md5": "24960c245a3a21b63be95314cc7467da",\n    "sha256": null,\n    "integrity": "Selected member SHA-256 and size are pinned; whole archive SHA-256 not measured."\n  },\n  "records": [\n    {\n      "image_id": "rice_000a5d20e06da761",\n      "relative_path": "images/IMG_1187_JPG.rf.38552e3f914b5b17b7d012228e5380c6.jpg",\n      "sha256": "802ba5298ed8ce69db41cb3cc75fda2b14521e329b144f1e6222d614a6ebb6f7",\n      "decoded_sha256": "a5a23e3b7105afa3469e669ae24b136f377915ad9bc688686cdf85a41021f29a",\n      "dhash": "96128a344d0b0f27",\n      "bytes": 48228,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_1187_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_000a5d20e06da761_0",\n          "bbox_xyxy": [\n            37.25,\n            513.0,\n            72.75,\n            572.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_000a5d20e06da761_1",\n          "bbox_xyxy": [\n            387.0,\n            150.5,\n            428.0,\n            201.50000000000003\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_000a5d20e06da761_2",\n          "bbox_xyxy": [\n            421.5,\n            289.5,\n            464.49999999999994,\n            335.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_000a5d20e06da761_3",\n          "bbox_xyxy": [\n            273.25,\n            413.25,\n            310.75,\n            461.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_000a5d20e06da761_4",\n          "bbox_xyxy": [\n            300.5,\n            271.75,\n            339.5,\n            317.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_000a5d20e06da761_5",\n          "bbox_xyxy": [\n            211.50000000000003,\n            225.75,\n            254.5,\n            288.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_000a5d20e06da761_6",\n          "bbox_xyxy": [\n            267.5,\n            110.25,\n            298.5,\n            169.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_000a5d20e06da761_7",\n          "bbox_xyxy": [\n            19.999999999999996,\n            273.5,\n            67.0,\n            313.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_000a5d20e06da761_8",\n          "bbox_xyxy": [\n            2.250000000000001,\n            68.0,\n            47.75,\n            113.99999999999999\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_000a5d20e06da761_9",\n          "bbox_xyxy": [\n            102.99999999999999,\n            261.5,\n            141.0,\n            314.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_000a5d20e06da761_10",\n          "bbox_xyxy": [\n            169.00000000000003,\n            376.5,\n            212.0,\n            425.50000000000006\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_000a5d20e06da761_11",\n          "bbox_xyxy": [\n            54.5,\n            143.0,\n            83.5,\n            205.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_000a5d20e06da761_12",\n          "bbox_xyxy": [\n            118.75,\n            150.5,\n            161.25,\n            194.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_000a5d20e06da761_13",\n          "bbox_xyxy": [\n            106.0,\n            23.999999999999993,\n            137.0,\n            88.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_000a5d20e06da761_14",\n          "bbox_xyxy": [\n            135.75,\n            515.75,\n            164.25,\n            571.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_000a5d20e06da761_15",\n          "bbox_xyxy": [\n            496.75,\n            104.50000000000001,\n            538.25,\n            146.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_000a5d20e06da761_16",\n          "bbox_xyxy": [\n            339.49999999999994,\n            232.0,\n            383.5,\n            279.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_000a5d20e06da761_17",\n          "bbox_xyxy": [\n            410.5,\n            20.25,\n            454.5,\n            80.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_000a5d20e06da761_18",\n          "bbox_xyxy": [\n            53.75,\n            401.0,\n            95.25,\n            443.00000000000006\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_000a5d20e06da761_19",\n          "bbox_xyxy": [\n            362.25,\n            423.00000000000006,\n            409.75,\n            466.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_000a5d20e06da761_20",\n          "bbox_xyxy": [\n            276.0,\n            0.2499999999999991,\n            315.0,\n            32.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_000a5d20e06da761_21",\n          "bbox_xyxy": [\n            310.0,\n            92.74999999999999,\n            352.0,\n            139.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_000a5d20e06da761_22",\n          "bbox_xyxy": [\n            259.75,\n            592.25,\n            281.25,\n            639.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_000a5d20e06da761_23",\n          "bbox_xyxy": [\n            500.50000000000006,\n            200.5,\n            533.5,\n            265.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_000a5d20e06da761_24",\n          "bbox_xyxy": [\n            504.0,\n            0.2499999999999991,\n            548.0,\n            45.74999999999999\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_000a5d20e06da761_25",\n          "bbox_xyxy": [\n            197.5,\n            99.75,\n            221.5,\n            167.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_000a5d20e06da761_26",\n          "bbox_xyxy": [\n            210.99999999999997,\n            0.0,\n            237.0,\n            8.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_1187_JPG.rf.38552e3f914b5b17b7d012228e5380c6.jpg",\n        "size": 48228,\n        "compressed": 47789,\n        "offset": 490486072,\n        "crc32": "38ed89e1",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_00265d1f786a5bc1",\n      "relative_path": "images/IMG_3994_JPG.rf.ff72c7b4bf32079884e219d2e8e31d8a.jpg",\n      "sha256": "0cb5e5157c17863d1de24f4175597e9ea8305bb018a61188d7aa932ca18baf54",\n      "decoded_sha256": "1870ea274479b29e55f83867d498ed11e91313bbfc205f8442e444745460b45a",\n      "dhash": "6464706861fbd46c",\n      "bytes": 25571,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_3994_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_00265d1f786a5bc1_0",\n          "bbox_xyxy": [\n            310.25,\n            479.49999999999994,\n            332.75,\n            553.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_00265d1f786a5bc1_1",\n          "bbox_xyxy": [\n            191.75,\n            62.25000000000001,\n            245.25000000000003,\n            145.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_00265d1f786a5bc1_2",\n          "bbox_xyxy": [\n            291.0,\n            38.5,\n            324.0,\n            131.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_00265d1f786a5bc1_3",\n          "bbox_xyxy": [\n            446.25,\n            33.25,\n            491.75,\n            89.75000000000001\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_00265d1f786a5bc1_4",\n          "bbox_xyxy": [\n            402.0,\n            129.25,\n            450.0,\n            175.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_00265d1f786a5bc1_5",\n          "bbox_xyxy": [\n            297.25,\n            149.0,\n            329.75,\n            200.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_00265d1f786a5bc1_6",\n          "bbox_xyxy": [\n            353.00000000000006,\n            234.5,\n            396.0,\n            274.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_00265d1f786a5bc1_7",\n          "bbox_xyxy": [\n            215.0,\n            182.5,\n            271.0,\n            210.49999999999997\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_00265d1f786a5bc1_8",\n          "bbox_xyxy": [\n            252.0,\n            256.5,\n            314.0,\n            305.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_00265d1f786a5bc1_9",\n          "bbox_xyxy": [\n            347.5,\n            333.75,\n            428.49999999999994,\n            375.25000000000006\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_00265d1f786a5bc1_10",\n          "bbox_xyxy": [\n            532.5,\n            408.5,\n            618.5,\n            443.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_00265d1f786a5bc1_11",\n          "bbox_xyxy": [\n            351.25,\n            411.5,\n            393.75,\n            459.49999999999994\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_00265d1f786a5bc1_12",\n          "bbox_xyxy": [\n            270.0,\n            353.00000000000006,\n            311.0,\n            411.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_00265d1f786a5bc1_13",\n          "bbox_xyxy": [\n            187.0,\n            317.0,\n            239.00000000000003,\n            360.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_00265d1f786a5bc1_14",\n          "bbox_xyxy": [\n            35.25,\n            325.75,\n            74.75,\n            386.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_00265d1f786a5bc1_15",\n          "bbox_xyxy": [\n            190.0,\n            449.5,\n            265.0,\n            474.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_00265d1f786a5bc1_16",\n          "bbox_xyxy": [\n            130.5,\n            483.00000000000006,\n            160.5,\n            541.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_00265d1f786a5bc1_17",\n          "bbox_xyxy": [\n            210.75,\n            560.75,\n            245.25000000000003,\n            613.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_00265d1f786a5bc1_18",\n          "bbox_xyxy": [\n            436.75,\n            537.25,\n            493.25,\n            581.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_00265d1f786a5bc1_19",\n          "bbox_xyxy": [\n            0.0,\n            214.74999999999997,\n            45.0,\n            273.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_3994_JPG.rf.ff72c7b4bf32079884e219d2e8e31d8a.jpg",\n        "size": 25571,\n        "compressed": 22823,\n        "offset": 587535726,\n        "crc32": "23f549fe",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_00a40547147261b0",\n      "relative_path": "images/IMG_2187_JPG.rf.9a34ebfeb57fbd5cf0ecf52edc851a75.jpg",\n      "sha256": "3ad78bb7037f57012f581aa9144905f4727350746a538beb7d5ac1540a9f576e",\n      "decoded_sha256": "58a8b94bae3b2371379f6c6133b9a7c217cd9cb887f675c6e4c5dda09c7e7519",\n      "dhash": "0570717979713112",\n      "bytes": 22360,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2187_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_00a40547147261b0_0",\n          "bbox_xyxy": [\n            174.0,\n            74.25,\n            499.0,\n            570.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_2187_JPG.rf.9a34ebfeb57fbd5cf0ecf52edc851a75.jpg",\n        "size": 22360,\n        "compressed": 20533,\n        "offset": 533880398,\n        "crc32": "14628d98",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_00f254bf89d663bb",\n      "relative_path": "images/IMG_1874_JPG.rf.9f9f28dd1f28123fc17d7fbc8fe99c9a.jpg",\n      "sha256": "1954c2f7e3a2713d22a0322865617df4d62091448cc5f4aedebac244027c0ab1",\n      "decoded_sha256": "222a082b1e02e4e9122e623badb84472e6e93ea0953622c4b6fdc28789d30e84",\n      "dhash": "563070707172304c",\n      "bytes": 19764,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_1874_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_00f254bf89d663bb_0",\n          "bbox_xyxy": [\n            150.25,\n            70.99999999999997,\n            529.75,\n            548.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_1874_JPG.rf.9f9f28dd1f28123fc17d7fbc8fe99c9a.jpg",\n        "size": 19764,\n        "compressed": 18651,\n        "offset": 502574279,\n        "crc32": "c71f0e1a",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_01148661d1d45f35",\n      "relative_path": "images/IMG_2068_JPG.rf.811f0f8e2c4a82f80e2716da2e8bd3ad.jpg",\n      "sha256": "193441b50dcabe936e2f99a9a65e06dff8507bbe64e321702187da980f9bb5a1",\n      "decoded_sha256": "50b8df9a16e819a1e7952dbb46a2f5a1d19d3a398d757fb4499d1a56683fcad5",\n      "dhash": "c6f17169697971b2",\n      "bytes": 30427,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2068_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_01148661d1d45f35_0",\n          "bbox_xyxy": [\n            104.5,\n            62.5,\n            506.50000000000006,\n            528.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_2068_JPG.rf.811f0f8e2c4a82f80e2716da2e8bd3ad.jpg",\n        "size": 30427,\n        "compressed": 29576,\n        "offset": 524468913,\n        "crc32": "f0c9ce42",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_01259d351b177814",\n      "relative_path": "images/IMG_1162_JPG.rf.15f5ae7050092220432d3d1b82236a8a.jpg",\n      "sha256": "b92bd82bf562cc19e1ac2c05b7c5d85322a2d92843be91e03db2838f6bd43954",\n      "decoded_sha256": "a9fcc7e704f5b61794f8dd536599ca6732a02c755c097378ca4335f6bf17b629",\n      "dhash": "468d0e2e4d0f0f2f",\n      "bytes": 37072,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_1162_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_01259d351b177814_0",\n          "bbox_xyxy": [\n            390.0,\n            215.75,\n            414.00000000000006,\n            252.24999999999997\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_01259d351b177814_1",\n          "bbox_xyxy": [\n            484.5,\n            106.25,\n            502.5,\n            143.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_01259d351b177814_2",\n          "bbox_xyxy": [\n            256.0,\n            59.75,\n            276.0,\n            99.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_01259d351b177814_3",\n          "bbox_xyxy": [\n            382.0,\n            107.5,\n            407.0,\n            144.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_01259d351b177814_4",\n          "bbox_xyxy": [\n            160.99999999999997,\n            57.0,\n            188.0,\n            88.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_01259d351b177814_5",\n          "bbox_xyxy": [\n            448.75,\n            301.5,\n            475.25000000000006,\n            338.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_01259d351b177814_6",\n          "bbox_xyxy": [\n            161.75,\n            7.5,\n            189.25,\n            37.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_01259d351b177814_7",\n          "bbox_xyxy": [\n            95.75,\n            13.75,\n            128.25,\n            43.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_01259d351b177814_8",\n          "bbox_xyxy": [\n            182.5,\n            349.75,\n            208.5,\n            379.25000000000006\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_01259d351b177814_9",\n          "bbox_xyxy": [\n            561.5,\n            78.5,\n            585.5,\n            109.50000000000001\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_01259d351b177814_10",\n          "bbox_xyxy": [\n            233.49999999999997,\n            129.75,\n            260.5,\n            162.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_01259d351b177814_11",\n          "bbox_xyxy": [\n            248.25,\n            0.2499999999999991,\n            267.75,\n            38.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_01259d351b177814_12",\n          "bbox_xyxy": [\n            253.25,\n            235.0,\n            276.75,\n            280.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_01259d351b177814_13",\n          "bbox_xyxy": [\n            478.49999999999994,\n            196.75,\n            507.5,\n            223.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_01259d351b177814_14",\n          "bbox_xyxy": [\n            295.0,\n            65.0,\n            323.0,\n            95.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_01259d351b177814_15",\n          "bbox_xyxy": [\n            71.5,\n            66.0,\n            93.5,\n            102.99999999999999\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_01259d351b177814_16",\n          "bbox_xyxy": [\n            126.25,\n            99.0,\n            150.75,\n            139.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_01259d351b177814_17",\n          "bbox_xyxy": [\n            190.5,\n            90.0,\n            216.50000000000003,\n            123.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_01259d351b177814_18",\n          "bbox_xyxy": [\n            99.25,\n            170.75,\n            126.75,\n            198.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_01259d351b177814_19",\n          "bbox_xyxy": [\n            515.75,\n            248.25000000000003,\n            542.25,\n            278.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_01259d351b177814_20",\n          "bbox_xyxy": [\n            440.50000000000006,\n            72.75,\n            463.5,\n            113.25000000000001\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_01259d351b177814_21",\n          "bbox_xyxy": [\n            7.999999999999998,\n            0.2499999999999991,\n            35.0,\n            28.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_01259d351b177814_22",\n          "bbox_xyxy": [\n            443.5,\n            179.50000000000003,\n            465.50000000000006,\n            217.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_01259d351b177814_23",\n          "bbox_xyxy": [\n            197.75,\n            280.0,\n            222.25,\n            312.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_01259d351b177814_24",\n          "bbox_xyxy": [\n            424.0,\n            364.75,\n            446.99999999999994,\n            402.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_01259d351b177814_25",\n          "bbox_xyxy": [\n            112.99999999999999,\n            247.75,\n            139.0,\n            275.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_01259d351b177814_26",\n          "bbox_xyxy": [\n            514.75,\n            0.0,\n            533.25,\n            40.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_01259d351b177814_27",\n          "bbox_xyxy": [\n            523.5,\n            179.0,\n            544.5,\n            220.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_01259d351b177814_28",\n          "bbox_xyxy": [\n            60.75,\n            113.75,\n            79.25,\n            154.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_01259d351b177814_29",\n          "bbox_xyxy": [\n            19.250000000000004,\n            197.75000000000003,\n            34.75,\n            239.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_01259d351b177814_30",\n          "bbox_xyxy": [\n            483.0,\n            495.99999999999994,\n            508.0,\n            525.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_01259d351b177814_31",\n          "bbox_xyxy": [\n            218.75,\n            605.5,\n            246.25,\n            631.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_1162_JPG.rf.15f5ae7050092220432d3d1b82236a8a.jpg",\n        "size": 37072,\n        "compressed": 36663,\n        "offset": 485904165,\n        "crc32": "f401ace8",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_0171eef12070e8ff",\n      "relative_path": "images/IMG_1988_JPG.rf.708f654f6481f6582a672dc7c3e76738.jpg",\n      "sha256": "5d792cced75e00e2d62bb1a762f18d167e8e30c64ad7b6cfa63c99c2cc572163",\n      "decoded_sha256": "80181b86b438be8c26dfd76217ee1ad039f972661f037b9c5a439f49d56a625e",\n      "dhash": "f1e9696971b2f429",\n      "bytes": 23584,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_1988_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_0171eef12070e8ff_0",\n          "bbox_xyxy": [\n            123.49999999999997,\n            0.0,\n            499.5,\n            552.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_1988_JPG.rf.708f654f6481f6582a672dc7c3e76738.jpg",\n        "size": 23584,\n        "compressed": 22910,\n        "offset": 507885182,\n        "crc32": "cf353fc0",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_017a99caec398785",\n      "relative_path": "images/IMG_0821_JPG.rf.00dd9d35c348ac0011ad356ee608f97a.jpg",\n      "sha256": "7396c2e132059feaca52a8bd5a57d5c61a2561e3a57c03067d294bc3b0b8caf0",\n      "decoded_sha256": "98f54c818a8944b1ea237bbcec9deb61f70874e599da7d37935097e8c075bdee",\n      "dhash": "17ca2517524855d4",\n      "bytes": 50270,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_0821_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_017a99caec398785_0",\n          "bbox_xyxy": [\n            89.25,\n            588.25,\n            140.75,\n            632.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_017a99caec398785_1",\n          "bbox_xyxy": [\n            114.0,\n            600.0,\n            156.0,\n            640.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_017a99caec398785_2",\n          "bbox_xyxy": [\n            168.75,\n            552.5,\n            199.24999999999997,\n            623.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_017a99caec398785_3",\n          "bbox_xyxy": [\n            200.25000000000003,\n            523.0,\n            225.75,\n            598.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_017a99caec398785_4",\n          "bbox_xyxy": [\n            176.0,\n            472.0,\n            239.0,\n            523.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_017a99caec398785_5",\n          "bbox_xyxy": [\n            164.5,\n            400.25,\n            193.49999999999997,\n            480.74999999999994\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_017a99caec398785_6",\n          "bbox_xyxy": [\n            220.75,\n            372.75,\n            257.25,\n            453.24999999999994\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_017a99caec398785_7",\n          "bbox_xyxy": [\n            158.25,\n            325.25,\n            201.75,\n            410.74999999999994\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_017a99caec398785_8",\n          "bbox_xyxy": [\n            286.5,\n            565.75,\n            353.5,\n            609.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_017a99caec398785_9",\n          "bbox_xyxy": [\n            358.25,\n            528.5,\n            403.75,\n            606.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_017a99caec398785_10",\n          "bbox_xyxy": [\n            314.0,\n            471.50000000000006,\n            380.0,\n            522.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_017a99caec398785_11",\n          "bbox_xyxy": [\n            437.0,\n            530.75,\n            478.0,\n            598.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_017a99caec398785_12",\n          "bbox_xyxy": [\n            422.5,\n            517.5,\n            469.5,\n            564.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_017a99caec398785_13",\n          "bbox_xyxy": [\n            474.75,\n            517.75,\n            504.25000000000006,\n            590.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_017a99caec398785_14",\n          "bbox_xyxy": [\n            461.50000000000006,\n            442.5,\n            512.5,\n            484.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_017a99caec398785_15",\n          "bbox_xyxy": [\n            501.99999999999994,\n            385.50000000000006,\n            554.0,\n            426.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_017a99caec398785_16",\n          "bbox_xyxy": [\n            481.25,\n            355.5,\n            533.75,\n            399.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_017a99caec398785_17",\n          "bbox_xyxy": [\n            572.75,\n            447.5,\n            606.25,\n            523.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_017a99caec398785_18",\n          "bbox_xyxy": [\n            539.0,\n            360.25,\n            590.0,\n            409.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_017a99caec398785_19",\n          "bbox_xyxy": [\n            559.75,\n            305.25,\n            620.25,\n            385.74999999999994\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_017a99caec398785_20",\n          "bbox_xyxy": [\n            538.0,\n            244.00000000000003,\n            586.0,\n            317.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_017a99caec398785_21",\n          "bbox_xyxy": [\n            581.75,\n            210.75,\n            631.25,\n            243.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_017a99caec398785_22",\n          "bbox_xyxy": [\n            606.0,\n            185.0,\n            640.0,\n            225.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_017a99caec398785_23",\n          "bbox_xyxy": [\n            565.25,\n            110.49999999999999,\n            624.75,\n            168.49999999999997\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_017a99caec398785_24",\n          "bbox_xyxy": [\n            531.75,\n            219.25,\n            585.25,\n            257.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_017a99caec398785_25",\n          "bbox_xyxy": [\n            514.25,\n            79.25,\n            546.75,\n            168.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_017a99caec398785_26",\n          "bbox_xyxy": [\n            458.5,\n            210.25,\n            514.5,\n            254.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_017a99caec398785_27",\n          "bbox_xyxy": [\n            399.5,\n            180.25000000000003,\n            452.5,\n            230.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_017a99caec398785_28",\n          "bbox_xyxy": [\n            365.5,\n            103.0,\n            417.5,\n            149.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_017a99caec398785_29",\n          "bbox_xyxy": [\n            353.24999999999994,\n            108.0,\n            377.75,\n            180.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_017a99caec398785_30",\n          "bbox_xyxy": [\n            315.0,\n            91.75,\n            360.0,\n            145.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_017a99caec398785_31",\n          "bbox_xyxy": [\n            383.00000000000006,\n            143.25,\n            426.0,\n            184.74999999999997\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_017a99caec398785_32",\n          "bbox_xyxy": [\n            303.24999999999994,\n            162.5,\n            337.75,\n            231.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_017a99caec398785_33",\n          "bbox_xyxy": [\n            267.0,\n            195.5,\n            319.0,\n            235.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_017a99caec398785_34",\n          "bbox_xyxy": [\n            235.5,\n            137.0,\n            290.5,\n            197.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_017a99caec398785_35",\n          "bbox_xyxy": [\n            185.25,\n            142.25,\n            239.75,\n            178.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_017a99caec398785_36",\n          "bbox_xyxy": [\n            200.0,\n            109.50000000000001,\n            257.0,\n            156.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_017a99caec398785_37",\n          "bbox_xyxy": [\n            180.25,\n            182.00000000000003,\n            224.75,\n            230.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_017a99caec398785_38",\n          "bbox_xyxy": [\n            153.5,\n            64.25,\n            182.5,\n            173.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_017a99caec398785_39",\n          "bbox_xyxy": [\n            110.25,\n            76.74999999999999,\n            164.75,\n            117.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_017a99caec398785_40",\n          "bbox_xyxy": [\n            64.75,\n            190.75,\n            116.25,\n            270.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_017a99caec398785_41",\n          "bbox_xyxy": [\n            133.5,\n            241.5,\n            172.5,\n            328.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_017a99caec398785_42",\n          "bbox_xyxy": [\n            37.0,\n            280.5,\n            76.0,\n            350.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_017a99caec398785_43",\n          "bbox_xyxy": [\n            7.5,\n            208.25,\n            53.5,\n            266.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_017a99caec398785_44",\n          "bbox_xyxy": [\n            38.5,\n            434.74999999999994,\n            106.5,\n            476.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_017a99caec398785_45",\n          "bbox_xyxy": [\n            9.249999999999998,\n            546.5,\n            58.75,\n            580.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_017a99caec398785_46",\n          "bbox_xyxy": [\n            207.24999999999997,\n            619.0,\n            266.75,\n            640.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_017a99caec398785_47",\n          "bbox_xyxy": [\n            408.24999999999994,\n            317.5,\n            462.75,\n            379.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_017a99caec398785_48",\n          "bbox_xyxy": [\n            428.75,\n            398.49999999999994,\n            462.24999999999994,\n            472.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_017a99caec398785_49",\n          "bbox_xyxy": [\n            370.5,\n            401.99999999999994,\n            427.5,\n            469.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_017a99caec398785_50",\n          "bbox_xyxy": [\n            337.75,\n            310.25,\n            378.24999999999994,\n            377.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_017a99caec398785_51",\n          "bbox_xyxy": [\n            299.25,\n            360.0,\n            346.75,\n            400.99999999999994\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_017a99caec398785_52",\n          "bbox_xyxy": [\n            249.00000000000003,\n            315.74999999999994,\n            297.0,\n            360.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_0821_JPG.rf.00dd9d35c348ac0011ad356ee608f97a.jpg",\n        "size": 50270,\n        "compressed": 50072,\n        "offset": 445802920,\n        "crc32": "4f8923bc",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_017b5f57ad0cf098",\n      "relative_path": "images/IMG_2729_JPG.rf.9d06ad5151fea038dd09a24ebf81959f.jpg",\n      "sha256": "1d851dde51de48238b23ff1f9524b59c2def8bf588758c7b526f742fd13e0187",\n      "decoded_sha256": "1575a5d81d0dae1b31872c170d59c63d90c653245419e88c054f4a0cf6c84853",\n      "dhash": "313179787831326c",\n      "bytes": 31558,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2729_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_017b5f57ad0cf098_0",\n          "bbox_xyxy": [\n            166.5,\n            35.249999999999986,\n            613.5,\n            614.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_2729_JPG.rf.9d06ad5151fea038dd09a24ebf81959f.jpg",\n        "size": 31558,\n        "compressed": 31043,\n        "offset": 581916759,\n        "crc32": "3f67d3a7",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_01e174f98aa70874",\n      "relative_path": "images/IMG_2474_JPG.rf.8993ebcccf33de37c00a83598d76497b.jpg",\n      "sha256": "be6b6426d9c3f7c4d738311a801864ac92454d90fd5c589bbc663ca9b63020d0",\n      "decoded_sha256": "ff73d8ea73d2af0407ec2d3f5196616bf9e010aacee7784fc7e90c443c884f09",\n      "dhash": "d971716169717294",\n      "bytes": 27152,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2474_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_01e174f98aa70874_0",\n          "bbox_xyxy": [\n            118.75,\n            20.749999999999993,\n            463.25,\n            560.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_2474_JPG.rf.8993ebcccf33de37c00a83598d76497b.jpg",\n        "size": 27152,\n        "compressed": 26438,\n        "offset": 558109486,\n        "crc32": "083329de",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_0200e8f2d56f85c9",\n      "relative_path": "images/IMG_4046_JPG.rf.d756f85c40be9d06d46406b9fa66abb4.jpg",\n      "sha256": "099066d9ea064189482cb9cd1dc34b5259247a29b43976cf718529c405e7d403",\n      "decoded_sha256": "1364968b3ee64b629e58ada77e6827f1a71c0deebd047c132f5c7aaa30640885",\n      "dhash": "e2ea3a3664e2321e",\n      "bytes": 46355,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_4046_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_0200e8f2d56f85c9_0",\n          "bbox_xyxy": [\n            447.25,\n            358.75,\n            477.75,\n            394.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0200e8f2d56f85c9_1",\n          "bbox_xyxy": [\n            550.0,\n            359.25,\n            588.0,\n            388.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0200e8f2d56f85c9_2",\n          "bbox_xyxy": [\n            163.5,\n            408.49999999999994,\n            198.5,\n            437.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0200e8f2d56f85c9_3",\n          "bbox_xyxy": [\n            267.0,\n            212.99999999999997,\n            296.0,\n            250.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0200e8f2d56f85c9_4",\n          "bbox_xyxy": [\n            51.0,\n            235.75,\n            94.0,\n            262.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0200e8f2d56f85c9_5",\n          "bbox_xyxy": [\n            341.5,\n            141.5,\n            378.5,\n            168.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_4046_JPG.rf.d756f85c40be9d06d46406b9fa66abb4.jpg",\n        "size": 46355,\n        "compressed": 44695,\n        "offset": 588540271,\n        "crc32": "3e0fa954",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_0206daa1bae3251b",\n      "relative_path": "images/IMG_2523_JPG.rf.76daf697a243520c259e4764f090611e.jpg",\n      "sha256": "5e01261b6d458c65eaacc87aaace22bb6cacf3900ce55be587262a6a245498e7",\n      "decoded_sha256": "1248a07f036e5ded0e7b592348de6e94052ea214236bd70bffc52f1595eddd58",\n      "dhash": "283070e0e0e0e060",\n      "bytes": 30982,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2523_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_0206daa1bae3251b_0",\n          "bbox_xyxy": [\n            127.0,\n            0.0,\n            498.0,\n            606.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_2523_JPG.rf.76daf697a243520c259e4764f090611e.jpg",\n        "size": 30982,\n        "compressed": 27502,\n        "offset": 562424921,\n        "crc32": "e58bac30",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_0216f8a97a50aa0b",\n      "relative_path": "images/IMG_2732_JPG.rf.ef3fc54c3fc88320fe1222704f3834b7.jpg",\n      "sha256": "e80234140b228102c8aad6d667a1c5a08a5b5298432e3d1947b5ce04943158cc",\n      "decoded_sha256": "869d5acbfdc376158d6c56bf0111507b4151bc2717ff65159daabed8afc637bf",\n      "dhash": "6b72b13979793131",\n      "bytes": 28082,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2732_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_0216f8a97a50aa0b_0",\n          "bbox_xyxy": [\n            182.00000000000003,\n            25.249999999999986,\n            490.0,\n            599.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_2732_JPG.rf.ef3fc54c3fc88320fe1222704f3834b7.jpg",\n        "size": 28082,\n        "compressed": 26810,\n        "offset": 582107649,\n        "crc32": "226fd729",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_028131df8871f0d5",\n      "relative_path": "images/IMG_1986_JPG.rf.7667e93f2bc600ff9b04b0d83bef108d.jpg",\n      "sha256": "40a49b564ca3ee28b58de5a870b9ce526441359c913a42013e4ff76d8e50206f",\n      "decoded_sha256": "287847542fdef1ba69315738007ddda3dedcfb0c541e877de39a02f44563008d",\n      "dhash": "7072716969713014",\n      "bytes": 21096,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_1986_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_028131df8871f0d5_0",\n          "bbox_xyxy": [\n            140.25,\n            41.25,\n            515.75,\n            556.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_1986_JPG.rf.7667e93f2bc600ff9b04b0d83bef108d.jpg",\n        "size": 21096,\n        "compressed": 19779,\n        "offset": 507671676,\n        "crc32": "3ca38f95",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_02b24b20f53eae77",\n      "relative_path": "images/IMG_2469_JPG.rf.b4e27b9a629d6cbf5efd29155ccb2ecd.jpg",\n      "sha256": "5c97fac7deaa4d2cae88c9729c85bed1500f0a1928b33dc3351a082cc488f5aa",\n      "decoded_sha256": "24bcca8776214b6a054a6b066f32daa364aad07733860987c1848ffa64eba81c",\n      "dhash": "4cf0f17969693132",\n      "bytes": 28028,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2469_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_02b24b20f53eae77_0",\n          "bbox_xyxy": [\n            136.25,\n            49.49999999999996,\n            530.75,\n            601.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_2469_JPG.rf.b4e27b9a629d6cbf5efd29155ccb2ecd.jpg",\n        "size": 28028,\n        "compressed": 27333,\n        "offset": 557548419,\n        "crc32": "fa823a2d",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_02e43da96331e069",\n      "relative_path": "images/IMG_1038_JPG.rf.ec6059f86b5f2b150dca45b89ffc05b2.jpg",\n      "sha256": "7cbb84dc291b101f180a3c12d9bc3a18cf1a025280480f64c2e357ccaac56892",\n      "decoded_sha256": "3c69a6d5de53106e3288af858a21b6b48967285c6b09bb68d6a5345afe993a11",\n      "dhash": "af534f6a2b8e8d0d",\n      "bytes": 42049,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_1038_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_02e43da96331e069_0",\n          "bbox_xyxy": [\n            216.75,\n            418.25,\n            272.25,\n            456.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_02e43da96331e069_1",\n          "bbox_xyxy": [\n            200.5,\n            213.5,\n            250.5,\n            253.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_02e43da96331e069_2",\n          "bbox_xyxy": [\n            196.0,\n            337.5,\n            251.0,\n            378.49999999999994\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_02e43da96331e069_3",\n          "bbox_xyxy": [\n            229.25,\n            13.75,\n            251.75,\n            120.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_02e43da96331e069_4",\n          "bbox_xyxy": [\n            0.0,\n            265.25,\n            40.0,\n            323.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_02e43da96331e069_5",\n          "bbox_xyxy": [\n            266.0,\n            248.75,\n            290.0,\n            334.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_02e43da96331e069_6",\n          "bbox_xyxy": [\n            517.5,\n            232.5,\n            567.5,\n            297.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_02e43da96331e069_7",\n          "bbox_xyxy": [\n            194.25,\n            464.25000000000006,\n            244.74999999999997,\n            537.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_02e43da96331e069_8",\n          "bbox_xyxy": [\n            73.75,\n            454.75,\n            116.25,\n            529.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_02e43da96331e069_9",\n          "bbox_xyxy": [\n            151.75,\n            420.25000000000006,\n            195.25000000000003,\n            488.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_02e43da96331e069_10",\n          "bbox_xyxy": [\n            498.00000000000006,\n            373.0,\n            536.0,\n            440.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_02e43da96331e069_11",\n          "bbox_xyxy": [\n            380.75,\n            233.5,\n            418.25,\n            326.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_02e43da96331e069_12",\n          "bbox_xyxy": [\n            124.5,\n            236.0,\n            155.5,\n            319.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_02e43da96331e069_13",\n          "bbox_xyxy": [\n            553.0,\n            312.50000000000006,\n            599.0,\n            381.50000000000006\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_02e43da96331e069_14",\n          "bbox_xyxy": [\n            392.5,\n            352.5,\n            443.49999999999994,\n            402.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_02e43da96331e069_15",\n          "bbox_xyxy": [\n            439.25,\n            540.0,\n            491.75,\n            604.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_02e43da96331e069_16",\n          "bbox_xyxy": [\n            329.25000000000006,\n            91.25,\n            369.75,\n            153.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_02e43da96331e069_17",\n          "bbox_xyxy": [\n            63.25,\n            165.75,\n            104.75000000000001,\n            219.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_02e43da96331e069_18",\n          "bbox_xyxy": [\n            58.24999999999999,\n            358.75,\n            88.75,\n            435.25000000000006\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_02e43da96331e069_19",\n          "bbox_xyxy": [\n            27.750000000000004,\n            564.0,\n            65.25,\n            640.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_02e43da96331e069_20",\n          "bbox_xyxy": [\n            88.50000000000001,\n            46.25,\n            134.5,\n            90.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_02e43da96331e069_21",\n          "bbox_xyxy": [\n            0.0,\n            119.25,\n            30.0,\n            207.75000000000003\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_02e43da96331e069_22",\n          "bbox_xyxy": [\n            153.74999999999997,\n            112.00000000000001,\n            194.24999999999997,\n            210.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_1038_JPG.rf.ec6059f86b5f2b150dca45b89ffc05b2.jpg",\n        "size": 42049,\n        "compressed": 41719,\n        "offset": 463975551,\n        "crc32": "38d93bbb",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_02e932cc3b66ac6f",\n      "relative_path": "images/IMG_1058_JPG.rf.204e782fb4c8bdb767bca894b1e6d01c.jpg",\n      "sha256": "52913f424db33ca89ef58b3fd6461fc6620bd16720729eabbd423680262e0d49",\n      "decoded_sha256": "d54417adbc27a64535568d166b396f395425df2e49b5a6b0caba159ed4c7d9c4",\n      "dhash": "2b536369241d180c",\n      "bytes": 51609,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_1058_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_02e932cc3b66ac6f_0",\n          "bbox_xyxy": [\n            223.25000000000003,\n            227.5,\n            253.75,\n            291.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_02e932cc3b66ac6f_1",\n          "bbox_xyxy": [\n            230.25000000000003,\n            296.75,\n            261.75,\n            376.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_02e932cc3b66ac6f_2",\n          "bbox_xyxy": [\n            303.75000000000006,\n            489.75,\n            360.25000000000006,\n            525.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_02e932cc3b66ac6f_3",\n          "bbox_xyxy": [\n            378.5,\n            80.0,\n            433.5,\n            113.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_02e932cc3b66ac6f_4",\n          "bbox_xyxy": [\n            408.0,\n            497.75000000000006,\n            458.0,\n            561.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_02e932cc3b66ac6f_5",\n          "bbox_xyxy": [\n            551.25,\n            83.75,\n            604.75,\n            110.24999999999999\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_02e932cc3b66ac6f_6",\n          "bbox_xyxy": [\n            442.5,\n            332.0,\n            496.50000000000006,\n            372.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_02e932cc3b66ac6f_7",\n          "bbox_xyxy": [\n            349.0,\n            500.0,\n            401.99999999999994,\n            562.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_02e932cc3b66ac6f_8",\n          "bbox_xyxy": [\n            456.5,\n            369.75,\n            493.5,\n            420.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_02e932cc3b66ac6f_9",\n          "bbox_xyxy": [\n            413.25,\n            576.25,\n            453.75,\n            606.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_02e932cc3b66ac6f_10",\n          "bbox_xyxy": [\n            522.25,\n            513.75,\n            567.75,\n            579.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_1058_JPG.rf.204e782fb4c8bdb767bca894b1e6d01c.jpg",\n        "size": 51609,\n        "compressed": 51429,\n        "offset": 467741924,\n        "crc32": "9cc378d7",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_0316eda3a78538ec",\n      "relative_path": "images/IMG_2414_JPG.rf.bdda6875b468d5f6a671582e94008023.jpg",\n      "sha256": "ffa1a67d768245ef7aff8f19644334d84f71c21a322675d5f6145e4265b20e67",\n      "decoded_sha256": "c9f7bfe08deea68a3bc32e324bf1cdee0ed8c227a3c14fe50bbb77a06fccc36a",\n      "dhash": "2030706878703010",\n      "bytes": 24036,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2414_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_0316eda3a78538ec_0",\n          "bbox_xyxy": [\n            175.00000000000003,\n            0.0,\n            524.0,\n            566.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_2414_JPG.rf.bdda6875b468d5f6a671582e94008023.jpg",\n        "size": 24036,\n        "compressed": 19652,\n        "offset": 553382964,\n        "crc32": "4f8a08f4",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_0323c8b81c19a7f1",\n      "relative_path": "images/IMG_2737_JPG.rf.493900f9b9d22fd04c31b8de619e9adf.jpg",\n      "sha256": "3773d6fd6f8eb39d0ec519999e395236aaf788eaf81ec0ad1c1d038d30007a33",\n      "decoded_sha256": "f78dab67ef08cae600145179cb5c5c6b7c377db3679d31cc7910df02b005a57f",\n      "dhash": "39b97878b8383162",\n      "bytes": 28560,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2737_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_0323c8b81c19a7f1_0",\n          "bbox_xyxy": [\n            148.25000000000003,\n            21.249999999999964,\n            513.75,\n            639.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_2737_JPG.rf.493900f9b9d22fd04c31b8de619e9adf.jpg",\n        "size": 28560,\n        "compressed": 27936,\n        "offset": 582582714,\n        "crc32": "f62a44d6",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_03dba5a29c688df2",\n      "relative_path": "images/IMG_1796_JPG.rf.91fe44721fc721a0ca26266e9c6836b0.jpg",\n      "sha256": "896221e4ae76d9a2ccd39db82c2eff1818c174db3698cd6433ce30eeac206136",\n      "decoded_sha256": "fa27037d9c2f372d271e253b8dacfd14ea84a46b12630a20d42e66af4ba6525e",\n      "dhash": "0060607060607008",\n      "bytes": 23343,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_1796_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_03dba5a29c688df2_0",\n          "bbox_xyxy": [\n            144.74999999999997,\n            57.5,\n            484.25,\n            577.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_1796_JPG.rf.91fe44721fc721a0ca26266e9c6836b0.jpg",\n        "size": 23343,\n        "compressed": 19099,\n        "offset": 499825112,\n        "crc32": "d2f8ea26",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_03e0a0c68c1d805f",\n      "relative_path": "images/20251107_142743_jpg.rf.4e4b74d388efecc5e54058fd27d66d11.jpg",\n      "sha256": "7461e57a388331c9dc0676a9ed77b505c881dcac47080288cf945f8ba92e364d",\n      "decoded_sha256": "2989db17273fa02deb4f5e47a3e677c8f672fad9b1cf9bc4b16d2fa4872f7cc3",\n      "dhash": "d0e0c4222024a4c0",\n      "bytes": 99827,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "20251107_142743_jpg",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_03e0a0c68c1d805f_0",\n          "bbox_xyxy": [\n            322.75000000000006,\n            520.0,\n            366.25,\n            532.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_03e0a0c68c1d805f_1",\n          "bbox_xyxy": [\n            485.25000000000006,\n            311.25,\n            518.75,\n            331.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_03e0a0c68c1d805f_2",\n          "bbox_xyxy": [\n            511.75000000000006,\n            250.5,\n            540.25,\n            276.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_03e0a0c68c1d805f_3",\n          "bbox_xyxy": [\n            618.0,\n            544.0,\n            640.0,\n            576.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_03e0a0c68c1d805f_4",\n          "bbox_xyxy": [\n            255.5,\n            584.25,\n            278.5,\n            616.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_03e0a0c68c1d805f_5",\n          "bbox_xyxy": [\n            453.25,\n            448.5,\n            478.75,\n            476.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_03e0a0c68c1d805f_6",\n          "bbox_xyxy": [\n            539.25,\n            446.5,\n            581.75,\n            468.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/20251107_142743_jpg.rf.4e4b74d388efecc5e54058fd27d66d11.jpg",\n        "size": 99827,\n        "compressed": 99653,\n        "offset": 321523462,\n        "crc32": "53d5a50e",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_042115e8352b07aa",\n      "relative_path": "images/6273997143085354525_jpg.rf.e82431a56c50b980b7cfb4f121a9ed73.jpg",\n      "sha256": "4b6be516e7fc358f6068a90cc80219a3fa3af9b4d99535547a428c5defde1a46",\n      "decoded_sha256": "e1a1d58b0e58a53eafb3020adce98b8e67a9baec38a7655d194a44e379309592",\n      "dhash": "03031b190d060d0d",\n      "bytes": 53305,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "6273997143085354525_jpg",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_042115e8352b07aa_0",\n          "bbox_xyxy": [\n            80.5,\n            106.75,\n            142.5,\n            151.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_042115e8352b07aa_1",\n          "bbox_xyxy": [\n            15.500000000000004,\n            206.75,\n            82.5,\n            258.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_042115e8352b07aa_2",\n          "bbox_xyxy": [\n            29.25,\n            420.99999999999994,\n            62.74999999999999,\n            465.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_042115e8352b07aa_3",\n          "bbox_xyxy": [\n            425.25,\n            315.5,\n            461.75000000000006,\n            376.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_042115e8352b07aa_4",\n          "bbox_xyxy": [\n            515.75,\n            400.25000000000006,\n            562.25,\n            473.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_042115e8352b07aa_5",\n          "bbox_xyxy": [\n            418.24999999999994,\n            538.75,\n            454.75,\n            592.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_042115e8352b07aa_6",\n          "bbox_xyxy": [\n            543.0,\n            141.25,\n            573.0,\n            188.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_042115e8352b07aa_7",\n          "bbox_xyxy": [\n            337.75,\n            226.75,\n            389.25000000000006,\n            268.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/6273997143085354525_jpg.rf.e82431a56c50b980b7cfb4f121a9ed73.jpg",\n        "size": 53305,\n        "compressed": 52991,\n        "offset": 421902195,\n        "crc32": "b090b6be",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_0463c87ab715a15c",\n      "relative_path": "images/IMG_2625_JPG.rf.bdce396e472ac0cb34afab36138a7d4f.jpg",\n      "sha256": "db0e1154fd8438d2779ebdd8329559764262d1a8e35b9b52ef26b0af4d6ea7e1",\n      "decoded_sha256": "610afc98dc3a7f835c14ffcdea8560769dd4e5feff1ee31b4f8f35ed7a89abd0",\n      "dhash": "323179697979b136",\n      "bytes": 28910,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2625_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_0463c87ab715a15c_0",\n          "bbox_xyxy": [\n            154.24999999999997,\n            68.25,\n            536.75,\n            639.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_2625_JPG.rf.bdce396e472ac0cb34afab36138a7d4f.jpg",\n        "size": 28910,\n        "compressed": 26776,\n        "offset": 571356697,\n        "crc32": "a3ac7f9d",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_04f1ae66d768fd74",\n      "relative_path": "images/IMG_2014_JPG.rf.0d11eedb533398f158d42601180a62a5.jpg",\n      "sha256": "5e48398a444b4607555f866738f73d49a619acd9f6e989034e7da65da095402a",\n      "decoded_sha256": "902a9761130aba28695cf6200908c6a62f6fb08466aaf1bafceea53c83f23e2a",\n      "dhash": "10f0f0f8f8703064",\n      "bytes": 25840,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2014_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_04f1ae66d768fd74_0",\n          "bbox_xyxy": [\n            108.25,\n            63.749999999999964,\n            475.75,\n            632.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_2014_JPG.rf.0d11eedb533398f158d42601180a62a5.jpg",\n        "size": 25840,\n        "compressed": 21977,\n        "offset": 509468340,\n        "crc32": "c7698bdd",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_052f2de5bcbf9ed2",\n      "relative_path": "images/IMG_4185_JPG.rf.4b73112fc1e46f0e59a061c5de3f9dbf.jpg",\n      "sha256": "bd8548bf29f9c690ae0167e06ea9ca6c3a28ea125e6178960ea05968ca8a78ff",\n      "decoded_sha256": "4bf3760dc5816b74c10f0682c530021703af4e7fc477d91522205be0a1c65f48",\n      "dhash": "b050a4a0c0f074d6",\n      "bytes": 50739,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_4185_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_052f2de5bcbf9ed2_0",\n          "bbox_xyxy": [\n            239.0,\n            468.75,\n            264.0,\n            507.24999999999994\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_052f2de5bcbf9ed2_1",\n          "bbox_xyxy": [\n            143.25,\n            444.49999999999994,\n            161.75,\n            496.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_052f2de5bcbf9ed2_2",\n          "bbox_xyxy": [\n            220.5,\n            409.25,\n            260.5,\n            438.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_052f2de5bcbf9ed2_3",\n          "bbox_xyxy": [\n            80.24999999999999,\n            364.0,\n            123.75,\n            383.00000000000006\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_052f2de5bcbf9ed2_4",\n          "bbox_xyxy": [\n            195.75,\n            378.5,\n            220.25000000000003,\n            416.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_052f2de5bcbf9ed2_5",\n          "bbox_xyxy": [\n            304.75,\n            424.0,\n            343.24999999999994,\n            451.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_052f2de5bcbf9ed2_6",\n          "bbox_xyxy": [\n            193.75,\n            306.25000000000006,\n            220.25,\n            362.75000000000006\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_052f2de5bcbf9ed2_7",\n          "bbox_xyxy": [\n            337.5,\n            287.0,\n            369.5,\n            313.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_052f2de5bcbf9ed2_8",\n          "bbox_xyxy": [\n            305.25,\n            242.5,\n            323.75,\n            291.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_052f2de5bcbf9ed2_9",\n          "bbox_xyxy": [\n            246.75,\n            201.25,\n            274.25,\n            236.74999999999997\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_052f2de5bcbf9ed2_10",\n          "bbox_xyxy": [\n            200.25000000000003,\n            163.49999999999997,\n            221.75,\n            210.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_052f2de5bcbf9ed2_11",\n          "bbox_xyxy": [\n            79.99999999999999,\n            306.25,\n            113.99999999999999,\n            333.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_052f2de5bcbf9ed2_12",\n          "bbox_xyxy": [\n            373.0,\n            147.75,\n            400.0,\n            183.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_052f2de5bcbf9ed2_13",\n          "bbox_xyxy": [\n            141.5,\n            94.75,\n            163.5,\n            130.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_052f2de5bcbf9ed2_14",\n          "bbox_xyxy": [\n            451.25,\n            152.24999999999997,\n            486.75,\n            175.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_052f2de5bcbf9ed2_15",\n          "bbox_xyxy": [\n            450.0,\n            376.5,\n            494.00000000000006,\n            393.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_052f2de5bcbf9ed2_16",\n          "bbox_xyxy": [\n            470.75,\n            211.25,\n            506.25,\n            251.74999999999997\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_052f2de5bcbf9ed2_17",\n          "bbox_xyxy": [\n            70.25,\n            225.75,\n            109.75,\n            247.24999999999997\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_052f2de5bcbf9ed2_18",\n          "bbox_xyxy": [\n            68.0,\n            199.24999999999997,\n            110.0,\n            223.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_052f2de5bcbf9ed2_19",\n          "bbox_xyxy": [\n            439.0,\n            505.25,\n            489.0,\n            522.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_4185_JPG.rf.4b73112fc1e46f0e59a061c5de3f9dbf.jpg",\n        "size": 50739,\n        "compressed": 50607,\n        "offset": 608003482,\n        "crc32": "bf51b86e",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_0545768c1fed50d7",\n      "relative_path": "images/IMG_0857_JPG.rf.50204a542c375e9c44131255449bd5d1.jpg",\n      "sha256": "da72a192a4477fca50e9b309fb1ca9cf8ca7ba67e4268ff241561113ced0f471",\n      "decoded_sha256": "47568667aef3c6647630040729740a466584468cdd71f7d86bd84eb8afe54b0a",\n      "dhash": "2c4a4e05211a1214",\n      "bytes": 47594,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_0857_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_0545768c1fed50d7_0",\n          "bbox_xyxy": [\n            50.25,\n            471.25,\n            139.75,\n            512.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0545768c1fed50d7_1",\n          "bbox_xyxy": [\n            150.25,\n            246.25,\n            194.75,\n            305.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0545768c1fed50d7_2",\n          "bbox_xyxy": [\n            500.75,\n            554.5,\n            581.25,\n            585.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0545768c1fed50d7_3",\n          "bbox_xyxy": [\n            182.75000000000003,\n            434.74999999999994,\n            264.25,\n            486.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0545768c1fed50d7_4",\n          "bbox_xyxy": [\n            25.250000000000004,\n            316.0,\n            118.75,\n            354.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0545768c1fed50d7_5",\n          "bbox_xyxy": [\n            421.25,\n            290.75,\n            500.75,\n            330.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0545768c1fed50d7_6",\n          "bbox_xyxy": [\n            452.5,\n            609.0,\n            514.5,\n            640.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0545768c1fed50d7_7",\n          "bbox_xyxy": [\n            504.25000000000006,\n            98.5,\n            577.75,\n            153.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0545768c1fed50d7_8",\n          "bbox_xyxy": [\n            287.25,\n            514.0,\n            364.75,\n            552.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0545768c1fed50d7_9",\n          "bbox_xyxy": [\n            181.25,\n            74.5,\n            237.75,\n            119.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0545768c1fed50d7_10",\n          "bbox_xyxy": [\n            546.75,\n            275.0,\n            613.25,\n            307.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0545768c1fed50d7_11",\n          "bbox_xyxy": [\n            9.999999999999996,\n            109.25,\n            87.0,\n            155.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0545768c1fed50d7_12",\n          "bbox_xyxy": [\n            391.25,\n            70.25,\n            429.74999999999994,\n            131.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0545768c1fed50d7_13",\n          "bbox_xyxy": [\n            326.99999999999994,\n            173.25,\n            406.0,\n            201.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0545768c1fed50d7_14",\n          "bbox_xyxy": [\n            314.0,\n            0.25000000000000133,\n            394.0,\n            20.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0545768c1fed50d7_15",\n          "bbox_xyxy": [\n            517.25,\n            435.0,\n            554.75,\n            495.99999999999994\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0545768c1fed50d7_16",\n          "bbox_xyxy": [\n            627.0,\n            532.75,\n            640.0,\n            565.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0545768c1fed50d7_17",\n          "bbox_xyxy": [\n            150.99999999999997,\n            158.25,\n            227.0,\n            216.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0545768c1fed50d7_18",\n          "bbox_xyxy": [\n            216.0,\n            29.250000000000004,\n            269.0,\n            89.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0545768c1fed50d7_19",\n          "bbox_xyxy": [\n            252.25,\n            560.25,\n            346.75,\n            589.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0545768c1fed50d7_20",\n          "bbox_xyxy": [\n            343.5,\n            419.74999999999994,\n            386.5,\n            486.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0545768c1fed50d7_21",\n          "bbox_xyxy": [\n            552.25,\n            167.75,\n            624.75,\n            221.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0545768c1fed50d7_22",\n          "bbox_xyxy": [\n            59.0,\n            211.0,\n            111.0,\n            281.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0545768c1fed50d7_23",\n          "bbox_xyxy": [\n            124.25,\n            548.75,\n            215.75,\n            585.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0545768c1fed50d7_24",\n          "bbox_xyxy": [\n            179.0,\n            615.0,\n            262.0,\n            640.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_0857_JPG.rf.50204a542c375e9c44131255449bd5d1.jpg",\n        "size": 47594,\n        "compressed": 47247,\n        "offset": 452110162,\n        "crc32": "a50375a3",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_054beca04a46d383",\n      "relative_path": "images/IMG_2742_JPG.rf.050b2c83c00a640ddab6c39032b9933b.jpg",\n      "sha256": "2b6c537e7ebff0782104ec5a854a890ae8279d476bed3f57ac2cd14da2cbaad7",\n      "decoded_sha256": "ed530fee253372c98b820106cef3b9a1efa6d1d185e83fc872bfe2b5b1c661e9",\n      "dhash": "4c7071e868787132",\n      "bytes": 34353,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2742_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_054beca04a46d383_0",\n          "bbox_xyxy": [\n            137.75000000000003,\n            43.49999999999998,\n            551.25,\n            609.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_2742_JPG.rf.050b2c83c00a640ddab6c39032b9933b.jpg",\n        "size": 34353,\n        "compressed": 33709,\n        "offset": 583069587,\n        "crc32": "1b2cfd0c",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_057f135aeeb434c0",\n      "relative_path": "images/6159142536404995643_jpg.rf.9471b29e4e1522d26d78fe528e404d01.jpg",\n      "sha256": "e745bd3ee74499fd3b16b219743da2f141902fc60045653aee00dc4150c648e9",\n      "decoded_sha256": "1b288e4c7102bc0213dffe10f4335dc3752de8be83347e78dc4e9d03d4a0bd31",\n      "dhash": "6cf4a3dcd4c6c66f",\n      "bytes": 30571,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "6159142536404995643_jpg",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_057f135aeeb434c0_0",\n          "bbox_xyxy": [\n            443.24999999999994,\n            262.75,\n            482.75,\n            305.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_057f135aeeb434c0_1",\n          "bbox_xyxy": [\n            189.0,\n            242.5,\n            225.0,\n            288.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_057f135aeeb434c0_2",\n          "bbox_xyxy": [\n            563.75,\n            161.25,\n            594.25,\n            200.75000000000003\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_057f135aeeb434c0_3",\n          "bbox_xyxy": [\n            486.0,\n            459.75,\n            516.0,\n            513.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_057f135aeeb434c0_4",\n          "bbox_xyxy": [\n            459.75,\n            63.25,\n            505.25,\n            91.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_057f135aeeb434c0_5",\n          "bbox_xyxy": [\n            204.75,\n            411.75000000000006,\n            251.25,\n            447.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_057f135aeeb434c0_6",\n          "bbox_xyxy": [\n            434.75,\n            340.25,\n            484.25000000000006,\n            372.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_057f135aeeb434c0_7",\n          "bbox_xyxy": [\n            131.5,\n            267.75,\n            162.5,\n            330.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_057f135aeeb434c0_8",\n          "bbox_xyxy": [\n            202.5,\n            33.50000000000001,\n            231.5,\n            79.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_057f135aeeb434c0_9",\n          "bbox_xyxy": [\n            92.00000000000001,\n            168.5,\n            121.0,\n            208.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_057f135aeeb434c0_10",\n          "bbox_xyxy": [\n            133.75,\n            480.0,\n            164.25,\n            526.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_057f135aeeb434c0_11",\n          "bbox_xyxy": [\n            136.25,\n            429.75,\n            167.75,\n            477.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_057f135aeeb434c0_12",\n          "bbox_xyxy": [\n            242.75000000000003,\n            465.25,\n            309.25,\n            494.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_057f135aeeb434c0_13",\n          "bbox_xyxy": [\n            315.0,\n            336.5,\n            353.0,\n            399.49999999999994\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_057f135aeeb434c0_14",\n          "bbox_xyxy": [\n            549.25,\n            423.5,\n            586.75,\n            493.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_057f135aeeb434c0_15",\n          "bbox_xyxy": [\n            579.75,\n            201.75,\n            613.25,\n            257.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_057f135aeeb434c0_16",\n          "bbox_xyxy": [\n            339.25,\n            238.0,\n            368.75,\n            283.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_057f135aeeb434c0_17",\n          "bbox_xyxy": [\n            375.0,\n            38.74999999999999,\n            429.00000000000006,\n            58.24999999999999\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_057f135aeeb434c0_18",\n          "bbox_xyxy": [\n            314.25,\n            117.75,\n            343.75,\n            165.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_057f135aeeb434c0_19",\n          "bbox_xyxy": [\n            212.5,\n            133.75,\n            251.5,\n            185.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/6159142536404995643_jpg.rf.9471b29e4e1522d26d78fe528e404d01.jpg",\n        "size": 30571,\n        "compressed": 30221,\n        "offset": 411519988,\n        "crc32": "d19721ca",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_0587f4ef63e00e6b",\n      "relative_path": "images/IMG_0768_JPG.rf.96f2014544b3658ede1e7cbbcc28847c.jpg",\n      "sha256": "b8a81c34097ae31383dbeb1da2af8bceb111c9d1a24add0fc87bba7ebf776132",\n      "decoded_sha256": "a681b01834cecc657ca1ee9a6f815dcd04b69b22246cbdba370d4968613dd808",\n      "dhash": "ee39d5b6b693ad24",\n      "bytes": 52483,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_0768_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_0587f4ef63e00e6b_0",\n          "bbox_xyxy": [\n            201.75,\n            267.5,\n            252.24999999999997,\n            359.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0587f4ef63e00e6b_1",\n          "bbox_xyxy": [\n            97.5,\n            263.5,\n            142.5,\n            342.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0587f4ef63e00e6b_2",\n          "bbox_xyxy": [\n            79.5,\n            64.24999999999999,\n            124.5,\n            129.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0587f4ef63e00e6b_3",\n          "bbox_xyxy": [\n            466.75000000000006,\n            137.0,\n            507.25,\n            210.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0587f4ef63e00e6b_4",\n          "bbox_xyxy": [\n            173.0,\n            144.5,\n            209.00000000000003,\n            216.50000000000003\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0587f4ef63e00e6b_5",\n          "bbox_xyxy": [\n            424.25,\n            78.5,\n            453.75,\n            164.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0587f4ef63e00e6b_6",\n          "bbox_xyxy": [\n            274.25,\n            162.0,\n            317.75,\n            240.99999999999997\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0587f4ef63e00e6b_7",\n          "bbox_xyxy": [\n            11.5,\n            75.5,\n            56.5,\n            157.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0587f4ef63e00e6b_8",\n          "bbox_xyxy": [\n            280.5,\n            335.25,\n            330.5,\n            401.75000000000006\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0587f4ef63e00e6b_9",\n          "bbox_xyxy": [\n            76.0,\n            197.0,\n            107.00000000000001,\n            273.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0587f4ef63e00e6b_10",\n          "bbox_xyxy": [\n            120.25,\n            164.0,\n            147.75,\n            251.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0587f4ef63e00e6b_11",\n          "bbox_xyxy": [\n            100.25,\n            2.0000000000000018,\n            155.75,\n            58.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0587f4ef63e00e6b_12",\n          "bbox_xyxy": [\n            170.75,\n            16.0,\n            217.24999999999997,\n            59.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0587f4ef63e00e6b_13",\n          "bbox_xyxy": [\n            350.5,\n            377.0,\n            399.5,\n            447.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0587f4ef63e00e6b_14",\n          "bbox_xyxy": [\n            606.25,\n            161.0,\n            639.75,\n            256.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0587f4ef63e00e6b_15",\n          "bbox_xyxy": [\n            584.0,\n            410.25000000000006,\n            625.0,\n            488.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0587f4ef63e00e6b_16",\n          "bbox_xyxy": [\n            269.5,\n            289.75,\n            310.5,\n            348.24999999999994\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0587f4ef63e00e6b_17",\n          "bbox_xyxy": [\n            62.25,\n            506.50000000000006,\n            112.75,\n            597.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0587f4ef63e00e6b_18",\n          "bbox_xyxy": [\n            428.49999999999994,\n            270.75,\n            482.5,\n            332.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0587f4ef63e00e6b_19",\n          "bbox_xyxy": [\n            25.249999999999993,\n            389.0,\n            61.75,\n            456.99999999999994\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0587f4ef63e00e6b_20",\n          "bbox_xyxy": [\n            539.5,\n            103.25,\n            598.5,\n            154.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0587f4ef63e00e6b_21",\n          "bbox_xyxy": [\n            417.5,\n            160.0,\n            444.5,\n            256.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0587f4ef63e00e6b_22",\n          "bbox_xyxy": [\n            520.0,\n            248.75,\n            579.0,\n            305.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0587f4ef63e00e6b_23",\n          "bbox_xyxy": [\n            96.5,\n            365.75,\n            134.5,\n            429.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0587f4ef63e00e6b_24",\n          "bbox_xyxy": [\n            408.25,\n            481.25,\n            461.75,\n            537.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0587f4ef63e00e6b_25",\n          "bbox_xyxy": [\n            286.75,\n            398.24999999999994,\n            336.25,\n            454.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0587f4ef63e00e6b_26",\n          "bbox_xyxy": [\n            347.75000000000006,\n            39.75,\n            401.25,\n            94.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0587f4ef63e00e6b_27",\n          "bbox_xyxy": [\n            365.25,\n            295.75,\n            407.75,\n            342.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0587f4ef63e00e6b_28",\n          "bbox_xyxy": [\n            182.5,\n            468.75,\n            239.50000000000003,\n            526.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0587f4ef63e00e6b_29",\n          "bbox_xyxy": [\n            212.75,\n            564.75,\n            260.25,\n            623.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0587f4ef63e00e6b_30",\n          "bbox_xyxy": [\n            349.25,\n            478.5,\n            396.75,\n            526.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0587f4ef63e00e6b_31",\n          "bbox_xyxy": [\n            488.24999999999994,\n            215.25,\n            537.75,\n            273.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0587f4ef63e00e6b_32",\n          "bbox_xyxy": [\n            348.24999999999994,\n            85.5,\n            387.75,\n            161.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0587f4ef63e00e6b_33",\n          "bbox_xyxy": [\n            524.5,\n            10.500000000000004,\n            563.5,\n            63.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0587f4ef63e00e6b_34",\n          "bbox_xyxy": [\n            318.75000000000006,\n            145.75,\n            355.25000000000006,\n            239.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0587f4ef63e00e6b_35",\n          "bbox_xyxy": [\n            174.75,\n            399.5,\n            221.25,\n            435.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0587f4ef63e00e6b_36",\n          "bbox_xyxy": [\n            57.75,\n            435.74999999999994,\n            100.25,\n            480.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0587f4ef63e00e6b_37",\n          "bbox_xyxy": [\n            565.75,\n            467.5,\n            598.25,\n            534.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0587f4ef63e00e6b_38",\n          "bbox_xyxy": [\n            434.0,\n            0.0,\n            469.0,\n            61.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0587f4ef63e00e6b_39",\n          "bbox_xyxy": [\n            478.00000000000006,\n            319.25,\n            519.0,\n            387.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0587f4ef63e00e6b_40",\n          "bbox_xyxy": [\n            30.250000000000004,\n            576.0,\n            58.75,\n            639.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0587f4ef63e00e6b_41",\n          "bbox_xyxy": [\n            69.25,\n            301.0,\n            105.75,\n            370.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0587f4ef63e00e6b_42",\n          "bbox_xyxy": [\n            250.75,\n            531.25,\n            299.25,\n            572.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0587f4ef63e00e6b_43",\n          "bbox_xyxy": [\n            276.75,\n            77.5,\n            313.25,\n            144.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0587f4ef63e00e6b_44",\n          "bbox_xyxy": [\n            234.5,\n            79.75,\n            280.5,\n            130.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0587f4ef63e00e6b_45",\n          "bbox_xyxy": [\n            329.0,\n            598.75,\n            369.0,\n            639.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0587f4ef63e00e6b_46",\n          "bbox_xyxy": [\n            536.75,\n            403.0,\n            574.25,\n            453.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0587f4ef63e00e6b_47",\n          "bbox_xyxy": [\n            105.75000000000001,\n            472.75,\n            150.25,\n            524.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0587f4ef63e00e6b_48",\n          "bbox_xyxy": [\n            464.0,\n            530.0,\n            523.0,\n            553.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0587f4ef63e00e6b_49",\n          "bbox_xyxy": [\n            504.49999999999994,\n            326.75,\n            558.5,\n            389.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0587f4ef63e00e6b_50",\n          "bbox_xyxy": [\n            463.49999999999994,\n            560.75,\n            502.5,\n            612.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0587f4ef63e00e6b_51",\n          "bbox_xyxy": [\n            577.75,\n            504.25000000000006,\n            622.25,\n            567.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0587f4ef63e00e6b_52",\n          "bbox_xyxy": [\n            314.0,\n            261.0,\n            343.00000000000006,\n            333.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0587f4ef63e00e6b_53",\n          "bbox_xyxy": [\n            533.5,\n            355.0,\n            575.5,\n            419.00000000000006\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_0768_JPG.rf.96f2014544b3658ede1e7cbbcc28847c.jpg",\n        "size": 52483,\n        "compressed": 52351,\n        "offset": 438824987,\n        "crc32": "7b17c3f0",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_0592ebb46bc6eebd",\n      "relative_path": "images/6159142536404995687_jpg.rf.f08094aa11e6e0effdca16527c3c6959.jpg",\n      "sha256": "e6dbe4196b1218bb95da90a8a2ad925d638f1da66b00b0067b50bb5ecdcae93e",\n      "decoded_sha256": "e26c2983dc2fa341ffe4b323ed5f9d9bf41b6addc82ad3a60a99c5cb83a690c8",\n      "dhash": "84b8c8e0d9c67471",\n      "bytes": 21623,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "6159142536404995687_jpg",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_0592ebb46bc6eebd_0",\n          "bbox_xyxy": [\n            178.75,\n            223.25,\n            227.25,\n            262.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0592ebb46bc6eebd_1",\n          "bbox_xyxy": [\n            484.00000000000006,\n            454.25000000000006,\n            520.0,\n            507.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0592ebb46bc6eebd_2",\n          "bbox_xyxy": [\n            136.0,\n            351.75000000000006,\n            175.0,\n            405.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0592ebb46bc6eebd_3",\n          "bbox_xyxy": [\n            166.5,\n            419.5,\n            206.5,\n            474.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0592ebb46bc6eebd_4",\n          "bbox_xyxy": [\n            277.5,\n            303.5,\n            322.5,\n            340.50000000000006\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0592ebb46bc6eebd_5",\n          "bbox_xyxy": [\n            390.25,\n            313.5,\n            429.75,\n            366.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0592ebb46bc6eebd_6",\n          "bbox_xyxy": [\n            363.25,\n            103.25,\n            406.75,\n            153.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0592ebb46bc6eebd_7",\n          "bbox_xyxy": [\n            281.25,\n            526.5,\n            338.75,\n            561.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0592ebb46bc6eebd_8",\n          "bbox_xyxy": [\n            555.5,\n            366.0,\n            592.5,\n            421.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0592ebb46bc6eebd_9",\n          "bbox_xyxy": [\n            267.75,\n            424.5,\n            306.25,\n            477.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/6159142536404995687_jpg.rf.f08094aa11e6e0effdca16527c3c6959.jpg",\n        "size": 21623,\n        "compressed": 20770,\n        "offset": 417221615,\n        "crc32": "dff19cd3",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_05c92603f2bb324a",\n      "relative_path": "images/IMG_4051_JPG.rf.c6cf5508c53d7238133184bf65d65578.jpg",\n      "sha256": "f4a89005e7e16dcd910c86ba869c3b02db2c8d7fcc404283fb121ab33bce12dd",\n      "decoded_sha256": "a2f1f3c41b561a5784f02b1f392e84ef7a7f810dcd29d52b0b70936ce9ff8ec0",\n      "dhash": "a1adccd8f9e8acf6",\n      "bytes": 47429,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_4051_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_05c92603f2bb324a_0",\n          "bbox_xyxy": [\n            458.5,\n            158.0,\n            499.49999999999994,\n            187.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05c92603f2bb324a_1",\n          "bbox_xyxy": [\n            146.75,\n            279.25,\n            170.25,\n            321.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05c92603f2bb324a_2",\n          "bbox_xyxy": [\n            229.25,\n            407.5,\n            265.75,\n            438.49999999999994\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05c92603f2bb324a_3",\n          "bbox_xyxy": [\n            371.75000000000006,\n            428.24999999999994,\n            405.25,\n            459.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05c92603f2bb324a_4",\n          "bbox_xyxy": [\n            497.25,\n            466.0,\n            519.75,\n            496.99999999999994\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05c92603f2bb324a_5",\n          "bbox_xyxy": [\n            464.0,\n            538.5,\n            506.0,\n            565.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_4051_JPG.rf.c6cf5508c53d7238133184bf65d65578.jpg",\n        "size": 47429,\n        "compressed": 47218,\n        "offset": 589200200,\n        "crc32": "6ea9d62b",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_05d249da3221ddcb",\n      "relative_path": "images/IMG_2511_JPG.rf.52928fb92f4a94e4e44b084a84e4b59e.jpg",\n      "sha256": "d87f529040e02270ad263cbefff7b6b914951cb956397cfc09a913d7d4aee6cb",\n      "decoded_sha256": "46d8da10293a685079ab55b04ea32a376e9a7e17135c1e82b38ceea943f9a0bc",\n      "dhash": "62727171717132b0",\n      "bytes": 26002,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2511_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_05d249da3221ddcb_0",\n          "bbox_xyxy": [\n            166.99999999999994,\n            24.750000000000014,\n            484.0,\n            577.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_2511_JPG.rf.52928fb92f4a94e4e44b084a84e4b59e.jpg",\n        "size": 26002,\n        "compressed": 24785,\n        "offset": 561350396,\n        "crc32": "492b1f2f",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_05d887bb38786154",\n      "relative_path": "images/IMG_2709_JPG.rf.f2e9039c0e31fd29f261c34a4670598f.jpg",\n      "sha256": "a6df22f32e3adb968e0a5065708d20494b40545af5466c3acfca389de3d5892f",\n      "decoded_sha256": "616cecc3ac3ca3cecc0ddd50f127fed1b018d4b01d7ae965e8ab463c786856af",\n      "dhash": "cd717161697930b2",\n      "bytes": 30381,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2709_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_05d887bb38786154_0",\n          "bbox_xyxy": [\n            130.49999999999997,\n            0.0,\n            502.5,\n            601.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_2709_JPG.rf.f2e9039c0e31fd29f261c34a4670598f.jpg",\n        "size": 30381,\n        "compressed": 29494,\n        "offset": 580260277,\n        "crc32": "bfaad21a",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_05ece00485f6635d",\n      "relative_path": "images/IMG_1149_JPG.rf.f817c8cbadf926541a6a5edf11480fed.jpg",\n      "sha256": "e6a1f0b0254e8348a2ffbb6be977d755f416c901b271335eb0f0cfade3016075",\n      "decoded_sha256": "ea9aef991cf28c38597cda0453efbb857da2dfd356fa2169b821c8e29082693f",\n      "dhash": "074f890214354313",\n      "bytes": 40872,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_1149_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_05ece00485f6635d_0",\n          "bbox_xyxy": [\n            430.25,\n            340.75,\n            462.75,\n            371.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05ece00485f6635d_1",\n          "bbox_xyxy": [\n            436.25,\n            471.0,\n            465.75,\n            504.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05ece00485f6635d_2",\n          "bbox_xyxy": [\n            458.5,\n            382.75,\n            479.49999999999994,\n            429.25000000000006\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05ece00485f6635d_3",\n          "bbox_xyxy": [\n            277.25,\n            218.25000000000003,\n            305.75,\n            258.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05ece00485f6635d_4",\n          "bbox_xyxy": [\n            473.25,\n            262.75,\n            508.75,\n            299.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05ece00485f6635d_5",\n          "bbox_xyxy": [\n            152.25,\n            474.49999999999994,\n            182.75,\n            506.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05ece00485f6635d_6",\n          "bbox_xyxy": [\n            218.5,\n            176.75,\n            242.5,\n            217.24999999999997\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05ece00485f6635d_7",\n          "bbox_xyxy": [\n            162.5,\n            361.75000000000006,\n            192.5,\n            390.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05ece00485f6635d_8",\n          "bbox_xyxy": [\n            407.75000000000006,\n            556.5,\n            436.25,\n            597.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05ece00485f6635d_9",\n          "bbox_xyxy": [\n            258.0,\n            485.5,\n            289.0,\n            522.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05ece00485f6635d_10",\n          "bbox_xyxy": [\n            346.5,\n            516.75,\n            379.49999999999994,\n            551.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05ece00485f6635d_11",\n          "bbox_xyxy": [\n            299.5,\n            357.5,\n            331.5,\n            388.49999999999994\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05ece00485f6635d_12",\n          "bbox_xyxy": [\n            295.0,\n            581.25,\n            327.0,\n            615.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05ece00485f6635d_13",\n          "bbox_xyxy": [\n            43.25,\n            446.25,\n            74.75,\n            479.74999999999994\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05ece00485f6635d_14",\n          "bbox_xyxy": [\n            203.75,\n            353.5,\n            226.25,\n            394.49999999999994\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05ece00485f6635d_15",\n          "bbox_xyxy": [\n            28.000000000000004,\n            258.0,\n            60.0,\n            290.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05ece00485f6635d_16",\n          "bbox_xyxy": [\n            244.75,\n            253.75,\n            265.25,\n            296.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05ece00485f6635d_17",\n          "bbox_xyxy": [\n            144.25,\n            190.0,\n            173.75,\n            225.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05ece00485f6635d_18",\n          "bbox_xyxy": [\n            414.25000000000006,\n            389.49999999999994,\n            439.75,\n            431.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05ece00485f6635d_19",\n          "bbox_xyxy": [\n            1.7500000000000004,\n            201.75,\n            28.25,\n            240.25000000000003\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05ece00485f6635d_20",\n          "bbox_xyxy": [\n            376.0,\n            269.0,\n            406.0,\n            316.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05ece00485f6635d_21",\n          "bbox_xyxy": [\n            81.25,\n            309.5,\n            106.75,\n            347.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05ece00485f6635d_22",\n          "bbox_xyxy": [\n            184.74999999999997,\n            507.25,\n            204.25,\n            549.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05ece00485f6635d_23",\n          "bbox_xyxy": [\n            292.0,\n            518.5,\n            315.0,\n            560.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05ece00485f6635d_24",\n          "bbox_xyxy": [\n            173.75,\n            142.5,\n            205.25,\n            178.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05ece00485f6635d_25",\n          "bbox_xyxy": [\n            251.75,\n            385.25,\n            277.25,\n            426.75000000000006\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05ece00485f6635d_26",\n          "bbox_xyxy": [\n            145.5,\n            270.75,\n            169.5,\n            320.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05ece00485f6635d_27",\n          "bbox_xyxy": [\n            366.75000000000006,\n            344.49999999999994,\n            397.25,\n            376.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05ece00485f6635d_28",\n          "bbox_xyxy": [\n            348.75,\n            617.0,\n            378.25,\n            640.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05ece00485f6635d_29",\n          "bbox_xyxy": [\n            131.5,\n            138.75,\n            156.5,\n            180.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05ece00485f6635d_30",\n          "bbox_xyxy": [\n            528.5,\n            355.5,\n            561.5,\n            400.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05ece00485f6635d_31",\n          "bbox_xyxy": [\n            97.5,\n            228.25000000000003,\n            117.5,\n            273.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05ece00485f6635d_32",\n          "bbox_xyxy": [\n            252.24999999999997,\n            598.5,\n            276.75,\n            639.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05ece00485f6635d_33",\n          "bbox_xyxy": [\n            102.75,\n            539.0,\n            128.25,\n            579.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05ece00485f6635d_34",\n          "bbox_xyxy": [\n            308.5,\n            310.74999999999994,\n            333.5,\n            350.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05ece00485f6635d_35",\n          "bbox_xyxy": [\n            216.0,\n            492.25,\n            239.0,\n            537.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05ece00485f6635d_36",\n          "bbox_xyxy": [\n            12.499999999999998,\n            313.25,\n            38.5,\n            350.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05ece00485f6635d_37",\n          "bbox_xyxy": [\n            27.75,\n            369.49999999999994,\n            48.24999999999999,\n            413.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05ece00485f6635d_38",\n          "bbox_xyxy": [\n            154.5,\n            593.25,\n            181.50000000000003,\n            633.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05ece00485f6635d_39",\n          "bbox_xyxy": [\n            82.74999999999999,\n            174.5,\n            111.25,\n            205.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05ece00485f6635d_40",\n          "bbox_xyxy": [\n            403.24999999999994,\n            189.00000000000003,\n            424.75,\n            238.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05ece00485f6635d_41",\n          "bbox_xyxy": [\n            147.75,\n            519.25,\n            171.25,\n            562.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05ece00485f6635d_42",\n          "bbox_xyxy": [\n            366.25,\n            462.75,\n            392.75000000000006,\n            507.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05ece00485f6635d_43",\n          "bbox_xyxy": [\n            81.74999999999999,\n            403.5,\n            105.25,\n            441.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05ece00485f6635d_44",\n          "bbox_xyxy": [\n            101.5,\n            392.5,\n            130.5,\n            421.50000000000006\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05ece00485f6635d_45",\n          "bbox_xyxy": [\n            575.5,\n            555.25,\n            605.5,\n            609.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05ece00485f6635d_46",\n          "bbox_xyxy": [\n            533.25,\n            480.0,\n            559.75,\n            514.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05ece00485f6635d_47",\n          "bbox_xyxy": [\n            200.0,\n            264.0,\n            227.99999999999997,\n            307.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05ece00485f6635d_48",\n          "bbox_xyxy": [\n            120.5,\n            607.5,\n            144.5,\n            639.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05ece00485f6635d_49",\n          "bbox_xyxy": [\n            185.75,\n            605.25,\n            204.25,\n            639.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05ece00485f6635d_50",\n          "bbox_xyxy": [\n            327.0,\n            245.0,\n            355.0,\n            276.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05ece00485f6635d_51",\n          "bbox_xyxy": [\n            45.0,\n            565.0,\n            68.0,\n            607.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05ece00485f6635d_52",\n          "bbox_xyxy": [\n            319.0,\n            426.0,\n            340.0,\n            451.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05ece00485f6635d_53",\n          "bbox_xyxy": [\n            0.2500000000000002,\n            490.0,\n            15.750000000000002,\n            517.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05ece00485f6635d_54",\n          "bbox_xyxy": [\n            528.75,\n            369.25,\n            561.25,\n            400.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_05ece00485f6635d_55",\n          "bbox_xyxy": [\n            286.75,\n            448.75,\n            319.25,\n            483.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_1149_JPG.rf.f817c8cbadf926541a6a5edf11480fed.jpg",\n        "size": 40872,\n        "compressed": 40585,\n        "offset": 483996417,\n        "crc32": "c454fb77",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_0617d26dcf45642f",\n      "relative_path": "images/IMG_1220_JPG.rf.2db1360fed00b56b6f05145675a2956b.jpg",\n      "sha256": "1f68d51429131a80dd59684ee65e9707277c2190eef7447e6915e3e11c79e65b",\n      "decoded_sha256": "6f75f5eb8e3bc3039c0adc6611eaf7883c1e1632b3bb3190bdd37109d49e5a5a",\n      "dhash": "2c4847974dd69561",\n      "bytes": 41020,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_1220_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_0617d26dcf45642f_0",\n          "bbox_xyxy": [\n            121.5,\n            281.75,\n            160.5,\n            323.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0617d26dcf45642f_1",\n          "bbox_xyxy": [\n            407.75,\n            447.25,\n            440.25,\n            492.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0617d26dcf45642f_2",\n          "bbox_xyxy": [\n            465.99999999999994,\n            528.75,\n            505.0,\n            571.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0617d26dcf45642f_3",\n          "bbox_xyxy": [\n            572.25,\n            261.5,\n            610.75,\n            310.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0617d26dcf45642f_4",\n          "bbox_xyxy": [\n            130.75,\n            97.25,\n            168.25,\n            139.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0617d26dcf45642f_5",\n          "bbox_xyxy": [\n            275.0,\n            433.75,\n            309.0,\n            482.24999999999994\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0617d26dcf45642f_6",\n          "bbox_xyxy": [\n            40.5,\n            272.75,\n            66.5,\n            328.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0617d26dcf45642f_7",\n          "bbox_xyxy": [\n            356.0,\n            5.249999999999999,\n            383.00000000000006,\n            60.74999999999999\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0617d26dcf45642f_8",\n          "bbox_xyxy": [\n            517.25,\n            399.0,\n            545.75,\n            464.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0617d26dcf45642f_9",\n          "bbox_xyxy": [\n            326.99999999999994,\n            563.25,\n            369.0,\n            606.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0617d26dcf45642f_10",\n          "bbox_xyxy": [\n            141.25,\n            145.5,\n            169.75,\n            195.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0617d26dcf45642f_11",\n          "bbox_xyxy": [\n            562.5,\n            49.0,\n            591.5,\n            106.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0617d26dcf45642f_12",\n          "bbox_xyxy": [\n            164.25,\n            208.5,\n            202.75000000000003,\n            246.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0617d26dcf45642f_13",\n          "bbox_xyxy": [\n            599.5,\n            436.99999999999994,\n            630.5,\n            501.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0617d26dcf45642f_14",\n          "bbox_xyxy": [\n            399.25,\n            18.25,\n            435.75,\n            56.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0617d26dcf45642f_15",\n          "bbox_xyxy": [\n            501.0,\n            35.24999999999999,\n            526.0,\n            91.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0617d26dcf45642f_16",\n          "bbox_xyxy": [\n            456.50000000000006,\n            282.0,\n            492.5,\n            333.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0617d26dcf45642f_17",\n          "bbox_xyxy": [\n            314.25,\n            478.24999999999994,\n            345.75,\n            539.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0617d26dcf45642f_18",\n          "bbox_xyxy": [\n            292.75,\n            259.75,\n            331.25,\n            307.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0617d26dcf45642f_19",\n          "bbox_xyxy": [\n            580.25,\n            173.5,\n            613.75,\n            203.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0617d26dcf45642f_20",\n          "bbox_xyxy": [\n            219.50000000000003,\n            61.49999999999999,\n            252.5,\n            102.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0617d26dcf45642f_21",\n          "bbox_xyxy": [\n            21.500000000000004,\n            125.5,\n            57.5,\n            177.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0617d26dcf45642f_22",\n          "bbox_xyxy": [\n            510.75,\n            239.75,\n            546.25,\n            286.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0617d26dcf45642f_23",\n          "bbox_xyxy": [\n            395.25,\n            118.75,\n            439.75,\n            164.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0617d26dcf45642f_24",\n          "bbox_xyxy": [\n            253.0,\n            172.24999999999997,\n            294.0,\n            215.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0617d26dcf45642f_25",\n          "bbox_xyxy": [\n            35.0,\n            63.75,\n            61.0,\n            118.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0617d26dcf45642f_26",\n          "bbox_xyxy": [\n            271.75,\n            106.00000000000001,\n            297.25,\n            160.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0617d26dcf45642f_27",\n          "bbox_xyxy": [\n            458.25,\n            384.5,\n            496.75,\n            420.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0617d26dcf45642f_28",\n          "bbox_xyxy": [\n            222.75,\n            591.0,\n            260.25,\n            630.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0617d26dcf45642f_29",\n          "bbox_xyxy": [\n            592.0,\n            557.75,\n            624.0,\n            606.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0617d26dcf45642f_30",\n          "bbox_xyxy": [\n            391.25,\n            291.25,\n            423.75,\n            337.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0617d26dcf45642f_31",\n          "bbox_xyxy": [\n            448.0,\n            33.75,\n            473.0,\n            90.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0617d26dcf45642f_32",\n          "bbox_xyxy": [\n            346.75,\n            110.75,\n            386.25,\n            166.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0617d26dcf45642f_33",\n          "bbox_xyxy": [\n            288.75,\n            0.0,\n            326.25,\n            20.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0617d26dcf45642f_34",\n          "bbox_xyxy": [\n            158.25,\n            395.75,\n            191.75,\n            441.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0617d26dcf45642f_35",\n          "bbox_xyxy": [\n            474.75,\n            121.25,\n            502.25,\n            180.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0617d26dcf45642f_36",\n          "bbox_xyxy": [\n            100.0,\n            498.25,\n            135.0,\n            536.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_1220_JPG.rf.2db1360fed00b56b6f05145675a2956b.jpg",\n        "size": 41020,\n        "compressed": 40830,\n        "offset": 496389564,\n        "crc32": "47b41d8b",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_061aed89f1b393fe",\n      "relative_path": "images/IMG_4228_JPG.rf.dbda1e88f6167ef3bbd191a8b031a99c.jpg",\n      "sha256": "50c9afc28c40bc431f115f9a5681e1902768ca9cb4451a7114ebb0f664bdad44",\n      "decoded_sha256": "9e98c7704fc4842dacdc98033f9df2d4eb51739f28d5b5c185d2e56ea4a7fe6d",\n      "dhash": "e8e5c1b0e9f491e1",\n      "bytes": 36568,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_4228_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_061aed89f1b393fe_0",\n          "bbox_xyxy": [\n            476.0,\n            407.5,\n            518.0,\n            433.49999999999994\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_061aed89f1b393fe_1",\n          "bbox_xyxy": [\n            418.75,\n            469.5,\n            459.25,\n            500.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_061aed89f1b393fe_2",\n          "bbox_xyxy": [\n            333.5,\n            495.75,\n            358.5,\n            539.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_061aed89f1b393fe_3",\n          "bbox_xyxy": [\n            261.25,\n            438.5,\n            301.75,\n            465.50000000000006\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_061aed89f1b393fe_4",\n          "bbox_xyxy": [\n            103.25,\n            247.5,\n            140.75,\n            272.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_061aed89f1b393fe_5",\n          "bbox_xyxy": [\n            324.0,\n            235.0,\n            359.0,\n            264.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_061aed89f1b393fe_6",\n          "bbox_xyxy": [\n            239.0,\n            321.75000000000006,\n            274.0,\n            355.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_061aed89f1b393fe_7",\n          "bbox_xyxy": [\n            440.25,\n            100.0,\n            467.75,\n            139.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_061aed89f1b393fe_8",\n          "bbox_xyxy": [\n            377.0,\n            24.5,\n            412.0,\n            50.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_061aed89f1b393fe_9",\n          "bbox_xyxy": [\n            153.25,\n            171.5,\n            192.75000000000003,\n            197.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_4228_JPG.rf.dbda1e88f6167ef3bbd191a8b031a99c.jpg",\n        "size": 36568,\n        "compressed": 36274,\n        "offset": 616149921,\n        "crc32": "bb83dcb0",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_0638b8fcdec80982",\n      "relative_path": "images/IMG_4223_JPG.rf.0114cba4231a0153d6fb79f3daacf84c.jpg",\n      "sha256": "4efda0e6e030dffd117102cdaa9635502e5beaff2f861d30f5d3ec015b7b6ebf",\n      "decoded_sha256": "968f0d38b02b786f5b7cb5f73d12224377758fbe32b93cdc56b8ed39b9c02370",\n      "dhash": "c19cb9a4e4f0ecec",\n      "bytes": 42374,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_4223_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_0638b8fcdec80982_0",\n          "bbox_xyxy": [\n            368.0,\n            200.25,\n            405.0,\n            223.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0638b8fcdec80982_1",\n          "bbox_xyxy": [\n            449.0,\n            111.0,\n            471.99999999999994,\n            149.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0638b8fcdec80982_2",\n          "bbox_xyxy": [\n            261.75,\n            194.25,\n            300.25,\n            217.75000000000003\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0638b8fcdec80982_3",\n          "bbox_xyxy": [\n            107.75,\n            303.5,\n            144.25,\n            327.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0638b8fcdec80982_4",\n          "bbox_xyxy": [\n            187.25,\n            315.75,\n            223.75,\n            339.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0638b8fcdec80982_5",\n          "bbox_xyxy": [\n            253.25,\n            390.0,\n            282.75,\n            420.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0638b8fcdec80982_6",\n          "bbox_xyxy": [\n            182.75000000000003,\n            403.00000000000006,\n            218.25,\n            426.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0638b8fcdec80982_7",\n          "bbox_xyxy": [\n            219.25,\n            495.5,\n            254.74999999999997,\n            517.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0638b8fcdec80982_8",\n          "bbox_xyxy": [\n            490.25,\n            292.0,\n            520.75,\n            318.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0638b8fcdec80982_9",\n          "bbox_xyxy": [\n            111.25,\n            188.49999999999997,\n            133.75,\n            224.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_4223_JPG.rf.0114cba4231a0153d6fb79f3daacf84c.jpg",\n        "size": 42374,\n        "compressed": 42174,\n        "offset": 614786600,\n        "crc32": "7922e7ab",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_06484ae5c899a926",\n      "relative_path": "images/IMG_0884_JPG.rf.0311c215f34bbc20fd50fde9bdcdf99e.jpg",\n      "sha256": "9d4d9c99a64015192d5b5c43a95a743ed0d18d199dd1c67ebab4fea590e8ef6d",\n      "decoded_sha256": "ee74c743f2b12fdd2a94c1dfc689781923f144f105ca60ef70f0bdb79b15a365",\n      "dhash": "dbb24b232e0d1615",\n      "bytes": 39613,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_0884_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_06484ae5c899a926_0",\n          "bbox_xyxy": [\n            278.75,\n            81.74999999999999,\n            334.25,\n            152.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06484ae5c899a926_1",\n          "bbox_xyxy": [\n            41.75000000000001,\n            0.0,\n            81.25,\n            62.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06484ae5c899a926_2",\n          "bbox_xyxy": [\n            79.75,\n            66.75,\n            135.25,\n            103.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06484ae5c899a926_3",\n          "bbox_xyxy": [\n            149.25,\n            12.750000000000004,\n            209.74999999999997,\n            66.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06484ae5c899a926_4",\n          "bbox_xyxy": [\n            387.0,\n            57.50000000000001,\n            427.0,\n            143.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06484ae5c899a926_5",\n          "bbox_xyxy": [\n            482.5,\n            87.74999999999999,\n            525.5,\n            174.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06484ae5c899a926_6",\n          "bbox_xyxy": [\n            532.25,\n            32.5,\n            567.75,\n            117.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06484ae5c899a926_7",\n          "bbox_xyxy": [\n            584.0,\n            0.25000000000000355,\n            638.0,\n            58.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06484ae5c899a926_8",\n          "bbox_xyxy": [\n            588.0,\n            127.0,\n            637.0,\n            193.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06484ae5c899a926_9",\n          "bbox_xyxy": [\n            541.0,\n            212.5,\n            595.0,\n            282.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06484ae5c899a926_10",\n          "bbox_xyxy": [\n            352.0,\n            177.75,\n            403.0,\n            252.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06484ae5c899a926_11",\n          "bbox_xyxy": [\n            28.75,\n            168.75,\n            110.25,\n            209.24999999999997\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06484ae5c899a926_12",\n          "bbox_xyxy": [\n            135.00000000000003,\n            149.0,\n            192.00000000000003,\n            215.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06484ae5c899a926_13",\n          "bbox_xyxy": [\n            190.75,\n            129.0,\n            224.25,\n            221.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06484ae5c899a926_14",\n          "bbox_xyxy": [\n            225.0,\n            143.49999999999997,\n            256.0,\n            234.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06484ae5c899a926_15",\n          "bbox_xyxy": [\n            0.2499999999999991,\n            255.5,\n            51.75000000000001,\n            366.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06484ae5c899a926_16",\n          "bbox_xyxy": [\n            100.5,\n            262.5,\n            147.5,\n            346.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06484ae5c899a926_17",\n          "bbox_xyxy": [\n            585.25,\n            392.75,\n            618.75,\n            467.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06484ae5c899a926_18",\n          "bbox_xyxy": [\n            508.75,\n            482.75,\n            548.25,\n            574.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06484ae5c899a926_19",\n          "bbox_xyxy": [\n            406.0,\n            449.75,\n            466.99999999999994,\n            504.25000000000006\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06484ae5c899a926_20",\n          "bbox_xyxy": [\n            415.25,\n            545.5,\n            472.75,\n            616.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06484ae5c899a926_21",\n          "bbox_xyxy": [\n            389.74999999999994,\n            318.5,\n            441.25,\n            398.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06484ae5c899a926_22",\n          "bbox_xyxy": [\n            321.25,\n            451.5,\n            370.75,\n            496.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06484ae5c899a926_23",\n          "bbox_xyxy": [\n            278.0,\n            517.25,\n            330.0,\n            604.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06484ae5c899a926_24",\n          "bbox_xyxy": [\n            202.25,\n            357.75000000000006,\n            238.75,\n            431.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06484ae5c899a926_25",\n          "bbox_xyxy": [\n            168.5,\n            386.75000000000006,\n            206.5,\n            442.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06484ae5c899a926_26",\n          "bbox_xyxy": [\n            12.25,\n            526.5,\n            57.75,\n            622.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06484ae5c899a926_27",\n          "bbox_xyxy": [\n            42.25000000000001,\n            394.0,\n            115.75,\n            471.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_0884_JPG.rf.0311c215f34bbc20fd50fde9bdcdf99e.jpg",\n        "size": 39613,\n        "compressed": 38658,\n        "offset": 456443802,\n        "crc32": "71619ef2",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_065a304fc91fe173",\n      "relative_path": "images/20251107_163248_jpg.rf.8c90756176cc2cf2a71c35563792599e.jpg",\n      "sha256": "e8c42df783a74a2c67fefb31a6ece947d2c9eab58ad6d1a14eb452258fe0248e",\n      "decoded_sha256": "60b8fbc3a3c6cbe1bc7eb20e6f07a9d55c59ae88d37bbc0f713fd14b8e93dd9d",\n      "dhash": "8f3b07cdf366399d",\n      "bytes": 104907,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "20251107_163248_jpg",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_065a304fc91fe173_0",\n          "bbox_xyxy": [\n            206.5,\n            176.75,\n            261.5,\n            207.24999999999997\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_065a304fc91fe173_1",\n          "bbox_xyxy": [\n            383.49999999999994,\n            328.75,\n            437.5,\n            351.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_065a304fc91fe173_2",\n          "bbox_xyxy": [\n            67.0,\n            540.25,\n            122.0,\n            559.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_065a304fc91fe173_3",\n          "bbox_xyxy": [\n            456.0,\n            253.75,\n            503.00000000000006,\n            278.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_065a304fc91fe173_4",\n          "bbox_xyxy": [\n            134.25,\n            244.5,\n            190.75,\n            264.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_065a304fc91fe173_5",\n          "bbox_xyxy": [\n            85.25,\n            302.25,\n            122.75,\n            335.74999999999994\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_065a304fc91fe173_6",\n          "bbox_xyxy": [\n            310.00000000000006,\n            292.75,\n            359.00000000000006,\n            312.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_065a304fc91fe173_7",\n          "bbox_xyxy": [\n            198.0,\n            145.0,\n            232.0,\n            179.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_065a304fc91fe173_8",\n          "bbox_xyxy": [\n            400.0,\n            383.49999999999994,\n            445.99999999999994,\n            417.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_065a304fc91fe173_9",\n          "bbox_xyxy": [\n            240.75,\n            299.25,\n            283.25,\n            329.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_065a304fc91fe173_10",\n          "bbox_xyxy": [\n            297.75,\n            399.5,\n            346.25,\n            435.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_065a304fc91fe173_11",\n          "bbox_xyxy": [\n            461.0,\n            613.0,\n            496.0,\n            640.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_065a304fc91fe173_12",\n          "bbox_xyxy": [\n            43.24999999999999,\n            122.74999999999999,\n            68.75,\n            156.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_065a304fc91fe173_13",\n          "bbox_xyxy": [\n            302.5,\n            346.25,\n            360.5,\n            378.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_065a304fc91fe173_14",\n          "bbox_xyxy": [\n            524.5,\n            519.0,\n            575.5,\n            533.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_065a304fc91fe173_15",\n          "bbox_xyxy": [\n            135.5,\n            293.25,\n            171.5,\n            330.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_065a304fc91fe173_16",\n          "bbox_xyxy": [\n            367.25,\n            88.75,\n            394.75,\n            132.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_065a304fc91fe173_17",\n          "bbox_xyxy": [\n            443.24999999999994,\n            483.75,\n            477.75,\n            508.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_065a304fc91fe173_18",\n          "bbox_xyxy": [\n            348.24999999999994,\n            467.5,\n            402.75,\n            495.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_065a304fc91fe173_19",\n          "bbox_xyxy": [\n            564.0,\n            145.5,\n            619.0,\n            169.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_065a304fc91fe173_20",\n          "bbox_xyxy": [\n            536.25,\n            229.25,\n            558.75,\n            252.75000000000003\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_065a304fc91fe173_21",\n          "bbox_xyxy": [\n            317.75,\n            599.0,\n            356.25,\n            634.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_065a304fc91fe173_22",\n          "bbox_xyxy": [\n            157.75,\n            453.24999999999994,\n            183.25,\n            477.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_065a304fc91fe173_23",\n          "bbox_xyxy": [\n            265.75,\n            521.25,\n            307.25,\n            532.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_065a304fc91fe173_24",\n          "bbox_xyxy": [\n            159.74999999999997,\n            339.0,\n            193.25,\n            356.99999999999994\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_065a304fc91fe173_25",\n          "bbox_xyxy": [\n            118.25,\n            275.5,\n            165.75,\n            288.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_065a304fc91fe173_26",\n          "bbox_xyxy": [\n            197.99999999999997,\n            408.75,\n            245.0,\n            427.24999999999994\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/20251107_163248_jpg.rf.8c90756176cc2cf2a71c35563792599e.jpg",\n        "size": 104907,\n        "compressed": 104763,\n        "offset": 341611603,\n        "crc32": "b1955185",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_06b492f311b1032e",\n      "relative_path": "images/IMG_0729_JPG.rf.ddf797a9526d4c937b6d3ee1526db283.jpg",\n      "sha256": "6475f911b03a6eea40e4c91c95239e568d7c5c4c1f9b36b789f53133343eea0e",\n      "decoded_sha256": "a44c8916ac3eed670cd0a293f170f7e28caff3cb38e69c49a3de5626ce1a7434",\n      "dhash": "5d939a6ae01969a3",\n      "bytes": 54480,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_0729_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_06b492f311b1032e_0",\n          "bbox_xyxy": [\n            288.0,\n            29.25,\n            357.0,\n            77.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06b492f311b1032e_1",\n          "bbox_xyxy": [\n            255.25000000000003,\n            63.250000000000014,\n            315.75,\n            122.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06b492f311b1032e_2",\n          "bbox_xyxy": [\n            368.25,\n            33.0,\n            461.75,\n            58.99999999999999\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06b492f311b1032e_3",\n          "bbox_xyxy": [\n            477.25,\n            168.25000000000003,\n            544.75,\n            228.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06b492f311b1032e_4",\n          "bbox_xyxy": [\n            139.0,\n            163.0,\n            220.0,\n            217.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06b492f311b1032e_5",\n          "bbox_xyxy": [\n            250.75,\n            160.5,\n            319.25,\n            216.50000000000003\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06b492f311b1032e_6",\n          "bbox_xyxy": [\n            230.0,\n            278.25,\n            319.0,\n            307.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06b492f311b1032e_7",\n          "bbox_xyxy": [\n            343.25,\n            273.0,\n            426.75,\n            331.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06b492f311b1032e_8",\n          "bbox_xyxy": [\n            108.5,\n            342.5,\n            170.5,\n            412.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06b492f311b1032e_9",\n          "bbox_xyxy": [\n            28.750000000000004,\n            283.5,\n            84.25,\n            346.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06b492f311b1032e_10",\n          "bbox_xyxy": [\n            40.75,\n            187.5,\n            98.25,\n            246.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06b492f311b1032e_11",\n          "bbox_xyxy": [\n            524.5,\n            251.25,\n            607.5,\n            283.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06b492f311b1032e_12",\n          "bbox_xyxy": [\n            407.25,\n            358.49999999999994,\n            459.75,\n            427.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06b492f311b1032e_13",\n          "bbox_xyxy": [\n            289.5,\n            369.0,\n            339.5,\n            431.99999999999994\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06b492f311b1032e_14",\n          "bbox_xyxy": [\n            338.75,\n            431.75000000000006,\n            410.25000000000006,\n            487.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06b492f311b1032e_15",\n          "bbox_xyxy": [\n            193.75,\n            511.75000000000006,\n            275.25,\n            560.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06b492f311b1032e_16",\n          "bbox_xyxy": [\n            13.75,\n            418.24999999999994,\n            110.25,\n            472.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06b492f311b1032e_17",\n          "bbox_xyxy": [\n            309.0,\n            110.25,\n            373.00000000000006,\n            155.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06b492f311b1032e_18",\n          "bbox_xyxy": [\n            397.25,\n            72.25,\n            486.75000000000006,\n            111.74999999999999\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06b492f311b1032e_19",\n          "bbox_xyxy": [\n            342.0,\n            46.75000000000001,\n            417.0,\n            92.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06b492f311b1032e_20",\n          "bbox_xyxy": [\n            514.25,\n            96.49999999999999,\n            597.75,\n            137.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06b492f311b1032e_21",\n          "bbox_xyxy": [\n            545.75,\n            134.0,\n            596.25,\n            189.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06b492f311b1032e_22",\n          "bbox_xyxy": [\n            356.5,\n            174.75,\n            421.5,\n            235.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06b492f311b1032e_23",\n          "bbox_xyxy": [\n            12.750000000000004,\n            119.25,\n            106.25,\n            170.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06b492f311b1032e_24",\n          "bbox_xyxy": [\n            115.75,\n            76.75,\n            173.25,\n            138.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06b492f311b1032e_25",\n          "bbox_xyxy": [\n            184.5,\n            12.999999999999998,\n            250.5,\n            65.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06b492f311b1032e_26",\n          "bbox_xyxy": [\n            28.0,\n            0.0,\n            116.00000000000001,\n            39.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06b492f311b1032e_27",\n          "bbox_xyxy": [\n            437.25,\n            0.2500000000000002,\n            524.75,\n            16.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06b492f311b1032e_28",\n          "bbox_xyxy": [\n            560.25,\n            0.2499999999999991,\n            639.75,\n            42.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06b492f311b1032e_29",\n          "bbox_xyxy": [\n            482.25,\n            277.0,\n            574.75,\n            325.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06b492f311b1032e_30",\n          "bbox_xyxy": [\n            202.0,\n            332.25,\n            274.0,\n            390.74999999999994\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_06b492f311b1032e_31",\n          "bbox_xyxy": [\n            164.24999999999997,\n            252.5,\n            254.75,\n            305.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_0729_JPG.rf.ddf797a9526d4c937b6d3ee1526db283.jpg",\n        "size": 54480,\n        "compressed": 54351,\n        "offset": 433186843,\n        "crc32": "943c1387",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_06c0734e7de092e4",\n      "relative_path": "images/IMG_2591_JPG.rf.25143f868ad1f5120efa5310548d3a09.jpg",\n      "sha256": "b7901bc8664afb95daf4c8730fa75cbbcbce4bde667878fa35f2d9a2af246323",\n      "decoded_sha256": "2878b55dc0b7b06344def41557d78b95385de9e10c2b833e192add2fa2b93077",\n      "dhash": "3030787878707050",\n      "bytes": 23889,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2591_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_06c0734e7de092e4_0",\n          "bbox_xyxy": [\n            140.99999999999997,\n            0.0,\n            482.0,\n            614.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_2591_JPG.rf.25143f868ad1f5120efa5310548d3a09.jpg",\n        "size": 23889,\n        "compressed": 19798,\n        "offset": 568216258,\n        "crc32": "b49f516a",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_0777642e923889f7",\n      "relative_path": "images/IMG_0842_JPG.rf.f123513e331e52e9961e2281d3f18ba9.jpg",\n      "sha256": "37052a45367821a4fe1631dd6e5481ab8026ad9be09785f6126f65841724cbcf",\n      "decoded_sha256": "941dfbfc80c0bdc3a294308486a402d94e3fd1e826a2182891be204ddfd2beb5",\n      "dhash": "4453417447594912",\n      "bytes": 53275,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_0842_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_0777642e923889f7_0",\n          "bbox_xyxy": [\n            179.00000000000003,\n            16.999999999999996,\n            258.0,\n            44.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0777642e923889f7_1",\n          "bbox_xyxy": [\n            423.00000000000006,\n            15.499999999999998,\n            471.0,\n            60.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0777642e923889f7_2",\n          "bbox_xyxy": [\n            451.25,\n            14.999999999999998,\n            523.75,\n            41.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0777642e923889f7_3",\n          "bbox_xyxy": [\n            455.74999999999994,\n            0.0,\n            515.25,\n            19.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0777642e923889f7_4",\n          "bbox_xyxy": [\n            436.5,\n            107.5,\n            491.5,\n            151.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0777642e923889f7_5",\n          "bbox_xyxy": [\n            605.0,\n            99.75,\n            640.0,\n            128.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0777642e923889f7_6",\n          "bbox_xyxy": [\n            566.0,\n            62.5,\n            625.0,\n            83.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0777642e923889f7_7",\n          "bbox_xyxy": [\n            551.25,\n            86.25,\n            586.75,\n            131.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0777642e923889f7_8",\n          "bbox_xyxy": [\n            536.5,\n            54.5,\n            570.5,\n            95.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0777642e923889f7_9",\n          "bbox_xyxy": [\n            431.99999999999994,\n            60.74999999999999,\n            466.0,\n            106.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0777642e923889f7_10",\n          "bbox_xyxy": [\n            154.0,\n            66.5,\n            201.0,\n            128.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0777642e923889f7_11",\n          "bbox_xyxy": [\n            195.0,\n            136.0,\n            249.0,\n            193.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0777642e923889f7_12",\n          "bbox_xyxy": [\n            245.75,\n            133.0,\n            319.25,\n            165.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0777642e923889f7_13",\n          "bbox_xyxy": [\n            298.75,\n            130.5,\n            364.25,\n            176.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0777642e923889f7_14",\n          "bbox_xyxy": [\n            348.5,\n            131.5,\n            415.50000000000006,\n            174.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0777642e923889f7_15",\n          "bbox_xyxy": [\n            231.75,\n            72.99999999999999,\n            294.25,\n            106.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0777642e923889f7_16",\n          "bbox_xyxy": [\n            322.24999999999994,\n            62.5,\n            393.75,\n            90.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0777642e923889f7_17",\n          "bbox_xyxy": [\n            305.00000000000006,\n            73.5,\n            389.00000000000006,\n            106.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0777642e923889f7_18",\n          "bbox_xyxy": [\n            410.0,\n            188.0,\n            485.0,\n            220.99999999999997\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0777642e923889f7_19",\n          "bbox_xyxy": [\n            376.25,\n            234.0,\n            408.75,\n            275.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0777642e923889f7_20",\n          "bbox_xyxy": [\n            417.5,\n            248.25,\n            490.5,\n            269.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0777642e923889f7_21",\n          "bbox_xyxy": [\n            254.25,\n            205.49999999999997,\n            336.75,\n            232.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0777642e923889f7_22",\n          "bbox_xyxy": [\n            284.5,\n            260.5,\n            367.5,\n            294.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0777642e923889f7_23",\n          "bbox_xyxy": [\n            303.25,\n            269.5,\n            370.75,\n            301.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0777642e923889f7_24",\n          "bbox_xyxy": [\n            142.5,\n            202.5,\n            222.5,\n            233.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0777642e923889f7_25",\n          "bbox_xyxy": [\n            217.25,\n            261.75,\n            252.75,\n            311.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0777642e923889f7_26",\n          "bbox_xyxy": [\n            265.25,\n            314.49999999999994,\n            336.75000000000006,\n            361.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0777642e923889f7_27",\n          "bbox_xyxy": [\n            170.99999999999997,\n            321.75000000000006,\n            217.0,\n            375.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0777642e923889f7_28",\n          "bbox_xyxy": [\n            136.0,\n            307.25,\n            175.0,\n            360.74999999999994\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0777642e923889f7_29",\n          "bbox_xyxy": [\n            306.25,\n            390.0,\n            345.75,\n            445.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0777642e923889f7_30",\n          "bbox_xyxy": [\n            427.0,\n            351.5,\n            508.0,\n            373.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0777642e923889f7_31",\n          "bbox_xyxy": [\n            367.25,\n            428.5,\n            427.75,\n            463.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0777642e923889f7_32",\n          "bbox_xyxy": [\n            364.75,\n            471.25,\n            434.25000000000006,\n            505.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0777642e923889f7_33",\n          "bbox_xyxy": [\n            448.75,\n            467.75,\n            512.25,\n            519.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0777642e923889f7_34",\n          "bbox_xyxy": [\n            423.5,\n            526.25,\n            516.5,\n            551.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0777642e923889f7_35",\n          "bbox_xyxy": [\n            348.00000000000006,\n            521.5,\n            419.0,\n            541.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0777642e923889f7_36",\n          "bbox_xyxy": [\n            350.74999999999994,\n            575.0,\n            400.25,\n            633.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0777642e923889f7_37",\n          "bbox_xyxy": [\n            518.75,\n            582.25,\n            588.25,\n            638.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0777642e923889f7_38",\n          "bbox_xyxy": [\n            532.25,\n            523.0,\n            620.75,\n            548.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0777642e923889f7_39",\n          "bbox_xyxy": [\n            564.0,\n            475.25,\n            605.0,\n            530.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0777642e923889f7_40",\n          "bbox_xyxy": [\n            454.0,\n            272.5,\n            541.0,\n            302.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0777642e923889f7_41",\n          "bbox_xyxy": [\n            535.75,\n            309.75,\n            614.25,\n            353.24999999999994\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0777642e923889f7_42",\n          "bbox_xyxy": [\n            540.25,\n            341.99999999999994,\n            578.75,\n            404.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0777642e923889f7_43",\n          "bbox_xyxy": [\n            550.25,\n            228.49999999999997,\n            603.75,\n            285.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0777642e923889f7_44",\n          "bbox_xyxy": [\n            520.5,\n            156.75,\n            570.5,\n            210.25000000000003\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0777642e923889f7_45",\n          "bbox_xyxy": [\n            487.75,\n            56.25,\n            564.25,\n            84.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0777642e923889f7_46",\n          "bbox_xyxy": [\n            386.5,\n            375.75,\n            458.5,\n            413.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0777642e923889f7_47",\n          "bbox_xyxy": [\n            123.75,\n            436.0,\n            227.25,\n            466.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0777642e923889f7_48",\n          "bbox_xyxy": [\n            145.5,\n            512.25,\n            196.50000000000003,\n            572.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0777642e923889f7_49",\n          "bbox_xyxy": [\n            258.5,\n            573.5,\n            340.5,\n            604.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_0842_JPG.rf.f123513e331e52e9961e2281d3f18ba9.jpg",\n        "size": 53275,\n        "compressed": 53135,\n        "offset": 449947420,\n        "crc32": "98bad530",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_07cdbd20de04e757",\n      "relative_path": "images/IMG_2666_JPG.rf.b66729f35dff6090151c096a032aee79.jpg",\n      "sha256": "96e03c5704fdc407ddc66f8761fb6a068d7685d7dbb94a4be9fa2f4ad517ea6f",\n      "decoded_sha256": "f09d4456474cdbae686f0f3364fc14d783b97d396148b225fdd071eb336d1d9b",\n      "dhash": "da39317031397970",\n      "bytes": 29466,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2666_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_07cdbd20de04e757_0",\n          "bbox_xyxy": [\n            199.50000000000003,\n            52.75000000000002,\n            537.5,\n            606.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_2666_JPG.rf.b66729f35dff6090151c096a032aee79.jpg",\n        "size": 29466,\n        "compressed": 28749,\n        "offset": 575951104,\n        "crc32": "5d3cecd5",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_086a32a853c01822",\n      "relative_path": "images/IMG_1059_JPG.rf.b126e6f0a6589bbe15fc8e03b1bddd6b.jpg",\n      "sha256": "42b807cb0c8e97058748e46873e6815b438fa7856e35d6ee7e046b8964823451",\n      "decoded_sha256": "0b0a4e8876f4c31898c5fd9b6f4b271f1434baffb959e6f8bd9b7a9d5777024a",\n      "dhash": "0d361d5b592a6e24",\n      "bytes": 52532,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_1059_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_086a32a853c01822_0",\n          "bbox_xyxy": [\n            591.25,\n            548.25,\n            639.75,\n            585.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_086a32a853c01822_1",\n          "bbox_xyxy": [\n            352.5,\n            268.25,\n            383.49999999999994,\n            334.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_086a32a853c01822_2",\n          "bbox_xyxy": [\n            572.5,\n            447.5,\n            634.5,\n            471.50000000000006\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_086a32a853c01822_3",\n          "bbox_xyxy": [\n            185.5,\n            497.0,\n            240.5,\n            538.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_086a32a853c01822_4",\n          "bbox_xyxy": [\n            385.25,\n            332.24999999999994,\n            430.74999999999994,\n            378.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_086a32a853c01822_5",\n          "bbox_xyxy": [\n            549.0,\n            218.0,\n            588.0,\n            307.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_086a32a853c01822_6",\n          "bbox_xyxy": [\n            288.0,\n            129.0,\n            336.0,\n            167.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_086a32a853c01822_7",\n          "bbox_xyxy": [\n            185.25,\n            307.75,\n            233.75,\n            346.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_086a32a853c01822_8",\n          "bbox_xyxy": [\n            419.0,\n            188.75,\n            473.00000000000006,\n            222.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_086a32a853c01822_9",\n          "bbox_xyxy": [\n            391.0,\n            0.0,\n            426.99999999999994,\n            11.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_086a32a853c01822_10",\n          "bbox_xyxy": [\n            378.5,\n            399.75,\n            433.5,\n            443.24999999999994\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_086a32a853c01822_11",\n          "bbox_xyxy": [\n            475.0,\n            561.75,\n            507.0,\n            625.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_086a32a853c01822_12",\n          "bbox_xyxy": [\n            391.0,\n            210.25,\n            436.0,\n            264.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_086a32a853c01822_13",\n          "bbox_xyxy": [\n            388.0,\n            426.25,\n            420.0,\n            491.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_1059_JPG.rf.b126e6f0a6589bbe15fc8e03b1bddd6b.jpg",\n        "size": 52532,\n        "compressed": 52144,\n        "offset": 468111413,\n        "crc32": "e856262f",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_08a8b33c11fcfa88",\n      "relative_path": "images/IMG_0912_JPG.rf.26624f419f45652632fa602b7c3291a9.jpg",\n      "sha256": "50348fd4e27bd81d067c4a459c9ca2515804335b3858cef87b5ee4fb75109212",\n      "decoded_sha256": "131e1a3fb45165a0f37583ad0787a68a3278fd6faf52c275f1cfc3bedbd1d006",\n      "dhash": "0d27269c36561ab7",\n      "bytes": 42854,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_0912_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_08a8b33c11fcfa88_0",\n          "bbox_xyxy": [\n            219.5,\n            113.5,\n            260.5,\n            166.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08a8b33c11fcfa88_1",\n          "bbox_xyxy": [\n            232.25,\n            167.5,\n            272.75,\n            224.50000000000003\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08a8b33c11fcfa88_2",\n          "bbox_xyxy": [\n            287.5,\n            146.0,\n            338.5,\n            225.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08a8b33c11fcfa88_3",\n          "bbox_xyxy": [\n            435.25000000000006,\n            52.000000000000014,\n            483.75,\n            129.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08a8b33c11fcfa88_4",\n          "bbox_xyxy": [\n            490.0,\n            100.24999999999999,\n            536.0,\n            161.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08a8b33c11fcfa88_5",\n          "bbox_xyxy": [\n            579.25,\n            45.74999999999999,\n            617.75,\n            110.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08a8b33c11fcfa88_6",\n          "bbox_xyxy": [\n            541.5,\n            137.25,\n            567.5,\n            221.74999999999997\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08a8b33c11fcfa88_7",\n          "bbox_xyxy": [\n            625.25,\n            259.75,\n            639.75,\n            336.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08a8b33c11fcfa88_8",\n          "bbox_xyxy": [\n            446.99999999999994,\n            231.5,\n            501.0,\n            267.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08a8b33c11fcfa88_9",\n          "bbox_xyxy": [\n            404.0,\n            217.0,\n            443.00000000000006,\n            298.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08a8b33c11fcfa88_10",\n          "bbox_xyxy": [\n            479.25,\n            268.25,\n            526.75,\n            303.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08a8b33c11fcfa88_11",\n          "bbox_xyxy": [\n            457.0,\n            305.25,\n            505.0,\n            374.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08a8b33c11fcfa88_12",\n          "bbox_xyxy": [\n            448.5,\n            381.75000000000006,\n            474.49999999999994,\n            450.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08a8b33c11fcfa88_13",\n          "bbox_xyxy": [\n            510.25,\n            384.0,\n            549.75,\n            439.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08a8b33c11fcfa88_14",\n          "bbox_xyxy": [\n            552.5,\n            368.0,\n            584.5,\n            442.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08a8b33c11fcfa88_15",\n          "bbox_xyxy": [\n            582.75,\n            442.0,\n            631.25,\n            518.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08a8b33c11fcfa88_16",\n          "bbox_xyxy": [\n            475.25,\n            461.0,\n            529.75,\n            508.00000000000006\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08a8b33c11fcfa88_17",\n          "bbox_xyxy": [\n            487.75,\n            508.75,\n            537.25,\n            559.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08a8b33c11fcfa88_18",\n          "bbox_xyxy": [\n            514.25,\n            593.0,\n            566.75,\n            640.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08a8b33c11fcfa88_19",\n          "bbox_xyxy": [\n            576.25,\n            553.75,\n            603.75,\n            623.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08a8b33c11fcfa88_20",\n          "bbox_xyxy": [\n            594.0,\n            590.25,\n            640.0,\n            639.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08a8b33c11fcfa88_21",\n          "bbox_xyxy": [\n            109.25,\n            256.5,\n            166.75,\n            332.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08a8b33c11fcfa88_22",\n          "bbox_xyxy": [\n            60.750000000000014,\n            536.75,\n            110.25,\n            621.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08a8b33c11fcfa88_23",\n          "bbox_xyxy": [\n            172.5,\n            382.5,\n            228.5,\n            464.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08a8b33c11fcfa88_24",\n          "bbox_xyxy": [\n            190.75,\n            556.25,\n            238.25,\n            603.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08a8b33c11fcfa88_25",\n          "bbox_xyxy": [\n            257.5,\n            579.25,\n            299.5,\n            639.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08a8b33c11fcfa88_26",\n          "bbox_xyxy": [\n            317.50000000000006,\n            491.5,\n            346.50000000000006,\n            560.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08a8b33c11fcfa88_27",\n          "bbox_xyxy": [\n            307.5,\n            545.5,\n            350.5,\n            608.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08a8b33c11fcfa88_28",\n          "bbox_xyxy": [\n            380.25,\n            448.75,\n            416.75000000000006,\n            539.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08a8b33c11fcfa88_29",\n          "bbox_xyxy": [\n            408.25,\n            593.25,\n            450.75,\n            639.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08a8b33c11fcfa88_30",\n          "bbox_xyxy": [\n            331.5,\n            287.5,\n            374.49999999999994,\n            349.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08a8b33c11fcfa88_31",\n          "bbox_xyxy": [\n            276.25,\n            345.0,\n            319.75,\n            415.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08a8b33c11fcfa88_32",\n          "bbox_xyxy": [\n            301.25,\n            397.0,\n            353.75,\n            463.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_0912_JPG.rf.26624f419f45652632fa602b7c3291a9.jpg",\n        "size": 42854,\n        "compressed": 42620,\n        "offset": 460567925,\n        "crc32": "216bc7db",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_08defe7751c9c429",\n      "relative_path": "images/IMG_1060_JPG.rf.12b467e711dbc8784ae5a9d1aab02b6c.jpg",\n      "sha256": "1a7a3d478eca1503df43e95814569629bc14fcc93202f440624c0ec61e8030b2",\n      "decoded_sha256": "8db607e0a8d31ee2ce5e74dda223161b3de7303a11cb37b1e774308fc293aa26",\n      "dhash": "6e932a661b2e290f",\n      "bytes": 50086,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_1060_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_08defe7751c9c429_0",\n          "bbox_xyxy": [\n            375.25,\n            336.25,\n            417.75,\n            411.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08defe7751c9c429_1",\n          "bbox_xyxy": [\n            196.25,\n            253.5,\n            230.75000000000003,\n            333.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08defe7751c9c429_2",\n          "bbox_xyxy": [\n            530.25,\n            422.5,\n            573.75,\n            469.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08defe7751c9c429_3",\n          "bbox_xyxy": [\n            195.5,\n            0.2499999999999991,\n            244.5,\n            34.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08defe7751c9c429_4",\n          "bbox_xyxy": [\n            518.25,\n            102.00000000000001,\n            562.75,\n            159.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08defe7751c9c429_5",\n          "bbox_xyxy": [\n            83.99999999999999,\n            81.74999999999999,\n            135.0,\n            127.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08defe7751c9c429_6",\n          "bbox_xyxy": [\n            226.0,\n            21.75,\n            286.0,\n            59.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08defe7751c9c429_7",\n          "bbox_xyxy": [\n            562.5,\n            71.5,\n            616.5,\n            111.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08defe7751c9c429_8",\n          "bbox_xyxy": [\n            582.0,\n            318.25,\n            614.0,\n            398.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08defe7751c9c429_9",\n          "bbox_xyxy": [\n            624.25,\n            89.24999999999999,\n            639.75,\n            122.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_1060_JPG.rf.12b467e711dbc8784ae5a9d1aab02b6c.jpg",\n        "size": 50086,\n        "compressed": 49890,\n        "offset": 468217798,\n        "crc32": "38ddbf54",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_08f11cb3638c7c9c",\n      "relative_path": "images/20251107_163309_jpg.rf.b90a53e13b8d803e41e265a1c4e97005.jpg",\n      "sha256": "aa40603d0106683959350f382f93236a7f216abeca4a2e997f21b2bb9b29f4b6",\n      "decoded_sha256": "b63c237726dea98160a3b523f7a2a8177ffde67e05dfc0a59e04d67d8c26c5ae",\n      "dhash": "eb7b3a1b9b97b931",\n      "bytes": 125130,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "20251107_163309_jpg",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_08f11cb3638c7c9c_0",\n          "bbox_xyxy": [\n            238.75,\n            23.750000000000004,\n            292.25,\n            74.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08f11cb3638c7c9c_1",\n          "bbox_xyxy": [\n            582.25,\n            429.0,\n            639.75,\n            461.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08f11cb3638c7c9c_2",\n          "bbox_xyxy": [\n            485.25,\n            301.25,\n            537.75,\n            320.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08f11cb3638c7c9c_3",\n          "bbox_xyxy": [\n            485.0,\n            431.75,\n            545.0,\n            468.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08f11cb3638c7c9c_4",\n          "bbox_xyxy": [\n            348.75,\n            6.000000000000001,\n            405.25000000000006,\n            48.00000000000001\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08f11cb3638c7c9c_5",\n          "bbox_xyxy": [\n            539.5,\n            503.49999999999994,\n            584.5,\n            542.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08f11cb3638c7c9c_6",\n          "bbox_xyxy": [\n            248.25,\n            405.0,\n            279.75,\n            438.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08f11cb3638c7c9c_7",\n          "bbox_xyxy": [\n            430.25,\n            428.24999999999994,\n            470.74999999999994,\n            464.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08f11cb3638c7c9c_8",\n          "bbox_xyxy": [\n            356.75,\n            359.5,\n            396.25,\n            389.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08f11cb3638c7c9c_9",\n          "bbox_xyxy": [\n            421.75,\n            257.25,\n            468.25,\n            292.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08f11cb3638c7c9c_10",\n          "bbox_xyxy": [\n            133.74999999999997,\n            36.49999999999999,\n            189.24999999999997,\n            65.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08f11cb3638c7c9c_11",\n          "bbox_xyxy": [\n            390.74999999999994,\n            115.25,\n            447.25,\n            144.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08f11cb3638c7c9c_12",\n          "bbox_xyxy": [\n            566.5,\n            312.49999999999994,\n            619.5,\n            343.49999999999994\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08f11cb3638c7c9c_13",\n          "bbox_xyxy": [\n            534.25,\n            320.50000000000006,\n            567.75,\n            363.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08f11cb3638c7c9c_14",\n          "bbox_xyxy": [\n            308.0,\n            348.00000000000006,\n            339.0,\n            389.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08f11cb3638c7c9c_15",\n          "bbox_xyxy": [\n            167.00000000000003,\n            323.75,\n            220.0,\n            350.25000000000006\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08f11cb3638c7c9c_16",\n          "bbox_xyxy": [\n            130.0,\n            335.5,\n            162.0,\n            375.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08f11cb3638c7c9c_17",\n          "bbox_xyxy": [\n            146.5,\n            414.49999999999994,\n            192.5,\n            438.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08f11cb3638c7c9c_18",\n          "bbox_xyxy": [\n            105.49999999999999,\n            591.25,\n            143.5,\n            615.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08f11cb3638c7c9c_19",\n          "bbox_xyxy": [\n            318.25,\n            578.5,\n            381.75,\n            605.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08f11cb3638c7c9c_20",\n          "bbox_xyxy": [\n            491.99999999999994,\n            150.0,\n            556.0,\n            171.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08f11cb3638c7c9c_21",\n          "bbox_xyxy": [\n            511.0,\n            227.5,\n            553.0,\n            266.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08f11cb3638c7c9c_22",\n          "bbox_xyxy": [\n            273.25,\n            242.75,\n            311.75,\n            271.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08f11cb3638c7c9c_23",\n          "bbox_xyxy": [\n            214.5,\n            149.25,\n            258.5,\n            170.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_08f11cb3638c7c9c_24",\n          "bbox_xyxy": [\n            299.5,\n            99.24999999999999,\n            357.5,\n            127.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/20251107_163309_jpg.rf.b90a53e13b8d803e41e265a1c4e97005.jpg",\n        "size": 125130,\n        "compressed": 125031,\n        "offset": 344729048,\n        "crc32": "3d805927",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_090652bf6a4f6c70",\n      "relative_path": "images/IMG_1164_JPG.rf.7960d225a5d5d1904e0e75344409a2a8.jpg",\n      "sha256": "9885920860c7e5ff67f587801b2569d6fc445e8dd5cfaf321b911641924a900f",\n      "decoded_sha256": "31c47cf44cef802bd581e3e28f671c7c9bbbe8c736665fce312ce3f91516eb31",\n      "dhash": "4bf3ab6e7555944e",\n      "bytes": 46967,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_1164_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_090652bf6a4f6c70_0",\n          "bbox_xyxy": [\n            261.5,\n            89.75,\n            312.5,\n            127.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_090652bf6a4f6c70_1",\n          "bbox_xyxy": [\n            288.5,\n            320.0,\n            348.5,\n            345.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_090652bf6a4f6c70_2",\n          "bbox_xyxy": [\n            254.50000000000003,\n            274.25,\n            312.5,\n            303.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_090652bf6a4f6c70_3",\n          "bbox_xyxy": [\n            346.25,\n            39.5,\n            401.75,\n            69.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_090652bf6a4f6c70_4",\n          "bbox_xyxy": [\n            274.25,\n            376.25,\n            317.75,\n            412.75000000000006\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_090652bf6a4f6c70_5",\n          "bbox_xyxy": [\n            381.0,\n            370.74999999999994,\n            421.99999999999994,\n            410.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_090652bf6a4f6c70_6",\n          "bbox_xyxy": [\n            529.5,\n            142.25,\n            592.5,\n            168.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_090652bf6a4f6c70_7",\n          "bbox_xyxy": [\n            313.75,\n            486.25,\n            356.25,\n            521.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_090652bf6a4f6c70_8",\n          "bbox_xyxy": [\n            132.50000000000003,\n            257.75,\n            194.50000000000003,\n            291.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_090652bf6a4f6c70_9",\n          "bbox_xyxy": [\n            63.75,\n            469.75,\n            121.25,\n            494.25000000000006\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_090652bf6a4f6c70_10",\n          "bbox_xyxy": [\n            440.5,\n            468.25,\n            482.5,\n            510.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_090652bf6a4f6c70_11",\n          "bbox_xyxy": [\n            239.25,\n            523.5,\n            292.75,\n            547.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_090652bf6a4f6c70_12",\n          "bbox_xyxy": [\n            368.75,\n            424.00000000000006,\n            414.25,\n            455.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_090652bf6a4f6c70_13",\n          "bbox_xyxy": [\n            52.74999999999999,\n            389.25,\n            89.24999999999999,\n            425.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_090652bf6a4f6c70_14",\n          "bbox_xyxy": [\n            132.5,\n            484.74999999999994,\n            186.5,\n            511.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_090652bf6a4f6c70_15",\n          "bbox_xyxy": [\n            273.75,\n            624.0,\n            321.25,\n            640.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_090652bf6a4f6c70_16",\n          "bbox_xyxy": [\n            437.5,\n            566.75,\n            488.49999999999994,\n            593.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_090652bf6a4f6c70_17",\n          "bbox_xyxy": [\n            0.0,\n            436.75,\n            48.0,\n            464.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_090652bf6a4f6c70_18",\n          "bbox_xyxy": [\n            172.0,\n            344.5,\n            225.99999999999997,\n            369.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_090652bf6a4f6c70_19",\n          "bbox_xyxy": [\n            1.4999999999999991,\n            330.50000000000006,\n            41.5,\n            368.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_090652bf6a4f6c70_20",\n          "bbox_xyxy": [\n            5.750000000000002,\n            536.0,\n            53.25,\n            580.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_090652bf6a4f6c70_21",\n          "bbox_xyxy": [\n            0.0,\n            215.75,\n            29.0,\n            253.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_090652bf6a4f6c70_22",\n          "bbox_xyxy": [\n            346.0,\n            523.75,\n            386.99999999999994,\n            566.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_090652bf6a4f6c70_23",\n          "bbox_xyxy": [\n            316.99999999999994,\n            126.5,\n            354.0,\n            164.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_090652bf6a4f6c70_24",\n          "bbox_xyxy": [\n            515.0,\n            614.0,\n            560.0,\n            640.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_090652bf6a4f6c70_25",\n          "bbox_xyxy": [\n            73.0,\n            321.25,\n            107.0,\n            358.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_090652bf6a4f6c70_26",\n          "bbox_xyxy": [\n            175.99999999999997,\n            25.0,\n            222.0,\n            56.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_090652bf6a4f6c70_27",\n          "bbox_xyxy": [\n            228.0,\n            221.0,\n            279.0,\n            250.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_090652bf6a4f6c70_28",\n          "bbox_xyxy": [\n            131.25,\n            418.25,\n            180.75,\n            448.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_090652bf6a4f6c70_29",\n          "bbox_xyxy": [\n            292.25,\n            442.24999999999994,\n            329.75,\n            478.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_090652bf6a4f6c70_30",\n          "bbox_xyxy": [\n            439.25,\n            332.0,\n            490.75,\n            370.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_090652bf6a4f6c70_31",\n          "bbox_xyxy": [\n            563.0,\n            416.25,\n            616.0,\n            442.75000000000006\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_090652bf6a4f6c70_32",\n          "bbox_xyxy": [\n            22.5,\n            283.75,\n            78.5,\n            305.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_090652bf6a4f6c70_33",\n          "bbox_xyxy": [\n            152.5,\n            141.0,\n            197.5,\n            167.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_090652bf6a4f6c70_34",\n          "bbox_xyxy": [\n            166.25,\n            580.5,\n            226.74999999999997,\n            613.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_090652bf6a4f6c70_35",\n          "bbox_xyxy": [\n            71.25,\n            193.75,\n            120.75,\n            222.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_090652bf6a4f6c70_36",\n          "bbox_xyxy": [\n            477.0,\n            245.5,\n            535.0,\n            276.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_090652bf6a4f6c70_37",\n          "bbox_xyxy": [\n            490.50000000000006,\n            476.0,\n            533.5,\n            517.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_090652bf6a4f6c70_38",\n          "bbox_xyxy": [\n            457.25,\n            425.0,\n            509.75,\n            447.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_090652bf6a4f6c70_39",\n          "bbox_xyxy": [\n            381.5,\n            246.50000000000003,\n            425.50000000000006,\n            280.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_090652bf6a4f6c70_40",\n          "bbox_xyxy": [\n            609.0,\n            471.25,\n            640.0,\n            500.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_090652bf6a4f6c70_41",\n          "bbox_xyxy": [\n            593.5,\n            237.00000000000003,\n            637.5,\n            265.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_090652bf6a4f6c70_42",\n          "bbox_xyxy": [\n            89.50000000000001,\n            142.50000000000003,\n            128.5,\n            179.50000000000003\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_090652bf6a4f6c70_43",\n          "bbox_xyxy": [\n            87.0,\n            522.0,\n            123.0,\n            559.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_090652bf6a4f6c70_44",\n          "bbox_xyxy": [\n            557.0,\n            40.50000000000001,\n            605.0,\n            78.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_090652bf6a4f6c70_45",\n          "bbox_xyxy": [\n            407.24999999999994,\n            81.25,\n            448.75,\n            115.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_090652bf6a4f6c70_46",\n          "bbox_xyxy": [\n            288.75,\n            320.25,\n            348.25,\n            344.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_090652bf6a4f6c70_47",\n          "bbox_xyxy": [\n            385.0,\n            186.75,\n            430.0,\n            230.25000000000003\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_090652bf6a4f6c70_48",\n          "bbox_xyxy": [\n            532.25,\n            292.75,\n            579.75,\n            328.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_090652bf6a4f6c70_49",\n          "bbox_xyxy": [\n            538.25,\n            331.75,\n            596.75,\n            361.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_090652bf6a4f6c70_50",\n          "bbox_xyxy": [\n            0.0,\n            596.0,\n            36.0,\n            640.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_1164_JPG.rf.7960d225a5d5d1904e0e75344409a2a8.jpg",\n        "size": 46967,\n        "compressed": 46747,\n        "offset": 486410519,\n        "crc32": "69349955",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_09462249aefcb182",\n      "relative_path": "images/6159142536404995645_jpg.rf.9a08531f4acea242fabd957a662b9fb1.jpg",\n      "sha256": "1f15f66cf654c12962a9cfa58e272544b88adb9f3f162e1d6ec851a1481e15f4",\n      "decoded_sha256": "937237eb0b88c871079e0999c7b86dd0255d49b72b3cd993b57a3234915ada4b",\n      "dhash": "a4aa40d0c4d6766e",\n      "bytes": 30698,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "6159142536404995645_jpg",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_09462249aefcb182_0",\n          "bbox_xyxy": [\n            196.75,\n            63.0,\n            232.24999999999997,\n            117.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09462249aefcb182_1",\n          "bbox_xyxy": [\n            218.5,\n            536.25,\n            272.5,\n            570.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09462249aefcb182_2",\n          "bbox_xyxy": [\n            424.75,\n            241.5,\n            463.24999999999994,\n            288.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09462249aefcb182_3",\n          "bbox_xyxy": [\n            342.5,\n            336.25,\n            396.50000000000006,\n            371.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09462249aefcb182_4",\n          "bbox_xyxy": [\n            215.5,\n            312.0,\n            243.49999999999997,\n            357.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09462249aefcb182_5",\n          "bbox_xyxy": [\n            481.99999999999994,\n            486.50000000000006,\n            521.0,\n            517.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09462249aefcb182_6",\n          "bbox_xyxy": [\n            257.5,\n            53.50000000000001,\n            309.5,\n            84.50000000000001\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09462249aefcb182_7",\n          "bbox_xyxy": [\n            68.25,\n            35.75,\n            135.75,\n            79.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09462249aefcb182_8",\n          "bbox_xyxy": [\n            126.75,\n            273.25,\n            164.25,\n            315.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09462249aefcb182_9",\n          "bbox_xyxy": [\n            171.5,\n            347.25,\n            218.5,\n            381.75000000000006\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09462249aefcb182_10",\n          "bbox_xyxy": [\n            450.25000000000006,\n            334.25,\n            488.75,\n            371.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09462249aefcb182_11",\n          "bbox_xyxy": [\n            264.0,\n            372.25,\n            318.0,\n            406.75000000000006\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09462249aefcb182_12",\n          "bbox_xyxy": [\n            336.75000000000006,\n            99.5,\n            405.25,\n            143.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09462249aefcb182_13",\n          "bbox_xyxy": [\n            169.25,\n            176.25,\n            205.75,\n            243.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09462249aefcb182_14",\n          "bbox_xyxy": [\n            312.0,\n            465.5,\n            355.0,\n            519.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09462249aefcb182_15",\n          "bbox_xyxy": [\n            272.0,\n            229.75,\n            323.0,\n            280.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09462249aefcb182_16",\n          "bbox_xyxy": [\n            140.5,\n            389.49999999999994,\n            176.5,\n            426.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09462249aefcb182_17",\n          "bbox_xyxy": [\n            254.25,\n            113.25,\n            285.75,\n            161.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09462249aefcb182_18",\n          "bbox_xyxy": [\n            235.25,\n            219.50000000000003,\n            273.75,\n            252.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09462249aefcb182_19",\n          "bbox_xyxy": [\n            337.25,\n            233.0,\n            380.74999999999994,\n            261.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/6159142536404995645_jpg.rf.9a08531f4acea242fabd957a662b9fb1.jpg",\n        "size": 30698,\n        "compressed": 30412,\n        "offset": 411732926,\n        "crc32": "982c9cee",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_09540b5531ad5450",\n      "relative_path": "images/20251107_150922_jpg.rf.5485a98c783630db84ec79e7e432f087.jpg",\n      "sha256": "dfc62ca0304df8fff026167db66f045afb152b67f07a59e4a5b8a48b64db5663",\n      "decoded_sha256": "8aa34e96e44e79bce1e6a1f1c85ede3c8345fc1e68e69e7198284396b5f2feb1",\n      "dhash": "ccddcfe7b787cd3f",\n      "bytes": 93128,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "20251107_150922_jpg",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_09540b5531ad5450_0",\n          "bbox_xyxy": [\n            429.5,\n            501.99999999999994,\n            469.5,\n            536.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09540b5531ad5450_1",\n          "bbox_xyxy": [\n            449.75,\n            6.000000000000001,\n            494.25000000000006,\n            29.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09540b5531ad5450_2",\n          "bbox_xyxy": [\n            556.5,\n            288.0,\n            572.5,\n            323.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09540b5531ad5450_3",\n          "bbox_xyxy": [\n            229.25,\n            322.5,\n            257.75,\n            357.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09540b5531ad5450_4",\n          "bbox_xyxy": [\n            490.74999999999994,\n            83.0,\n            542.25,\n            98.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09540b5531ad5450_5",\n          "bbox_xyxy": [\n            493.25,\n            278.0,\n            525.75,\n            307.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09540b5531ad5450_6",\n          "bbox_xyxy": [\n            203.75,\n            206.25,\n            228.25000000000003,\n            245.75000000000003\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09540b5531ad5450_7",\n          "bbox_xyxy": [\n            503.5,\n            41.25,\n            531.5,\n            62.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09540b5531ad5450_8",\n          "bbox_xyxy": [\n            586.5,\n            119.75,\n            618.5,\n            128.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/20251107_150922_jpg.rf.5485a98c783630db84ec79e7e432f087.jpg",\n        "size": 93128,\n        "compressed": 93015,\n        "offset": 322176629,\n        "crc32": "6ab7d7e4",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_097e59c6453c2d9f",\n      "relative_path": "images/IMG_2149_JPG.rf.95cdcee1a02b6b5bbde32b33bad575de.jpg",\n      "sha256": "fcc2198674e920e30079018cc1742eb77bf74b7b1832374146147552a45dc626",\n      "decoded_sha256": "35a7f5539d4e0558fb461faff38eef7988c9d02a89f5f0a5350dd1c5c886b295",\n      "dhash": "0812327171797130",\n      "bytes": 23746,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2149_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_097e59c6453c2d9f_0",\n          "bbox_xyxy": [\n            172.99999999999997,\n            125.24999999999999,\n            466.0,\n            595.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_2149_JPG.rf.95cdcee1a02b6b5bbde32b33bad575de.jpg",\n        "size": 23746,\n        "compressed": 22109,\n        "offset": 530644864,\n        "crc32": "04c8d89e",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_09bc1de2806d5c39",\n      "relative_path": "images/IMG_2420_JPG.rf.87bab17ea6143dec575c30ce41d401f1.jpg",\n      "sha256": "22b60c4831b2a25bf757a441d4917ba9daada26b2d72062ea4140d5e2e4c04af",\n      "decoded_sha256": "3f1e0771e95816aa0d7c21320479f33e37464d684174d2d86d8893791f9a2c48",\n      "dhash": "6c31716969713014",\n      "bytes": 29693,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2420_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_09bc1de2806d5c39_0",\n          "bbox_xyxy": [\n            142.5,\n            41.25,\n            525.5,\n            563.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_2420_JPG.rf.87bab17ea6143dec575c30ce41d401f1.jpg",\n        "size": 29693,\n        "compressed": 28834,\n        "offset": 554172992,\n        "crc32": "affd7503",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_09ee36b272bd2213",\n      "relative_path": "images/IMG_0743_JPG.rf.7372d149da3dc724d6991739677aee8b.jpg",\n      "sha256": "c7eac13f5c6532ce8003496247be6b2965eda444df04e707484053a4282ef5f4",\n      "decoded_sha256": "ec504247299e8c0c97881af1f544ca819d8309bda83b6ce15713cf51d3897668",\n      "dhash": "d785d2520e55698b",\n      "bytes": 67525,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_0743_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_09ee36b272bd2213_0",\n          "bbox_xyxy": [\n            76.0,\n            576.75,\n            154.0,\n            630.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_1",\n          "bbox_xyxy": [\n            149.5,\n            618.25,\n            221.50000000000003,\n            639.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_2",\n          "bbox_xyxy": [\n            161.74999999999997,\n            510.74999999999994,\n            241.25,\n            550.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_3",\n          "bbox_xyxy": [\n            410.25000000000006,\n            172.99999999999997,\n            453.75,\n            235.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_4",\n          "bbox_xyxy": [\n            499.00000000000006,\n            0.0,\n            575.0,\n            42.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_5",\n          "bbox_xyxy": [\n            381.25,\n            348.5,\n            456.75,\n            391.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_6",\n          "bbox_xyxy": [\n            435.75,\n            534.5,\n            503.25,\n            552.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_7",\n          "bbox_xyxy": [\n            305.50000000000006,\n            513.5,\n            373.5,\n            548.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_8",\n          "bbox_xyxy": [\n            282.25,\n            267.0,\n            363.75,\n            291.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_9",\n          "bbox_xyxy": [\n            381.25,\n            67.75,\n            456.75,\n            111.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_10",\n          "bbox_xyxy": [\n            462.25,\n            135.75,\n            517.75,\n            174.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_11",\n          "bbox_xyxy": [\n            461.50000000000006,\n            597.0,\n            542.5,\n            634.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_12",\n          "bbox_xyxy": [\n            459.0,\n            626.0,\n            536.0,\n            640.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_13",\n          "bbox_xyxy": [\n            295.75,\n            592.25,\n            377.25,\n            638.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_14",\n          "bbox_xyxy": [\n            280.25,\n            9.749999999999996,\n            350.75,\n            51.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_15",\n          "bbox_xyxy": [\n            534.5,\n            29.000000000000004,\n            581.5,\n            79.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_16",\n          "bbox_xyxy": [\n            548.0,\n            1.9999999999999996,\n            627.0,\n            37.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_17",\n          "bbox_xyxy": [\n            589.0,\n            38.25,\n            640.0,\n            68.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_18",\n          "bbox_xyxy": [\n            590.0,\n            175.99999999999997,\n            640.0,\n            228.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_19",\n          "bbox_xyxy": [\n            503.25,\n            347.0,\n            581.75,\n            388.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_20",\n          "bbox_xyxy": [\n            603.0,\n            346.75000000000006,\n            640.0,\n            392.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_21",\n          "bbox_xyxy": [\n            616.0,\n            392.24999999999994,\n            640.0,\n            428.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_22",\n          "bbox_xyxy": [\n            592.0,\n            430.74999999999994,\n            640.0,\n            480.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_23",\n          "bbox_xyxy": [\n            477.25,\n            420.50000000000006,\n            537.75,\n            471.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_24",\n          "bbox_xyxy": [\n            379.0,\n            556.75,\n            446.0,\n            620.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_25",\n          "bbox_xyxy": [\n            392.24999999999994,\n            623.0,\n            463.75,\n            640.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_26",\n          "bbox_xyxy": [\n            212.0,\n            586.0,\n            279.0,\n            640.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_27",\n          "bbox_xyxy": [\n            96.75,\n            415.25,\n            178.25,\n            459.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_28",\n          "bbox_xyxy": [\n            201.25,\n            460.74999999999994,\n            280.75,\n            510.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_29",\n          "bbox_xyxy": [\n            255.75,\n            477.5,\n            315.25,\n            532.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_30",\n          "bbox_xyxy": [\n            173.0,\n            334.75,\n            230.99999999999997,\n            384.25000000000006\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_31",\n          "bbox_xyxy": [\n            227.25,\n            332.25,\n            302.75,\n            375.74999999999994\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_32",\n          "bbox_xyxy": [\n            118.5,\n            246.50000000000003,\n            171.5,\n            294.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_33",\n          "bbox_xyxy": [\n            112.75,\n            282.25,\n            182.25,\n            321.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_34",\n          "bbox_xyxy": [\n            181.0,\n            227.00000000000003,\n            235.0,\n            280.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_35",\n          "bbox_xyxy": [\n            215.0,\n            258.25,\n            288.0,\n            314.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_36",\n          "bbox_xyxy": [\n            20.5,\n            153.25,\n            76.5,\n            227.75000000000003\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_37",\n          "bbox_xyxy": [\n            18.499999999999996,\n            196.75,\n            111.5,\n            234.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_38",\n          "bbox_xyxy": [\n            132.25,\n            179.25,\n            227.75,\n            236.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_39",\n          "bbox_xyxy": [\n            118.75,\n            158.74999999999997,\n            184.25,\n            194.24999999999997\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_40",\n          "bbox_xyxy": [\n            125.0,\n            55.75,\n            171.0,\n            117.25000000000001\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_41",\n          "bbox_xyxy": [\n            18.25,\n            73.0,\n            70.75,\n            129.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_42",\n          "bbox_xyxy": [\n            14.000000000000004,\n            0.0,\n            85.0,\n            56.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_43",\n          "bbox_xyxy": [\n            124.0,\n            10.999999999999998,\n            196.0,\n            37.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_44",\n          "bbox_xyxy": [\n            12.750000000000004,\n            400.25,\n            85.25,\n            450.74999999999994\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_45",\n          "bbox_xyxy": [\n            420.74999999999994,\n            39.75,\n            512.25,\n            74.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_46",\n          "bbox_xyxy": [\n            585.25,\n            71.0,\n            639.75,\n            110.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_47",\n          "bbox_xyxy": [\n            514.25,\n            72.75,\n            575.75,\n            118.25000000000001\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_48",\n          "bbox_xyxy": [\n            442.5,\n            85.25,\n            505.5,\n            135.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_49",\n          "bbox_xyxy": [\n            300.74999999999994,\n            60.25,\n            355.25,\n            124.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_50",\n          "bbox_xyxy": [\n            518.0,\n            132.5,\n            584.0,\n            187.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_51",\n          "bbox_xyxy": [\n            579.0,\n            119.5,\n            640.0,\n            157.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_52",\n          "bbox_xyxy": [\n            453.25,\n            200.75,\n            531.75,\n            244.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_53",\n          "bbox_xyxy": [\n            496.25,\n            218.25000000000003,\n            571.75,\n            273.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_54",\n          "bbox_xyxy": [\n            597.0,\n            286.25,\n            640.0,\n            346.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_55",\n          "bbox_xyxy": [\n            593.0,\n            248.0,\n            640.0,\n            289.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_56",\n          "bbox_xyxy": [\n            539.75,\n            270.0,\n            600.25,\n            331.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_57",\n          "bbox_xyxy": [\n            474.0,\n            292.25,\n            547.0,\n            332.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_58",\n          "bbox_xyxy": [\n            480.25,\n            304.0,\n            529.75,\n            356.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_59",\n          "bbox_xyxy": [\n            382.5,\n            271.0,\n            457.5,\n            329.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_60",\n          "bbox_xyxy": [\n            311.75,\n            307.0,\n            388.25,\n            341.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_61",\n          "bbox_xyxy": [\n            322.0,\n            216.25,\n            400.0,\n            274.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_62",\n          "bbox_xyxy": [\n            416.99999999999994,\n            388.0,\n            489.0,\n            440.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_63",\n          "bbox_xyxy": [\n            255.0,\n            365.25000000000006,\n            341.0,\n            418.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_64",\n          "bbox_xyxy": [\n            303.0,\n            400.0,\n            392.0,\n            435.99999999999994\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_65",\n          "bbox_xyxy": [\n            260.5,\n            410.5,\n            323.5,\n            464.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_66",\n          "bbox_xyxy": [\n            279.25,\n            448.00000000000006,\n            359.75,\n            476.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_67",\n          "bbox_xyxy": [\n            434.0,\n            441.75,\n            499.0,\n            498.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_68",\n          "bbox_xyxy": [\n            356.99999999999994,\n            472.25,\n            451.0,\n            511.75000000000006\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_69",\n          "bbox_xyxy": [\n            376.75,\n            498.5,\n            434.25,\n            558.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_70",\n          "bbox_xyxy": [\n            546.0,\n            451.25,\n            624.0,\n            507.75000000000006\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_71",\n          "bbox_xyxy": [\n            580.0,\n            511.0,\n            633.0,\n            562.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_72",\n          "bbox_xyxy": [\n            493.00000000000006,\n            511.0,\n            559.0,\n            566.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_73",\n          "bbox_xyxy": [\n            565.5,\n            580.25,\n            631.5,\n            639.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_74",\n          "bbox_xyxy": [\n            35.5,\n            479.75,\n            110.5,\n            537.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_75",\n          "bbox_xyxy": [\n            106.49999999999999,\n            352.0,\n            172.5,\n            412.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_76",\n          "bbox_xyxy": [\n            59.25,\n            311.75,\n            117.74999999999999,\n            365.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_77",\n          "bbox_xyxy": [\n            18.249999999999996,\n            344.00000000000006,\n            83.75,\n            390.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_78",\n          "bbox_xyxy": [\n            1.7499999999999982,\n            268.25,\n            73.25,\n            320.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_79",\n          "bbox_xyxy": [\n            212.99999999999997,\n            85.5,\n            285.0,\n            141.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_80",\n          "bbox_xyxy": [\n            158.0,\n            123.5,\n            233.0,\n            168.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_81",\n          "bbox_xyxy": [\n            81.5,\n            97.25,\n            133.5,\n            151.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_82",\n          "bbox_xyxy": [\n            74.5,\n            39.0,\n            117.5,\n            100.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_83",\n          "bbox_xyxy": [\n            164.25,\n            49.25,\n            247.75000000000003,\n            86.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_84",\n          "bbox_xyxy": [\n            143.25,\n            18.749999999999996,\n            217.75000000000003,\n            52.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_85",\n          "bbox_xyxy": [\n            169.25,\n            0.2500000000000002,\n            265.75,\n            12.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_86",\n          "bbox_xyxy": [\n            310.25,\n            121.0,\n            396.75000000000006,\n            168.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_87",\n          "bbox_xyxy": [\n            299.0,\n            163.5,\n            396.99999999999994,\n            206.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_88",\n          "bbox_xyxy": [\n            220.75000000000003,\n            176.25,\n            310.25,\n            217.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_09ee36b272bd2213_89",\n          "bbox_xyxy": [\n            161.0,\n            422.75,\n            245.0,\n            460.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_0743_JPG.rf.7372d149da3dc724d6991739677aee8b.jpg",\n        "size": 67525,\n        "compressed": 67399,\n        "offset": 434652201,\n        "crc32": "44c63c6d",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_0a2afa3797982379",\n      "relative_path": "images/IMG_1049_JPG.rf.bf28d022e4a1c210db81ad45583b52ad.jpg",\n      "sha256": "5aab14e4de149a625283afee85d5a6d73490fa830db4bccc3efaf4e406fe754b",\n      "decoded_sha256": "6a62fd7d0edc9466172a4c7b6b2727382e108ccf64bf4dac039817edef44fa21",\n      "dhash": "06728707291d9b97",\n      "bytes": 39989,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_1049_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_0a2afa3797982379_0",\n          "bbox_xyxy": [\n            455.75,\n            243.5,\n            511.25,\n            302.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0a2afa3797982379_1",\n          "bbox_xyxy": [\n            271.25,\n            120.0,\n            321.75,\n            168.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0a2afa3797982379_2",\n          "bbox_xyxy": [\n            479.25,\n            517.0,\n            533.75,\n            559.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0a2afa3797982379_3",\n          "bbox_xyxy": [\n            358.5,\n            88.25,\n            396.5,\n            158.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0a2afa3797982379_4",\n          "bbox_xyxy": [\n            565.0,\n            310.25,\n            613.0,\n            349.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0a2afa3797982379_5",\n          "bbox_xyxy": [\n            474.0,\n            131.0,\n            524.0,\n            174.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0a2afa3797982379_6",\n          "bbox_xyxy": [\n            591.75,\n            459.0,\n            631.25,\n            522.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0a2afa3797982379_7",\n          "bbox_xyxy": [\n            105.0,\n            547.0,\n            142.0,\n            621.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0a2afa3797982379_8",\n          "bbox_xyxy": [\n            232.75,\n            222.5,\n            286.25,\n            262.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0a2afa3797982379_9",\n          "bbox_xyxy": [\n            79.5,\n            179.75,\n            130.5,\n            220.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0a2afa3797982379_10",\n          "bbox_xyxy": [\n            183.75,\n            403.75,\n            225.25,\n            462.24999999999994\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0a2afa3797982379_11",\n          "bbox_xyxy": [\n            427.25,\n            384.25,\n            490.74999999999994,\n            415.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0a2afa3797982379_12",\n          "bbox_xyxy": [\n            138.75,\n            220.5,\n            195.25,\n            261.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0a2afa3797982379_13",\n          "bbox_xyxy": [\n            376.75,\n            484.25000000000006,\n            413.25,\n            537.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0a2afa3797982379_14",\n          "bbox_xyxy": [\n            540.75,\n            458.24999999999994,\n            588.25,\n            502.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0a2afa3797982379_15",\n          "bbox_xyxy": [\n            596.0,\n            369.75,\n            640.0,\n            404.25000000000006\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0a2afa3797982379_16",\n          "bbox_xyxy": [\n            226.25,\n            320.75,\n            287.75,\n            381.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0a2afa3797982379_17",\n          "bbox_xyxy": [\n            532.0,\n            74.5,\n            567.0,\n            152.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0a2afa3797982379_18",\n          "bbox_xyxy": [\n            258.5,\n            564.0,\n            311.5,\n            595.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0a2afa3797982379_19",\n          "bbox_xyxy": [\n            348.75,\n            293.00000000000006,\n            390.25000000000006,\n            361.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0a2afa3797982379_20",\n          "bbox_xyxy": [\n            38.75,\n            0.2499999999999991,\n            60.25,\n            71.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0a2afa3797982379_21",\n          "bbox_xyxy": [\n            194.0,\n            78.25,\n            230.0,\n            138.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0a2afa3797982379_22",\n          "bbox_xyxy": [\n            333.75,\n            498.75,\n            375.25000000000006,\n            560.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0a2afa3797982379_23",\n          "bbox_xyxy": [\n            7.000000000000002,\n            85.75,\n            50.0,\n            149.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0a2afa3797982379_24",\n          "bbox_xyxy": [\n            314.5,\n            335.0,\n            360.5,\n            365.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0a2afa3797982379_25",\n          "bbox_xyxy": [\n            407.5,\n            404.5,\n            466.50000000000006,\n            450.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0a2afa3797982379_26",\n          "bbox_xyxy": [\n            110.0,\n            16.250000000000004,\n            158.0,\n            81.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0a2afa3797982379_27",\n          "bbox_xyxy": [\n            327.25,\n            432.24999999999994,\n            372.75,\n            498.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0a2afa3797982379_28",\n          "bbox_xyxy": [\n            464.0,\n            0.0,\n            517.0,\n            42.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0a2afa3797982379_29",\n          "bbox_xyxy": [\n            5.249999999999999,\n            330.75,\n            34.75,\n            418.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0a2afa3797982379_30",\n          "bbox_xyxy": [\n            70.75,\n            453.00000000000006,\n            117.25000000000001,\n            509.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0a2afa3797982379_31",\n          "bbox_xyxy": [\n            64.75,\n            337.25,\n            135.25,\n            367.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0a2afa3797982379_32",\n          "bbox_xyxy": [\n            529.5,\n            536.5,\n            576.5,\n            569.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_1049_JPG.rf.bf28d022e4a1c210db81ad45583b52ad.jpg",\n        "size": 39989,\n        "compressed": 39497,\n        "offset": 466026552,\n        "crc32": "8be1b885",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_0a74ae2476c1994b",\n      "relative_path": "images/IMG_1769_JPG.rf.eb99c2ec71bd1644fd4659d145b26034.jpg",\n      "sha256": "a8bf289b89a9bb71e1ba6aa2f9bc7c6c2e1a028c688d7efae4a8d694fbefffb5",\n      "decoded_sha256": "10d7b0116d410f6e08c2caabb9da66c585e85508141bcdb31cdc865b254637d4",\n      "dhash": "1430706060703010",\n      "bytes": 22882,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_1769_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_0a74ae2476c1994b_0",\n          "bbox_xyxy": [\n            176.25000000000003,\n            17.5,\n            517.75,\n            593.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_1769_JPG.rf.eb99c2ec71bd1644fd4659d145b26034.jpg",\n        "size": 22882,\n        "compressed": 18587,\n        "offset": 499060394,\n        "crc32": "f57ccc9b",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_0ab1c332485b75a9",\n      "relative_path": "images/IMG_2147_JPG.rf.d20723597d569c9cf7c3a0180c39abd7.jpg",\n      "sha256": "65e19e6a357314d348a7da540519c6bc49483a3bf419d3f1e00e4287a4569430",\n      "decoded_sha256": "a15ff7e45e986ddc53b4819fc2234d02a8e1f28123e76e63085b7a4afb804381",\n      "dhash": "b672716961e1f130",\n      "bytes": 27342,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2147_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_0ab1c332485b75a9_0",\n          "bbox_xyxy": [\n            72.0,\n            12.000000000000028,\n            510.99999999999994,\n            640.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_2147_JPG.rf.d20723597d569c9cf7c3a0180c39abd7.jpg",\n        "size": 27342,\n        "compressed": 26536,\n        "offset": 530421967,\n        "crc32": "187fb6c8",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_0acb2fa262203da2",\n      "relative_path": "images/IMG_4003_JPG.rf.950ddd366a8884acebc6d63d46ed09d0.jpg",\n      "sha256": "de4f52b68b305da5d5ec8a999a62c24d87825a65c901059900d0e892d9b6e9b1",\n      "decoded_sha256": "d6a28ff29845fb7f4697e45a76be77894037f2f453311b534d94ce4e6000e432",\n      "dhash": "10d484a4ea723a68",\n      "bytes": 22172,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_4003_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_0acb2fa262203da2_0",\n          "bbox_xyxy": [\n            222.99999999999997,\n            553.0,\n            285.0,\n            610.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0acb2fa262203da2_1",\n          "bbox_xyxy": [\n            343.0,\n            550.5,\n            398.0,\n            586.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0acb2fa262203da2_2",\n          "bbox_xyxy": [\n            271.25,\n            421.75000000000006,\n            326.75,\n            455.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0acb2fa262203da2_3",\n          "bbox_xyxy": [\n            358.49999999999994,\n            380.74999999999994,\n            417.5,\n            417.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0acb2fa262203da2_4",\n          "bbox_xyxy": [\n            447.5,\n            434.00000000000006,\n            513.5,\n            465.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0acb2fa262203da2_5",\n          "bbox_xyxy": [\n            509.5,\n            436.5,\n            567.5,\n            470.50000000000006\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0acb2fa262203da2_6",\n          "bbox_xyxy": [\n            481.75,\n            317.25,\n            569.25,\n            352.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0acb2fa262203da2_7",\n          "bbox_xyxy": [\n            381.5,\n            239.5,\n            436.5,\n            299.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0acb2fa262203da2_8",\n          "bbox_xyxy": [\n            239.5,\n            304.75,\n            278.5,\n            354.25000000000006\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0acb2fa262203da2_9",\n          "bbox_xyxy": [\n            187.25,\n            182.0,\n            222.75,\n            234.00000000000003\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0acb2fa262203da2_10",\n          "bbox_xyxy": [\n            134.25,\n            343.5,\n            203.75,\n            378.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0acb2fa262203da2_11",\n          "bbox_xyxy": [\n            463.0,\n            127.74999999999999,\n            488.0,\n            189.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0acb2fa262203da2_12",\n          "bbox_xyxy": [\n            434.75,\n            215.0,\n            497.25,\n            252.00000000000003\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0acb2fa262203da2_13",\n          "bbox_xyxy": [\n            302.75,\n            204.25,\n            334.25000000000006,\n            267.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0acb2fa262203da2_14",\n          "bbox_xyxy": [\n            258.25,\n            245.0,\n            293.75,\n            297.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0acb2fa262203da2_15",\n          "bbox_xyxy": [\n            284.5,\n            293.24999999999994,\n            329.5,\n            352.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0acb2fa262203da2_16",\n          "bbox_xyxy": [\n            296.75,\n            85.49999999999999,\n            339.25,\n            138.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0acb2fa262203da2_17",\n          "bbox_xyxy": [\n            166.25,\n            97.0,\n            208.75,\n            175.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0acb2fa262203da2_18",\n          "bbox_xyxy": [\n            38.25,\n            223.5,\n            117.75,\n            247.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0acb2fa262203da2_19",\n          "bbox_xyxy": [\n            85.25,\n            169.25,\n            139.75,\n            204.74999999999997\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_4003_JPG.rf.950ddd366a8884acebc6d63d46ed09d0.jpg",\n        "size": 22172,\n        "compressed": 17921,\n        "offset": 587960360,\n        "crc32": "1ed77b7e",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_0ae9e4ae734a10e7",\n      "relative_path": "images/IMG_4108_JPG.rf.833cbe1c2f445662714c02ac8128c4f6.jpg",\n      "sha256": "8b5a886640292b34a3db0aea0d8be2cae6ffede592d9de20d27910aafc49fd29",\n      "decoded_sha256": "8e49d760d8f0953c48dd1cfcff4c15e3e4f9d0a8759a2a3b16bc70851b7d1e79",\n      "dhash": "5d5b6a787dda5b6b",\n      "bytes": 43606,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_4108_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_0ae9e4ae734a10e7_0",\n          "bbox_xyxy": [\n            216.75,\n            339.75,\n            255.25000000000003,\n            382.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0ae9e4ae734a10e7_1",\n          "bbox_xyxy": [\n            332.75,\n            430.25,\n            378.24999999999994,\n            470.74999999999994\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0ae9e4ae734a10e7_2",\n          "bbox_xyxy": [\n            452.75,\n            401.25,\n            474.25000000000006,\n            456.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0ae9e4ae734a10e7_3",\n          "bbox_xyxy": [\n            501.25,\n            274.5,\n            529.75,\n            329.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0ae9e4ae734a10e7_4",\n          "bbox_xyxy": [\n            451.25,\n            334.25,\n            482.75000000000006,\n            376.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0ae9e4ae734a10e7_5",\n          "bbox_xyxy": [\n            387.25,\n            296.5,\n            432.75,\n            359.49999999999994\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0ae9e4ae734a10e7_6",\n          "bbox_xyxy": [\n            434.75,\n            278.5,\n            465.25,\n            313.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0ae9e4ae734a10e7_7",\n          "bbox_xyxy": [\n            406.25,\n            243.25,\n            447.75000000000006,\n            267.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0ae9e4ae734a10e7_8",\n          "bbox_xyxy": [\n            277.75,\n            269.5,\n            314.25,\n            290.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0ae9e4ae734a10e7_9",\n          "bbox_xyxy": [\n            300.0,\n            323.25,\n            331.0,\n            365.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0ae9e4ae734a10e7_10",\n          "bbox_xyxy": [\n            346.0,\n            380.75,\n            391.99999999999994,\n            408.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0ae9e4ae734a10e7_11",\n          "bbox_xyxy": [\n            390.5,\n            212.5,\n            410.5,\n            264.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_4108_JPG.rf.833cbe1c2f445662714c02ac8128c4f6.jpg",\n        "size": 43606,\n        "compressed": 43386,\n        "offset": 599607038,\n        "crc32": "46ca7859",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_0af75f59238a0795",\n      "relative_path": "images/IMG_2535_JPG.rf.3071ee6fb9c9bf0cf9d200dcc6f9dc6a.jpg",\n      "sha256": "1df0bb71b787278195caae63eb195934f519810aec1c64b87f29438331271d34",\n      "decoded_sha256": "03d6e4080ddc58d25ee389f5ddb50a09e09bfc600a48c19b5e44480723b93eae",\n      "dhash": "487070707070b0b0",\n      "bytes": 23208,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2535_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_0af75f59238a0795_0",\n          "bbox_xyxy": [\n            142.75,\n            26.25,\n            460.25,\n            591.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_2535_JPG.rf.3071ee6fb9c9bf0cf9d200dcc6f9dc6a.jpg",\n        "size": 23208,\n        "compressed": 20237,\n        "offset": 563193404,\n        "crc32": "e03787a8",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_0b1246ba0ceb9787",\n      "relative_path": "images/20251107_163251_jpg.rf.2ad766bb8966d7826201662e94c82529.jpg",\n      "sha256": "71fd0c34e830a3fdd7fc85d482efa2690fb8143686ec49057fbb5f0c4e64be6f",\n      "decoded_sha256": "a26b702d145fe9bb4528ac66961a8afb15da17a43fdbc461824b132d83ce68d5",\n      "dhash": "3821a22606066232",\n      "bytes": 126519,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "20251107_163251_jpg",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_0b1246ba0ceb9787_0",\n          "bbox_xyxy": [\n            443.75,\n            310.25,\n            464.25,\n            346.75000000000006\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0b1246ba0ceb9787_1",\n          "bbox_xyxy": [\n            8.250000000000002,\n            386.5,\n            60.75,\n            409.49999999999994\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0b1246ba0ceb9787_2",\n          "bbox_xyxy": [\n            58.75,\n            461.75000000000006,\n            88.25,\n            495.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0b1246ba0ceb9787_3",\n          "bbox_xyxy": [\n            78.74999999999999,\n            143.5,\n            135.25,\n            165.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0b1246ba0ceb9787_4",\n          "bbox_xyxy": [\n            213.0,\n            10.0,\n            263.0,\n            50.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0b1246ba0ceb9787_5",\n          "bbox_xyxy": [\n            314.0,\n            24.250000000000004,\n            368.00000000000006,\n            69.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0b1246ba0ceb9787_6",\n          "bbox_xyxy": [\n            96.00000000000001,\n            399.25,\n            142.0,\n            430.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0b1246ba0ceb9787_7",\n          "bbox_xyxy": [\n            244.25,\n            535.0,\n            295.75,\n            560.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0b1246ba0ceb9787_8",\n          "bbox_xyxy": [\n            159.0,\n            395.0,\n            196.0,\n            427.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0b1246ba0ceb9787_9",\n          "bbox_xyxy": [\n            28.25,\n            292.75,\n            71.75,\n            318.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0b1246ba0ceb9787_10",\n          "bbox_xyxy": [\n            71.5,\n            286.5,\n            91.5,\n            328.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0b1246ba0ceb9787_11",\n          "bbox_xyxy": [\n            92.25,\n            269.5,\n            142.75,\n            292.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0b1246ba0ceb9787_12",\n          "bbox_xyxy": [\n            153.5,\n            239.50000000000003,\n            196.5,\n            277.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0b1246ba0ceb9787_13",\n          "bbox_xyxy": [\n            84.25,\n            208.0,\n            115.75,\n            243.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0b1246ba0ceb9787_14",\n          "bbox_xyxy": [\n            409.25,\n            31.0,\n            456.75,\n            54.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0b1246ba0ceb9787_15",\n          "bbox_xyxy": [\n            345.0,\n            139.0,\n            385.99999999999994,\n            162.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0b1246ba0ceb9787_16",\n          "bbox_xyxy": [\n            255.75,\n            89.25,\n            308.25,\n            120.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0b1246ba0ceb9787_17",\n          "bbox_xyxy": [\n            177.0,\n            108.75,\n            219.00000000000003,\n            134.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0b1246ba0ceb9787_18",\n          "bbox_xyxy": [\n            470.0,\n            73.74999999999999,\n            498.0,\n            100.24999999999999\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0b1246ba0ceb9787_19",\n          "bbox_xyxy": [\n            525.5,\n            156.50000000000003,\n            575.5,\n            180.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0b1246ba0ceb9787_20",\n          "bbox_xyxy": [\n            298.25,\n            222.0,\n            333.75,\n            253.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0b1246ba0ceb9787_21",\n          "bbox_xyxy": [\n            332.5,\n            374.75,\n            357.5,\n            405.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0b1246ba0ceb9787_22",\n          "bbox_xyxy": [\n            411.75,\n            383.00000000000006,\n            456.25,\n            406.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0b1246ba0ceb9787_23",\n          "bbox_xyxy": [\n            275.0,\n            321.75,\n            304.0,\n            359.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0b1246ba0ceb9787_24",\n          "bbox_xyxy": [\n            221.75,\n            326.5,\n            258.25,\n            360.50000000000006\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0b1246ba0ceb9787_25",\n          "bbox_xyxy": [\n            386.0,\n            302.25,\n            431.0,\n            316.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0b1246ba0ceb9787_26",\n          "bbox_xyxy": [\n            456.25,\n            548.5,\n            494.74999999999994,\n            572.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0b1246ba0ceb9787_27",\n          "bbox_xyxy": [\n            365.50000000000006,\n            587.0,\n            411.5,\n            609.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0b1246ba0ceb9787_28",\n          "bbox_xyxy": [\n            474.49999999999994,\n            619.25,\n            523.5,\n            639.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/20251107_163251_jpg.rf.2ad766bb8966d7826201662e94c82529.jpg",\n        "size": 126519,\n        "compressed": 126416,\n        "offset": 342065765,\n        "crc32": "36e368f0",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_0b391a4936d60796",\n      "relative_path": "images/IMG_2318_JPG.rf.a2a19d819e2dccf4c8002f6cfc26956c.jpg",\n      "sha256": "6d36c59c7ddabe30519c9cfd2acb3afe741857449f4cce8cba41a13303b134e9",\n      "decoded_sha256": "8aeee7722c5c48d1d03681cd0be1a0e5f95b08101521555b9e5f337daf7b662e",\n      "dhash": "323070707971319a",\n      "bytes": 26305,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2318_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_0b391a4936d60796_0",\n          "bbox_xyxy": [\n            161.75,\n            25.0,\n            535.25,\n            610.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_2318_JPG.rf.a2a19d819e2dccf4c8002f6cfc26956c.jpg",\n        "size": 26305,\n        "compressed": 25505,\n        "offset": 544673050,\n        "crc32": "0d172f54",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_0b7d4ac1e484d546",\n      "relative_path": "images/6159142536404995619_jpg.rf.df4598db7988891f4291e82ff7565e47.jpg",\n      "sha256": "c441dbde61b040ab4b26d29c1266a2ae42c8ef2e8be9ad57470bbdc433b2a7e3",\n      "decoded_sha256": "099aad55c3b960a2f324e15d8b31574339a215099a876b138418a8f72887802c",\n      "dhash": "81a9a56599a5899b",\n      "bytes": 62505,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "6159142536404995619_jpg",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_0b7d4ac1e484d546_0",\n          "bbox_xyxy": [\n            452.0,\n            455.74999999999994,\n            503.0,\n            497.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0b7d4ac1e484d546_1",\n          "bbox_xyxy": [\n            159.0,\n            270.25,\n            207.0,\n            325.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0b7d4ac1e484d546_2",\n          "bbox_xyxy": [\n            245.75000000000003,\n            249.25,\n            306.25,\n            272.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0b7d4ac1e484d546_3",\n          "bbox_xyxy": [\n            432.75,\n            257.75,\n            490.25,\n            301.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0b7d4ac1e484d546_4",\n          "bbox_xyxy": [\n            97.0,\n            523.0,\n            156.0,\n            555.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0b7d4ac1e484d546_5",\n          "bbox_xyxy": [\n            38.5,\n            236.75,\n            97.5,\n            292.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0b7d4ac1e484d546_6",\n          "bbox_xyxy": [\n            59.0,\n            365.25000000000006,\n            126.0,\n            403.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0b7d4ac1e484d546_7",\n          "bbox_xyxy": [\n            370.25,\n            526.75,\n            414.75,\n            578.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0b7d4ac1e484d546_8",\n          "bbox_xyxy": [\n            96.75,\n            0.2500000000000002,\n            154.25,\n            12.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0b7d4ac1e484d546_9",\n          "bbox_xyxy": [\n            246.5,\n            426.0,\n            281.5,\n            489.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0b7d4ac1e484d546_10",\n          "bbox_xyxy": [\n            98.25000000000001,\n            27.0,\n            134.75,\n            91.00000000000001\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0b7d4ac1e484d546_11",\n          "bbox_xyxy": [\n            241.5,\n            92.25,\n            281.5,\n            141.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0b7d4ac1e484d546_12",\n          "bbox_xyxy": [\n            408.00000000000006,\n            100.25,\n            454.0,\n            155.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/6159142536404995619_jpg.rf.df4598db7988891f4291e82ff7565e47.jpg",\n        "size": 62505,\n        "compressed": 62305,\n        "offset": 409038177,\n        "crc32": "95a95b29",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_0bcc92d3dce91c35",\n      "relative_path": "images/IMG_2181_JPG.rf.e129dfe0c3fafebdc0ddace5003e57fe.jpg",\n      "sha256": "932afbbe126c0e9e0bc6242b949e77c187884db4e87db9f13a58f7505320cb9d",\n      "decoded_sha256": "0fedaca5e5dd7a41817b79a74cbbc6990531f6879bf41155d701743a874d6cb7",\n      "dhash": "7070707070703230",\n      "bytes": 25829,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2181_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_0bcc92d3dce91c35_0",\n          "bbox_xyxy": [\n            90.75,\n            0.0,\n            587.25,\n            614.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_2181_JPG.rf.e129dfe0c3fafebdc0ddace5003e57fe.jpg",\n        "size": 25829,\n        "compressed": 21861,\n        "offset": 533286554,\n        "crc32": "6775415b",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_0c13ed3e330068ca",\n      "relative_path": "images/IMG_4209_JPG.rf.95a0e6a45b50c01a689ccf511473f0a6.jpg",\n      "sha256": "01cd638138a3b4cf7c0166d7ba7482439e3d89353c5267ce1e0a4b2bcbf4af3d",\n      "decoded_sha256": "e265436f2a3d4c9887875d35bfe21e607983076d513cadd34d9327c4685e6e47",\n      "dhash": "a6ca555a7471b5ed",\n      "bytes": 47881,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_4209_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_0c13ed3e330068ca_0",\n          "bbox_xyxy": [\n            245.99999999999997,\n            52.25,\n            278.0,\n            93.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c13ed3e330068ca_1",\n          "bbox_xyxy": [\n            180.0,\n            92.25,\n            234.0,\n            110.75000000000001\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c13ed3e330068ca_2",\n          "bbox_xyxy": [\n            85.75,\n            50.74999999999999,\n            118.25,\n            86.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c13ed3e330068ca_3",\n          "bbox_xyxy": [\n            137.75,\n            136.75,\n            188.25,\n            173.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c13ed3e330068ca_4",\n          "bbox_xyxy": [\n            153.25,\n            216.50000000000003,\n            198.75,\n            244.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c13ed3e330068ca_5",\n          "bbox_xyxy": [\n            169.75,\n            405.75,\n            207.25,\n            433.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c13ed3e330068ca_6",\n          "bbox_xyxy": [\n            88.25,\n            483.25,\n            151.75,\n            506.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c13ed3e330068ca_7",\n          "bbox_xyxy": [\n            98.0,\n            532.0,\n            135.0,\n            555.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c13ed3e330068ca_8",\n          "bbox_xyxy": [\n            235.5,\n            476.99999999999994,\n            261.5,\n            514.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c13ed3e330068ca_9",\n          "bbox_xyxy": [\n            338.5,\n            502.25,\n            363.5,\n            546.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c13ed3e330068ca_10",\n          "bbox_xyxy": [\n            434.74999999999994,\n            586.75,\n            476.25,\n            631.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c13ed3e330068ca_11",\n          "bbox_xyxy": [\n            465.5,\n            453.75,\n            512.5,\n            490.25000000000006\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c13ed3e330068ca_12",\n          "bbox_xyxy": [\n            389.25000000000006,\n            408.0,\n            412.75,\n            455.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c13ed3e330068ca_13",\n          "bbox_xyxy": [\n            435.50000000000006,\n            376.99999999999994,\n            473.5,\n            406.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c13ed3e330068ca_14",\n          "bbox_xyxy": [\n            314.5,\n            299.0,\n            373.5,\n            323.00000000000006\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c13ed3e330068ca_15",\n          "bbox_xyxy": [\n            513.25,\n            279.0,\n            552.75,\n            308.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c13ed3e330068ca_16",\n          "bbox_xyxy": [\n            431.75000000000006,\n            204.0,\n            477.25,\n            231.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c13ed3e330068ca_17",\n          "bbox_xyxy": [\n            397.5,\n            255.0,\n            433.49999999999994,\n            289.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c13ed3e330068ca_18",\n          "bbox_xyxy": [\n            340.5,\n            82.0,\n            379.5,\n            112.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c13ed3e330068ca_19",\n          "bbox_xyxy": [\n            480.25,\n            63.00000000000001,\n            542.75,\n            105.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_4209_JPG.rf.95a0e6a45b50c01a689ccf511473f0a6.jpg",\n        "size": 47881,\n        "compressed": 47514,\n        "offset": 612286461,\n        "crc32": "f37c56e7",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_0c4051118b21f1fb",\n      "relative_path": "images/6100321344378571847_jpg.rf.29dbd2e04805cf8155defda485922ff9.jpg",\n      "sha256": "3f46b8e2da707416c7637eb9faddb8e56eff8045193fb8037076bb2970df6341",\n      "decoded_sha256": "8fcd83d3891540451095a76ad28ddd6729647bdeb1802c10559ff97995edf560",\n      "dhash": "478b9d0dc7975b93",\n      "bytes": 132422,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "6100321344378571847_jpg",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_0c4051118b21f1fb_0",\n          "bbox_xyxy": [\n            153.75,\n            485.25,\n            186.25,\n            511.75000000000006\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c4051118b21f1fb_1",\n          "bbox_xyxy": [\n            180.5,\n            220.25,\n            206.50000000000003,\n            262.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c4051118b21f1fb_2",\n          "bbox_xyxy": [\n            323.00000000000006,\n            524.5,\n            361.0,\n            544.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c4051118b21f1fb_3",\n          "bbox_xyxy": [\n            579.25,\n            129.0,\n            602.75,\n            166.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c4051118b21f1fb_4",\n          "bbox_xyxy": [\n            583.25,\n            603.25,\n            603.75,\n            631.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c4051118b21f1fb_5",\n          "bbox_xyxy": [\n            276.5,\n            99.0,\n            314.5,\n            126.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c4051118b21f1fb_6",\n          "bbox_xyxy": [\n            198.5,\n            70.75,\n            227.5,\n            111.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c4051118b21f1fb_7",\n          "bbox_xyxy": [\n            515.5,\n            84.75,\n            534.5,\n            105.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c4051118b21f1fb_8",\n          "bbox_xyxy": [\n            425.50000000000006,\n            13.5,\n            453.5,\n            46.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c4051118b21f1fb_9",\n          "bbox_xyxy": [\n            546.0,\n            124.75,\n            570.0,\n            143.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c4051118b21f1fb_10",\n          "bbox_xyxy": [\n            316.25,\n            583.0,\n            340.75,\n            613.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c4051118b21f1fb_11",\n          "bbox_xyxy": [\n            31.250000000000004,\n            0.2499999999999991,\n            51.75000000000001,\n            29.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c4051118b21f1fb_12",\n          "bbox_xyxy": [\n            95.5,\n            617.25,\n            124.5,\n            639.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c4051118b21f1fb_13",\n          "bbox_xyxy": [\n            341.5,\n            591.75,\n            375.50000000000006,\n            624.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c4051118b21f1fb_14",\n          "bbox_xyxy": [\n            184.0,\n            0.0,\n            210.0,\n            30.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c4051118b21f1fb_15",\n          "bbox_xyxy": [\n            0.0,\n            240.49999999999997,\n            28.0,\n            267.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c4051118b21f1fb_16",\n          "bbox_xyxy": [\n            19.0,\n            269.0,\n            51.0,\n            292.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c4051118b21f1fb_17",\n          "bbox_xyxy": [\n            229.24999999999997,\n            365.50000000000006,\n            263.75,\n            393.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/6100321344378571847_jpg.rf.29dbd2e04805cf8155defda485922ff9.jpg",\n        "size": 132422,\n        "compressed": 132321,\n        "offset": 395621582,\n        "crc32": "a8528947",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_0c4b4b17c66ba65b",\n      "relative_path": "images/6273997143085354574_jpg.rf.566b2325ad37951d1d1ea41166983628.jpg",\n      "sha256": "37cbb0c672966b0b050a0a64216470657c08ded9363887c9735396ed84870fea",\n      "decoded_sha256": "a04d04f5056318bd2e8bfa126067c6f7d52bdefe73131c97c74874b511947d2f",\n      "dhash": "d49d09a517368b1a",\n      "bytes": 50246,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "6273997143085354574_jpg",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_0c4b4b17c66ba65b_0",\n          "bbox_xyxy": [\n            103.5,\n            511.5,\n            166.5,\n            564.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c4b4b17c66ba65b_1",\n          "bbox_xyxy": [\n            529.25,\n            577.25,\n            579.75,\n            622.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c4b4b17c66ba65b_2",\n          "bbox_xyxy": [\n            368.49999999999994,\n            529.5,\n            417.5,\n            574.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c4b4b17c66ba65b_3",\n          "bbox_xyxy": [\n            527.25,\n            416.0,\n            569.75,\n            453.00000000000006\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c4b4b17c66ba65b_4",\n          "bbox_xyxy": [\n            282.75,\n            384.75,\n            321.25,\n            424.25000000000006\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c4b4b17c66ba65b_5",\n          "bbox_xyxy": [\n            358.75,\n            160.25,\n            404.25,\n            187.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c4b4b17c66ba65b_6",\n          "bbox_xyxy": [\n            430.74999999999994,\n            105.25,\n            487.25,\n            131.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c4b4b17c66ba65b_7",\n          "bbox_xyxy": [\n            285.75,\n            15.249999999999995,\n            310.25,\n            66.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c4b4b17c66ba65b_8",\n          "bbox_xyxy": [\n            417.24999999999994,\n            23.75,\n            453.75,\n            66.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c4b4b17c66ba65b_9",\n          "bbox_xyxy": [\n            534.5,\n            41.00000000000001,\n            561.5,\n            87.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c4b4b17c66ba65b_10",\n          "bbox_xyxy": [\n            123.25,\n            8.5,\n            163.75,\n            54.49999999999999\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c4b4b17c66ba65b_11",\n          "bbox_xyxy": [\n            110.0,\n            121.0,\n            167.0,\n            164.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c4b4b17c66ba65b_12",\n          "bbox_xyxy": [\n            83.5,\n            264.75,\n            128.5,\n            306.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/6273997143085354574_jpg.rf.566b2325ad37951d1d1ea41166983628.jpg",\n        "size": 50246,\n        "compressed": 49863,\n        "offset": 431614520,\n        "crc32": "2b41d4d3",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_0c58ca3762c01433",\n      "relative_path": "images/IMG_1215_JPG.rf.87ec574af4362b00d42eb8241c5c5ced.jpg",\n      "sha256": "ccdd80ab4b79f636d0d6c461b66f5ba56fa9271b60d2dc3bac263a56a27578b3",\n      "decoded_sha256": "eecdab23758b7e7375192ad246c241e553f3e83800a23cbf48f4361e26aae001",\n      "dhash": "360a178869b58d49",\n      "bytes": 50385,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_1215_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_0c58ca3762c01433_0",\n          "bbox_xyxy": [\n            306.0,\n            319.75,\n            345.0,\n            362.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c58ca3762c01433_1",\n          "bbox_xyxy": [\n            256.25,\n            3.999999999999999,\n            284.75,\n            67.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c58ca3762c01433_2",\n          "bbox_xyxy": [\n            361.75,\n            273.0,\n            404.25,\n            315.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c58ca3762c01433_3",\n          "bbox_xyxy": [\n            194.25,\n            64.5,\n            226.75,\n            120.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c58ca3762c01433_4",\n          "bbox_xyxy": [\n            312.25,\n            196.75,\n            339.75,\n            255.25000000000003\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c58ca3762c01433_5",\n          "bbox_xyxy": [\n            89.0,\n            0.2499999999999991,\n            129.0,\n            32.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c58ca3762c01433_6",\n          "bbox_xyxy": [\n            0.0,\n            141.24999999999997,\n            36.0,\n            196.74999999999997\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c58ca3762c01433_7",\n          "bbox_xyxy": [\n            125.25,\n            117.25,\n            155.75,\n            167.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c58ca3762c01433_8",\n          "bbox_xyxy": [\n            356.0,\n            347.5,\n            386.99999999999994,\n            397.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c58ca3762c01433_9",\n          "bbox_xyxy": [\n            103.5,\n            268.25,\n            143.5,\n            329.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c58ca3762c01433_10",\n          "bbox_xyxy": [\n            270.5,\n            597.0,\n            298.5,\n            640.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c58ca3762c01433_11",\n          "bbox_xyxy": [\n            236.74999999999997,\n            325.75,\n            271.25,\n            374.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c58ca3762c01433_12",\n          "bbox_xyxy": [\n            48.25,\n            96.75,\n            86.75,\n            140.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c58ca3762c01433_13",\n          "bbox_xyxy": [\n            508.5,\n            116.00000000000001,\n            538.5,\n            177.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c58ca3762c01433_14",\n          "bbox_xyxy": [\n            550.25,\n            306.25,\n            582.75,\n            351.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c58ca3762c01433_15",\n          "bbox_xyxy": [\n            337.75,\n            455.0,\n            374.25000000000006,\n            499.00000000000006\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c58ca3762c01433_16",\n          "bbox_xyxy": [\n            267.5,\n            153.5,\n            291.5,\n            212.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c58ca3762c01433_17",\n          "bbox_xyxy": [\n            0.2500000000000002,\n            274.0,\n            17.75,\n            322.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c58ca3762c01433_18",\n          "bbox_xyxy": [\n            440.0,\n            356.5,\n            464.00000000000006,\n            410.50000000000006\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c58ca3762c01433_19",\n          "bbox_xyxy": [\n            433.25,\n            462.5,\n            471.75,\n            501.50000000000006\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c58ca3762c01433_20",\n          "bbox_xyxy": [\n            518.75,\n            10.0,\n            546.25,\n            65.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c58ca3762c01433_21",\n          "bbox_xyxy": [\n            249.50000000000003,\n            259.25,\n            282.5,\n            306.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c58ca3762c01433_22",\n          "bbox_xyxy": [\n            97.25,\n            420.0,\n            120.75000000000001,\n            482.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c58ca3762c01433_23",\n          "bbox_xyxy": [\n            179.5,\n            297.25,\n            206.50000000000003,\n            354.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c58ca3762c01433_24",\n          "bbox_xyxy": [\n            606.25,\n            186.0,\n            639.75,\n            230.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c58ca3762c01433_25",\n          "bbox_xyxy": [\n            16.0,\n            0.0,\n            58.00000000000001,\n            32.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c58ca3762c01433_26",\n          "bbox_xyxy": [\n            465.99999999999994,\n            0.0,\n            500.0,\n            23.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c58ca3762c01433_27",\n          "bbox_xyxy": [\n            205.5,\n            166.74999999999997,\n            228.49999999999997,\n            231.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c58ca3762c01433_28",\n          "bbox_xyxy": [\n            229.0,\n            427.5,\n            254.0,\n            488.49999999999994\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c58ca3762c01433_29",\n          "bbox_xyxy": [\n            589.5,\n            430.25,\n            627.5,\n            471.75000000000006\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c58ca3762c01433_30",\n          "bbox_xyxy": [\n            470.75,\n            500.25,\n            498.25,\n            566.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c58ca3762c01433_31",\n          "bbox_xyxy": [\n            288.25,\n            419.0,\n            312.75,\n            471.99999999999994\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c58ca3762c01433_32",\n          "bbox_xyxy": [\n            552.25,\n            537.0,\n            579.75,\n            599.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c58ca3762c01433_33",\n          "bbox_xyxy": [\n            426.25,\n            261.5,\n            458.75,\n            312.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c58ca3762c01433_34",\n          "bbox_xyxy": [\n            424.75,\n            585.75,\n            452.25,\n            639.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c58ca3762c01433_35",\n          "bbox_xyxy": [\n            530.75,\n            211.5,\n            557.25,\n            268.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c58ca3762c01433_36",\n          "bbox_xyxy": [\n            322.5,\n            30.000000000000004,\n            366.50000000000006,\n            73.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c58ca3762c01433_37",\n          "bbox_xyxy": [\n            435.50000000000006,\n            172.5,\n            466.5,\n            227.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c58ca3762c01433_38",\n          "bbox_xyxy": [\n            29.25,\n            315.25,\n            57.74999999999999,\n            371.75000000000006\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c58ca3762c01433_39",\n          "bbox_xyxy": [\n            364.49999999999994,\n            161.5,\n            388.5,\n            213.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c58ca3762c01433_40",\n          "bbox_xyxy": [\n            476.75,\n            400.74999999999994,\n            509.25,\n            447.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c58ca3762c01433_41",\n          "bbox_xyxy": [\n            409.0,\n            46.75000000000001,\n            444.0,\n            92.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c58ca3762c01433_42",\n          "bbox_xyxy": [\n            184.25,\n            538.75,\n            217.75000000000003,\n            587.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c58ca3762c01433_43",\n          "bbox_xyxy": [\n            363.5,\n            565.25,\n            394.49999999999994,\n            618.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c58ca3762c01433_44",\n          "bbox_xyxy": [\n            129.0,\n            516.25,\n            163.0,\n            561.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c58ca3762c01433_45",\n          "bbox_xyxy": [\n            601.0,\n            577.0,\n            640.0,\n            612.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c58ca3762c01433_46",\n          "bbox_xyxy": [\n            68.5,\n            184.75,\n            102.5,\n            232.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c58ca3762c01433_47",\n          "bbox_xyxy": [\n            18.750000000000004,\n            33.75000000000001,\n            50.25,\n            79.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c58ca3762c01433_48",\n          "bbox_xyxy": [\n            43.00000000000001,\n            528.25,\n            80.0,\n            579.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c58ca3762c01433_49",\n          "bbox_xyxy": [\n            166.25,\n            0.0,\n            201.74999999999997,\n            23.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c58ca3762c01433_50",\n          "bbox_xyxy": [\n            603.75,\n            0.0,\n            631.25,\n            29.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c58ca3762c01433_51",\n          "bbox_xyxy": [\n            136.5,\n            216.25,\n            173.5,\n            266.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c58ca3762c01433_52",\n          "bbox_xyxy": [\n            146.0,\n            406.5,\n            175.0,\n            460.50000000000006\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c58ca3762c01433_53",\n          "bbox_xyxy": [\n            0.2500000000000002,\n            439.49999999999994,\n            10.750000000000002,\n            481.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_1215_JPG.rf.87ec574af4362b00d42eb8241c5c5ced.jpg",\n        "size": 50385,\n        "compressed": 50219,\n        "offset": 495621626,\n        "crc32": "3919829d",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_0c592aa578b3fbba",\n      "relative_path": "images/IMG_0830_JPG.rf.e198012532683323b2f958a3a9526a6f.jpg",\n      "sha256": "9a774353f2d23203d31bc0613ccd0b13455035a0393730bee8fb14c4b0f7aa76",\n      "decoded_sha256": "0870a97263043ef8a020d4309ff3c386ad9ee89172726ef1ab01defafa47a049",\n      "dhash": "06342d5272c76c4d",\n      "bytes": 52294,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_0830_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_0c592aa578b3fbba_0",\n          "bbox_xyxy": [\n            514.5,\n            118.75,\n            601.5,\n            166.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c592aa578b3fbba_1",\n          "bbox_xyxy": [\n            414.0,\n            102.75,\n            466.99999999999994,\n            175.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c592aa578b3fbba_2",\n          "bbox_xyxy": [\n            96.25,\n            136.0,\n            158.75,\n            203.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c592aa578b3fbba_3",\n          "bbox_xyxy": [\n            0.0,\n            24.499999999999996,\n            29.0,\n            87.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c592aa578b3fbba_4",\n          "bbox_xyxy": [\n            0.0,\n            58.5,\n            84.0,\n            108.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c592aa578b3fbba_5",\n          "bbox_xyxy": [\n            24.000000000000004,\n            217.99999999999997,\n            147.0,\n            260.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c592aa578b3fbba_6",\n          "bbox_xyxy": [\n            17.5,\n            127.25,\n            87.5,\n            189.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c592aa578b3fbba_7",\n          "bbox_xyxy": [\n            0.0,\n            169.0,\n            45.0,\n            246.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c592aa578b3fbba_8",\n          "bbox_xyxy": [\n            499.5,\n            3.999999999999999,\n            554.5,\n            67.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c592aa578b3fbba_9",\n          "bbox_xyxy": [\n            371.0,\n            172.5,\n            471.0,\n            203.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c592aa578b3fbba_10",\n          "bbox_xyxy": [\n            378.25,\n            210.25,\n            485.75,\n            237.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c592aa578b3fbba_11",\n          "bbox_xyxy": [\n            474.5,\n            223.75,\n            577.5,\n            274.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c592aa578b3fbba_12",\n          "bbox_xyxy": [\n            529.0,\n            181.0,\n            640.0,\n            236.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c592aa578b3fbba_13",\n          "bbox_xyxy": [\n            511.75,\n            299.0,\n            579.25,\n            359.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c592aa578b3fbba_14",\n          "bbox_xyxy": [\n            583.25,\n            358.00000000000006,\n            639.75,\n            411.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c592aa578b3fbba_15",\n          "bbox_xyxy": [\n            371.99999999999994,\n            489.75,\n            469.0,\n            525.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c592aa578b3fbba_16",\n          "bbox_xyxy": [\n            504.49999999999994,\n            470.74999999999994,\n            558.5,\n            537.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c592aa578b3fbba_17",\n          "bbox_xyxy": [\n            435.99999999999994,\n            507.75,\n            515.0,\n            548.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c592aa578b3fbba_18",\n          "bbox_xyxy": [\n            380.25,\n            553.75,\n            489.75,\n            598.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c592aa578b3fbba_19",\n          "bbox_xyxy": [\n            467.24999999999994,\n            622.0,\n            558.75,\n            640.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c592aa578b3fbba_20",\n          "bbox_xyxy": [\n            555.75,\n            595.0,\n            633.25,\n            640.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c592aa578b3fbba_21",\n          "bbox_xyxy": [\n            474.25,\n            557.25,\n            556.75,\n            609.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c592aa578b3fbba_22",\n          "bbox_xyxy": [\n            260.5,\n            613.0,\n            323.5,\n            640.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c592aa578b3fbba_23",\n          "bbox_xyxy": [\n            152.5,\n            551.5,\n            226.5,\n            617.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c592aa578b3fbba_24",\n          "bbox_xyxy": [\n            223.49999999999997,\n            488.75,\n            285.5,\n            552.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c592aa578b3fbba_25",\n          "bbox_xyxy": [\n            0.0,\n            530.0,\n            49.0,\n            598.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c592aa578b3fbba_26",\n          "bbox_xyxy": [\n            55.25,\n            423.75,\n            135.75,\n            479.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c592aa578b3fbba_27",\n          "bbox_xyxy": [\n            180.0,\n            381.75,\n            267.0,\n            421.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c592aa578b3fbba_28",\n          "bbox_xyxy": [\n            121.00000000000001,\n            424.0,\n            220.0,\n            471.99999999999994\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c592aa578b3fbba_29",\n          "bbox_xyxy": [\n            326.0,\n            319.0,\n            381.0,\n            381.99999999999994\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c592aa578b3fbba_30",\n          "bbox_xyxy": [\n            475.99999999999994,\n            362.75,\n            535.0,\n            432.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c592aa578b3fbba_31",\n          "bbox_xyxy": [\n            401.0,\n            242.25,\n            471.99999999999994,\n            267.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c592aa578b3fbba_32",\n          "bbox_xyxy": [\n            314.25,\n            223.75,\n            402.75,\n            277.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c592aa578b3fbba_33",\n          "bbox_xyxy": [\n            239.24999999999997,\n            236.50000000000003,\n            344.75,\n            279.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c592aa578b3fbba_34",\n          "bbox_xyxy": [\n            164.75,\n            201.50000000000003,\n            245.25,\n            274.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c592aa578b3fbba_35",\n          "bbox_xyxy": [\n            222.24999999999997,\n            130.25,\n            296.75,\n            186.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0c592aa578b3fbba_36",\n          "bbox_xyxy": [\n            288.0,\n            97.5,\n            384.0,\n            156.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_0830_JPG.rf.e198012532683323b2f958a3a9526a6f.jpg",\n        "size": 52294,\n        "compressed": 52135,\n        "offset": 447898125,\n        "crc32": "720f8898",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_0c9d1826dbfabcac",\n      "relative_path": "images/IMG_2280_JPG.rf.267d3f74213d0ef74421a972d2fb21fa.jpg",\n      "sha256": "3cdcf7e44f201440e79a344989484d40a92cc1507ad4abb427c617661b104615",\n      "decoded_sha256": "1e671e2e92c5f250b76e878f154f41d434614dc1b33f719b3d39b2ffdd00633b",\n      "dhash": "6072307868703030",\n      "bytes": 26242,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2280_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_0c9d1826dbfabcac_0",\n          "bbox_xyxy": [\n            152.50000000000003,\n            58.999999999999986,\n            556.5,\n            617.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_2280_JPG.rf.267d3f74213d0ef74421a972d2fb21fa.jpg",\n        "size": 26242,\n        "compressed": 22203,\n        "offset": 541563992,\n        "crc32": "f033eb78",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_0cf4003dc2590333",\n      "relative_path": "images/IMG_1982_JPG.rf.97fd60ec9631f82ac8373e32aff82c07.jpg",\n      "sha256": "e4797f07a804d44479d9b6fdce4fdb1de6375d8c568ed9bb3925dcab87936186",\n      "decoded_sha256": "92fd20e955fa8fc27db3869113f93912580c5f0262a565eaa9e82526c7ecd942",\n      "dhash": "32f1e979797172cc",\n      "bytes": 26353,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_1982_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_0cf4003dc2590333_0",\n          "bbox_xyxy": [\n            114.5,\n            0.0,\n            496.50000000000006,\n            631.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_1982_JPG.rf.97fd60ec9631f82ac8373e32aff82c07.jpg",\n        "size": 26353,\n        "compressed": 25626,\n        "offset": 507334819,\n        "crc32": "5c640455",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_0d375b362534785f",\n      "relative_path": "images/IMG_2232_JPG.rf.1b6c4c6d0a3466f46575518e408715b1.jpg",\n      "sha256": "d4350e606cb7ed91eb147efdc28ecbad7e99903a72636611343e9919f4830897",\n      "decoded_sha256": "5cf66445d3e7103589d4055d33ca800309144a06e28901c3927cda8be5e65d53",\n      "dhash": "b27169e96971314c",\n      "bytes": 31335,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2232_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_0d375b362534785f_0",\n          "bbox_xyxy": [\n            125.00000000000003,\n            29.00000000000002,\n            519.0,\n            615.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_2232_JPG.rf.1b6c4c6d0a3466f46575518e408715b1.jpg",\n        "size": 31335,\n        "compressed": 30764,\n        "offset": 537332598,\n        "crc32": "ab125850",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_0d9b37ed46b23bd0",\n      "relative_path": "images/IMG_1170_JPG.rf.352bf09d5d4f8d132dccf08acf5f1e3f.jpg",\n      "sha256": "96d0e214c32519fd914bd224e0dd29856e2d9c3f4b480510badb7fb2808864e4",\n      "decoded_sha256": "97ce09dde6fb6dae8ad4345b069b4a3384b4793572daed2d954606c64374dcd7",\n      "dhash": "6cd3dc74abb42ed6",\n      "bytes": 39250,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_1170_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_0d9b37ed46b23bd0_0",\n          "bbox_xyxy": [\n            302.0,\n            91.25,\n            337.0,\n            141.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0d9b37ed46b23bd0_1",\n          "bbox_xyxy": [\n            278.25,\n            412.25,\n            315.75,\n            456.75000000000006\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0d9b37ed46b23bd0_2",\n          "bbox_xyxy": [\n            504.75,\n            460.0,\n            538.25,\n            522.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0d9b37ed46b23bd0_3",\n          "bbox_xyxy": [\n            390.75,\n            456.25,\n            431.25,\n            500.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0d9b37ed46b23bd0_4",\n          "bbox_xyxy": [\n            136.75,\n            448.0,\n            168.25,\n            495.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0d9b37ed46b23bd0_5",\n          "bbox_xyxy": [\n            290.75,\n            277.25,\n            340.25,\n            321.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0d9b37ed46b23bd0_6",\n          "bbox_xyxy": [\n            231.0,\n            381.0,\n            275.0,\n            421.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0d9b37ed46b23bd0_7",\n          "bbox_xyxy": [\n            396.25,\n            344.74999999999994,\n            438.75,\n            386.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0d9b37ed46b23bd0_8",\n          "bbox_xyxy": [\n            437.5,\n            272.25,\n            480.5,\n            325.74999999999994\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0d9b37ed46b23bd0_9",\n          "bbox_xyxy": [\n            518.5,\n            587.0,\n            553.5,\n            640.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0d9b37ed46b23bd0_10",\n          "bbox_xyxy": [\n            516.75,\n            0.2499999999999991,\n            560.25,\n            28.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0d9b37ed46b23bd0_11",\n          "bbox_xyxy": [\n            463.75,\n            423.75,\n            490.25000000000006,\n            487.24999999999994\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0d9b37ed46b23bd0_12",\n          "bbox_xyxy": [\n            348.75,\n            351.0,\n            380.25000000000006,\n            401.99999999999994\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0d9b37ed46b23bd0_13",\n          "bbox_xyxy": [\n            138.75,\n            87.75,\n            179.25,\n            125.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0d9b37ed46b23bd0_14",\n          "bbox_xyxy": [\n            283.25,\n            223.25000000000003,\n            330.75,\n            268.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0d9b37ed46b23bd0_15",\n          "bbox_xyxy": [\n            586.0,\n            336.0,\n            624.0,\n            391.99999999999994\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0d9b37ed46b23bd0_16",\n          "bbox_xyxy": [\n            252.00000000000003,\n            478.25,\n            275.0,\n            536.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0d9b37ed46b23bd0_17",\n          "bbox_xyxy": [\n            203.5,\n            271.0,\n            231.5,\n            324.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0d9b37ed46b23bd0_18",\n          "bbox_xyxy": [\n            147.75,\n            192.75000000000003,\n            181.25,\n            238.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0d9b37ed46b23bd0_19",\n          "bbox_xyxy": [\n            358.00000000000006,\n            251.0,\n            384.0,\n            306.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0d9b37ed46b23bd0_20",\n          "bbox_xyxy": [\n            49.25000000000001,\n            499.25,\n            78.75,\n            563.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0d9b37ed46b23bd0_21",\n          "bbox_xyxy": [\n            182.0,\n            551.5,\n            213.0,\n            613.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0d9b37ed46b23bd0_22",\n          "bbox_xyxy": [\n            364.00000000000006,\n            0.25000000000000133,\n            405.0,\n            20.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0d9b37ed46b23bd0_23",\n          "bbox_xyxy": [\n            437.5,\n            553.5,\n            465.5,\n            611.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0d9b37ed46b23bd0_24",\n          "bbox_xyxy": [\n            288.0,\n            592.0,\n            317.0,\n            640.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0d9b37ed46b23bd0_25",\n          "bbox_xyxy": [\n            459.0,\n            8.5,\n            496.99999999999994,\n            55.50000000000001\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0d9b37ed46b23bd0_26",\n          "bbox_xyxy": [\n            363.75,\n            135.25000000000003,\n            392.24999999999994,\n            191.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0d9b37ed46b23bd0_27",\n          "bbox_xyxy": [\n            353.49999999999994,\n            600.25,\n            387.5,\n            639.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0d9b37ed46b23bd0_28",\n          "bbox_xyxy": [\n            483.25,\n            182.75000000000003,\n            523.75,\n            219.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0d9b37ed46b23bd0_29",\n          "bbox_xyxy": [\n            530.5,\n            345.0,\n            567.5,\n            390.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0d9b37ed46b23bd0_30",\n          "bbox_xyxy": [\n            325.0,\n            440.50000000000006,\n            364.00000000000006,\n            481.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0d9b37ed46b23bd0_31",\n          "bbox_xyxy": [\n            412.75,\n            0.25000000000000133,\n            455.25,\n            35.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0d9b37ed46b23bd0_32",\n          "bbox_xyxy": [\n            121.0,\n            624.0,\n            147.0,\n            640.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0d9b37ed46b23bd0_33",\n          "bbox_xyxy": [\n            550.25,\n            112.75,\n            588.75,\n            163.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0d9b37ed46b23bd0_34",\n          "bbox_xyxy": [\n            216.5,\n            65.0,\n            257.5,\n            107.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0d9b37ed46b23bd0_35",\n          "bbox_xyxy": [\n            110.75,\n            383.75,\n            149.25,\n            424.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0d9b37ed46b23bd0_36",\n          "bbox_xyxy": [\n            562.0,\n            496.5,\n            593.0,\n            549.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0d9b37ed46b23bd0_37",\n          "bbox_xyxy": [\n            537.75,\n            224.75,\n            570.25,\n            287.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0d9b37ed46b23bd0_38",\n          "bbox_xyxy": [\n            606.0,\n            76.0,\n            640.0,\n            110.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0d9b37ed46b23bd0_39",\n          "bbox_xyxy": [\n            449.25,\n            163.0,\n            475.75,\n            215.99999999999997\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_1170_JPG.rf.352bf09d5d4f8d132dccf08acf5f1e3f.jpg",\n        "size": 39250,\n        "compressed": 38424,\n        "offset": 487413556,\n        "crc32": "cfb2c6ac",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_0dd21c13a593263d",\n      "relative_path": "images/IMG_2674_JPG.rf.67b71c1e38786d9cc7f51176f2da0859.jpg",\n      "sha256": "116b081fee48527337ea8926688afbf23b706f20cb604ea485e095256823f455",\n      "decoded_sha256": "18923356bba93e7b7b6e1a870a38e7c842da78c53a3776c3319cda8482117e31",\n      "dhash": "3470707868707030",\n      "bytes": 23552,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2674_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_0dd21c13a593263d_0",\n          "bbox_xyxy": [\n            126.25000000000003,\n            2.5,\n            547.75,\n            587.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_2674_JPG.rf.67b71c1e38786d9cc7f51176f2da0859.jpg",\n        "size": 23552,\n        "compressed": 19414,\n        "offset": 576703940,\n        "crc32": "566b74dc",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_0e1dc663c5dda10a",\n      "relative_path": "images/IMG_2696_JPG.rf.be73d2ced9d0169e38826b465bd24c4d.jpg",\n      "sha256": "addc252667b66b5bf3d49b792c2cabd9794a151db9cd1f5a7fa6855f16d8617b",\n      "decoded_sha256": "d5eb02baaee840642f82efc021994d6f70e9b9ce4d603009002066a1b2cfde55",\n      "dhash": "6cf0716969e97172",\n      "bytes": 30881,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2696_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_0e1dc663c5dda10a_0",\n          "bbox_xyxy": [\n            121.49999999999999,\n            20.249999999999986,\n            524.5,\n            620.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_2696_JPG.rf.be73d2ced9d0169e38826b465bd24c4d.jpg",\n        "size": 30881,\n        "compressed": 29928,\n        "offset": 579242768,\n        "crc32": "ada437d2",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_0e893645533005b7",\n      "relative_path": "images/IMG_2050_JPG.rf.843e2f7ce73871fb266f8288ee295774.jpg",\n      "sha256": "f7fe2bd4b188fd8dd651ac582419b3838f6e7a4af810765c7ae5c14202441048",\n      "decoded_sha256": "fb532389b67fcf4eceb75a1890cd0155b90bd718b752c5bb8c6659141bf142a1",\n      "dhash": "ccf0317979697931",\n      "bytes": 22958,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2050_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_0e893645533005b7_0",\n          "bbox_xyxy": [\n            155.0,\n            54.00000000000002,\n            523.0,\n            640.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_2050_JPG.rf.843e2f7ce73871fb266f8288ee295774.jpg",\n        "size": 22958,\n        "compressed": 21286,\n        "offset": 522862526,\n        "crc32": "fe706009",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_0eff2c417ecff326",\n      "relative_path": "images/6095710224539913288_jpg.rf.4a0de89a01ff65566605d911546028c6.jpg",\n      "sha256": "9d1c51f34ae307c6a08a2c6354c5f1478765197c7ec909f851a5a0f22e0c653f",\n      "decoded_sha256": "4f4f682c70783667a6c4ce79aae39180780948d82f98a2a2c2e268219085305e",\n      "dhash": "471336250f0f2fae",\n      "bytes": 130874,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "6095710224539913288_jpg",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_0eff2c417ecff326_0",\n          "bbox_xyxy": [\n            184.5,\n            173.0,\n            233.49999999999997,\n            217.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0eff2c417ecff326_1",\n          "bbox_xyxy": [\n            473.5,\n            185.75,\n            519.5,\n            212.24999999999997\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0eff2c417ecff326_2",\n          "bbox_xyxy": [\n            316.0,\n            14.0,\n            370.0,\n            41.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0eff2c417ecff326_3",\n          "bbox_xyxy": [\n            265.75,\n            364.5,\n            316.25,\n            385.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0eff2c417ecff326_4",\n          "bbox_xyxy": [\n            454.5,\n            232.75,\n            487.5,\n            281.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0eff2c417ecff326_5",\n          "bbox_xyxy": [\n            477.0,\n            411.99999999999994,\n            510.0,\n            444.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0eff2c417ecff326_6",\n          "bbox_xyxy": [\n            291.0,\n            88.5,\n            326.99999999999994,\n            121.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0eff2c417ecff326_7",\n          "bbox_xyxy": [\n            563.25,\n            76.25,\n            588.75,\n            115.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0eff2c417ecff326_8",\n          "bbox_xyxy": [\n            273.25,\n            459.5,\n            319.75,\n            479.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0eff2c417ecff326_9",\n          "bbox_xyxy": [\n            401.5,\n            118.0,\n            449.49999999999994,\n            145.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0eff2c417ecff326_10",\n          "bbox_xyxy": [\n            488.5,\n            567.75,\n            523.5,\n            596.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0eff2c417ecff326_11",\n          "bbox_xyxy": [\n            397.75,\n            3.999999999999999,\n            430.25,\n            46.99999999999999\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0eff2c417ecff326_12",\n          "bbox_xyxy": [\n            395.74999999999994,\n            335.50000000000006,\n            427.25,\n            361.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0eff2c417ecff326_13",\n          "bbox_xyxy": [\n            425.99999999999994,\n            548.25,\n            460.0,\n            565.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0eff2c417ecff326_14",\n          "bbox_xyxy": [\n            369.0,\n            406.0,\n            398.00000000000006,\n            443.00000000000006\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0eff2c417ecff326_15",\n          "bbox_xyxy": [\n            236.25,\n            40.74999999999999,\n            264.75,\n            66.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0eff2c417ecff326_16",\n          "bbox_xyxy": [\n            433.5,\n            355.74999999999994,\n            453.5,\n            397.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/6095710224539913288_jpg.rf.4a0de89a01ff65566605d911546028c6.jpg",\n        "size": 130874,\n        "compressed": 130772,\n        "offset": 354998893,\n        "crc32": "5ff874a7",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_0f47eb30b5bc876b",\n      "relative_path": "images/IMG_3970_JPG.rf.f70b019f30f83f8d5876e45c6729d57f.jpg",\n      "sha256": "4f1caf02c6718b68a80e7ff52ed9e465535059b115d7bcf27325a72d7e24fb56",\n      "decoded_sha256": "116eac58e27cce224f83fc5b754577bd88f02304af689549a1052401f8b01380",\n      "dhash": "b4b1ad957252499f",\n      "bytes": 28745,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_3970_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_0f47eb30b5bc876b_0",\n          "bbox_xyxy": [\n            544.75,\n            533.0,\n            620.25,\n            588.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0f47eb30b5bc876b_1",\n          "bbox_xyxy": [\n            98.50000000000001,\n            11.749999999999998,\n            167.5,\n            78.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0f47eb30b5bc876b_2",\n          "bbox_xyxy": [\n            153.0,\n            227.25,\n            214.00000000000003,\n            294.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0f47eb30b5bc876b_3",\n          "bbox_xyxy": [\n            365.5,\n            100.0,\n            419.5,\n            167.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0f47eb30b5bc876b_4",\n          "bbox_xyxy": [\n            86.25,\n            561.5,\n            152.75,\n            619.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0f47eb30b5bc876b_5",\n          "bbox_xyxy": [\n            158.74999999999997,\n            474.5,\n            204.24999999999997,\n            539.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0f47eb30b5bc876b_6",\n          "bbox_xyxy": [\n            11.25,\n            429.25000000000006,\n            67.75,\n            502.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0f47eb30b5bc876b_7",\n          "bbox_xyxy": [\n            430.0,\n            11.25,\n            504.00000000000006,\n            83.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0f47eb30b5bc876b_8",\n          "bbox_xyxy": [\n            342.75,\n            484.75,\n            435.25,\n            520.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0f47eb30b5bc876b_9",\n          "bbox_xyxy": [\n            416.5,\n            164.25,\n            474.49999999999994,\n            215.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0f47eb30b5bc876b_10",\n          "bbox_xyxy": [\n            311.5,\n            382.25,\n            372.5,\n            432.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0f47eb30b5bc876b_11",\n          "bbox_xyxy": [\n            300.75,\n            83.75,\n            343.25,\n            142.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0f47eb30b5bc876b_12",\n          "bbox_xyxy": [\n            294.5,\n            243.75,\n            360.5,\n            301.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0f47eb30b5bc876b_13",\n          "bbox_xyxy": [\n            74.5,\n            217.75000000000003,\n            137.5,\n            264.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0f47eb30b5bc876b_14",\n          "bbox_xyxy": [\n            254.5,\n            12.75,\n            311.5,\n            54.25000000000001\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0f47eb30b5bc876b_15",\n          "bbox_xyxy": [\n            230.5,\n            156.5,\n            289.5,\n            199.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0f47eb30b5bc876b_16",\n          "bbox_xyxy": [\n            412.75,\n            282.25,\n            474.25000000000006,\n            333.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0f47eb30b5bc876b_17",\n          "bbox_xyxy": [\n            497.5,\n            362.75,\n            572.5,\n            413.24999999999994\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0f47eb30b5bc876b_18",\n          "bbox_xyxy": [\n            223.0,\n            472.5,\n            279.0,\n            524.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0f47eb30b5bc876b_19",\n          "bbox_xyxy": [\n            171.75,\n            341.99999999999994,\n            269.25,\n            376.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_3970_JPG.rf.f70b019f30f83f8d5876e45c6729d57f.jpg",\n        "size": 28745,\n        "compressed": 28397,\n        "offset": 586500166,\n        "crc32": "cd56a016",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_0fac0f1bcf2589fd",\n      "relative_path": "images/IMG_1051_JPG.rf.7a44ff27ee7fd310b3d7e39ae229e6f7.jpg",\n      "sha256": "af3f40ee3b1e130d7223a75bbc756b30790f1cf3e2659d3ab696401fc6cf9a50",\n      "decoded_sha256": "0a2cab060546a1c15cecc250b70d88ea15f6ba54179b2c10205bc37e866115ac",\n      "dhash": "bbabd7f557d70f3f",\n      "bytes": 53552,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_1051_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_0fac0f1bcf2589fd_0",\n          "bbox_xyxy": [\n            151.5,\n            188.5,\n            197.5,\n            233.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0fac0f1bcf2589fd_1",\n          "bbox_xyxy": [\n            220.75,\n            434.5,\n            257.25,\n            482.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0fac0f1bcf2589fd_2",\n          "bbox_xyxy": [\n            0.0,\n            450.0,\n            18.0,\n            504.00000000000006\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0fac0f1bcf2589fd_3",\n          "bbox_xyxy": [\n            171.5,\n            527.5,\n            221.5,\n            576.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0fac0f1bcf2589fd_4",\n          "bbox_xyxy": [\n            164.5,\n            347.24999999999994,\n            226.50000000000003,\n            388.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0fac0f1bcf2589fd_5",\n          "bbox_xyxy": [\n            418.25,\n            230.25,\n            483.75,\n            279.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0fac0f1bcf2589fd_6",\n          "bbox_xyxy": [\n            220.5,\n            223.0,\n            286.5,\n            255.99999999999997\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0fac0f1bcf2589fd_7",\n          "bbox_xyxy": [\n            2.250000000000001,\n            544.25,\n            59.75,\n            585.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0fac0f1bcf2589fd_8",\n          "bbox_xyxy": [\n            287.25,\n            228.0,\n            363.75,\n            268.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0fac0f1bcf2589fd_9",\n          "bbox_xyxy": [\n            103.25,\n            426.75000000000006,\n            158.75,\n            467.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0fac0f1bcf2589fd_10",\n          "bbox_xyxy": [\n            162.5,\n            378.0,\n            199.50000000000003,\n            423.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0fac0f1bcf2589fd_11",\n          "bbox_xyxy": [\n            319.0,\n            428.00000000000006,\n            350.0,\n            481.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0fac0f1bcf2589fd_12",\n          "bbox_xyxy": [\n            305.50000000000006,\n            295.25,\n            368.5,\n            329.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0fac0f1bcf2589fd_13",\n          "bbox_xyxy": [\n            210.25000000000003,\n            104.0,\n            250.75,\n            149.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0fac0f1bcf2589fd_14",\n          "bbox_xyxy": [\n            32.75,\n            608.0,\n            96.25,\n            640.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0fac0f1bcf2589fd_15",\n          "bbox_xyxy": [\n            505.0,\n            184.5,\n            578.0,\n            209.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0fac0f1bcf2589fd_16",\n          "bbox_xyxy": [\n            117.25000000000001,\n            79.25,\n            198.75,\n            105.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0fac0f1bcf2589fd_17",\n          "bbox_xyxy": [\n            0.0,\n            293.0,\n            23.0,\n            329.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0fac0f1bcf2589fd_18",\n          "bbox_xyxy": [\n            95.5,\n            298.0,\n            185.5,\n            323.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0fac0f1bcf2589fd_19",\n          "bbox_xyxy": [\n            54.75,\n            496.5,\n            130.25,\n            529.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0fac0f1bcf2589fd_20",\n          "bbox_xyxy": [\n            5.2500000000000036,\n            0.0,\n            82.75,\n            22.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0fac0f1bcf2589fd_21",\n          "bbox_xyxy": [\n            337.25,\n            58.5,\n            397.75,\n            111.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_1051_JPG.rf.7a44ff27ee7fd310b3d7e39ae229e6f7.jpg",\n        "size": 53552,\n        "compressed": 53373,\n        "offset": 466424783,\n        "crc32": "97d3f254",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_0fb112618d1e6eca",\n      "relative_path": "images/IMG_2264_JPG.rf.eabb2aa0cde6ed92806e5a64a5ba183a.jpg",\n      "sha256": "3b7712e55f3337e1f1c5b570ebaf06ff8405f294ec1d8b9450b530eecb9e9c71",\n      "decoded_sha256": "afd71484b6215f5c93d6a6852cee1206a640ed2d3ea4d37d4047e39cb54e9a15",\n      "dhash": "a53819b130383919",\n      "bytes": 25920,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2264_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_0fb112618d1e6eca_0",\n          "bbox_xyxy": [\n            230.99999999999997,\n            36.75000000000001,\n            512.0,\n            580.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_2264_JPG.rf.eabb2aa0cde6ed92806e5a64a5ba183a.jpg",\n        "size": 25920,\n        "compressed": 25148,\n        "offset": 540539818,\n        "crc32": "bf5479f2",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_0fc8f40c95906929",\n      "relative_path": "images/6100321344378571859_jpg.rf.adaf73d3e549467f3952c1ad90550114.jpg",\n      "sha256": "cfc0639538a760963b5f78b1ae5142c2dd0124faf8cfd40d5c14f16ea42b6218",\n      "decoded_sha256": "03306a6778d7a5435e66110e7984ef03c5a29797b22d103a7ef4d542f9debf06",\n      "dhash": "060e1e260d0b132f",\n      "bytes": 122075,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "6100321344378571859_jpg",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_0fc8f40c95906929_0",\n          "bbox_xyxy": [\n            491.0,\n            214.0,\n            518.0,\n            271.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0fc8f40c95906929_1",\n          "bbox_xyxy": [\n            329.75,\n            499.5,\n            359.25000000000006,\n            519.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0fc8f40c95906929_2",\n          "bbox_xyxy": [\n            77.0,\n            452.75,\n            101.0,\n            479.25000000000006\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0fc8f40c95906929_3",\n          "bbox_xyxy": [\n            241.74999999999997,\n            427.5,\n            266.25,\n            453.49999999999994\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0fc8f40c95906929_4",\n          "bbox_xyxy": [\n            517.25,\n            22.750000000000004,\n            546.75,\n            41.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0fc8f40c95906929_5",\n          "bbox_xyxy": [\n            462.5,\n            12.0,\n            482.5,\n            43.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0fc8f40c95906929_6",\n          "bbox_xyxy": [\n            561.25,\n            611.25,\n            580.75,\n            639.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0fc8f40c95906929_7",\n          "bbox_xyxy": [\n            605.5,\n            388.75,\n            629.5,\n            416.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0fc8f40c95906929_8",\n          "bbox_xyxy": [\n            215.25000000000003,\n            48.5,\n            241.75,\n            71.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0fc8f40c95906929_9",\n          "bbox_xyxy": [\n            409.49999999999994,\n            177.75000000000003,\n            438.5,\n            198.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0fc8f40c95906929_10",\n          "bbox_xyxy": [\n            92.25000000000001,\n            156.5,\n            108.75,\n            183.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0fc8f40c95906929_11",\n          "bbox_xyxy": [\n            140.0,\n            422.75,\n            169.0,\n            445.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0fc8f40c95906929_12",\n          "bbox_xyxy": [\n            142.0,\n            458.0,\n            171.0,\n            477.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0fc8f40c95906929_13",\n          "bbox_xyxy": [\n            229.24999999999997,\n            576.25,\n            248.75,\n            606.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0fc8f40c95906929_14",\n          "bbox_xyxy": [\n            491.25,\n            114.5,\n            514.75,\n            144.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0fc8f40c95906929_15",\n          "bbox_xyxy": [\n            348.75,\n            197.25,\n            370.25000000000006,\n            224.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0fc8f40c95906929_16",\n          "bbox_xyxy": [\n            111.74999999999999,\n            41.75,\n            140.25,\n            63.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/6100321344378571859_jpg.rf.adaf73d3e549467f3952c1ad90550114.jpg",\n        "size": 122075,\n        "compressed": 121966,\n        "offset": 400043886,\n        "crc32": "bc070ea0",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_10191efb12095085",\n      "relative_path": "images/IMG_2185_JPG.rf.57c82882d908c4225485d4a447382440.jpg",\n      "sha256": "189c4c1a0237116f378e767a045ae0ba282aab0a6cc3216ec170f1f891843288",\n      "decoded_sha256": "ebdbd20de64d19e08a7b672cd159daf20ab8ec87fd749e4ea3f502fa2e5fb81f",\n      "dhash": "8570717079713056",\n      "bytes": 24054,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2185_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_10191efb12095085_0",\n          "bbox_xyxy": [\n            155.25,\n            60.249999999999986,\n            534.75,\n            568.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_2185_JPG.rf.57c82882d908c4225485d4a447382440.jpg",\n        "size": 24054,\n        "compressed": 22149,\n        "offset": 533655364,\n        "crc32": "77267d43",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_1027f010093a278e",\n      "relative_path": "images/6273997143085354567_jpg.rf.e63fb30e7b471869e261831993978acb.jpg",\n      "sha256": "0e79c96080f0db3781ef61ed5fd7f967421fd12eb468af1dc23e5a2194270b25",\n      "decoded_sha256": "d1322e9b466939ccf2ee1bb8d7028a6a13c63f02ab7cc277b99723e8f5573c9e",\n      "dhash": "db5563ced993a489",\n      "bytes": 38913,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "6273997143085354567_jpg",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_1027f010093a278e_0",\n          "bbox_xyxy": [\n            324.74999999999994,\n            381.25,\n            361.25,\n            434.74999999999994\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1027f010093a278e_1",\n          "bbox_xyxy": [\n            403.75,\n            303.25,\n            452.24999999999994,\n            343.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1027f010093a278e_2",\n          "bbox_xyxy": [\n            122.25,\n            286.0,\n            184.75,\n            325.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1027f010093a278e_3",\n          "bbox_xyxy": [\n            39.0,\n            142.0,\n            75.0,\n            188.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1027f010093a278e_4",\n          "bbox_xyxy": [\n            201.25,\n            143.5,\n            235.75000000000003,\n            198.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1027f010093a278e_5",\n          "bbox_xyxy": [\n            308.75000000000006,\n            173.0,\n            365.25000000000006,\n            213.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1027f010093a278e_6",\n          "bbox_xyxy": [\n            243.75,\n            187.5,\n            284.25,\n            252.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1027f010093a278e_7",\n          "bbox_xyxy": [\n            331.25,\n            40.5,\n            384.74999999999994,\n            94.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1027f010093a278e_8",\n          "bbox_xyxy": [\n            123.5,\n            40.25,\n            173.5,\n            83.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1027f010093a278e_9",\n          "bbox_xyxy": [\n            66.74999999999999,\n            389.75,\n            117.25,\n            447.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1027f010093a278e_10",\n          "bbox_xyxy": [\n            180.5,\n            382.25,\n            256.5,\n            420.74999999999994\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1027f010093a278e_11",\n          "bbox_xyxy": [\n            242.5,\n            497.75,\n            310.5,\n            560.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1027f010093a278e_12",\n          "bbox_xyxy": [\n            85.75,\n            571.25,\n            139.25,\n            608.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1027f010093a278e_13",\n          "bbox_xyxy": [\n            374.25,\n            572.0,\n            420.75,\n            640.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1027f010093a278e_14",\n          "bbox_xyxy": [\n            465.74999999999994,\n            505.74999999999994,\n            520.25,\n            537.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1027f010093a278e_15",\n          "bbox_xyxy": [\n            546.25,\n            330.5,\n            608.75,\n            380.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1027f010093a278e_16",\n          "bbox_xyxy": [\n            496.75000000000006,\n            227.5,\n            557.25,\n            270.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1027f010093a278e_17",\n          "bbox_xyxy": [\n            567.25,\n            159.75,\n            598.75,\n            211.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1027f010093a278e_18",\n          "bbox_xyxy": [\n            451.0,\n            99.0,\n            499.0,\n            166.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1027f010093a278e_19",\n          "bbox_xyxy": [\n            83.5,\n            496.75000000000006,\n            123.5,\n            550.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/6273997143085354567_jpg.rf.e63fb30e7b471869e261831993978acb.jpg",\n        "size": 38913,\n        "compressed": 38544,\n        "offset": 430069806,\n        "crc32": "9c44b108",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_103b67b0d368c89d",\n      "relative_path": "images/6159142536404995625_jpg.rf.c571316ede4ea986263e86a14f767239.jpg",\n      "sha256": "c2da31657e14c2b53b1dbf86fb27b53a100970b2a44cc7a59a42bcc9d1cf2266",\n      "decoded_sha256": "5736bc31b1bd7c24c9a67b87299a5d96df875f03d4838a5671ab73865ebc2e19",\n      "dhash": "9a9e8d486d4d5343",\n      "bytes": 43802,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "6159142536404995625_jpg",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_103b67b0d368c89d_0",\n          "bbox_xyxy": [\n            344.0,\n            282.25,\n            401.0,\n            314.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_103b67b0d368c89d_1",\n          "bbox_xyxy": [\n            476.25,\n            272.0,\n            510.75,\n            317.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_103b67b0d368c89d_2",\n          "bbox_xyxy": [\n            202.25,\n            198.5,\n            232.75,\n            247.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_103b67b0d368c89d_3",\n          "bbox_xyxy": [\n            395.0,\n            393.25,\n            435.99999999999994,\n            425.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_103b67b0d368c89d_4",\n          "bbox_xyxy": [\n            229.25,\n            370.25,\n            280.75,\n            399.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_103b67b0d368c89d_5",\n          "bbox_xyxy": [\n            519.0,\n            78.75,\n            544.0,\n            141.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_103b67b0d368c89d_6",\n          "bbox_xyxy": [\n            435.5,\n            120.25,\n            470.5,\n            170.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_103b67b0d368c89d_7",\n          "bbox_xyxy": [\n            430.50000000000006,\n            186.74999999999997,\n            473.5,\n            216.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_103b67b0d368c89d_8",\n          "bbox_xyxy": [\n            110.0,\n            64.75,\n            142.0,\n            113.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_103b67b0d368c89d_9",\n          "bbox_xyxy": [\n            94.0,\n            174.25,\n            134.0,\n            216.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_103b67b0d368c89d_10",\n          "bbox_xyxy": [\n            336.5,\n            41.49999999999999,\n            379.49999999999994,\n            75.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_103b67b0d368c89d_11",\n          "bbox_xyxy": [\n            143.25,\n            418.00000000000006,\n            175.75,\n            466.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_103b67b0d368c89d_12",\n          "bbox_xyxy": [\n            182.0,\n            508.5,\n            238.0,\n            530.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_103b67b0d368c89d_13",\n          "bbox_xyxy": [\n            445.0,\n            381.75000000000006,\n            498.0,\n            410.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_103b67b0d368c89d_14",\n          "bbox_xyxy": [\n            307.25,\n            520.0,\n            352.75,\n            552.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/6159142536404995625_jpg.rf.c571316ede4ea986263e86a14f767239.jpg",\n        "size": 43802,\n        "compressed": 43586,\n        "offset": 410196819,\n        "crc32": "0b319e4e",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_1158223d7cd82377",\n      "relative_path": "images/IMG_2574_JPG.rf.bc38f1b3441cc8ac53bfba559838d097.jpg",\n      "sha256": "60134413a8a46e29c34ed42fac55a8cad99343eb5c1182e791db86af95a4acfd",\n      "decoded_sha256": "ab63ee9841fd05521e36e002bc5ad4b8ff5f8046d05e19bef716dfa06498d0dd",\n      "dhash": "f9f1f0f4f0f07972",\n      "bytes": 29134,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2574_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_1158223d7cd82377_0",\n          "bbox_xyxy": [\n            91.74999999999997,\n            5.999999999999979,\n            561.25,\n            640.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_2574_JPG.rf.bc38f1b3441cc8ac53bfba559838d097.jpg",\n        "size": 29134,\n        "compressed": 27647,\n        "offset": 566687855,\n        "crc32": "0412c4c6",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_1187605f1c23744d",\n      "relative_path": "images/IMG_20260110_151133_jpg.rf.8379cac498211b30d196313684050067.jpg",\n      "sha256": "ff1d8870946bca5706c1ead825aa84f851b9a30d53912f6e7f793cbcb86fe703",\n      "decoded_sha256": "204896bb2b16c5fd91456ae47e14e0c46c1c6c7bd9642e34ae87227e0ba1278c",\n      "dhash": "ead27091dc649b9a",\n      "bytes": 49822,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_20260110_151133_jpg",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_1187605f1c23744d_0",\n          "bbox_xyxy": [\n            321.0,\n            277.5,\n            381.0,\n            328.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1187605f1c23744d_1",\n          "bbox_xyxy": [\n            90.25,\n            263.5,\n            149.75,\n            308.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1187605f1c23744d_2",\n          "bbox_xyxy": [\n            416.0,\n            371.5,\n            463.00000000000006,\n            425.50000000000006\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1187605f1c23744d_3",\n          "bbox_xyxy": [\n            454.75,\n            357.0,\n            513.25,\n            395.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1187605f1c23744d_4",\n          "bbox_xyxy": [\n            501.0,\n            40.75,\n            551.0,\n            74.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1187605f1c23744d_5",\n          "bbox_xyxy": [\n            235.25,\n            412.5,\n            279.75,\n            439.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1187605f1c23744d_6",\n          "bbox_xyxy": [\n            95.75,\n            531.75,\n            139.25,\n            564.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1187605f1c23744d_7",\n          "bbox_xyxy": [\n            377.0,\n            567.0,\n            415.0,\n            608.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1187605f1c23744d_8",\n          "bbox_xyxy": [\n            478.25,\n            398.24999999999994,\n            533.75,\n            427.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1187605f1c23744d_9",\n          "bbox_xyxy": [\n            296.5,\n            168.25,\n            343.5,\n            199.74999999999997\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_20260110_151133_jpg.rf.8379cac498211b30d196313684050067.jpg",\n        "size": 49822,\n        "compressed": 49651,\n        "offset": 520540312,\n        "crc32": "a62e80de",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_11d5f3bdc0fae223",\n      "relative_path": "images/IMG_1956_JPG.rf.729abc6ea7d8575d7e72932aaa6079cc.jpg",\n      "sha256": "a1b9a0efefd9dd2c5bfb9d693bede7c087fba73328ceb68fa6917dbc70850884",\n      "decoded_sha256": "a197d4d8e8f9a96a41c8ab376416615b579d46d1a1a59d4d2bf798e486e89403",\n      "dhash": "3371616171723365",\n      "bytes": 23673,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_1956_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_11d5f3bdc0fae223_0",\n          "bbox_xyxy": [\n            128.5,\n            29.250000000000007,\n            497.5,\n            562.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_1956_JPG.rf.729abc6ea7d8575d7e72932aaa6079cc.jpg",\n        "size": 23673,\n        "compressed": 22689,\n        "offset": 505369841,\n        "crc32": "151bf160",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_1259a93710c38786",\n      "relative_path": "images/6100321344378571818_jpg.rf.44e31fbd42cf5675f333f5f198f3c680.jpg",\n      "sha256": "7e1ae4b9ca418915e634436492be8e9aac6f8ae8747e20272b16b809e9abc104",\n      "decoded_sha256": "e8e82882d8dbe00385f4d9a6599d2651d76c9d43c79d5c5af8989471e4d00a07",\n      "dhash": "ab924e131b0ba767",\n      "bytes": 109279,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "6100321344378571818_jpg",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_1259a93710c38786_0",\n          "bbox_xyxy": [\n            630.25,\n            22.249999999999996,\n            639.75,\n            73.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1259a93710c38786_1",\n          "bbox_xyxy": [\n            41.0,\n            48.5,\n            83.0,\n            75.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1259a93710c38786_2",\n          "bbox_xyxy": [\n            0.0,\n            79.25,\n            18.0,\n            103.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1259a93710c38786_3",\n          "bbox_xyxy": [\n            251.75,\n            550.75,\n            282.25,\n            572.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1259a93710c38786_4",\n          "bbox_xyxy": [\n            300.0,\n            542.75,\n            328.0,\n            564.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1259a93710c38786_5",\n          "bbox_xyxy": [\n            78.25000000000001,\n            62.25,\n            112.75,\n            96.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1259a93710c38786_6",\n          "bbox_xyxy": [\n            152.0,\n            160.0,\n            180.0,\n            181.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1259a93710c38786_7",\n          "bbox_xyxy": [\n            45.0,\n            210.0,\n            79.0,\n            235.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1259a93710c38786_8",\n          "bbox_xyxy": [\n            73.25,\n            105.25,\n            106.75,\n            126.74999999999999\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1259a93710c38786_9",\n          "bbox_xyxy": [\n            368.24999999999994,\n            27.249999999999996,\n            389.75,\n            54.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1259a93710c38786_10",\n          "bbox_xyxy": [\n            483.25,\n            465.0,\n            510.75,\n            488.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1259a93710c38786_11",\n          "bbox_xyxy": [\n            396.0,\n            0.0,\n            424.0,\n            14.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1259a93710c38786_12",\n          "bbox_xyxy": [\n            515.75,\n            494.75,\n            542.25,\n            518.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1259a93710c38786_13",\n          "bbox_xyxy": [\n            165.75,\n            12.5,\n            184.25,\n            40.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1259a93710c38786_14",\n          "bbox_xyxy": [\n            498.75,\n            150.25,\n            528.25,\n            168.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1259a93710c38786_15",\n          "bbox_xyxy": [\n            98.99999999999999,\n            0.25000000000000133,\n            125.0,\n            21.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1259a93710c38786_16",\n          "bbox_xyxy": [\n            291.5,\n            76.25000000000001,\n            310.5,\n            104.75000000000001\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1259a93710c38786_17",\n          "bbox_xyxy": [\n            406.99999999999994,\n            444.74999999999994,\n            431.0,\n            466.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1259a93710c38786_18",\n          "bbox_xyxy": [\n            233.75,\n            128.25,\n            260.25,\n            146.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1259a93710c38786_19",\n          "bbox_xyxy": [\n            603.75,\n            448.75,\n            624.25,\n            480.25000000000006\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1259a93710c38786_20",\n          "bbox_xyxy": [\n            198.0,\n            190.25,\n            217.0,\n            218.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/6100321344378571818_jpg.rf.44e31fbd42cf5675f333f5f198f3c680.jpg",\n        "size": 109279,\n        "compressed": 109166,\n        "offset": 390453071,\n        "crc32": "df74e232",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_126d5c4c37f438cd",\n      "relative_path": "images/IMG_4088_JPG.rf.a7b91d4367ee32ed32413c2eeea94e61.jpg",\n      "sha256": "13aa7ef1f798a8cfb0ad78a466111e83310ef902d09943bd1dd1cee582966977",\n      "decoded_sha256": "20257ac9541b7a3827b48cb42793d7d06fbab3933f5cc8346e4d39229043afb8",\n      "dhash": "596325c45b33a38d",\n      "bytes": 54009,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_4088_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_126d5c4c37f438cd_0",\n          "bbox_xyxy": [\n            324.25000000000006,\n            174.74999999999997,\n            379.75,\n            203.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_126d5c4c37f438cd_1",\n          "bbox_xyxy": [\n            223.25,\n            197.75,\n            254.74999999999997,\n            257.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_126d5c4c37f438cd_2",\n          "bbox_xyxy": [\n            278.5,\n            216.75,\n            311.5,\n            254.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_126d5c4c37f438cd_3",\n          "bbox_xyxy": [\n            397.75,\n            282.0,\n            430.25,\n            323.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_126d5c4c37f438cd_4",\n          "bbox_xyxy": [\n            170.25,\n            276.25,\n            214.75,\n            305.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_126d5c4c37f438cd_5",\n          "bbox_xyxy": [\n            288.75,\n            418.5,\n            333.25,\n            446.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_126d5c4c37f438cd_6",\n          "bbox_xyxy": [\n            180.25,\n            398.25,\n            228.75,\n            433.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_126d5c4c37f438cd_7",\n          "bbox_xyxy": [\n            284.75,\n            455.25,\n            310.25,\n            507.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_126d5c4c37f438cd_8",\n          "bbox_xyxy": [\n            210.5,\n            513.5,\n            234.5,\n            555.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_126d5c4c37f438cd_9",\n          "bbox_xyxy": [\n            238.75,\n            565.25,\n            269.25,\n            607.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_126d5c4c37f438cd_10",\n          "bbox_xyxy": [\n            112.75,\n            565.75,\n            150.25,\n            598.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_126d5c4c37f438cd_11",\n          "bbox_xyxy": [\n            89.0,\n            470.50000000000006,\n            142.0,\n            491.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_4088_JPG.rf.a7b91d4367ee32ed32413c2eeea94e61.jpg",\n        "size": 54009,\n        "compressed": 53870,\n        "offset": 596679483,\n        "crc32": "a53f5091",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_12c92f009295083c",\n      "relative_path": "images/IMG_2587_JPG.rf.009b2de96d0523244eb54d03d230d4d0.jpg",\n      "sha256": "bf1fcd76c39e333828aa3649d3dc64becaa5ae77c70ef734d29acbc073e23bf0",\n      "decoded_sha256": "24e2b43d547bbd649f76e5f5a1dc085ee8aad6ebc4d9230f37e696839f89daf6",\n      "dhash": "34f1f1696969f1b2",\n      "bytes": 32561,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2587_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_12c92f009295083c_0",\n          "bbox_xyxy": [\n            63.24999999999999,\n            0.7499999999999929,\n            501.75,\n            622.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_2587_JPG.rf.009b2de96d0523244eb54d03d230d4d0.jpg",\n        "size": 32561,\n        "compressed": 31619,\n        "offset": 567739440,\n        "crc32": "5a48cee4",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_12e4f6294165e4e9",\n      "relative_path": "images/IMG_2688_JPG.rf.be3167e800c764af60c4219d64dda4ad.jpg",\n      "sha256": "5387e335550084c32fa08bb3ad11e92b2c18da533ac3f44ffc0a8ffa8753ca4e",\n      "decoded_sha256": "3e7c93476c10a3cd2b69c10882bf3094e1df46c5d64dacc547b3bc6d54130952",\n      "dhash": "f1f1e9e8e8f0b273",\n      "bytes": 32352,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2688_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_12e4f6294165e4e9_0",\n          "bbox_xyxy": [\n            38.75,\n            0.2500000000000213,\n            568.25,\n            560.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_2688_JPG.rf.be3167e800c764af60c4219d64dda4ad.jpg",\n        "size": 32352,\n        "compressed": 32024,\n        "offset": 578282289,\n        "crc32": "75cd58e3",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_13829f12b8af7e78",\n      "relative_path": "images/IMG_2726_JPG.rf.e53776afcc13b37f25adcaa6516dbcd4.jpg",\n      "sha256": "57d1bf2acb4a060299cba4e161030c51844d941a184b0dae50500e502aab6c8c",\n      "decoded_sha256": "655d4cad460b7c49fd6fe16f4fcaedc018314c37f4786732dcaf5647af9f1db0",\n      "dhash": "b270787879b1e968",\n      "bytes": 27753,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2726_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_13829f12b8af7e78_0",\n          "bbox_xyxy": [\n            183.0,\n            5.500000000000007,\n            549.0,\n            598.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_2726_JPG.rf.e53776afcc13b37f25adcaa6516dbcd4.jpg",\n        "size": 27753,\n        "compressed": 27121,\n        "offset": 581656899,\n        "crc32": "66d99f1d",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_141821564fb1e6a2",\n      "relative_path": "images/IMG_2657_JPG.rf.b463eb17b36ef136c8d19496564a7a41.jpg",\n      "sha256": "0a7cef4d67968a0e517b7b4ebd876c3cd98be52b3c40fc07ba214ec89fc87853",\n      "decoded_sha256": "6d349a1bb0c6b5a33e1bf210f1e00abda394a05359ce443d4bd7b69829a802fb",\n      "dhash": "b231797979713034",\n      "bytes": 26304,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2657_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_141821564fb1e6a2_0",\n          "bbox_xyxy": [\n            165.0,\n            40.0,\n            520.0,\n            587.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_2657_JPG.rf.b463eb17b36ef136c8d19496564a7a41.jpg",\n        "size": 26304,\n        "compressed": 25707,\n        "offset": 574939527,\n        "crc32": "49da2ec4",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_144fe44a89602088",\n      "relative_path": "images/IMG_4167_JPG.rf.90405e7a0fc7546a343513992c84fc03.jpg",\n      "sha256": "bcead8d6cc0ca02de9c2aab1faa9b1ae3896e5c0d141cec8c937164927f89988",\n      "decoded_sha256": "89d70657a8d31edfa6a2a872e6f186ab63f6f464e01caad233b327f62052afb5",\n      "dhash": "396159f9e3f3f7db",\n      "bytes": 42098,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_4167_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_144fe44a89602088_0",\n          "bbox_xyxy": [\n            484.0,\n            488.0,\n            501.99999999999994,\n            533.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_144fe44a89602088_1",\n          "bbox_xyxy": [\n            442.25,\n            354.25,\n            464.75,\n            401.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_144fe44a89602088_2",\n          "bbox_xyxy": [\n            386.25,\n            423.75,\n            413.75,\n            468.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_144fe44a89602088_3",\n          "bbox_xyxy": [\n            286.75,\n            379.74999999999994,\n            307.25,\n            431.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_144fe44a89602088_4",\n          "bbox_xyxy": [\n            210.25,\n            341.75,\n            234.75,\n            386.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_144fe44a89602088_5",\n          "bbox_xyxy": [\n            161.25,\n            330.25,\n            204.75,\n            360.74999999999994\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_144fe44a89602088_6",\n          "bbox_xyxy": [\n            209.24999999999997,\n            250.99999999999997,\n            243.75,\n            298.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_144fe44a89602088_7",\n          "bbox_xyxy": [\n            169.74999999999997,\n            173.25,\n            188.25,\n            220.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_144fe44a89602088_8",\n          "bbox_xyxy": [\n            401.75000000000006,\n            194.25,\n            425.25,\n            234.74999999999997\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_144fe44a89602088_9",\n          "bbox_xyxy": [\n            477.5,\n            167.75,\n            499.5,\n            210.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_4167_JPG.rf.90405e7a0fc7546a343513992c84fc03.jpg",\n        "size": 42098,\n        "compressed": 41642,\n        "offset": 604828079,\n        "crc32": "475025a3",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_145282b28c60d708",\n      "relative_path": "images/IMG_2671_JPG.rf.16f91f1cd29ebd0a56d3549b21ffe4c3.jpg",\n      "sha256": "a03926bed526eb22396d5fbea640815db84b8d9dc2d43ce923e67fafdbcfa43a",\n      "decoded_sha256": "b8dd36debb11afa608066a8eb60749a0f4e09ac71272b224ea279e0b2ddce95b",\n      "dhash": "3071616961617110",\n      "bytes": 27249,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2671_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_145282b28c60d708_0",\n          "bbox_xyxy": [\n            169.49999999999994,\n            41.499999999999986,\n            506.5,\n            638.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_2671_JPG.rf.16f91f1cd29ebd0a56d3549b21ffe4c3.jpg",\n        "size": 27249,\n        "compressed": 26574,\n        "offset": 576302684,\n        "crc32": "efc232af",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_1497b30647d40dc4",\n      "relative_path": "images/IMG_2418_JPG.rf.078f8464405c905ef95664ddef594c4c.jpg",\n      "sha256": "e545073aaa50563cbafbba5fac20af9d8058866445e4ce11af8b25fbd2573db0",\n      "decoded_sha256": "437a26a26c453761f77df7c3f9b72fd8457d8e94866e8b0cd8fca741b8c4598e",\n      "dhash": "51b9793939387166",\n      "bytes": 28643,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2418_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_1497b30647d40dc4_0",\n          "bbox_xyxy": [\n            176.0,\n            3.249999999999993,\n            519.0,\n            627.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_2418_JPG.rf.078f8464405c905ef95664ddef594c4c.jpg",\n        "size": 28643,\n        "compressed": 27333,\n        "offset": 553845150,\n        "crc32": "71cb3884",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_14a361cd40feeee0",\n      "relative_path": "images/IMG_2311_JPG.rf.268747dbec5ab87d8b0ec26562f3d6ca.jpg",\n      "sha256": "0277b20d6fd3f31fd16ae84b34ece228c421b955bc4876d7b5017b8565a298a2",\n      "decoded_sha256": "1d0092bd6579d49e719025a815f556028ca8975bd5ebbf66807600fc1599d531",\n      "dhash": "d9e071e1e0e06061",\n      "bytes": 29211,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2311_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_14a361cd40feeee0_0",\n          "bbox_xyxy": [\n            109.99999999999997,\n            49.00000000000002,\n            536.0,\n            640.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_2311_JPG.rf.268747dbec5ab87d8b0ec26562f3d6ca.jpg",\n        "size": 29211,\n        "compressed": 28295,\n        "offset": 543736047,\n        "crc32": "6074d263",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_14aaabf38831b6aa",\n      "relative_path": "images/IMG_2275_JPG.rf.26707eb8145c0263d9ea8c1e4c5b82c2.jpg",\n      "sha256": "6a8f890b0ec2a398681fd405a575bfa7f943ac449f63b397c80c679953af9b17",\n      "decoded_sha256": "9516e43c9d8b0bfa025317bfa796a99006863ec5be3738447d5c5179a68f8989",\n      "dhash": "4070303978687132",\n      "bytes": 34014,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2275_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_14aaabf38831b6aa_0",\n          "bbox_xyxy": [\n            148.75,\n            55.49999999999997,\n            523.25,\n            637.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_2275_JPG.rf.26707eb8145c0263d9ea8c1e4c5b82c2.jpg",\n        "size": 34014,\n        "compressed": 32808,\n        "offset": 541316917,\n        "crc32": "7e42f951",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_15476420c63554ae",\n      "relative_path": "images/IMG_20260110_144820_jpg.rf.05f122a5cf6e65f7082188e630056d8b.jpg",\n      "sha256": "a1e131f154f5d73e8dfb5cf3ffd80e8f9a1260c3b9a24ed919036ccb7ad5f174",\n      "decoded_sha256": "177934799f525c28aeeca17c309c5fec03fc1baeb04c0e086073dc86352b7f5e",\n      "dhash": "c51595a663f94917",\n      "bytes": 60150,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_20260110_144820_jpg",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_15476420c63554ae_0",\n          "bbox_xyxy": [\n            125.49999999999999,\n            202.75,\n            168.49999999999997,\n            232.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_15476420c63554ae_1",\n          "bbox_xyxy": [\n            121.25,\n            179.0,\n            155.75,\n            216.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_15476420c63554ae_2",\n          "bbox_xyxy": [\n            217.75,\n            297.5,\n            270.25,\n            323.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_15476420c63554ae_3",\n          "bbox_xyxy": [\n            243.75,\n            315.25,\n            288.25,\n            340.74999999999994\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_15476420c63554ae_4",\n          "bbox_xyxy": [\n            199.5,\n            326.0,\n            246.50000000000003,\n            371.99999999999994\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_15476420c63554ae_5",\n          "bbox_xyxy": [\n            438.00000000000006,\n            175.75,\n            499.0,\n            220.25000000000003\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_15476420c63554ae_6",\n          "bbox_xyxy": [\n            439.25,\n            94.5,\n            500.75,\n            149.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_15476420c63554ae_7",\n          "bbox_xyxy": [\n            276.5,\n            236.5,\n            317.5,\n            278.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_15476420c63554ae_8",\n          "bbox_xyxy": [\n            538.0,\n            306.25,\n            584.0,\n            333.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_15476420c63554ae_9",\n          "bbox_xyxy": [\n            407.24999999999994,\n            454.74999999999994,\n            443.75,\n            481.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_20260110_144820_jpg.rf.05f122a5cf6e65f7082188e630056d8b.jpg",\n        "size": 60150,\n        "compressed": 59971,\n        "offset": 512074808,\n        "crc32": "c5ceb293",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_157a08df44c81776",\n      "relative_path": "images/IMG_0760_JPG.rf.aca7818ab5e1f7f5d46311b1315b3592.jpg",\n      "sha256": "0b94b0e7c3809f031dd1331fab092172c230e2e114b04a0d2125e3aa9afbe0a4",\n      "decoded_sha256": "26676dfd28842fd0f13f6af4320ba71e36ee333f1197d7907f4967334a666a0b",\n      "dhash": "188c9ed7270d4727",\n      "bytes": 45053,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_0760_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_157a08df44c81776_0",\n          "bbox_xyxy": [\n            82.25,\n            9.499999999999996,\n            134.75,\n            53.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_157a08df44c81776_1",\n          "bbox_xyxy": [\n            57.000000000000014,\n            96.0,\n            104.0,\n            154.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_157a08df44c81776_2",\n          "bbox_xyxy": [\n            27.750000000000004,\n            9.0,\n            75.25,\n            77.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_157a08df44c81776_3",\n          "bbox_xyxy": [\n            167.25,\n            228.5,\n            204.75,\n            307.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_157a08df44c81776_4",\n          "bbox_xyxy": [\n            228.0,\n            190.0,\n            283.0,\n            237.00000000000003\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_157a08df44c81776_5",\n          "bbox_xyxy": [\n            286.5,\n            211.75,\n            336.5,\n            275.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_157a08df44c81776_6",\n          "bbox_xyxy": [\n            319.5,\n            267.25,\n            369.5,\n            342.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_157a08df44c81776_7",\n          "bbox_xyxy": [\n            339.75,\n            173.49999999999997,\n            399.25000000000006,\n            220.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_157a08df44c81776_8",\n          "bbox_xyxy": [\n            383.5,\n            105.49999999999999,\n            436.5,\n            168.49999999999997\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_157a08df44c81776_9",\n          "bbox_xyxy": [\n            393.25,\n            35.75,\n            461.75,\n            73.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_157a08df44c81776_10",\n          "bbox_xyxy": [\n            339.0,\n            19.75,\n            403.00000000000006,\n            79.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_157a08df44c81776_11",\n          "bbox_xyxy": [\n            452.0,\n            72.0,\n            498.0,\n            152.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_157a08df44c81776_12",\n          "bbox_xyxy": [\n            434.5,\n            204.25,\n            495.5,\n            237.75000000000003\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_157a08df44c81776_13",\n          "bbox_xyxy": [\n            399.5,\n            223.25,\n            460.5,\n            281.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_157a08df44c81776_14",\n          "bbox_xyxy": [\n            513.5,\n            156.75,\n            566.5,\n            225.25000000000003\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_157a08df44c81776_15",\n          "bbox_xyxy": [\n            582.5,\n            215.99999999999997,\n            625.5,\n            293.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_157a08df44c81776_16",\n          "bbox_xyxy": [\n            526.5,\n            259.5,\n            588.5,\n            305.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_157a08df44c81776_17",\n          "bbox_xyxy": [\n            584.75,\n            275.0,\n            627.25,\n            380.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_157a08df44c81776_18",\n          "bbox_xyxy": [\n            415.99999999999994,\n            377.5,\n            480.0,\n            459.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_157a08df44c81776_19",\n          "bbox_xyxy": [\n            452.5,\n            468.5,\n            510.5,\n            535.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_157a08df44c81776_20",\n          "bbox_xyxy": [\n            533.0,\n            473.75,\n            600.0,\n            535.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_157a08df44c81776_21",\n          "bbox_xyxy": [\n            239.75,\n            316.75,\n            302.25,\n            383.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_157a08df44c81776_22",\n          "bbox_xyxy": [\n            122.5,\n            157.0,\n            170.5,\n            247.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_157a08df44c81776_23",\n          "bbox_xyxy": [\n            99.0,\n            190.75,\n            134.0,\n            273.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_157a08df44c81776_24",\n          "bbox_xyxy": [\n            30.0,\n            327.25,\n            76.0,\n            426.75000000000006\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_157a08df44c81776_25",\n          "bbox_xyxy": [\n            113.99999999999999,\n            501.50000000000006,\n            170.0,\n            592.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_157a08df44c81776_26",\n          "bbox_xyxy": [\n            0.2499999999999991,\n            518.25,\n            59.75,\n            585.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_0760_JPG.rf.aca7818ab5e1f7f5d46311b1315b3592.jpg",\n        "size": 45053,\n        "compressed": 44887,\n        "offset": 437064776,\n        "crc32": "5df328c9",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_158e0f3c206108e0",\n      "relative_path": "images/IMG_2012_JPG.rf.769b131ffc62dc0275856646241c7447.jpg",\n      "sha256": "4c0a70cbea8a0a4196891de978674de4065dc37e725407a0d24ad5d27333c6fb",\n      "decoded_sha256": "2215ec57ddf716126b38c69ab0bc5b7a18a37b324246d5a151366ff9e9e496c1",\n      "dhash": "0618707878687030",\n      "bytes": 25172,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2012_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_158e0f3c206108e0_0",\n          "bbox_xyxy": [\n            155.5,\n            86.24999999999997,\n            526.5,\n            639.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_2012_JPG.rf.769b131ffc62dc0275856646241c7447.jpg",\n        "size": 25172,\n        "compressed": 21080,\n        "offset": 509377068,\n        "crc32": "d256fc7e",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_15964be9985f0e50",\n      "relative_path": "images/IMG_1936_JPG.rf.4b8220ec3f5ff977bbc5867fd09a5c2a.jpg",\n      "sha256": "17e5de175163d43bfdc360456f07f0ff9bbaed583c6495839f4ddc46d02f2439",\n      "decoded_sha256": "5247f1fe4cda172e6a94a4aa5d7299b162212b7623eaa2168a145f72724f9b1c",\n      "dhash": "2c7031797871b95a",\n      "bytes": 27160,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_1936_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_15964be9985f0e50_0",\n          "bbox_xyxy": [\n            173.5,\n            40.99999999999998,\n            526.5,\n            582.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_1936_JPG.rf.4b8220ec3f5ff977bbc5867fd09a5c2a.jpg",\n        "size": 27160,\n        "compressed": 26619,\n        "offset": 504453572,\n        "crc32": "e2b1ac96",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_1596b1eced388dfa",\n      "relative_path": "images/IMG_2097_JPG.rf.452fbc356ff502ff63c3193d9cc23f8c.jpg",\n      "sha256": "aeae3f334aee90beb90aa2229f42dc8f9f1d0219d0a9a68d4f53c50d2a78899b",\n      "decoded_sha256": "937dbe3aa14c3a1659feadbb1ff6ef340c5893087eb40ef914edadc68effcaa0",\n      "dhash": "f03130b0f0703034",\n      "bytes": 28348,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2097_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_1596b1eced388dfa_0",\n          "bbox_xyxy": [\n            81.00000000000001,\n            12.999999999999972,\n            520.0,\n            640.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_2097_JPG.rf.452fbc356ff502ff63c3193d9cc23f8c.jpg",\n        "size": 28348,\n        "compressed": 25416,\n        "offset": 526666379,\n        "crc32": "b7229648",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_15dedbc773d10a24",\n      "relative_path": "images/IMG_1200_JPG.rf.2691a37156d39a269d7abc17027ee5e4.jpg",\n      "sha256": "077b436f97eef04645299ad71a0e191a4337227740a9d81a07a1fb5c5c3b9c26",\n      "decoded_sha256": "dd6186447ece4ee5a805beb359121d2a65375f8e61b33dd4285e829899d25430",\n      "dhash": "4d4e5d132a9aa849",\n      "bytes": 45504,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_1200_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_15dedbc773d10a24_0",\n          "bbox_xyxy": [\n            546.25,\n            265.75,\n            581.75,\n            324.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_15dedbc773d10a24_1",\n          "bbox_xyxy": [\n            277.75,\n            386.5,\n            309.25,\n            446.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_15dedbc773d10a24_2",\n          "bbox_xyxy": [\n            401.75000000000006,\n            443.25,\n            447.25,\n            481.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_15dedbc773d10a24_3",\n          "bbox_xyxy": [\n            333.24999999999994,\n            112.5,\n            372.75,\n            178.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_15dedbc773d10a24_4",\n          "bbox_xyxy": [\n            231.25,\n            484.25000000000006,\n            274.75,\n            532.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_15dedbc773d10a24_5",\n          "bbox_xyxy": [\n            107.00000000000001,\n            449.25,\n            139.0,\n            516.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_15dedbc773d10a24_6",\n          "bbox_xyxy": [\n            352.75,\n            343.75,\n            388.24999999999994,\n            408.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_15dedbc773d10a24_7",\n          "bbox_xyxy": [\n            170.75000000000003,\n            208.25,\n            211.25,\n            260.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_15dedbc773d10a24_8",\n          "bbox_xyxy": [\n            147.0,\n            0.0,\n            195.0,\n            37.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_15dedbc773d10a24_9",\n          "bbox_xyxy": [\n            161.5,\n            372.75,\n            207.5,\n            429.25000000000006\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_15dedbc773d10a24_10",\n          "bbox_xyxy": [\n            221.50000000000003,\n            120.75,\n            264.5,\n            169.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_15dedbc773d10a24_11",\n          "bbox_xyxy": [\n            484.75,\n            330.25,\n            515.25,\n            384.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_15dedbc773d10a24_12",\n          "bbox_xyxy": [\n            25.75,\n            141.24999999999997,\n            63.25,\n            206.74999999999997\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_15dedbc773d10a24_13",\n          "bbox_xyxy": [\n            160.75,\n            577.0,\n            204.25,\n            620.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_15dedbc773d10a24_14",\n          "bbox_xyxy": [\n            114.75,\n            336.75,\n            157.25,\n            379.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_15dedbc773d10a24_15",\n          "bbox_xyxy": [\n            56.25,\n            360.5,\n            99.75,\n            405.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_15dedbc773d10a24_16",\n          "bbox_xyxy": [\n            356.75,\n            505.25,\n            393.25,\n            566.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_15dedbc773d10a24_17",\n          "bbox_xyxy": [\n            3.2500000000000018,\n            58.00000000000001,\n            56.75,\n            103.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_15dedbc773d10a24_18",\n          "bbox_xyxy": [\n            152.75,\n            100.24999999999999,\n            185.25,\n            166.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_15dedbc773d10a24_19",\n          "bbox_xyxy": [\n            457.75000000000006,\n            521.0,\n            496.25,\n            582.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_15dedbc773d10a24_20",\n          "bbox_xyxy": [\n            606.0,\n            516.0,\n            640.0,\n            560.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_15dedbc773d10a24_21",\n          "bbox_xyxy": [\n            69.0,\n            263.75,\n            113.0,\n            302.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_15dedbc773d10a24_22",\n          "bbox_xyxy": [\n            288.5,\n            235.99999999999997,\n            318.5,\n            307.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_15dedbc773d10a24_23",\n          "bbox_xyxy": [\n            6.249999999999998,\n            254.5,\n            39.75,\n            316.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_15dedbc773d10a24_24",\n          "bbox_xyxy": [\n            465.0,\n            0.0,\n            513.0,\n            23.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_15dedbc773d10a24_25",\n          "bbox_xyxy": [\n            513.5,\n            463.0,\n            554.5,\n            502.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_15dedbc773d10a24_26",\n          "bbox_xyxy": [\n            553.75,\n            615.25,\n            605.25,\n            639.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_15dedbc773d10a24_27",\n          "bbox_xyxy": [\n            536.25,\n            72.00000000000001,\n            569.75,\n            154.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_15dedbc773d10a24_28",\n          "bbox_xyxy": [\n            341.75000000000006,\n            219.5,\n            382.25,\n            286.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_15dedbc773d10a24_29",\n          "bbox_xyxy": [\n            596.25,\n            213.5,\n            639.75,\n            252.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_15dedbc773d10a24_30",\n          "bbox_xyxy": [\n            591.75,\n            0.0,\n            632.25,\n            17.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_15dedbc773d10a24_31",\n          "bbox_xyxy": [\n            619.0,\n            314.0,\n            640.0,\n            374.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_15dedbc773d10a24_32",\n          "bbox_xyxy": [\n            394.0,\n            567.25,\n            438.00000000000006,\n            611.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_15dedbc773d10a24_33",\n          "bbox_xyxy": [\n            245.0,\n            0.0,\n            289.0,\n            39.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_15dedbc773d10a24_34",\n          "bbox_xyxy": [\n            1.9999999999999973,\n            0.0,\n            49.0,\n            24.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_15dedbc773d10a24_35",\n          "bbox_xyxy": [\n            398.25,\n            41.99999999999999,\n            440.75,\n            99.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_15dedbc773d10a24_36",\n          "bbox_xyxy": [\n            119.0,\n            508.5,\n            148.0,\n            584.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_15dedbc773d10a24_37",\n          "bbox_xyxy": [\n            462.5,\n            160.75,\n            497.5,\n            217.24999999999997\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_15dedbc773d10a24_38",\n          "bbox_xyxy": [\n            447.75000000000006,\n            36.49999999999999,\n            486.25,\n            95.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_15dedbc773d10a24_39",\n          "bbox_xyxy": [\n            41.99999999999999,\n            610.0,\n            75.0,\n            640.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_15dedbc773d10a24_40",\n          "bbox_xyxy": [\n            0.0,\n            329.5,\n            23.0,\n            390.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_15dedbc773d10a24_41",\n          "bbox_xyxy": [\n            12.249999999999998,\n            496.5,\n            45.75,\n            554.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_15dedbc773d10a24_42",\n          "bbox_xyxy": [\n            424.75,\n            288.5,\n            462.25,\n            332.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_15dedbc773d10a24_43",\n          "bbox_xyxy": [\n            231.25,\n            335.99999999999994,\n            272.75,\n            395.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_1200_JPG.rf.2691a37156d39a269d7abc17027ee5e4.jpg",\n        "size": 45504,\n        "compressed": 44840,\n        "offset": 492358769,\n        "crc32": "a7596395",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_16158f8bf85b2702",\n      "relative_path": "images/IMG_2105_JPG.rf.b478d1bc1ad015f5e3bb16b8bf1cf499.jpg",\n      "sha256": "138e7ddab96dfb1253c6176d02728e6a5aba8569de1532ff2f393c5fc119db65",\n      "decoded_sha256": "ccf3627cac1dbe962214f481989118fb0d69e992efc2a7e4759682892bc961b6",\n      "dhash": "5430706969717132",\n      "bytes": 26179,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2105_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_16158f8bf85b2702_0",\n          "bbox_xyxy": [\n            169.74999999999997,\n            16.000000000000014,\n            541.25,\n            589.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_2105_JPG.rf.b478d1bc1ad015f5e3bb16b8bf1cf499.jpg",\n        "size": 26179,\n        "compressed": 24451,\n        "offset": 527433580,\n        "crc32": "50f55fea",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_16e3e22e755ef103",\n      "relative_path": "images/IMG_2076_JPG.rf.0cda08ed03d9c822f46323963c821188.jpg",\n      "sha256": "e2e35fd1651df9e913917ab7c7c38587b097dd3f13361ef072317c48980b909a",\n      "decoded_sha256": "6c6c538f78418e87508ce83d066879e707edd034d575ca9f6a82317437762ef9",\n      "dhash": "1030706870783030",\n      "bytes": 24804,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2076_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_16e3e22e755ef103_0",\n          "bbox_xyxy": [\n            158.75,\n            80.99999999999997,\n            483.25,\n            617.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_2076_JPG.rf.0cda08ed03d9c822f46323963c821188.jpg",\n        "size": 24804,\n        "compressed": 20628,\n        "offset": 525031822,\n        "crc32": "157d5f06",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_172a5c20023dc5cf",\n      "relative_path": "images/IMG_1957_JPG.rf.f814d4e9cbb416903bd4c685c94d273d.jpg",\n      "sha256": "8d2b6552137b48d3c9072f46cea4b53d5bf31bd11b1407c41f5a763c7face223",\n      "decoded_sha256": "3b22feecb212c916eab535c7effcb816cc809d78edf795d0f722e42892bd5041",\n      "dhash": "2c30706070707030",\n      "bytes": 26447,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_1957_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_172a5c20023dc5cf_0",\n          "bbox_xyxy": [\n            142.75,\n            17.749999999999986,\n            481.25,\n            602.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_1957_JPG.rf.f814d4e9cbb416903bd4c685c94d273d.jpg",\n        "size": 26447,\n        "compressed": 22415,\n        "offset": 505537629,\n        "crc32": "2f67086a",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_1745a1358847c012",\n      "relative_path": "images/IMG_20260110_145547_jpg.rf.32166fc9cdd2f756d3a59bcd3c82bf61.jpg",\n      "sha256": "27ef1bef86752ecb932f05bb8010b5e03e3ce87c066bd34b9d7921c9653bc58c",\n      "decoded_sha256": "34e90b1281c95a5a7ba703e8318cc1a513c3982cf21739f8337da81bf79b6ec4",\n      "dhash": "211d3bba8b4f97ab",\n      "bytes": 32740,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_20260110_145547_jpg",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_1745a1358847c012_0",\n          "bbox_xyxy": [\n            152.0,\n            385.0,\n            197.0,\n            462.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1745a1358847c012_1",\n          "bbox_xyxy": [\n            239.50000000000003,\n            14.75,\n            277.5,\n            75.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1745a1358847c012_2",\n          "bbox_xyxy": [\n            430.50000000000006,\n            410.50000000000006,\n            471.5,\n            491.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1745a1358847c012_3",\n          "bbox_xyxy": [\n            510.0,\n            324.25000000000006,\n            561.0,\n            392.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1745a1358847c012_4",\n          "bbox_xyxy": [\n            273.0,\n            242.0,\n            345.0,\n            287.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1745a1358847c012_5",\n          "bbox_xyxy": [\n            460.50000000000006,\n            120.0,\n            488.5,\n            176.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1745a1358847c012_6",\n          "bbox_xyxy": [\n            516.0,\n            217.25,\n            547.0,\n            272.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1745a1358847c012_7",\n          "bbox_xyxy": [\n            214.00000000000003,\n            280.75,\n            242.0,\n            337.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1745a1358847c012_8",\n          "bbox_xyxy": [\n            64.50000000000001,\n            483.75,\n            96.5,\n            534.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1745a1358847c012_9",\n          "bbox_xyxy": [\n            523.5,\n            427.0,\n            551.5,\n            475.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1745a1358847c012_10",\n          "bbox_xyxy": [\n            531.5,\n            581.0,\n            560.5,\n            621.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1745a1358847c012_11",\n          "bbox_xyxy": [\n            229.74999999999997,\n            597.0,\n            279.25,\n            631.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_20260110_145547_jpg.rf.32166fc9cdd2f756d3a59bcd3c82bf61.jpg",\n        "size": 32740,\n        "compressed": 31532,\n        "offset": 513698898,\n        "crc32": "0b1351ca",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_174fdcca5f64e62c",\n      "relative_path": "images/IMG_1996_JPG.rf.739a72d5d9450ba28d8f07d9eadee204.jpg",\n      "sha256": "5230a080a0ebd954bec831d971e7306bbc474a5a72c5b8a94a771060f795ff70",\n      "decoded_sha256": "dd2907410bc1cd7c9348fb68cf3125327cda5c51590ff54fe41b6375640f2b5d",\n      "dhash": "1030387878705044",\n      "bytes": 21749,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_1996_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_174fdcca5f64e62c_0",\n          "bbox_xyxy": [\n            154.25,\n            17.5,\n            525.75,\n            591.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_1996_JPG.rf.739a72d5d9450ba28d8f07d9eadee204.jpg",\n        "size": 21749,\n        "compressed": 17311,\n        "offset": 508323616,\n        "crc32": "5154e31f",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_177427b6fd5a3fd2",\n      "relative_path": "images/6095710224539913297_jpg.rf.0ca131d56cbaeb4b702913c1ef329568.jpg",\n      "sha256": "be12e8479dbb463372567af545b44475b4fb8bc16fd0da2ac7edfe7377e64827",\n      "decoded_sha256": "cf467cfb05dcf25519c53ed76ae0ae83c0a4f22e5172c6c04aca940449dd833a",\n      "dhash": "896e78f9b478f8f3",\n      "bytes": 138464,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "6095710224539913297_jpg",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_177427b6fd5a3fd2_0",\n          "bbox_xyxy": [\n            613.0,\n            471.25,\n            640.0,\n            497.75000000000006\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_177427b6fd5a3fd2_1",\n          "bbox_xyxy": [\n            94.75000000000001,\n            554.0,\n            126.25,\n            575.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_177427b6fd5a3fd2_2",\n          "bbox_xyxy": [\n            363.24999999999994,\n            295.0,\n            387.75,\n            316.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_177427b6fd5a3fd2_3",\n          "bbox_xyxy": [\n            201.25,\n            481.25,\n            229.75,\n            506.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_177427b6fd5a3fd2_4",\n          "bbox_xyxy": [\n            509.25000000000006,\n            239.75,\n            537.75,\n            261.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_177427b6fd5a3fd2_5",\n          "bbox_xyxy": [\n            533.25,\n            221.25,\n            546.75,\n            243.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_177427b6fd5a3fd2_6",\n          "bbox_xyxy": [\n            21.500000000000004,\n            227.24999999999997,\n            47.5,\n            245.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_177427b6fd5a3fd2_7",\n          "bbox_xyxy": [\n            71.75,\n            144.0,\n            91.25,\n            164.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_177427b6fd5a3fd2_8",\n          "bbox_xyxy": [\n            88.0,\n            573.5,\n            122.0,\n            594.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_177427b6fd5a3fd2_9",\n          "bbox_xyxy": [\n            334.74999999999994,\n            605.0,\n            366.25,\n            622.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_177427b6fd5a3fd2_10",\n          "bbox_xyxy": [\n            383.75,\n            526.5,\n            400.25000000000006,\n            557.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_177427b6fd5a3fd2_11",\n          "bbox_xyxy": [\n            442.0,\n            151.25,\n            472.0,\n            169.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_177427b6fd5a3fd2_12",\n          "bbox_xyxy": [\n            526.25,\n            112.75,\n            561.75,\n            133.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_177427b6fd5a3fd2_13",\n          "bbox_xyxy": [\n            563.5,\n            24.0,\n            604.5,\n            51.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_177427b6fd5a3fd2_14",\n          "bbox_xyxy": [\n            394.75,\n            530.0,\n            415.25,\n            562.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_177427b6fd5a3fd2_15",\n          "bbox_xyxy": [\n            390.25000000000006,\n            260.0,\n            423.75,\n            285.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_177427b6fd5a3fd2_16",\n          "bbox_xyxy": [\n            405.25,\n            0.2500000000000002,\n            435.74999999999994,\n            15.750000000000002\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_177427b6fd5a3fd2_17",\n          "bbox_xyxy": [\n            516.0,\n            459.00000000000006,\n            548.0,\n            485.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_177427b6fd5a3fd2_18",\n          "bbox_xyxy": [\n            370.74999999999994,\n            362.25,\n            400.25,\n            381.75000000000006\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_177427b6fd5a3fd2_19",\n          "bbox_xyxy": [\n            438.5,\n            293.25,\n            469.49999999999994,\n            312.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_177427b6fd5a3fd2_20",\n          "bbox_xyxy": [\n            622.0,\n            336.0,\n            640.0,\n            354.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_177427b6fd5a3fd2_21",\n          "bbox_xyxy": [\n            502.5,\n            66.75,\n            539.5,\n            95.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/6095710224539913297_jpg.rf.0ca131d56cbaeb4b702913c1ef329568.jpg",\n        "size": 138464,\n        "compressed": 138364,\n        "offset": 358960149,\n        "crc32": "28c9a331",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_1780e5633611aec2",\n      "relative_path": "images/IMG_1194_JPG.rf.fb2db0931cd5ad4ee1a37ee33d94d136.jpg",\n      "sha256": "2247adb52dc95385500d2138bd429dad3ec55de72e00d94bd4c030d94f752b68",\n      "decoded_sha256": "6a3e87d9e9531bde7831770367e3e4eab658cbc5876b5e68589dc29cc9bf9a42",\n      "dhash": "ab8daa09944d6d4f",\n      "bytes": 47479,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_1194_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_1780e5633611aec2_0",\n          "bbox_xyxy": [\n            168.25,\n            87.75,\n            215.75,\n            132.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1780e5633611aec2_1",\n          "bbox_xyxy": [\n            501.25,\n            153.75000000000003,\n            543.75,\n            198.25000000000003\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1780e5633611aec2_2",\n          "bbox_xyxy": [\n            91.0,\n            140.75,\n            127.00000000000001,\n            192.24999999999997\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1780e5633611aec2_3",\n          "bbox_xyxy": [\n            273.5,\n            435.75,\n            305.5,\n            496.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1780e5633611aec2_4",\n          "bbox_xyxy": [\n            511.5,\n            38.75000000000001,\n            544.5,\n            89.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1780e5633611aec2_5",\n          "bbox_xyxy": [\n            180.0,\n            368.75,\n            217.99999999999997,\n            434.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1780e5633611aec2_6",\n          "bbox_xyxy": [\n            217.5,\n            203.25,\n            261.5,\n            244.74999999999997\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1780e5633611aec2_7",\n          "bbox_xyxy": [\n            437.0,\n            246.50000000000003,\n            465.0,\n            310.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1780e5633611aec2_8",\n          "bbox_xyxy": [\n            575.75,\n            78.25000000000001,\n            609.25,\n            147.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1780e5633611aec2_9",\n          "bbox_xyxy": [\n            352.75,\n            181.75,\n            402.25,\n            225.25000000000003\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1780e5633611aec2_10",\n          "bbox_xyxy": [\n            574.5,\n            0.0,\n            607.5,\n            56.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1780e5633611aec2_11",\n          "bbox_xyxy": [\n            194.75,\n            513.0,\n            235.25,\n            565.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1780e5633611aec2_12",\n          "bbox_xyxy": [\n            429.25,\n            72.75,\n            455.75,\n            146.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1780e5633611aec2_13",\n          "bbox_xyxy": [\n            285.5,\n            149.75,\n            327.5,\n            191.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1780e5633611aec2_14",\n          "bbox_xyxy": [\n            349.75,\n            29.25,\n            382.25,\n            81.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1780e5633611aec2_15",\n          "bbox_xyxy": [\n            83.99999999999999,\n            0.2500000000000002,\n            133.0,\n            15.750000000000002\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1780e5633611aec2_16",\n          "bbox_xyxy": [\n            159.75,\n            244.50000000000003,\n            206.25,\n            282.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1780e5633611aec2_17",\n          "bbox_xyxy": [\n            93.5,\n            66.5,\n            138.5,\n            106.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1780e5633611aec2_18",\n          "bbox_xyxy": [\n            0.0,\n            245.99999999999997,\n            28.0,\n            292.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1780e5633611aec2_19",\n          "bbox_xyxy": [\n            81.25,\n            273.0,\n            115.75,\n            330.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1780e5633611aec2_20",\n          "bbox_xyxy": [\n            21.750000000000004,\n            123.5,\n            66.25,\n            159.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1780e5633611aec2_21",\n          "bbox_xyxy": [\n            372.25,\n            271.5,\n            410.74999999999994,\n            317.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1780e5633611aec2_22",\n          "bbox_xyxy": [\n            248.0,\n            39.25,\n            278.0,\n            101.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1780e5633611aec2_23",\n          "bbox_xyxy": [\n            311.0,\n            309.0,\n            356.99999999999994,\n            361.99999999999994\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1780e5633611aec2_24",\n          "bbox_xyxy": [\n            619.0,\n            198.75,\n            640.0,\n            264.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1780e5633611aec2_25",\n          "bbox_xyxy": [\n            408.00000000000006,\n            0.0,\n            436.0,\n            34.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1780e5633611aec2_26",\n          "bbox_xyxy": [\n            608.0,\n            0.0,\n            640.0,\n            22.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1780e5633611aec2_27",\n          "bbox_xyxy": [\n            255.5,\n            294.5,\n            295.5,\n            337.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1780e5633611aec2_28",\n          "bbox_xyxy": [\n            393.25,\n            450.0,\n            435.75,\n            485.99999999999994\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1780e5633611aec2_29",\n          "bbox_xyxy": [\n            534.75,\n            297.75,\n            556.25,\n            362.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1780e5633611aec2_30",\n          "bbox_xyxy": [\n            0.0,\n            1.0000000000000009,\n            15.0,\n            61.99999999999999\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_1194_JPG.rf.fb2db0931cd5ad4ee1a37ee33d94d136.jpg",\n        "size": 47479,\n        "compressed": 47242,\n        "offset": 491576082,\n        "crc32": "05669f5b",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_17a22401b20a81c8",\n      "relative_path": "images/IMG_2203_JPG.rf.977434be92caeae31a65c0af408d1674.jpg",\n      "sha256": "9563ea5faaee5c00e9b35c97ce92e837338c97d23312341363ca3c32fb6d9bd8",\n      "decoded_sha256": "0aafc48e66e5cb7c1180ef4d78343b72672aeac7ee90aba59f53e63f9ba82567",\n      "dhash": "c0e060f0e0e0e070",\n      "bytes": 24089,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2203_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_17a22401b20a81c8_0",\n          "bbox_xyxy": [\n            56.000000000000014,\n            76.75,\n            524.0,\n            635.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_2203_JPG.rf.977434be92caeae31a65c0af408d1674.jpg",\n        "size": 24089,\n        "compressed": 20263,\n        "offset": 534840599,\n        "crc32": "1f628141",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_17d97dbabc914fd3",\n      "relative_path": "images/IMG_0822_JPG.rf.039b5a404affdf358a96d65973403e2a.jpg",\n      "sha256": "1bd72b58d298535081b3ef1258d577a7c0cc3d3eb54c21144e495fa07790b5dd",\n      "decoded_sha256": "6318c30bc3c8e7341b9fb19b36c124cca9dc5bf1aefe73215098a7e416dab43d",\n      "dhash": "1f36b553139caeaf",\n      "bytes": 50149,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_0822_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_17d97dbabc914fd3_0",\n          "bbox_xyxy": [\n            402.5,\n            524.0,\n            425.5,\n            593.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17d97dbabc914fd3_1",\n          "bbox_xyxy": [\n            424.25000000000006,\n            539.5,\n            464.75,\n            602.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17d97dbabc914fd3_2",\n          "bbox_xyxy": [\n            427.75,\n            518.5,\n            477.25,\n            570.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17d97dbabc914fd3_3",\n          "bbox_xyxy": [\n            386.25,\n            450.5,\n            439.74999999999994,\n            487.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17d97dbabc914fd3_4",\n          "bbox_xyxy": [\n            494.5,\n            529.25,\n            544.5,\n            605.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17d97dbabc914fd3_5",\n          "bbox_xyxy": [\n            529.25,\n            472.5,\n            589.75,\n            523.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17d97dbabc914fd3_6",\n          "bbox_xyxy": [\n            549.5,\n            567.25,\n            614.5,\n            612.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17d97dbabc914fd3_7",\n          "bbox_xyxy": [\n            555.0,\n            362.5,\n            602.0,\n            395.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17d97dbabc914fd3_8",\n          "bbox_xyxy": [\n            437.24999999999994,\n            323.5,\n            493.75,\n            386.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17d97dbabc914fd3_9",\n          "bbox_xyxy": [\n            514.25,\n            300.75,\n            566.75,\n            373.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17d97dbabc914fd3_10",\n          "bbox_xyxy": [\n            599.0,\n            306.0,\n            640.0,\n            359.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17d97dbabc914fd3_11",\n          "bbox_xyxy": [\n            579.5,\n            186.5,\n            632.5,\n            236.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17d97dbabc914fd3_12",\n          "bbox_xyxy": [\n            564.5,\n            159.5,\n            597.5,\n            229.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17d97dbabc914fd3_13",\n          "bbox_xyxy": [\n            609.0,\n            132.25,\n            640.0,\n            192.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17d97dbabc914fd3_14",\n          "bbox_xyxy": [\n            534.25,\n            71.24999999999999,\n            582.75,\n            132.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17d97dbabc914fd3_15",\n          "bbox_xyxy": [\n            522.25,\n            99.24999999999999,\n            547.75,\n            179.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17d97dbabc914fd3_16",\n          "bbox_xyxy": [\n            478.75,\n            90.25,\n            532.25,\n            145.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17d97dbabc914fd3_17",\n          "bbox_xyxy": [\n            466.75,\n            137.5,\n            513.25,\n            174.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17d97dbabc914fd3_18",\n          "bbox_xyxy": [\n            443.5,\n            179.25,\n            498.5,\n            224.74999999999997\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17d97dbabc914fd3_19",\n          "bbox_xyxy": [\n            378.0,\n            212.25,\n            443.0,\n            257.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17d97dbabc914fd3_20",\n          "bbox_xyxy": [\n            352.0,\n            79.75,\n            392.0,\n            171.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17d97dbabc914fd3_21",\n          "bbox_xyxy": [\n            269.25,\n            112.25000000000001,\n            330.75,\n            173.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17d97dbabc914fd3_22",\n          "bbox_xyxy": [\n            231.0,\n            190.75000000000003,\n            294.0,\n            231.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17d97dbabc914fd3_23",\n          "bbox_xyxy": [\n            268.25,\n            221.5,\n            320.75,\n            253.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17d97dbabc914fd3_24",\n          "bbox_xyxy": [\n            308.75000000000006,\n            246.75,\n            360.25000000000006,\n            320.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17d97dbabc914fd3_25",\n          "bbox_xyxy": [\n            306.5,\n            225.75,\n            370.50000000000006,\n            264.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17d97dbabc914fd3_26",\n          "bbox_xyxy": [\n            306.5,\n            371.0,\n            363.5,\n            416.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17d97dbabc914fd3_27",\n          "bbox_xyxy": [\n            346.75,\n            393.0,\n            399.25,\n            435.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17d97dbabc914fd3_28",\n          "bbox_xyxy": [\n            362.5,\n            353.00000000000006,\n            420.5,\n            411.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17d97dbabc914fd3_29",\n          "bbox_xyxy": [\n            284.75,\n            316.5,\n            335.25,\n            385.50000000000006\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17d97dbabc914fd3_30",\n          "bbox_xyxy": [\n            191.74999999999997,\n            250.75,\n            246.25,\n            300.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17d97dbabc914fd3_31",\n          "bbox_xyxy": [\n            152.0,\n            279.5,\n            204.00000000000003,\n            318.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17d97dbabc914fd3_32",\n          "bbox_xyxy": [\n            128.5,\n            312.5,\n            166.5,\n            397.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17d97dbabc914fd3_33",\n          "bbox_xyxy": [\n            125.0,\n            380.0,\n            170.0,\n            440.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17d97dbabc914fd3_34",\n          "bbox_xyxy": [\n            74.0,\n            387.75,\n            123.0,\n            453.24999999999994\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17d97dbabc914fd3_35",\n          "bbox_xyxy": [\n            16.750000000000004,\n            376.25,\n            67.25,\n            434.74999999999994\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17d97dbabc914fd3_36",\n          "bbox_xyxy": [\n            0.0,\n            315.25,\n            63.0,\n            362.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17d97dbabc914fd3_37",\n          "bbox_xyxy": [\n            0.2499999999999991,\n            245.75,\n            49.75,\n            318.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17d97dbabc914fd3_38",\n          "bbox_xyxy": [\n            84.75,\n            200.49999999999997,\n            145.25,\n            242.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17d97dbabc914fd3_39",\n          "bbox_xyxy": [\n            70.0,\n            587.0,\n            138.0,\n            640.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17d97dbabc914fd3_40",\n          "bbox_xyxy": [\n            78.5,\n            495.25,\n            116.5,\n            584.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17d97dbabc914fd3_41",\n          "bbox_xyxy": [\n            58.5,\n            439.0,\n            90.5,\n            503.00000000000006\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17d97dbabc914fd3_42",\n          "bbox_xyxy": [\n            100.49999999999999,\n            534.75,\n            136.5,\n            614.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17d97dbabc914fd3_43",\n          "bbox_xyxy": [\n            203.0,\n            612.0,\n            254.00000000000003,\n            640.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17d97dbabc914fd3_44",\n          "bbox_xyxy": [\n            184.00000000000003,\n            498.0,\n            252.0,\n            538.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17d97dbabc914fd3_45",\n          "bbox_xyxy": [\n            301.0,\n            460.0,\n            338.0,\n            535.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17d97dbabc914fd3_46",\n          "bbox_xyxy": [\n            475.50000000000006,\n            394.75,\n            528.5,\n            474.25000000000006\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17d97dbabc914fd3_47",\n          "bbox_xyxy": [\n            441.25,\n            400.0,\n            474.74999999999994,\n            480.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_0822_JPG.rf.039b5a404affdf358a96d65973403e2a.jpg",\n        "size": 50149,\n        "compressed": 49953,\n        "offset": 446058148,\n        "crc32": "7a05fa3d",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_17f42b353a5146c4",\n      "relative_path": "images/6100321344378571869_jpg.rf.a00cb974b5b8be3b68c6f081f6d0e9d7.jpg",\n      "sha256": "38f28c292241ba430b81935cd9fc2f267900c486b640dec722bac57247451c06",\n      "decoded_sha256": "a809ccc460c12ac403627f8e791caf80d6342be2720b71341ce75dc6c2e4c079",\n      "dhash": "b9552746168fcbcf",\n      "bytes": 132697,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "6100321344378571869_jpg",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_17f42b353a5146c4_0",\n          "bbox_xyxy": [\n            336.0,\n            161.5,\n            366.0,\n            182.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17f42b353a5146c4_1",\n          "bbox_xyxy": [\n            250.0,\n            158.74999999999997,\n            276.0,\n            184.24999999999997\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17f42b353a5146c4_2",\n          "bbox_xyxy": [\n            549.25,\n            413.5,\n            576.75,\n            440.50000000000006\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17f42b353a5146c4_3",\n          "bbox_xyxy": [\n            56.5,\n            452.25,\n            99.5,\n            474.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17f42b353a5146c4_4",\n          "bbox_xyxy": [\n            86.5,\n            166.50000000000003,\n            105.49999999999999,\n            195.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17f42b353a5146c4_5",\n          "bbox_xyxy": [\n            35.5,\n            427.75,\n            50.5,\n            469.25000000000006\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17f42b353a5146c4_6",\n          "bbox_xyxy": [\n            35.75,\n            597.0,\n            75.25,\n            617.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17f42b353a5146c4_7",\n          "bbox_xyxy": [\n            419.5,\n            57.5,\n            440.5,\n            87.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17f42b353a5146c4_8",\n          "bbox_xyxy": [\n            372.25,\n            43.5,\n            401.75000000000006,\n            72.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17f42b353a5146c4_9",\n          "bbox_xyxy": [\n            145.75,\n            163.25000000000003,\n            176.25,\n            183.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17f42b353a5146c4_10",\n          "bbox_xyxy": [\n            418.00000000000006,\n            459.00000000000006,\n            449.0,\n            480.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17f42b353a5146c4_11",\n          "bbox_xyxy": [\n            215.5,\n            272.75,\n            243.49999999999997,\n            290.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17f42b353a5146c4_12",\n          "bbox_xyxy": [\n            186.5,\n            435.74999999999994,\n            212.5,\n            460.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17f42b353a5146c4_13",\n          "bbox_xyxy": [\n            178.25000000000003,\n            111.75,\n            198.75,\n            141.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17f42b353a5146c4_14",\n          "bbox_xyxy": [\n            500.5,\n            602.0,\n            530.5,\n            624.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17f42b353a5146c4_15",\n          "bbox_xyxy": [\n            140.25,\n            258.25,\n            171.75,\n            277.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17f42b353a5146c4_16",\n          "bbox_xyxy": [\n            574.5,\n            238.5,\n            603.5,\n            261.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17f42b353a5146c4_17",\n          "bbox_xyxy": [\n            254.24999999999997,\n            31.25,\n            283.75,\n            52.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17f42b353a5146c4_18",\n          "bbox_xyxy": [\n            461.75,\n            9.0,\n            494.25,\n            27.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17f42b353a5146c4_19",\n          "bbox_xyxy": [\n            36.25000000000001,\n            225.25,\n            61.75000000000001,\n            244.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17f42b353a5146c4_20",\n          "bbox_xyxy": [\n            484.75,\n            96.49999999999999,\n            513.25,\n            117.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17f42b353a5146c4_21",\n          "bbox_xyxy": [\n            201.74999999999997,\n            252.0,\n            231.25,\n            271.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17f42b353a5146c4_22",\n          "bbox_xyxy": [\n            545.0,\n            352.75,\n            577.0,\n            372.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17f42b353a5146c4_23",\n          "bbox_xyxy": [\n            327.24999999999994,\n            67.75,\n            358.75,\n            87.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17f42b353a5146c4_24",\n          "bbox_xyxy": [\n            399.74999999999994,\n            20.0,\n            426.25,\n            49.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17f42b353a5146c4_25",\n          "bbox_xyxy": [\n            376.25,\n            3.75,\n            397.75000000000006,\n            28.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17f42b353a5146c4_26",\n          "bbox_xyxy": [\n            159.5,\n            310.5,\n            189.5,\n            335.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17f42b353a5146c4_27",\n          "bbox_xyxy": [\n            409.5,\n            606.5,\n            442.5,\n            624.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17f42b353a5146c4_28",\n          "bbox_xyxy": [\n            469.49999999999994,\n            226.0,\n            501.5,\n            244.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17f42b353a5146c4_29",\n          "bbox_xyxy": [\n            92.0,\n            584.0,\n            120.0,\n            603.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17f42b353a5146c4_30",\n          "bbox_xyxy": [\n            474.25000000000006,\n            397.0,\n            497.75,\n            427.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17f42b353a5146c4_31",\n          "bbox_xyxy": [\n            410.50000000000006,\n            381.25,\n            426.5,\n            406.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17f42b353a5146c4_32",\n          "bbox_xyxy": [\n            529.0,\n            220.0,\n            562.0,\n            237.99999999999997\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17f42b353a5146c4_33",\n          "bbox_xyxy": [\n            403.0,\n            525.75,\n            430.0,\n            552.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17f42b353a5146c4_34",\n          "bbox_xyxy": [\n            0.0,\n            187.0,\n            25.0,\n            215.99999999999997\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17f42b353a5146c4_35",\n          "bbox_xyxy": [\n            612.75,\n            59.49999999999999,\n            627.25,\n            87.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17f42b353a5146c4_36",\n          "bbox_xyxy": [\n            583.0,\n            198.75,\n            609.0,\n            227.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17f42b353a5146c4_37",\n          "bbox_xyxy": [\n            252.25,\n            0.0,\n            279.75,\n            15.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17f42b353a5146c4_38",\n          "bbox_xyxy": [\n            0.2499999999999991,\n            82.0,\n            24.75,\n            103.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17f42b353a5146c4_39",\n          "bbox_xyxy": [\n            495.0,\n            112.75,\n            515.0,\n            143.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17f42b353a5146c4_40",\n          "bbox_xyxy": [\n            263.75,\n            195.0,\n            290.25,\n            215.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17f42b353a5146c4_41",\n          "bbox_xyxy": [\n            502.5,\n            56.49999999999999,\n            524.5,\n            85.49999999999999\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17f42b353a5146c4_42",\n          "bbox_xyxy": [\n            584.0,\n            627.0,\n            606.0,\n            640.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17f42b353a5146c4_43",\n          "bbox_xyxy": [\n            386.75,\n            71.74999999999999,\n            413.25,\n            102.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17f42b353a5146c4_44",\n          "bbox_xyxy": [\n            301.75,\n            342.25,\n            334.25,\n            362.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17f42b353a5146c4_45",\n          "bbox_xyxy": [\n            567.0,\n            270.5,\n            596.0,\n            293.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17f42b353a5146c4_46",\n          "bbox_xyxy": [\n            522.5,\n            266.5,\n            548.5,\n            293.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17f42b353a5146c4_47",\n          "bbox_xyxy": [\n            434.75,\n            124.5,\n            467.25,\n            145.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17f42b353a5146c4_48",\n          "bbox_xyxy": [\n            293.5,\n            71.5,\n            318.5,\n            98.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17f42b353a5146c4_49",\n          "bbox_xyxy": [\n            242.5,\n            291.25,\n            259.5,\n            321.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17f42b353a5146c4_50",\n          "bbox_xyxy": [\n            285.25,\n            296.75,\n            317.75,\n            317.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17f42b353a5146c4_51",\n          "bbox_xyxy": [\n            17.499999999999996,\n            619.25,\n            38.5,\n            639.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17f42b353a5146c4_52",\n          "bbox_xyxy": [\n            621.0,\n            112.5,\n            640.0,\n            142.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17f42b353a5146c4_53",\n          "bbox_xyxy": [\n            31.500000000000004,\n            341.25,\n            51.5,\n            368.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/6100321344378571869_jpg.rf.a00cb974b5b8be3b68c6f081f6d0e9d7.jpg",\n        "size": 132697,\n        "compressed": 132594,\n        "offset": 404869211,\n        "crc32": "3e1ae4b4",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_17fee450a4e82ffa",\n      "relative_path": "images/IMG_0888_JPG.rf.9521cd963bb8fe755e025386c147639b.jpg",\n      "sha256": "dd4ebd4a79233577b262bbfffa315003cc80267d917002a120ea060f3b61a83e",\n      "decoded_sha256": "e8f46132b42349ec1657c4230bbcadb8fb5ea964b69248d198044de59a0ce7ac",\n      "dhash": "1717174f150b132d",\n      "bytes": 42133,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_0888_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_17fee450a4e82ffa_0",\n          "bbox_xyxy": [\n            303.5,\n            65.25,\n            340.50000000000006,\n            124.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fee450a4e82ffa_1",\n          "bbox_xyxy": [\n            323.75,\n            150.5,\n            351.25,\n            208.49999999999997\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fee450a4e82ffa_2",\n          "bbox_xyxy": [\n            265.0,\n            198.25,\n            311.0,\n            239.74999999999997\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fee450a4e82ffa_3",\n          "bbox_xyxy": [\n            195.5,\n            92.0,\n            240.5,\n            138.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fee450a4e82ffa_4",\n          "bbox_xyxy": [\n            105.24999999999999,\n            131.75,\n            153.75,\n            200.25000000000003\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fee450a4e82ffa_5",\n          "bbox_xyxy": [\n            558.0,\n            129.0,\n            606.0,\n            170.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fee450a4e82ffa_6",\n          "bbox_xyxy": [\n            607.75,\n            213.0,\n            634.25,\n            267.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fee450a4e82ffa_7",\n          "bbox_xyxy": [\n            530.5,\n            210.75,\n            567.5,\n            265.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fee450a4e82ffa_8",\n          "bbox_xyxy": [\n            474.75,\n            122.0,\n            519.25,\n            190.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fee450a4e82ffa_9",\n          "bbox_xyxy": [\n            395.5,\n            218.25,\n            450.5,\n            261.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fee450a4e82ffa_10",\n          "bbox_xyxy": [\n            473.25,\n            270.75,\n            508.75,\n            334.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fee450a4e82ffa_11",\n          "bbox_xyxy": [\n            488.5,\n            339.75,\n            514.5,\n            408.24999999999994\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fee450a4e82ffa_12",\n          "bbox_xyxy": [\n            421.0,\n            367.0,\n            453.00000000000006,\n            435.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fee450a4e82ffa_13",\n          "bbox_xyxy": [\n            500.75,\n            454.75,\n            548.25,\n            498.24999999999994\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fee450a4e82ffa_14",\n          "bbox_xyxy": [\n            572.0,\n            501.75,\n            613.0,\n            539.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fee450a4e82ffa_15",\n          "bbox_xyxy": [\n            576.5,\n            359.5,\n            623.5,\n            402.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fee450a4e82ffa_16",\n          "bbox_xyxy": [\n            332.25,\n            256.5,\n            374.75,\n            306.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fee450a4e82ffa_17",\n          "bbox_xyxy": [\n            323.75,\n            307.25,\n            369.25,\n            354.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fee450a4e82ffa_18",\n          "bbox_xyxy": [\n            315.00000000000006,\n            389.00000000000006,\n            364.00000000000006,\n            430.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fee450a4e82ffa_19",\n          "bbox_xyxy": [\n            228.25000000000003,\n            258.0,\n            258.75,\n            317.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fee450a4e82ffa_20",\n          "bbox_xyxy": [\n            96.5683735,\n            137.92904,\n            305.12848049999997,\n            460.92312\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fee450a4e82ffa_21",\n          "bbox_xyxy": [\n            143.5,\n            260.0,\n            175.5,\n            335.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fee450a4e82ffa_22",\n          "bbox_xyxy": [\n            87.0,\n            296.5,\n            123.0,\n            360.50000000000006\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fee450a4e82ffa_23",\n          "bbox_xyxy": [\n            5.999999999999998,\n            134.0,\n            45.0,\n            180.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fee450a4e82ffa_24",\n          "bbox_xyxy": [\n            12.249999999999996,\n            183.49999999999997,\n            58.75,\n            234.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fee450a4e82ffa_25",\n          "bbox_xyxy": [\n            0.0,\n            268.0,\n            47.0,\n            304.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fee450a4e82ffa_26",\n          "bbox_xyxy": [\n            375.25,\n            433.25,\n            419.75,\n            466.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fee450a4e82ffa_27",\n          "bbox_xyxy": [\n            405.25,\n            523.5,\n            456.75000000000006,\n            554.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fee450a4e82ffa_28",\n          "bbox_xyxy": [\n            328.75,\n            503.0,\n            373.25,\n            560.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fee450a4e82ffa_29",\n          "bbox_xyxy": [\n            301.25,\n            484.5,\n            328.75,\n            565.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fee450a4e82ffa_30",\n          "bbox_xyxy": [\n            275.25,\n            497.5,\n            300.75,\n            568.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fee450a4e82ffa_31",\n          "bbox_xyxy": [\n            213.5,\n            441.25,\n            257.5,\n            495.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fee450a4e82ffa_32",\n          "bbox_xyxy": [\n            149.75,\n            519.5,\n            195.25,\n            576.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fee450a4e82ffa_33",\n          "bbox_xyxy": [\n            138.75,\n            412.5,\n            168.25,\n            487.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fee450a4e82ffa_34",\n          "bbox_xyxy": [\n            20.0,\n            383.0,\n            48.0,\n            458.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fee450a4e82ffa_35",\n          "bbox_xyxy": [\n            55.74999999999999,\n            434.49999999999994,\n            91.25,\n            506.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fee450a4e82ffa_36",\n          "bbox_xyxy": [\n            0.2499999999999991,\n            533.75,\n            34.75,\n            598.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_0888_JPG.rf.9521cd963bb8fe755e025386c147639b.jpg",\n        "size": 42133,\n        "compressed": 41928,\n        "offset": 457288892,\n        "crc32": "6c2ce6aa",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_18066acb1b029873",\n      "relative_path": "images/IMG_2521_JPG.rf.ae1331f0bf1471acc1e23a04015559d1.jpg",\n      "sha256": "06a26305aed1b2ea2731320488db83b64cf18dfbd57d430619d9eced4c4a07dd",\n      "decoded_sha256": "18690300f956aa9fbe87fb816b4d72524142bd306bc94356205600289c88f02d",\n      "dhash": "3070686868703448",\n      "bytes": 25037,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2521_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_18066acb1b029873_0",\n          "bbox_xyxy": [\n            144.00000000000003,\n            34.00000000000002,\n            482.0,\n            598.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_2521_JPG.rf.ae1331f0bf1471acc1e23a04015559d1.jpg",\n        "size": 25037,\n        "compressed": 20814,\n        "offset": 562160867,\n        "crc32": "b41a0759",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_181782f602764a19",\n      "relative_path": "images/6159142536404995690_jpg.rf.9c4806d6ab643fd7c6af7098c6914e69.jpg",\n      "sha256": "263de50a095f5464750ba27777e134d3fae7317d1a6400bf8e303c54bdc75036",\n      "decoded_sha256": "37c2cb265822a3cb99a3044ea8b0d56554db57f216b628e6db2e20f208d9367f",\n      "dhash": "464604444246c606",\n      "bytes": 18079,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "6159142536404995690_jpg",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_181782f602764a19_0",\n          "bbox_xyxy": [\n            177.75000000000003,\n            82.25000000000001,\n            234.25,\n            138.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181782f602764a19_1",\n          "bbox_xyxy": [\n            468.75,\n            206.0,\n            514.25,\n            271.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181782f602764a19_2",\n          "bbox_xyxy": [\n            510.75,\n            402.25,\n            559.25,\n            479.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181782f602764a19_3",\n          "bbox_xyxy": [\n            478.5,\n            516.75,\n            531.5,\n            589.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181782f602764a19_4",\n          "bbox_xyxy": [\n            172.75000000000003,\n            477.75,\n            218.25,\n            548.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181782f602764a19_5",\n          "bbox_xyxy": [\n            496.25,\n            66.0,\n            561.75,\n            110.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181782f602764a19_6",\n          "bbox_xyxy": [\n            11.000000000000005,\n            396.0,\n            72.0,\n            441.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181782f602764a19_7",\n          "bbox_xyxy": [\n            131.0,\n            429.75,\n            187.0,\n            488.24999999999994\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181782f602764a19_8",\n          "bbox_xyxy": [\n            177.00000000000003,\n            329.25000000000006,\n            235.0,\n            379.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181782f602764a19_9",\n          "bbox_xyxy": [\n            159.25,\n            289.0,\n            221.75,\n            330.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/6159142536404995690_jpg.rf.9c4806d6ab643fd7c6af7098c6914e69.jpg",\n        "size": 18079,\n        "compressed": 13809,\n        "offset": 417706861,\n        "crc32": "fd093957",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_18761f97d783eb06",\n      "relative_path": "images/IMG_2228_JPG.rf.35ac1b7d8cb7efcf44db7cbec7b3420a.jpg",\n      "sha256": "5ea269ac54584634590c9c6c3de4c86c0ec7ca6f534a614fe58e67fbb4037e4e",\n      "decoded_sha256": "011b680fbe778e6baaa9b78093758d1c25c7ca54f4365c783eb9357b1758a812",\n      "dhash": "4870706869687073",\n      "bytes": 31401,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2228_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_18761f97d783eb06_0",\n          "bbox_xyxy": [\n            121.74999999999997,\n            8.999999999999986,\n            544.25,\n            637.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_2228_JPG.rf.35ac1b7d8cb7efcf44db7cbec7b3420a.jpg",\n        "size": 31401,\n        "compressed": 30072,\n        "offset": 536774174,\n        "crc32": "77af1186",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_18ab1c1ee4644e73",\n      "relative_path": "images/6179275676641332323_jpg.rf.7dd0fc6ac00dd2808bbb73d7154c2ff6.jpg",\n      "sha256": "781d023f91fa57be2ebdcc39ca462720a0b7eaae393b459e28da04877725cb4a",\n      "decoded_sha256": "84fcdd69bef120f736e8cff10f9698e68bf8e1bec375529400bbc85ef36ab888",\n      "dhash": "8ba7ab9793970719",\n      "bytes": 118484,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "6179275676641332323_jpg",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_18ab1c1ee4644e73_0",\n          "bbox_xyxy": [\n            536.75,\n            471.25,\n            575.25,\n            500.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_18ab1c1ee4644e73_1",\n          "bbox_xyxy": [\n            0.0,\n            537.0,\n            42.0,\n            591.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_18ab1c1ee4644e73_2",\n          "bbox_xyxy": [\n            0.0,\n            469.5,\n            58.0,\n            519.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_18ab1c1ee4644e73_3",\n          "bbox_xyxy": [\n            292.49999999999994,\n            409.49999999999994,\n            353.49999999999994,\n            443.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_18ab1c1ee4644e73_4",\n          "bbox_xyxy": [\n            0.0,\n            66.0,\n            16.0,\n            100.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_18ab1c1ee4644e73_5",\n          "bbox_xyxy": [\n            451.0,\n            109.50000000000001,\n            481.0,\n            138.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_18ab1c1ee4644e73_6",\n          "bbox_xyxy": [\n            379.0,\n            15.25,\n            433.00000000000006,\n            61.75000000000001\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_18ab1c1ee4644e73_7",\n          "bbox_xyxy": [\n            271.75,\n            480.25,\n            325.25,\n            541.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_18ab1c1ee4644e73_8",\n          "bbox_xyxy": [\n            509.0,\n            378.5,\n            549.0,\n            443.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_18ab1c1ee4644e73_9",\n          "bbox_xyxy": [\n            200.49999999999997,\n            75.5,\n            237.5,\n            100.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_18ab1c1ee4644e73_10",\n          "bbox_xyxy": [\n            31.75,\n            340.99999999999994,\n            74.25,\n            380.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_18ab1c1ee4644e73_11",\n          "bbox_xyxy": [\n            537.0,\n            0.0,\n            582.0,\n            22.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_18ab1c1ee4644e73_12",\n          "bbox_xyxy": [\n            109.0,\n            196.25,\n            174.0,\n            234.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_18ab1c1ee4644e73_13",\n          "bbox_xyxy": [\n            134.25,\n            229.75,\n            201.75,\n            266.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_18ab1c1ee4644e73_14",\n          "bbox_xyxy": [\n            541.75,\n            24.750000000000007,\n            580.25,\n            68.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_18ab1c1ee4644e73_15",\n          "bbox_xyxy": [\n            54.75,\n            155.25,\n            101.25,\n            193.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_18ab1c1ee4644e73_16",\n          "bbox_xyxy": [\n            0.25000000000000133,\n            232.0,\n            25.75,\n            276.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_18ab1c1ee4644e73_17",\n          "bbox_xyxy": [\n            141.0,\n            0.0,\n            195.0,\n            19.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_18ab1c1ee4644e73_18",\n          "bbox_xyxy": [\n            535.25,\n            69.75,\n            581.75,\n            101.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_18ab1c1ee4644e73_19",\n          "bbox_xyxy": [\n            3.75,\n            105.5,\n            48.25,\n            166.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_18ab1c1ee4644e73_20",\n          "bbox_xyxy": [\n            358.25,\n            386.0,\n            385.75,\n            408.00000000000006\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_18ab1c1ee4644e73_21",\n          "bbox_xyxy": [\n            279.0,\n            58.00000000000001,\n            313.0,\n            80.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_18ab1c1ee4644e73_22",\n          "bbox_xyxy": [\n            97.5,\n            263.5,\n            145.5,\n            296.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_18ab1c1ee4644e73_23",\n          "bbox_xyxy": [\n            505.5,\n            93.0,\n            560.5,\n            134.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_18ab1c1ee4644e73_24",\n          "bbox_xyxy": [\n            288.75,\n            11.749999999999998,\n            317.25,\n            54.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_18ab1c1ee4644e73_25",\n          "bbox_xyxy": [\n            90.0,\n            587.0,\n            120.0,\n            640.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_18ab1c1ee4644e73_26",\n          "bbox_xyxy": [\n            199.0,\n            0.0,\n            240.0,\n            21.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_18ab1c1ee4644e73_27",\n          "bbox_xyxy": [\n            209.74999999999997,\n            492.75,\n            278.25,\n            529.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_18ab1c1ee4644e73_28",\n          "bbox_xyxy": [\n            354.0,\n            259.25,\n            391.99999999999994,\n            277.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_18ab1c1ee4644e73_29",\n          "bbox_xyxy": [\n            34.25,\n            13.499999999999996,\n            80.75,\n            62.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_18ab1c1ee4644e73_30",\n          "bbox_xyxy": [\n            404.25,\n            147.75000000000003,\n            456.75,\n            179.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_18ab1c1ee4644e73_31",\n          "bbox_xyxy": [\n            503.25,\n            50.25,\n            531.75,\n            78.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_18ab1c1ee4644e73_32",\n          "bbox_xyxy": [\n            420.0,\n            196.50000000000003,\n            449.00000000000006,\n            224.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_18ab1c1ee4644e73_33",\n          "bbox_xyxy": [\n            48.24999999999999,\n            618.25,\n            93.75,\n            639.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_18ab1c1ee4644e73_34",\n          "bbox_xyxy": [\n            450.75,\n            144.0,\n            491.25,\n            174.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_18ab1c1ee4644e73_35",\n          "bbox_xyxy": [\n            293.0,\n            441.25,\n            344.0,\n            475.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_18ab1c1ee4644e73_36",\n          "bbox_xyxy": [\n            110.75,\n            58.0,\n            159.25,\n            118.99999999999999\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_18ab1c1ee4644e73_37",\n          "bbox_xyxy": [\n            585.0,\n            0.0,\n            610.0,\n            10.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_18ab1c1ee4644e73_38",\n          "bbox_xyxy": [\n            95.5,\n            378.75,\n            124.5,\n            445.25000000000006\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_18ab1c1ee4644e73_39",\n          "bbox_xyxy": [\n            102.74999999999999,\n            126.25,\n            136.25,\n            159.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_18ab1c1ee4644e73_40",\n          "bbox_xyxy": [\n            94.24999999999999,\n            554.5,\n            127.75,\n            584.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_18ab1c1ee4644e73_41",\n          "bbox_xyxy": [\n            552.25,\n            279.5,\n            608.75,\n            309.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_18ab1c1ee4644e73_42",\n          "bbox_xyxy": [\n            220.99999999999997,\n            210.5,\n            258.0,\n            235.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_18ab1c1ee4644e73_43",\n          "bbox_xyxy": [\n            576.0,\n            208.25,\n            640.0,\n            242.75000000000003\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_18ab1c1ee4644e73_44",\n          "bbox_xyxy": [\n            174.5,\n            609.0,\n            241.50000000000003,\n            640.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_18ab1c1ee4644e73_45",\n          "bbox_xyxy": [\n            158.5,\n            470.50000000000006,\n            203.5,\n            498.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_18ab1c1ee4644e73_46",\n          "bbox_xyxy": [\n            114.25,\n            310.75,\n            168.75,\n            374.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_18ab1c1ee4644e73_47",\n          "bbox_xyxy": [\n            418.75,\n            101.0,\n            435.25000000000006,\n            127.00000000000001\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_18ab1c1ee4644e73_48",\n          "bbox_xyxy": [\n            217.75000000000003,\n            31.250000000000004,\n            243.25,\n            51.75000000000001\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_18ab1c1ee4644e73_49",\n          "bbox_xyxy": [\n            337.75,\n            593.25,\n            374.25000000000006,\n            612.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_18ab1c1ee4644e73_50",\n          "bbox_xyxy": [\n            400.74999999999994,\n            376.75,\n            455.25,\n            426.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_18ab1c1ee4644e73_51",\n          "bbox_xyxy": [\n            210.75,\n            234.25,\n            237.24999999999997,\n            261.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_18ab1c1ee4644e73_52",\n          "bbox_xyxy": [\n            168.25,\n            267.0,\n            222.75000000000003,\n            315.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_18ab1c1ee4644e73_53",\n          "bbox_xyxy": [\n            491.5,\n            186.25,\n            540.5,\n            218.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_18ab1c1ee4644e73_54",\n          "bbox_xyxy": [\n            124.25,\n            411.25,\n            165.75,\n            443.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_18ab1c1ee4644e73_55",\n          "bbox_xyxy": [\n            601.5,\n            369.25,\n            631.5,\n            401.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_18ab1c1ee4644e73_56",\n          "bbox_xyxy": [\n            607.0,\n            266.0,\n            640.0,\n            289.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/6179275676641332323_jpg.rf.7dd0fc6ac00dd2808bbb73d7154c2ff6.jpg",\n        "size": 118484,\n        "compressed": 118380,\n        "offset": 419447135,\n        "crc32": "373a40b7",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_19044d34d566c840",\n      "relative_path": "images/IMG_2456_JPG.rf.45e9e4e3eb6c8cee25218987662d5d32.jpg",\n      "sha256": "19410a2e65d6875240f284cf2ece004a5350628c07c9dd9000902c44cba089cf",\n      "decoded_sha256": "3376132757849bf65af9d576d5521374ad671380fa098c8e818110595706384d",\n      "dhash": "4c703839797131d2",\n      "bytes": 30207,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2456_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_19044d34d566c840_0",\n          "bbox_xyxy": [\n            147.75000000000003,\n            50.74999999999996,\n            541.25,\n            590.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/IMG_2456_JPG.rf.45e9e4e3eb6c8cee25218987662d5d32.jpg",\n        "size": 30207,\n        "compressed": 29489,\n        "offset": 556455250,\n        "crc32": "6fe0efa0",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_190f4e687fb5372f",\n      "relative_path": "images/6100321344378571830_jpg.rf.4487d1fddda7eaefacac4270cd1cf0aa.jpg",\n      "sha256": "4d5f598022665c4fb09cb98fc46f3890abe63d3b27f4a098deb4cd72078f1054",\n      "decoded_sha256": "477b44021b86221e1fb96d8c4dfea3aefa959a3ca7ba6095dd3e5a45ed7e6ff7",\n      "dhash": "46441622264d6467",\n      "bytes": 113237,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "6100321344378571830_jpg",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "train",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_190f4e687fb5372f_0",\n          "bbox_xyxy": [\n            408.0,\n            183.25,\n            433.0,\n            209.74999999999997\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_190f4e687fb5372f_1",\n          "bbox_xyxy": [\n            254.25,\n            177.0,\n            290.75,\n            203.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_190f4e687fb5372f_2",\n          "bbox_xyxy": [\n            574.25,\n            351.5,\n            598.75,\n            371.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_190f4e687fb5372f_3",\n          "bbox_xyxy": [\n            39.75,\n            257.75,\n            85.25,\n            282.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_190f4e687fb5372f_4",\n          "bbox_xyxy": [\n            150.75,\n            561.75,\n            178.25,\n            583.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_190f4e687fb5372f_5",\n          "bbox_xyxy": [\n            46.75,\n            546.25,\n            78.25,\n            560.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_190f4e687fb5372f_6",\n          "bbox_xyxy": [\n            400.74999999999994,\n            262.0,\n            425.25,\n            278.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_190f4e687fb5372f_7",\n          "bbox_xyxy": [\n            539.5,\n            329.5,\n            563.5,\n            355.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_190f4e687fb5372f_8",\n          "bbox_xyxy": [\n            498.24999999999994,\n            105.0,\n            527.75,\n            122.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_190f4e687fb5372f_9",\n          "bbox_xyxy": [\n            495.75,\n            615.25,\n            529.25,\n            632.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_190f4e687fb5372f_10",\n          "bbox_xyxy": [\n            556.75,\n            269.0,\n            574.25,\n            293.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_190f4e687fb5372f_11",\n          "bbox_xyxy": [\n            74.0,\n            285.5,\n            95.0,\n            310.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_190f4e687fb5372f_12",\n          "bbox_xyxy": [\n            152.0,\n            169.25,\n            182.0,\n            187.75000000000003\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_190f4e687fb5372f_13",\n          "bbox_xyxy": [\n            524.25,\n            50.0,\n            551.75,\n            68.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_190f4e687fb5372f_14",\n          "bbox_xyxy": [\n            577.0,\n            626.25,\n            601.0,\n            639.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_190f4e687fb5372f_15",\n          "bbox_xyxy": [\n            299.5,\n            537.75,\n            315.5,\n            565.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_190f4e687fb5372f_16",\n          "bbox_xyxy": [\n            471.0,\n            155.25,\n            486.0,\n            179.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_190f4e687fb5372f_17",\n          "bbox_xyxy": [\n            280.0,\n            296.5,\n            297.0,\n            323.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_190f4e687fb5372f_18",\n          "bbox_xyxy": [\n            482.5,\n            147.75,\n            507.5,\n            167.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/train/images/6100321344378571830_jpg.rf.4487d1fddda7eaefacac4270cd1cf0aa.jpg",\n        "size": 113237,\n        "compressed": 113128,\n        "offset": 394630135,\n        "crc32": "d14295a6",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_00aceb5ac830af30",\n      "relative_path": "images/IMG_1832_JPG.rf.8b648e9a8ecb800e4c0ff5e88193bf78.jpg",\n      "sha256": "786c6ba34dbb8611b2f403b23ad6bb9a0694ee751f38a1c69e8c4e1d164024d4",\n      "decoded_sha256": "1311aa9ee75889503179f0b99f2f9de1fe2adce20a1843d8141f6a3f6cef55a3",\n      "dhash": "4670707171713130",\n      "bytes": 24394,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_1832_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "validation",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_00aceb5ac830af30_0",\n          "bbox_xyxy": [\n            196.99999999999994,\n            94.74999999999997,\n            464.0,\n            551.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/valid/images/IMG_1832_JPG.rf.8b648e9a8ecb800e4c0ff5e88193bf78.jpg",\n        "size": 24394,\n        "compressed": 23242,\n        "offset": 629758303,\n        "crc32": "838372db",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_084453ad6831479a",\n      "relative_path": "images/IMG_2684_JPG.rf.496a57e20da06ad21c8acdddbe6716b3.jpg",\n      "sha256": "f6790d55c7e4b55f2eff9ee74161cf7d9ca7b85a0d7f691651f596b66d83d7c7",\n      "decoded_sha256": "9b600c5f3e2023af94a0d854a379982783f81836f61e9872a862130cc9a53317",\n      "dhash": "3231393070703859",\n      "bytes": 30875,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2684_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "validation",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_084453ad6831479a_0",\n          "bbox_xyxy": [\n            184.5,\n            3.000000000000007,\n            535.5,\n            578.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/valid/images/IMG_2684_JPG.rf.496a57e20da06ad21c8acdddbe6716b3.jpg",\n        "size": 30875,\n        "compressed": 30448,\n        "offset": 632465951,\n        "crc32": "916c75b4",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_088a43c5a44ad335",\n      "relative_path": "images/IMG_2412_JPG.rf.71c47555b5636597782efa299c4451af.jpg",\n      "sha256": "f1f0b54166c618e1166d02fc5da757bbc442682fa87b9cb9bc8dc90590b20ca3",\n      "decoded_sha256": "7d1bfa7441c28d5f48377c2329cf89b23fad39352883c09deeacdc1b1581d70a",\n      "dhash": "2430317969793132",\n      "bytes": 28782,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2412_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "validation",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_088a43c5a44ad335_0",\n          "bbox_xyxy": [\n            151.0,\n            65.49999999999997,\n            536.0,\n            592.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/valid/images/IMG_2412_JPG.rf.71c47555b5636597782efa299c4451af.jpg",\n        "size": 28782,\n        "compressed": 28238,\n        "offset": 631499878,\n        "crc32": "b1c06b6d",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_0a2cc2f62edcf7a7",\n      "relative_path": "images/IMG_2227_JPG.rf.fc1ea8fb9304c3780709501367c167b1.jpg",\n      "sha256": "b6a85a6bf92608149bcfa96d0535846a9f238b4923ec87084c5404e68ce0dfca",\n      "decoded_sha256": "1916397e821bd9f23bcce005cf1b7bdef1166ed4a8f9f5ab91b4ad0f502c66ee",\n      "dhash": "a4303139797931b0",\n      "bytes": 28618,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2227_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "validation",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_0a2cc2f62edcf7a7_0",\n          "bbox_xyxy": [\n            212.25,\n            60.75000000000003,\n            507.75,\n            566.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/valid/images/IMG_2227_JPG.rf.fc1ea8fb9304c3780709501367c167b1.jpg",\n        "size": 28618,\n        "compressed": 27502,\n        "offset": 630764870,\n        "crc32": "4e883b53",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_0f2d137f64873648",\n      "relative_path": "images/IMG_2484_JPG.rf.c30203f8aee6b16b75841bf171c184fb.jpg",\n      "sha256": "22572e8c0ed2d894931e930495d88c0b77da5f46eef82a243a467429d50ec203",\n      "decoded_sha256": "cb47859a6ae9e78dd67fb9278e0af4a080a3e52e9c45f9cdeac9f2d9bfe9c437",\n      "dhash": "1070696979713032",\n      "bytes": 25630,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2484_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "validation",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_0f2d137f64873648_0",\n          "bbox_xyxy": [\n            170.5,\n            23.249999999999993,\n            519.5,\n            530.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/valid/images/IMG_2484_JPG.rf.c30203f8aee6b16b75841bf171c184fb.jpg",\n        "size": 25630,\n        "compressed": 24944,\n        "offset": 631760680,\n        "crc32": "a99f16a6",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_11fec0dca61110e1",\n      "relative_path": "images/IMG_1156_JPG.rf.4e704b009f5c95f8dd0eefd39e62cfe8.jpg",\n      "sha256": "2ece020eea1ad7e3b54d2f5acf6402a20ada465998d7f4063f071745a8d75c31",\n      "decoded_sha256": "31e63070cd226941ef021d72aab23e419c321b59c0bd4064d457cfa27d9a5761",\n      "dhash": "0793091513379511",\n      "bytes": 43340,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_1156_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "validation",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_11fec0dca61110e1_0",\n          "bbox_xyxy": [\n            349.0,\n            131.5,\n            374.0,\n            174.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_11fec0dca61110e1_1",\n          "bbox_xyxy": [\n            55.5,\n            439.25,\n            85.5,\n            471.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_11fec0dca61110e1_2",\n          "bbox_xyxy": [\n            314.75,\n            412.5,\n            345.25,\n            453.49999999999994\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_11fec0dca61110e1_3",\n          "bbox_xyxy": [\n            417.0,\n            200.75,\n            440.0,\n            249.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_11fec0dca61110e1_4",\n          "bbox_xyxy": [\n            182.0,\n            455.74999999999994,\n            217.0,\n            490.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_11fec0dca61110e1_5",\n          "bbox_xyxy": [\n            225.75000000000003,\n            420.25,\n            246.25,\n            465.74999999999994\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_11fec0dca61110e1_6",\n          "bbox_xyxy": [\n            339.5,\n            455.0,\n            372.5,\n            490.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_11fec0dca61110e1_7",\n          "bbox_xyxy": [\n            388.75,\n            354.75,\n            416.25,\n            389.25000000000006\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_11fec0dca61110e1_8",\n          "bbox_xyxy": [\n            201.50000000000003,\n            336.25,\n            224.5,\n            381.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_11fec0dca61110e1_9",\n          "bbox_xyxy": [\n            426.5,\n            451.5,\n            446.5,\n            496.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_11fec0dca61110e1_10",\n          "bbox_xyxy": [\n            416.75,\n            130.25,\n            446.25,\n            161.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_11fec0dca61110e1_11",\n          "bbox_xyxy": [\n            388.5,\n            527.5,\n            421.5,\n            561.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_11fec0dca61110e1_12",\n          "bbox_xyxy": [\n            206.25,\n            598.75,\n            238.75,\n            627.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_11fec0dca61110e1_13",\n          "bbox_xyxy": [\n            277.5,\n            240.25000000000003,\n            302.5,\n            290.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_11fec0dca61110e1_14",\n          "bbox_xyxy": [\n            14.249999999999998,\n            306.99999999999994,\n            48.75,\n            339.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_11fec0dca61110e1_15",\n          "bbox_xyxy": [\n            468.75,\n            252.0,\n            497.24999999999994,\n            283.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_11fec0dca61110e1_16",\n          "bbox_xyxy": [\n            13.249999999999996,\n            374.00000000000006,\n            37.75,\n            420.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_11fec0dca61110e1_17",\n          "bbox_xyxy": [\n            298.0,\n            366.5,\n            328.0,\n            404.49999999999994\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_11fec0dca61110e1_18",\n          "bbox_xyxy": [\n            129.75,\n            230.25,\n            161.25,\n            268.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_11fec0dca61110e1_19",\n          "bbox_xyxy": [\n            4.249999999999998,\n            155.25000000000003,\n            33.75,\n            186.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_11fec0dca61110e1_20",\n          "bbox_xyxy": [\n            240.5,\n            168.5,\n            268.5,\n            203.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_11fec0dca61110e1_21",\n          "bbox_xyxy": [\n            138.0,\n            590.75,\n            172.0,\n            622.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_11fec0dca61110e1_22",\n          "bbox_xyxy": [\n            75.25,\n            300.5,\n            99.75,\n            347.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_11fec0dca61110e1_23",\n          "bbox_xyxy": [\n            127.99999999999999,\n            82.5,\n            159.0,\n            111.49999999999999\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_11fec0dca61110e1_24",\n          "bbox_xyxy": [\n            93.0,\n            451.25,\n            115.0,\n            499.74999999999994\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_11fec0dca61110e1_25",\n          "bbox_xyxy": [\n            112.25000000000001,\n            167.0,\n            148.75,\n            203.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_11fec0dca61110e1_26",\n          "bbox_xyxy": [\n            168.49999999999997,\n            532.5,\n            189.5,\n            575.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_11fec0dca61110e1_27",\n          "bbox_xyxy": [\n            262.75,\n            477.24999999999994,\n            284.25,\n            518.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_11fec0dca61110e1_28",\n          "bbox_xyxy": [\n            13.249999999999996,\n            63.99999999999999,\n            42.75,\n            100.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_11fec0dca61110e1_29",\n          "bbox_xyxy": [\n            99.24999999999999,\n            366.5,\n            127.75,\n            399.49999999999994\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_11fec0dca61110e1_30",\n          "bbox_xyxy": [\n            272.25,\n            532.0,\n            296.75,\n            575.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_11fec0dca61110e1_31",\n          "bbox_xyxy": [\n            37.25,\n            248.0,\n            63.75,\n            281.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_11fec0dca61110e1_32",\n          "bbox_xyxy": [\n            97.99999999999999,\n            543.5,\n            121.0,\n            588.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_11fec0dca61110e1_33",\n          "bbox_xyxy": [\n            308.00000000000006,\n            321.75,\n            336.0,\n            354.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_11fec0dca61110e1_34",\n          "bbox_xyxy": [\n            87.25000000000001,\n            79.0,\n            118.75,\n            112.99999999999999\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_11fec0dca61110e1_35",\n          "bbox_xyxy": [\n            332.0,\n            578.0,\n            355.0,\n            610.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_11fec0dca61110e1_36",\n          "bbox_xyxy": [\n            351.75000000000006,\n            215.0,\n            382.25,\n            260.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_11fec0dca61110e1_37",\n          "bbox_xyxy": [\n            149.25,\n            121.5,\n            183.75,\n            163.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_11fec0dca61110e1_38",\n          "bbox_xyxy": [\n            304.25,\n            496.75,\n            336.75,\n            529.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_11fec0dca61110e1_39",\n          "bbox_xyxy": [\n            298.25,\n            561.0,\n            331.75,\n            593.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_11fec0dca61110e1_40",\n          "bbox_xyxy": [\n            53.5,\n            368.25,\n            82.5,\n            395.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_11fec0dca61110e1_41",\n          "bbox_xyxy": [\n            402.0,\n            584.25,\n            435.0,\n            618.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_11fec0dca61110e1_42",\n          "bbox_xyxy": [\n            147.25,\n            308.5,\n            170.75,\n            347.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_11fec0dca61110e1_43",\n          "bbox_xyxy": [\n            41.0,\n            188.49999999999997,\n            66.0,\n            229.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_11fec0dca61110e1_44",\n          "bbox_xyxy": [\n            127.5,\n            478.0,\n            153.5,\n            525.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_11fec0dca61110e1_45",\n          "bbox_xyxy": [\n            483.5,\n            442.25,\n            514.5,\n            477.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_11fec0dca61110e1_46",\n          "bbox_xyxy": [\n            202.75,\n            550.75,\n            226.25,\n            582.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_11fec0dca61110e1_47",\n          "bbox_xyxy": [\n            267.75,\n            80.75000000000001,\n            296.25,\n            130.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_11fec0dca61110e1_48",\n          "bbox_xyxy": [\n            31.25,\n            583.75,\n            46.75,\n            633.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_11fec0dca61110e1_49",\n          "bbox_xyxy": [\n            173.49999999999997,\n            416.25,\n            199.5,\n            456.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_11fec0dca61110e1_50",\n          "bbox_xyxy": [\n            36.0,\n            495.25,\n            69.0,\n            525.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_11fec0dca61110e1_51",\n          "bbox_xyxy": [\n            260.0,\n            408.75,\n            292.0,\n            443.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_11fec0dca61110e1_52",\n          "bbox_xyxy": [\n            456.25,\n            523.5,\n            483.75,\n            561.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_11fec0dca61110e1_53",\n          "bbox_xyxy": [\n            108.5,\n            619.25,\n            139.5,\n            639.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/valid/images/IMG_1156_JPG.rf.4e704b009f5c95f8dd0eefd39e62cfe8.jpg",\n        "size": 43340,\n        "compressed": 43124,\n        "offset": 629349079,\n        "crc32": "bb5d1b33",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_127983f2e771bf55",\n      "relative_path": "images/IMG_2116_JPG.rf.4664377f4a48e3aa767e48875dec7a9c.jpg",\n      "sha256": "f47f0188d2bf5de653b756b21af6be0db5f5c2ae934aca125d2530a2e6dc3c54",\n      "decoded_sha256": "f8fd207af5099512b6240beb7044d8dd5f1cc2394ce765302fa9ad0831d9d90b",\n      "dhash": "4cf0716169697032",\n      "bytes": 28490,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2116_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "validation",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_127983f2e771bf55_0",\n          "bbox_xyxy": [\n            154.0,\n            71.50000000000003,\n            486.0,\n            587.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/valid/images/IMG_2116_JPG.rf.4664377f4a48e3aa767e48875dec7a9c.jpg",\n        "size": 28490,\n        "compressed": 27290,\n        "offset": 630498311,\n        "crc32": "7881bf92",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_139353b68c8897c4",\n      "relative_path": "images/IMG_4192_JPG.rf.8331a2e5f05aa50ba4a2911c425d9e0f.jpg",\n      "sha256": "c53b7a2ab9492ff88cdabacf2355d4fe8b21674a92874b5f1a7c9d8f48e9407c",\n      "decoded_sha256": "7c03bf2d98cc3bd76bcd52f3780584882d937c7494bb6071f9da53818c5da5f5",\n      "dhash": "2842c2b8c46c4031",\n      "bytes": 52759,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_4192_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "validation",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_139353b68c8897c4_0",\n          "bbox_xyxy": [\n            537.5,\n            122.74999999999999,\n            561.5,\n            164.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_139353b68c8897c4_1",\n          "bbox_xyxy": [\n            312.75,\n            175.49999999999997,\n            357.25,\n            202.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_139353b68c8897c4_2",\n          "bbox_xyxy": [\n            127.25,\n            131.75,\n            182.75,\n            154.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_139353b68c8897c4_3",\n          "bbox_xyxy": [\n            122.75,\n            164.00000000000003,\n            149.25,\n            208.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_139353b68c8897c4_4",\n          "bbox_xyxy": [\n            201.0,\n            162.5,\n            230.0,\n            200.49999999999997\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_139353b68c8897c4_5",\n          "bbox_xyxy": [\n            90.0,\n            238.25,\n            116.00000000000001,\n            274.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_139353b68c8897c4_6",\n          "bbox_xyxy": [\n            257.75,\n            232.75,\n            279.25,\n            270.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_139353b68c8897c4_7",\n          "bbox_xyxy": [\n            364.0,\n            248.25000000000003,\n            394.0,\n            283.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_139353b68c8897c4_8",\n          "bbox_xyxy": [\n            393.24999999999994,\n            249.74999999999997,\n            414.75,\n            289.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_139353b68c8897c4_9",\n          "bbox_xyxy": [\n            265.25,\n            327.0,\n            290.75,\n            378.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_139353b68c8897c4_10",\n          "bbox_xyxy": [\n            364.25000000000006,\n            472.0,\n            409.75,\n            490.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_139353b68c8897c4_11",\n          "bbox_xyxy": [\n            290.25,\n            552.75,\n            324.75,\n            573.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_139353b68c8897c4_12",\n          "bbox_xyxy": [\n            308.0,\n            572.0,\n            339.0,\n            599.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_139353b68c8897c4_13",\n          "bbox_xyxy": [\n            190.25000000000003,\n            534.75,\n            231.75,\n            564.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_139353b68c8897c4_14",\n          "bbox_xyxy": [\n            153.0,\n            490.25,\n            190.0,\n            529.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_139353b68c8897c4_15",\n          "bbox_xyxy": [\n            134.0,\n            379.49999999999994,\n            158.0,\n            426.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_139353b68c8897c4_16",\n          "bbox_xyxy": [\n            129.25,\n            298.25,\n            160.75,\n            330.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_139353b68c8897c4_17",\n          "bbox_xyxy": [\n            108.75,\n            280.25,\n            156.25,\n            301.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_139353b68c8897c4_18",\n          "bbox_xyxy": [\n            50.50000000000001,\n            224.25,\n            93.5,\n            256.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_139353b68c8897c4_19",\n          "bbox_xyxy": [\n            210.99999999999997,\n            102.5,\n            237.0,\n            146.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_139353b68c8897c4_20",\n          "bbox_xyxy": [\n            271.25,\n            498.25,\n            292.75,\n            526.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/valid/images/IMG_4192_JPG.rf.8331a2e5f05aa50ba4a2911c425d9e0f.jpg",\n        "size": 52759,\n        "compressed": 52614,\n        "offset": 633013349,\n        "crc32": "b9d1f0db",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_14a73bbb1f722b0f",\n      "relative_path": "images/IMG_4207_JPG.rf.fe7711bb92c13c1fdda8cf4826a60e84.jpg",\n      "sha256": "ed89c8c7d365912610d3936358466205ce3e8dff526ebd23141a7cdd5a1b5580",\n      "decoded_sha256": "46d9a5c3e70bf6aa7c6ed98064f065317ba952fcfd8a1e4f31d616e5f91d6d92",\n      "dhash": "d47cd975ccc2e430",\n      "bytes": 59349,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_4207_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "validation",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_14a73bbb1f722b0f_0",\n          "bbox_xyxy": [\n            136.25,\n            55.75,\n            193.75,\n            79.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_14a73bbb1f722b0f_1",\n          "bbox_xyxy": [\n            254.75,\n            67.75,\n            286.25,\n            112.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_14a73bbb1f722b0f_2",\n          "bbox_xyxy": [\n            167.75,\n            191.75,\n            207.25,\n            227.24999999999997\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_14a73bbb1f722b0f_3",\n          "bbox_xyxy": [\n            300.0,\n            307.75,\n            319.0,\n            354.25000000000006\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_14a73bbb1f722b0f_4",\n          "bbox_xyxy": [\n            364.0,\n            324.74999999999994,\n            391.0,\n            361.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_14a73bbb1f722b0f_5",\n          "bbox_xyxy": [\n            448.00000000000006,\n            317.75,\n            474.0,\n            356.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_14a73bbb1f722b0f_6",\n          "bbox_xyxy": [\n            442.0,\n            241.25,\n            470.0,\n            277.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_14a73bbb1f722b0f_7",\n          "bbox_xyxy": [\n            355.75,\n            401.75,\n            409.25,\n            434.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_14a73bbb1f722b0f_8",\n          "bbox_xyxy": [\n            494.75,\n            394.25000000000006,\n            527.25,\n            437.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_14a73bbb1f722b0f_9",\n          "bbox_xyxy": [\n            437.75,\n            494.5,\n            479.25000000000006,\n            525.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_14a73bbb1f722b0f_10",\n          "bbox_xyxy": [\n            342.75000000000006,\n            535.0,\n            401.25,\n            564.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_14a73bbb1f722b0f_11",\n          "bbox_xyxy": [\n            476.75,\n            568.0,\n            518.25,\n            593.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_14a73bbb1f722b0f_12",\n          "bbox_xyxy": [\n            481.75,\n            511.0,\n            504.25,\n            551.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_14a73bbb1f722b0f_13",\n          "bbox_xyxy": [\n            375.0,\n            580.25,\n            429.00000000000006,\n            594.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_14a73bbb1f722b0f_14",\n          "bbox_xyxy": [\n            155.25000000000003,\n            534.75,\n            206.75,\n            567.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_14a73bbb1f722b0f_15",\n          "bbox_xyxy": [\n            123.75,\n            428.75,\n            162.25,\n            450.25000000000006\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_14a73bbb1f722b0f_16",\n          "bbox_xyxy": [\n            102.5,\n            340.25000000000006,\n            129.5,\n            373.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_14a73bbb1f722b0f_17",\n          "bbox_xyxy": [\n            440.5,\n            136.0,\n            475.5,\n            157.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_14a73bbb1f722b0f_18",\n          "bbox_xyxy": [\n            483.75,\n            69.75,\n            523.25,\n            99.24999999999999\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/valid/images/IMG_4207_JPG.rf.fe7711bb92c13c1fdda8cf4826a60e84.jpg",\n        "size": 59349,\n        "compressed": 59218,\n        "offset": 633066136,\n        "crc32": "aa3fbdf7",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_14d154ffdf4762be",\n      "relative_path": "images/IMG_1780_JPG.rf.1e89aa5e6e83a127176aa166decf747f.jpg",\n      "sha256": "5c03a17eada429c5d2a2e4851fa77a2c220935171a77a1f17dd03daa7722a517",\n      "decoded_sha256": "3a8aaf1acbd5c9585df147d9205578855c3043e73bac46069df78a6027a414e4",\n      "dhash": "24b0717969717264",\n      "bytes": 26210,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_1780_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "validation",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_14d154ffdf4762be_0",\n          "bbox_xyxy": [\n            185.0,\n            3.75,\n            449.0,\n            567.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/valid/images/IMG_1780_JPG.rf.1e89aa5e6e83a127176aa166decf747f.jpg",\n        "size": 26210,\n        "compressed": 25409,\n        "offset": 629585032,\n        "crc32": "1c6a4b95",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_15d16e95a9155b0b",\n      "relative_path": "images/IMG_2465_JPG.rf.cb87da4b8b2f28fecb9667fe13a38075.jpg",\n      "sha256": "a5c057c008c14a5150c178bfbfc350f01c14a151e4a4f1490e86973c6d9b7cf2",\n      "decoded_sha256": "0615038226bf3dac65330d7c37c3cc8f96dca61dfbd88c9c7a87556f7fce8b67",\n      "dhash": "881831313139b1b2",\n      "bytes": 21907,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2465_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "validation",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_15d16e95a9155b0b_0",\n          "bbox_xyxy": [\n            187.99999999999997,\n            90.75,\n            498.0,\n            622.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/valid/images/IMG_2465_JPG.rf.cb87da4b8b2f28fecb9667fe13a38075.jpg",\n        "size": 21907,\n        "compressed": 20828,\n        "offset": 631657191,\n        "crc32": "8a14da4b",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_1918a97383232f6c",\n      "relative_path": "images/IMG_3901_JPG.rf.adaa50af2aceb9d3d32674009fc7f452.jpg",\n      "sha256": "7a3d4f6613aebebfd4fca927906f37ae11b4f8b5f60468101928245702f3ebed",\n      "decoded_sha256": "59d7758885c125a84d60573c0388742984b30f587cb257f8fcca74b21ada180a",\n      "dhash": "24e9d6c642b86265",\n      "bytes": 27066,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_3901_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "validation",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_1918a97383232f6c_0",\n          "bbox_xyxy": [\n            142.50000000000003,\n            156.75,\n            194.50000000000003,\n            206.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1918a97383232f6c_1",\n          "bbox_xyxy": [\n            201.50000000000003,\n            134.5,\n            245.5,\n            190.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1918a97383232f6c_2",\n          "bbox_xyxy": [\n            294.5,\n            138.0,\n            349.5,\n            214.00000000000003\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1918a97383232f6c_3",\n          "bbox_xyxy": [\n            181.25,\n            349.0,\n            214.75,\n            404.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1918a97383232f6c_4",\n          "bbox_xyxy": [\n            181.25,\n            309.25,\n            234.75,\n            342.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1918a97383232f6c_5",\n          "bbox_xyxy": [\n            319.0,\n            320.25,\n            371.99999999999994,\n            374.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1918a97383232f6c_6",\n          "bbox_xyxy": [\n            303.0,\n            442.5,\n            342.0,\n            495.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1918a97383232f6c_7",\n          "bbox_xyxy": [\n            347.5,\n            425.50000000000006,\n            405.5,\n            466.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1918a97383232f6c_8",\n          "bbox_xyxy": [\n            198.25,\n            476.99999999999994,\n            241.75,\n            524.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1918a97383232f6c_9",\n          "bbox_xyxy": [\n            534.0,\n            426.0,\n            575.0,\n            494.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1918a97383232f6c_10",\n          "bbox_xyxy": [\n            438.00000000000006,\n            397.0,\n            514.0,\n            442.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1918a97383232f6c_11",\n          "bbox_xyxy": [\n            537.75,\n            319.0,\n            576.25,\n            371.99999999999994\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1918a97383232f6c_12",\n          "bbox_xyxy": [\n            409.74999999999994,\n            122.75,\n            441.25,\n            177.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1918a97383232f6c_13",\n          "bbox_xyxy": [\n            344.0,\n            78.5,\n            391.0,\n            123.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1918a97383232f6c_14",\n          "bbox_xyxy": [\n            432.0,\n            48.24999999999999,\n            488.0,\n            87.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1918a97383232f6c_15",\n          "bbox_xyxy": [\n            560.25,\n            67.25,\n            600.75,\n            141.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1918a97383232f6c_16",\n          "bbox_xyxy": [\n            557.25,\n            167.0,\n            624.75,\n            215.99999999999997\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1918a97383232f6c_17",\n          "bbox_xyxy": [\n            235.25,\n            30.5,\n            289.75,\n            94.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1918a97383232f6c_18",\n          "bbox_xyxy": [\n            485.25,\n            183.75,\n            521.75,\n            239.24999999999997\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1918a97383232f6c_19",\n          "bbox_xyxy": [\n            51.25,\n            422.25,\n            102.75,\n            490.74999999999994\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/valid/images/IMG_3901_JPG.rf.adaa50af2aceb9d3d32674009fc7f452.jpg",\n        "size": 27066,\n        "compressed": 26646,\n        "offset": 632694690,\n        "crc32": "6c04a37f",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_1cd1bbb5c6afc891",\n      "relative_path": "images/IMG_2266_JPG.rf.f5348076bf5f73fde74fcbfabec393fc.jpg",\n      "sha256": "901b7a1f4d0d986e0da79db8a341cef14d0ffc1d8b02e54ab82eb8390df545ae",\n      "decoded_sha256": "5b2727e4c9925f3756911ef6697a1d12cd078aa45ad833ef92bc2a0aa3da791a",\n      "dhash": "2c3232717939b132",\n      "bytes": 26097,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2266_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "validation",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_1cd1bbb5c6afc891_0",\n          "bbox_xyxy": [\n            165.25000000000003,\n            58.25000000000003,\n            503.75,\n            588.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/valid/images/IMG_2266_JPG.rf.f5348076bf5f73fde74fcbfabec393fc.jpg",\n        "size": 26097,\n        "compressed": 25139,\n        "offset": 630904088,\n        "crc32": "25221d9e",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_1d0b1e4f1cada6db",\n      "relative_path": "images/IMG_2585_JPG.rf.177eac873ad4ae2228901ea51f7b08eb.jpg",\n      "sha256": "d771325e1f6188aa131a6fa3f80fda74236c9489619d156e25c5cf3163cce3ca",\n      "decoded_sha256": "cee3fe49e9de6b907396fe4ddaae3080579b7cef56179a3f3c519fbbb1b6a879",\n      "dhash": "0c7030b9393931b0",\n      "bytes": 25509,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2585_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "validation",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_1d0b1e4f1cada6db_0",\n          "bbox_xyxy": [\n            192.5,\n            72.00000000000003,\n            575.5,\n            640.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/valid/images/IMG_2585_JPG.rf.177eac873ad4ae2228901ea51f7b08eb.jpg",\n        "size": 25509,\n        "compressed": 24892,\n        "offset": 632254855,\n        "crc32": "5e8e54ba",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_1e2c62124ceb9f8f",\n      "relative_path": "images/IMG_1216_JPG.rf.12279563c2e883eaf053f60ac6a23939.jpg",\n      "sha256": "88b1c332c006f6fed144f8c8dc2862071aab2ec48111e619715db20e8f4ea0e1",\n      "decoded_sha256": "f50e878fb15e3d2bca8b8112646af069d0a88d9588c4b4f48a0c848d70aa0d47",\n      "dhash": "580a18120597172e",\n      "bytes": 52589,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_1216_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "validation",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_1e2c62124ceb9f8f_0",\n          "bbox_xyxy": [\n            171.5,\n            55.0,\n            198.5,\n            115.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2c62124ceb9f8f_1",\n          "bbox_xyxy": [\n            321.5,\n            32.25,\n            359.49999999999994,\n            79.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2c62124ceb9f8f_2",\n          "bbox_xyxy": [\n            531.75,\n            239.25,\n            563.25,\n            291.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2c62124ceb9f8f_3",\n          "bbox_xyxy": [\n            163.0,\n            199.0,\n            188.0,\n            255.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2c62124ceb9f8f_4",\n          "bbox_xyxy": [\n            350.0,\n            227.25,\n            380.0,\n            272.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2c62124ceb9f8f_5",\n          "bbox_xyxy": [\n            417.25,\n            184.00000000000003,\n            447.75,\n            238.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2c62124ceb9f8f_6",\n          "bbox_xyxy": [\n            455.0,\n            131.0,\n            485.99999999999994,\n            178.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2c62124ceb9f8f_7",\n          "bbox_xyxy": [\n            462.75,\n            39.5,\n            502.25,\n            75.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2c62124ceb9f8f_8",\n          "bbox_xyxy": [\n            295.25,\n            161.50000000000003,\n            322.75,\n            210.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2c62124ceb9f8f_9",\n          "bbox_xyxy": [\n            429.25000000000006,\n            311.75,\n            462.75,\n            359.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2c62124ceb9f8f_10",\n          "bbox_xyxy": [\n            179.5,\n            362.5,\n            216.50000000000003,\n            410.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2c62124ceb9f8f_11",\n          "bbox_xyxy": [\n            388.75,\n            352.5,\n            417.24999999999994,\n            410.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2c62124ceb9f8f_12",\n          "bbox_xyxy": [\n            226.25,\n            112.0,\n            257.75,\n            165.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2c62124ceb9f8f_13",\n          "bbox_xyxy": [\n            390.50000000000006,\n            39.25,\n            428.5,\n            76.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2c62124ceb9f8f_14",\n          "bbox_xyxy": [\n            113.0,\n            358.5,\n            150.0,\n            399.49999999999994\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2c62124ceb9f8f_15",\n          "bbox_xyxy": [\n            223.0,\n            211.25,\n            247.0,\n            272.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2c62124ceb9f8f_16",\n          "bbox_xyxy": [\n            173.0,\n            300.25,\n            204.00000000000003,\n            345.74999999999994\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2c62124ceb9f8f_17",\n          "bbox_xyxy": [\n            75.0,\n            385.5,\n            103.0,\n            432.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2c62124ceb9f8f_18",\n          "bbox_xyxy": [\n            553.25,\n            63.24999999999999,\n            581.75,\n            118.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2c62124ceb9f8f_19",\n          "bbox_xyxy": [\n            306.75,\n            309.0,\n            345.25,\n            366.99999999999994\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2c62124ceb9f8f_20",\n          "bbox_xyxy": [\n            361.75,\n            143.75000000000003,\n            399.25,\n            183.25000000000003\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2c62124ceb9f8f_21",\n          "bbox_xyxy": [\n            97.5,\n            80.5,\n            134.5,\n            122.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2c62124ceb9f8f_22",\n          "bbox_xyxy": [\n            368.75,\n            558.5,\n            402.24999999999994,\n            608.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2c62124ceb9f8f_23",\n          "bbox_xyxy": [\n            276.0,\n            260.0,\n            315.0,\n            306.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2c62124ceb9f8f_24",\n          "bbox_xyxy": [\n            244.25,\n            336.75000000000006,\n            271.75,\n            392.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2c62124ceb9f8f_25",\n          "bbox_xyxy": [\n            67.0,\n            594.25,\n            96.0,\n            639.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2c62124ceb9f8f_26",\n          "bbox_xyxy": [\n            500.74999999999994,\n            304.5,\n            527.25,\n            360.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2c62124ceb9f8f_27",\n          "bbox_xyxy": [\n            235.0,\n            568.0,\n            267.0,\n            615.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2c62124ceb9f8f_28",\n          "bbox_xyxy": [\n            57.75,\n            314.0,\n            97.25,\n            354.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2c62124ceb9f8f_29",\n          "bbox_xyxy": [\n            0.0,\n            218.25,\n            25.0,\n            271.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2c62124ceb9f8f_30",\n          "bbox_xyxy": [\n            86.25,\n            488.25,\n            120.75,\n            528.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2c62124ceb9f8f_31",\n          "bbox_xyxy": [\n            118.75,\n            240.75,\n            145.25,\n            299.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2c62124ceb9f8f_32",\n          "bbox_xyxy": [\n            326.25,\n            454.49999999999994,\n            350.75,\n            516.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2c62124ceb9f8f_33",\n          "bbox_xyxy": [\n            288.25,\n            546.5,\n            319.75,\n            591.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2c62124ceb9f8f_34",\n          "bbox_xyxy": [\n            573.0,\n            390.5,\n            606.0,\n            432.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2c62124ceb9f8f_35",\n          "bbox_xyxy": [\n            468.5,\n            571.0,\n            506.5,\n            614.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2c62124ceb9f8f_36",\n          "bbox_xyxy": [\n            249.74999999999997,\n            27.0,\n            284.25,\n            72.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2c62124ceb9f8f_37",\n          "bbox_xyxy": [\n            144.75,\n            453.5,\n            166.25,\n            504.49999999999994\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2c62124ceb9f8f_38",\n          "bbox_xyxy": [\n            275.5,\n            441.0,\n            303.5,\n            494.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2c62124ceb9f8f_39",\n          "bbox_xyxy": [\n            5.0,\n            304.5,\n            37.0,\n            353.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2c62124ceb9f8f_40",\n          "bbox_xyxy": [\n            506.5,\n            478.5,\n            541.5,\n            516.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2c62124ceb9f8f_41",\n          "bbox_xyxy": [\n            72.25,\n            200.49999999999997,\n            94.75,\n            257.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2c62124ceb9f8f_42",\n          "bbox_xyxy": [\n            557.75,\n            605.0,\n            582.25,\n            640.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2c62124ceb9f8f_43",\n          "bbox_xyxy": [\n            1.0000000000000009,\n            395.0,\n            26.0,\n            445.99999999999994\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2c62124ceb9f8f_44",\n          "bbox_xyxy": [\n            198.25,\n            462.5,\n            224.74999999999997,\n            521.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2c62124ceb9f8f_45",\n          "bbox_xyxy": [\n            515.75,\n            149.75,\n            548.25,\n            187.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2c62124ceb9f8f_46",\n          "bbox_xyxy": [\n            611.0,\n            503.24999999999994,\n            640.0,\n            547.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2c62124ceb9f8f_47",\n          "bbox_xyxy": [\n            19.0,\n            98.75,\n            51.99999999999999,\n            141.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2c62124ceb9f8f_48",\n          "bbox_xyxy": [\n            0.25000000000000133,\n            496.25,\n            30.75,\n            531.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2c62124ceb9f8f_49",\n          "bbox_xyxy": [\n            437.5,\n            472.25,\n            469.5,\n            512.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2c62124ceb9f8f_50",\n          "bbox_xyxy": [\n            397.25,\n            79.25,\n            426.75000000000006,\n            128.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2c62124ceb9f8f_51",\n          "bbox_xyxy": [\n            416.50000000000006,\n            0.0,\n            442.5,\n            22.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2c62124ceb9f8f_52",\n          "bbox_xyxy": [\n            329.0,\n            0.2500000000000002,\n            361.99999999999994,\n            19.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2c62124ceb9f8f_53",\n          "bbox_xyxy": [\n            14.25,\n            613.25,\n            37.75,\n            639.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2c62124ceb9f8f_54",\n          "bbox_xyxy": [\n            625.0,\n            315.75,\n            640.0,\n            353.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/valid/images/IMG_1216_JPG.rf.12279563c2e883eaf053f60ac6a23939.jpg",\n        "size": 52589,\n        "compressed": 52436,\n        "offset": 629532423,\n        "crc32": "385c5ebd",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_1e97c64af149f99e",\n      "relative_path": "images/IMG_2127_JPG.rf.e42115be8ff93778efa39104fe8167ca.jpg",\n      "sha256": "52f0661e90eacbe2196c1c534b9cbe3ae2646d0d21e880940377b64e710b5788",\n      "decoded_sha256": "c005cfede2d8119c866ae138f49b76e2dc1c43c14cd7d72cb37231a97864bbf8",\n      "dhash": "6872713171717212",\n      "bytes": 26000,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2127_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "validation",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_1e97c64af149f99e_0",\n          "bbox_xyxy": [\n            146.25,\n            36.000000000000014,\n            480.75,\n            546.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/valid/images/IMG_2127_JPG.rf.e42115be8ff93778efa39104fe8167ca.jpg",\n        "size": 26000,\n        "compressed": 25301,\n        "offset": 630552906,\n        "crc32": "54a68ad8",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_1fef1ffa54893b98",\n      "relative_path": "images/IMG_2362_JPG.rf.821b35a82f3b4a4e921e7420eb2e6792.jpg",\n      "sha256": "ba475e7ca7b87c5a2cae8e28050f52a404d9d1a820aa717228b107ed3634878f",\n      "decoded_sha256": "63c511d16a93b28ef331300fed494bf65208ecc481a912d432713826532ca0fc",\n      "dhash": "cd7071697971b193",\n      "bytes": 26480,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2362_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "validation",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_1fef1ffa54893b98_0",\n          "bbox_xyxy": [\n            178.49999999999997,\n            50.50000000000001,\n            474.5,\n            569.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/valid/images/IMG_2362_JPG.rf.821b35a82f3b4a4e921e7420eb2e6792.jpg",\n        "size": 26480,\n        "compressed": 25779,\n        "offset": 631245495,\n        "crc32": "c3621d14",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_22340ee80593e4f8",\n      "relative_path": "images/IMG_1029_JPG.rf.4b0e156ae3d2dc2e7c03d797dbe77774.jpg",\n      "sha256": "085006785ef405d12e4c7021d1d99f670e743f776263f2f1b7f2d58ecc8223e9",\n      "decoded_sha256": "51545aa551601faeed340806e8b986d7e92f237548e41fd486b324844864f26a",\n      "dhash": "0e0d150f2543b70d",\n      "bytes": 47633,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_1029_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "validation",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_22340ee80593e4f8_0",\n          "bbox_xyxy": [\n            435.74999999999994,\n            77.25,\n            480.25,\n            118.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_22340ee80593e4f8_1",\n          "bbox_xyxy": [\n            510.25,\n            46.49999999999999,\n            564.75,\n            80.49999999999999\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_22340ee80593e4f8_2",\n          "bbox_xyxy": [\n            186.25,\n            417.75,\n            213.75,\n            487.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_22340ee80593e4f8_3",\n          "bbox_xyxy": [\n            359.75,\n            544.0,\n            382.25,\n            608.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_22340ee80593e4f8_4",\n          "bbox_xyxy": [\n            584.5,\n            79.0,\n            633.5,\n            121.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_22340ee80593e4f8_5",\n          "bbox_xyxy": [\n            375.0,\n            255.25,\n            395.99999999999994,\n            314.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_22340ee80593e4f8_6",\n          "bbox_xyxy": [\n            357.0,\n            450.75,\n            383.0,\n            521.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_22340ee80593e4f8_7",\n          "bbox_xyxy": [\n            262.5,\n            489.75,\n            288.5,\n            568.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_22340ee80593e4f8_8",\n          "bbox_xyxy": [\n            563.25,\n            441.0,\n            602.75,\n            506.99999999999994\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_22340ee80593e4f8_9",\n          "bbox_xyxy": [\n            137.75,\n            555.5,\n            171.25,\n            626.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_22340ee80593e4f8_10",\n          "bbox_xyxy": [\n            191.25,\n            477.5,\n            233.75,\n            521.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_22340ee80593e4f8_11",\n          "bbox_xyxy": [\n            496.25,\n            427.5,\n            536.75,\n            455.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_22340ee80593e4f8_12",\n          "bbox_xyxy": [\n            620.25,\n            346.0,\n            639.75,\n            418.00000000000006\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_22340ee80593e4f8_13",\n          "bbox_xyxy": [\n            91.25,\n            510.25000000000006,\n            135.75,\n            558.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_22340ee80593e4f8_14",\n          "bbox_xyxy": [\n            392.75,\n            49.5,\n            428.24999999999994,\n            113.50000000000001\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_22340ee80593e4f8_15",\n          "bbox_xyxy": [\n            295.5,\n            195.75,\n            336.5,\n            247.24999999999997\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_22340ee80593e4f8_16",\n          "bbox_xyxy": [\n            554.0,\n            244.5,\n            582.0,\n            324.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_22340ee80593e4f8_17",\n          "bbox_xyxy": [\n            453.25,\n            574.0,\n            495.75,\n            616.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_22340ee80593e4f8_18",\n          "bbox_xyxy": [\n            579.75,\n            582.5,\n            597.25,\n            639.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_22340ee80593e4f8_19",\n          "bbox_xyxy": [\n            525.5,\n            487.0,\n            573.5,\n            543.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_22340ee80593e4f8_20",\n          "bbox_xyxy": [\n            504.25000000000006,\n            318.0,\n            537.75,\n            385.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_22340ee80593e4f8_21",\n          "bbox_xyxy": [\n            406.25,\n            596.75,\n            450.75,\n            639.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_22340ee80593e4f8_22",\n          "bbox_xyxy": [\n            235.0,\n            330.74999999999994,\n            262.0,\n            387.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_22340ee80593e4f8_23",\n          "bbox_xyxy": [\n            441.0,\n            320.25,\n            463.00000000000006,\n            391.75000000000006\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_22340ee80593e4f8_24",\n          "bbox_xyxy": [\n            291.0,\n            347.25,\n            345.0,\n            384.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_22340ee80593e4f8_25",\n          "bbox_xyxy": [\n            397.0,\n            367.5,\n            435.0,\n            424.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_22340ee80593e4f8_26",\n          "bbox_xyxy": [\n            235.25,\n            383.5,\n            283.75,\n            450.50000000000006\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_22340ee80593e4f8_27",\n          "bbox_xyxy": [\n            290.0,\n            441.0,\n            342.0,\n            514.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_22340ee80593e4f8_28",\n          "bbox_xyxy": [\n            445.0,\n            518.75,\n            502.0,\n            556.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_22340ee80593e4f8_29",\n          "bbox_xyxy": [\n            458.25,\n            247.0,\n            480.75,\n            316.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_22340ee80593e4f8_30",\n          "bbox_xyxy": [\n            422.75,\n            137.25,\n            469.25000000000006,\n            181.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/valid/images/IMG_1029_JPG.rf.4b0e156ae3d2dc2e7c03d797dbe77774.jpg",\n        "size": 47633,\n        "compressed": 47360,\n        "offset": 628960920,\n        "crc32": "defcb9d2",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_22cb3e7f8bc6679f",\n      "relative_path": "images/IMG_2298_JPG.rf.e90f1002370ccd5d4fcccac9b801abb0.jpg",\n      "sha256": "e914786414e465f0d3d17700179d69c9692986307790165969216ef16f19d4c2",\n      "decoded_sha256": "3584782969c77c0c386279980b3b04f2aa808c2259d1853fce3975a30d6460ae",\n      "dhash": "041030707078b132",\n      "bytes": 27020,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2298_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "validation",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_22cb3e7f8bc6679f_0",\n          "bbox_xyxy": [\n            211.74999999999997,\n            61.000000000000014,\n            506.25,\n            570.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/valid/images/IMG_2298_JPG.rf.e90f1002370ccd5d4fcccac9b801abb0.jpg",\n        "size": 27020,\n        "compressed": 24496,\n        "offset": 631010503,\n        "crc32": "f695e0a0",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_247ce8f4dd61227d",\n      "relative_path": "images/IMG_4148_JPG.rf.40faad70375a3a5ce5040c6975e057cd.jpg",\n      "sha256": "58162b3bcede7eb0e8068c2c25531694ccebb60a61d528cf40b14a5dffb8b8d4",\n      "decoded_sha256": "5b7fcbe854ad458af992f12fb398bd501431d2e2904bffb46cc21252e1f25aa6",\n      "dhash": "60d254ecdcf89a9e",\n      "bytes": 46589,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_4148_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "validation",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_247ce8f4dd61227d_0",\n          "bbox_xyxy": [\n            158.25,\n            217.75,\n            203.75,\n            242.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_247ce8f4dd61227d_1",\n          "bbox_xyxy": [\n            168.25,\n            315.5,\n            189.74999999999997,\n            354.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_247ce8f4dd61227d_2",\n          "bbox_xyxy": [\n            247.75000000000003,\n            266.0,\n            289.25,\n            304.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_247ce8f4dd61227d_3",\n          "bbox_xyxy": [\n            168.5,\n            406.99999999999994,\n            201.5,\n            434.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_247ce8f4dd61227d_4",\n          "bbox_xyxy": [\n            170.75,\n            501.0,\n            218.25,\n            521.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_247ce8f4dd61227d_5",\n          "bbox_xyxy": [\n            367.5,\n            423.25,\n            407.5,\n            455.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_247ce8f4dd61227d_6",\n          "bbox_xyxy": [\n            385.75,\n            384.75,\n            429.25,\n            398.24999999999994\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_247ce8f4dd61227d_7",\n          "bbox_xyxy": [\n            437.75,\n            295.75,\n            473.24999999999994,\n            325.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_247ce8f4dd61227d_8",\n          "bbox_xyxy": [\n            441.25,\n            160.0,\n            473.75,\n            201.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_247ce8f4dd61227d_9",\n          "bbox_xyxy": [\n            250.0,\n            416.25,\n            268.0,\n            456.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/valid/images/IMG_4148_JPG.rf.40faad70375a3a5ce5040c6975e057cd.jpg",\n        "size": 46589,\n        "compressed": 46405,\n        "offset": 632966771,\n        "crc32": "ef04e7d2",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_26683ad92cea74b0",\n      "relative_path": "images/IMG_2278_JPG.rf.354f092e70c2ecd377504fee819d0055.jpg",\n      "sha256": "47838cf3d727250dc9614b680658fa81cf6ee7130190d59a3a8e68efdad4331b",\n      "decoded_sha256": "2929ed3f4ca6f3bba42e6400ab08318a131d38cadf632b6f60a0658f0f8ced40",\n      "dhash": "24b9317171b13919",\n      "bytes": 26993,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2278_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "validation",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_26683ad92cea74b0_0",\n          "bbox_xyxy": [\n            171.50000000000003,\n            33.49999999999998,\n            510.5,\n            627.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/valid/images/IMG_2278_JPG.rf.354f092e70c2ecd377504fee819d0055.jpg",\n        "size": 26993,\n        "compressed": 26198,\n        "offset": 630929400,\n        "crc32": "b58595ef",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_27d8228821955104",\n      "relative_path": "images/6095710224539913295_jpg.rf.0838d48f2d0d5b7e541bdea2efc445bb.jpg",\n      "sha256": "d8c7a187247b87c626e02edbff447c3763b44f0e4b778f70f6bf40e8213ddc99",\n      "decoded_sha256": "0bc8dae28511c56c4ccf04d83822e52b745ad25881e3af4cbf54705cc93dc03c",\n      "dhash": "64ac3c1e160c0406",\n      "bytes": 155209,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "6095710224539913295_jpg",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "validation",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_27d8228821955104_0",\n          "bbox_xyxy": [\n            323.00000000000006,\n            454.75,\n            354.0,\n            475.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_27d8228821955104_1",\n          "bbox_xyxy": [\n            294.75,\n            370.5,\n            317.25,\n            397.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_27d8228821955104_2",\n          "bbox_xyxy": [\n            182.00000000000003,\n            7.5,\n            195.0,\n            44.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/valid/images/6095710224539913295_jpg.rf.0838d48f2d0d5b7e541bdea2efc445bb.jpg",\n        "size": 155209,\n        "compressed": 155105,\n        "offset": 627336515,\n        "crc32": "992f28dd",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_28f0a264588074ea",\n      "relative_path": "images/IMG_2437_JPG.rf.98f99ef5a0e646f2136df4b6ad0ec30f.jpg",\n      "sha256": "63ab1b1a2a2eeeb805e4d63acef6e3530c1c2503537d3bb81b64cb53779f5d60",\n      "decoded_sha256": "ab785e01cbbb8b994ea88d4109d6a27d36d3c4b6a7a89c54821edaef2839430c",\n      "dhash": "2413317179793152",\n      "bytes": 23201,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2437_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "validation",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_28f0a264588074ea_0",\n          "bbox_xyxy": [\n            172.00000000000003,\n            67.25000000000001,\n            545.0,\n            616.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/valid/images/IMG_2437_JPG.rf.98f99ef5a0e646f2136df4b6ad0ec30f.jpg",\n        "size": 23201,\n        "compressed": 20889,\n        "offset": 631612148,\n        "crc32": "1267cdb4",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_29a787dcdfe9e940",\n      "relative_path": "images/IMG_1912_JPG.rf.9fe36e5bedf719ce2cd22e159462732d.jpg",\n      "sha256": "9fb3932f7065bd02ecc1b9650482a0cb79dfb32611d2fdace25539425156882d",\n      "decoded_sha256": "ff7f2d05213402af1943587ee80dc3838091d548b8acda472c139cffb5ed0df6",\n      "dhash": "4872716969717090",\n      "bytes": 25323,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_1912_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "validation",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_29a787dcdfe9e940_0",\n          "bbox_xyxy": [\n            135.0,\n            49.25000000000001,\n            475.0,\n            572.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/valid/images/IMG_1912_JPG.rf.9fe36e5bedf719ce2cd22e159462732d.jpg",\n        "size": 25323,\n        "compressed": 24462,\n        "offset": 629807065,\n        "crc32": "0cd45bd3",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_2a16d70e87ef0510",\n      "relative_path": "images/IMG_2261_JPG.rf.cf5ebb67f7b4afee7f0ce8e379ff0c48.jpg",\n      "sha256": "265218a9ea92152f3c4e60cc532c13cc916bae394a24c996205db6176af232cb",\n      "decoded_sha256": "b1279670f07d740b1e28109174a5d1477c1f65764e1109f60f83e6d040b2099b",\n      "dhash": "6931313979713232",\n      "bytes": 26176,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2261_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "validation",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_2a16d70e87ef0510_0",\n          "bbox_xyxy": [\n            151.74999999999997,\n            39.49999999999999,\n            486.25,\n            579.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/valid/images/IMG_2261_JPG.rf.cf5ebb67f7b4afee7f0ce8e379ff0c48.jpg",\n        "size": 26176,\n        "compressed": 25175,\n        "offset": 630878740,\n        "crc32": "216435b5",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_2a3fcf2f8a37672f",\n      "relative_path": "images/IMG_2686_JPG.rf.01be43d2093645264df2c37ee6b6b54c.jpg",\n      "sha256": "b2ea7e880f10a8f05ebf87816a96b0d184f26bd88598860a510e39eee97e5f51",\n      "decoded_sha256": "850661b29be3df91872a848e9d7f8d256d4ecfe97b7665c18783dc9c202ef74d",\n      "dhash": "693131797871b130",\n      "bytes": 30481,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2686_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "validation",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_2a3fcf2f8a37672f_0",\n          "bbox_xyxy": [\n            134.00000000000003,\n            0.7499999999999929,\n            502.0,\n            615.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/valid/images/IMG_2686_JPG.rf.01be43d2093645264df2c37ee6b6b54c.jpg",\n        "size": 30481,\n        "compressed": 29953,\n        "offset": 632496572,\n        "crc32": "3b432720",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_2ccacb88edb995d4",\n      "relative_path": "images/IMG_2560_JPG.rf.ed843a13efce168cf0a41a1b797cc368.jpg",\n      "sha256": "83394baa2b7061bcd0497f39f97b40c7e2be76a25bfd469ec5e1698415eb53f5",\n      "decoded_sha256": "ca35fdf912dd67067bbb84acbc95633f1f9f68d8f8ace40ebbbe64a27d8e102d",\n      "dhash": "0d51716878703133",\n      "bytes": 26921,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2560_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "validation",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_2ccacb88edb995d4_0",\n          "bbox_xyxy": [\n            163.25000000000003,\n            43.75,\n            548.75,\n            579.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/valid/images/IMG_2560_JPG.rf.ed843a13efce168cf0a41a1b797cc368.jpg",\n        "size": 26921,\n        "compressed": 26332,\n        "offset": 632152206,\n        "crc32": "cd035f20",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_31575d4abd36614e",\n      "relative_path": "images/IMG_2058_JPG.rf.ee46acff7f4d3770236e650977e31c65.jpg",\n      "sha256": "481f3c34cadf1e0c1746f409b7090745703c70760a13d31eed20ca494f44dbac",\n      "decoded_sha256": "5efe5f2325650965dc636ff96a7da7a0ddd6148aecf02eff5ea7d5689c70fe86",\n      "dhash": "c9f27179797171b0",\n      "bytes": 28148,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2058_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "validation",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_31575d4abd36614e_0",\n          "bbox_xyxy": [\n            123.75000000000003,\n            22.250000000000014,\n            530.25,\n            598.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/valid/images/IMG_2058_JPG.rf.ee46acff7f4d3770236e650977e31c65.jpg",\n        "size": 28148,\n        "compressed": 27636,\n        "offset": 630356529,\n        "crc32": "c21dc6a9",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_31bb9f8cb586e50c",\n      "relative_path": "images/IMG_2728_JPG.rf.57660745dbf8828d0b88244bacde5bd0.jpg",\n      "sha256": "bc1d7515ed74e6ea62ec76c5abaf6f8fbf2d9f51d7f1aefae2bd4f6a4faf843d",\n      "decoded_sha256": "7ce4eaba590d92b43bd1748a777c12f8725a207ba4949c69e414f8cd3474a95c",\n      "dhash": "68b0f169697931b2",\n      "bytes": 31448,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2728_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "validation",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_31bb9f8cb586e50c_0",\n          "bbox_xyxy": [\n            73.25,\n            22.75000000000002,\n            500.75,\n            639.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/valid/images/IMG_2728_JPG.rf.57660745dbf8828d0b88244bacde5bd0.jpg",\n        "size": 31448,\n        "compressed": 30538,\n        "offset": 632581089,\n        "crc32": "5f66fbd0",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_322c5319a02be8ef",\n      "relative_path": "images/IMG_0968_JPG.rf.d32f9daee2a5c62787769c57070233e3.jpg",\n      "sha256": "2e44191a6bcc47c93de550d89adf964fbed620fa8bd8b7615eefe674e475ea99",\n      "decoded_sha256": "e4a18246d5fdbf09146a686edee476d40cc8da7164abc5631dc28317000deefc",\n      "dhash": "482b858c21851350",\n      "bytes": 71502,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_0968_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "validation",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_322c5319a02be8ef_0",\n          "bbox_xyxy": [\n            528.0,\n            168.0,\n            573.0,\n            218.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_322c5319a02be8ef_1",\n          "bbox_xyxy": [\n            500.99999999999994,\n            264.75,\n            565.0,\n            290.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_322c5319a02be8ef_2",\n          "bbox_xyxy": [\n            360.25000000000006,\n            228.25,\n            443.75,\n            250.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_322c5319a02be8ef_3",\n          "bbox_xyxy": [\n            337.0,\n            101.5,\n            407.0,\n            161.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_322c5319a02be8ef_4",\n          "bbox_xyxy": [\n            213.0,\n            89.50000000000001,\n            308.0,\n            121.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_322c5319a02be8ef_5",\n          "bbox_xyxy": [\n            184.74999999999997,\n            116.74999999999999,\n            239.25,\n            162.24999999999997\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_322c5319a02be8ef_6",\n          "bbox_xyxy": [\n            144.0,\n            168.75,\n            200.0,\n            220.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_322c5319a02be8ef_7",\n          "bbox_xyxy": [\n            36.75,\n            122.5,\n            85.24999999999999,\n            182.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_322c5319a02be8ef_8",\n          "bbox_xyxy": [\n            18.249999999999996,\n            309.75,\n            73.75,\n            368.24999999999994\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_322c5319a02be8ef_9",\n          "bbox_xyxy": [\n            61.5,\n            369.49999999999994,\n            134.5,\n            413.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_322c5319a02be8ef_10",\n          "bbox_xyxy": [\n            122.0,\n            433.0,\n            186.0,\n            502.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_322c5319a02be8ef_11",\n          "bbox_xyxy": [\n            78.75,\n            460.74999999999994,\n            124.25,\n            525.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_322c5319a02be8ef_12",\n          "bbox_xyxy": [\n            156.0,\n            554.25,\n            215.0,\n            602.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_322c5319a02be8ef_13",\n          "bbox_xyxy": [\n            258.5,\n            572.25,\n            298.5,\n            637.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_322c5319a02be8ef_14",\n          "bbox_xyxy": [\n            538.75,\n            482.5,\n            589.25,\n            537.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_322c5319a02be8ef_15",\n          "bbox_xyxy": [\n            532.75,\n            427.25,\n            579.25,\n            477.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_322c5319a02be8ef_16",\n          "bbox_xyxy": [\n            466.5,\n            459.5,\n            515.5,\n            512.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_322c5319a02be8ef_17",\n          "bbox_xyxy": [\n            566.0,\n            369.25,\n            617.0,\n            416.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_322c5319a02be8ef_18",\n          "bbox_xyxy": [\n            470.25000000000006,\n            373.25,\n            513.75,\n            438.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_322c5319a02be8ef_19",\n          "bbox_xyxy": [\n            376.75000000000006,\n            322.75,\n            432.25,\n            393.24999999999994\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_322c5319a02be8ef_20",\n          "bbox_xyxy": [\n            428.24999999999994,\n            319.0,\n            479.75,\n            365.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_322c5319a02be8ef_21",\n          "bbox_xyxy": [\n            574.0,\n            316.0,\n            640.0,\n            336.99999999999994\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_322c5319a02be8ef_22",\n          "bbox_xyxy": [\n            262.75,\n            313.5,\n            337.25,\n            340.50000000000006\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_322c5319a02be8ef_23",\n          "bbox_xyxy": [\n            226.75,\n            316.0,\n            272.25,\n            366.99999999999994\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_322c5319a02be8ef_24",\n          "bbox_xyxy": [\n            209.0,\n            282.5,\n            256.0,\n            317.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_322c5319a02be8ef_25",\n          "bbox_xyxy": [\n            103.0,\n            186.25,\n            144.0,\n            245.75000000000003\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_322c5319a02be8ef_26",\n          "bbox_xyxy": [\n            111.25,\n            249.25,\n            207.75,\n            292.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_322c5319a02be8ef_27",\n          "bbox_xyxy": [\n            251.25,\n            360.74999999999994,\n            297.75,\n            427.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_322c5319a02be8ef_28",\n          "bbox_xyxy": [\n            379.49999999999994,\n            536.25,\n            448.5,\n            568.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_322c5319a02be8ef_29",\n          "bbox_xyxy": [\n            547.5,\n            43.75,\n            602.5,\n            96.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/valid/images/IMG_0968_JPG.rf.d32f9daee2a5c62787769c57070233e3.jpg",\n        "size": 71502,\n        "compressed": 71369,\n        "offset": 628829876,\n        "crc32": "328f4628",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_340d00d000c5edaa",\n      "relative_path": "images/IMG_1012_JPG.rf.712fd1c6c1b90a66c3a6f2a1e9293c31.jpg",\n      "sha256": "2a437a8c245d632c1a22da26fb58f0aee9dbbcb68cff04c291c90d7df2f6f529",\n      "decoded_sha256": "caf26dbede8d03396511ca1e47ea79b5d59b993d8111bc318c1ea5bb3130a365",\n      "dhash": "284c88506c7b3b1b",\n      "bytes": 59459,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_1012_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "validation",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_340d00d000c5edaa_0",\n          "bbox_xyxy": [\n            454.49999999999994,\n            46.5,\n            491.5,\n            128.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_340d00d000c5edaa_1",\n          "bbox_xyxy": [\n            402.5,\n            70.5,\n            442.5,\n            154.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_340d00d000c5edaa_2",\n          "bbox_xyxy": [\n            456.99999999999994,\n            185.5,\n            491.0,\n            261.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_340d00d000c5edaa_3",\n          "bbox_xyxy": [\n            393.24999999999994,\n            180.0,\n            437.75,\n            246.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_340d00d000c5edaa_4",\n          "bbox_xyxy": [\n            335.74999999999994,\n            235.0,\n            397.25,\n            277.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_340d00d000c5edaa_5",\n          "bbox_xyxy": [\n            440.99999999999994,\n            336.25,\n            500.0,\n            396.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_340d00d000c5edaa_6",\n          "bbox_xyxy": [\n            348.0,\n            392.75,\n            415.0,\n            479.25000000000006\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_340d00d000c5edaa_7",\n          "bbox_xyxy": [\n            385.74999999999994,\n            316.25,\n            442.25,\n            383.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_340d00d000c5edaa_8",\n          "bbox_xyxy": [\n            257.75,\n            395.74999999999994,\n            305.25,\n            485.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_340d00d000c5edaa_9",\n          "bbox_xyxy": [\n            179.5,\n            359.75,\n            221.50000000000003,\n            457.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_340d00d000c5edaa_10",\n          "bbox_xyxy": [\n            9.75,\n            287.25,\n            53.24999999999999,\n            388.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_340d00d000c5edaa_11",\n          "bbox_xyxy": [\n            33.5,\n            166.75,\n            90.5,\n            219.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_340d00d000c5edaa_12",\n          "bbox_xyxy": [\n            87.75,\n            93.75,\n            155.25,\n            167.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_340d00d000c5edaa_13",\n          "bbox_xyxy": [\n            153.75,\n            156.75,\n            177.25,\n            254.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_340d00d000c5edaa_14",\n          "bbox_xyxy": [\n            167.75,\n            264.5,\n            222.25,\n            321.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_340d00d000c5edaa_15",\n          "bbox_xyxy": [\n            240.0,\n            292.5,\n            281.0,\n            342.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_340d00d000c5edaa_16",\n          "bbox_xyxy": [\n            211.5,\n            59.249999999999986,\n            262.5,\n            124.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/valid/images/IMG_1012_JPG.rf.712fd1c6c1b90a66c3a6f2a1e9293c31.jpg",\n        "size": 59459,\n        "compressed": 59329,\n        "offset": 628901418,\n        "crc32": "8e5ed89d",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_34315fee6d85c210",\n      "relative_path": "images/IMG_2299_JPG.rf.080624817749bcc480a05b98b5ce4772.jpg",\n      "sha256": "c91ba36ff4aa774fbc34ea0a26b3aa01e91b70fe5a783d427c7d93749d2f909f",\n      "decoded_sha256": "5d822c311aee854547ac505187c2aa60dd34ebca9542117e822e4fbe599acfac",\n      "dhash": "0812313179b13152",\n      "bytes": 23700,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2299_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "validation",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_34315fee6d85c210_0",\n          "bbox_xyxy": [\n            183.25,\n            68.25000000000003,\n            556.75,\n            575.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/valid/images/IMG_2299_JPG.rf.080624817749bcc480a05b98b5ce4772.jpg",\n        "size": 23700,\n        "compressed": 20997,\n        "offset": 631035172,\n        "crc32": "4b13b627",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_349ec21352f6ebf2",\n      "relative_path": "images/IMG_4052_JPG.rf.36655c8aa0b4fafa73de8a2876125120.jpg",\n      "sha256": "589ddd3f2dad434e46805b35b077534187795ec0dd89a89a58be494d0d01ebed",\n      "decoded_sha256": "5de11c7abce3dee4ea7da46dcf95069f115963f1903a63145b5bb0c19581e8df",\n      "dhash": "49a958f8fea4b871",\n      "bytes": 54463,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_4052_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "validation",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_349ec21352f6ebf2_0",\n          "bbox_xyxy": [\n            159.5,\n            375.0,\n            198.49999999999997,\n            400.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_349ec21352f6ebf2_1",\n          "bbox_xyxy": [\n            291.5,\n            390.50000000000006,\n            314.5,\n            431.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_349ec21352f6ebf2_2",\n          "bbox_xyxy": [\n            389.49999999999994,\n            501.25,\n            421.5,\n            531.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_349ec21352f6ebf2_3",\n          "bbox_xyxy": [\n            393.75,\n            348.5,\n            435.25000000000006,\n            373.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_349ec21352f6ebf2_4",\n          "bbox_xyxy": [\n            509.49999999999994,\n            335.75,\n            538.5,\n            373.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_349ec21352f6ebf2_5",\n          "bbox_xyxy": [\n            230.5,\n            457.5,\n            273.5,\n            484.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/valid/images/IMG_4052_JPG.rf.36655c8aa0b4fafa73de8a2876125120.jpg",\n        "size": 54463,\n        "compressed": 53836,\n        "offset": 632721509,\n        "crc32": "477ce6c8",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_34faa1d6c7e6e468",\n      "relative_path": "images/IMG_1947_JPG.rf.fa0796d7035fc07b38fec7e43ffc9d86.jpg",\n      "sha256": "3b0e574f1b4ba609b171027734e9e079af4c2eb09dcdad7a7b20a218528e75cf",\n      "decoded_sha256": "6812c4b572a9dc7357b19e17668a45513b5086a6529bb72931b1ee3734d632ac",\n      "dhash": "0812717961713132",\n      "bytes": 29384,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_1947_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "validation",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_34faa1d6c7e6e468_0",\n          "bbox_xyxy": [\n            176.0,\n            43.500000000000014,\n            501.0,\n            572.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/valid/images/IMG_1947_JPG.rf.fa0796d7035fc07b38fec7e43ffc9d86.jpg",\n        "size": 29384,\n        "compressed": 28651,\n        "offset": 629856696,\n        "crc32": "835a603c",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_3581dc4c449ac601",\n      "relative_path": "images/IMG_2011_JPG.rf.a9df6df0288da332aaa6009f8989fb47.jpg",\n      "sha256": "8cbad8881fe3c5e4dabe6acc26346baa7a93a7e69e70275202a429ed18a27016",\n      "decoded_sha256": "d5c96ebc14e82fa4bdb35592297580961da873f2977178daa7884988b80a2339",\n      "dhash": "843270727131325a",\n      "bytes": 21787,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2011_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "validation",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_3581dc4c449ac601_0",\n          "bbox_xyxy": [\n            204.5,\n            57.5,\n            458.49999999999994,\n            556.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/valid/images/IMG_2011_JPG.rf.a9df6df0288da332aaa6009f8989fb47.jpg",\n        "size": 21787,\n        "compressed": 20874,\n        "offset": 630082536,\n        "crc32": "c17638ca",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_374fc184290d5c72",\n      "relative_path": "images/IMG_2235_JPG.rf.57ec126763e15563675ddc8502f36ad3.jpg",\n      "sha256": "c09f6eb10a6cd8ad29b99a15c216147c3a967aec32ff764ead5ba38c076e98a9",\n      "decoded_sha256": "17908bfe24c18004d4bf199eb3b61f0d6225cd83955b4750df58bc2b5fa2cadd",\n      "dhash": "7432797979713044",\n      "bytes": 31023,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2235_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "validation",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_374fc184290d5c72_0",\n          "bbox_xyxy": [\n            157.0,\n            0.0,\n            493.0,\n            540.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/valid/images/IMG_2235_JPG.rf.57ec126763e15563675ddc8502f36ad3.jpg",\n        "size": 31023,\n        "compressed": 30020,\n        "offset": 630792545,\n        "crc32": "641bb4fa",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_389ef48f221fb56b",\n      "relative_path": "images/IMG_2640_JPG.rf.9343fdd1dc6f921d8e90d26e02c172ac.jpg",\n      "sha256": "20efd9c416956c689b3866c944de49f5ed1c0fbe14f9be3b16f0e268d1c43f9b",\n      "decoded_sha256": "9047b2120bfd47e9162df94ed746b893b045f5deb40483c3dbc8d99720582836",\n      "dhash": "0a72727971717032",\n      "bytes": 24185,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2640_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "validation",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_389ef48f221fb56b_0",\n          "bbox_xyxy": [\n            156.5,\n            51.499999999999986,\n            452.5,\n            606.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/valid/images/IMG_2640_JPG.rf.9343fdd1dc6f921d8e90d26e02c172ac.jpg",\n        "size": 24185,\n        "compressed": 22605,\n        "offset": 632359446,\n        "crc32": "0cf2e3d7",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_38b9c68d8df822e3",\n      "relative_path": "images/6095710224539913562_jpg.rf.7099237d33b0eb5cc5c381c8b734deb5.jpg",\n      "sha256": "7c8cbffe92ffec1ccb87c3e3966df1c1349793a5e06e95ba7235ca73a8a2236b",\n      "decoded_sha256": "a4d8b66d739cda37aff036f85f43322db599b3d29a71fd84d57d52f6a38881a3",\n      "dhash": "4fcfbf8f1fafadcf",\n      "bytes": 136973,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "6095710224539913562_jpg",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "validation",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_38b9c68d8df822e3_0",\n          "bbox_xyxy": [\n            174.74999999999997,\n            152.49999999999997,\n            209.25,\n            185.49999999999997\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_38b9c68d8df822e3_1",\n          "bbox_xyxy": [\n            154.75,\n            416.25,\n            179.25,\n            440.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_38b9c68d8df822e3_2",\n          "bbox_xyxy": [\n            522.75,\n            450.50000000000006,\n            559.25,\n            473.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_38b9c68d8df822e3_3",\n          "bbox_xyxy": [\n            122.5,\n            593.0,\n            159.5,\n            619.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_38b9c68d8df822e3_4",\n          "bbox_xyxy": [\n            465.0,\n            389.25,\n            502.0,\n            433.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_38b9c68d8df822e3_5",\n          "bbox_xyxy": [\n            443.25,\n            559.5,\n            481.75,\n            598.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_38b9c68d8df822e3_6",\n          "bbox_xyxy": [\n            585.75,\n            385.25,\n            628.25,\n            416.75000000000006\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_38b9c68d8df822e3_7",\n          "bbox_xyxy": [\n            401.99999999999994,\n            355.5,\n            436.0,\n            404.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_38b9c68d8df822e3_8",\n          "bbox_xyxy": [\n            459.0,\n            442.75,\n            493.00000000000006,\n            475.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_38b9c68d8df822e3_9",\n          "bbox_xyxy": [\n            217.25,\n            99.75,\n            243.75,\n            138.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_38b9c68d8df822e3_10",\n          "bbox_xyxy": [\n            0.2499999999999991,\n            327.75,\n            23.75,\n            355.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_38b9c68d8df822e3_11",\n          "bbox_xyxy": [\n            368.5,\n            58.00000000000001,\n            413.5,\n            86.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_38b9c68d8df822e3_12",\n          "bbox_xyxy": [\n            596.0,\n            350.5,\n            636.0,\n            377.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_38b9c68d8df822e3_13",\n          "bbox_xyxy": [\n            0.7499999999999996,\n            302.5,\n            23.25,\n            327.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_38b9c68d8df822e3_14",\n          "bbox_xyxy": [\n            592.0,\n            459.0,\n            640.0,\n            486.99999999999994\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_38b9c68d8df822e3_15",\n          "bbox_xyxy": [\n            74.24999999999999,\n            218.0,\n            99.75,\n            235.99999999999997\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_38b9c68d8df822e3_16",\n          "bbox_xyxy": [\n            254.75,\n            249.00000000000003,\n            280.25,\n            287.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_38b9c68d8df822e3_17",\n          "bbox_xyxy": [\n            297.5,\n            493.75,\n            314.5,\n            515.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_38b9c68d8df822e3_18",\n          "bbox_xyxy": [\n            415.25,\n            475.75,\n            459.75,\n            498.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_38b9c68d8df822e3_19",\n          "bbox_xyxy": [\n            591.75,\n            609.25,\n            615.25,\n            638.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_38b9c68d8df822e3_20",\n          "bbox_xyxy": [\n            283.25,\n            584.75,\n            305.75,\n            600.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_38b9c68d8df822e3_21",\n          "bbox_xyxy": [\n            198.0,\n            598.0,\n            217.0,\n            624.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_38b9c68d8df822e3_22",\n          "bbox_xyxy": [\n            5.5,\n            355.0,\n            34.5,\n            374.00000000000006\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_38b9c68d8df822e3_23",\n          "bbox_xyxy": [\n            531.75,\n            378.24999999999994,\n            555.25,\n            404.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_38b9c68d8df822e3_24",\n          "bbox_xyxy": [\n            563.0,\n            61.0,\n            590.0,\n            88.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_38b9c68d8df822e3_25",\n          "bbox_xyxy": [\n            625.0,\n            124.0,\n            640.0,\n            144.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_38b9c68d8df822e3_26",\n          "bbox_xyxy": [\n            267.25,\n            83.75,\n            291.75,\n            103.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_38b9c68d8df822e3_27",\n          "bbox_xyxy": [\n            611.0,\n            427.75,\n            637.0,\n            458.24999999999994\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_38b9c68d8df822e3_28",\n          "bbox_xyxy": [\n            549.25,\n            15.25,\n            578.75,\n            37.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_38b9c68d8df822e3_29",\n          "bbox_xyxy": [\n            522.5,\n            563.0,\n            554.5,\n            588.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_38b9c68d8df822e3_30",\n          "bbox_xyxy": [\n            271.75,\n            545.0,\n            287.25,\n            564.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_38b9c68d8df822e3_31",\n          "bbox_xyxy": [\n            618.0,\n            551.25,\n            639.0,\n            577.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_38b9c68d8df822e3_32",\n          "bbox_xyxy": [\n            419.0,\n            228.75,\n            428.00000000000006,\n            250.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/valid/images/6095710224539913562_jpg.rf.7099237d33b0eb5cc5c381c8b734deb5.jpg",\n        "size": 136973,\n        "compressed": 136869,\n        "offset": 627663768,\n        "crc32": "56420ba2",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_3a4e13b2c75c9d9f",\n      "relative_path": "images/IMG_2572_JPG.rf.3ddd11fc686f44935e2eb6583c8fd324.jpg",\n      "sha256": "60bb5e71e7a3409d3f2aedc78849708e79581e150fb5381c6d0720bfd3dabdbe",\n      "decoded_sha256": "b1782e9d829edd89192f93d6b4e6c50c37cf7ba2d24122cdd80643fb2a57c9aa",\n      "dhash": "68f938b830b0b959",\n      "bytes": 25108,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2572_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "validation",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_3a4e13b2c75c9d9f_0",\n          "bbox_xyxy": [\n            159.0,\n            15.749999999999993,\n            517.0,\n            590.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/valid/images/IMG_2572_JPG.rf.3ddd11fc686f44935e2eb6583c8fd324.jpg",\n        "size": 25108,\n        "compressed": 23351,\n        "offset": 632205030,\n        "crc32": "214ddb84",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_3d8739ffb0cad942",\n      "relative_path": "images/IMG_1042_JPG.rf.deb2a9a6719e29ad3fb044d68d08127a.jpg",\n      "sha256": "a8e8fefc06ef60b73e3ef7bdc1b30380ba38ee62c02eba15213d082005e6fb33",\n      "decoded_sha256": "d31a3dc821797349644b1d68ed7d70b4386e80ed07ab4f17a496bacd034ec7c1",\n      "dhash": "560726651a0d1916",\n      "bytes": 44991,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_1042_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "validation",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_3d8739ffb0cad942_0",\n          "bbox_xyxy": [\n            130.0,\n            34.5,\n            175.0,\n            87.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_3d8739ffb0cad942_1",\n          "bbox_xyxy": [\n            240.25000000000003,\n            167.25,\n            286.75,\n            212.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_3d8739ffb0cad942_2",\n          "bbox_xyxy": [\n            282.25,\n            19.0,\n            345.74999999999994,\n            76.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_3d8739ffb0cad942_3",\n          "bbox_xyxy": [\n            450.0,\n            275.0,\n            504.00000000000006,\n            305.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_3d8739ffb0cad942_4",\n          "bbox_xyxy": [\n            309.49999999999994,\n            77.25,\n            371.5,\n            118.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_3d8739ffb0cad942_5",\n          "bbox_xyxy": [\n            187.5,\n            256.25,\n            245.49999999999997,\n            305.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_3d8739ffb0cad942_6",\n          "bbox_xyxy": [\n            29.5,\n            354.75,\n            84.5,\n            408.24999999999994\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_3d8739ffb0cad942_7",\n          "bbox_xyxy": [\n            443.5,\n            83.25,\n            470.50000000000006,\n            163.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_3d8739ffb0cad942_8",\n          "bbox_xyxy": [\n            350.5,\n            305.75,\n            389.5,\n            397.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_3d8739ffb0cad942_9",\n          "bbox_xyxy": [\n            469.0,\n            566.75,\n            527.0,\n            615.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_3d8739ffb0cad942_10",\n          "bbox_xyxy": [\n            510.75,\n            152.0,\n            561.25,\n            238.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_3d8739ffb0cad942_11",\n          "bbox_xyxy": [\n            437.75,\n            429.75,\n            497.25,\n            510.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_3d8739ffb0cad942_12",\n          "bbox_xyxy": [\n            517.75,\n            346.0,\n            577.25,\n            398.00000000000006\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_3d8739ffb0cad942_13",\n          "bbox_xyxy": [\n            309.75,\n            533.5,\n            349.25000000000006,\n            610.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_3d8739ffb0cad942_14",\n          "bbox_xyxy": [\n            58.50000000000001,\n            415.50000000000006,\n            124.5,\n            486.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_3d8739ffb0cad942_15",\n          "bbox_xyxy": [\n            354.5,\n            500.50000000000006,\n            415.5,\n            566.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_3d8739ffb0cad942_16",\n          "bbox_xyxy": [\n            194.75,\n            89.5,\n            245.25,\n            140.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_3d8739ffb0cad942_17",\n          "bbox_xyxy": [\n            150.5,\n            431.5,\n            186.50000000000003,\n            496.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_3d8739ffb0cad942_18",\n          "bbox_xyxy": [\n            469.0,\n            0.0,\n            522.0,\n            32.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_3d8739ffb0cad942_19",\n          "bbox_xyxy": [\n            32.75,\n            263.5,\n            77.25,\n            305.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_3d8739ffb0cad942_20",\n          "bbox_xyxy": [\n            277.75,\n            190.49999999999997,\n            323.25,\n            288.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_3d8739ffb0cad942_21",\n          "bbox_xyxy": [\n            143.25,\n            151.24999999999997,\n            182.75000000000003,\n            236.74999999999997\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_3d8739ffb0cad942_22",\n          "bbox_xyxy": [\n            54.0,\n            131.5,\n            106.0,\n            194.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_3d8739ffb0cad942_23",\n          "bbox_xyxy": [\n            550.75,\n            34.0,\n            583.25,\n            121.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/valid/images/IMG_1042_JPG.rf.deb2a9a6719e29ad3fb044d68d08127a.jpg",\n        "size": 44991,\n        "compressed": 44807,\n        "offset": 629008453,\n        "crc32": "c558f803",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_00ff28b138a6f576",\n      "relative_path": "images/IMG_1910_JPG.rf.acec3d9cfbdf275197acfde159217c29.jpg",\n      "sha256": "b26f82f583d25ee60883ae0a8ce7577bdd86b510838b019fdcad1caa33738086",\n      "decoded_sha256": "a785f348b93b88db473dc6df4a0b5dc610674be9c9a1f30366627aa5a038caef",\n      "dhash": "2a68707262606060",\n      "bytes": 27180,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_1910_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "test",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_00ff28b138a6f576_0",\n          "bbox_xyxy": [\n            133.75,\n            90.99999999999997,\n            473.25,\n            622.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/test/images/IMG_1910_JPG.rf.acec3d9cfbdf275197acfde159217c29.jpg",\n        "size": 27180,\n        "compressed": 25724,\n        "offset": 317549633,\n        "crc32": "9ccf20cd",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_015e2132aff2ed5f",\n      "relative_path": "images/20251107_160252_jpg.rf.b1fe0f275943f347c5b6669a68462610.jpg",\n      "sha256": "a63bf0425980e2511556ce39062e374e2df4fe60269d0708118cd6b290973145",\n      "decoded_sha256": "a56e5433cd52424c035f2949f649a54f9fa48c0e5998d699c7e49c0d517297af",\n      "dhash": "f5313b0e0e0c360f",\n      "bytes": 148446,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "20251107_160252_jpg",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "test",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_015e2132aff2ed5f_0",\n          "bbox_xyxy": [\n            177.99999999999997,\n            276.0,\n            195.0,\n            306.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_015e2132aff2ed5f_1",\n          "bbox_xyxy": [\n            219.00000000000003,\n            212.75,\n            252.0,\n            236.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_015e2132aff2ed5f_2",\n          "bbox_xyxy": [\n            388.75,\n            207.25,\n            417.24999999999994,\n            228.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_015e2132aff2ed5f_3",\n          "bbox_xyxy": [\n            496.25,\n            162.25,\n            524.75,\n            187.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_015e2132aff2ed5f_4",\n          "bbox_xyxy": [\n            359.25,\n            119.75,\n            380.75,\n            145.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_015e2132aff2ed5f_5",\n          "bbox_xyxy": [\n            159.5,\n            53.75,\n            180.5,\n            78.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_015e2132aff2ed5f_6",\n          "bbox_xyxy": [\n            108.75,\n            168.75,\n            149.25,\n            182.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_015e2132aff2ed5f_7",\n          "bbox_xyxy": [\n            557.25,\n            79.25,\n            594.75,\n            103.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_015e2132aff2ed5f_8",\n          "bbox_xyxy": [\n            405.0,\n            35.0,\n            447.0,\n            46.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_015e2132aff2ed5f_9",\n          "bbox_xyxy": [\n            54.5,\n            538.75,\n            70.5,\n            555.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_015e2132aff2ed5f_10",\n          "bbox_xyxy": [\n            257.0,\n            610.5,\n            312.0,\n            627.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_015e2132aff2ed5f_11",\n          "bbox_xyxy": [\n            198.25,\n            461.25,\n            227.75000000000003,\n            494.74999999999994\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/test/images/20251107_160252_jpg.rf.b1fe0f275943f347c5b6669a68462610.jpg",\n        "size": 148446,\n        "compressed": 148353,\n        "offset": 313225793,\n        "crc32": "f5ed93c5",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_045099d9eaa116bd",\n      "relative_path": "images/IMG_2379_JPG.rf.240a26338583e781518ac73e435635e0.jpg",\n      "sha256": "8a2b98c0c78f52f6acdb97b82e2c232e8533600418048ab4ef38abed17f35287",\n      "decoded_sha256": "7b005112a2e5efe33c90c81cabbcf94cd31ed66412ead34e31d970a9ed0421d8",\n      "dhash": "44707171717171b6",\n      "bytes": 27673,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2379_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "test",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_045099d9eaa116bd_0",\n          "bbox_xyxy": [\n            134.50000000000003,\n            63.75,\n            532.5,\n            608.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/test/images/IMG_2379_JPG.rf.240a26338583e781518ac73e435635e0.jpg",\n        "size": 27673,\n        "compressed": 26364,\n        "offset": 318609738,\n        "crc32": "406a4a49",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_089ba40dac04ce61",\n      "relative_path": "images/6273997143085354522_jpg.rf.0798a4f8432e805053468fec98a75290.jpg",\n      "sha256": "b2668de363132fb0fbbd7ac3c52025889f2bb1add38fa5b57e603e0022e89e60",\n      "decoded_sha256": "a6257fe72c478ff040a120c2678f858156c1f13cfa5036d71ed4da54c7d4c8bb",\n      "dhash": "81a797d2e5c909c1",\n      "bytes": 49650,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "6273997143085354522_jpg",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "test",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_089ba40dac04ce61_0",\n          "bbox_xyxy": [\n            104.25,\n            110.25,\n            156.75,\n            150.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_089ba40dac04ce61_1",\n          "bbox_xyxy": [\n            219.5,\n            98.5,\n            275.5,\n            126.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_089ba40dac04ce61_2",\n          "bbox_xyxy": [\n            127.99999999999999,\n            227.24999999999997,\n            185.99999999999997,\n            276.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_089ba40dac04ce61_3",\n          "bbox_xyxy": [\n            287.75,\n            112.25,\n            340.25,\n            156.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_089ba40dac04ce61_4",\n          "bbox_xyxy": [\n            308.75000000000006,\n            229.5,\n            355.25000000000006,\n            258.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_089ba40dac04ce61_5",\n          "bbox_xyxy": [\n            351.25,\n            311.25000000000006,\n            425.75,\n            342.75000000000006\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_089ba40dac04ce61_6",\n          "bbox_xyxy": [\n            404.49999999999994,\n            441.99999999999994,\n            431.5,\n            481.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_089ba40dac04ce61_7",\n          "bbox_xyxy": [\n            159.25,\n            382.0,\n            187.75000000000003,\n            425.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_089ba40dac04ce61_8",\n          "bbox_xyxy": [\n            463.75,\n            309.75,\n            494.25,\n            354.25000000000006\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_089ba40dac04ce61_9",\n          "bbox_xyxy": [\n            256.25,\n            332.25,\n            305.75,\n            365.74999999999994\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_089ba40dac04ce61_10",\n          "bbox_xyxy": [\n            27.75,\n            497.75000000000006,\n            72.25,\n            536.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_089ba40dac04ce61_11",\n          "bbox_xyxy": [\n            521.5,\n            252.00000000000003,\n            565.5,\n            285.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/test/images/6273997143085354522_jpg.rf.0798a4f8432e805053468fec98a75290.jpg",\n        "size": 49650,\n        "compressed": 49354,\n        "offset": 315476086,\n        "crc32": "98c8ff8a",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_0a7e8d9e71ba934e",\n      "relative_path": "images/IMG_2453_JPG.rf.b913bc9660dc336328d8dcd06ee86c10.jpg",\n      "sha256": "671e84445d3cee55f9ecf7fd2eab9532e5472ebf9ad2ab763e022cb8919094bb",\n      "decoded_sha256": "13444ea620ccb33cef69fc314ddc8b02696b5f37a3a71e15a5828e05283f584c",\n      "dhash": "0072716969717230",\n      "bytes": 31542,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2453_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "test",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_0a7e8d9e71ba934e_0",\n          "bbox_xyxy": [\n            169.00000000000003,\n            97.5,\n            478.0,\n            577.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/test/images/IMG_2453_JPG.rf.b913bc9660dc336328d8dcd06ee86c10.jpg",\n        "size": 31542,\n        "compressed": 30302,\n        "offset": 318955150,\n        "crc32": "07c292f1",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_0b235e3cccc93131",\n      "relative_path": "images/IMG_4151_JPG.rf.a7738cf5c4eb708fc7285691d8870e43.jpg",\n      "sha256": "27c7e492c4b939d48cee9eae526519295c4a63818f34b20a4e829f2c2c304b10",\n      "decoded_sha256": "ccff369545805159d81ac4a0602a80029772c1da63eafddff8bca5c217c11ba2",\n      "dhash": "4662c47456e57c7d",\n      "bytes": 53665,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_4151_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "test",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_0b235e3cccc93131_0",\n          "bbox_xyxy": [\n            245.49999999999997,\n            263.5,\n            282.5,\n            296.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0b235e3cccc93131_1",\n          "bbox_xyxy": [\n            166.74999999999997,\n            325.50000000000006,\n            211.25,\n            351.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0b235e3cccc93131_2",\n          "bbox_xyxy": [\n            224.25,\n            393.75,\n            247.75000000000003,\n            444.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0b235e3cccc93131_3",\n          "bbox_xyxy": [\n            195.75,\n            473.49999999999994,\n            225.25000000000003,\n            517.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0b235e3cccc93131_4",\n          "bbox_xyxy": [\n            454.5,\n            413.49999999999994,\n            502.5,\n            437.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0b235e3cccc93131_5",\n          "bbox_xyxy": [\n            305.0,\n            309.0,\n            332.0,\n            361.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0b235e3cccc93131_6",\n          "bbox_xyxy": [\n            320.5,\n            387.0,\n            374.5,\n            418.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0b235e3cccc93131_7",\n          "bbox_xyxy": [\n            465.5,\n            283.75,\n            509.5,\n            317.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0b235e3cccc93131_8",\n          "bbox_xyxy": [\n            470.5,\n            353.0,\n            525.5,\n            382.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0b235e3cccc93131_9",\n          "bbox_xyxy": [\n            416.5,\n            552.0,\n            465.50000000000006,\n            583.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/test/images/IMG_4151_JPG.rf.a7738cf5c4eb708fc7285691d8870e43.jpg",\n        "size": 53665,\n        "compressed": 53517,\n        "offset": 320265724,\n        "crc32": "8fbad98d",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_0cede29efc7af87c",\n      "relative_path": "images/IMG_1077_JPG.rf.bbd85e1dcd8a0d7d74d3006523fc1440.jpg",\n      "sha256": "51abf4f96c55d08839114fae35f208f36e4e279d0560c67836cf3a9e3f7d479c",\n      "decoded_sha256": "35b1a63ea16ee6d8a2a5845b0efb9110d0096f9b14b3578afdcbf74e8ec24896",\n      "dhash": "8102021316480911",\n      "bytes": 56198,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_1077_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "test",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_0cede29efc7af87c_0",\n          "bbox_xyxy": [\n            85.25,\n            384.0,\n            129.75,\n            429.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0cede29efc7af87c_1",\n          "bbox_xyxy": [\n            528.75,\n            361.25,\n            591.25,\n            393.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0cede29efc7af87c_2",\n          "bbox_xyxy": [\n            307.0,\n            297.0,\n            346.0,\n            343.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0cede29efc7af87c_3",\n          "bbox_xyxy": [\n            200.5,\n            479.75,\n            250.5,\n            514.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0cede29efc7af87c_4",\n          "bbox_xyxy": [\n            318.75,\n            141.75,\n            398.25,\n            175.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0cede29efc7af87c_5",\n          "bbox_xyxy": [\n            345.5,\n            559.25,\n            404.5,\n            579.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0cede29efc7af87c_6",\n          "bbox_xyxy": [\n            314.0,\n            0.0,\n            376.0,\n            17.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0cede29efc7af87c_7",\n          "bbox_xyxy": [\n            248.75,\n            45.5,\n            298.25,\n            85.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0cede29efc7af87c_8",\n          "bbox_xyxy": [\n            129.5,\n            497.0,\n            192.5,\n            513.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0cede29efc7af87c_9",\n          "bbox_xyxy": [\n            568.75,\n            572.0,\n            597.25,\n            622.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_0cede29efc7af87c_10",\n          "bbox_xyxy": [\n            29.25,\n            344.75,\n            112.74999999999999,\n            360.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/test/images/IMG_1077_JPG.rf.bbd85e1dcd8a0d7d74d3006523fc1440.jpg",\n        "size": 56198,\n        "compressed": 55985,\n        "offset": 316540541,\n        "crc32": "0e612833",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_0cfde140bf5bc257",\n      "relative_path": "images/IMG_2213_JPG.rf.de0925d4251bbbe9b686f0a1f3ba4fd3.jpg",\n      "sha256": "fefecf58c818deb5f64af4cfb2434b9fba7e665be2c7976f53538b7faf10e9fa",\n      "decoded_sha256": "6fb74838af446103b1e7d16026c84de61d7cfbb4215957fea43d8ebbe1e782d1",\n      "dhash": "4470717969713232",\n      "bytes": 30899,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2213_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "test",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_0cfde140bf5bc257_0",\n          "bbox_xyxy": [\n            142.5,\n            65.49999999999997,\n            522.5,\n            532.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/test/images/IMG_2213_JPG.rf.de0925d4251bbbe9b686f0a1f3ba4fd3.jpg",\n        "size": 30899,\n        "compressed": 29510,\n        "offset": 318334380,\n        "crc32": "5ddb746a",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_0eb2105ecb98f8b2",\n      "relative_path": "images/IMG_2091_JPG.rf.e801399effea72c6df68f44a73335420.jpg",\n      "sha256": "8ce9fa64405351ba5cc531613fdb12bd67d0152296eb24f186ab72d6e67f5f76",\n      "decoded_sha256": "8bc107a7102b6d8e8c1b22fc5ec786ddfa73067f0327eb30b8f684afeba380d9",\n      "dhash": "055171796979b132",\n      "bytes": 28336,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2091_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "test",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_0eb2105ecb98f8b2_0",\n          "bbox_xyxy": [\n            163.75000000000003,\n            87.5,\n            485.25000000000006,\n            575.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/test/images/IMG_2091_JPG.rf.e801399effea72c6df68f44a73335420.jpg",\n        "size": 28336,\n        "compressed": 27674,\n        "offset": 318004615,\n        "crc32": "0aeca05d",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_0efe653d5ea0065a",\n      "relative_path": "images/IMG_2196_JPG.rf.2edacd3f2b7ec55e62bc2bdb91eae6f8.jpg",\n      "sha256": "ace482d80d4484e144490851c99da53c485121d18bca7e234e7a962dc179bd10",\n      "decoded_sha256": "aa11e0a961ae7cd0b207b96cfd9a7e25938e7964e10c79abb01d726d483c48c7",\n      "dhash": "4430716960607234",\n      "bytes": 27470,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2196_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "test",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_0efe653d5ea0065a_0",\n          "bbox_xyxy": [\n            176.0,\n            86.50000000000003,\n            486.0,\n            557.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/test/images/IMG_2196_JPG.rf.2edacd3f2b7ec55e62bc2bdb91eae6f8.jpg",\n        "size": 27470,\n        "compressed": 26562,\n        "offset": 318230474,\n        "crc32": "0413cdb0",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_0f7789049723a9b0",\n      "relative_path": "images/IMG_2552_JPG.rf.82820c3c4acea7292e16b494baf4afce.jpg",\n      "sha256": "13b2d5db16438d114aab161f2d71d56cdb1f62f1e698a4e59ac9a07d7c7de3bd",\n      "decoded_sha256": "3869e367f99772d54ed5323b66873924361276a45e61e5b701ed90f3209a499c",\n      "dhash": "6430796969713132",\n      "bytes": 28446,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2552_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "test",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_0f7789049723a9b0_0",\n          "bbox_xyxy": [\n            153.5,\n            33.75,\n            505.50000000000006,\n            568.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/test/images/IMG_2552_JPG.rf.82820c3c4acea7292e16b494baf4afce.jpg",\n        "size": 28446,\n        "compressed": 27829,\n        "offset": 319211614,\n        "crc32": "194983b4",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_1000566d831bd227",\n      "relative_path": "images/IMG_2189_JPG.rf.64708c2cda09b201d9567a15b65c15d0.jpg",\n      "sha256": "26900c5352ff60040bf2383f768d0705f778c0b11af86a01be0acdf2c7e051bf",\n      "decoded_sha256": "a02bd7a18e3c2cec2d984953a559d19b93a9e12a1904ac5d3fccca5efd27005c",\n      "dhash": "2431317371713214",\n      "bytes": 22733,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2189_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "test",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_1000566d831bd227_0",\n          "bbox_xyxy": [\n            171.99999999999994,\n            64.25,\n            519.0,\n            561.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/test/images/IMG_2189_JPG.rf.64708c2cda09b201d9567a15b65c15d0.jpg",\n        "size": 22733,\n        "compressed": 21868,\n        "offset": 318208434,\n        "crc32": "7f5476fc",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_10d9f546f64ace8a",\n      "relative_path": "images/IMG_2513_JPG.rf.ee489cd362a55a9145797517231483ad.jpg",\n      "sha256": "927b2283e1b19c9456b569e8da6384dfc980d454e5e34cf1325780c44efe89d6",\n      "decoded_sha256": "82994247e13c8b572a2e095cfc83dd94bf95c3c71e7e813efec96158de794f5b",\n      "dhash": "253070717170b130",\n      "bytes": 27543,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2513_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "test",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_10d9f546f64ace8a_0",\n          "bbox_xyxy": [\n            161.75000000000006,\n            37.499999999999964,\n            512.25,\n            608.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/test/images/IMG_2513_JPG.rf.ee489cd362a55a9145797517231483ad.jpg",\n        "size": 27543,\n        "compressed": 26846,\n        "offset": 319072247,\n        "crc32": "cb687c25",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_10f4a8930b4748f9",\n      "relative_path": "images/IMG_2616_JPG.rf.daa5481e376f030035915a746e7cbfd1.jpg",\n      "sha256": "35960b2a042f260f8c0ff8f18cabfa581532b0a8fc49d836782b428caafaf429",\n      "decoded_sha256": "01d7074c84b66a1147f09501383031f4ed8311f41b18f33cc43c6bfd4f2d63c0",\n      "dhash": "9838f0f0b438594a",\n      "bytes": 30305,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2616_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "test",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_10f4a8930b4748f9_0",\n          "bbox_xyxy": [\n            193.25,\n            0.0,\n            556.75,\n            595.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/test/images/IMG_2616_JPG.rf.daa5481e376f030035915a746e7cbfd1.jpg",\n        "size": 30305,\n        "compressed": 29074,\n        "offset": 319420112,\n        "crc32": "83200694",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_112cc5b65a38337a",\n      "relative_path": "images/6159142536404995681_jpg.rf.486e97683e58928bc194af89115a90c3.jpg",\n      "sha256": "fd369edaefc990781c90ebc2b57c0e56230c242ce174a44f9ad7778a488bcd33",\n      "decoded_sha256": "fc315a36f86765ae0cd912d59758d8e1f5df02ba97d4c9be11c42bf38f14971b",\n      "dhash": "1549416961756d29",\n      "bytes": 47851,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "6159142536404995681_jpg",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "test",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_112cc5b65a38337a_0",\n          "bbox_xyxy": [\n            253.5,\n            335.25000000000006,\n            306.5,\n            368.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_112cc5b65a38337a_1",\n          "bbox_xyxy": [\n            454.25,\n            435.50000000000006,\n            505.75,\n            476.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_112cc5b65a38337a_2",\n          "bbox_xyxy": [\n            194.74999999999997,\n            494.49999999999994,\n            244.25,\n            541.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_112cc5b65a38337a_3",\n          "bbox_xyxy": [\n            405.0,\n            507.75,\n            454.00000000000006,\n            550.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_112cc5b65a38337a_4",\n          "bbox_xyxy": [\n            305.0,\n            404.0,\n            363.0,\n            451.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_112cc5b65a38337a_5",\n          "bbox_xyxy": [\n            219.25,\n            402.25,\n            249.74999999999997,\n            456.75000000000006\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_112cc5b65a38337a_6",\n          "bbox_xyxy": [\n            189.5,\n            315.25,\n            248.49999999999997,\n            362.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_112cc5b65a38337a_7",\n          "bbox_xyxy": [\n            377.75,\n            279.5,\n            433.24999999999994,\n            304.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_112cc5b65a38337a_8",\n          "bbox_xyxy": [\n            278.25,\n            222.25,\n            324.75,\n            253.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_112cc5b65a38337a_9",\n          "bbox_xyxy": [\n            177.75000000000003,\n            128.75,\n            238.25,\n            163.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/test/images/6159142536404995681_jpg.rf.486e97683e58928bc194af89115a90c3.jpg",\n        "size": 47851,\n        "compressed": 47576,\n        "offset": 315428327,\n        "crc32": "90d01db3",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_1183e9c7f4be502a",\n      "relative_path": "images/IMG_2153_JPG.rf.92fd79912e6de0bf88d8b86bf3ec002f.jpg",\n      "sha256": "e7077fd39f5607f15ac15d09de3b66ed438a4d45d4a21c6e19f598c1473a3d9c",\n      "decoded_sha256": "e1b6d327bbbedb6b3a459f8159d2b300b6c9d0764509d2861ab918d6972609b9",\n      "dhash": "66f8b83834387819",\n      "bytes": 26896,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2153_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "test",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_1183e9c7f4be502a_0",\n          "bbox_xyxy": [\n            194.5,\n            22.5,\n            550.5,\n            575.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/test/images/IMG_2153_JPG.rf.92fd79912e6de0bf88d8b86bf3ec002f.jpg",\n        "size": 26896,\n        "compressed": 26333,\n        "offset": 318109589,\n        "crc32": "05967021",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_11f8bf03e53d340f",\n      "relative_path": "images/IMG_2174_JPG.rf.7c3da364352e1c2465d5f8878a5e5328.jpg",\n      "sha256": "e17940b1a49e4aec582e80ac4b6ece9b7f73da4fd028d9ef0e3054c05fe353a8",\n      "decoded_sha256": "729a01e357dd7e31b6e3d61bcf1ed4550410f5a9497e51ff48f99f52ed253feb",\n      "dhash": "0c50707171713132",\n      "bytes": 24191,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2174_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "test",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_11f8bf03e53d340f_0",\n          "bbox_xyxy": [\n            150.5,\n            73.49999999999997,\n            468.5,\n            587.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/test/images/IMG_2174_JPG.rf.7c3da364352e1c2465d5f8878a5e5328.jpg",\n        "size": 24191,\n        "compressed": 23253,\n        "offset": 318185009,\n        "crc32": "171109f0",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_142d39c152cdad73",\n      "relative_path": "images/IMG_1092_JPG.rf.b526f22db4c711c71f2abb8fd8a82820.jpg",\n      "sha256": "8f241df984cbc7b276c20aa4a6b595ec6c907badfab14e648a1b4ba9c698d3c2",\n      "decoded_sha256": "2382edad4ea4c2f1d93f195ae19d4cb31efd76ad32a207db4a1649f39f995500",\n      "dhash": "8b46570b950d27a3",\n      "bytes": 55609,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_1092_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "test",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_142d39c152cdad73_0",\n          "bbox_xyxy": [\n            192.5,\n            568.75,\n            238.5,\n            609.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_142d39c152cdad73_1",\n          "bbox_xyxy": [\n            469.0,\n            50.0,\n            494.0,\n            110.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_142d39c152cdad73_2",\n          "bbox_xyxy": [\n            192.75,\n            205.0,\n            210.25,\n            271.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_142d39c152cdad73_3",\n          "bbox_xyxy": [\n            424.25,\n            381.0,\n            473.75,\n            419.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_142d39c152cdad73_4",\n          "bbox_xyxy": [\n            367.75,\n            64.5,\n            410.25,\n            119.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_142d39c152cdad73_5",\n          "bbox_xyxy": [\n            246.25,\n            23.25,\n            282.75,\n            79.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_142d39c152cdad73_6",\n          "bbox_xyxy": [\n            196.25,\n            173.75,\n            225.75000000000003,\n            233.25000000000003\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_142d39c152cdad73_7",\n          "bbox_xyxy": [\n            267.75,\n            524.25,\n            314.25,\n            562.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_142d39c152cdad73_8",\n          "bbox_xyxy": [\n            96.00000000000001,\n            597.5,\n            135.0,\n            632.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_142d39c152cdad73_9",\n          "bbox_xyxy": [\n            455.75,\n            621.25,\n            503.25,\n            639.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_142d39c152cdad73_10",\n          "bbox_xyxy": [\n            179.5,\n            141.25,\n            229.5,\n            171.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_142d39c152cdad73_11",\n          "bbox_xyxy": [\n            165.25000000000003,\n            484.25,\n            185.75,\n            538.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_142d39c152cdad73_12",\n          "bbox_xyxy": [\n            234.25,\n            513.75,\n            270.75,\n            559.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_142d39c152cdad73_13",\n          "bbox_xyxy": [\n            224.24999999999997,\n            448.0,\n            248.75,\n            512.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_142d39c152cdad73_14",\n          "bbox_xyxy": [\n            203.25,\n            271.25,\n            240.75,\n            299.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_142d39c152cdad73_15",\n          "bbox_xyxy": [\n            105.5,\n            406.25,\n            160.5,\n            440.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_142d39c152cdad73_16",\n          "bbox_xyxy": [\n            76.75,\n            230.5,\n            103.25,\n            311.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_142d39c152cdad73_17",\n          "bbox_xyxy": [\n            321.25,\n            157.25,\n            362.75000000000006,\n            212.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_142d39c152cdad73_18",\n          "bbox_xyxy": [\n            295.75,\n            168.5,\n            339.25,\n            206.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_142d39c152cdad73_19",\n          "bbox_xyxy": [\n            273.5,\n            117.75,\n            296.5,\n            180.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/test/images/IMG_1092_JPG.rf.b526f22db4c711c71f2abb8fd8a82820.jpg",\n        "size": 55609,\n        "compressed": 55467,\n        "offset": 316643645,\n        "crc32": "ce4b0735",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_1516716c1da90e40",\n      "relative_path": "images/IMG_2365_JPG.rf.532ad918ae1effd56489935fb04dea76.jpg",\n      "sha256": "dfb809abe81a6aefebedb800dd6cc63b26b9665f0a5d0d7184d46da4fcbe2677",\n      "decoded_sha256": "fe6b86474fbe39b182e2ad32feef99960d68add801fbfd12851369d8bf34403e",\n      "dhash": "4d32716969717132",\n      "bytes": 24820,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2365_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "test",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_1516716c1da90e40_0",\n          "bbox_xyxy": [\n            115.24999999999999,\n            26.999999999999993,\n            493.75,\n            607.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/test/images/IMG_2365_JPG.rf.532ad918ae1effd56489935fb04dea76.jpg",\n        "size": 24820,\n        "compressed": 23993,\n        "offset": 318585573,\n        "crc32": "f1846f29",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_16c033fa2020b0ed",\n      "relative_path": "images/6159142536404995658_jpg.rf.331b2b823e4b474fae8b6a85430a71ad.jpg",\n      "sha256": "046a61d5129bcfc27042882ee6494b83633a99bf54d1b8f8bae75a79a25d3088",\n      "decoded_sha256": "2e0cec0495418aaed6de6b9dd71ae3a78a6b2a74988b14355aae29d7d084a602",\n      "dhash": "a5e6a4b1ddd5d1d7",\n      "bytes": 35648,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "6159142536404995658_jpg",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "test",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_16c033fa2020b0ed_0",\n          "bbox_xyxy": [\n            93.5,\n            159.5,\n            119.50000000000001,\n            203.49999999999997\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_16c033fa2020b0ed_1",\n          "bbox_xyxy": [\n            242.5,\n            82.5,\n            275.5,\n            115.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_16c033fa2020b0ed_2",\n          "bbox_xyxy": [\n            401.0,\n            309.49999999999994,\n            436.99999999999994,\n            336.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_16c033fa2020b0ed_3",\n          "bbox_xyxy": [\n            170.25000000000003,\n            500.25,\n            201.75,\n            540.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_16c033fa2020b0ed_4",\n          "bbox_xyxy": [\n            330.75,\n            233.5,\n            366.25,\n            258.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_16c033fa2020b0ed_5",\n          "bbox_xyxy": [\n            138.0,\n            119.75,\n            165.99999999999997,\n            157.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_16c033fa2020b0ed_6",\n          "bbox_xyxy": [\n            433.5,\n            465.50000000000006,\n            463.5,\n            513.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_16c033fa2020b0ed_7",\n          "bbox_xyxy": [\n            509.25,\n            114.5,\n            540.75,\n            154.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_16c033fa2020b0ed_8",\n          "bbox_xyxy": [\n            321.75,\n            533.25,\n            363.25,\n            562.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_16c033fa2020b0ed_9",\n          "bbox_xyxy": [\n            307.50000000000006,\n            289.0,\n            341.50000000000006,\n            319.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_16c033fa2020b0ed_10",\n          "bbox_xyxy": [\n            451.5,\n            209.75,\n            480.50000000000006,\n            245.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_16c033fa2020b0ed_11",\n          "bbox_xyxy": [\n            240.75,\n            248.25,\n            290.25,\n            279.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_16c033fa2020b0ed_12",\n          "bbox_xyxy": [\n            68.75000000000001,\n            251.74999999999997,\n            107.25000000000001,\n            296.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_16c033fa2020b0ed_13",\n          "bbox_xyxy": [\n            455.0,\n            80.25,\n            505.0,\n            117.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_16c033fa2020b0ed_14",\n          "bbox_xyxy": [\n            436.25,\n            378.49999999999994,\n            487.75000000000006,\n            397.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_16c033fa2020b0ed_15",\n          "bbox_xyxy": [\n            301.75,\n            478.75,\n            358.25,\n            501.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_16c033fa2020b0ed_16",\n          "bbox_xyxy": [\n            129.0,\n            430.0,\n            178.0,\n            463.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_16c033fa2020b0ed_17",\n          "bbox_xyxy": [\n            221.25,\n            119.0,\n            248.75,\n            172.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_16c033fa2020b0ed_18",\n          "bbox_xyxy": [\n            353.75,\n            113.25000000000001,\n            386.25,\n            157.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/test/images/6159142536404995658_jpg.rf.331b2b823e4b474fae8b6a85430a71ad.jpg",\n        "size": 35648,\n        "compressed": 35359,\n        "offset": 315352961,\n        "crc32": "1648d6ef",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_17fbab7bed876663",\n      "relative_path": "images/IMG_0771_JPG.rf.145fa03b1b24682722b6b5908fda1e34.jpg",\n      "sha256": "3bb1f5745f9db812d4bdb0f5adb60b619a1209e0a637f70af2aa583b809ee6fd",\n      "decoded_sha256": "9a93800cf82fa71c23088feba77ba2d36d7c2c427349ec63d489aba97aa3710e",\n      "dhash": "2c26b49a36665594",\n      "bytes": 50049,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_0771_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "test",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_17fbab7bed876663_0",\n          "bbox_xyxy": [\n            390.0,\n            40.74999999999999,\n            454.00000000000006,\n            71.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fbab7bed876663_1",\n          "bbox_xyxy": [\n            242.75000000000003,\n            111.0,\n            289.25,\n            158.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fbab7bed876663_2",\n          "bbox_xyxy": [\n            67.25,\n            137.0,\n            132.75,\n            170.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fbab7bed876663_3",\n          "bbox_xyxy": [\n            13.75,\n            148.5,\n            100.25,\n            182.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fbab7bed876663_4",\n          "bbox_xyxy": [\n            2.9999999999999982,\n            50.74999999999999,\n            83.99999999999999,\n            96.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fbab7bed876663_5",\n          "bbox_xyxy": [\n            314.75,\n            155.25,\n            367.25,\n            215.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fbab7bed876663_6",\n          "bbox_xyxy": [\n            464.75,\n            151.24999999999997,\n            515.25,\n            206.74999999999997\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fbab7bed876663_7",\n          "bbox_xyxy": [\n            543.75,\n            146.0,\n            614.25,\n            193.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fbab7bed876663_8",\n          "bbox_xyxy": [\n            486.75,\n            86.25,\n            543.25,\n            144.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fbab7bed876663_9",\n          "bbox_xyxy": [\n            453.75,\n            35.25,\n            525.25,\n            77.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fbab7bed876663_10",\n          "bbox_xyxy": [\n            217.5,\n            58.00000000000001,\n            288.5,\n            105.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fbab7bed876663_11",\n          "bbox_xyxy": [\n            56.99999999999999,\n            78.0,\n            110.0,\n            130.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fbab7bed876663_12",\n          "bbox_xyxy": [\n            68.0,\n            233.49999999999997,\n            143.0,\n            294.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fbab7bed876663_13",\n          "bbox_xyxy": [\n            266.25,\n            180.0,\n            320.75,\n            237.00000000000003\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fbab7bed876663_14",\n          "bbox_xyxy": [\n            190.0,\n            281.25,\n            235.0,\n            332.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fbab7bed876663_15",\n          "bbox_xyxy": [\n            258.25,\n            310.5,\n            347.75,\n            340.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fbab7bed876663_16",\n          "bbox_xyxy": [\n            365.0,\n            246.0,\n            410.99999999999994,\n            300.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fbab7bed876663_17",\n          "bbox_xyxy": [\n            450.25,\n            338.0,\n            527.75,\n            385.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fbab7bed876663_18",\n          "bbox_xyxy": [\n            531.5,\n            304.75,\n            604.5,\n            357.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fbab7bed876663_19",\n          "bbox_xyxy": [\n            291.75,\n            411.5,\n            329.25,\n            466.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fbab7bed876663_20",\n          "bbox_xyxy": [\n            514.25,\n            429.25,\n            581.75,\n            483.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fbab7bed876663_21",\n          "bbox_xyxy": [\n            168.75,\n            514.75,\n            254.24999999999997,\n            552.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fbab7bed876663_22",\n          "bbox_xyxy": [\n            315.0,\n            513.75,\n            370.0,\n            567.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fbab7bed876663_23",\n          "bbox_xyxy": [\n            107.00000000000001,\n            561.25,\n            194.0,\n            595.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fbab7bed876663_24",\n          "bbox_xyxy": [\n            21.500000000000004,\n            304.0,\n            101.5,\n            346.99999999999994\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fbab7bed876663_25",\n          "bbox_xyxy": [\n            5.500000000000007,\n            376.5,\n            87.5,\n            411.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fbab7bed876663_26",\n          "bbox_xyxy": [\n            183.25,\n            398.0,\n            236.75,\n            457.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fbab7bed876663_27",\n          "bbox_xyxy": [\n            406.0,\n            464.25,\n            479.0,\n            515.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fbab7bed876663_28",\n          "bbox_xyxy": [\n            424.25,\n            537.5,\n            495.75,\n            585.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fbab7bed876663_29",\n          "bbox_xyxy": [\n            536.0,\n            579.75,\n            581.0,\n            627.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fbab7bed876663_30",\n          "bbox_xyxy": [\n            284.99999999999994,\n            587.0,\n            390.99999999999994,\n            640.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fbab7bed876663_31",\n          "bbox_xyxy": [\n            28.75,\n            551.75,\n            111.25,\n            605.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_17fbab7bed876663_32",\n          "bbox_xyxy": [\n            436.75,\n            203.75,\n            516.25,\n            243.25000000000003\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/test/images/IMG_0771_JPG.rf.145fa03b1b24682722b6b5908fda1e34.jpg",\n        "size": 50049,\n        "compressed": 49908,\n        "offset": 315839590,\n        "crc32": "8c71eccd",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_181820ebba6cf9a7",\n      "relative_path": "images/IMG_1139_JPG.rf.e8db0b2ee5cddca3b9dc9d50e7e4e7e1.jpg",\n      "sha256": "34d605cb34c0c5321f9306a66922f5652cde29b8da9a3455293713d57460c10b",\n      "decoded_sha256": "4b6c0aa0430df7ce90e872d718ecf2d81ba5c66749a32a072bdace24d13e5ea8",\n      "dhash": "2f27990e2c0c1c09",\n      "bytes": 42535,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_1139_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "test",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_181820ebba6cf9a7_0",\n          "bbox_xyxy": [\n            265.0,\n            110.25,\n            284.0,\n            149.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_1",\n          "bbox_xyxy": [\n            158.0,\n            237.0,\n            178.0,\n            273.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_2",\n          "bbox_xyxy": [\n            236.75,\n            72.0,\n            258.25,\n            105.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_3",\n          "bbox_xyxy": [\n            345.0,\n            219.5,\n            370.0,\n            245.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_4",\n          "bbox_xyxy": [\n            195.25,\n            300.0,\n            218.75,\n            335.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_5",\n          "bbox_xyxy": [\n            186.5,\n            212.75000000000003,\n            206.5,\n            243.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_6",\n          "bbox_xyxy": [\n            348.0,\n            139.25,\n            368.0,\n            173.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_7",\n          "bbox_xyxy": [\n            235.25000000000003,\n            526.25,\n            260.75,\n            559.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_8",\n          "bbox_xyxy": [\n            499.25000000000006,\n            448.49999999999994,\n            524.75,\n            482.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_9",\n          "bbox_xyxy": [\n            386.0,\n            374.25,\n            408.00000000000006,\n            405.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_10",\n          "bbox_xyxy": [\n            203.25,\n            111.0,\n            221.75,\n            144.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_11",\n          "bbox_xyxy": [\n            58.75,\n            513.5,\n            83.25,\n            553.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_12",\n          "bbox_xyxy": [\n            446.25,\n            333.75,\n            471.75,\n            360.25000000000006\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_13",\n          "bbox_xyxy": [\n            130.25,\n            175.75,\n            153.75,\n            200.25000000000003\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_14",\n          "bbox_xyxy": [\n            164.5,\n            113.50000000000001,\n            189.5,\n            142.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_15",\n          "bbox_xyxy": [\n            118.75,\n            566.25,\n            146.25,\n            592.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_16",\n          "bbox_xyxy": [\n            236.25,\n            422.75,\n            262.75,\n            454.25000000000006\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_17",\n          "bbox_xyxy": [\n            109.50000000000001,\n            355.75,\n            128.5,\n            394.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_18",\n          "bbox_xyxy": [\n            241.0,\n            486.5,\n            266.0,\n            515.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_19",\n          "bbox_xyxy": [\n            107.25,\n            301.75,\n            130.75,\n            339.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_20",\n          "bbox_xyxy": [\n            25.250000000000004,\n            476.25,\n            48.75,\n            508.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_21",\n          "bbox_xyxy": [\n            511.75000000000006,\n            296.0,\n            535.25,\n            326.99999999999994\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_22",\n          "bbox_xyxy": [\n            295.25,\n            451.0,\n            318.75,\n            483.00000000000006\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_23",\n          "bbox_xyxy": [\n            454.25,\n            472.0,\n            478.75,\n            505.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_24",\n          "bbox_xyxy": [\n            340.74999999999994,\n            454.25,\n            360.25,\n            483.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_25",\n          "bbox_xyxy": [\n            168.0,\n            189.75,\n            189.00000000000003,\n            220.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_26",\n          "bbox_xyxy": [\n            465.99999999999994,\n            522.0,\n            490.0,\n            557.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_27",\n          "bbox_xyxy": [\n            324.74999999999994,\n            560.25,\n            356.25,\n            588.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_28",\n          "bbox_xyxy": [\n            268.5,\n            367.75,\n            290.5,\n            400.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_29",\n          "bbox_xyxy": [\n            83.5,\n            171.74999999999997,\n            109.50000000000001,\n            196.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_30",\n          "bbox_xyxy": [\n            468.25,\n            390.99999999999994,\n            495.75,\n            420.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_31",\n          "bbox_xyxy": [\n            380.0,\n            490.5,\n            405.99999999999994,\n            519.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_32",\n          "bbox_xyxy": [\n            12.25,\n            356.0,\n            37.75,\n            384.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_33",\n          "bbox_xyxy": [\n            82.00000000000001,\n            104.75000000000001,\n            101.0,\n            141.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_34",\n          "bbox_xyxy": [\n            429.5,\n            496.0,\n            455.5,\n            527.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_35",\n          "bbox_xyxy": [\n            504.75,\n            341.25,\n            527.25,\n            372.75000000000006\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_36",\n          "bbox_xyxy": [\n            411.25,\n            338.00000000000006,\n            430.75,\n            376.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_37",\n          "bbox_xyxy": [\n            494.0,\n            480.5,\n            521.0,\n            510.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_38",\n          "bbox_xyxy": [\n            46.75000000000001,\n            203.25,\n            61.25,\n            240.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_39",\n          "bbox_xyxy": [\n            448.00000000000006,\n            278.0,\n            469.0,\n            316.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_40",\n          "bbox_xyxy": [\n            512.75,\n            521.75,\n            531.25,\n            565.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_41",\n          "bbox_xyxy": [\n            284.0,\n            172.5,\n            308.0,\n            213.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_42",\n          "bbox_xyxy": [\n            38.25,\n            148.25,\n            59.75000000000001,\n            184.74999999999997\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_43",\n          "bbox_xyxy": [\n            243.25000000000003,\n            202.0,\n            263.75,\n            238.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_44",\n          "bbox_xyxy": [\n            177.24999999999997,\n            399.49999999999994,\n            201.75,\n            426.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_45",\n          "bbox_xyxy": [\n            134.25,\n            431.5,\n            155.75,\n            466.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_46",\n          "bbox_xyxy": [\n            390.5,\n            216.50000000000003,\n            415.5,\n            249.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_47",\n          "bbox_xyxy": [\n            81.25,\n            437.24999999999994,\n            105.75,\n            463.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_48",\n          "bbox_xyxy": [\n            389.5,\n            258.25,\n            407.5,\n            295.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_49",\n          "bbox_xyxy": [\n            110.5,\n            494.25,\n            134.5,\n            538.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_50",\n          "bbox_xyxy": [\n            490.25,\n            258.75,\n            507.75,\n            296.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_51",\n          "bbox_xyxy": [\n            167.5,\n            293.0,\n            188.5,\n            319.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_52",\n          "bbox_xyxy": [\n            424.25,\n            573.75,\n            453.75,\n            601.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_53",\n          "bbox_xyxy": [\n            107.99999999999999,\n            233.0,\n            131.0,\n            255.99999999999997\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_54",\n          "bbox_xyxy": [\n            237.0,\n            601.5,\n            262.0,\n            639.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_55",\n          "bbox_xyxy": [\n            364.5,\n            404.75,\n            382.5,\n            442.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_56",\n          "bbox_xyxy": [\n            76.5,\n            55.0,\n            97.5,\n            87.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_57",\n          "bbox_xyxy": [\n            361.5,\n            311.25,\n            379.49999999999994,\n            350.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_58",\n          "bbox_xyxy": [\n            126.25,\n            379.25,\n            151.75,\n            405.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_59",\n          "bbox_xyxy": [\n            183.25,\n            521.75,\n            201.75,\n            561.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_60",\n          "bbox_xyxy": [\n            11.5,\n            304.0,\n            29.499999999999996,\n            339.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_61",\n          "bbox_xyxy": [\n            114.50000000000001,\n            102.25,\n            136.5,\n            129.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_62",\n          "bbox_xyxy": [\n            429.74999999999994,\n            155.00000000000003,\n            446.25,\n            192.00000000000003\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_63",\n          "bbox_xyxy": [\n            34.25,\n            608.5,\n            61.75,\n            637.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_64",\n          "bbox_xyxy": [\n            30.25,\n            265.75,\n            47.75,\n            297.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_65",\n          "bbox_xyxy": [\n            241.5,\n            350.0,\n            268.5,\n            375.99999999999994\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_66",\n          "bbox_xyxy": [\n            371.0,\n            564.0,\n            398.00000000000006,\n            599.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_67",\n          "bbox_xyxy": [\n            45.5,\n            289.25,\n            71.5,\n            322.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_68",\n          "bbox_xyxy": [\n            83.75,\n            255.25,\n            102.25000000000001,\n            289.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_69",\n          "bbox_xyxy": [\n            325.0,\n            510.74999999999994,\n            350.99999999999994,\n            542.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_70",\n          "bbox_xyxy": [\n            50.0,\n            356.50000000000006,\n            75.0,\n            392.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_71",\n          "bbox_xyxy": [\n            304.75,\n            314.25,\n            329.25,\n            350.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_181820ebba6cf9a7_72",\n          "bbox_xyxy": [\n            478.00000000000006,\n            620.0,\n            506.0,\n            640.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/test/images/IMG_1139_JPG.rf.e8db0b2ee5cddca3b9dc9d50e7e4e7e1.jpg",\n        "size": 42535,\n        "compressed": 42316,\n        "offset": 316842002,\n        "crc32": "ce896162",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_1b3bff70af4223e5",\n      "relative_path": "images/IMG_1144_JPG.rf.c253cc7f8c5120023c49a69f2a0c411b.jpg",\n      "sha256": "df0236ed3ce4ec1a9e54d8dd43a42f6ebfd21c5fe56cd2f73531d13d525be268",\n      "decoded_sha256": "ced07b76209764ef48adfedf153684fe4584df24892780e7e3efde5f1b7e937d",\n      "dhash": "2bce472d32541b95",\n      "bytes": 50917,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_1144_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "test",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_1b3bff70af4223e5_0",\n          "bbox_xyxy": [\n            108.75,\n            434.5,\n            145.25,\n            467.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_1",\n          "bbox_xyxy": [\n            453.24999999999994,\n            305.25,\n            487.75,\n            339.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_2",\n          "bbox_xyxy": [\n            459.75,\n            419.5,\n            488.24999999999994,\n            467.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_3",\n          "bbox_xyxy": [\n            281.25,\n            511.0,\n            313.75,\n            553.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_4",\n          "bbox_xyxy": [\n            488.75,\n            497.5,\n            524.25,\n            531.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_5",\n          "bbox_xyxy": [\n            76.75,\n            77.75,\n            98.25,\n            124.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_6",\n          "bbox_xyxy": [\n            557.75,\n            86.75,\n            583.25,\n            138.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_7",\n          "bbox_xyxy": [\n            474.49999999999994,\n            244.5,\n            506.5,\n            276.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_8",\n          "bbox_xyxy": [\n            284.5,\n            353.25,\n            310.5,\n            398.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_9",\n          "bbox_xyxy": [\n            455.25,\n            569.0,\n            480.74999999999994,\n            620.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_10",\n          "bbox_xyxy": [\n            98.75,\n            230.0,\n            127.25000000000001,\n            278.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_11",\n          "bbox_xyxy": [\n            340.75,\n            476.0,\n            368.25,\n            514.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_12",\n          "bbox_xyxy": [\n            348.24999999999994,\n            163.49999999999997,\n            377.75,\n            214.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_13",\n          "bbox_xyxy": [\n            147.75,\n            141.25,\n            177.25,\n            189.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_14",\n          "bbox_xyxy": [\n            337.5,\n            371.75000000000006,\n            368.49999999999994,\n            412.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_15",\n          "bbox_xyxy": [\n            90.5,\n            316.75,\n            119.5,\n            361.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_16",\n          "bbox_xyxy": [\n            56.5,\n            566.0,\n            88.5,\n            600.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_17",\n          "bbox_xyxy": [\n            487.5,\n            565.25,\n            516.5,\n            602.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_18",\n          "bbox_xyxy": [\n            324.25,\n            574.75,\n            358.75,\n            611.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_19",\n          "bbox_xyxy": [\n            439.00000000000006,\n            230.5,\n            460.0,\n            279.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_20",\n          "bbox_xyxy": [\n            582.75,\n            189.25,\n            611.25,\n            225.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_21",\n          "bbox_xyxy": [\n            532.5,\n            421.25,\n            560.5,\n            467.75000000000006\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_22",\n          "bbox_xyxy": [\n            19.5,\n            364.25,\n            48.5,\n            398.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_23",\n          "bbox_xyxy": [\n            130.25,\n            595.75,\n            157.75,\n            639.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_24",\n          "bbox_xyxy": [\n            471.5,\n            22.25,\n            503.5,\n            51.75000000000001\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_25",\n          "bbox_xyxy": [\n            19.75,\n            477.5,\n            45.25,\n            525.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_26",\n          "bbox_xyxy": [\n            257.5,\n            216.0,\n            286.5,\n            254.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_27",\n          "bbox_xyxy": [\n            302.5,\n            216.0,\n            325.5,\n            259.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_28",\n          "bbox_xyxy": [\n            188.5,\n            111.00000000000001,\n            212.5,\n            160.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_29",\n          "bbox_xyxy": [\n            353.5,\n            321.75000000000006,\n            395.50000000000006,\n            355.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_30",\n          "bbox_xyxy": [\n            0.0,\n            430.0,\n            25.0,\n            467.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_31",\n          "bbox_xyxy": [\n            215.99999999999997,\n            428.0,\n            247.0,\n            470.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_32",\n          "bbox_xyxy": [\n            174.25,\n            592.75,\n            204.74999999999997,\n            628.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_33",\n          "bbox_xyxy": [\n            374.5,\n            554.25,\n            410.5,\n            590.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_34",\n          "bbox_xyxy": [\n            415.0,\n            258.0,\n            439.00000000000006,\n            306.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_35",\n          "bbox_xyxy": [\n            547.75,\n            307.25,\n            573.25,\n            353.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_36",\n          "bbox_xyxy": [\n            501.0,\n            136.25,\n            539.0,\n            170.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_37",\n          "bbox_xyxy": [\n            97.00000000000001,\n            598.0,\n            119.0,\n            640.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_38",\n          "bbox_xyxy": [\n            465.75,\n            149.25,\n            499.25,\n            180.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_39",\n          "bbox_xyxy": [\n            182.00000000000003,\n            212.75000000000003,\n            215.0,\n            243.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_40",\n          "bbox_xyxy": [\n            402.5,\n            14.749999999999996,\n            433.49999999999994,\n            56.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_41",\n          "bbox_xyxy": [\n            221.0,\n            359.25000000000006,\n            244.0,\n            409.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_42",\n          "bbox_xyxy": [\n            309.49999999999994,\n            316.75,\n            331.5,\n            360.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_43",\n          "bbox_xyxy": [\n            418.0,\n            404.0,\n            445.0,\n            443.00000000000006\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_44",\n          "bbox_xyxy": [\n            29.5,\n            267.25,\n            55.5,\n            311.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_45",\n          "bbox_xyxy": [\n            134.0,\n            303.5,\n            161.0,\n            350.50000000000006\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_46",\n          "bbox_xyxy": [\n            251.25,\n            567.75,\n            280.75,\n            607.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_47",\n          "bbox_xyxy": [\n            221.25,\n            519.25,\n            251.74999999999997,\n            560.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_48",\n          "bbox_xyxy": [\n            97.0,\n            519.0,\n            128.0,\n            562.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_49",\n          "bbox_xyxy": [\n            386.5,\n            148.25,\n            410.50000000000006,\n            192.75000000000003\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_50",\n          "bbox_xyxy": [\n            535.0,\n            28.499999999999996,\n            571.0,\n            62.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_51",\n          "bbox_xyxy": [\n            218.0,\n            157.0,\n            242.0,\n            202.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_52",\n          "bbox_xyxy": [\n            280.0,\n            166.0,\n            311.0,\n            201.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_53",\n          "bbox_xyxy": [\n            485.75,\n            370.25,\n            513.25,\n            419.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_54",\n          "bbox_xyxy": [\n            394.25,\n            486.0,\n            421.75,\n            537.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_55",\n          "bbox_xyxy": [\n            320.75,\n            39.0,\n            351.25,\n            72.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_56",\n          "bbox_xyxy": [\n            47.75,\n            177.75,\n            70.25,\n            221.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_57",\n          "bbox_xyxy": [\n            278.5,\n            605.25,\n            311.5,\n            636.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_58",\n          "bbox_xyxy": [\n            570.0,\n            566.25,\n            592.0,\n            619.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_59",\n          "bbox_xyxy": [\n            42.25,\n            457.0,\n            68.75,\n            497.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1b3bff70af4223e5_60",\n          "bbox_xyxy": [\n            159.25,\n            477.5,\n            201.75,\n            518.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/test/images/IMG_1144_JPG.rf.c253cc7f8c5120023c49a69f2a0c411b.jpg",\n        "size": 50917,\n        "compressed": 50776,\n        "offset": 316884490,\n        "crc32": "fe32012c",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_1bebd35cebe80e65",\n      "relative_path": "images/IMG_2077_JPG.rf.2a82b2ef1205309192a7f702af330b19.jpg",\n      "sha256": "cefe909b7c017d70bef23deb567d2a5b2ec275b27d7e111c2c201c32e7170b7c",\n      "decoded_sha256": "afe726d1198956617ebbc4154f4e39814235efdea7088f154bfa248991f23409",\n      "dhash": "2cb1333070703032",\n      "bytes": 27554,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2077_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "test",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_1bebd35cebe80e65_0",\n          "bbox_xyxy": [\n            206.0,\n            110.50000000000004,\n            526.0,\n            608.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/test/images/IMG_2077_JPG.rf.2a82b2ef1205309192a7f702af330b19.jpg",\n        "size": 27554,\n        "compressed": 26820,\n        "offset": 317977623,\n        "crc32": "a341cbc3",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_1d37a37c89bbe81c",\n      "relative_path": "images/IMG_4170_JPG.rf.048f84546a438657f582f643409bd309.jpg",\n      "sha256": "bf8223392d8b18460e6a885acf12d2bd55d785e313fb641ae0e1384e8b3854bb",\n      "decoded_sha256": "911f3bbe7999d01a9e89dd493f807958f6c26e145dac8eff15f7a0fe07cbba03",\n      "dhash": "666bf1990ec9d292",\n      "bytes": 61696,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_4170_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "test",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_1d37a37c89bbe81c_0",\n          "bbox_xyxy": [\n            113.25,\n            152.25,\n            151.75,\n            208.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1d37a37c89bbe81c_1",\n          "bbox_xyxy": [\n            259.25,\n            213.0,\n            290.75,\n            257.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1d37a37c89bbe81c_2",\n          "bbox_xyxy": [\n            268.75,\n            283.5,\n            328.25,\n            303.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1d37a37c89bbe81c_3",\n          "bbox_xyxy": [\n            413.5,\n            191.0,\n            470.50000000000006,\n            231.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1d37a37c89bbe81c_4",\n          "bbox_xyxy": [\n            416.0,\n            286.25,\n            444.0,\n            342.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1d37a37c89bbe81c_5",\n          "bbox_xyxy": [\n            479.0,\n            411.75,\n            534.0,\n            449.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1d37a37c89bbe81c_6",\n          "bbox_xyxy": [\n            398.5,\n            415.25,\n            425.50000000000006,\n            460.74999999999994\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1d37a37c89bbe81c_7",\n          "bbox_xyxy": [\n            197.5,\n            346.75,\n            217.5,\n            389.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1d37a37c89bbe81c_8",\n          "bbox_xyxy": [\n            98.25000000000001,\n            555.25,\n            122.75,\n            611.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1d37a37c89bbe81c_9",\n          "bbox_xyxy": [\n            177.75,\n            581.0,\n            195.25,\n            631.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/test/images/IMG_4170_JPG.rf.048f84546a438657f582f643409bd309.jpg",\n        "size": 61696,\n        "compressed": 61559,\n        "offset": 320369978,\n        "crc32": "e3b04498",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_1e2158bb63117540",\n      "relative_path": "images/IMG_1172_JPG.rf.125f13a7d8fa84ffdb5c7473aa369ca7.jpg",\n      "sha256": "bba110aa64b9ba8b84642b5f4afaa52a1611d6106e37bca3ec2f7064aea3e3e2",\n      "decoded_sha256": "8e4d928c5fe3945597bddeb7d91ab9184501e766fae139f6954749a0a596f210",\n      "dhash": "0e172d7356bd2ab3",\n      "bytes": 42906,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_1172_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "test",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_1e2158bb63117540_0",\n          "bbox_xyxy": [\n            297.75,\n            565.75,\n            333.25,\n            603.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2158bb63117540_1",\n          "bbox_xyxy": [\n            237.25,\n            554.0,\n            273.75,\n            593.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2158bb63117540_2",\n          "bbox_xyxy": [\n            232.0,\n            465.25,\n            267.0,\n            505.74999999999994\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2158bb63117540_3",\n          "bbox_xyxy": [\n            448.75,\n            71.0,\n            483.25,\n            121.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2158bb63117540_4",\n          "bbox_xyxy": [\n            514.0,\n            166.25,\n            544.0,\n            209.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2158bb63117540_5",\n          "bbox_xyxy": [\n            351.25,\n            265.5,\n            383.75,\n            320.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2158bb63117540_6",\n          "bbox_xyxy": [\n            505.5,\n            79.25,\n            542.5,\n            115.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2158bb63117540_7",\n          "bbox_xyxy": [\n            383.75,\n            371.5,\n            417.24999999999994,\n            419.49999999999994\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2158bb63117540_8",\n          "bbox_xyxy": [\n            489.25000000000006,\n            382.5,\n            517.75,\n            425.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2158bb63117540_9",\n          "bbox_xyxy": [\n            358.0,\n            574.5,\n            388.0,\n            611.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2158bb63117540_10",\n          "bbox_xyxy": [\n            437.75,\n            409.0,\n            480.25,\n            464.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2158bb63117540_11",\n          "bbox_xyxy": [\n            452.5,\n            177.24999999999997,\n            476.50000000000006,\n            231.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2158bb63117540_12",\n          "bbox_xyxy": [\n            139.75,\n            296.25,\n            167.25,\n            346.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2158bb63117540_13",\n          "bbox_xyxy": [\n            541.25,\n            375.0,\n            571.75,\n            417.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2158bb63117540_14",\n          "bbox_xyxy": [\n            93.0,\n            571.5,\n            117.0,\n            623.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2158bb63117540_15",\n          "bbox_xyxy": [\n            304.0,\n            322.75,\n            332.0,\n            368.24999999999994\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2158bb63117540_16",\n          "bbox_xyxy": [\n            189.75,\n            603.25,\n            236.25,\n            639.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2158bb63117540_17",\n          "bbox_xyxy": [\n            465.75,\n            305.25,\n            498.25,\n            346.75000000000006\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2158bb63117540_18",\n          "bbox_xyxy": [\n            540.0,\n            26.500000000000004,\n            574.0,\n            61.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2158bb63117540_19",\n          "bbox_xyxy": [\n            514.75,\n            478.5,\n            547.25,\n            525.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2158bb63117540_20",\n          "bbox_xyxy": [\n            570.25,\n            477.0,\n            591.75,\n            532.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2158bb63117540_21",\n          "bbox_xyxy": [\n            364.25000000000006,\n            417.75000000000006,\n            397.75,\n            456.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2158bb63117540_22",\n          "bbox_xyxy": [\n            209.25,\n            309.0,\n            237.75000000000003,\n            349.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2158bb63117540_23",\n          "bbox_xyxy": [\n            541.25,\n            301.99999999999994,\n            574.75,\n            344.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2158bb63117540_24",\n          "bbox_xyxy": [\n            616.0,\n            378.0,\n            640.0,\n            422.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2158bb63117540_25",\n          "bbox_xyxy": [\n            383.75,\n            506.75,\n            418.25,\n            539.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2158bb63117540_26",\n          "bbox_xyxy": [\n            125.75000000000001,\n            443.75,\n            160.25000000000003,\n            479.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2158bb63117540_27",\n          "bbox_xyxy": [\n            309.0,\n            419.25000000000006,\n            336.0,\n            462.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2158bb63117540_28",\n          "bbox_xyxy": [\n            244.25,\n            193.0,\n            280.75,\n            225.99999999999997\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2158bb63117540_29",\n          "bbox_xyxy": [\n            268.25,\n            258.0,\n            303.75,\n            305.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2158bb63117540_30",\n          "bbox_xyxy": [\n            552.0,\n            588.5,\n            582.0,\n            628.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2158bb63117540_31",\n          "bbox_xyxy": [\n            390.25,\n            170.25,\n            420.74999999999994,\n            222.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2158bb63117540_32",\n          "bbox_xyxy": [\n            324.0,\n            135.25,\n            353.00000000000006,\n            179.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2158bb63117540_33",\n          "bbox_xyxy": [\n            463.0,\n            540.25,\n            498.0,\n            581.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2158bb63117540_34",\n          "bbox_xyxy": [\n            510.50000000000006,\n            581.25,\n            551.5,\n            621.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2158bb63117540_35",\n          "bbox_xyxy": [\n            415.5,\n            617.5,\n            434.5,\n            639.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2158bb63117540_36",\n          "bbox_xyxy": [\n            594.75,\n            588.0,\n            621.25,\n            638.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2158bb63117540_37",\n          "bbox_xyxy": [\n            603.75,\n            207.25,\n            626.25,\n            264.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2158bb63117540_38",\n          "bbox_xyxy": [\n            621.25,\n            158.25,\n            639.75,\n            200.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2158bb63117540_39",\n          "bbox_xyxy": [\n            585.75,\n            105.25,\n            619.25,\n            142.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2158bb63117540_40",\n          "bbox_xyxy": [\n            404.5,\n            0.0,\n            439.5,\n            23.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_1e2158bb63117540_41",\n          "bbox_xyxy": [\n            355.0,\n            58.25,\n            382.0,\n            89.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/test/images/IMG_1172_JPG.rf.125f13a7d8fa84ffdb5c7473aa369ca7.jpg",\n        "size": 42906,\n        "compressed": 42688,\n        "offset": 317067019,\n        "crc32": "f72d0d8a",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_207f348f48f2c2a6",\n      "relative_path": "images/IMG_2120_JPG.rf.861fc0acb02a64552d5ac1b04e7ac481.jpg",\n      "sha256": "87071433016af4fa2d5a510f176a1dc50909551d2959388c0c829a7090b830c7",\n      "decoded_sha256": "bd321e8c41d892cb4f1d437d29ba557f113e01f271708f402acb599c8699fd82",\n      "dhash": "4830706171713030",\n      "bytes": 25979,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2120_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "test",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_207f348f48f2c2a6_0",\n          "bbox_xyxy": [\n            149.5,\n            46.25,\n            545.5,\n            596.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/test/images/IMG_2120_JPG.rf.861fc0acb02a64552d5ac1b04e7ac481.jpg",\n        "size": 25979,\n        "compressed": 25375,\n        "offset": 318056761,\n        "crc32": "ea75e845",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_222d965e84d05f12",\n      "relative_path": "images/IMG_1186_JPG.rf.7ddf9172c32a92a9ddaef119e2cdb0a2.jpg",\n      "sha256": "e5c82ad62882d96ed9b7de85c1d3dc9b1f92ecf525ac9f06818593bcaf013b59",\n      "decoded_sha256": "3dcaab3fe759c6bb3f6b68082798222f1639fccf810d5e16c38096805d978ab8",\n      "dhash": "121d1f451b2b171b",\n      "bytes": 46035,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_1186_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "test",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_222d965e84d05f12_0",\n          "bbox_xyxy": [\n            337.5,\n            196.5,\n            377.5,\n            237.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_222d965e84d05f12_1",\n          "bbox_xyxy": [\n            380.75,\n            327.5,\n            418.25,\n            380.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_222d965e84d05f12_2",\n          "bbox_xyxy": [\n            61.99999999999999,\n            467.5,\n            94.0,\n            527.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_222d965e84d05f12_3",\n          "bbox_xyxy": [\n            449.0,\n            223.75,\n            491.0,\n            277.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_222d965e84d05f12_4",\n          "bbox_xyxy": [\n            575.25,\n            342.5,\n            611.75,\n            387.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_222d965e84d05f12_5",\n          "bbox_xyxy": [\n            199.5,\n            221.74999999999997,\n            228.49999999999997,\n            281.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_222d965e84d05f12_6",\n          "bbox_xyxy": [\n            299.99999999999994,\n            321.75000000000006,\n            345.99999999999994,\n            362.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_222d965e84d05f12_7",\n          "bbox_xyxy": [\n            90.5,\n            164.5,\n            119.5,\n            224.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_222d965e84d05f12_8",\n          "bbox_xyxy": [\n            578.5,\n            517.25,\n            621.5,\n            561.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_222d965e84d05f12_9",\n          "bbox_xyxy": [\n            54.0,\n            589.25,\n            84.0,\n            639.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_222d965e84d05f12_10",\n          "bbox_xyxy": [\n            84.5,\n            317.25,\n            110.5,\n            379.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_222d965e84d05f12_11",\n          "bbox_xyxy": [\n            333.75,\n            431.99999999999994,\n            359.25,\n            494.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_222d965e84d05f12_12",\n          "bbox_xyxy": [\n            276.75,\n            516.75,\n            322.25,\n            563.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_222d965e84d05f12_13",\n          "bbox_xyxy": [\n            322.75,\n            65.5,\n            357.25,\n            122.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_222d965e84d05f12_14",\n          "bbox_xyxy": [\n            538.75,\n            482.25,\n            567.25,\n            540.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_222d965e84d05f12_15",\n          "bbox_xyxy": [\n            394.74999999999994,\n            447.5,\n            436.25,\n            493.49999999999994\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_222d965e84d05f12_16",\n          "bbox_xyxy": [\n            380.25,\n            550.25,\n            409.75,\n            624.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_222d965e84d05f12_17",\n          "bbox_xyxy": [\n            553.5,\n            199.5,\n            591.5,\n            245.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_222d965e84d05f12_18",\n          "bbox_xyxy": [\n            25.999999999999996,\n            205.75,\n            75.0,\n            255.25000000000003\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_222d965e84d05f12_19",\n          "bbox_xyxy": [\n            470.0,\n            480.25,\n            495.0,\n            545.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_222d965e84d05f12_20",\n          "bbox_xyxy": [\n            0.0,\n            343.00000000000006,\n            33.0,\n            389.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_222d965e84d05f12_21",\n          "bbox_xyxy": [\n            142.0,\n            441.5,\n            167.0,\n            503.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_222d965e84d05f12_22",\n          "bbox_xyxy": [\n            153.5,\n            329.5,\n            196.5,\n            370.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_222d965e84d05f12_23",\n          "bbox_xyxy": [\n            544.75,\n            15.249999999999995,\n            563.25,\n            66.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_222d965e84d05f12_24",\n          "bbox_xyxy": [\n            234.74999999999997,\n            409.25,\n            263.25,\n            470.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_222d965e84d05f12_25",\n          "bbox_xyxy": [\n            550.75,\n            617.0,\n            579.25,\n            640.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_222d965e84d05f12_26",\n          "bbox_xyxy": [\n            0.25000000000000133,\n            582.5,\n            30.75,\n            619.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_222d965e84d05f12_27",\n          "bbox_xyxy": [\n            140.25,\n            601.25,\n            182.75,\n            639.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_222d965e84d05f12_28",\n          "bbox_xyxy": [\n            610.0,\n            381.0,\n            640.0,\n            426.99999999999994\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_222d965e84d05f12_29",\n          "bbox_xyxy": [\n            420.50000000000006,\n            75.5,\n            448.5,\n            129.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_222d965e84d05f12_30",\n          "bbox_xyxy": [\n            489.0,\n            364.5,\n            527.0,\n            424.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/test/images/IMG_1186_JPG.rf.7ddf9172c32a92a9ddaef119e2cdb0a2.jpg",\n        "size": 46035,\n        "compressed": 45806,\n        "offset": 317151867,\n        "crc32": "1f53a716",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_2237683a85b6ea4b",\n      "relative_path": "images/IMG_2595_JPG.rf.d602d18963ca9a1c37a9ede4aa6e2ed3.jpg",\n      "sha256": "1a50f175b85a21ec1f13f00fcbac30fa6a029bce763bbcccc25432eb0fc267ed",\n      "decoded_sha256": "2593c5f514bbca169b5fece2f4efc6209d77251b56c0f712a9ea245fb35d39a2",\n      "dhash": "0a72717961717170",\n      "bytes": 27712,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2595_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "test",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_2237683a85b6ea4b_0",\n          "bbox_xyxy": [\n            151.25,\n            27.500000000000036,\n            454.75,\n            616.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/test/images/IMG_2595_JPG.rf.d602d18963ca9a1c37a9ede4aa6e2ed3.jpg",\n        "size": 27712,\n        "compressed": 27190,\n        "offset": 319331795,\n        "crc32": "19e0899f",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_23b37646a7a9b21b",\n      "relative_path": "images/IMG_4180_JPG.rf.3f8ae42257e8a8bb85f385f8ba61e1ae.jpg",\n      "sha256": "7db6f10e1d0f6d9aba96d8dd5be50b24e86cf9a81bf531f1c03bd3835deab4b8",\n      "decoded_sha256": "a1a17bb16b4a33ce7b339ee7b28d8613d107999225e54f4a76a29c1f77cfe313",\n      "dhash": "965258f4d0da9021",\n      "bytes": 48005,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_4180_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "test",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_23b37646a7a9b21b_0",\n          "bbox_xyxy": [\n            168.0,\n            97.25,\n            200.99999999999997,\n            152.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_23b37646a7a9b21b_1",\n          "bbox_xyxy": [\n            190.5,\n            168.5,\n            238.49999999999997,\n            198.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_23b37646a7a9b21b_2",\n          "bbox_xyxy": [\n            269.25,\n            164.5,\n            315.75,\n            184.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_23b37646a7a9b21b_3",\n          "bbox_xyxy": [\n            314.75,\n            201.5,\n            367.25,\n            221.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_23b37646a7a9b21b_4",\n          "bbox_xyxy": [\n            207.0,\n            130.5,\n            242.0,\n            154.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_23b37646a7a9b21b_5",\n          "bbox_xyxy": [\n            153.25,\n            181.75,\n            184.74999999999997,\n            220.25000000000003\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_23b37646a7a9b21b_6",\n          "bbox_xyxy": [\n            234.0,\n            245.25,\n            274.0,\n            269.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_23b37646a7a9b21b_7",\n          "bbox_xyxy": [\n            278.5,\n            228.75,\n            305.5,\n            261.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_23b37646a7a9b21b_8",\n          "bbox_xyxy": [\n            182.75,\n            285.5,\n            217.25,\n            311.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_23b37646a7a9b21b_9",\n          "bbox_xyxy": [\n            304.75,\n            330.74999999999994,\n            344.25000000000006,\n            360.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_23b37646a7a9b21b_10",\n          "bbox_xyxy": [\n            479.25000000000006,\n            228.49999999999997,\n            514.75,\n            255.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_23b37646a7a9b21b_11",\n          "bbox_xyxy": [\n            530.0,\n            404.75,\n            547.0,\n            448.24999999999994\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_23b37646a7a9b21b_12",\n          "bbox_xyxy": [\n            359.75,\n            455.0,\n            380.25,\n            490.99999999999994\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_23b37646a7a9b21b_13",\n          "bbox_xyxy": [\n            449.0,\n            488.5,\n            463.00000000000006,\n            536.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_23b37646a7a9b21b_14",\n          "bbox_xyxy": [\n            328.75,\n            522.25,\n            364.25,\n            545.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_23b37646a7a9b21b_15",\n          "bbox_xyxy": [\n            272.5,\n            522.0,\n            304.5,\n            550.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_23b37646a7a9b21b_16",\n          "bbox_xyxy": [\n            105.49999999999999,\n            500.25,\n            153.5,\n            524.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_23b37646a7a9b21b_17",\n          "bbox_xyxy": [\n            282.0,\n            432.75,\n            319.0,\n            459.25000000000006\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_23b37646a7a9b21b_18",\n          "bbox_xyxy": [\n            118.0,\n            368.25,\n            160.0,\n            383.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_23b37646a7a9b21b_19",\n          "bbox_xyxy": [\n            335.50000000000006,\n            289.0,\n            363.5,\n            330.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_23b37646a7a9b21b_20",\n          "bbox_xyxy": [\n            141.0,\n            233.49999999999997,\n            176.0,\n            269.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/test/images/IMG_4180_JPG.rf.3f8ae42257e8a8bb85f385f8ba61e1ae.jpg",\n        "size": 48005,\n        "compressed": 47824,\n        "offset": 320431709,\n        "crc32": "dca2c1a1",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_24dffac79439f76e",\n      "relative_path": "images/IMG_2497_JPG.rf.c70019c1b105042b87560240fbe949c5.jpg",\n      "sha256": "fa04c4cdb8c80a5c7a1117ee997b09568c53721ac48b19e1db94eac4c16b0504",\n      "decoded_sha256": "a5065683689053fde375d5f7b02a4175bad204a6c70d2eeec2555e286a15733d",\n      "dhash": "4d327168687172b0",\n      "bytes": 30475,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2497_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "test",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_24dffac79439f76e_0",\n          "bbox_xyxy": [\n            104.25,\n            12.250000000000014,\n            540.75,\n            567.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/test/images/IMG_2497_JPG.rf.c70019c1b105042b87560240fbe949c5.jpg",\n        "size": 30475,\n        "compressed": 30030,\n        "offset": 319042045,\n        "crc32": "5d5af802",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_253cc2e343935772",\n      "relative_path": "images/IMG_1783_JPG.rf.5769906ec198f740aacde7f40e0ff406.jpg",\n      "sha256": "489336d7cd209f88f7d91085c892c9fc461b782c82bf7d092e17820017d2e4c3",\n      "decoded_sha256": "078a12e641fd9e8c20aebfea2e07694ab38353c68b0a6c62a72a83a04ee8eb61",\n      "dhash": "647030797078f0f0",\n      "bytes": 30099,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_1783_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "test",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_253cc2e343935772_0",\n          "bbox_xyxy": [\n            150.0,\n            87.5,\n            530.0,\n            612.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/test/images/IMG_1783_JPG.rf.5769906ec198f740aacde7f40e0ff406.jpg",\n        "size": 30099,\n        "compressed": 29627,\n        "offset": 317398220,\n        "crc32": "862cb1a2",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_2ad443d154cfb4e4",\n      "relative_path": "images/IMG_4169_JPG.rf.fd3cab1dde2ed0530cc5ee6b41fd28d5.jpg",\n      "sha256": "8560b5ee69d9ed5fec21ae79ca2cd51ee6017b74e3a57a2687b1296dac4539dd",\n      "decoded_sha256": "502e53cb0109defc77e478c91f4a26c13b4718f8a53dd4a44e58950c885f62cc",\n      "dhash": "7366d4d3d4d06867",\n      "bytes": 50544,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_4169_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "test",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_2ad443d154cfb4e4_0",\n          "bbox_xyxy": [\n            463.0,\n            520.25,\n            505.0,\n            553.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2ad443d154cfb4e4_1",\n          "bbox_xyxy": [\n            368.5,\n            501.75000000000006,\n            410.50000000000006,\n            537.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2ad443d154cfb4e4_2",\n          "bbox_xyxy": [\n            386.75,\n            484.75,\n            439.25,\n            510.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2ad443d154cfb4e4_3",\n          "bbox_xyxy": [\n            286.75,\n            478.25,\n            309.25,\n            525.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2ad443d154cfb4e4_4",\n          "bbox_xyxy": [\n            180.0,\n            438.75,\n            205.0,\n            480.25000000000006\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2ad443d154cfb4e4_5",\n          "bbox_xyxy": [\n            166.74999999999997,\n            391.75,\n            201.25,\n            423.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2ad443d154cfb4e4_6",\n          "bbox_xyxy": [\n            70.75,\n            333.24999999999994,\n            109.25,\n            352.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2ad443d154cfb4e4_7",\n          "bbox_xyxy": [\n            396.0,\n            265.5,\n            426.99999999999994,\n            307.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2ad443d154cfb4e4_8",\n          "bbox_xyxy": [\n            305.00000000000006,\n            426.0,\n            359.00000000000006,\n            451.99999999999994\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2ad443d154cfb4e4_9",\n          "bbox_xyxy": [\n            323.00000000000006,\n            209.0,\n            356.0,\n            250.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/test/images/IMG_4169_JPG.rf.fd3cab1dde2ed0530cc5ee6b41fd28d5.jpg",\n        "size": 50544,\n        "compressed": 50393,\n        "offset": 320319413,\n        "crc32": "c4d12e17",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_2b93cb41c285981f",\n      "relative_path": "images/IMG_3896_JPG.rf.e60eb6c163e10364a4164c7d7d357c9a.jpg",\n      "sha256": "df2290cd3a920194da299910f5f4492370c6ad81897e7219c96d92c889ff8adf",\n      "decoded_sha256": "ef7d20301f5e080ba64b49c9a37f5d85739e3c124443fed0c5196af1463b0288",\n      "dhash": "b498d0d8e8cca1f0",\n      "bytes": 26138,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_3896_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "test",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_2b93cb41c285981f_0",\n          "bbox_xyxy": [\n            463.00000000000006,\n            303.25,\n            524.0,\n            374.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2b93cb41c285981f_1",\n          "bbox_xyxy": [\n            443.0,\n            389.0,\n            503.0,\n            426.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2b93cb41c285981f_2",\n          "bbox_xyxy": [\n            207.75,\n            366.5,\n            277.25,\n            424.49999999999994\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2b93cb41c285981f_3",\n          "bbox_xyxy": [\n            238.0,\n            444.5,\n            303.0,\n            522.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2b93cb41c285981f_4",\n          "bbox_xyxy": [\n            78.25,\n            64.0,\n            159.75,\n            109.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2b93cb41c285981f_5",\n          "bbox_xyxy": [\n            156.75,\n            150.25,\n            234.25,\n            209.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2b93cb41c285981f_6",\n          "bbox_xyxy": [\n            82.5,\n            497.0,\n            160.5,\n            557.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2b93cb41c285981f_7",\n          "bbox_xyxy": [\n            390.5,\n            41.25,\n            477.5,\n            101.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2b93cb41c285981f_8",\n          "bbox_xyxy": [\n            268.5,\n            142.75,\n            339.49999999999994,\n            216.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2b93cb41c285981f_9",\n          "bbox_xyxy": [\n            218.25,\n            519.5,\n            280.75,\n            560.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2b93cb41c285981f_10",\n          "bbox_xyxy": [\n            254.74999999999997,\n            273.0,\n            319.25,\n            358.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2b93cb41c285981f_11",\n          "bbox_xyxy": [\n            348.49999999999994,\n            264.25,\n            407.5,\n            303.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2b93cb41c285981f_12",\n          "bbox_xyxy": [\n            272.5,\n            544.0,\n            361.5,\n            588.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2b93cb41c285981f_13",\n          "bbox_xyxy": [\n            143.0,\n            239.24999999999997,\n            192.0,\n            288.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2b93cb41c285981f_14",\n          "bbox_xyxy": [\n            439.25,\n            4.500000000000002,\n            488.75,\n            57.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2b93cb41c285981f_15",\n          "bbox_xyxy": [\n            106.49999999999999,\n            396.25,\n            155.5,\n            454.74999999999994\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2b93cb41c285981f_16",\n          "bbox_xyxy": [\n            290.75,\n            11.25,\n            347.25,\n            55.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2b93cb41c285981f_17",\n          "bbox_xyxy": [\n            348.00000000000006,\n            138.5,\n            379.0,\n            200.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2b93cb41c285981f_18",\n          "bbox_xyxy": [\n            426.25,\n            201.75,\n            483.75,\n            264.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2b93cb41c285981f_19",\n          "bbox_xyxy": [\n            379.25,\n            350.0,\n            433.75,\n            413.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2b93cb41c285981f_20",\n          "bbox_xyxy": [\n            147.25,\n            398.75,\n            194.75,\n            451.25\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/test/images/IMG_3896_JPG.rf.e60eb6c163e10364a4164c7d7d357c9a.jpg",\n        "size": 26138,\n        "compressed": 25607,\n        "offset": 319859453,\n        "crc32": "4b9c5457",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_2c68c512f851fdc0",\n      "relative_path": "images/IMG_4090_JPG.rf.febfe6ce8ced7f2fc3926ffa5afe3459.jpg",\n      "sha256": "c4ded1b5173f896a99ecb863f6bbcc2b8d584c04be119542eed8d0e7ff3740b8",\n      "decoded_sha256": "288370784f08165be964cf0260682ef0f868edd8ed377a7dd2979b0c246a850f",\n      "dhash": "a6baf4f0e8b4baec",\n      "bytes": 44844,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_4090_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "test",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_2c68c512f851fdc0_0",\n          "bbox_xyxy": [\n            304.25,\n            131.25,\n            347.75,\n            156.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2c68c512f851fdc0_1",\n          "bbox_xyxy": [\n            379.0,\n            131.0,\n            419.0,\n            159.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2c68c512f851fdc0_2",\n          "bbox_xyxy": [\n            302.25,\n            224.25,\n            340.74999999999994,\n            251.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2c68c512f851fdc0_3",\n          "bbox_xyxy": [\n            245.75,\n            246.75,\n            288.25,\n            289.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2c68c512f851fdc0_4",\n          "bbox_xyxy": [\n            348.24999999999994,\n            253.25000000000003,\n            399.75,\n            273.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2c68c512f851fdc0_5",\n          "bbox_xyxy": [\n            411.75000000000006,\n            300.25,\n            442.25,\n            335.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2c68c512f851fdc0_6",\n          "bbox_xyxy": [\n            330.25000000000006,\n            323.25,\n            368.75,\n            351.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2c68c512f851fdc0_7",\n          "bbox_xyxy": [\n            232.0,\n            327.25,\n            258.0,\n            380.74999999999994\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2c68c512f851fdc0_8",\n          "bbox_xyxy": [\n            260.0,\n            408.5,\n            285.0,\n            448.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2c68c512f851fdc0_9",\n          "bbox_xyxy": [\n            296.5,\n            433.25,\n            346.5,\n            466.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_FEMALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2c68c512f851fdc0_10",\n          "bbox_xyxy": [\n            351.75,\n            428.00000000000006,\n            409.25,\n            446.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2c68c512f851fdc0_11",\n          "bbox_xyxy": [\n            356.5,\n            489.5,\n            401.5,\n            517.5\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/test/images/IMG_4090_JPG.rf.febfe6ce8ced7f2fc3926ffa5afe3459.jpg",\n        "size": 44844,\n        "compressed": 44592,\n        "offset": 320060650,\n        "crc32": "3f2675f4",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_2d9c2f3033bca640",\n      "relative_path": "images/IMG_1031_JPG.rf.ce93899af3ce718b3837dcfc63f13ed8.jpg",\n      "sha256": "dfced1f3f6fdaa573377b99f0912df06579e38c361f433b5c3dd9b65e0fb6888",\n      "decoded_sha256": "ea8684609ef6adc1f1b5e7bc1a489824968fdd49b40b7ece839168ced9e3212d",\n      "dhash": "462d2b5e12931a2b",\n      "bytes": 46022,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_1031_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "test",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_2d9c2f3033bca640_0",\n          "bbox_xyxy": [\n            307.25,\n            325.5,\n            356.75000000000006,\n            392.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2d9c2f3033bca640_1",\n          "bbox_xyxy": [\n            475.50000000000006,\n            506.75,\n            533.5,\n            561.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2d9c2f3033bca640_2",\n          "bbox_xyxy": [\n            145.75,\n            326.0,\n            186.25,\n            403.00000000000006\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2d9c2f3033bca640_3",\n          "bbox_xyxy": [\n            393.24999999999994,\n            133.0,\n            447.75,\n            179.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2d9c2f3033bca640_4",\n          "bbox_xyxy": [\n            442.5,\n            60.24999999999999,\n            482.5,\n            138.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2d9c2f3033bca640_5",\n          "bbox_xyxy": [\n            267.75,\n            409.0,\n            320.25,\n            453.00000000000006\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2d9c2f3033bca640_6",\n          "bbox_xyxy": [\n            108.25,\n            426.0,\n            146.75,\n            513.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2d9c2f3033bca640_7",\n          "bbox_xyxy": [\n            403.75,\n            352.5,\n            444.25,\n            420.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2d9c2f3033bca640_8",\n          "bbox_xyxy": [\n            381.99999999999994,\n            458.49999999999994,\n            416.0,\n            542.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2d9c2f3033bca640_9",\n          "bbox_xyxy": [\n            204.5,\n            481.99999999999994,\n            244.5,\n            559.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2d9c2f3033bca640_10",\n          "bbox_xyxy": [\n            535.0,\n            344.75,\n            578.0,\n            407.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2d9c2f3033bca640_11",\n          "bbox_xyxy": [\n            173.25,\n            220.75,\n            222.75000000000003,\n            277.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2d9c2f3033bca640_12",\n          "bbox_xyxy": [\n            409.49999999999994,\n            271.75,\n            466.5,\n            320.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2d9c2f3033bca640_13",\n          "bbox_xyxy": [\n            110.5,\n            216.5,\n            159.5,\n            247.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2d9c2f3033bca640_14",\n          "bbox_xyxy": [\n            352.5,\n            524.5,\n            404.5,\n            576.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2d9c2f3033bca640_15",\n          "bbox_xyxy": [\n            254.74999999999997,\n            134.25,\n            284.25,\n            229.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2d9c2f3033bca640_16",\n          "bbox_xyxy": [\n            204.5,\n            128.75,\n            250.5,\n            210.25\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2d9c2f3033bca640_17",\n          "bbox_xyxy": [\n            524.0,\n            0.2499999999999991,\n            572.0,\n            65.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2d9c2f3033bca640_18",\n          "bbox_xyxy": [\n            121.00000000000001,\n            6.999999999999997,\n            175.0,\n            54.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2d9c2f3033bca640_19",\n          "bbox_xyxy": [\n            41.5,\n            337.0,\n            94.5,\n            368.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2d9c2f3033bca640_20",\n          "bbox_xyxy": [\n            15.499999999999998,\n            135.5,\n            66.5,\n            196.50000000000003\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2d9c2f3033bca640_21",\n          "bbox_xyxy": [\n            330.5,\n            230.49999999999997,\n            387.5,\n            267.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2d9c2f3033bca640_22",\n          "bbox_xyxy": [\n            490.25,\n            238.5,\n            537.75,\n            283.5\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2d9c2f3033bca640_23",\n          "bbox_xyxy": [\n            76.0,\n            113.25,\n            121.0,\n            175.75\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        },\n        {\n          "object_id": "rice_2d9c2f3033bca640_24",\n          "bbox_xyxy": [\n            280.0,\n            0.0,\n            308.0,\n            55.0\n          ],\n          "species": "white_stemborer",\n          "original_label": "WSB_MALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/test/images/IMG_1031_JPG.rf.ce93899af3ce718b3837dcfc63f13ed8.jpg",\n        "size": 46022,\n        "compressed": 45813,\n        "offset": 316377986,\n        "crc32": "04ad5a69",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_2e627baaf19feb91",\n      "relative_path": "images/IMG_2461_JPG.rf.16574f0568aefcf27d43bcd5dcd06dd4.jpg",\n      "sha256": "274909ec733281997a7676643b0a8f7cfddfc0ae3712f90d3220db8a10673c58",\n      "decoded_sha256": "9fffc7c701da0d673c5c66fe7878dfa873e6e91c93cc235da6bb2cbfc8d19eb9",\n      "dhash": "2832327171713032",\n      "bytes": 28319,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2461_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "test",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_2e627baaf19feb91_0",\n          "bbox_xyxy": [\n            169.50000000000003,\n            23.000000000000007,\n            479.5,\n            572.0\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/test/images/IMG_2461_JPG.rf.16574f0568aefcf27d43bcd5dcd06dd4.jpg",\n        "size": 28319,\n        "compressed": 27051,\n        "offset": 318985624,\n        "crc32": "ea9bd713",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_2f11d9ff5cee0f16",\n      "relative_path": "images/IMG_2702_JPG.rf.6021a3efcbd063cc885f79a3b0eba5f5.jpg",\n      "sha256": "a876a131ba5152b02cd33f8c6afb87bb602cd8827fd66ee53788045b9db6d9b1",\n      "decoded_sha256": "573ef418f770695461eff653a4338dca178be23e9e0bfc3aeb2d0cb8825b6431",\n      "dhash": "0c1a717878797132",\n      "bytes": 28544,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2702_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "test",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_2f11d9ff5cee0f16_0",\n          "bbox_xyxy": [\n            184.0,\n            49.24999999999997,\n            511.0,\n            611.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/test/images/IMG_2702_JPG.rf.6021a3efcbd063cc885f79a3b0eba5f5.jpg",\n        "size": 28544,\n        "compressed": 27793,\n        "offset": 319603979,\n        "crc32": "901cf2bc",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_30fe8a693294be5c",\n      "relative_path": "images/IMG_2659_JPG.rf.f8b9ee8cf3f828a7627b71220b114109.jpg",\n      "sha256": "2a5869560f3ae87a8879837df01ab3098596471856035eb081cba8c4b0e681a5",\n      "decoded_sha256": "8d39438a169e38a2da746474aaf7d6b391d4e0b7618282e947f6a5f93d6f39ca",\n      "dhash": "0c19397178393152",\n      "bytes": 24325,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2659_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "test",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_30fe8a693294be5c_0",\n          "bbox_xyxy": [\n            186.0,\n            57.24999999999998,\n            524.0,\n            588.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/test/images/IMG_2659_JPG.rf.f8b9ee8cf3f828a7627b71220b114109.jpg",\n        "size": 24325,\n        "compressed": 23550,\n        "offset": 319504107,\n        "crc32": "17c92a63",\n        "method": 8\n      }\n    },\n    {\n      "image_id": "rice_318827171d21944e",\n      "relative_path": "images/IMG_2725_JPG.rf.8b45704da1a11291bc47d816cb5daf3b.jpg",\n      "sha256": "78163f9aa22900cdefcb80d9ef0dabdfaa7c92367aac7b94015cc569d6cacb84",\n      "decoded_sha256": "e68ccbe7847134ba98a80748175786d94f93b01717a6edc7ad813a983013f1ec",\n      "dhash": "2d31b1b979b939b9",\n      "bytes": 32959,\n      "width": 640,\n      "height": 640,\n      "capture_group_id": "IMG_2725_JPG",\n      "grouping_basis": "source_filename_family_not_verified_capture",\n      "split": "test",\n      "source_record": "https://doi.org/10.5281/zenodo.20066074",\n      "licence": "CC-BY-4.0",\n      "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n      "annotation_status": "published_unverified",\n      "objects": [\n        {\n          "object_id": "rice_318827171d21944e_0",\n          "bbox_xyxy": [\n            189.0,\n            30.25000000000002,\n            504.0,\n            638.75\n          ],\n          "species": "rice_black_bug",\n          "original_label": "RBB_FEMALE",\n          "ignore": false\n        }\n      ],\n      "zip_member": {\n        "name": "datasets/02_detection_dataset/test/images/IMG_2725_JPG.rf.8b45704da1a11291bc47d816cb5daf3b.jpg",\n        "size": 32959,\n        "compressed": 32433,\n        "offset": 319711944,\n        "crc32": "dc5ea665",\n        "method": 8\n      }\n    }\n  ],\n  "crops": [],\n  "selection_salt": "dimer-ph-rice-pests-v1",\n  "source_split_preserved": true,\n  "exclusions": [\n    {\n      "family": "6095710224539913189_jpg",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "6095710224539913190_jpg",\n      "reason": "no_valid_box_member",\n      "images": 1\n    },\n    {\n      "family": "6095710224539913191_jpg",\n      "reason": "no_valid_box_member",\n      "images": 1\n    },\n    {\n      "family": "6095710224539913192_jpg",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "6095710224539913193_jpg",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "6095710224539913194_jpg",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "6095710224539913215_jpg",\n      "reason": "no_valid_box_member",\n      "images": 1\n    },\n    {\n      "family": "6095710224539913216_jpg",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "6095710224539913217_jpg",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "6095710224539913218_jpg",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "6095710224539913286_jpg",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "6095710224539913299_jpg",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "6095710224539913487_jpg",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "6095710224539913490_jpg",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "6095710224539913493_jpg",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "6095710224539913494_jpg",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "6095710224539913495_jpg",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "6095710224539913496_jpg",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "6095710224539913497_jpg",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "6095710224539913499_jpg",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "6095710224539913500_jpg",\n      "reason": "no_valid_box_member",\n      "images": 1\n    },\n    {\n      "family": "6095710224539913502_jpg",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "6095710224539913509_jpg",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "6095710224539913510_jpg",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "6095710224539913511_jpg",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "6095710224539913512_jpg",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "6095710224539913514_jpg",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "6095710224539913517_jpg",\n      "reason": "no_valid_box_member",\n      "images": 1\n    },\n    {\n      "family": "6095710224539913518_jpg",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "6095710224539913519_jpg",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "6095710224539913520_jpg",\n      "reason": "no_valid_box_member",\n      "images": 1\n    },\n    {\n      "family": "6095710224539913521_jpg",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "6095710224539913523_jpg",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "6095710224539913524_jpg",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "6095710224539913554_jpg",\n      "reason": "no_valid_box_member",\n      "images": 1\n    },\n    {\n      "family": "6095710224539913558_jpg",\n      "reason": "no_valid_box_member",\n      "images": 1\n    },\n    {\n      "family": "6095710224539913559_jpg",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "6095710224539913560_jpg",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "6095710224539913561_jpg",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "6095710224539913563_jpg",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "6095710224539913564_jpg",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "6095710224539913565_jpg",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "6095710224539913566_jpg",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "6095710224539913567_jpg",\n      "reason": "no_valid_box_member",\n      "images": 1\n    },\n    {\n      "family": "6095710224539913568_jpg",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "6095710224539913569_jpg",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "6095710224539913571_jpg",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "6100321344378571883_jpg",\n      "reason": "no_valid_box_member",\n      "images": 1\n    },\n    {\n      "family": "6100321344378571884_jpg",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "6179275676641332334_jpg",\n      "reason": "family_contains_more_than_100_targets",\n      "images": 5\n    },\n    {\n      "family": "IMG_0730_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_0778_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_0779_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_0780_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_0781_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_0782_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_0783_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_0784_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_0785_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_0786_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_0787_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_0793_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_0800_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_0820_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_0825_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_0828_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_0829_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_0831_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_0832_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_0836_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_0838_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_0840_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_0841_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_0843_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_0845_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_0850_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_0860_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_0861_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_0863_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_0864_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_0865_JPG",\n      "reason": "no_valid_box_member",\n      "images": 1\n    },\n    {\n      "family": "IMG_0867_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_0869_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_0872_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_0874_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_0876_JPG",\n      "reason": "no_valid_box_member",\n      "images": 1\n    },\n    {\n      "family": "IMG_0878_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_0879_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_0881_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_0883_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_0891_JPG",\n      "reason": "no_valid_box_member",\n      "images": 1\n    },\n    {\n      "family": "IMG_1028_JPG",\n      "reason": "no_valid_box_member",\n      "images": 1\n    },\n    {\n      "family": "IMG_1034_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_1035_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_1037_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_1043_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_1044_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_1045_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_1047_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_1053_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_1056_JPG",\n      "reason": "no_valid_box_member",\n      "images": 1\n    },\n    {\n      "family": "IMG_1062_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_1063_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_1074_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_1075_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_1080_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_1082_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_1083_JPG",\n      "reason": "no_valid_box_member",\n      "images": 1\n    },\n    {\n      "family": "IMG_1086_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_1096_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_1097_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_1107_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_1108_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_1109_JPG",\n      "reason": "no_valid_box_member",\n      "images": 10\n    },\n    {\n      "family": "IMG_1110_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_1128_JPG",\n      "reason": "no_valid_box_member",\n      "images": 1\n    },\n    {\n      "family": "IMG_1131_JPG",\n      "reason": "source_family_cross_split",\n      "images": 6\n    },\n    {\n      "family": "IMG_1137_JPG",\n      "reason": "no_valid_box_member",\n      "images": 1\n    },\n    {\n      "family": "IMG_1142_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_1151_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_1155_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_1176_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_1177_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_1179_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_1189_JPG",\n      "reason": "no_valid_box_member",\n      "images": 1\n    },\n    {\n      "family": "IMG_1197_JPG",\n      "reason": "no_valid_box_member",\n      "images": 1\n    },\n    {\n      "family": "IMG_1201_JPG",\n      "reason": "no_valid_box_member",\n      "images": 1\n    },\n    {\n      "family": "IMG_1203_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_1221_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_1223_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_1226_JPG",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_20260110_144752_jpg",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_20260110_145616_jpg",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_20260110_150009_jpg",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_20260110_150630_jpg",\n      "reason": "no_valid_box_member",\n      "images": 5\n    },\n    {\n      "family": "IMG_2681_JPG",\n      "reason": "conservative_dhash_or_exact_duplicate",\n      "against": "IMG_2535_JPG"\n    },\n    {\n      "family": "IMG_2290_JPG",\n      "reason": "conservative_dhash_or_exact_duplicate",\n      "against": "IMG_2414_JPG"\n    },\n    {\n      "family": "IMG_2306_JPG",\n      "reason": "conservative_dhash_or_exact_duplicate",\n      "against": "IMG_2414_JPG"\n    },\n    {\n      "family": "IMG_2041_JPG",\n      "reason": "conservative_dhash_or_exact_duplicate",\n      "against": "IMG_2105_JPG"\n    },\n    {\n      "family": "IMG_2114_JPG",\n      "reason": "conservative_dhash_or_exact_duplicate",\n      "against": "IMG_2116_JPG"\n    },\n    {\n      "family": "IMG_2483_JPG",\n      "reason": "conservative_dhash_or_exact_duplicate",\n      "against": "IMG_2552_JPG"\n    }\n  ],\n  "limitations": [\n    "Source filename families are not verified independent capture events.",\n    "Published boxes and species-sex labels are not verified human-complete references.",\n    "Source images are processed Roboflow exports, not untouched camera originals.",\n    "Capture location is absent; Philippine affiliation does not establish every capture origin."\n  ],\n  "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0.",\n  "license_url": "https://creativecommons.org/licenses/by/4.0/",\n  "source_metadata": {\n    "doi": "10.5281/zenodo.20066074",\n    "version": "1",\n    "licence": "cc-by-4.0",\n    "capture_origin_verified": false,\n    "human_annotation_completeness_verified": false\n  }\n}\n', 'dataset_audit.json': '{\n  "scope": "exploratory_published_annotations",\n  "qualification": {\n    "passed": true,\n    "scope": "exploratory_published_annotations",\n    "capture_independence_verified": false,\n    "human_reference_completeness_verified": false,\n    "field_surveillance_qualified": false,\n    "meaning": "Passed the user-approved exploratory published-annotation benchmark data checks only."\n  },\n  "manifest_sha256": "272ff5523de026d4e154d3e008ed39a046e434bb6c9e2974c2c222c71f8bf7a1",\n  "gates": {\n    "rights": {\n      "status": "passed",\n      "evidence": "Zenodo record API license cc-by-4.0; user screenshot confirms CC BY 4.0."\n    },\n    "original_images": {\n      "status": "passed",\n      "evidence": "Selected whole detection images downloaded and decoded; these are 640x640 processed Roboflow exports, not original camera files."\n    },\n    "crop_lineage": {\n      "status": "passed",\n      "evidence": "Canonical crops derive from manifest boxes with unique image/object IDs; all 91577 distributed ROI filenames map to source detection image names, with zero split mismatches. Canonical run does not use distributed ROI pixels."\n    },\n    "capacity": {\n      "status": "passed",\n      "images": {\n        "train": 120,\n        "validation": 40,\n        "test": 40\n      },\n      "instances": {\n        "train": {\n          "rice_black_bug": 713,\n          "white_stemborer": 991\n        },\n        "validation": {\n          "rice_black_bug": 186,\n          "white_stemborer": 164\n        },\n        "test": {\n          "rice_black_bug": 271,\n          "white_stemborer": 184\n        }\n      },\n      "groups": {\n        "train": 120,\n        "validation": 40,\n        "test": 40\n      },\n      "group_kind": "source-filename families only"\n    },\n    "philippine_capture_provenance": {\n      "status": "unresolved",\n      "evidence": "PhilRice Midsayap creator affiliations; all7098 location values absent."\n    },\n    "capture_independence": {\n      "status": "unresolved",\n      "evidence": "All7098 capture_date,capture_time,light_trap_id,camera_device,operator,notes fields absent; filename grouping cannot establish specimen independence."\n    },\n    "reference_completeness": {\n      "status": "unresolved",\n      "evidence": "Published YOLO box table; no human verification/completeness documentation."\n    }\n  },\n  "source_inventory": {\n    "archive_entries": 112940,\n    "archive_files": 112898,\n    "detection_images": 7098,\n    "detection_label_files": 7097,\n    "raw_folder_images": 7068,\n    "raw_manifest_rows": 7098,\n    "roi_images": 91577,\n    "annotation_rows": 91577,\n    "source_filename_families": 1685,\n    "cross_role_source_families": 1,\n    "images_above_100_targets": 4,\n    "maximum_targets": 106\n  },\n  "selection": {\n    "salt": "dimer-ph-rice-pests-v1",\n    "source_roles_preserved": true,\n    "variant_rule": "Within each source stem retain minimum SHA256(salt + colon + filename); rank families by SHA256(salt + colon + stem).",\n    "sample_rule": "Role quotas120/40/40; conservative exclusion of exact hashes, decoded hashes or dHash Hamming distance <=4 to any previously selected image; refill using next deterministic candidate.",\n    "exclusions": [\n      {\n        "family": "6095710224539913189_jpg",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "6095710224539913190_jpg",\n        "reason": "no_valid_box_member",\n        "images": 1\n      },\n      {\n        "family": "6095710224539913191_jpg",\n        "reason": "no_valid_box_member",\n        "images": 1\n      },\n      {\n        "family": "6095710224539913192_jpg",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "6095710224539913193_jpg",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "6095710224539913194_jpg",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "6095710224539913215_jpg",\n        "reason": "no_valid_box_member",\n        "images": 1\n      },\n      {\n        "family": "6095710224539913216_jpg",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "6095710224539913217_jpg",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "6095710224539913218_jpg",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "6095710224539913286_jpg",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "6095710224539913299_jpg",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "6095710224539913487_jpg",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "6095710224539913490_jpg",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "6095710224539913493_jpg",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "6095710224539913494_jpg",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "6095710224539913495_jpg",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "6095710224539913496_jpg",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "6095710224539913497_jpg",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "6095710224539913499_jpg",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "6095710224539913500_jpg",\n        "reason": "no_valid_box_member",\n        "images": 1\n      },\n      {\n        "family": "6095710224539913502_jpg",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "6095710224539913509_jpg",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "6095710224539913510_jpg",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "6095710224539913511_jpg",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "6095710224539913512_jpg",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "6095710224539913514_jpg",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "6095710224539913517_jpg",\n        "reason": "no_valid_box_member",\n        "images": 1\n      },\n      {\n        "family": "6095710224539913518_jpg",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "6095710224539913519_jpg",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "6095710224539913520_jpg",\n        "reason": "no_valid_box_member",\n        "images": 1\n      },\n      {\n        "family": "6095710224539913521_jpg",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "6095710224539913523_jpg",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "6095710224539913524_jpg",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "6095710224539913554_jpg",\n        "reason": "no_valid_box_member",\n        "images": 1\n      },\n      {\n        "family": "6095710224539913558_jpg",\n        "reason": "no_valid_box_member",\n        "images": 1\n      },\n      {\n        "family": "6095710224539913559_jpg",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "6095710224539913560_jpg",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "6095710224539913561_jpg",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "6095710224539913563_jpg",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "6095710224539913564_jpg",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "6095710224539913565_jpg",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "6095710224539913566_jpg",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "6095710224539913567_jpg",\n        "reason": "no_valid_box_member",\n        "images": 1\n      },\n      {\n        "family": "6095710224539913568_jpg",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "6095710224539913569_jpg",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "6095710224539913571_jpg",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "6100321344378571883_jpg",\n        "reason": "no_valid_box_member",\n        "images": 1\n      },\n      {\n        "family": "6100321344378571884_jpg",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "6179275676641332334_jpg",\n        "reason": "family_contains_more_than_100_targets",\n        "images": 5\n      },\n      {\n        "family": "IMG_0730_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_0778_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_0779_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_0780_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_0781_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_0782_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_0783_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_0784_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_0785_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_0786_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_0787_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_0793_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_0800_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_0820_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_0825_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_0828_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_0829_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_0831_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_0832_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_0836_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_0838_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_0840_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_0841_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_0843_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_0845_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_0850_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_0860_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_0861_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_0863_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_0864_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_0865_JPG",\n        "reason": "no_valid_box_member",\n        "images": 1\n      },\n      {\n        "family": "IMG_0867_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_0869_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_0872_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_0874_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_0876_JPG",\n        "reason": "no_valid_box_member",\n        "images": 1\n      },\n      {\n        "family": "IMG_0878_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_0879_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_0881_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_0883_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_0891_JPG",\n        "reason": "no_valid_box_member",\n        "images": 1\n      },\n      {\n        "family": "IMG_1028_JPG",\n        "reason": "no_valid_box_member",\n        "images": 1\n      },\n      {\n        "family": "IMG_1034_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_1035_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_1037_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_1043_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_1044_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_1045_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_1047_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_1053_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_1056_JPG",\n        "reason": "no_valid_box_member",\n        "images": 1\n      },\n      {\n        "family": "IMG_1062_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_1063_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_1074_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_1075_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_1080_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_1082_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_1083_JPG",\n        "reason": "no_valid_box_member",\n        "images": 1\n      },\n      {\n        "family": "IMG_1086_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_1096_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_1097_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_1107_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_1108_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_1109_JPG",\n        "reason": "no_valid_box_member",\n        "images": 10\n      },\n      {\n        "family": "IMG_1110_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_1128_JPG",\n        "reason": "no_valid_box_member",\n        "images": 1\n      },\n      {\n        "family": "IMG_1131_JPG",\n        "reason": "source_family_cross_split",\n        "images": 6\n      },\n      {\n        "family": "IMG_1137_JPG",\n        "reason": "no_valid_box_member",\n        "images": 1\n      },\n      {\n        "family": "IMG_1142_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_1151_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_1155_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_1176_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_1177_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_1179_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_1189_JPG",\n        "reason": "no_valid_box_member",\n        "images": 1\n      },\n      {\n        "family": "IMG_1197_JPG",\n        "reason": "no_valid_box_member",\n        "images": 1\n      },\n      {\n        "family": "IMG_1201_JPG",\n        "reason": "no_valid_box_member",\n        "images": 1\n      },\n      {\n        "family": "IMG_1203_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_1221_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_1223_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_1226_JPG",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_20260110_144752_jpg",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_20260110_145616_jpg",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_20260110_150009_jpg",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_20260110_150630_jpg",\n        "reason": "no_valid_box_member",\n        "images": 5\n      },\n      {\n        "family": "IMG_2681_JPG",\n        "reason": "conservative_dhash_or_exact_duplicate",\n        "against": "IMG_2535_JPG"\n      },\n      {\n        "family": "IMG_2290_JPG",\n        "reason": "conservative_dhash_or_exact_duplicate",\n        "against": "IMG_2414_JPG"\n      },\n      {\n        "family": "IMG_2306_JPG",\n        "reason": "conservative_dhash_or_exact_duplicate",\n        "against": "IMG_2414_JPG"\n      },\n      {\n        "family": "IMG_2041_JPG",\n        "reason": "conservative_dhash_or_exact_duplicate",\n        "against": "IMG_2105_JPG"\n      },\n      {\n        "family": "IMG_2114_JPG",\n        "reason": "conservative_dhash_or_exact_duplicate",\n        "against": "IMG_2116_JPG"\n      },\n      {\n        "family": "IMG_2483_JPG",\n        "reason": "conservative_dhash_or_exact_duplicate",\n        "against": "IMG_2552_JPG"\n      }\n    ],\n    "discarded_source_crops": true,\n    "missing_label_policy": "Canonical references use published bounding_box_annotations.csv; missing label file is never interpreted as zero targets.",\n    "target_free_images": 0\n  },\n  "source_metadata_files": {\n    "acquisition_metadata.csv": {\n      "sha256": "968072410d1c4608fcad3aa388ce7e4a4f5bff08e762b4c4a495b20f71298102",\n      "bytes": 1715179\n    },\n    "bounding_box_annotations.csv": {\n      "sha256": "87e78d239cd5eb59097113cf409cb6d2efac9081e9d9f419eec510ab93ada084",\n      "bytes": 34666466\n    },\n    "class_dictionary.csv": {\n      "sha256": "e1a239d7476fd963e17ed2a209e29db0b542ccf7529a7da33d64058a3b216b05",\n      "bytes": 760\n    },\n    "image_quality_flags.csv": {\n      "sha256": "169ef17d9e076f65a9dc043f0bb0d211f55016c4ac77643d958986c338e4b175",\n      "bytes": 17804691\n    },\n    "split_manifest.csv": {\n      "sha256": "a61eb3502131e1f8a609c21b1e784161a21575943e03696a8edc891e91633159",\n      "bytes": 1218249\n    }\n  },\n  "integrity": {\n    "central_directory_sha256": "6047989ee5e1b4fe244b2065c3cea1b2093d10d70d0066711f0e77db94c08804",\n    "full_archive_downloaded": false,\n    "archive_md5_publisher": "24960c245a3a21b63be95314cc7467da",\n    "member_crc_and_size_verified": true,\n    "selected_member_sha256_pinned": true\n  },\n  "limitations": [\n    "Source filename families are not verified independent capture events.",\n    "Published boxes and species-sex labels are not verified human-complete references.",\n    "Source images are processed Roboflow exports, not untouched camera originals.",\n    "Capture location is absent; Philippine affiliation does not establish every capture origin.",\n    "Single-insect and multi-insect photographs are mixed. Dorsal/ventral views may represent the same specimen across filename families.",\n    "dHash is an approximate duplicate screen and cannot prove biological or acquisition independence.",\n    "Source resize/contrast/augmentation cannot be undone; no claim to unprocessed camera imagery.",\n    "All performance is agreement with published annotations, not independently verified pest count accuracy."\n  ]\n}\n', 'mirror.json': '{\n  "format_version": 1,\n  "url": "https://github.com/kurtvalcorza/detr-detection-pipeline/releases/download/rice-capstone-sample-v1/rice_capstone_sample_v1.zip",\n  "bytes": 8502987,\n  "sha256": "48b44850c62ff07cce97154180965b9c57855cd5bb58f95b41da3bbb4e513e5d",\n  "members": 200,\n  "layout": "ZIP_STORED; one member per manifest record at its relative_path; sorted names; timestamps 1980-01-01; no directory entries or extra fields",\n  "source": "zenodo 10.5281/zenodo.20066074",\n  "source_archive": {\n    "bytes": 1005857218,\n    "md5": "24960c245a3a21b63be95314cc7467da",\n    "sha256": "f4a0409977e0615e34f3cf2b936a5ca9dd3a91b153d239e314334ea40c2537f7"\n  },\n  "integrity": "Every member is byte-identical to the Zenodo ZIP member named in data_manifest.json and is re-verified against that manifest (bytes, SHA-256, format, dimensions). Zenodo remains the source of record and the fallback.",\n  "licence": "CC-BY-4.0",\n  "attribution": "Deleña, Reymark; Balleras, Gina; Dequiña, Brent Jason; Jumawan, Rexter; Flores, Christine. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species–sex recognition of rice black bug and white stemborer. Version 1. Zenodo, DOI 10.5281/zenodo.20066074. CC BY 4.0."\n}\n', 'model_manifest.json': '{\n  "detr": {\n    "format": "dimer_hf_snapshot",\n    "formatVersion": 1,\n    "modelKey": "detr-resnet-50",\n    "modelId": "facebook/detr-resnet-50",\n    "revision": "1d5f47bd3bdd2c4bbfa585418ffe6da5028b4c0b",\n    "files": [\n      {\n        "path": "README.md",\n        "bytes": 5891,\n        "sha256": "bc7d74d96a9101113c06be3a90975510dd19671382eb6fdfc5d1edbb042a723b"\n      },\n      {\n        "path": "config.json",\n        "bytes": 4592,\n        "sha256": "e7bcf3992363f27717a863f14b193140ad2e41d4338ee012730e58a92cae17e6"\n      },\n      {\n        "path": "model.safetensors",\n        "bytes": 166587896,\n        "sha256": "830f5e2eeaada8c8c8281779dcc8ab12833972eb8514ed0a35be6c1d4420ad81"\n      },\n      {\n        "path": "preprocessor_config.json",\n        "bytes": 290,\n        "sha256": "0673fea2a6d3cf92cdbab3c7426c0ecdf8a4729a2a4d5199033dcd66a2b8759b"\n      }\n    ],\n    "totalBytes": 166598669\n  },\n  "bioclip": {\n    "format": "dimer_hf_snapshot",\n    "formatVersion": 1,\n    "modelKey": "bioclip-2",\n    "modelId": "imageomics/bioclip-2",\n    "revision": "2957b322090f9cb17ae72c71981c7218a28d81e0",\n    "files": [\n      {\n        "path": "README.md",\n        "bytes": 20723,\n        "sha256": "807e4daa2b3486bc542a7c796c84a117be02818e671c934a66cd1898c3c098c1"\n      },\n      {\n        "path": "open_clip_config.json",\n        "bytes": 534,\n        "sha256": "1bf947e96e943fe50efd5c3e26c37f843a2fa3c358967719a68c8a6d17ce68c8"\n      },\n      {\n        "path": "open_clip_model.safetensors",\n        "bytes": 1710517724,\n        "sha256": "b7b2bf6fbc95799e42630e394cf95803892ab447c1a8ab629dbc82fbeaf7dfef"\n      }\n    ],\n    "totalBytes": 1710538981\n  },\n  "siglip": {\n    "format": "dimer_hf_snapshot",\n    "formatVersion": 1,\n    "modelKey": "siglip2-base-patch16-224",\n    "modelId": "google/siglip2-base-patch16-224",\n    "revision": "5ffaac51d5e2f3367f7dab0cad4be4cb07c0caa2",\n    "files": [\n      {\n        "path": "README.md",\n        "bytes": 3367,\n        "sha256": "77fd3e4cc34abaa0cf891f03fc73be58cd0e36ce09741cb0fe096342e4d8867f"\n      },\n      {\n        "path": "config.json",\n        "bytes": 253,\n        "sha256": "fe8b5fe6d5734360678fd71c11c21e1ea3364bd8598d34295d9206335973ffd7"\n      },\n      {\n        "path": "model.safetensors",\n        "bytes": 1500800904,\n        "sha256": "612923381c76ec5a9bed335d1c48827e3f2e506ac31b044b63b2031fadee6a0b"\n      },\n      {\n        "path": "preprocessor_config.json",\n        "bytes": 394,\n        "sha256": "9b36b57ebaf20f09bf4c22100ccc21877ea6bfe5aead0c00c59f8af8ccefacfc"\n      },\n      {\n        "path": "special_tokens_map.json",\n        "bytes": 636,\n        "sha256": "baec30ea10906f16adb8c18af7a34023002c1746542612b8b41c9f09e1351351"\n      },\n      {\n        "path": "tokenizer.json",\n        "bytes": 34363039,\n        "sha256": "cb9140fae3ac5122c972d37adf83e1248471a38147ad76f8215c8872c6fd8322"\n      },\n      {\n        "path": "tokenizer.model",\n        "bytes": 4241003,\n        "sha256": "61a7b147390c64585d6c3543dd6fc636906c9af3865a5548f27f31aee1d4c8e2"\n      },\n      {\n        "path": "tokenizer_config.json",\n        "bytes": 47164,\n        "sha256": "14afe629fe4959b9e0d51e1852b8d9f7ad074f90a1a7125a4fcdd17f06e78fc8"\n      }\n    ],\n    "totalBytes": 1539456760\n  }\n}\n', 'requirements.txt': '# This file was autogenerated by uv via the following command:\n#    uv pip compile tools/rice-requirements.in --python-version 3.12 --python-platform x86_64-manylinux_2_28 --generate-hashes --only-binary :all: --output-file tools/rice-requirements.lock\ncertifi==2026.7.22 \\\n    --hash=sha256:62f22742b58a1a33014a2b6b706588a8d7e2a88ae7bd1a6ebe8c992928483775 \\\n    --hash=sha256:741e2c3b351ddf169a738da9f2c048608ff7f2c5cc02f1ebc6b118bb090d5d55\n    # via requests\ncharset-normalizer==3.5.1 \\\n    --hash=sha256:00668ebb0609751758682eb0b5857e7c35b9f00e84dfdef062e103244ec94d45 \\\n    --hash=sha256:012a22b88a77ca2e59b98ac5889b0deb604147666032f45e6d6e217634d2550d \\\n    --hash=sha256:01e93745f7f219b703b60ba7afead36cfc4242782be5af484673fc500df12da5 \\\n    --hash=sha256:04368edf83514385ffc3e1cfd4546e595f4f1272dd23ba437a93a9cc3741d47b \\\n    --hash=sha256:0722590aabf9dc6a6c0343d523c05458fa2b5047dbe6302fd526bb570600753f \\\n    --hash=sha256:07ffd07412fc5d5e84cd8952acf9ff7e4ed7a708e69d1bada19d8ba91711353f \\\n    --hash=sha256:09a7bba9f739468c8e78c36a75c33768e53cb1959fc638f510454c14683f00d5 \\\n    --hash=sha256:0b2b1b3fa5670c127b246df1d0c059defd41f689a868a3b9d79df9b1cac42d22 \\\n    --hash=sha256:0c6dfb5ca6723eeed15aa8e564a014d69fcb8812f94eef11fe3631e0508199f5 \\\n    --hash=sha256:0d929fc574b4d6fd9e7c0f5c2ede8716a41911923aa7fa5fce38e0818aa4a1ac \\\n    --hash=sha256:13e3afe97712e8887cd516e960c63f0b93122971e5b5e4b2622fe7701771e838 \\\n    --hash=sha256:15f024313246a4ed976c60f440bb8d257815513a681d212ff74fd46f7d715a90 \\\n    --hash=sha256:195ce897c6153c0700078142cf8efe3e6454ca4cf4357499e4078dfd83396626 \\\n    --hash=sha256:19a3dd5aa73cef1c99687c4fc57db016a9c17104ae1185da88ba566a5d3bebe4 \\\n    --hash=sha256:1d1c7a53a6c2103925cdd6d7229f8c567379f211c869793df679f2e9f738c369 \\\n    --hash=sha256:1f5883d77fd409a261abb5dc8ccbe335720d798b1de4abb3b1d47ccbbc76b53b \\\n    --hash=sha256:21b82d8082f6f5e7f456ef0bd16323d08de1266efbfeb476e64b2a91d1471a4e \\\n    --hash=sha256:252d099029bcbea642f2a06c4ed5046bdf8b5a8150b64afa5e027e88b106e5ee \\\n    --hash=sha256:256dd4d85d9e4dc595e2bc983c980e73f62ddeb3165c58b4c3dfe78c5c8548c1 \\\n    --hash=sha256:26422d45fd13551cf564c58932f7d72b4f58b93b0fcf18c35ba6be12b46bb102 \\\n    --hash=sha256:2679de311c7946dde5d3b6f44941844133ff5c7cb86099c0061ab1e8901c20a8 \\\n    --hash=sha256:29880d17a8eb0b5cfdfd8944b468322928059aa35f1f5fa8ff22b149ec0b42f8 \\\n    --hash=sha256:2bced4061f000f7187254a02ad3433ae17eaf991747ceea2f478422590a5bba9 \\\n    --hash=sha256:2e9cf9253119d8e5d111f05d71626786fd3d6193817316eab1ca088cdb8593cf \\\n    --hash=sha256:2f06b7eae9dbe77fe1d644ca244dad508de8d302870a43f3c559b521270938a0 \\\n    --hash=sha256:2f293479cce755c75f1697e87c409b7ae4c555c7dfecb6e988ad13abba943031 \\\n    --hash=sha256:329fc3ccb63ad22d867d84c2adea759a64079a37ba4a343433b02c7a2816871e \\\n    --hash=sha256:343fb4f2821043bd87095f7b08a1a181febc8e36ac64212143bbfd0a0e1bc235 \\\n    --hash=sha256:3588e376b3ea2eea84976f67273d679f229e24c66dce7b82ae45aef04ff6e072 \\\n    --hash=sha256:35aea775dc2bd5f54cd84a1cd2696cc3207c479cb9cf0bd346f0d343e4300ddb \\\n    --hash=sha256:35fe081843b35aad20ffeccec3eeffbe637b15d14f3fb22cc1b59cd8ec17e93c \\\n    --hash=sha256:36047af20e17097c3bb9476c2b7655f2f7aa51322c0ba58c07695bedf755a950 \\\n    --hash=sha256:3617ac3cfd8b9888f145ad89dd6e692285834b0201c6074a5eeaad3fd4d668c2 \\\n    --hash=sha256:366ec70f5547c640d3ce1985722490f23faf4eb5216a7eeba78277490e78dacb \\\n    --hash=sha256:394fea06235c8543390050ed5f529187074b029fb027213f6c46ac11ab5d950e \\\n    --hash=sha256:3d27167433c0d5f18dc850f07d0b3816221984fecdc405d6c157a6f0b8f8e9e6 \\\n    --hash=sha256:3e5e1224c0a6a90e05843e07adfec669edebec17801c67072f51e59561d63c0b \\\n    --hash=sha256:41876ee62a3dddf48ff1121ad8f0798032aa03f2fd35f21f34a4cab14f18d8d2 \\\n    --hash=sha256:433c5a81eade63b47e522303bad236f59dba55ea6951746f5558355eeed8c75d \\\n    --hash=sha256:4582c27e8c889d64811987b5967fbd3ae0c823fe1fd933b543d55ac20bb475fa \\\n    --hash=sha256:485a0d363cafefcd2538a73c7c838daa2035f09b2c9f9b5e3133f80c6aeb84c2 \\\n    --hash=sha256:494b70049a4d69aec6e8137c13af4cf8db8c9f9820a1392ac293b0dd2987a818 \\\n    --hash=sha256:496846868fea80e479324862fa877f02411f2fd0f83b79ccee2607aa68b2a032 \\\n    --hash=sha256:4abdc5f9ad448c1ecbfae2974b820535d6bc6e7eef63babbab3d81cf46968c71 \\\n    --hash=sha256:4b599739b93b2cbeded49645ae3c8d1405c29ddfbceac1545c87a3f9580a9e96 \\\n    --hash=sha256:4bea7f8ebe90bbd7f0e4a2de42ca6924ba23e3e76418c408ff82f1d46fabd687 \\\n    --hash=sha256:4c4fb141a727957c93edfe5c32a26ceb6b5f6461d67146e2d39f51e16170bea8 \\\n    --hash=sha256:4c9548dc78002099910abaebc0a72ac58b7d30931869e0351c09b507dff4ece3 \\\n    --hash=sha256:4d26f14f041e83dd8edfd61f4cd4fa7285d31798b5bf1f28e70c367ba6c41d61 \\\n    --hash=sha256:4f298bdadb8f0b9e5672877f647d1be9373ef5320c9e2f049795e26cad28b6a9 \\\n    --hash=sha256:52ec005752a56ae79547a05c0139ca2501a0c866390b6115008456b9f0e7cde1 \\\n    --hash=sha256:55261ac0d2941c42f196dd576f543d87a8ee03cd6f5e30dfb4d807b2e3b9121a \\\n    --hash=sha256:56490c595a28b1bb27dfc583e816152a9767721ef58b2c03b13f954d2f707420 \\\n    --hash=sha256:58d3e12c88e0950bca850ae1f7c256055c097639c2edb9eb123af9807d8b15e4 \\\n    --hash=sha256:58d4aa13a59c969dbfdf9e6a9560e242cbfd9e8a8f50c2747714df1a423adf65 \\\n    --hash=sha256:59171c6e45bf07d0d5cab3b0bf81d945035530f6873398b3b531c31184d46663 \\\n    --hash=sha256:5b6d1386bf0096d26d3a863dc0a487a5b4eb9aa93cf5ba69683d29dde6b9d60f \\\n    --hash=sha256:5c0ea61a470e070686aa30892fed79e297d2c8d0ab46b8bcdf027d38c51da591 \\\n    --hash=sha256:5c84bec0ab5ae0c64bfe73a7d2adcb5ce73b467523fc27fd6a28ab2aa6cbe35a \\\n    --hash=sha256:5ca0555312ae2fe82715cada7fac375530c2f3349e1eaa1bcb33d0283ac79a18 \\\n    --hash=sha256:5d8531a6569d025f68e2321e7638fb7978f23db58e5f69f56913837aae03816e \\\n    --hash=sha256:5e2d0e146dcb57034f8b97dc58d2d512cb90aba253960ce449f695fec6a82c6f \\\n    --hash=sha256:5fc45d653ea8c9a20479167e11d4a0f8cb2fa3470737ab6f9c827532313187b7 \\\n    --hash=sha256:6117b84ea48435e5356dc737f5121485c30920ba43375fa7b434fd753df0eac3 \\\n    --hash=sha256:6199d5606e2bbf2b096cf64d03f8b6790c91081d5ac866b8e7bb6422738cc60c \\\n    --hash=sha256:62b55f6722735a6c472f88361cde6640608773d9443cebdbb51abf436a1fcdd3 \\\n    --hash=sha256:687c9ca3035544b113bea2055e180af96fb63c0c476e22a9180f51925186e7b7 \\\n    --hash=sha256:6b7430cf5728e68f6c462254009a6ef4086e1bea43cf2f57aa9c55fb4f50ff96 \\\n    --hash=sha256:6ba32c4d2abf1d2fe7cf27d280f4cca5664233b0f885549c7761719eb977f486 \\\n    --hash=sha256:6c9cdde8becb25a7fde49924511aa2644d6f8081cc8df8e9452724303348d8e3 \\\n    --hash=sha256:6df0ec430f9a831772c23ca5a224cba36517a58a84bb32c32bb59a9fa67c47f6 \\\n    --hash=sha256:6e2912d4babbc65196ac13c2f53468dc57fb8b9c25ef913e8c59ddf7c6dc0e1b \\\n    --hash=sha256:6e5e4d73d588ca5ed09df1b7dcd1b203d1df3c542e3f50d126c947d432b10731 \\\n    --hash=sha256:70055ff39b97c99e7ae40ea3e393fb62aa2e44dbd9b29f8d14f42fb0025c3959 \\\n    --hash=sha256:706bfd38730a5ac7a365793269a00f4e988178cec121391f4248d84ad8c972e9 \\\n    --hash=sha256:7235dc28fc6dd9d832ac7c7bce95367dedb85929f17368a0c2bee1e080b9acbf \\\n    --hash=sha256:774d157f112367ff4abd29019f38f023c24e00e56edc7829c20e358a5a913ad8 \\\n    --hash=sha256:77efcff2b23071c349402ac1066667a3d011f62398d81408c9b88ad991747c9e \\\n    --hash=sha256:789b8982559ae28dad2356519f841655756cdcd96616410590ae0b17454ee64f \\\n    --hash=sha256:7ac76cf9afd34929d76eb7fcb63be476a4853d8a96f0dcf2d0db68a0cbdf9885 \\\n    --hash=sha256:7c0c10730342b0c9b35dd1d619beb8214e520bd96a1f870f452680b238aab3e0 \\\n    --hash=sha256:823f82903d189af463d7df250ef1f7f696f3cee08cc8d91deb565e8d425f6506 \\\n    --hash=sha256:838648accb3a7fd9803fd45c87bce8509648eb0c11bc34e216141300977244f2 \\\n    --hash=sha256:854066be00447fa8de2ccbbe893e2ffc4b123ef16d897af794c1e18bd4a714b0 \\\n    --hash=sha256:85d5855daafc240cc045c026d7a15fd198a09b0fc8ff6f5ecbb5297b509cb11e \\\n    --hash=sha256:85de3134b5379856e323ba37c19c9256d39425f7b76a63af52b09fb4664c2e8f \\\n    --hash=sha256:87e4f41d375c0b9be2fb5251aee4b8a689169e134535aed81bf085c3b647451e \\\n    --hash=sha256:88ca277405c2d3b71c4e1c2ee0e7966e807bcba86a69d11e19ba199d18ae4491 \\\n    --hash=sha256:88e85ab89cb822c1e635f51d6d32e488f94e002e70e2f492bdb8b945543f345a \\\n    --hash=sha256:8ac8c94b6539074e0f40899301273ac8402b9b3e01c7b7ba269ff30340aaaf20 \\\n    --hash=sha256:8fe532b3c966d1fb794e0698e4589d0444017ae77fc0b31edea13c0e35bcc449 \\\n    --hash=sha256:9085f87b0e38a2b92b8923059b4e8789fe40d9279712d15dcc670048d77079af \\\n    --hash=sha256:90b7481fb62fbe172c558bc6fd1c4c98d82004a54a7551f20e11ac9bf0b8708c \\\n    --hash=sha256:92caef967d287a407085d61176fce4012b1dd62daed4eb6d5ceb26d3d2538712 \\\n    --hash=sha256:9362dd90aa7dab48c0054a21187791ccf05473f7dba5d92b8033ae62164675e7 \\\n    --hash=sha256:94d78ecec2605a8d0398b0f365d5f12a63248438516f5dac536a5eff7337df4a \\\n    --hash=sha256:94fbf1c0c6cc0d3d5e50f9a9313a8cdca90dd696d34b381cd1704f8c9e939f20 \\\n    --hash=sha256:950f23cb393f85543777b0433f082cddd25b51ab398eac7971146495679efe5f \\\n    --hash=sha256:96eefc178f8636b9c760c5829345307fd81cfae9ab1e80997dbddeb0f54ee9a3 \\\n    --hash=sha256:96fef3e886d6a9874b14f27fc193fbdc69d5d8035783d86aa4e1cea594e695f9 \\\n    --hash=sha256:977cdbd483a9cff38179bea4fd754289a6f2195c7abd414aba85410b3e66cc5e \\\n    --hash=sha256:978eab16f55b4ab2c2a745be9a0a840bf8f09a7f227d9c76eb30214d078865a5 \\\n    --hash=sha256:994e883d17c559cdfd38c84003c8b27d25424a1077272a17e7cd27bfe0bf57b2 \\\n    --hash=sha256:9ac4444d8d4fd4c4bd08bf451ed3167aa9e7ec6cdb41b648794f1d1103652e36 \\\n    --hash=sha256:9b5db6052055d34d41230fb78d7c439c23dc536a9896f6cb039e8dd92cfc1263 \\\n    --hash=sha256:9d9a0dc7cbe9bec24c3f767c9122c41fe5a1bc43f47cd099d00d393e09769de4 \\\n    --hash=sha256:9dbdd9205662134957cf0c324f639bdc5031c0ca056e2369e238db75187c0f11 \\\n    --hash=sha256:9eea3ab2597a5e65fe65296e2d6a84570845a6b55532d90333d740d48bbc850a \\\n    --hash=sha256:a2028475ba855475b8b4d3cfeb4994269c967aea8b9892dfba907f4263a863a3 \\\n    --hash=sha256:a3a370082ce34d0612f421e15fe011c53bb1feff21a26d06ad4fb244dab5a375 \\\n    --hash=sha256:a545775cfe815855ea32d7c27731d79da358ef2055b4a25830231b1622dd18aa \\\n    --hash=sha256:a5cbd90ecf0fc62e64726917ad083b73001f0563657a87ec3c0b504e277dc90d \\\n    --hash=sha256:a6d095662e73e74f0a49988e0593373e243e3a52e27bfeea0a859e88acf4a0f5 \\\n    --hash=sha256:a6dac12ff6b846103483683f60c5f8fee205121adc58ffd87e90a90a3af69e99 \\\n    --hash=sha256:a951ad59cad9145664a730d3036b40b844e74d2d3683da40111463cd3a83845d \\\n    --hash=sha256:aa1099b956fb795e686d073568f6dc002a0bb89765ea6d5b055dd7d9bf1b116c \\\n    --hash=sha256:aa2bb0b37202dca27175591f761108b5d34096ade1191ffe4808bdf6b1571488 \\\n    --hash=sha256:aae2ee51122d3ae968a3837d97dc24a0aeebb0dea23694422cd172bd30017cd6 \\\n    --hash=sha256:ab743e9bc90c1f73552ec33e10e3331315acd2c397b36065b591b0181de533cc \\\n    --hash=sha256:ac00177c4831ffa650f8609e4bdddd5fe09c03b1c0c47acece7e6ea20421598b \\\n    --hash=sha256:ac13b004224fb341e1e25a1ed5e19d32f57cdb2a403e01f003b46f051a550f6f \\\n    --hash=sha256:acaf604462bf330b0d07e7a07c1d6e4adac79e5fb13e9c5140590542cafacc00 \\\n    --hash=sha256:ae31a1a1db2ee6cc2942fccaf695c934bc7f3db9f2133a3fef1f367cf1a4ab10 \\\n    --hash=sha256:ae4a097991662cd4fff0ddc74e0fe7874f82e00042fa0ea00855645ed0c79598 \\\n    --hash=sha256:aea996a6aba25260827c9ea511d1addfde2da9eb686ac961838509086188b7e6 \\\n    --hash=sha256:b39b69b347e5e47a3b5b8cfc005c68c1ba347474e3960236c4944a8ecd174962 \\\n    --hash=sha256:b54e7e13267d49ffbfe68e25b3cbd774dab38fa37238f71265e91b36146eb21c \\\n    --hash=sha256:b9af956078716df40d985fb0dfeb2c2120c5ca92ba4ff4b388acfd01cdc14d08 \\\n    --hash=sha256:ba2f37ee79e6338845261a3c5b1784e5d1acdff2c0785b284f1b633033d136ab \\\n    --hash=sha256:ba501e667c17d8411f98e67a022d9604ef179aff0e459b7e292c796837c13573 \\\n    --hash=sha256:baf3775a2635e5a11fbd5e4e64ee69c7e86875d224a5c72aca4c141064589a90 \\\n    --hash=sha256:bb57753e36e4855b8ca375069482250a6246372331a3e4f3407eaebb007443f5 \\\n    --hash=sha256:bd6c173f04743d483881bffa1478d5a4624475b8cd1d2194956a75548e191c18 \\\n    --hash=sha256:be47f99644b208bff7766314013f9acf57b056b04191d570d68ad14022cf5b1d \\\n    --hash=sha256:c010f5581d9c612804cc59fcf7b524b707fbcb72828551237ab545bb5c7034af \\\n    --hash=sha256:c1dcc36dcb96abc02236e182d17e0f71430152a6c2c7447421da2d2dc144edea \\\n    --hash=sha256:c428c6c31eb5f4277d7f8eccaf767fbd548ddd5ce3c8b4f4cbbfab3d96b5904c \\\n    --hash=sha256:c658c50ac0c98cd755a2dd50b7977d3bca7df401dcc47fbdfa87db53ef7d4e8b \\\n    --hash=sha256:c71fb0d56c920c269cd3e2e3fe7c610e3f1fdb21a6ce60efa6430ff63676cea6 \\\n    --hash=sha256:c7b742bf31c88566b4bb6335a7f393bb322e580b6bb98df7bd0c25e6e3519ce8 \\\n    --hash=sha256:cc0329df4caaceb950d2f580b5ac716a377f7059624a0bafaeaf8a218c6ed774 \\\n    --hash=sha256:cc5d36d96478aa9c60654bd932525bf32964c62a7281eafdf16d85003a8d6004 \\\n    --hash=sha256:ce854f5f478050ade5a238731c4ca985a7d3b3cb53ff600a9b5c3b689b5f0a7a \\\n    --hash=sha256:ced3fdd71aaa83ce593746c2edb42b7a59cb4c19c8b5c407781c72e493aae55a \\\n    --hash=sha256:cee5dd7c6fb5dd52a0fe2a740f9bc6e3593f5f8b1788bde49de02086f30182b2 \\\n    --hash=sha256:cfa1c0cc3a8f9f53f1243a5a99ac36fd003880199383b37672e86ddda9cb07e2 \\\n    --hash=sha256:d1ee1e296209fdce05b81b663250eefa02213a2da7b41bf26f7829b8ba3545aa \\\n    --hash=sha256:d59b75732e9b6f27388e10c14b0259cc5f2e48c78627d185e6a177b58ad3cffe \\\n    --hash=sha256:d63600d620ad0064c3a748b950ac5ea38a80190e5498532efefa4b7b3f1da1f3 \\\n    --hash=sha256:dd732602a7009217f658d5863d12d79d373a4de0eebc111094bcdd3bb8e0a6cc \\\n    --hash=sha256:e06efa066f7dbadbc84ebc126a97c452a6451dfcf589d89d788484949e1cf795 \\\n    --hash=sha256:e199fb99720074809a7720f1c0b4d919eea8b87e88713e0f8f602f7bef543d9d \\\n    --hash=sha256:e4b018dc5a0eee4676e38fe84a47a427816c590b93b55d9025274ec4d6ffc2dc \\\n    --hash=sha256:e6621fb2a4988d6e53eedc455e5903e2679f3967b8acb3d639f1b63c14a2e893 \\\n    --hash=sha256:e71c909f353863b2b89c83de2ebed71ea6d0df8a6ef65a128193c5e650766bef \\\n    --hash=sha256:e90251c0c7bdd54a100a0dce3c07b7e637278c93af29dbf78ebb89a58c4bac7d \\\n    --hash=sha256:e9fbdce1e47394b09bc9f26ab117dfc8d6491977a11d86f592bb42c779db2fda \\\n    --hash=sha256:eb12fb2ba69ffa05f8695f61c69e591dc4b4a12ac3757ac8af8adb259bf56d17 \\\n    --hash=sha256:eda059b6bc8bc0812d626fd91a7ce01bf583df0a61296eff390fd94141a34e30 \\\n    --hash=sha256:f03ac127268b43ef4fe9e6ab6794a6794b49485a0cc0c1db79876d2f33f75bc7 \\\n    --hash=sha256:f298e218441525d3794428b4c8b8fb8662c6d3ea79925d4807ee6b9a96a3bca5 \\\n    --hash=sha256:f5542f9b941279d82d41eb0aa9f98eba36fe4df5c7086c651df7944935b37182 \\\n    --hash=sha256:f6f7deae3feb4edfa2efaf7c574fe88cbf055038a6abdb40188e4fff66d5699f \\\n    --hash=sha256:f9b1e28d0e8dbfa858abdba91d6b547beaf2df1a59bec6da6faae7b96a4991a9 \\\n    --hash=sha256:f9f8405c2c758532c74fed975dbee57be1f31a6e865c031870c79a6ed3212ada \\\n    --hash=sha256:fa48b1b63d639f9483e0633e092f5851e2348c352f1f9bb6c8182f87884ef876 \\\n    --hash=sha256:fb78f6e7fcd8ad785d28cd577168bc1aaee827b25bb8755638f694794ea98f0a \\\n    --hash=sha256:fbc597639158fd7c14d55e808718848319540f51b0e6746e3eefa59723a4a348 \\\n    --hash=sha256:fce8cbd4997efeb450bd298b54f755dcdff18d496f7a5ddbb4867c6d7c88fdc3 \\\n    --hash=sha256:fd0350afdc3aabd5576f60ea109228bd5538139713c7b094c5cd27c73a98bc6f \\\n    --hash=sha256:fd0a274c0e5f9a21565cd9d3dd749b61f96b7aa1e20a93aa1ba4029518f2e5c0 \\\n    --hash=sha256:fdb8a068947befafba9952162645dc2fecaeb400e64584829ed5e9b2fbe21a7f\n    # via requests\ncloudpickle==3.1.2 \\\n    --hash=sha256:7fda9eb655c9c230dab534f1983763de5835249750e85fbcef43aaa30a9a2414 \\\n    --hash=sha256:9acb47f6afd73f60dc1df93bb801b472f05ff42fa6c84167d25cb206be1fbf4a\n    # via joblib\ncontourpy==1.4.0 \\\n    --hash=sha256:018227e134b73090b06f911e773a526c77c9af426be29065500ce096a5accd4c \\\n    --hash=sha256:072a702e2e178f4fcf96f04775d0c059e4c917925abf0d3208ebb6865df3c23d \\\n    --hash=sha256:0c7a4c2716a4e98342221954416a836cca77c996c14ddbf22b5f01d5d93ca09c \\\n    --hash=sha256:107ec46f7aa1664266d69b1181aadc953d58d39f7d8d648fa5b795d4a060b0de \\\n    --hash=sha256:181bea01bc742734ae672fa00c717d855dd35e1f029536406538c52e7b0cd73d \\\n    --hash=sha256:186ba929df36d61b6127da2e89cd1357e4cb79aca647381ab1a6feb0b152877b \\\n    --hash=sha256:196759a3e4db60e1546167e361090ebb6d199aa3aafc9e34ea17a5ac3b23e814 \\\n    --hash=sha256:20156f5a1ac4f8ce02656e39a61e82164a3d359796dc8026f75b062783d500e1 \\\n    --hash=sha256:20da2f86bfaed60dbba729fe738c8241b1dd22b6cf7e8cbbf30df290bd04ba28 \\\n    --hash=sha256:2deb580178ca19437a84bd77e4bc2cd91a8ccad212413a83c273900868b5978c \\\n    --hash=sha256:2e741a39dfc96babe1722561e81351aeec104526c57dbe91c167c1db606c2d55 \\\n    --hash=sha256:3863ef2e2b13fe93f8c0ebb08ee400cb07153b8b0e91c5acb26e4537f283634f \\\n    --hash=sha256:3e0a2392533e5e2abdf277c951047516acc8d2f3c42b2dd9dc34c907c9ebabf2 \\\n    --hash=sha256:404dbcd9233513dfd1323b66ea593fef90e899f658f6b8a9ed8e93bd0ca669db \\\n    --hash=sha256:417be048f7e122cefbe2a34c51a1f2b0410f8eb35796f794d45475ddb7872d9d \\\n    --hash=sha256:423bd5f4b3f11d54a8c597234e513382e3454bb106b8ec7ae32ba4ed63f8d99b \\\n    --hash=sha256:432e89f835cf01f8d2123a130a3126b17ee6e0348e4d62c0973872b30aa873bf \\\n    --hash=sha256:43c3ccbb32c6294b183dcc8e8c46dacf5ecef809497e3d48a5be298eef185ad0 \\\n    --hash=sha256:43d80072a32299bf945de0a6dd4e034ea0162e832261de2e07a274a5adbba9c9 \\\n    --hash=sha256:4bffcb2e8cc5c0e837631acab59dd04265fdac376550e7c3e310523877c68cb7 \\\n    --hash=sha256:510f7d93d94cf6ebf4e2cd0640bea16ab8affac7547230175b6745b596ce0599 \\\n    --hash=sha256:5450f091ac1be0be3ad3a2a3b3f23b5e443e78c670ced4fd347d626f92a28fd2 \\\n    --hash=sha256:5545ff38229c15d11d822fc6f7415b932da9f4975f53e305b7b96ebfb01f889c \\\n    --hash=sha256:5a4c89c7f38a0d7a94356d74e3391073c4325c4d47ad2fd065c3fe7dbae16c0c \\\n    --hash=sha256:61624f6722480aa7e168fd746164e7bfdf48f8ccb6542f1613741200dc37e2b1 \\\n    --hash=sha256:618137ec5778fb76d494c9ec854b104a377fddba0d128bef7136cce2c3bf0df8 \\\n    --hash=sha256:630d32f06cedf37b7f1dd2b6a12e4492826bd477a4c9c7b2e2fee6ce3a2cea76 \\\n    --hash=sha256:64039341e2d8804f1a13bda8c69083eab935167bbd7e856cf096241f17d5bd45 \\\n    --hash=sha256:6507a016976a75a15aee47809a34605de6244ccc056e6164bf9fb14b27eecad4 \\\n    --hash=sha256:6ced1670fafee703b8277ab1645072a226e74f167166eae146fb743916119b67 \\\n    --hash=sha256:6e697d94e69f499ff6bebb899cae97a58d5d14f0e1fe9568b43a0248d2f9af8c \\\n    --hash=sha256:6f963ea9f0d68d2b6a02d861e3c0aae09ae25ba21f243fcf281e9fed468b078d \\\n    --hash=sha256:738c44fa71735a617f36da58e32810512407d283d5d1bb5b1cecb00d7eeba7cf \\\n    --hash=sha256:758e7496cec3195fc28a28945eab3ceef348c47fb95c7afc2eca2f3901209c8c \\\n    --hash=sha256:78ed5c5f962b3e156109f0531b174a65ba80d13cc681a70227b12524c162e184 \\\n    --hash=sha256:79b06c60d5e569ce12c5da8f0c51217cdc484386e2349fa717c30fafb56d2871 \\\n    --hash=sha256:7f861af508fca2384eef392bc1d8377cfa349b2b854cb11b45d89c75780e2561 \\\n    --hash=sha256:80c7fc8e7ac217777ce7ba2db3cf8478652b5a4b335283111012ff9eaac9d842 \\\n    --hash=sha256:86d05cec773c9507a3950122e0e40ce77c23c75ceeb2fc189514e71893cbb34b \\\n    --hash=sha256:875f42444c9cf48d56f724f2637e60d0f73b3b12c9041e1484580a233edf9591 \\\n    --hash=sha256:90feb8da006803a95ba573fddc9536357d0c3faddc3a4c9e22816ca49af52878 \\\n    --hash=sha256:912c6afaa106e2f74ba22b30416b77ef7eb94e3bdc5a4df51ab147845dba9b6d \\\n    --hash=sha256:96019f059bcb3774acc0104be3360a57b36d33eaaa2e1d7f77c800ca8e07618b \\\n    --hash=sha256:9c0e07c691f3b3321913ed9b8161c50ea5f77fadd006f52f5e755359b0dcbfc5 \\\n    --hash=sha256:a03b32c2e7eda8b17757c022162c13c86f5c3d6f937dddf9ba3c3ff7b513953b \\\n    --hash=sha256:a1c8a74744fa746eeaa12f0f9ed7f8b4add9d8f1b14d95e3b5e133675eac888f \\\n    --hash=sha256:a3e67bc1a6a4618a1dac7c3053a9ffece5ddfa2046b2670b342cf430bb0b87d1 \\\n    --hash=sha256:a58b5f130afec61a093d743cdda435cc4047748b6791627e34a68f41ec3a9b07 \\\n    --hash=sha256:a5e2bb871b90cd7a52bdee63bf80ef7acb46398d27efc66d7679240016efc5aa \\\n    --hash=sha256:abd0f7e51ecc52bf8c95713bd80b1c17c516a71fa391c54475308a4377903c2b \\\n    --hash=sha256:b7794ea07cab575633daad8d6e963b662053391022294d3aacc1d9ba9ef54114 \\\n    --hash=sha256:b9dc493e7924e5aa32d89d0b8e50986280f3d4b284968a792787bd7bcffd3cad \\\n    --hash=sha256:be85d160e7c795113c2a93254dde2fc7e86b375322570eee78b420969426973c \\\n    --hash=sha256:bee6ff96653e0807cd0f8eaef00174a031e3a2effa5c20f49a5a8d574b05af23 \\\n    --hash=sha256:c76f3a5318164db7d9401132fc1b5364b784c613c93fa506e3b5e6d1bf353ec8 \\\n    --hash=sha256:c8be3392b84cf43373a488874dc1ebf46f60784a70977f92586e18019d71c7ed \\\n    --hash=sha256:c927ec747633e68c9920bcaad48b0bcefe648812b822f18c34e2d0e2270a3d2a \\\n    --hash=sha256:cc87f2ffa84a49a77d76cb792f28a48ff3b05a222aa0b2bddbd839ba78e1d5ca \\\n    --hash=sha256:dd59df9e2fb0aff8bd7fd1adb9f349b7c835245a9d0f18c2f2deeb537190f2b1 \\\n    --hash=sha256:ddf5a2596d716fd3793434844caf31abc7a40b1e8718431420c89858268fb909 \\\n    --hash=sha256:de503609fdb71f597634ca64fb4bd2f13d27c97f140e912b1e8ed93544840df0 \\\n    --hash=sha256:e21d1d4a9db5b3a793649c7ab0ddb4697f2818929eec871a7b9c45fce7b2a1ea \\\n    --hash=sha256:e439ab450c93455feb0218532ded3e946ec7a9b5f459069f122cfa44b89229f6 \\\n    --hash=sha256:ea09dc704029930cbd097f75cb0cc1db9852adbcf1a151fac7e9559f7bcbfd19 \\\n    --hash=sha256:ef47fef9a912c77e3d84b014f701e5a9340f25703e9ed3bcebe37f91fb69dc49 \\\n    --hash=sha256:ef9440f6f8506246269a82734f5ff9e2e4c5e775b3996fc883cc491c5257eca6 \\\n    --hash=sha256:f1219a8898523cba821085f2da8a1b962cc696325763f09d7f93538ba43b2d70 \\\n    --hash=sha256:f863c6100bf926cf47d13f3cd75f9bb8ebd98aaab230eeb4468b91f98f39a6b3 \\\n    --hash=sha256:fa1b787362a3856e63dd2b89449f6c384d55ee1beb8f94f4bcc871b40f02462c \\\n    --hash=sha256:fc9feef8f1f001c5b87decadc67c4a5d1eebb62ca39c4763d1237ff62cf2b707\n    # via matplotlib\ncuda-bindings==13.4.3 \\\n    --hash=sha256:044c03b056dcc5cecfad426a071187dd9e1e6817fbb363bc2f3a7520170b3e67 \\\n    --hash=sha256:0deff5b22462bb410859684299b1fc6a621780c43a4729342aa6ec28783485b4 \\\n    --hash=sha256:130ff1daae550db2cef559ba477f3a63d040756bd135f5deba4b5bdc4e110252 \\\n    --hash=sha256:145cc9b02dfb7bcc3288701b6f19c69c590e4451c62916ba0a8a3db9a64a91a8 \\\n    --hash=sha256:1fd7d8459b364aedc11f3e59703453ced823135f78a9111ca70feef8d56d4d21 \\\n    --hash=sha256:3d7a6506e625be59cdc119ef4423afa0fc3a4830b129dc010cbd51324b93d66d \\\n    --hash=sha256:4796864ce829bd95ef2ef0d23c6ba21bb64e08f7fab0a377302ed1affb6605c7 \\\n    --hash=sha256:52d7f3f5f7f014dddddc66cd802ed0ddf65ae99ad42b5619fae7b82e5ddd6771 \\\n    --hash=sha256:6c6bb1a4c4f7520e062669c6f3605a7016b2e65ba51b8922f92edd91d5fadf8d \\\n    --hash=sha256:7e11cfe8fec4c85ce79feda18124971c52596f0cbd642a94f5dafc257124a4b3 \\\n    --hash=sha256:81eef62bbb95cb4a705fb423b2ad1c63d716edb352ee2af0c28bf8a29cc8258c \\\n    --hash=sha256:a9ea13b9cfe711515ae83b5efc99101af4d3f8ad882f99724a839c8b5417fb7e \\\n    --hash=sha256:b89d6e738494b7b95c38e3413f86d32c24682c8e714870531a5a2b193a2fc50a \\\n    --hash=sha256:bbacde6f75665b197016b986164cfdaa33b17515e5e635a63ddb75926aaa71c3 \\\n    --hash=sha256:bc51990309b416e0780a11a921ac597ef9c0e52e1bdf5ae0b8e8c4766b66442b \\\n    --hash=sha256:bfbd3f7d4ac04dd41dc49121b9e408c8283992f47124c2290ecb79bbbadcca8e \\\n    --hash=sha256:c2af7e69d2557fdb4e5fc1169159fd08da7e861aa84fb83307a1a0396b6b0973 \\\n    --hash=sha256:d5f72bcfcdf3be23e1da3c792f68f508586f48d037bca8b10f552c4cca5971f2 \\\n    --hash=sha256:d6eb969920e28f66f8fc3b0b3afcb6e09381cc96bf8e8158d774e9488ae89980 \\\n    --hash=sha256:d7c6c9f46fca7f3fc61959ef9a2398ac656172145b43f408e0a6492360cf1c0c \\\n    --hash=sha256:df8b3767facca8acde216460df684dfc3826d519096c13adfd3030a55870dc79 \\\n    --hash=sha256:e52f66340785a51b8b77f4487329de188f3cabbf37c62e68f410a8299e8677ae \\\n    --hash=sha256:f8519603001c92bf83e7095df3b8211e3999a9c4ded096b57de0f3ff52b66368\n    # via torch\ncuda-pathfinder==1.8.2 \\\n    --hash=sha256:4e65059febdb4d19d5cbc4798677e19db2b582f2f702f457b609e571690d357e\n    # via cuda-bindings\ncuda-toolkit==13.0.2 \\\n    --hash=sha256:b198824cf2f54003f50d64ada3a0f184b42ca0846c1c94192fa269ecd97a66eb\n    # via torch\ncycler==0.12.1 \\\n    --hash=sha256:85cef7cff222d8644161529808465972e51340599459b8ac3ccbac5a854e0d30 \\\n    --hash=sha256:88bb128f02ba341da8ef447245a9e138fae777f6a23943da4540077d3601eb1c\n    # via matplotlib\nfilelock==4.0.4 \\\n    --hash=sha256:0df72be195ca7892216d16f2edce8d9b93a571f02402972020a8cff84c594c7b \\\n    --hash=sha256:90999ed63a26ccf86b93b959ab10cf1017f422d816be454ed54cbed263e71ab5\n    # via\n    #   huggingface-hub\n    #   torch\n    #   transformers\nfonttools==4.66.0 \\\n    --hash=sha256:02117d05dddb51e39b5c5a6eed0702ed8e0e1c341cb138101b2446eb479858f8 \\\n    --hash=sha256:03922992966a1830b94a750961d1ccfe1a7375d792ca61077a5afb719811788a \\\n    --hash=sha256:056641fcedeaad24b92e343e35e9fd1337200c102450678894b17928482d010d \\\n    --hash=sha256:0640e69b00e089d6b82e0a54e3a03af84b775f1f0b2cbecc1b0d7fe3980a194a \\\n    --hash=sha256:0cf33d1041364ebb7a9958db757e213c6d2a238eb41755fbf935db028c2db20f \\\n    --hash=sha256:107aae7c38f05561fef5103baa98a44400b5f4c69d87839c9a5f8125405e7d9a \\\n    --hash=sha256:135ce8738a6341fbefeedc061e20c2889350e84405abea7e61b1c9d87e1a1f2f \\\n    --hash=sha256:15315302c620e6ce8582934e07932dcf442fc3f59ab208d30564200b9a2f7ea4 \\\n    --hash=sha256:1d120ea0f5260b04e9b5ac0d9239efde4c23d990347d5d5889cdd37221726fec \\\n    --hash=sha256:221defad3b1949c1fdde2558d1fe780d42f926b683a71a46a4495a0e6a6614bd \\\n    --hash=sha256:2b278e2abe596872b3c9fe611f956ded85588537974e48e2adbe2d6ee159e099 \\\n    --hash=sha256:35684b562df7154d7a0ebfa512db9199c7fb0a93b5abfb59dac9d022dbee7aaa \\\n    --hash=sha256:379b94fe10651d26caf398eb646b3ecde886a9c504d3cce5e46164d58a78e32d \\\n    --hash=sha256:3889fdbe63e85dbf2a0d9d7c9d90746827ce60af2cdd24f0bb4738380a9540eb \\\n    --hash=sha256:398bfc5ce9055e0b91a6189f8cf462d4230b38f522b2ceaecb4f34cbf4fb5480 \\\n    --hash=sha256:3ea59aaec135c96c4cff20d79fdf1d10623993712a77990585a42cfd6a42bc5c \\\n    --hash=sha256:4513753259f06316bdc2039c9bf3129e74923c1cfad0722b4728a9ce19beeeb2 \\\n    --hash=sha256:4bdaf80590fcccb1b9d8e3b8ca72225defb14d03b79788f59781ce1bb66049b9 \\\n    --hash=sha256:5dda4087a7f8f0b985995489b3c0115bd117f18c0fc236013a0e78e837bb9ac2 \\\n    --hash=sha256:6371e7ed43cc9e23550fe4a1b72af3205275f28dcecac8c09b80a4582e326802 \\\n    --hash=sha256:6627f48130cdbe04c7e1189bff4192886ecda757f575626ee8468f6580eaecf3 \\\n    --hash=sha256:67ea5af3ca60e1e5c9b2841b1f6dba5ef16aab581bf463abb68515447154ab61 \\\n    --hash=sha256:6b11180c4166a4fc353bb02b147322b7c454927bba4f5040ccdef109a73f9051 \\\n    --hash=sha256:700c448f190cc7bbd9f059bd96131f4770e90938c9692550b142153bea73b14c \\\n    --hash=sha256:8009e736e1491c569392b2e9782c30dcb6ac879def102c8c2e06add8d2adc10f \\\n    --hash=sha256:850d50b347f31667d3a277a6e66a4219236db9d98afadb2993b8addf9bd4f593 \\\n    --hash=sha256:85e2960a29e6882567b981ba408b5a5ca2d8865b8f67240e1b420348997a13be \\\n    --hash=sha256:8739d77810aeb6d2ecd55cc2e4864d98a00c317fb6131a6d22ca58b3f50cf22a \\\n    --hash=sha256:886f3029ea10362c592d2e1a5326a44645b85a2a0be6fdb7c9edbdb60cedb0ab \\\n    --hash=sha256:8af85dfc2ad564f95b3e650024578bb2268bc4d827d418aab9650bbde4b6926a \\\n    --hash=sha256:8ea6265535b25f8254868f5a212a1c9dbd88c2c4050f3df18d9174949df0d6d9 \\\n    --hash=sha256:9b0198962bbba64a88bf15dc5d95e4bab8c5208dede22a10fb06280406ed9874 \\\n    --hash=sha256:a5a0624aaefb0a29e4ed7500a98732a1b0ec2d3040fe86cc8837f82e3fe4f446 \\\n    --hash=sha256:a9a45a23b020e667499fb2dbce0f1373b1b1b8cf88e47f3031e69ef3ea21f3ee \\\n    --hash=sha256:ada87643b20a8fa5763e6dd4fa0f5e2901cbdbe0833deb6d5c5a9c3186782b94 \\\n    --hash=sha256:aea0cc5609a5f2a2500f91bd6e1989fdd22da0f225be00dc1c4cec775838a7d0 \\\n    --hash=sha256:b02686b2e47115b2378ab6156abd984c776c6faa4f56898fbcc1c30ee387937d \\\n    --hash=sha256:b432b6b54a0f3be118c6243312c826aa086aa9bcb150160be4f0bc8a21ea3aaa \\\n    --hash=sha256:b46500e4d6f5708a9127ea12902ba54cb0ec594fc6bec649afc53154a1125b2b \\\n    --hash=sha256:ba65eb3e2c46ad0922d84ef78e287c2f48b7a8cae4420728bf3b8e583282d6d7 \\\n    --hash=sha256:bc7b7ddc1a1f46c363354304e9a8dd93722e4a6a24f785015650898dacf40df9 \\\n    --hash=sha256:c144e68572af1dfcb05fe065827d0920d4118b265504928e1d2262e3701a25a1 \\\n    --hash=sha256:c3a66c749b69e9abd92e519ccbdea7210522b0fffdba3f6492a8fab461d4708a \\\n    --hash=sha256:c519433e8632284dbe64ee4c81b609a17e205bc1afff285553723964880d1c89 \\\n    --hash=sha256:ce818581070527883e3678b92b1ac7ed3569e48c80f51b89b264f97d84be2608 \\\n    --hash=sha256:cf0033f343cb1592fb24ae3eaf8b58156f208884ac16991094293ee94c585654 \\\n    --hash=sha256:cffa168522a0e057d3a5628c8ebd43ddf66a34613f42e116bffb85b0b32ce5a0 \\\n    --hash=sha256:d2d10f89c4bf7cd42b84695da617f9130fa594647ee74673bf8c8d001fa04ef9 \\\n    --hash=sha256:dbee639a23c4e067aedfbcf84d2e466a71a1a1156ad04d6d09023791ed167495 \\\n    --hash=sha256:e4677025ee40b3b1420387ca85f55545b8095f7596c3d5d59552dbee19c91f07 \\\n    --hash=sha256:ee4f85d1341af630d787514eef1a0e1ad883f9d6c11a4c1465faeb9cd0bb5ced \\\n    --hash=sha256:ee7b9f6c835ea1a9beb5a35c8eb0c62b6a3290f8105d3d0c29a0a18c90f9b8ba \\\n    --hash=sha256:ef0610dfe7bb5bf574d9bdad6f597403ebc9807d124ac6f148d7604b2609be98 \\\n    --hash=sha256:f014bbf05f30731bf8b9f5c7d2b4c0e4f28f375926f95b187f85e0bbceb16029 \\\n    --hash=sha256:f2032ac005e14c56e774fd19f2bc1c22c796c0c3b39d91741c0e3f9575944d38 \\\n    --hash=sha256:fa995d96ced196388d8ed7a67ee001b181459d230e24bb68b193ae859f5609da \\\n    --hash=sha256:fd3dcb69d65c05a2fffa5d9b743b77e0cdc13832665ba435065493dad2b57f83 \\\n    --hash=sha256:fd9297a5f670f3e59ae697289ddad5902d3c4a86188e83e7823bde06334a560c\n    # via matplotlib\nfsspec==2026.9.0 \\\n    --hash=sha256:0f08147951c8cb31d844c3547d631053b127863b60be04cf06e121333ee0e2fe \\\n    --hash=sha256:8dd6e646e99ea382bd85f97a45e6b526a442d79423a7dc673f1e2756d05fcb5f\n    # via\n    #   huggingface-hub\n    #   torch\nftfy==6.3.1 \\\n    --hash=sha256:7c70eb532015cd2f9adb53f101fb6c7945988d023a085d127d1573dc49dd0083 \\\n    --hash=sha256:9b3c3d90f84fb267fe64d375a07b7f8912d817cf86009ae134aa03e1819506ec\n    # via\n    #   -r tools/rice-requirements.in\n    #   open-clip-torch\nhf-xet==1.6.0 \\\n    --hash=sha256:0e6e21fa3cdfcdcd76748564bf593870a5e013f47d97cf10aed63aa222cff5b7 \\\n    --hash=sha256:23379c2f9ec8696d952b16414a2bae72cad86a52df869b050698ba60f538c675 \\\n    --hash=sha256:2e58454a340b3556dfa4972d5451aff4fba8dd42a236600ba1a1d2b1514f0fef \\\n    --hash=sha256:35cec30d75c6f9eb9c16a77cef68e85a103b72e24d4b473714ec9ff06428bab9 \\\n    --hash=sha256:3dc3e35441ba395006af5aaacc40ef2e603c51ef46c3530b9156185f00935ea3 \\\n    --hash=sha256:4fc74352a17015bd0ee90038bc9efe38db894cde45f268b6712b04fce8cd0acb \\\n    --hash=sha256:5153e6bb103ad49d6ea9f1b2e230db5a2ea32551ad09a706d2f61d7c7c80d80e \\\n    --hash=sha256:5789835d7c6bc9436962853192082374297fb72d7eff7e7762ec25ceb7e25338 \\\n    --hash=sha256:633dc0cd71d32da58ab8c03ad38e2fac452c15c2b0a2866ebf6ededfe0a5061d \\\n    --hash=sha256:70cbb9c896901600128cb9b6f06e132954fbede1db30f31f7c6c63f84cb7c31d \\\n    --hash=sha256:75765820ce4700db3750c94acc8fe27c5fae4c9ec000a0dbac3ca082acf97765 \\\n    --hash=sha256:8fb4f71cba6129110c3374a33f919001ff130488fc23553698e34cc1c2a1198c \\\n    --hash=sha256:948f15d3a9545cfe5932f6bd8b440f6ae630aee108f14b7bd6c561f7c2dcc522 \\\n    --hash=sha256:d62671bb130879cef0ee4c9ebe47a14af6c66ec53e6d84dc15936e5ffdfac82f \\\n    --hash=sha256:f0906082d9932ae0c0057fa194041c22b4e2cdb46b2592ef3b91f020d62a081a \\\n    --hash=sha256:f2f7278c05c22fd60cb436cda1269649b3e81db65ecdc8496e5e164aa4143e7b \\\n    --hash=sha256:fb4fadde1b2b70bf4c0c14a6dccbe7194b1c28947fefd5bbe3fed9d940676c3b\n    # via huggingface-hub\nhuggingface-hub==0.36.2 \\\n    --hash=sha256:1934304d2fb224f8afa3b87007d58501acfda9215b334eed53072dd5e815ff7a \\\n    --hash=sha256:48f0c8eac16145dfce371e9d2d7772854a4f591bcb56c9cf548accf531d54270\n    # via\n    #   -r tools/rice-requirements.in\n    #   open-clip-torch\n    #   timm\n    #   tokenizers\n    #   transformers\nidna==3.20 \\\n    --hash=sha256:a7db850025b95ded1eae8a46181a1a6c56c92c96f0e2b005d9ff8dc0210cab44 \\\n    --hash=sha256:ab7ae7122974553370f0bdb919e1a960b2cd1bc1ef0276416d896db81c14582c\n    # via requests\njinja2==3.1.6 \\\n    --hash=sha256:0137fb05990d35f1275a587e9aee6d56da821fc83491a0fb838183be43f66d6d \\\n    --hash=sha256:85ece4451f492d0c13c5dd7c13a64681a86afae63a5f347908daf103ce6d2f67\n    # via torch\njoblib==1.6.0 \\\n    --hash=sha256:2ccc96785b12046c08fd6d55839c12857831b54a3c1673ffadd2f04bfc4eda03 \\\n    --hash=sha256:3dbbf9f6e4b592a2357b854608e980fe6390d131d7a82f011a377ef2ebef7aba\n    # via scikit-learn\nkiwisolver==1.5.1 \\\n    --hash=sha256:007a5553dfc4f4e8d184f588a0200e2cd4b63a59cc8796df3c39909e679dc7a0 \\\n    --hash=sha256:0324cd2567259b7a095f6cf18a52b0ffc6f3de9e69528ff1bc0e7a37bd43ff1a \\\n    --hash=sha256:0627b9bceb9c3cdcf12b8a18655eedfed2692b038df27423383c120d0b7dc2d6 \\\n    --hash=sha256:06a6917674de9e0fe3f66f5430787f59a9f2ddb64af9b714eaec547e29ef5c19 \\\n    --hash=sha256:072bdb15a3c19a5b5dbc8f8fb1f4e1884bf4f3507eeb4cc6334401274d37a5c0 \\\n    --hash=sha256:0a4faea5c6db201c6a21391d2ac926ea97acf7dacdbc3c417189e1adb1a00837 \\\n    --hash=sha256:0ba9527afc80ae3d7814ed98b6572d02bf85eaf48065678342c5f0c6dab7a8c7 \\\n    --hash=sha256:0d8924877ce22e17326a99a418c3c82037da078df3c6a260b13eca677444e6e7 \\\n    --hash=sha256:0ebdef3eae5336568147c39a55be6a2036ffde53faa9ca2d978989ae7c2da12c \\\n    --hash=sha256:1209042a623ddfda5497e4066c7b77651dde8e1d3a9dd97599dc7e97f3b9b78c \\\n    --hash=sha256:16895f553ee6620a827d2da56b871f835fb70b9216cca5d188e885caf6e3bd23 \\\n    --hash=sha256:17851e5dad4484be0cbccbde3b15331deae036de9aebd45eed964487802b172f \\\n    --hash=sha256:1798e83840c3f627246104c4d8a9639c60fa068adf9ce92b61791781fa8a68c1 \\\n    --hash=sha256:18170a77ddfecf40ec60d0928268dc95880c881864e015a8f34094ed18b9b9ad \\\n    --hash=sha256:186884a58486651e3c217b6acea0a53eaa9498fdd472057c46f2f0fb5c25aad5 \\\n    --hash=sha256:18a0cfb124546a4c2e6087c5f3029c7f44b37c85b142e0ced71f73a7599ac208 \\\n    --hash=sha256:1983f0974a750a6f6556f368ba11105d1d8369c735b944747c9f12ae5aea7aae \\\n    --hash=sha256:1a7587dc335f2c0f5bd577fd0540bd16c66006bdb60f759a1059f025e6c4f071 \\\n    --hash=sha256:1acc7e5b7ef05e9da8bb70cd6c7c4513090213d2e1ad9720f599f0bf6c52aec5 \\\n    --hash=sha256:1d852545c4d0e35a72728d072cbaa59e2fa7dd84bdf01e068d670dd0ceb58eb6 \\\n    --hash=sha256:1ed0f5e49d0ceff8b72190824d9e59c062fbbc02c231b853112c78474b3f5ec2 \\\n    --hash=sha256:1fff05e239575b1481b6ed1a782f6fad616efbf1f0b1f44e6e85c4dfe426e483 \\\n    --hash=sha256:21e46b23a2da695c364124817bc01d970effd5483147f8d66a6a7167e3f6b851 \\\n    --hash=sha256:22d5e5aaad6be121f2515765e3b1c444352cb8eb4c86510801db8f2e50757316 \\\n    --hash=sha256:2551cf9917af48ee7c4b29cc82320489508cf96fd26a51f6fc124de661cd44c7 \\\n    --hash=sha256:255605693a483db7bd5c79f60437f7bf658f7f520d61aa42722e32257c941951 \\\n    --hash=sha256:26e8268480be5061d509e29669d59103c067a26377a56491630ece11762e3858 \\\n    --hash=sha256:27add358abe374ebaa3b8763ef380bc99051b5a4b18d94878366a9e4f59efef0 \\\n    --hash=sha256:2ae70bc59790d2af72a3f76f24b272403e135070340281108b447cb77ea70819 \\\n    --hash=sha256:2e10ae1bba1899188b33557c10d73affcc12033edd18adddb57d209039976a4c \\\n    --hash=sha256:3221f78211074f561c44ca42eac0619828171bec15a2c4cf6f7747d07df76e8e \\\n    --hash=sha256:34633ecf50d16187ab8e5528b7a2530f2feb4e23f300db4672538b51cfc5cd38 \\\n    --hash=sha256:34ec467940442c9943016fb2d4c81d1ba84351eeca2f1a78f8bc87f1ba0d414c \\\n    --hash=sha256:37f801b5d7cc0e5a548921308e059fd2b057bb42972b591cfa3049f95423c4ed \\\n    --hash=sha256:38f6e0deb4d0a4615efe0c4efc5990b06ae450ab50a0b321c0b078b6d238c083 \\\n    --hash=sha256:3c24cd69455e1b00ddf770c13b6e2c33e07d6dc3f2d34add0bf9277c5c6bbd46 \\\n    --hash=sha256:3cc210010fd2f438a3ed430b45f1b501fd13a8618bf984dc2c5ce5b69b78752e \\\n    --hash=sha256:3fa5855898f6d3d01b72ccd48a2d65cbdee301251603fefe34e2025bddba219c \\\n    --hash=sha256:416ba7ff9f233b7036689bb5a3783537e838ad483f63558d2a800f75afe738b1 \\\n    --hash=sha256:431dc224a1a92a5c8f582d96e505196a3b5997a7271076678da2dfde67b77e9a \\\n    --hash=sha256:43844c1a7ad6d723d5b5b4c4fc7f5bd399c40e288120d16257c7c9e8765c6e85 \\\n    --hash=sha256:44b8faef94f1857e77fa0238f3390ff1ac51d2ea20a487e2e452a59fd2b5f5ca \\\n    --hash=sha256:470d420f98d368d6f010633a20659b544c5fdfa5329e6b70219f2ef08fd4a7ef \\\n    --hash=sha256:482676e5bd48d70ac99d9fc78863469845421e01184fa83f1f9366dc49f7e974 \\\n    --hash=sha256:4d4ca09bf13cff792b1884f64b98ee6c2467930d632233be25c56b442d99f10e \\\n    --hash=sha256:5025e36fb4fb275cef0a4e30dbb11cb4ae61d1c83deb90189cb5d7e4cafd6b55 \\\n    --hash=sha256:509735237ae0d849e8a843551d423d2500d2e0a9ac1611a145658b29c0fb9f85 \\\n    --hash=sha256:534f02c1abb31ed6dbd3515545285c330b2f12d00fdb1fdb71658b9ca5a13a6a \\\n    --hash=sha256:5978c3340f16a35c30f8ab2fa7bcf559973c55f1a5ef6970e1f621acf3c4db13 \\\n    --hash=sha256:5b973887ff782cfd6b67c9904ad8ca542e0bc5e4961503408b423b5a688b4d38 \\\n    --hash=sha256:5c490db2168a508088f59140dd392556a54b8bd1048fc6383c8baff13c359673 \\\n    --hash=sha256:5d142e352eb13facc7dd047489aebdff6ba78576c239f1ea04931979caaf0567 \\\n    --hash=sha256:5daa1f19e097050b9c4d9a78fcc9263cb96c9dfae08037ddc1b7c4ad1889f2a2 \\\n    --hash=sha256:61e9a64c7635095a6bfe483e2ff055d437c59bd45f3617a228b37277f0185d62 \\\n    --hash=sha256:63fb7294b768f444eb4b068965f2662f28c2fd4161e23bd60fcf3ff27b74c046 \\\n    --hash=sha256:685929988b208a911f1285e2f8ed54210b0d681a3dc0f03e00d599d291986e7e \\\n    --hash=sha256:6a797a1cefc8b9c93170db580337e1fe3d011ad18b1299943231279406342048 \\\n    --hash=sha256:6b92f60017dda7d877fdc546438b5e28f31c523264f49cf5a48c1d0ce1a0dfbc \\\n    --hash=sha256:70ed9a45c7484d2b30cdacf60d220f494a1763b9fec1ad03285c6553fa0889f2 \\\n    --hash=sha256:719a35fa1156db3640555f95ebb94f60a444e64d1c69626b0edef5df78eba225 \\\n    --hash=sha256:74ad5c3dad54a4641b4c28cd15ded70899d04459c6c7aeacafea716be97cce6d \\\n    --hash=sha256:74ea337e0ec3f6f342a36a4f1b5cd94dd9affddcd28ba9aae2905af932ee8c6b \\\n    --hash=sha256:75d9b1cf8258462dbdc1eeda718c96ea7f079324c09067f6daabfcf37712b7fe \\\n    --hash=sha256:77a4c8187a5948d7f8795adb765a3c7b553d07d86d88e43038fc32fc1fb9a3f3 \\\n    --hash=sha256:7824b5e8bdbf0bccb4ccd37bbb115849a1dc45437fb4de8351385ed07c437ee0 \\\n    --hash=sha256:7d38b0c279c3032e8c9cc013b405c6df8e1668dbf15465779aa7f15f61201812 \\\n    --hash=sha256:7e9c01d3dd7ceba4d1d436cc021d40d592466e40b9bc7f5d83dc4e98a5c9cd8c \\\n    --hash=sha256:7fd82debf43c6acd0a94359d232f6bb516ee13f269a7993736a9ac9f988bb5d9 \\\n    --hash=sha256:824c3d763a05ea9e9003610145186b0e9848c7584a5575c79bac5a8e7cd80bad \\\n    --hash=sha256:828f75af2b0080c8a972e75f649ab46af008e92c6104a57a759157200b835b75 \\\n    --hash=sha256:83f78128fa28705fa85d01c59771c72fe81c11bd0e6155edbb9f818983a7d761 \\\n    --hash=sha256:876bbfd276473d3daffe30e8c975df4ed9429967b41a6cb362dbb5155b6f13ad \\\n    --hash=sha256:886fc26012f0e8b5f69d1cfe6d711f6b11f194621539bf8e6bb1c25c5dc82724 \\\n    --hash=sha256:8a34616dc2521cc8dc1d7d081734da63539f021ac0450ce950908340c6e7aa2f \\\n    --hash=sha256:8a708a47ade1fe19e8371d5da076bac0dd4b0a5a7985ad6c637f7f7e361b6baa \\\n    --hash=sha256:8af9b142ad719ae3a911ebf616bc4b78b32bbab84d6a40d3ad2f129670509957 \\\n    --hash=sha256:8bf4df63592c2a66b4f8edc5df2544998c288aa02f96ce0acd880cd1de8c8127 \\\n    --hash=sha256:8de6f2a4ce7e7bd27d23dd94abf0ccafe0e0e5cc9c764b0577191f2c25f08f26 \\\n    --hash=sha256:8f8fddb8e323bd6eee4e54e69a39243beab22689070f4c66b472c4cc88bb89d8 \\\n    --hash=sha256:8fca690b00c4c48f6c2a547b0160ed511357093a4e4c9b47e0fadf3128066d89 \\\n    --hash=sha256:9506e892bcc3b409831d363c6f53e5985e1c8d1f6f6b0256d00358684ff85378 \\\n    --hash=sha256:958254518717542d02d0688d0d20cbf771da5e415e6f49543f92481c850a4540 \\\n    --hash=sha256:95a02752aa032eef4aed01cda6d9b687c669bd0396bf4519eef8bba22a286720 \\\n    --hash=sha256:96c30002424670b5e1e46495c2b8cbffef39cf77c1d79e76462029d50339785b \\\n    --hash=sha256:98b208a7cc42c803445ef551d6753cc42a5ea13e9cab1ee66cd8b9cb70195330 \\\n    --hash=sha256:9b3092d8992a1d69b7a59c3e39f35e1b9be327a17f68a7c35fc17329e337d6f2 \\\n    --hash=sha256:9e51c119992ea8820706871c30a4642ec76de20ae82f9b50b9a45517d8e9f810 \\\n    --hash=sha256:a5716a33bfabb2c6ce27b6cf03253467b3804f83e215f4d202685cf93c6c9874 \\\n    --hash=sha256:a5a00665d1a0e26763a7338d7e911d4598fbc1d50dd0d6b7919b7dc6c5d6569f \\\n    --hash=sha256:a5ca5aebae78a0bc13c1943af4af615d4966c5b650b05d5aa83b50e427196fee \\\n    --hash=sha256:a7b85b2cc6ea45e5f7e8c9a30bc9fabd47cda09106cbb4b967335c3e6c43b69d \\\n    --hash=sha256:a83ee7107df13abe42a54a6654670eef9bb39425cf2e27f65e0007465e1286ab \\\n    --hash=sha256:aa7d00b1700966d2917e54d278aba86897890ca9276dd8b76cf6446b6c181b92 \\\n    --hash=sha256:ab620eb663952455271ac37f9aaad86b73c969c02f11f53cea405b38e96a4300 \\\n    --hash=sha256:ad8b9671348d7c8716715652ae11f85ed0eb99e265a2df2ca490577d69860b2c \\\n    --hash=sha256:aefe930d113798330e9462f7874542977869c0613cba3262e2de3a8d5dee8f3a \\\n    --hash=sha256:b03af77d77e50edba2030fd5f7c352ff209314b09030a3cba7c14edf9a09a444 \\\n    --hash=sha256:b390aec180a7c054919c04898835e1c77bced23ea8383eb2c570213bf25d1a86 \\\n    --hash=sha256:b3d78f7bb2b9d9a30345be1474b9aaa8685430b54afb51ba3639b5c6c11e9ed6 \\\n    --hash=sha256:b5664603a253efd3a75716d793d1d3a6a82723b61dc6db767b2460bbbeec4c0f \\\n    --hash=sha256:b69602970994a2ed8bbfa78c2f0394a7435226c6040489702d9f0a0ad0c07052 \\\n    --hash=sha256:b6ae6a0328f0bc035741820fdeecdcd67bf4694eee03972e843663107122f450 \\\n    --hash=sha256:bad20d4c69c851c982a1e3606f4c293edfd5a87885786c50082412240c4b1ffd \\\n    --hash=sha256:bb7c99f0673c03017a3ee01e54a5c2617a05468b11eabe513b0080e063ed95b1 \\\n    --hash=sha256:bebb89489b279b2f5661bbbb2abcc87bcd4a46607bb4a5c966f04f1db6b8df9a \\\n    --hash=sha256:bfd1de989b3330420e29de39352f5c049905c9e3ee67233a50d550e3d652c148 \\\n    --hash=sha256:c2306e8bb53601979fcb3fa09cc65e031876d9ae01eff2fcbcd7a84ef94d5bc1 \\\n    --hash=sha256:c3a4e41e3096bf1f0f1b76e2ffd6d828d6547f574f702d59bdbef7acfa59db9c \\\n    --hash=sha256:c6834b92dd2428e2dd85ef3d85f723d3c12f20aaf43a2ddd4f944ca25d833408 \\\n    --hash=sha256:c90d3022d8a94778939cda8638c6c8da8fa757b8958dad7ec868ce29c87681b8 \\\n    --hash=sha256:ca307d6c259e5c98d3cb9ade55342b47a6839762caf2536f3d7b46ee660cc82e \\\n    --hash=sha256:ca7f6fe0f37ca978a1e5eb7a3a68e6413f417e78e838324947ffd420202b198b \\\n    --hash=sha256:cb6fae641357ed2f6e533c0d3c6504a4a5703621a50c89459e46051d56b61140 \\\n    --hash=sha256:cdaeeb6c350106df6bf9d873395973e5f066a9713200b72cd64f55d0a3eafab6 \\\n    --hash=sha256:cea20da04494e662b83c872683bf4ff2345206043d036315ed0e924b652e7294 \\\n    --hash=sha256:cea90547bfd93807e0013a004dc76552be44fad3bc1cc2b38610a9e889ed098f \\\n    --hash=sha256:d09037ca068d784ebc4aec290ef952ca27ac15dd9c0b5801a88c6e1096b83e6b \\\n    --hash=sha256:d27c2123977cb9269c30a49ba45f03a4323017ef693e19db4ec9dbe1299a3002 \\\n    --hash=sha256:d50de98e8d807dc31822fff96f50293163a62418eb65487a21b42713d72ed0b7 \\\n    --hash=sha256:d66a64dd5dec136040ec2ae94aa026a912ee60fdd45bc28d3db30037fd809e88 \\\n    --hash=sha256:d79308fa689fac89cbcfbd4dbfc80b5f95c54c5a7fd4d194be221f9d33d026e6 \\\n    --hash=sha256:da3275833be0edbaf4830fae08bae3dc7219f40ce0c37eaa6c25825957e06612 \\\n    --hash=sha256:dc1a26b8e53395a01c2c611e58602fa47461f136fba7cd5542e6db6d64be1839 \\\n    --hash=sha256:dc23390afe9f4ef9ac3bcc72a03a56eebbde03f4c571a32cb38f859cff9a6524 \\\n    --hash=sha256:e05c2f7925f1d88778e53cb44f14e0223204a3bdd09a41664750363acfb1f2ef \\\n    --hash=sha256:e12dfea7f5fc2a34a9080efbf79c4c44eb380ec5b9c6fea09407e08f0d1e941d \\\n    --hash=sha256:e4e4523d6f336708d732516e6cfca7796cf3d96c9474eb5aecf6165f2f1fefc3 \\\n    --hash=sha256:e4e49f7e1a4e7191bdf9dc67a974db714501b1fc52c24324103d06a86abd5c08 \\\n    --hash=sha256:e68e151428b5384f766cd25739bf77c7e4a3dc93b5ded7a12118d9fbfdf78ab6 \\\n    --hash=sha256:e8e4d953faaded9ec7ede36824e9814082d22d4c7b1eafbfa079ecba8cd0d076 \\\n    --hash=sha256:ee9df1f0d77b9c6e94f4ac0fec533fbddd5ea3a327807f18d7b069ae019ded80 \\\n    --hash=sha256:f0a887b6565bbfe80efde2b7f6e8890d7d9bbdb11bdb17028a3690c32fe0621f \\\n    --hash=sha256:f0f4a42db92d6ec7677ab9d12830a2a8ec145a9c6d15db2b593466bc875c78d7 \\\n    --hash=sha256:f1303ef2eec81262a4b708c3e858afe58d7c75ad91c1c05266eda7673369859a \\\n    --hash=sha256:f1d56ec54d257d05e0b50f5780d967540cd07beeaf9e5f645b26d50cce79f4d8 \\\n    --hash=sha256:f4167e87b397f273dc2356fcf1eaf50a6bac51e6105f45103ef7129c8efb0255 \\\n    --hash=sha256:f76fc85bd054c806960f917ec0f329e24e436f1712267d90588e4c39890caa63 \\\n    --hash=sha256:f942903fde7363d1d879057ec5de01310efda2597161784d752fa9953a01a71a \\\n    --hash=sha256:f9b1c4900736e489a812c529100de4b8fb617d4db075e931e213c57424b83d9b \\\n    --hash=sha256:fc271a6f0a2126958f4090e5507b9da5848927dae331f8f763bd4aa642b3d2cd \\\n    --hash=sha256:febcce10f2bcdbb80b4ea919238a6a4ac13dbc4c7cadbe8d5d75c3682f8b5404\n    # via matplotlib\nmarkupsafe==3.0.3 \\\n    --hash=sha256:0303439a41979d9e74d18ff5e2dd8c43ed6c6001fd40e5bf2e43f7bd9bbc523f \\\n    --hash=sha256:068f375c472b3e7acbe2d5318dea141359e6900156b5b2ba06a30b169086b91a \\\n    --hash=sha256:0bf2a864d67e76e5c9a34dc26ec616a66b9888e25e7b9460e1c76d3293bd9dbf \\\n    --hash=sha256:0db14f5dafddbb6d9208827849fad01f1a2609380add406671a26386cdf15a19 \\\n    --hash=sha256:0eb9ff8191e8498cca014656ae6b8d61f39da5f95b488805da4bb029cccbfbaf \\\n    --hash=sha256:0f4b68347f8c5eab4a13419215bdfd7f8c9b19f2b25520968adfad23eb0ce60c \\\n    --hash=sha256:1085e7fbddd3be5f89cc898938f42c0b3c711fdcb37d75221de2666af647c175 \\\n    --hash=sha256:116bb52f642a37c115f517494ea5feb03889e04df47eeff5b130b1808ce7c219 \\\n    --hash=sha256:12c63dfb4a98206f045aa9563db46507995f7ef6d83b2f68eda65c307c6829eb \\\n    --hash=sha256:133a43e73a802c5562be9bbcd03d090aa5a1fe899db609c29e8c8d815c5f6de6 \\\n    --hash=sha256:1353ef0c1b138e1907ae78e2f6c63ff67501122006b0f9abad68fda5f4ffc6ab \\\n    --hash=sha256:15d939a21d546304880945ca1ecb8a039db6b4dc49b2c5a400387cdae6a62e26 \\\n    --hash=sha256:177b5253b2834fe3678cb4a5f0059808258584c559193998be2601324fdeafb1 \\\n    --hash=sha256:1872df69a4de6aead3491198eaf13810b565bdbeec3ae2dc8780f14458ec73ce \\\n    --hash=sha256:1b4b79e8ebf6b55351f0d91fe80f893b4743f104bff22e90697db1590e47a218 \\\n    --hash=sha256:1b52b4fb9df4eb9ae465f8d0c228a00624de2334f216f178a995ccdcf82c4634 \\\n    --hash=sha256:1ba88449deb3de88bd40044603fafffb7bc2b055d626a330323a9ed736661695 \\\n    --hash=sha256:1cc7ea17a6824959616c525620e387f6dd30fec8cb44f649e31712db02123dad \\\n    --hash=sha256:218551f6df4868a8d527e3062d0fb968682fe92054e89978594c28e642c43a73 \\\n    --hash=sha256:26a5784ded40c9e318cfc2bdb30fe164bdb8665ded9cd64d500a34fb42067b1c \\\n    --hash=sha256:2713baf880df847f2bece4230d4d094280f4e67b1e813eec43b4c0e144a34ffe \\\n    --hash=sha256:2a15a08b17dd94c53a1da0438822d70ebcd13f8c3a95abe3a9ef9f11a94830aa \\\n    --hash=sha256:2f981d352f04553a7171b8e44369f2af4055f888dfb147d55e42d29e29e74559 \\\n    --hash=sha256:32001d6a8fc98c8cb5c947787c5d08b0a50663d139f1305bac5885d98d9b40fa \\\n    --hash=sha256:3524b778fe5cfb3452a09d31e7b5adefeea8c5be1d43c4f810ba09f2ceb29d37 \\\n    --hash=sha256:3537e01efc9d4dccdf77221fb1cb3b8e1a38d5428920e0657ce299b20324d758 \\\n    --hash=sha256:35add3b638a5d900e807944a078b51922212fb3dedb01633a8defc4b01a3c85f \\\n    --hash=sha256:38664109c14ffc9e7437e86b4dceb442b0096dfe3541d7864d9cbe1da4cf36c8 \\\n    --hash=sha256:3a7e8ae81ae39e62a41ec302f972ba6ae23a5c5396c8e60113e9066ef893da0d \\\n    --hash=sha256:3b562dd9e9ea93f13d53989d23a7e775fdfd1066c33494ff43f5418bc8c58a5c \\\n    --hash=sha256:457a69a9577064c05a97c41f4e65148652db078a3a509039e64d3467b9e7ef97 \\\n    --hash=sha256:4bd4cd07944443f5a265608cc6aab442e4f74dff8088b0dfc8238647b8f6ae9a \\\n    --hash=sha256:4e885a3d1efa2eadc93c894a21770e4bc67899e3543680313b09f139e149ab19 \\\n    --hash=sha256:4faffd047e07c38848ce017e8725090413cd80cbc23d86e55c587bf979e579c9 \\\n    --hash=sha256:509fa21c6deb7a7a273d629cf5ec029bc209d1a51178615ddf718f5918992ab9 \\\n    --hash=sha256:5678211cb9333a6468fb8d8be0305520aa073f50d17f089b5b4b477ea6e67fdc \\\n    --hash=sha256:591ae9f2a647529ca990bc681daebdd52c8791ff06c2bfa05b65163e28102ef2 \\\n    --hash=sha256:5a7d5dc5140555cf21a6fefbdbf8723f06fcd2f63ef108f2854de715e4422cb4 \\\n    --hash=sha256:69c0b73548bc525c8cb9a251cddf1931d1db4d2258e9599c28c07ef3580ef354 \\\n    --hash=sha256:6b5420a1d9450023228968e7e6a9ce57f65d148ab56d2313fcd589eee96a7a50 \\\n    --hash=sha256:722695808f4b6457b320fdc131280796bdceb04ab50fe1795cd540799ebe1698 \\\n    --hash=sha256:729586769a26dbceff69f7a7dbbf59ab6572b99d94576a5592625d5b411576b9 \\\n    --hash=sha256:77f0643abe7495da77fb436f50f8dab76dbc6e5fd25d39589a0f1fe6548bfa2b \\\n    --hash=sha256:795e7751525cae078558e679d646ae45574b47ed6e7771863fcc079a6171a0fc \\\n    --hash=sha256:7be7b61bb172e1ed687f1754f8e7484f1c8019780f6f6b0786e76bb01c2ae115 \\\n    --hash=sha256:7c3fb7d25180895632e5d3148dbdc29ea38ccb7fd210aa27acbd1201a1902c6e \\\n    --hash=sha256:7e68f88e5b8799aa49c85cd116c932a1ac15caaa3f5db09087854d218359e485 \\\n    --hash=sha256:83891d0e9fb81a825d9a6d61e3f07550ca70a076484292a70fde82c4b807286f \\\n    --hash=sha256:8485f406a96febb5140bfeca44a73e3ce5116b2501ac54fe953e488fb1d03b12 \\\n    --hash=sha256:8709b08f4a89aa7586de0aadc8da56180242ee0ada3999749b183aa23df95025 \\\n    --hash=sha256:8f71bc33915be5186016f675cd83a1e08523649b0e33efdb898db577ef5bb009 \\\n    --hash=sha256:915c04ba3851909ce68ccc2b8e2cd691618c4dc4c4232fb7982bca3f41fd8c3d \\\n    --hash=sha256:949b8d66bc381ee8b007cd945914c721d9aba8e27f71959d750a46f7c282b20b \\\n    --hash=sha256:94c6f0bb423f739146aec64595853541634bde58b2135f27f61c1ffd1cd4d16a \\\n    --hash=sha256:9a1abfdc021a164803f4d485104931fb8f8c1efd55bc6b748d2f5774e78b62c5 \\\n    --hash=sha256:9b79b7a16f7fedff2495d684f2b59b0457c3b493778c9eed31111be64d58279f \\\n    --hash=sha256:a320721ab5a1aba0a233739394eb907f8c8da5c98c9181d1161e77a0c8e36f2d \\\n    --hash=sha256:a4afe79fb3de0b7097d81da19090f4df4f8d3a2b3adaa8764138aac2e44f3af1 \\\n    --hash=sha256:ad2cf8aa28b8c020ab2fc8287b0f823d0a7d8630784c31e9ee5edea20f406287 \\\n    --hash=sha256:b8512a91625c9b3da6f127803b166b629725e68af71f8184ae7e7d54686a56d6 \\\n    --hash=sha256:bc51efed119bc9cfdf792cdeaa4d67e8f6fcccab66ed4bfdd6bde3e59bfcbb2f \\\n    --hash=sha256:bdc919ead48f234740ad807933cdf545180bfbe9342c2bb451556db2ed958581 \\\n    --hash=sha256:bdd37121970bfd8be76c5fb069c7751683bdf373db1ed6c010162b2a130248ed \\\n    --hash=sha256:be8813b57049a7dc738189df53d69395eba14fb99345e0a5994914a3864c8a4b \\\n    --hash=sha256:c0c0b3ade1c0b13b936d7970b1d37a57acde9199dc2aecc4c336773e1d86049c \\\n    --hash=sha256:c47a551199eb8eb2121d4f0f15ae0f923d31350ab9280078d1e5f12b249e0026 \\\n    --hash=sha256:c4ffb7ebf07cfe8931028e3e4c85f0357459a3f9f9490886198848f4fa002ec8 \\\n    --hash=sha256:ccfcd093f13f0f0b7fdd0f198b90053bf7b2f02a3927a30e63f3ccc9df56b676 \\\n    --hash=sha256:d2ee202e79d8ed691ceebae8e0486bd9a2cd4794cec4824e1c99b6f5009502f6 \\\n    --hash=sha256:d53197da72cc091b024dd97249dfc7794d6a56530370992a5e1a08983ad9230e \\\n    --hash=sha256:d6dd0be5b5b189d31db7cda48b91d7e0a9795f31430b7f271219ab30f1d3ac9d \\\n    --hash=sha256:d88b440e37a16e651bda4c7c2b930eb586fd15ca7406cb39e211fcff3bf3017d \\\n    --hash=sha256:de8a88e63464af587c950061a5e6a67d3632e36df62b986892331d4620a35c01 \\\n    --hash=sha256:df2449253ef108a379b8b5d6b43f4b1a8e81a061d6537becd5582fba5f9196d7 \\\n    --hash=sha256:e1c1493fb6e50ab01d20a22826e57520f1284df32f2d8601fdd90b6304601419 \\\n    --hash=sha256:e1cf1972137e83c5d4c136c43ced9ac51d0e124706ee1c8aa8532c1287fa8795 \\\n    --hash=sha256:e2103a929dfa2fcaf9bb4e7c091983a49c9ac3b19c9061b6d5427dd7d14d81a1 \\\n    --hash=sha256:e56b7d45a839a697b5eb268c82a71bd8c7f6c94d6fd50c3d577fa39a9f1409f5 \\\n    --hash=sha256:e8afc3f2ccfa24215f8cb28dcf43f0113ac3c37c2f0f0806d8c70e4228c5cf4d \\\n    --hash=sha256:e8fc20152abba6b83724d7ff268c249fa196d8259ff481f3b1476383f8f24e42 \\\n    --hash=sha256:eaa9599de571d72e2daf60164784109f19978b327a3910d3e9de8c97b5b70cfe \\\n    --hash=sha256:ec15a59cf5af7be74194f7ab02d0f59a62bdcf1a537677ce67a2537c9b87fcda \\\n    --hash=sha256:f190daf01f13c72eac4efd5c430a8de82489d9cff23c364c3ea822545032993e \\\n    --hash=sha256:f34c41761022dd093b4b6896d4810782ffbabe30f2d443ff5f083e0cbbb8c737 \\\n    --hash=sha256:f3e98bb3798ead92273dc0e5fd0f31ade220f59a266ffd8a4f6065e0a3ce0523 \\\n    --hash=sha256:f42d0984e947b8adf7dd6dde396e720934d12c506ce84eea8476409563607591 \\\n    --hash=sha256:f71a396b3bf33ecaa1626c255855702aca4d3d9fea5e051b41ac59a9c1c41edc \\\n    --hash=sha256:f9e130248f4462aaa8e2552d547f36ddadbeaa573879158d721bbd33dfe4743a \\\n    --hash=sha256:fed51ac40f757d41b7c48425901843666a6677e3e8eb0abcff09e4ba6e664f50\n    # via jinja2\nmatplotlib==3.10.8 \\\n    --hash=sha256:00270d217d6b20d14b584c521f810d60c5c78406dc289859776550df837dcda7 \\\n    --hash=sha256:0a33deb84c15ede243aead39f77e990469fff93ad1521163305095b77b72ce4a \\\n    --hash=sha256:113bb52413ea508ce954a02c10ffd0d565f9c3bc7f2eddc27dfe1731e71c7b5f \\\n    --hash=sha256:12d90df9183093fcd479f4172ac26b322b1248b15729cb57f42f71f24c7e37a3 \\\n    --hash=sha256:15d30132718972c2c074cd14638c7f4592bd98719e2308bccea40e0538bc0cb5 \\\n    --hash=sha256:18821ace09c763ec93aef5eeff087ee493a24051936d7b9ebcad9662f66501f9 \\\n    --hash=sha256:1ae029229a57cd1e8fe542485f27e7ca7b23aa9e8944ddb4985d0bc444f1eca2 \\\n    --hash=sha256:2299372c19d56bcd35cf05a2738308758d32b9eaed2371898d8f5bd33f084aa3 \\\n    --hash=sha256:238b7ce5717600615c895050239ec955d91f321c209dd110db988500558e70d6 \\\n    --hash=sha256:24d50994d8c5816ddc35411e50a86ab05f575e2530c02752e02538122613371f \\\n    --hash=sha256:25d380fe8b1dc32cf8f0b1b448470a77afb195438bafdf1d858bfb876f3edf7b \\\n    --hash=sha256:2c1998e92cd5999e295a731bcb2911c75f597d937341f3030cc24ef2733d78a8 \\\n    --hash=sha256:2cf5bd12cecf46908f286d7838b2abc6c91cda506c0445b8223a7c19a00df008 \\\n    --hash=sha256:32f8dce744be5569bebe789e46727946041199030db8aeb2954d26013a0eb26b \\\n    --hash=sha256:37b3c1cc42aa184b3f738cfa18c1c1d72fd496d85467a6cf7b807936d39aa656 \\\n    --hash=sha256:3a48a78d2786784cc2413e57397981fb45c79e968d99656706018d6e62e57958 \\\n    --hash=sha256:3ab4aabc72de4ff77b3ec33a6d78a68227bf1123465887f9905ba79184a1cc04 \\\n    --hash=sha256:3c624e43ed56313651bc18a47f838b60d7b8032ed348911c54906b130b20071b \\\n    --hash=sha256:3f2e409836d7f5ac2f1c013110a4d50b9f7edc26328c108915f9075d7d7a91b6 \\\n    --hash=sha256:3f5c3e4da343bba819f0234186b9004faba952cc420fbc522dc4e103c1985908 \\\n    --hash=sha256:41703cc95688f2516b480f7f339d8851a6035f18e100ee6a32bc0b8536a12a9c \\\n    --hash=sha256:495672de149445ec1b772ff2c9ede9b769e3cb4f0d0aa7fa730d7f59e2d4e1c1 \\\n    --hash=sha256:4cf267add95b1c88300d96ca837833d4112756045364f5c734a2276038dae27d \\\n    --hash=sha256:56271f3dac49a88d7fca5060f004d9d22b865f743a12a23b1e937a0be4818ee1 \\\n    --hash=sha256:595ba4d8fe983b88f0eec8c26a241e16d6376fe1979086232f481f8f3f67494c \\\n    --hash=sha256:5f62550b9a30afde8c1c3ae450e5eb547d579dd69b25c2fc7a1c67f934c1717a \\\n    --hash=sha256:646d95230efb9ca614a7a594d4fcacde0ac61d25e37dd51710b36477594963ce \\\n    --hash=sha256:64fcc24778ca0404ce0cb7b6b77ae1f4c7231cdd60e6778f999ee05cbd581b9a \\\n    --hash=sha256:6be43b667360fef5c754dda5d25a32e6307a03c204f3c0fc5468b78fa87b4160 \\\n    --hash=sha256:6da7c2ce169267d0d066adcf63758f0604aa6c3eebf67458930f9d9b79ad1db1 \\\n    --hash=sha256:83d282364ea9f3e52363da262ce32a09dfe241e4080dcedda3c0db059d3c1f11 \\\n    --hash=sha256:9153c3292705be9f9c64498a8872118540c3f4123d1a1c840172edf262c8be4a \\\n    --hash=sha256:99eefd13c0dc3b3c1b4d561c1169e65fe47aab7b8158754d7c084088e2329466 \\\n    --hash=sha256:a0a7f52498f72f13d4a25ea70f35f4cb60642b466cbb0a9be951b5bc3f45a486 \\\n    --hash=sha256:a2b336e2d91a3d7006864e0990c83b216fcdca64b5a6484912902cef87313d78 \\\n    --hash=sha256:a48f2b74020919552ea25d222d5cc6af9ca3f4eb43a93e14d068457f545c2a17 \\\n    --hash=sha256:ad3d9833a64cf48cc4300f2b406c3d0f4f4724a91c0bd5640678a6ba7c102077 \\\n    --hash=sha256:b44d07310e404ba95f8c25aa5536f154c0a8ec473303535949e52eb71d0a1565 \\\n    --hash=sha256:b53285e65d4fa4c86399979e956235deb900be5baa7fc1218ea67fbfaeaadd6f \\\n    --hash=sha256:b5a2b97dbdc7d4f353ebf343744f1d1f1cca8aa8bfddb4262fcf4306c3761d50 \\\n    --hash=sha256:b9a5ca4ac220a0cdd1ba6bcba3608547117d30468fefce49bb26f55c1a3d5c58 \\\n    --hash=sha256:bab485bcf8b1c7d2060b4fcb6fc368a9e6f4cd754c9c2fea281f4be21df394a2 \\\n    --hash=sha256:c108a1d6fa78a50646029cb6d49808ff0fc1330fda87fa6f6250c6b5369b6645 \\\n    --hash=sha256:d56a1efd5bfd61486c8bc968fa18734464556f0fb8e51690f4ac25d85cbbbbc2 \\\n    --hash=sha256:d9050fee89a89ed57b4fb2c1bfac9a3d0c57a0d55aed95949eedbc42070fea39 \\\n    --hash=sha256:dd80ecb295460a5d9d260df63c43f4afbdd832d725a531f008dad1664f458adf \\\n    --hash=sha256:e8ea3e2d4066083e264e75c829078f9e149fa119d27e19acd503de65e0b13149 \\\n    --hash=sha256:eb3823f11823deade26ce3b9f40dcb4a213da7a670013929f31d5f5ed1055b22 \\\n    --hash=sha256:ee40c27c795bda6a5292e9cff9890189d32f7e3a0bf04e0e3c9430c4a00c37df \\\n    --hash=sha256:efb30e3baaea72ce5928e32bab719ab4770099079d66726a62b11b1ef7273be4 \\\n    --hash=sha256:f254d118d14a7f99d616271d6c3c27922c092dac11112670b157798b89bf4933 \\\n    --hash=sha256:f89c151aab2e2e23cb3fe0acad1e8b82841fd265379c4cecd0f3fcb34c15e0f6 \\\n    --hash=sha256:f97aeb209c3d2511443f8797e3e5a569aebb040d4f8bc79aa3ee78a8fb9e3dd8 \\\n    --hash=sha256:f9b587c9c7274c1613a30afabf65a272114cd6cdbe67b3406f818c79d7ab2e2a \\\n    --hash=sha256:fb061f596dad3a0f52b60dc6a5dec4a0c300dec41e058a7efe09256188d170b7\n    # via -r tools/rice-requirements.in\nmpmath==1.3.0 \\\n    --hash=sha256:7a28eb2a9774d00c7bc92411c19a89209d5da7c4c9a9e227be8330a23a25b91f \\\n    --hash=sha256:a0b2b9fe80bbcd81a6647ff13108738cfb482d481d826cc0e02f5b35e5c88d2c\n    # via sympy\nnetworkx==3.7 \\\n    --hash=sha256:e3fd2c13a7814cee3746340d8d7f8598a67f16a58bf47fb7f8793fab6efca1b0 \\\n    --hash=sha256:fd77a511bd90f39f3d016351345b52cf5319b813bdca01de3f755d3cca62e96a\n    # via torch\nnumpy==2.5.3 \\\n    --hash=sha256:012e66aca395d795496446e52aeeb5866312a5d4d3f27da270e5a0b43f70dc5c \\\n    --hash=sha256:09d5a423c71ad5feb5625844ad58050e35df43871004b52ac9c0ad44a56775be \\\n    --hash=sha256:09ffa5d903faeaa5c4dd05009cf81c8bab9f2cb37c548b8d39b65b4cfa7c97f7 \\\n    --hash=sha256:0a59a421a32580a009e8a1751345bf829631b990dc1794b80514ab722b435def \\\n    --hash=sha256:116f96cadd935c6122e9228d676fe7ede19e741f5c8bb1c3cddbe0c51ccebea2 \\\n    --hash=sha256:1302b90c0e52281681b2975adfe8a860cb7b12216a27b4b0b4207c44bf7bccf0 \\\n    --hash=sha256:15aa985ac73a8db02db7663381aa109510449d3819d37206caed27b33a65a8a6 \\\n    --hash=sha256:1aad64d99730d013cfc6debafed22783b4fc5a7f4b8bc744d2d8cf7dcc880551 \\\n    --hash=sha256:1c80eabb4035ecf4ca9cd49cde8a9fdd69a729e63e6474887d1523ade7aa277f \\\n    --hash=sha256:1f3ed25271581281f2fccb1adcedfcde4c07362eec69189b50baf6f90e3ae159 \\\n    --hash=sha256:1fb6f8fb9ff0b3a69f52c66ce397b0246583e9f28616231b0e32ca49259a5fa6 \\\n    --hash=sha256:214045a5bf00113a146ab9ee9730c44501af6723cdf1f6830932f7b5ef2e7af0 \\\n    --hash=sha256:26e15e4aecd8617dfbaecb37d223e365d7b39411fba20454be2670a96aa74cb5 \\\n    --hash=sha256:2c25dfa72943e4336ddb6b0ee4277b47a0c85bede0807530ec68103bf58e2c10 \\\n    --hash=sha256:2d8240cb4c16fd831074aa2b2cf9fc54664d826341d61c372245b96a74a49a9a \\\n    --hash=sha256:350ba9783ce969cf9f7ce6e6a9a58e1a6e2a19ca025b7ee448c4db727706212a \\\n    --hash=sha256:4c8a6d2ebce6305fd82fbefca827775437147052a976ee7c94b36a0c1b52ac6c \\\n    --hash=sha256:4f8929ee6c96bfbd7b4ed2032e0c03af86fe1826740ab61ddabf9072d06e57ff \\\n    --hash=sha256:536f963710a4e63934d80ac0dc4f478804a83e9a84b6828018f25d09953ada33 \\\n    --hash=sha256:54a115e5a73b8fc44f0cebef486365a1894b5c9760685d4558b72b7c3eb846e0 \\\n    --hash=sha256:595d020938c84e320bcf40ad71089e108eac0d377cd018e14a8c094f39e98d85 \\\n    --hash=sha256:66a78fe4556c60aceda5916f9eacd638b18e9e681016ec302dcb4682d6d4d034 \\\n    --hash=sha256:6b05c171afb3aa07adbd20abc00aea86fe375beb0fdb9ef780ec5b7f63bab1c0 \\\n    --hash=sha256:6cef4bb1706dfec49243c05d921eefb4e190d41e2528b30d8035ea1f36b4c24a \\\n    --hash=sha256:6f24021b9f22bc6301c37b196974a92c1c18dccedb6fef3dd252e95f2d6adbe4 \\\n    --hash=sha256:71b39d9f935b6ec0f8753e3e2afb51e3efba6f2e05b68b32a40754d24bcd4a3c \\\n    --hash=sha256:71cad2b2a7451ab79d8f5e71b453485b6775963d5cf794179144a7463fe6e8ec \\\n    --hash=sha256:76c2c1e6bfa5c84adc6434dfbf013aa92096a7985221762c8f11fedfd20fff58 \\\n    --hash=sha256:8617bbfae4486cf99c9f899966699428d19da931d06ca94ad3da986c76e15997 \\\n    --hash=sha256:86bff898a431c0fb71f7610b75726e75a54d47b37edc9d537f48de63bb3c0b90 \\\n    --hash=sha256:8e4dd766076855b5ff7ea52fa5f07ce26286726e0f8bff446b7739d02e6ea204 \\\n    --hash=sha256:92f30e89b8ee0ecf363033576c422b2f58fed6a80bed0aa48dff6d14c654663e \\\n    --hash=sha256:93e1f5447e2b1e479d7bd74701e84746b86450cff1fc368b132d195e2b8f8211 \\\n    --hash=sha256:9a37475425b431b4d060f23b4f52cd2f3aef6bc7c654bd760adf0040eec9d435 \\\n    --hash=sha256:9deb49575e5b0b94ed72c8a64ec4d033381adc27e9060ae842971f697ba96104 \\\n    --hash=sha256:a5fa86b80fd24bcd1aff83ad23be44ea323de3f787be8f8b15d4a65621e25321 \\\n    --hash=sha256:a6391fafaba97500887132cd582abc6e19452b1ac775a47caa7b24490e152058 \\\n    --hash=sha256:a72f874bc9e10e4b8f80426fb49716d5141f64442a0c8418065093ec8017fbb0 \\\n    --hash=sha256:ac7bb1c52d445bd4f8f7f97fefe6abc3a084dc4d63df50d79b17fa2b78e89297 \\\n    --hash=sha256:adc1ada2662f8a5f960b8a10d9986897e7499ef07e06d4cfe7197f8cce923c07 \\\n    --hash=sha256:b00eefbcf0f292945c4b4dec2ae845389ef5bcdcd596e6e4328051db5b5ba694 \\\n    --hash=sha256:b0521d0f4aebb6e06189451025fa17a913287b13c03d5fe05c017333b654ea5b \\\n    --hash=sha256:b5d93cf48f687479941d12b69c873ad2cc76bbd487f0091c2200636497f34034 \\\n    --hash=sha256:b7e18c623bb5c95acb3b3328861272816ba199fb531921c5d6d0b675f1fde9e3 \\\n    --hash=sha256:bd4cb9ad3c7889b9b3fe0a9a9fb5d2ed26f9879bff2608d9f01aed147a20d231 \\\n    --hash=sha256:be5a8381859b6da607c84f4f7d6847725f1cf1853ef8a2c9e115b7d58bef47dc \\\n    --hash=sha256:befa1ae5bd6030b3f512b43ff3fa5290bbed6b84411a44244b14adf835f5b89d \\\n    --hash=sha256:bf63afbe037eb5d2fe87fbcc7778e61da53ebaf21d938a4515aa73b62532a5d4 \\\n    --hash=sha256:c00abe94c1a69d75d827dcf1c025b25c8a45d230b3bcd77a9020883a1b047653 \\\n    --hash=sha256:c2381f82999704f818e2c987a865050e285ec3621262c66d40f5a96c8f899f8e \\\n    --hash=sha256:c76d5dde9f445058f83d0c02af00557a4db91de9a9a57c0df87d1535001d654b \\\n    --hash=sha256:cb189f09db39283b26bfd061ec16189e14f71c6755207f72a0f7540867afe5b9 \\\n    --hash=sha256:ccb32e0525d29e8b0572eb84c9a57af0e7a4e615726927506f55063c62414034 \\\n    --hash=sha256:ccbc4665079665c3cf3bab4db9f6b095370cd6437d66be549b6c2a1fd19e1958 \\\n    --hash=sha256:d1c89973648c85069c5046ad460f7b8a00218b29a2e42359ac8cc63e9ab94832 \\\n    --hash=sha256:df2d5874ff183595a4ba404edd04f6bd9b5505c1d7708573f6a6c17489a67563 \\\n    --hash=sha256:e01c918ac3d48e18a927cf7b14a26a3e29ff2bdf2eacb976da0aecd6a43ed034 \\\n    --hash=sha256:e6ab667ba76450084eb64013762c438ea76d9d29cc676dcd6c2e9892ba37f841 \\\n    --hash=sha256:e931e4f499e0dc7ef29d269a8e5b35dd722e5d14be07df6240166ea7c6532fae \\\n    --hash=sha256:f54660b0eb6b0b9f36e7fe1cdfdff472028dd0d14acd9b9b65098efbad059469 \\\n    --hash=sha256:f59a878c33d6b88122d80d239bb3b845d58708750b0cb06a09aebb9b18ec696c \\\n    --hash=sha256:f7fabeb6cea87d65f3b926de33d03fb016cfdc29314c90974383b5582ae72891 \\\n    --hash=sha256:f9579f383d1bf9df80081e72760e84960a7fd4f88cf0c9e535a8597c9bb646f5 \\\n    --hash=sha256:f9a2353b37a1a9e78fd82b27ad7e2a32a2d036604d18f02b05e3136c62ca3b09 \\\n    --hash=sha256:fc36dc566135b5eceec4cf89758fcb719266a019ef07dae1754ae7c9f617ef3e \\\n    --hash=sha256:ffdc76bfcae6b255dff75202c5e7feaf95b40246bc0a17944facc1fecf9f79ab\n    # via\n    #   -r tools/rice-requirements.in\n    #   contourpy\n    #   matplotlib\n    #   pycocotools\n    #   scikit-learn\n    #   scipy\n    #   torchvision\n    #   transformers\nnvidia-cublas==13.1.0.3 \\\n    --hash=sha256:2a3b94a37def342471c59fad7856caee4926809a72dd5270155d6a31b5b277be \\\n    --hash=sha256:c86fc7f7ae36d7528288c5d88098edcb7b02c633d262e7ddbb86b0ad91be5df2 \\\n    --hash=sha256:ee8722c1f0145ab246bccb9e452153b5e0515fd094c3678df50b2a0888b8b171\n    # via\n    #   cuda-toolkit\n    #   nvidia-cudnn-cu13\n    #   nvidia-cusolver\nnvidia-cuda-cupti==13.0.85 \\\n    --hash=sha256:4eb01c08e859bf924d222250d2e8f8b8ff6d3db4721288cf35d14252a4d933c8 \\\n    --hash=sha256:683f58d301548deeefcb8f6fac1b8d907691b9d8b18eccab417f51e362102f00 \\\n    --hash=sha256:796bd679890ee55fb14a94629b698b6db54bcfd833d391d5e94017dd9d7d3151\n    # via cuda-toolkit\nnvidia-cuda-nvrtc==13.0.88 \\\n    --hash=sha256:6bcd4e7f8e205cbe644f5a98f2f799bef9556fefc89dd786e79a16312ce49872 \\\n    --hash=sha256:ad9b6d2ead2435f11cbb6868809d2adeeee302e9bb94bcf0539c7a40d80e8575 \\\n    --hash=sha256:d27f20a0ca67a4bb34268a5e951033496c5b74870b868bacd046b1b8e0c3267b\n    # via cuda-toolkit\nnvidia-cuda-runtime==13.0.96 \\\n    --hash=sha256:7f82250d7782aa23b6cfe765ecc7db554bd3c2870c43f3d1821f1d18aebf0548 \\\n    --hash=sha256:ef9bcbe90493a2b9d810e43d249adb3d02e98dd30200d86607d8d02687c43f55 \\\n    --hash=sha256:f79298c8a098cec150a597c8eba58ecdab96e3bdc4b9bc4f9983635031740492\n    # via cuda-toolkit\nnvidia-cudnn-cu13==9.19.0.56 \\\n    --hash=sha256:40d8c375005bcb01495f8edf375230b203a411a0c05fb6dc92a3781edcb23eac \\\n    --hash=sha256:6ed29ffaee1176c612daf442e4dd6cfeb6a0caa43ddcbeb59da94953030b1be4 \\\n    --hash=sha256:d20e1734305e9d68889a96e3f35094d733ff1f83932ebe462753973e53a572bf\n    # via torch\nnvidia-cufft==12.0.0.61 \\\n    --hash=sha256:2708c852ef8cd89d1d2068bdbece0aa188813a0c934db3779b9b1faa8442e5f5 \\\n    --hash=sha256:2abce5b39d2f5ae12730fb7e5db6696533e36c26e2d3e8fd1750bdd2853364eb \\\n    --hash=sha256:6c44f692dce8fd5ffd3e3df134b6cdb9c2f72d99cf40b62c32dde45eea9ddad3\n    # via cuda-toolkit\nnvidia-cufile==1.15.1.6 \\\n    --hash=sha256:08a3ecefae5a01c7f5117351c64f17c7c62efa5fffdbe24fc7d298da19cd0b44 \\\n    --hash=sha256:bdc0deedc61f548bddf7733bdc216456c2fdb101d020e1ab4b88d232d5e2f6d1\n    # via cuda-toolkit\nnvidia-curand==10.4.0.35 \\\n    --hash=sha256:133df5a7509c3e292aaa2b477afd0194f06ce4ea24d714d616ff36439cee349a \\\n    --hash=sha256:1aee33a5da6e1db083fe2b90082def8915f30f3248d5896bcec36a579d941bfc \\\n    --hash=sha256:65b1710aa6961d326b411e314b374290904c5ddf41dc3f766ebc3f1d7d4ca69f\n    # via cuda-toolkit\nnvidia-cusolver==12.0.4.66 \\\n    --hash=sha256:02c2457eaa9e39de20f880f4bd8820e6a1cfb9f9a34f820eb12a155aa5bc92d2 \\\n    --hash=sha256:0a759da5dea5c0ea10fd307de75cdeb59e7ea4fcb8add0924859b944babf1112 \\\n    --hash=sha256:16515bd33a8e76bb54d024cfa068fa68d30e80fc34b9e1090813ea9362e0cb65\n    # via cuda-toolkit\nnvidia-cusparse==12.6.3.3 \\\n    --hash=sha256:2b3c89c88d01ee0e477cb7f82ef60a11a4bcd57b6b87c33f789350b59759360b \\\n    --hash=sha256:80bcc4662f23f1054ee334a15c72b8940402975e0eab63178fc7e670aa59472c \\\n    --hash=sha256:cbcf42feb737bd7ec15b4c0a63e62351886bd3f975027b8815d7f720a2b5ea79\n    # via\n    #   cuda-toolkit\n    #   nvidia-cusolver\nnvidia-cusparselt-cu13==0.8.0 \\\n    --hash=sha256:25e30a8a7323935d4ad0340b95a0b69926eee755767e8e0b1cf8dd85b197d3fd \\\n    --hash=sha256:400c6ed1cf6780fc6efedd64ec9f1345871767e6a1a0a552a1ea0578117ea77c \\\n    --hash=sha256:e80212ed7b1afc97102fbb2b5c82487aa73f6a0edfa6d26c5a152593e520bb8f\n    # via torch\nnvidia-nccl-cu13==2.28.9 \\\n    --hash=sha256:01c873ba1626b54caa12272ed228dc5b2781545e0ae8ba3f432a8ef1c6d78643 \\\n    --hash=sha256:e4553a30f34195f3fa1da02a6da3d6337d28f2003943aa0a3d247bbc25fefc42\n    # via torch\nnvidia-nvjitlink==13.0.88 \\\n    --hash=sha256:13a74f429e23b921c1109976abefacc69835f2f433ebd323d3946e11d804e47b \\\n    --hash=sha256:634e96e3da9ef845ae744097a1f289238ecf946ce0b82e93cdce14b9782e682f \\\n    --hash=sha256:e931536ccc7d467a98ba1d8b89ff7fa7f1fa3b13f2b0069118cd7f47bff07d0c\n    # via\n    #   cuda-toolkit\n    #   nvidia-cufft\n    #   nvidia-cusolver\n    #   nvidia-cusparse\nnvidia-nvshmem-cu13==3.4.5 \\\n    --hash=sha256:290f0a2ee94c9f3687a02502f3b9299a9f9fe826e6d0287ee18482e78d495b80 \\\n    --hash=sha256:6dc2a197f38e5d0376ad52cd1a2a3617d3cdc150fd5966f4aee9bcebb1d68fe9\n    # via torch\nnvidia-nvtx==13.0.85 \\\n    --hash=sha256:4936d1d6780fbe68db454f5e72a42ff64d1fd6397df9f363ae786930fd5c1cd4 \\\n    --hash=sha256:cb7780edb6b14107373c835bf8b72e7a178bac7367e23da7acb108f973f157a6 \\\n    --hash=sha256:d66ea44254dd3c6eacc300047af6e1288d2269dd072b417e0adffbf479e18519\n    # via cuda-toolkit\nopen-clip-torch==3.3.0 \\\n    --hash=sha256:904b1a9f909df8281bb3de60ab95491cd2994a509177ea4f9d6292a84fe24d6d \\\n    --hash=sha256:c549ad5ed6bfc119cc11105033c0a2b9d7a2a4afeb40a58a09aab3da1a0043ce\n    # via -r tools/rice-requirements.in\npackaging==26.3 \\\n    --hash=sha256:94edc256424af38762eb31306eed28beb9f0efc50a8837492c9d6fd6004aed79 \\\n    --hash=sha256:d7193f7c8e4e93f444fde0262bf90af30e16fa0ad0ad44cb553c87339b23cd1c\n    # via\n    #   huggingface-hub\n    #   matplotlib\n    #   transformers\npillow==12.3.0 \\\n    --hash=sha256:00808c5e14ef63ac5161091d242999076604ff74b883423a11e5d7bbb38bf756 \\\n    --hash=sha256:04f01d28a6aaff387bf842a13be313df23ba0597a44f1a976c9feb3c6ff4711a \\\n    --hash=sha256:06ff022112bc9cbf83b60f8e028d94ad87b60621706487e65f673de61610ab59 \\\n    --hash=sha256:0740a512dc522224c77d9aa5a8d70d8b7d73fb91f2c21125d8d025d3b8990e45 \\\n    --hash=sha256:0847a763afefb695bc912d7c131e7e0632d4edc1d8698f58ddabec8e46b8b6d3 \\\n    --hash=sha256:0dd2064cbc55aaec028ef5fbb60fa47bb6c3e7918e07ff17935284b227a9d2df \\\n    --hash=sha256:0feb2e9d6ad6c9e3c06effe9d00f3f1e618a6643273576b016f591e9315a7139 \\\n    --hash=sha256:10e41f0fbf1eec8cfd234b8fe17a4caac7c9d0db4c204d3c173a8f9f6ef3232b \\\n    --hash=sha256:1182d52bc2d5e5d7d0949503aa7e36d12f42205dc287e4883f407b1988820d39 \\\n    --hash=sha256:164b31cd1a0490ab6efae01aa5df49da7061be0af1b30e035b6e9a1bfe34ee6e \\\n    --hash=sha256:1657923d2d45afb66526e5b933e5b3052e6bdea196c90d3abb2424e18c77dae8 \\\n    --hash=sha256:186941b6aef820ad110fb01fb06eb925374dc3a21b17e37ec9a53b250c6fe2d1 \\\n    --hash=sha256:1cca606cd25738df4ed873d5ad46bbdb3d83b5cbca291f6b4ff13a4df6b0bbe8 \\\n    --hash=sha256:21900ce7ba264168cd50defae43cd75d25c833ad4ad6e73ffc5596d12e25ac89 \\\n    --hash=sha256:236ff70b9312fb68943c703aa842ca6a758abfa45ac187a5e7c1452e96ef72b5 \\\n    --hash=sha256:23aceaa007d6172b02c277f0cd359c79492bbb14f7072b4ede9fbcaf20648130 \\\n    --hash=sha256:23d27a3e0307ec2244cc51e7287b919aa68d097504ebe19df4e76a98a3eea5bd \\\n    --hash=sha256:24870b09b224f7ae3c39ed07d10e819d06f8720bc551847b1d623832b5b0e28d \\\n    --hash=sha256:251bf95b67017e27b13d82f5b326234ca62d70f9cf4c2b9032de2358a3b12c7b \\\n    --hash=sha256:25b9b82bb22e6e2b3cd07b39c68b7b862001226cb3dff7130d1cb914121b39ed \\\n    --hash=sha256:28ce87c5ab450a9dd970b52e5aca5fe63ed432d18a2eaddd1979a00a1ba24ace \\\n    --hash=sha256:300557495eb45ebb8aec96c2da9c4be642fbf7cd937278b4013ba894ea8eb0eb \\\n    --hash=sha256:30f2aa603c41533cc25c05acd0da21636e84a315768feb631c937177db558931 \\\n    --hash=sha256:331b624368d4f1d069149002f25f44bc61c8919ce8ddb3c45bdad8f6e2d89510 \\\n    --hash=sha256:37d6d0a00072fd2948eb22bce7e1475f34569d90c87c59f7a2ec59541b77f7a6 \\\n    --hash=sha256:37dc8f7bbb66efe481bb60defacef820c950c24713fb44962ed6aa2a50966de1 \\\n    --hash=sha256:3b8182a766685eaa002637e28b4ec8d6b18819a0c71f579bf0dbaa5830297cce \\\n    --hash=sha256:3edce1d53195db527e0191f84b71d02022de0540bf43a16ed734ed7537b07385 \\\n    --hash=sha256:446c34dcc4324b084a53b705127dc15717b22c5e140ae0a3c38349d4efec071e \\\n    --hash=sha256:4998562bf62a445225f22e07c896bb04b35b1b1f2eb6d760584c9c51d7a5f78c \\\n    --hash=sha256:4b0a7fe987b14c31ebda6083f74f22b561fd3739bc0ac51e019622e3d72668c7 \\\n    --hash=sha256:4e8c2a84d977f50b9daed6eeaf3baef67d00d5d74d932288f02cb94518ee3ace \\\n    --hash=sha256:4f883547d4b7f0495ebe7056b0cc2aea76094e7a4abc8e933540f3271df27d9c \\\n    --hash=sha256:514435a37670e3e5e08f3945b68718b6ed329bb84367777e16f9f4dfe1e61a0f \\\n    --hash=sha256:53aa02d20d10c3d814d536aa4e5ac9b84ca0ff5a88377963b085ad6822f93e64 \\\n    --hash=sha256:5594fc43d548a7ed94949d139aa1341b270f1863f11cfd37f5a6c8b778a6b67f \\\n    --hash=sha256:571b9fcb07b97ef3a492028fb3d2dc0993ca23a06138b0315286566d29ef718a \\\n    --hash=sha256:57b3d78c95ba9059768b10e28b813002261d3f3dfc55cc48b0c988f625175827 \\\n    --hash=sha256:5afb51d599ea772b8365ae807ae557f18bccfe46ab261fd1c2a9ed700fc6eb17 \\\n    --hash=sha256:6b02afb9b97f65fbca5f31db6a2a3ba21aa93030225f150fa3f249717e938fb4 \\\n    --hash=sha256:6c0016e7b354317c4e9e525b937ac8596c38d2d232b419529b9cd7a1cd46e39a \\\n    --hash=sha256:71d6097b330eea8fd15097780c8e89cb1a8ce7838669f48c5bacd6f663dd4701 \\\n    --hash=sha256:756c768d0c9c2955feb7a56c37ea24aea2e369f8d36a88da270b6a9f19e62b5e \\\n    --hash=sha256:78cb2c6865a35ab8ff8b75fd122f6033b92a62c82801110e48ddd6c936a45d91 \\\n    --hash=sha256:7a743ff716f746fc19a9557f60dab1600d4613255f8a7aeb3cdde4db7eb15a66 \\\n    --hash=sha256:85f998ea1848bc6757289e739cfbdda3a04adfd58b02fc018ce54d754a5ce468 \\\n    --hash=sha256:8728f216dcdb6e6d555cf971cb34076139ad74b31fc2c14da4fafc741c5f6217 \\\n    --hash=sha256:877c3f311ff35410f690861c4409e7ccbf0cd2f878e50628a28e5a0bb689e658 \\\n    --hash=sha256:8cd2f7bdda092d99c9fc2fb7391354f306d01443d22785d0cbfafa2e2c8bb418 \\\n    --hash=sha256:8e95e1385e4998ae9694eeaa4730ba5457ff61185b3a55e2e7bea0880aef452a \\\n    --hash=sha256:962864dc93511324d51ddbb5b9f8731bf71675b93ca612a07441896f4688fb8c \\\n    --hash=sha256:9cf95fe4d0f84c82d282745d9bb08ad9f926efa00be4697e767b814ce40d4330 \\\n    --hash=sha256:9e881fca225083806662a5c43d627d215f258ff43c890f831966c7d7ba9c7402 \\\n    --hash=sha256:a2b55dd6b2a4c4b7d87ffa56bdb33fdc5fdb9a462173861a7bc097f17d91cb09 \\\n    --hash=sha256:a45650e8ce7fafffd731db8550230db6b0d306d181a90b67d3e6bca2f1990930 \\\n    --hash=sha256:a876864214e136f0eb367788dbd7df045f4806801518e2cfe9e13229cfe06d8f \\\n    --hash=sha256:ae26d61dfa7a47befdc7572b521024e8745f3d809bd95ca9505a7bba9ef849ec \\\n    --hash=sha256:af8d94b0db561cf68b88a267c5c44b49e134f525d0dc2cb7ed413a66bc23559a \\\n    --hash=sha256:b343699e8308bdc51978310e1c959c584e7869cc8c40780058c87da7781a1e94 \\\n    --hash=sha256:b3c777e849237620b022f7f297dd67705f9f5cf1685f09f02e46f93e92725468 \\\n    --hash=sha256:b629de27fda84b42cde7edef0d85f13b958b47f6e9bbcbba9b673c562a89bd8b \\\n    --hash=sha256:ba09209fbe443b4acccebe845d8a138b89a8f4fbaeedd44953490b5315d5e965 \\\n    --hash=sha256:ba54cfebe86920a559a7c4d6b9050791c20513650a1952ebe3368c7dc70306f8 \\\n    --hash=sha256:bcb46e2f9feff8d06323983bd83ed00c201fdcab3d74973e7072a889b3979fcd \\\n    --hash=sha256:bcc33feacfaefce60c12fd500a277533bdc02b10a19f7f6d348763d8140bbba7 \\\n    --hash=sha256:bf16ba1b4d0b6b7c8e534936632270cf70eb00dbe09005bc345b2677b726855c \\\n    --hash=sha256:cf1845d02ad822a369a49f2bb9345b1614744267682e7a03527dc3bf6eea1777 \\\n    --hash=sha256:d69141514cc30b774ceea5e3ed3a6635c8d8a96edf664689b890f4089111fb35 \\\n    --hash=sha256:d9c7f76c0673154f044e9d78c8655fb4213f6ca31a836df48b40fe5d187717b9 \\\n    --hash=sha256:dbce0b29841537a2fa4a214c2bbf14de3587c9680caa9b4e217568472490b28f \\\n    --hash=sha256:dc624f6bc473dacdf7ef7eb8678d0d08edf15cd94fad6ae5c7d6cc67a4e4902f \\\n    --hash=sha256:e158cb00350dc278f3b91551101aa7d12415a66ebf2c91d8d5ac14e56ddd3ad0 \\\n    --hash=sha256:e491916b378fba47242221bb9ead245211b70d504f495d105d17b14a24b4907c \\\n    --hash=sha256:e795b7eb908249c4e43c7c99fac7c2c75dab0c43566e37db472a355f63693d71 \\\n    --hash=sha256:e7e480451b9fa137494bccd3a7d69adbe8ac65a87d97be61e11f1b1050a5bac3 \\\n    --hash=sha256:e91206ee562682b51b98ef4b26a6ef48fd84e15fd4c4bc5ec768eb641d206838 \\\n    --hash=sha256:e9871b1ffbfa9656b60aeee92ed5136a5742696006fa322b29ea3d8da0ecc9cf \\\n    --hash=sha256:e9aeb04d6aef139de265b29683e119b638208f88cf73cdd1658aa07221165321 \\\n    --hash=sha256:ebaea975e03d3141d9d3a507df75c9b3ec90fa9d2ffd07567b3a978d9d790b26 \\\n    --hash=sha256:f0606c8bf2cdefea14a43530f7657cbbb7ecf1c4222512492ef4a4434a9501ec \\\n    --hash=sha256:f13c32a3abd6079a66d9526e18dad9b6d280384d49d7c54040cd57b6424041d9 \\\n    --hash=sha256:f7401aebd7f581d7f83a439d87d474999317ee099218e5ad25d125290990ba65 \\\n    --hash=sha256:fa4ecea169a355be7a3ade2c783e2ed12f0e40d2c5621cda8b3297faf7fbb9f5 \\\n    --hash=sha256:fbd139c8447d25dd750ab79ee274cc5e1fe80fc56340ab10b18a195e1b6eca3e \\\n    --hash=sha256:fdafc9cce40277e0f7a0feabce0ee50dd2fa1800f3b38015e51296b5e814048d \\\n    --hash=sha256:fe3cca2e4e8a592be0f269a1ca4835c25199d9f3ce815c8491048f785b0a0198 \\\n    --hash=sha256:ffd0c5368496f41b0944be820fcb7a838aa6e623d250b01acf2643939c3f99d7\n    # via\n    #   -r tools/rice-requirements.in\n    #   matplotlib\n    #   torchvision\nprotobuf==6.33.5 \\\n    --hash=sha256:3093804752167bcab3998bec9f1048baae6e29505adaf1afd14a37bddede533c \\\n    --hash=sha256:69915a973dd0f60f31a08b8318b73eab2bd6a392c79184b3612226b0a3f8ec02 \\\n    --hash=sha256:6ddcac2a081f8b7b9642c09406bc6a4290128fce5f471cddd165960bb9119e5c \\\n    --hash=sha256:8afa18e1d6d20af15b417e728e9f60f3aa108ee76f23c3b2c07a2c3b546d3afd \\\n    --hash=sha256:8f04fa32763dcdb4973d537d6b54e615cc61108c7cb38fe59310c3192d29510a \\\n    --hash=sha256:9b71e0281f36f179d00cbcb119cb19dec4d14a81393e5ea220f64b286173e190 \\\n    --hash=sha256:a3157e62729aafb8df6da2c03aa5c0937c7266c626ce11a278b6eb7963c4e37c \\\n    --hash=sha256:a5cb85982d95d906df1e2210e58f8e4f1e3cdc088e52c921a041f9c9a0386de5 \\\n    --hash=sha256:cbf16ba3350fb7b889fca858fb215967792dc125b35c7976ca4818bee3521cf0 \\\n    --hash=sha256:d71b040839446bac0f4d162e758bea99c8251161dae9d0983a3b88dee345153b\n    # via -r tools/rice-requirements.in\npycocotools==2.0.11 \\\n    --hash=sha256:04480330df5013f6edd94891a0ee8294274185f1b5093d1b0f23d51778f0c0e9 \\\n    --hash=sha256:08c79789fd79e801ae4ecfcfeec32b31e36254e7a2b4019af28c104975d5e730 \\\n    --hash=sha256:1192de413a23b4b94199197e8f8dbe1277cc24e6e9847bee6a71be3d8e543963 \\\n    --hash=sha256:18ba75ff58cedb33a85ce2c18f1452f1fe20c9dd59925eec5300b2bf6205dbe1 \\\n    --hash=sha256:34254d76da85576fcaf5c1f3aa9aae16b8cb15418334ba4283b800796bd1993d \\\n    --hash=sha256:368244f30eb8d6cae7003aa2c0831fbdf0153664a32859ec7fbceea52bfb6878 \\\n    --hash=sha256:484d33515353186aadba9e2a290d81b107275cdb9565084e31a5568a52a0b120 \\\n    --hash=sha256:4fc9889e819452b9c142036e1eabac8a13a8bd552d8beba299a57e0da6bfa1ec \\\n    --hash=sha256:63026e11a56211058d0e84e8263f74cbccd5e786fac18d83fd221ecb9819fcc7 \\\n    --hash=sha256:693417797f0377fd094eb815c0a1e7d1c3c0251b71e3b3779fce3b3cf24793c5 \\\n    --hash=sha256:6a2f4f036f5bfdaf8c9625279051e921721fc9d27f92c97a3f3355a07ba38513 \\\n    --hash=sha256:78bae4a9de9d34c4759754a848dfb3306f9ef1c2fcb12164ffbd3d013d008321 \\\n    --hash=sha256:7cd4cdfd2c676f30838aa0b1047441892fb4f97d70bf3df480bcc7a18a64d7d4 \\\n    --hash=sha256:7fd4121766cc057133534679c0ec3f9023dbd96e9b31cf95c86a069ebdac2b65 \\\n    --hash=sha256:81bdceebb4c64e9265213e2d733808a12f9c18dfb14457323cc6b9af07fa0e61 \\\n    --hash=sha256:83d896f4310379849dfcfa7893afb0ff21f4f3cdb04ab3f61b05dd98953dd0ad \\\n    --hash=sha256:87af87b8d06d5b852a885a319d9362dca3bed9f8bbcc3feb6513acb1f88ea242 \\\n    --hash=sha256:89e853425018e2c2920ee0f2112cf7c140a1dcf5f4f49abd9c2da112c3e0f4b3 \\\n    --hash=sha256:8cedb8ccb97ffe9ed2c8c259234fa69f4f1e8665afe3a02caf93f6ef2952c07f \\\n    --hash=sha256:8e159232adae3aef6b4e2d37b008bff107b26e9ed3b48e70ea6482302834bd34 \\\n    --hash=sha256:9dc8b388984c72aa84b1a68933f196ce71ab114c59232d0eab20c97cc1300875 \\\n    --hash=sha256:a1c05f91ccc658dfe01325267209c4b435da1722c93eeb5749fabc1d087b6882 \\\n    --hash=sha256:a2b018497ec198ffc737dd7e6306a2e69999779ca619a9e12950e4506e410c3e \\\n    --hash=sha256:a2e9634bc7cadfb01c88e0b98589aaf0bd12983c7927bde93f19c0103e5441f4 \\\n    --hash=sha256:a6b13baf6bfcf881b6d6ac6e23c776f87a68304cd86e53d1d6b9afa31e363c4e \\\n    --hash=sha256:a82d1c9ed83f75da0b3f244f2a3cf559351a283307bd9b79a4ee2b93ab3231dd \\\n    --hash=sha256:ac8aa17263e6489aa521f9fa91e959dfe0ea3a5519fde2cbf547312cdce7559e \\\n    --hash=sha256:b6a07071c441d0f5e480a8f287106191582e40289d4e242dfe684e0c8a751088 \\\n    --hash=sha256:bd7a1e19ef56a828a94bace673372071d334a9232cd32ae3cd48845a04d45c4f \\\n    --hash=sha256:c230f5e7b14bd19085217b4f40bba81bf14a182b150b8e9fab1c15d504ade343 \\\n    --hash=sha256:c3546b93b39943347c4f5b0694b5824105cbe2174098a416bcad4acd9c21e957 \\\n    --hash=sha256:ca9f120f719ec405ad0c74ccfdb8402b0c37bd5f88ab5b6482a0de2efd5a36f4 \\\n    --hash=sha256:d6b88557166794a24acd03f50296f2b95adf3e4206b4b8995e6bdca925c9cb72 \\\n    --hash=sha256:e21311ea71f85591680d8992858e2d44a2a156dc3b2bf1c5c901c4a19348177b \\\n    --hash=sha256:e40a3a898c6e5340b8d70cf7984868b9bff8c3d80187de9a3b661d504d665978 \\\n    --hash=sha256:e72527fdc00985d29dbe31d17b19cd2d16fbad7b01e974c567b593d5844de710 \\\n    --hash=sha256:eba35d6e06caaea28ce65a4b92e4343ecbb9abf4915f7ef7fca989b80839111c \\\n    --hash=sha256:eebd723503a2eb2c8b285f56ea3be1d9f3875cd7c40d945358a428db94f14015 \\\n    --hash=sha256:efd1694b2075f2f10c5828f10f6e6c4e44368841fd07dae385c3aa015c8e25f9 \\\n    --hash=sha256:f78cbb1a32d061fcad4bdba083de70a39a21c1c3d9235a3f77d8f007541ec5ef \\\n    --hash=sha256:f7eb43b79448476b094240450420b7425d06e297880144b8ea6f01e9b4340e43 \\\n    --hash=sha256:fd72b9734e6084b217c1fc3945bfd4ec05bdc75a44e4f0c461a91442bb804973 \\\n    --hash=sha256:ffe806ce535f5996445188f9a35643791dc54beabc61bd81e2b03367356d604f\n    # via -r tools/rice-requirements.in\npyparsing==3.3.3 \\\n    --hash=sha256:928ae7e20211f3b6f3915a72f06a0cfd29ab9d24279dd6346b6b1a7146397d36 \\\n    --hash=sha256:ece8c00a69cf01b45d0b1dedabb469c90d8caf996d4fda40f147627a122849a4\n    # via matplotlib\npython-dateutil==2.9.0.post0 \\\n    --hash=sha256:37dd54208da7e1cd875388217d5e00ebd4179249f90fb72437e91a35459a0ad3 \\\n    --hash=sha256:a8b2bc7bffae282281c8140a97d3aa9c14da0b136dfe83f850eea9a5f7470427\n    # via matplotlib\npyyaml==6.0.3 \\\n    --hash=sha256:00c4bdeba853cc34e7dd471f16b4114f4162dc03e6b7afcc2128711f0eca823c \\\n    --hash=sha256:0150219816b6a1fa26fb4699fb7daa9caf09eb1999f3b70fb6e786805e80375a \\\n    --hash=sha256:02893d100e99e03eda1c8fd5c441d8c60103fd175728e23e431db1b589cf5ab3 \\\n    --hash=sha256:02ea2dfa234451bbb8772601d7b8e426c2bfa197136796224e50e35a78777956 \\\n    --hash=sha256:0f29edc409a6392443abf94b9cf89ce99889a1dd5376d94316ae5145dfedd5d6 \\\n    --hash=sha256:10892704fc220243f5305762e276552a0395f7beb4dbf9b14ec8fd43b57f126c \\\n    --hash=sha256:16249ee61e95f858e83976573de0f5b2893b3677ba71c9dd36b9cf8be9ac6d65 \\\n    --hash=sha256:1d37d57ad971609cf3c53ba6a7e365e40660e3be0e5175fa9f2365a379d6095a \\\n    --hash=sha256:1ebe39cb5fc479422b83de611d14e2c0d3bb2a18bbcb01f229ab3cfbd8fee7a0 \\\n    --hash=sha256:214ed4befebe12df36bcc8bc2b64b396ca31be9304b8f59e25c11cf94a4c033b \\\n    --hash=sha256:2283a07e2c21a2aa78d9c4442724ec1eb15f5e42a723b99cb3d822d48f5f7ad1 \\\n    --hash=sha256:22ba7cfcad58ef3ecddc7ed1db3409af68d023b7f940da23c6c2a1890976eda6 \\\n    --hash=sha256:27c0abcb4a5dac13684a37f76e701e054692a9b2d3064b70f5e4eb54810553d7 \\\n    --hash=sha256:28c8d926f98f432f88adc23edf2e6d4921ac26fb084b028c733d01868d19007e \\\n    --hash=sha256:2e71d11abed7344e42a8849600193d15b6def118602c4c176f748e4583246007 \\\n    --hash=sha256:34d5fcd24b8445fadc33f9cf348c1047101756fd760b4dacb5c3e99755703310 \\\n    --hash=sha256:37503bfbfc9d2c40b344d06b2199cf0e96e97957ab1c1b546fd4f87e53e5d3e4 \\\n    --hash=sha256:3c5677e12444c15717b902a5798264fa7909e41153cdf9ef7ad571b704a63dd9 \\\n    --hash=sha256:3ff07ec89bae51176c0549bc4c63aa6202991da2d9a6129d7aef7f1407d3f295 \\\n    --hash=sha256:41715c910c881bc081f1e8872880d3c650acf13dfa8214bad49ed4cede7c34ea \\\n    --hash=sha256:418cf3f2111bc80e0933b2cd8cd04f286338bb88bdc7bc8e6dd775ebde60b5e0 \\\n    --hash=sha256:44edc647873928551a01e7a563d7452ccdebee747728c1080d881d68af7b997e \\\n    --hash=sha256:4a2e8cebe2ff6ab7d1050ecd59c25d4c8bd7e6f400f5f82b96557ac0abafd0ac \\\n    --hash=sha256:4ad1906908f2f5ae4e5a8ddfce73c320c2a1429ec52eafd27138b7f1cbe341c9 \\\n    --hash=sha256:501a031947e3a9025ed4405a168e6ef5ae3126c59f90ce0cd6f2bfc477be31b7 \\\n    --hash=sha256:5190d403f121660ce8d1d2c1bb2ef1bd05b5f68533fc5c2ea899bd15f4399b35 \\\n    --hash=sha256:5498cd1645aa724a7c71c8f378eb29ebe23da2fc0d7a08071d89469bf1d2defb \\\n    --hash=sha256:5cf4e27da7e3fbed4d6c3d8e797387aaad68102272f8f9752883bc32d61cb87b \\\n    --hash=sha256:5e0b74767e5f8c593e8c9b5912019159ed0533c70051e9cce3e8b6aa699fcd69 \\\n    --hash=sha256:5ed875a24292240029e4483f9d4a4b8a1ae08843b9c54f43fcc11e404532a8a5 \\\n    --hash=sha256:5fcd34e47f6e0b794d17de1b4ff496c00986e1c83f7ab2fb8fcfe9616ff7477b \\\n    --hash=sha256:5fdec68f91a0c6739b380c83b951e2c72ac0197ace422360e6d5a959d8d97b2c \\\n    --hash=sha256:6344df0d5755a2c9a276d4473ae6b90647e216ab4757f8426893b5dd2ac3f369 \\\n    --hash=sha256:64386e5e707d03a7e172c0701abfb7e10f0fb753ee1d773128192742712a98fd \\\n    --hash=sha256:652cb6edd41e718550aad172851962662ff2681490a8a711af6a4d288dd96824 \\\n    --hash=sha256:66291b10affd76d76f54fad28e22e51719ef9ba22b29e1d7d03d6777a9174198 \\\n    --hash=sha256:66e1674c3ef6f541c35191caae2d429b967b99e02040f5ba928632d9a7f0f065 \\\n    --hash=sha256:6adc77889b628398debc7b65c073bcb99c4a0237b248cacaf3fe8a557563ef6c \\\n    --hash=sha256:79005a0d97d5ddabfeeea4cf676af11e647e41d81c9a7722a193022accdb6b7c \\\n    --hash=sha256:7c6610def4f163542a622a73fb39f534f8c101d690126992300bf3207eab9764 \\\n    --hash=sha256:7f047e29dcae44602496db43be01ad42fc6f1cc0d8cd6c83d342306c32270196 \\\n    --hash=sha256:8098f252adfa6c80ab48096053f512f2321f0b998f98150cea9bd23d83e1467b \\\n    --hash=sha256:850774a7879607d3a6f50d36d04f00ee69e7fc816450e5f7e58d7f17f1ae5c00 \\\n    --hash=sha256:8d1fab6bb153a416f9aeb4b8763bc0f22a5586065f86f7664fc23339fc1c1fac \\\n    --hash=sha256:8da9669d359f02c0b91ccc01cac4a67f16afec0dac22c2ad09f46bee0697eba8 \\\n    --hash=sha256:8dc52c23056b9ddd46818a57b78404882310fb473d63f17b07d5c40421e47f8e \\\n    --hash=sha256:9149cad251584d5fb4981be1ecde53a1ca46c891a79788c0df828d2f166bda28 \\\n    --hash=sha256:93dda82c9c22deb0a405ea4dc5f2d0cda384168e466364dec6255b293923b2f3 \\\n    --hash=sha256:96b533f0e99f6579b3d4d4995707cf36df9100d67e0c8303a0c55b27b5f99bc5 \\\n    --hash=sha256:9c57bb8c96f6d1808c030b1687b9b5fb476abaa47f0db9c0101f5e9f394e97f4 \\\n    --hash=sha256:9c7708761fccb9397fe64bbc0395abcae8c4bf7b0eac081e12b809bf47700d0b \\\n    --hash=sha256:9f3bfb4965eb874431221a3ff3fdcddc7e74e3b07799e0e84ca4a0f867d449bf \\\n    --hash=sha256:a33284e20b78bd4a18c8c2282d549d10bc8408a2a7ff57653c0cf0b9be0afce5 \\\n    --hash=sha256:a80cb027f6b349846a3bf6d73b5e95e782175e52f22108cfa17876aaeff93702 \\\n    --hash=sha256:b30236e45cf30d2b8e7b3e85881719e98507abed1011bf463a8fa23e9c3e98a8 \\\n    --hash=sha256:b3bc83488de33889877a0f2543ade9f70c67d66d9ebb4ac959502e12de895788 \\\n    --hash=sha256:b865addae83924361678b652338317d1bd7e79b1f4596f96b96c77a5a34b34da \\\n    --hash=sha256:b8bb0864c5a28024fac8a632c443c87c5aa6f215c0b126c449ae1a150412f31d \\\n    --hash=sha256:ba1cc08a7ccde2d2ec775841541641e4548226580ab850948cbfda66a1befcdc \\\n    --hash=sha256:bdb2c67c6c1390b63c6ff89f210c8fd09d9a1217a465701eac7316313c915e4c \\\n    --hash=sha256:c1ff362665ae507275af2853520967820d9124984e0f7466736aea23d8611fba \\\n    --hash=sha256:c2514fceb77bc5e7a2f7adfaa1feb2fb311607c9cb518dbc378688ec73d8292f \\\n    --hash=sha256:c3355370a2c156cffb25e876646f149d5d68f5e0a3ce86a5084dd0b64a994917 \\\n    --hash=sha256:c458b6d084f9b935061bc36216e8a69a7e293a2f1e68bf956dcd9e6cbcd143f5 \\\n    --hash=sha256:d0eae10f8159e8fdad514efdc92d74fd8d682c933a6dd088030f3834bc8e6b26 \\\n    --hash=sha256:d76623373421df22fb4cf8817020cbb7ef15c725b9d5e45f17e189bfc384190f \\\n    --hash=sha256:ebc55a14a21cb14062aa4162f906cd962b28e2e9ea38f9b4391244cd8de4ae0b \\\n    --hash=sha256:eda16858a3cab07b80edaf74336ece1f986ba330fdb8ee0d6c0d68fe82bc96be \\\n    --hash=sha256:ee2922902c45ae8ccada2c5b501ab86c36525b883eff4255313a253a3160861c \\\n    --hash=sha256:efd7b85f94a6f21e4932043973a7ba2613b059c4a000551892ac9f1d11f5baf3 \\\n    --hash=sha256:f7057c9a337546edc7973c0d3ba84ddcdf0daa14533c2065749c9075001090e6 \\\n    --hash=sha256:fa160448684b4e94d80416c0fa4aac48967a969efe22931448d853ada8baf926 \\\n    --hash=sha256:fc09d0aa354569bc501d4e787133afc08552722d3ab34836a80547331bb5d4a0\n    # via\n    #   huggingface-hub\n    #   timm\n    #   transformers\nregex==2026.9.10 \\\n    --hash=sha256:030fa9e23624e39b3b94e46b90a5abd1a1678eb2f58fcdd3fd6c27526bf91c7e \\\n    --hash=sha256:032da15431c890d376f53547f0a6219f4f4cd19f3e4f11bdc321453b5bd207e4 \\\n    --hash=sha256:044bd4639b6bb409ec9e5d8b7accd57e02b4c4a4e2eafde916f8ae8006b3e40b \\\n    --hash=sha256:048a89ee797db10160bd2bd519286577a6b43a100279bd4b7d8456a3d69c80a0 \\\n    --hash=sha256:05fb018cfe7144585fc83882405906ff84994a2d154afc2509ecc7752c51f864 \\\n    --hash=sha256:07b45ba5c94b8fcb30cb6c56a11f715c57533a3017964504322ea52690a27b72 \\\n    --hash=sha256:0aa7589394230e0f0a422ab6b90841ff12c87e855e7aaf75d192a54a5f124548 \\\n    --hash=sha256:0acee94b480dd853e39434aa9a575f95385b1b4b8fa3feae56db363ca5cad782 \\\n    --hash=sha256:0b9ba3b2765cdfe18f0f561a69f78a69701f2896654a81c711108d35d14e5099 \\\n    --hash=sha256:0c32480f3371b75068decaf9e5da72c224e953830dd71e36e06cf80e30ea39d8 \\\n    --hash=sha256:1270cdec69248592bbe38a0b263ed58d907b891bd2b93703e225c317e421bda1 \\\n    --hash=sha256:13c52fc377792675f604a207a2ae5958c080f6854f7698d40d9ff034d95b1e76 \\\n    --hash=sha256:14caa05ce39ec70437af5aac8814c50ee6628f4a90353871c059692f448a164f \\\n    --hash=sha256:1562aabd9d4eb09bd88a62ad97ed06800094b529ac43419e43020b9cefec79b0 \\\n    --hash=sha256:175cf49ce7a994c88b8f15e3cb17cdb66a48ebb2d36de736b8205033db950f89 \\\n    --hash=sha256:1aa309ab7ba89a62d6cf70dbd38d4176440bce3c7001ab86256704cf4c18c6eb \\\n    --hash=sha256:1ad10a135fa0b4e4a462a61d07c6654d7518cfdb5cb8da08f9ff7d61384af1fe \\\n    --hash=sha256:1b891f77554bff991804cee24b78b40789f7d5993a24c7907bc7025fd2a70c8d \\\n    --hash=sha256:1e321e2c84f0e52c457f5ea5944f796d6e8e09cb99738ea98dcc1bfe402a128d \\\n    --hash=sha256:1e954e246466d5a1a78f563ce8364b5d7cb19e7adb0ccdec8f9c9610083187bc \\\n    --hash=sha256:1f0a8b4928823bc8b217a1ab7bf3d90598909dec9a70fbbfe9a52cc4eca55990 \\\n    --hash=sha256:1fbc8314436353e097c050e11b01a6c11433579437ed0579730157676ef59e2f \\\n    --hash=sha256:20e8bfb07ad79a282f8b95b56fe67f9750b1b7f775724e4ba1f23cb296115ce4 \\\n    --hash=sha256:217e98ba5fc8908ed8ffd4ebac04753a0c831067cbfb495b9821b94cc61eaa76 \\\n    --hash=sha256:239620b0e0681669367c0e218c8eb2551d9f8fe3b9fccfc8d0003377804e8348 \\\n    --hash=sha256:23ac9a28180f274d7dd7651fa131ad5b02d343b75df4b040737f0356223895dd \\\n    --hash=sha256:2479171edccced52ef02b899558f88ab2c235fe05b93180fdcae1670aacd89e1 \\\n    --hash=sha256:24d12a625a37c89c2b09303402a06942f55f071b95a7916a49c17034c3d47cd5 \\\n    --hash=sha256:2dd9286093c71afc8f55ef035c5b9d2776641fd72c6535f1febc92d0b0be9666 \\\n    --hash=sha256:2e67f8843f0e4b931f1fa860bf3bbe4134b714c0155cc5c7c0d7ea450230aae0 \\\n    --hash=sha256:31e4df2b11d48f61d511019bc1ee9b477055f17c352b68fe72db7a98b14d603c \\\n    --hash=sha256:3264132d576847ab5f88bb83e7debe67854bf165b3ea613bd467312b6099536a \\\n    --hash=sha256:3540734dbe241ebb3b87d5713781f6749a3e4d45480f506aa5fb5cbb0c37d249 \\\n    --hash=sha256:35ba3bab0c45079735f55ac61526774de1d84bc4a0333cc554e1a4ab74913924 \\\n    --hash=sha256:3a66e40a1a20de96a2fee00ed67e11012b62d85b277688258677fd19997addb7 \\\n    --hash=sha256:3bdeed3318a8eb2bbadc9c56347e0ff651639e934a47e168d05a3b12929fd0e7 \\\n    --hash=sha256:3fb4ae8cf83ef4e9addd43b2da31a9f45be816a8036fae8af59c8998b72718e2 \\\n    --hash=sha256:4971776b4f2bd7fd9a83eceb2cb2592cbe2924f639fe8045e6a9de5ba4bfcf25 \\\n    --hash=sha256:4a761ea45f2ad74c575ef5850ea514cef97302a552d3c7c9d1a1a870d4661d6c \\\n    --hash=sha256:4c66d54042a14a503907d81861b8a5235e6d1f03d4fbc1d8767f652eaf957ac1 \\\n    --hash=sha256:4db7d00c4afbfbb55b8e17b1e371da11418ea9389b030acec63c1fa4c7ad4b86 \\\n    --hash=sha256:4f0407474ffac8e5e89d93ca41d60891e29f0ab8423eb66ff292d850a86a0843 \\\n    --hash=sha256:53e182b6b04d0011909b47d51a2d72d908de07c7b1c7f16b3adda2204d723bc1 \\\n    --hash=sha256:5847e22bbf959764d776937d791d034cc2d19b787e361c88d97e859e8dc68502 \\\n    --hash=sha256:58c01f7b81079cf0817ba831ff4d9eff5d28be4a3ac76c353e6f09bd63f4c386 \\\n    --hash=sha256:58da726d3e766c0b3f5a3997dfaf0275898a1107b8191cdd6b0437fe45fd817d \\\n    --hash=sha256:5bef622850cf760154719d4e0d74b0a855962432995168e250069899ae12fe8f \\\n    --hash=sha256:5ccd139b2061132e7b265cfb4b4721baeb9f8928b81415304abf1ec7e3181c26 \\\n    --hash=sha256:5cef9f3d14796500ea834c41dbe688f1f6b23c7024dc23e8a794d7ebaf5d71d0 \\\n    --hash=sha256:63bb62cf62217dc38c8a6b2b61b165b0e4eb8fa93b0aba12139251c0986a8fa3 \\\n    --hash=sha256:681ed38664b64c6617d3c3c332018d1948c77e139c5ea667c1886efa671e426f \\\n    --hash=sha256:6888065672b341e5246f391ec16dc258a29218ac784172fd67c30d941544755b \\\n    --hash=sha256:6aebdd9a946de328b3f6f61dbf48dd064a36eb6dddf96e34ae6651d37f6e9383 \\\n    --hash=sha256:6afcad14310f1311d077553ed374b42a5e538f85a8c884b4e38e52de091c8077 \\\n    --hash=sha256:6b34a778c695d24e77c140e3b4c95da69282e34f2f6b02b55656aa4a0379f643 \\\n    --hash=sha256:6fd555fc9abef50c530869690b2daca054c8811a7aff632d11f9a7b2590b2742 \\\n    --hash=sha256:71879292c9c7ac67b1680345b16daba1be937cb027362cfa04e68f65db2dcfdd \\\n    --hash=sha256:75242f44a3e283106077be4ab717bc535e4701c9d54ad69e195945c22f137a1d \\\n    --hash=sha256:75aa39d3f4f1650eea84e46b0d8cefe77dd5478c10e3d0aaf0b0f00493475a7a \\\n    --hash=sha256:75f9297b16fcb588a1f8d8a55dabef3c0c20b0c7bac43c87ceaaaf1a825c12f4 \\\n    --hash=sha256:79e9432995e14c749d34209413de5e621ec8e67789bf4f46dbfabea9d06a2406 \\\n    --hash=sha256:7abb38b8c40f3a235235a44da452c64b7b5c1d650ec6351027db0e090804f2e5 \\\n    --hash=sha256:7dcad477c49c4c626a6c4fcd71b39a971aa217060cc40a6569fd24edcc0fa509 \\\n    --hash=sha256:7e6c0b5ec6ddee4032247585dc491b0fa58627745b66a705728703a3f0331231 \\\n    --hash=sha256:7f8f10015866608fe4c043cec2e4fe4c39a94bb50e45091de4cdf4004b9ae4b0 \\\n    --hash=sha256:866de9f98df0611d7b62b3a8729d3284a64c0cc6edd90bb95a533e443a4939cb \\\n    --hash=sha256:87f5f75c109f08f5c602d68e1af54cead8165189c727b6ac946b30b9833a3ba4 \\\n    --hash=sha256:880ac684c27176464c00c3fdc456116364f5ebc70da07aad0c2d4a7ba45e98db \\\n    --hash=sha256:88b02aa8d0ec9b6189fe933d425775882271c23700ac11fd26d1779b0f56fde3 \\\n    --hash=sha256:8ba1f78bd4fef2d8f84b894ec28ac3481afe6cc07aaa253ad4717ef7b3fe6bcb \\\n    --hash=sha256:8c07021a4faa3f092869adbd1f35cdc7a592276c807aeebc3ceb8ff1a638f0b4 \\\n    --hash=sha256:8d5c4518235a2ec1611e57af85fa488d529c1106aacff12adadcedf8687012cd \\\n    --hash=sha256:8e127d9a80cbf1c3276bb465c6d047e8705e97b58c2b8f2f0c0a69c336b44b37 \\\n    --hash=sha256:94c5ce3bc41d226b4eb89ca3f842b2e28c031487fb1f34eb2153d98235831325 \\\n    --hash=sha256:94d096369b7cd96d15343fef5257fe39eff9d0e8758b92a0e15e358b92cdb2fc \\\n    --hash=sha256:968c1e33edd9a104d1bf24c8d476c72de7e3839ae7f894b37e9e4f4739fdeeca \\\n    --hash=sha256:990797e765d89a423880052c68b61c31afe701de94a8c060f61c40605ca6c727 \\\n    --hash=sha256:9ce239acb15843ab03976626af810a4424b0409689ec2bbc52088ab5479ab487 \\\n    --hash=sha256:9d772586951d7d6a5d162d48f414065e483b1c81ab38fd8ed97c78b05883421a \\\n    --hash=sha256:9fbd2e5d8002dc49a6129fb321ec51c57a025e752ed525ddce0ba9223c4350a7 \\\n    --hash=sha256:a41693eb3fc4b92e6127d113813c6c395237f7edd3224abf67609af48c690d11 \\\n    --hash=sha256:abbfc1c33bf8efddcc43844aba61e036d74a918680dc3ce8ce2538b004eda0f9 \\\n    --hash=sha256:b298cdc33c5cc6969ff07f0fba19cc73e0fd8576373c50935feadaca2f6b4405 \\\n    --hash=sha256:b43456de605c8ee77eb75f07bc1ee44ba27f9cee22207deb77d495e954b7d953 \\\n    --hash=sha256:b71649169a9fcf30b395ee01047fa7ad6654a4c900ca75b23c04dedcce6a1f8c \\\n    --hash=sha256:b91c37551bf39d75116c02b146956f65b9aa0337a4a652f4ae186983789d4001 \\\n    --hash=sha256:b9d36b03dc362aa40ffaaec9d9bd75e87763529563ec008c43b0e07782f5be7a \\\n    --hash=sha256:bafa41b0dd63669e5c0f8adf3d24819efeb73c847f492eb011212eb352e69041 \\\n    --hash=sha256:bb7774924f8cd69f49cba0b3c2d679a6326f777e0e67d130ad5203e4df53f0d3 \\\n    --hash=sha256:bf29611e5376fec8f795879bb5c6153a76c3a292573d173c26784042b01eb840 \\\n    --hash=sha256:c014641157e9049b0603b8daa5343bd408d9b757b709aaa0f373cd3fab2d7944 \\\n    --hash=sha256:c103b3b14e011774af4fb7e4617ad4d72b9171905cd3b231a70a4efd76e477d7 \\\n    --hash=sha256:c22df8dd6373bbe3898e77429ffc85594300e39d752fd0e68a31e59d37899376 \\\n    --hash=sha256:c25a754bb81a2edcfc3b65eda50f017d736f818112ed43e8aafd595cb00678ae \\\n    --hash=sha256:c32818b28bcd153b25b63038348a9fe9b9fbcddb60df43f204c3ab55eeb57f77 \\\n    --hash=sha256:c37fa93bf18bf4f90b01c0fa9f11ea567ee4b7dd8bf96e63663e5edc37aa38cf \\\n    --hash=sha256:c3d95d7d9538b5b726dd6fcd7b6117a71e6565202f6d64f5845fb4d8f203f533 \\\n    --hash=sha256:c8fbd9cb30c68c1686b94029b9ef845d5870d3d65baf66cb126b676849b9d72b \\\n    --hash=sha256:cb76a9c4e07a6a47849726af0ed14c41741a182f097f134a8cf29c1bc0f4dde8 \\\n    --hash=sha256:ce7c118cb102975f974585688357a717ffbf9dddd64ab0bb1bc93eb5b367cf95 \\\n    --hash=sha256:cf377960d2ac37d987394a9dbaa75e91338c41a46d41e1d25e90125e7b3ee2dc \\\n    --hash=sha256:d278ad30ec83b6b9202685b0f80b741a51ea3ca7f0595ebda96e7628b6398876 \\\n    --hash=sha256:d2d377fd1cad611b806cdd732d86b65f536c768209890cb442556548daa65a23 \\\n    --hash=sha256:d414c411c06fe0009eac33488fb1591c66b5c2673e342e452e7bb2fe63da8194 \\\n    --hash=sha256:d8c668af8f7bdb1d18739c27d30cd9f4b371495a883f75a002fb7a39d740fecd \\\n    --hash=sha256:dce932f8e3ba936475ea3d0d8b59f7b050a9e206e994f53f8fd80299871e87da \\\n    --hash=sha256:debc629e98b95abaea1cf3057ca296151f348c697c9b8a59d18013adb302c0dd \\\n    --hash=sha256:e0dc78251154b66dc60211563fc115345da332eaa881e4e2523fb1edae3772f4 \\\n    --hash=sha256:e5e4a6e0734a685d13b9685622bb503bdbb2927f8b0df025a5085f0ea067475b \\\n    --hash=sha256:e6b99181d184d0f5c7b36b8d12b94d1e9499cce6246594331f9edc5d2ea9fceb \\\n    --hash=sha256:e7327795089ddb44912dce1434e1d7244be2e9fb48fcc2d6782936af7a3062db \\\n    --hash=sha256:ebb2ba68e4641a994061f70bf44ed448fba0b9b1d18c94ffb9efc1cca805b39b \\\n    --hash=sha256:ec8855f08c17895a26fbf5f19ed829722e19b34a96629e49a43c92974924026b \\\n    --hash=sha256:ecb2e7acb18f8cc4a67f0ad986c0af291ea4dd385d0614ba9bc09d7f8bbb478c \\\n    --hash=sha256:ef4c0a9dfdc90581b90b1b95a8c3d1557f8ff8f5a2a53536d26314de699d1468 \\\n    --hash=sha256:ef4ce69ff97fbb44b46751cfea5e859ad0b66d1a50abf34954f0645f51e81671 \\\n    --hash=sha256:ef5a059ea1c6ee5d1c7e99a2484e628608d010921efe876c6f0e2029d2f35eca \\\n    --hash=sha256:f0e2e5d23448b660d60a6ed85c46cc03b4b48bd276b8f4041d4a5fe2a4a0626b \\\n    --hash=sha256:f2374c27deb189b282ec7e16106752c22ad39b056bbd8018960b1e4cc95d67a1 \\\n    --hash=sha256:f2f43bf4e47ff7ce9e585558706d698c6204d0f80bf2207766382ed817c8e9f4 \\\n    --hash=sha256:f5c629df03adec31ee505dda3c8988f106c9390e4cbd343600036eb8b3d6724f \\\n    --hash=sha256:f70b9f0e39c2dba1d9da6bf7ef7c377cad7277f8440e9a69be05ede529ff024c \\\n    --hash=sha256:f7d4656e17ab736e9415a6442a345bfc97bb8b7dcce47884bb74a37f70f08d0c \\\n    --hash=sha256:f8bdec659a8fa7af51a32b224b3b7c02bc415d54ffd35187b1d224176b17d607 \\\n    --hash=sha256:faa911fbbcf8ac90bda0e0657d60768e3390954ef0588211d63a22add1cb1cd1 \\\n    --hash=sha256:fbc4e2f3cb7ce8436154e6483079e7d35eeb321a952fa936e180300630d8b873 \\\n    --hash=sha256:fd6bd89b9fc06018d35851cab0240adb7dd84d51941b19f6574ac90cd54e3ae5 \\\n    --hash=sha256:ff4d7b14ea19e50c8d9d6d83f45bd9b45cbb624c07ac1fa54db0a019049abed7 \\\n    --hash=sha256:ff6b3267318661dfddf6b3628663e00e5946bd0a5c8fa678537a1401f0388f91 \\\n    --hash=sha256:ffc2da104e43db716ce30cef9f28049a1faa6aca385dd8771b033268d0730b07\n    # via\n    #   -r tools/rice-requirements.in\n    #   open-clip-torch\n    #   transformers\nrequests==2.34.2 \\\n    --hash=sha256:2a0d60c172f83ac6ab31e4554906c0f3b3588d37b5cb939b1c061f4907e278e0 \\\n    --hash=sha256:f288924cae4e29463698d6d60bc6a4da69c89185ad1e0bcc4104f584e960b9ed\n    # via\n    #   huggingface-hub\n    #   transformers\nsafetensors==0.8.0 \\\n    --hash=sha256:040070828e36dc8e122178bbbd5830ff9e97920affb84cbe0f46442497bed358 \\\n    --hash=sha256:096ec1a98435df7beb08853bb5aa9081a84f23d0adc67ed1a0a10550f608373f \\\n    --hash=sha256:2ddf52eac562eda224f99acfa7889d02968c1fd59a5b011ae7d8137c37e9c02d \\\n    --hash=sha256:3ae091f16662658bdc019a4ff6cb4c085bb7d725eb5978b183ffd265863b6d2d \\\n    --hash=sha256:4124502b78f03534117c848f87a39b8f31e577b15eff423bf8bfb95f2a8c30d0 \\\n    --hash=sha256:4a95ae2b05d7726d751da4ebf626a2ca782b706e101bd894c95bc2450b1cffcc \\\n    --hash=sha256:7a46e5ff292c356d6991e60942ba7f79817682d3a2cef0702136448cb9c4d235 \\\n    --hash=sha256:7bc0a787ba8a35be368ee3574edfa2b1ad389eebd0a72e482ae275490e3f6c98 \\\n    --hash=sha256:87eec7ffed2b809f05a398a8becb7d013f19f7837cd15d9748580d6cf30dbaf4 \\\n    --hash=sha256:8e080062fcde23be189565e1c3305d16751a218ecf9412c8601e64204eb6f846 \\\n    --hash=sha256:8e9f537aa183a38ace122d27303dcd986b26bd2a7591f9181d7f0c396f4677ca \\\n    --hash=sha256:c554f85858e05226d3c2828e32395e677434685d6d94594a41643361c5e837f0 \\\n    --hash=sha256:c80201d22cbf405b80647a60ada77bba06c8fba2da2743ba1e89cdcc39a81f25 \\\n    --hash=sha256:f7838e5135a406ad3e02efdcb8cf2e5397d368b0154537c4fec682dbc544d452 \\\n    --hash=sha256:fabaf3e0f18a6618d9b36560682562157f77c2b71fcffc7b432be2baed9d753d \\\n    --hash=sha256:fcdd41ec4628fee5799f807c73c353629130fbd942aa23d83c623dd6c9d52d78 \\\n    --hash=sha256:fd6f3f93c9a0a7cc2788ee63fb763353d4bd2e89b0751bc78fcf7dda00bea774\n    # via\n    #   -r tools/rice-requirements.in\n    #   open-clip-torch\n    #   timm\n    #   transformers\nscikit-learn==1.8.0 \\\n    --hash=sha256:00d6f1d66fbcf4eba6e356e1420d33cc06c70a45bb1363cd6f6a8e4ebbbdece2 \\\n    --hash=sha256:0d6ae97234d5d7079dc0040990a6f7aeb97cb7fa7e8945f1999a429b23569e0a \\\n    --hash=sha256:146b4d36f800c013d267b29168813f7a03a43ecd2895d04861f1240b564421da \\\n    --hash=sha256:15fc3b5d19cc2be65404786857f2e13c70c83dd4782676dd6814e3b89dc8f5b9 \\\n    --hash=sha256:2838551e011a64e3053ad7618dda9310175f7515f1742fa2d756f7c874c05961 \\\n    --hash=sha256:29ffc74089f3d5e87dfca4c2c8450f88bdc61b0fc6ed5d267f3988f19a1309f6 \\\n    --hash=sha256:2de443b9373b3b615aec1bb57f9baa6bb3a9bd093f1269ba95c17d870422b271 \\\n    --hash=sha256:35c007dedb2ffe38fe3ee7d201ebac4a2deccd2408e8621d53067733e3c74809 \\\n    --hash=sha256:3bad7565bc9cf37ce19a7c0d107742b320c1285df7aab1a6e2d28780df167242 \\\n    --hash=sha256:4496bb2cf7a43ce1a2d7524a79e40bc5da45cf598dbf9545b7e8316ccba47bb4 \\\n    --hash=sha256:4511be56637e46c25721e83d1a9cea9614e7badc7040c4d573d75fbe257d6fd7 \\\n    --hash=sha256:5025ce924beccb28298246e589c691fe1b8c1c96507e6d27d12c5fadd85bfd76 \\\n    --hash=sha256:56079a99c20d230e873ea40753102102734c5953366972a71d5cb39a32bc40c6 \\\n    --hash=sha256:5e30adb87f0cc81c7690a84f7932dd66be5bac57cfe16b91cb9151683a4a2d3b \\\n    --hash=sha256:5fb63362b5a7ddab88e52b6dbb47dac3fd7dafeee740dc6c8d8a446ddedade8e \\\n    --hash=sha256:6b595b07a03069a2b1740dc08c2299993850ea81cce4fe19b2421e0c970de6b7 \\\n    --hash=sha256:72358cce49465d140cc4e7792015bb1f0296a9742d5622c67e31399b75468b9e \\\n    --hash=sha256:74b66d8689d52ed04c271e1329f0c61635bcaf5b926db9b12d58914cdc01fe57 \\\n    --hash=sha256:7cc267b6108f0a1499a734167282c00c4ebf61328566b55ef262d48e9849c735 \\\n    --hash=sha256:80832434a6cc114f5219211eec13dcbc16c2bac0e31ef64c6d346cde3cf054cb \\\n    --hash=sha256:8c497fff237d7b4e07e9ef1a640887fa4fb765647f86fbe00f969ff6280ce2bb \\\n    --hash=sha256:8fdf95767f989b0cfedb85f7ed8ca215d4be728031f56ff5a519ee1e3276dc2e \\\n    --hash=sha256:9bccbb3b40e3de10351f8f5068e105d0f4083b1a65fa07b6634fbc401a6287fd \\\n    --hash=sha256:a0bcfe4d0d14aec44921545fd2af2338c7471de9cb701f1da4c9d85906ab847a \\\n    --hash=sha256:a69525355a641bf8ef136a7fa447672fb54fe8d60cab5538d9eb7c6438543fb9 \\\n    --hash=sha256:ada8121bcb4dac28d930febc791a69f7cb1673c8495e5eee274190b73a4559c1 \\\n    --hash=sha256:bf97c10a3f5a7543f9b88cbf488d33d175e9146115a451ae34568597ba33dcde \\\n    --hash=sha256:c22a2da7a198c28dd1a6e1136f19c830beab7fdca5b3e5c8bba8394f8a5c45b3 \\\n    --hash=sha256:c2656924ec73e5939c76ac4c8b026fc203b83d8900362eb2599d8aee80e4880f \\\n    --hash=sha256:c57b1b610bd1f40ba43970e11ce62821c2e6569e4d74023db19c6b26f246cb3b \\\n    --hash=sha256:eddde82a035681427cbedded4e6eff5e57fa59216c2e3e90b10b19ab1d0a65c3 \\\n    --hash=sha256:edec98c5e7c128328124a029bceb09eda2d526997780fef8d65e9a69eead963e \\\n    --hash=sha256:ee787491dbfe082d9c3013f01f5991658b0f38aa8177e4cd4bf434c58f551702 \\\n    --hash=sha256:f28dd15c6bb0b66ba09728cf09fd8736c304be29409bd8445a080c1280619e8c \\\n    --hash=sha256:f984ca4b14914e6b4094c5d52a32ea16b49832c03bd17a110f004db3c223e8e1 \\\n    --hash=sha256:fb65db5d7531bccf3a4f6bec3462223bea71384e2cda41da0f10b7c292b9e7c4 \\\n    --hash=sha256:fe1c011a640a9f0791146011dfd3c7d9669785f9fed2b2a5f9e207536cf5c2fd\n    # via -r tools/rice-requirements.in\nscipy==1.17.1 \\\n    --hash=sha256:010f4333c96c9bb1a4516269e33cb5917b08ef2166d5556ca2fd9f082a9e6ea0 \\\n    --hash=sha256:02ae3b274fde71c5e92ac4d54bc06c42d80e399fec704383dcd99b301df37458 \\\n    --hash=sha256:08b900519463543aa604a06bec02461558a6e1cef8fdbb8098f77a48a83c8118 \\\n    --hash=sha256:131f5aaea57602008f9822e2115029b55d4b5f7c070287699fe45c661d051e39 \\\n    --hash=sha256:158dd96d2207e21c966063e1635b1063cd7787b627b6f07305315dd73d9c679e \\\n    --hash=sha256:1cc682cea2ae55524432f3cdff9e9a3be743d52a7443d0cba9017c23c87ae2f6 \\\n    --hash=sha256:1f95b894f13729334fb990162e911c9e5dc1ab390c58aa6cbecb389c5b5e28ec \\\n    --hash=sha256:200e1050faffacc162be6a486a984a0497866ec54149a01270adc8a59b7c7d21 \\\n    --hash=sha256:2040ad4d1795a0ae89bfc7e8429677f365d45aa9fd5e4587cf1ea737f927b4a1 \\\n    --hash=sha256:2b64ca7d4aee0102a97f3ba22124052b4bd2152522355073580bf4845e2550b6 \\\n    --hash=sha256:2ceb2d3e01c5f1d83c4189737a42d9cb2fc38a6eeed225e7515eef71ad301dce \\\n    --hash=sha256:35c3a56d2ef83efc372eaec584314bd0ef2e2f0d2adb21c55e6ad5b344c0dcb8 \\\n    --hash=sha256:37425bc9175607b0268f493d79a292c39f9d001a357bebb6b88fdfaff13f6448 \\\n    --hash=sha256:3877ac408e14da24a6196de0ddcace62092bfc12a83823e92e49e40747e52c19 \\\n    --hash=sha256:3fd1fcdab3ea951b610dc4cef356d416d5802991e7e32b5254828d342f7b7e0b \\\n    --hash=sha256:41b71f4a3a4cab9d366cd9065b288efc4d4f3c0b37a91a8e0947fb5bd7f31d87 \\\n    --hash=sha256:43af8d1f3bea642559019edfe64e9b11192a8978efbd1539d7bc2aaa23d92de4 \\\n    --hash=sha256:45abad819184f07240d8a696117a7aacd39787af9e0b719d00285549ed19a1e9 \\\n    --hash=sha256:4b400bdc6f79fa02a4d86640310dde87a21fba0c979efff5248908c6f15fad1b \\\n    --hash=sha256:4eb6c25dd62ee8d5edf68a8e1c171dd71c292fdae95d8aeb3dd7d7de4c364082 \\\n    --hash=sha256:581b2264fc0aa555f3f435a5944da7504ea3a065d7029ad60e7c3d1ae09c5464 \\\n    --hash=sha256:5cf36e801231b6a2059bf354720274b7558746f3b1a4efb43fcf557ccd484a87 \\\n    --hash=sha256:5e3c5c011904115f88a39308379c17f91546f77c1667cea98739fe0fccea804c \\\n    --hash=sha256:6609bc224e9568f65064cfa72edc0f24ee6655b47575954ec6339534b2798369 \\\n    --hash=sha256:6e3dcd57ab780c741fde8dc68619de988b966db759a3c3152e8e9142c26295ad \\\n    --hash=sha256:6fac755ca3d2c3edcb22f479fceaa241704111414831ddd3bc6056e18516892f \\\n    --hash=sha256:744b2bf3640d907b79f3fd7874efe432d1cf171ee721243e350f55234b4cec4c \\\n    --hash=sha256:74cbb80d93260fe2ffa334efa24cb8f2f0f622a9b9febf8b483c0b865bfb3475 \\\n    --hash=sha256:766e0dc5a616d026a3a1cffa379af959671729083882f50307e18175797b3dfd \\\n    --hash=sha256:7bdf2da170b67fdf10bca777614b1c7d96ae3ca5794fd9587dce41eb2966e866 \\\n    --hash=sha256:7ff200bf9d24f2e4d5dc6ee8c3ac64d739d3a89e2326ba68aaf6c4a2b838fd7d \\\n    --hash=sha256:844e165636711ef41f80b4103ed234181646b98a53c8f05da12ca5ca289134f6 \\\n    --hash=sha256:8a604bae87c6195d8b1045eddece0514d041604b14f2727bbc2b3020172045eb \\\n    --hash=sha256:94055a11dfebe37c656e70317e1996dc197e1a15bbcc351bcdd4610e128fe1ca \\\n    --hash=sha256:95d8e012d8cb8816c226aef832200b1d45109ed4464303e997c5b13122b297c0 \\\n    --hash=sha256:9cdc1a2fcfd5c52cfb3045feb399f7b3ce822abdde3a193a6b9a60b3cb5854ca \\\n    --hash=sha256:9ecb4efb1cd6e8c4afea0daa91a87fbddbce1b99d2895d151596716c0b2e859d \\\n    --hash=sha256:a3472cfbca0a54177d0faa68f697d8ba4c80bbdc19908c3465556d9f7efce9ee \\\n    --hash=sha256:a4328d245944d09fd639771de275701ccadf5f781ba0ff092ad141e017eccda4 \\\n    --hash=sha256:a48a72c77a310327f6a3a920092fa2b8fd03d7deaa60f093038f22d98e096717 \\\n    --hash=sha256:a720477885a9d2411f94a93d16f9d89bad0f28ca23c3f8daa521e2dcc3f44d49 \\\n    --hash=sha256:a77cbd07b940d326d39a1d1b37817e2ee4d79cb30e7338f3d0cddffae70fcaa2 \\\n    --hash=sha256:a9956e4d4f4a301ebf6cde39850333a6b6110799d470dbbb1e25326ac447f52a \\\n    --hash=sha256:adb2642e060a6549c343603a3851ba76ef0b74cc8c079a9a58121c7ec9fe2350 \\\n    --hash=sha256:beeda3d4ae615106d7094f7e7cef6218392e4465cc95d25f900bebabfded0950 \\\n    --hash=sha256:c80be5ede8f3f8eded4eff73cc99a25c388ce98e555b17d31da05287015ffa5b \\\n    --hash=sha256:cc90d2e9c7e5c7f1a482c9875007c095c3194b1cfedca3c2f3291cdc2bc7c086 \\\n    --hash=sha256:cd96a1898c0a47be4520327e01f874acfd61fb48a9420f8aa9f6483412ffa444 \\\n    --hash=sha256:d2650c1fb97e184d12d8ba010493ee7b322864f7d3d00d3f9bb97d9c21de4068 \\\n    --hash=sha256:d30e57c72013c2a4fe441c2fcb8e77b14e152ad48b5464858e07e2ad9fbfceff \\\n    --hash=sha256:d59c30000a16d8edc7e64152e30220bfbd724c9bbb08368c054e24c651314f0a \\\n    --hash=sha256:dbc12c9f3d185f5c737d801da555fb74b3dcfa1a50b66a1a93e09190f41fab50 \\\n    --hash=sha256:e18f12c6b0bc5a592ed23d3f7b891f68fd7f8241d69b7883769eb5d5dfb52696 \\\n    --hash=sha256:e19ebea31758fac5893a2ac360fedd00116cbb7628e650842a6691ba7ca28a21 \\\n    --hash=sha256:e30bdeaa5deed6bc27b4cc490823cd0347d7dae09119b8803ae576ea0ce52e4c \\\n    --hash=sha256:eb092099205ef62cd1782b006658db09e2fed75bffcae7cc0d44052d8aa0f484 \\\n    --hash=sha256:eee2cfda04c00a857206a4330f0c5e3e56535494e30ca445eb19ec624ae75118 \\\n    --hash=sha256:f4115102802df98b2b0db3cce5cb9b92572633a1197c77b7553e5203f284a5b3 \\\n    --hash=sha256:f590cd684941912d10becc07325a3eeb77886fe981415660d9265c4c418d0bea \\\n    --hash=sha256:f8885db0bc2bffa59d5c1b72fad7a6a92d3e80e7257f967dd81abb553a90d293 \\\n    --hash=sha256:fcb310ddb270a06114bb64bbe53c94926b943f5b7f0842194d585c65eb4edd76\n    # via\n    #   -r tools/rice-requirements.in\n    #   scikit-learn\nsentencepiece==0.2.1 \\\n    --hash=sha256:010f025a544ef770bb395091d57cb94deb9652d8972e0d09f71d85d5a0816c8c \\\n    --hash=sha256:017f97b274d4b0baa84b2dc743bf4517be81156f413bb24f12aacacde378e5ab \\\n    --hash=sha256:01e6912125cb45d3792f530a4d38f8e21bf884d6b4d4ade1b2de5cf7a8d2a52b \\\n    --hash=sha256:02593eca45440ef39247cee8c47322a34bdcc1d8ae83ad28ba5a899a2cf8d79a \\\n    --hash=sha256:097f3394e99456e9e4efba1737c3749d7e23563dd1588ce71a3d007f25475fff \\\n    --hash=sha256:0a0d15781a171d188b661ae4bde1d998c303f6bd8621498c50c671bd45a4798e \\\n    --hash=sha256:0a81799d0a68d618e89063fb423c3001a034c893069135ffe51fee439ae474d6 \\\n    --hash=sha256:0c0f672da370cc490e4c59d89e12289778310a0e71d176c541e4834759e1ae07 \\\n    --hash=sha256:0cdfecef430d985f1c2bcbfff3defd1d95dae876fbd0173376012d2d7d24044b \\\n    --hash=sha256:105e36e75cbac1292642045458e8da677b2342dcd33df503e640f0b457cb6751 \\\n    --hash=sha256:10ed3dab2044c47f7a2e7b4969b0c430420cdd45735d78c8f853191fa0e3148b \\\n    --hash=sha256:1855f57db07b51fb51ed6c9c452f570624d2b169b36f0f79ef71a6e6c618cd8b \\\n    --hash=sha256:2005242a16d2dc3ac5fe18aa7667549134d37854823df4c4db244752453b78a8 \\\n    --hash=sha256:22c4ebcb3c6ab1496ab1c37c79ef7bb563b8726f29548c30773b7a4cb152df1a \\\n    --hash=sha256:251874d720ac7f28024a168501f3c7bb15d1802245f6e66de565f18bbb9b5eaa \\\n    --hash=sha256:27e38eee653abc3d387862e67bc5c8b6f428cd604e688b85d29170b7e725c26c \\\n    --hash=sha256:2af5a1fb05013332ad94343b8b5f3973e006a2dde2dfba55a819549e054e2f0f \\\n    --hash=sha256:2f27ae6deea72efdb6f361750c92f6c21fd0ad087445082770cc34015213c526 \\\n    --hash=sha256:33f068c9382dc2e7c228eedfd8163b52baa86bb92f50d0488bf2b7da7032e484 \\\n    --hash=sha256:39f8651bd10974eafb9834ce30d9bcf5b73e1fc798a7f7d2528f9820ca86e119 \\\n    --hash=sha256:3d165fbb9bf8fba35f1946ba2617c3f9995679f07438325f07c026d53f33e746 \\\n    --hash=sha256:477c81505db072b3ab627e7eab972ea1025331bd3a92bacbf798df2b75ea86ec \\\n    --hash=sha256:4cdc7c36234fda305e85c32949c5211faaf8dd886096c7cea289ddc12a2d02de \\\n    --hash=sha256:4f5a3e0d9f445ed9d66c0fec47d4b23d12cfc858b407a03c194c1b26c2ac2a63 \\\n    --hash=sha256:56dd39a3c4d6493db3cdca7e8cc68c6b633f0d4195495cbadfcf5af8a22d05a6 \\\n    --hash=sha256:57cae326c8727de58c85977b175af132a7138d84c764635d7e71bbee7e774133 \\\n    --hash=sha256:5d0350b686c320068702116276cfb26c066dc7e65cfef173980b11bb4d606719 \\\n    --hash=sha256:5e4366c97b68218fd30ea72d70c525e6e78a6c0a88650f57ac4c43c63b234a9d \\\n    --hash=sha256:60937c959e6f44159fdd9f56fbdd302501f96114a5ba436829496d5f32d8de3f \\\n    --hash=sha256:6356d0986b8b8dc351b943150fcd81a1c6e6e4d439772e8584c64230e58ca987 \\\n    --hash=sha256:6d297a1748d429ba8534eebe5535448d78b8acc32d00a29b49acf28102eeb094 \\\n    --hash=sha256:733e59ff1794d26db706cd41fc2d7ca5f6c64a820709cb801dc0ea31780d64ab \\\n    --hash=sha256:8138cec27c2f2282f4a34d9a016e3374cd40e5c6e9cb335063db66a0a3b71fad \\\n    --hash=sha256:814978ac05130dd5812b4b03215c766bc6abaef13e7bd72bc534e4d1e12e9a4c \\\n    --hash=sha256:82d9ead6591015f009cb1be1cb1c015d5e6f04046dbb8c9588b931e869a29728 \\\n    --hash=sha256:881b2e44b14fc19feade3cbed314be37de639fc415375cefaa5bc81a4be137fd \\\n    --hash=sha256:891ade6503dd93d418c03993f7d6a8aa20260c422cefff5096b9068185e67642 \\\n    --hash=sha256:89a3ea015517c42c0341d0d962f3e6aaf2cf10d71b1932d475c44ba48d00aa2b \\\n    --hash=sha256:8dd4b477a7b069648d19363aad0cab9bad2f4e83b2d179be668efa672500dc94 \\\n    --hash=sha256:8f8ba89a3acb3dc1ae90f65ec1894b0b9596fdb98ab003ff38e058f898b39bc7 \\\n    --hash=sha256:9076430ac25dfa7147d9d05751dbc66a04bc1aaac371c07f84952979ea59f0d0 \\\n    --hash=sha256:92b3816aa2339355fda2c8c4e021a5de92180b00aaccaf5e2808972e77a4b22f \\\n    --hash=sha256:99f955df238021bf11f0fc37cdb54fd5e5b5f7fd30ecc3d93fb48b6815437167 \\\n    --hash=sha256:a19adcec27c524cb7069a1c741060add95f942d1cbf7ad0d104dffa0a7d28a2b \\\n    --hash=sha256:a483fd29a34c3e34c39ac5556b0a90942bec253d260235729e50976f5dba1068 \\\n    --hash=sha256:ac650534e2251083c5f75dde4ff28896ce7c8904133dc8fef42780f4d5588fcd \\\n    --hash=sha256:ad8493bea8432dae8d6830365352350f3b4144415a1d09c4c8cb8d30cf3b6c3c \\\n    --hash=sha256:afefe50a0cdcb4f2fd9733cb52001a2c164181ee2d82c32d38f5b1b326a8528c \\\n    --hash=sha256:b3616ad246f360e52c85781e47682d31abfb6554c779e42b65333d4b5f44ecc0 \\\n    --hash=sha256:b81a24733726e3678d2db63619acc5a8dccd074f7aa7a54ecd5ca33ca6d2d596 \\\n    --hash=sha256:c415c9de1447e0a74ae3fdb2e52f967cb544113a3a5ce3a194df185cbc1f962f \\\n    --hash=sha256:c6c8f42949f419ff8c7e9960dbadcfbc982d7b5efc2f6748210d3dd53a7de062 \\\n    --hash=sha256:c7f0fd2f2693309e6628aeeb2e2faf6edd221134dfccac3308ca0de01f8dab47 \\\n    --hash=sha256:c7f54a31cde6fa5cb030370566f68152a742f433f8d2be458463d06c208aef33 \\\n    --hash=sha256:c83b85ab2d6576607f31df77ff86f28182be4a8de6d175d2c33ca609925f5da1 \\\n    --hash=sha256:caa4e560c72c151da80036aecc2159e51a7fd8ae9efebefd96860460ce6bd025 \\\n    --hash=sha256:d3233770f78e637dc8b1fda2cd7c3b99ec77e7505041934188a4e7fe751de3b0 \\\n    --hash=sha256:d7b670879c370d350557edabadbad1f6561a9e6968126e6debca4029e5547820 \\\n    --hash=sha256:d8b1d91545578852f128650b8cce4ec20f93d39b378ff554ebe66290f2dabb92 \\\n    --hash=sha256:d9381351182ff9888cc80e41c632e7e274b106f450de33d67a9e8f6043da6f76 \\\n    --hash=sha256:daeb5e9e9fcad012324807856113708614d534f596d5008638eb9b40112cd9e4 \\\n    --hash=sha256:dcd8161eee7b41aae57ded06272905dbd680a0a04b91edd0f64790c796b2f706 \\\n    --hash=sha256:e10fa50bdbaa5e2445dbd387979980d391760faf0ec99a09bd7780ff37eaec44 \\\n    --hash=sha256:e37e4b4c4a11662b5db521def4e44d4d30ae69a1743241412a93ae40fdcab4bb \\\n    --hash=sha256:e52144670738b4b477fade6c2a9b6af71a8d0094514c9853ac9f6fc1fcfabae7\n    # via -r tools/rice-requirements.in\nsetuptools==81.0.0 \\\n    --hash=sha256:487b53915f52501f0a79ccfd0c02c165ffe06631443a886740b91af4b7a5845a \\\n    --hash=sha256:fdd925d5c5d9f62e4b74b30d6dd7828ce236fd6ed998a08d81de62ce5a6310d6\n    # via torch\nsix==1.17.0 \\\n    --hash=sha256:4721f391ed90541fddacab5acf947aa0d3dc7d27b2e1e8eda2be8970586c3274 \\\n    --hash=sha256:ff70335d468e7eb6ec65b95b99d3a2836546063f63acc5171de367e834932a81\n    # via python-dateutil\nsympy==1.14.0 \\\n    --hash=sha256:d3d3fe8df1e5a0b42f0e7bdf50541697dbe7d23746e894990c030e2b05e72517 \\\n    --hash=sha256:e091cc3e99d2141a0ba2847328f5479b05d94a6635cb96148ccb3f34671bd8f5\n    # via torch\nthreadpoolctl==3.7.0 \\\n    --hash=sha256:61348cfb77d53b9242e0017029244b559b810c142ced65b4e21eeca1843959a7 \\\n    --hash=sha256:cd8b60b5641b45c67bbf73c64c843235fc2d8a480c87389f52f5dbee893b86be\n    # via scikit-learn\ntimm==1.0.29 \\\n    --hash=sha256:1af8d12bb7e15c5f96e98d60b7b8319cd7f31730778bf1af58e912a7ce171576 \\\n    --hash=sha256:4266486e355c9beb9432e689de580d09e2efe7fa92a9a79e76c03e7763eac724\n    # via\n    #   -r tools/rice-requirements.in\n    #   open-clip-torch\ntokenizers==0.22.2 \\\n    --hash=sha256:143b999bdc46d10febb15cbffb4207ddd1f410e2c755857b5a0797961bbdc113 \\\n    --hash=sha256:1a62ba2c5faa2dd175aaeed7b15abf18d20266189fb3406c5d0550dd34dd5f37 \\\n    --hash=sha256:1c774b1276f71e1ef716e5486f21e76333464f47bece56bbd554485982a9e03e \\\n    --hash=sha256:1e418a55456beedca4621dbab65a318981467a2b188e982a23e117f115ce5001 \\\n    --hash=sha256:1e50f8554d504f617d9e9d6e4c2c2884a12b388a97c5c77f0bc6cf4cd032feee \\\n    --hash=sha256:2249487018adec45d6e3554c71d46eb39fa8ea67156c640f7513eb26f318cec7 \\\n    --hash=sha256:25b85325d0815e86e0bac263506dd114578953b7b53d7de09a6485e4a160a7dd \\\n    --hash=sha256:29c30b83d8dcd061078b05ae0cb94d3c710555fbb44861139f9f83dcca3dc3e4 \\\n    --hash=sha256:319f659ee992222f04e58f84cbf407cfa66a65fe3a8de44e8ad2bc53e7d99012 \\\n    --hash=sha256:369cc9fc8cc10cb24143873a0d95438bb8ee257bb80c71989e3ee290e8d72c67 \\\n    --hash=sha256:37ae80a28c1d3265bb1f22464c856bd23c02a05bb211e56d0c5301a435be6c1a \\\n    --hash=sha256:38337540fbbddff8e999d59970f3c6f35a82de10053206a7562f1ea02d046fa5 \\\n    --hash=sha256:473b83b915e547aa366d1eee11806deaf419e17be16310ac0a14077f1e28f917 \\\n    --hash=sha256:544dd704ae7238755d790de45ba8da072e9af3eea688f698b137915ae959281c \\\n    --hash=sha256:64d94e84f6660764e64e7e0b22baa72f6cd942279fdbb21d46abd70d179f0195 \\\n    --hash=sha256:753d47ebd4542742ef9261d9da92cd545b2cacbb48349a1225466745bb866ec4 \\\n    --hash=sha256:791135ee325f2336f498590eb2f11dc5c295232f288e75c99a36c5dbce63088a \\\n    --hash=sha256:9ce725d22864a1e965217204946f830c37876eee3b2ba6fc6255e8e903d5fcbc \\\n    --hash=sha256:a6bf3f88c554a2b653af81f3204491c818ae2ac6fbc09e76ef4773351292bc92 \\\n    --hash=sha256:bfb88f22a209ff7b40a576d5324bf8286b519d7358663db21d6246fb17eea2d5 \\\n    --hash=sha256:c9ea31edff2968b44a88f97d784c2f16dc0729b8b143ed004699ebca91f05c48 \\\n    --hash=sha256:df6c4265b289083bf710dff49bc51ef252f9d5be33a45ee2bed151114a56207b \\\n    --hash=sha256:e10bf9113d209be7cd046d40fbabbaf3278ff6d18eb4da4c500443185dc1896c \\\n    --hash=sha256:f01a9c019878532f98927d2bacb79bbb404b43d3437455522a00a30718cdedb5\n    # via transformers\ntorch==2.11.0 \\\n    --hash=sha256:01018087326984a33b64e04c8cb5c2795f9120e0d775ada1f6638840227b04d7 \\\n    --hash=sha256:0f68f4ac6d95d12e896c3b7a912b5871619542ec54d3649cf48cc1edd4dd2756 \\\n    --hash=sha256:1b32ceda909818a03b112006709b02be1877240c31750a8d9c6b7bf5f2d8a6e5 \\\n    --hash=sha256:1e6debd97ccd3205bbb37eb806a9d8219e1139d15419982c09e23ef7d4369d18 \\\n    --hash=sha256:2658f34ce7e2dabf4ec73b45e2ca68aedad7a5be87ea756ad656eaf32bf1e1ea \\\n    --hash=sha256:2b4e811728bd0cc58fb2b0948fe939a1ee2bf1422f6025be2fca4c7bd9d79718 \\\n    --hash=sha256:2bb3cc54bd0dea126b0060bb1ec9de0f9c7f7342d93d436646516b0330cd5be7 \\\n    --hash=sha256:2c0d7fcfbc0c4e8bb5ebc3907cbc0c6a0da1b8f82b1fc6e14e914fa0b9baf74e \\\n    --hash=sha256:4b5866312ee6e52ea625cd211dcb97d6a2cdc1131a5f15cc0d87eec948f6dd34 \\\n    --hash=sha256:4cf8687f4aec3900f748d553483ef40e0ac38411c3c48d0a86a438f6d7a99b18 \\\n    --hash=sha256:4dc8b3809469b6c30b411bb8c4cad3828efd26236153d9beb6a3ec500f211a60 \\\n    --hash=sha256:4dda3b3f52d121063a731ddb835f010dc137b920d7fec2778e52f60d8e4bf0cd \\\n    --hash=sha256:563ed3d25542d7e7bbc5b235ccfacfeb97fb470c7fee257eae599adb8005c8a2 \\\n    --hash=sha256:63a68fa59de8f87acc7e85a5478bb2dddbb3392b7593ec3e78827c793c4b73fd \\\n    --hash=sha256:73e24aaf8f36ab90d95cd1761208b2eb70841c2a9ca1a3f9061b39fc5331b708 \\\n    --hash=sha256:7aa2f9bbc6d4595ba72138026b2074be1233186150e9292865e04b7a63b8c67a \\\n    --hash=sha256:7b6a60d48062809f58595509c524b88e6ddec3ebe25833d6462eeab81e5f2ce4 \\\n    --hash=sha256:8245477871c3700d4370352ffec94b103cfcb737229445cf9946cddb7b2ca7cd \\\n    --hash=sha256:8b394322f49af4362d4f80e424bcaca7efcd049619af03a4cf4501520bdf0fb4 \\\n    --hash=sha256:98bb213c3084cfe176302949bdc360074b18a9da7ab59ef2edc9d9f742504778 \\\n    --hash=sha256:a97b94bbf62992949b4730c6cd2cc9aee7b335921ee8dc207d930f2ed09ae2db \\\n    --hash=sha256:ab9a8482f475f9ba20e12db84b0e55e2f58784bdca43a854a6ccd3fd4b9f75e6 \\\n    --hash=sha256:b2a43985ff5ef6ddd923bbcf99943e5f58059805787c5c9a2622bf05ca2965b0 \\\n    --hash=sha256:b3c712ae6fb8e7a949051a953fc412fe0a6940337336c3b6f905e905dac5157f \\\n    --hash=sha256:cc89b9b173d9adfab59fd227f0ab5e5516d9a52b658ae41d64e59d2e55a418db \\\n    --hash=sha256:d91aac77f24082809d2c5a93f52a5f085032740a1ebc9252a7b052ef5a4fddc6 \\\n    --hash=sha256:f99924682ef0aa6a4ab3b1b76f40dc6e273fca09f367d15a524266db100a723f \\\n    --hash=sha256:fbf39280699d1b869f55eac536deceaa1b60bd6788ba74f399cc67e60a5fab10\n    # via\n    #   -r tools/rice-requirements.in\n    #   open-clip-torch\n    #   timm\n    #   torchvision\ntorchvision==0.26.0 \\\n    --hash=sha256:0f3e572efe62ad645017ea847e0b5e4f2f638d4e39f05bc011d1eb9ac68d4806 \\\n    --hash=sha256:114bec0c0e98aa4ba446f63e2fe7a2cbca37b39ac933987ee4804f65de121800 \\\n    --hash=sha256:1c55dc8affbcc0eb2060fbabbe996ae9e5839b24bb6419777f17848945a411b1 \\\n    --hash=sha256:2adfbe438473236191ff077a4a9a0c767436879c89628aa97137e959b0c11a94 \\\n    --hash=sha256:358fc4726d0c08615b6d83b3149854f11efb2a564ed1acb6fce882e151412d23 \\\n    --hash=sha256:3daf9cc149cf3cdcbd4df9c59dae69ffca86c6823250442c3bbfd63fc2e26c61 \\\n    --hash=sha256:406557718e62fdf10f5706e88d8a5ec000f872da913bf629aab9297622585547 \\\n    --hash=sha256:4280c35ec8cba1fcc8294fb87e136924708726864c379e4c54494797d86bc474 \\\n    --hash=sha256:55bd6ad4ae77be01ba67a410b05b51f53b0d0ee45f146eb6a0dfb9007e70ab3c \\\n    --hash=sha256:5d63dd43162691258b1b3529b9041bac7d54caa37eae0925f997108268cbf7c4 \\\n    --hash=sha256:7058c5878262937e876f20c25867b33724586aa4499e2853b2d52b99a5e51953 \\\n    --hash=sha256:7993c01648e7c61d191b018e84d38fe0825c8fcb2720cd0f37caf7ba14404aa1 \\\n    --hash=sha256:8008474855623c6ba52876589dc52df0aa66e518c25eca841445348e5f79844c \\\n    --hash=sha256:82c3965eca27e86a316e31e4c3e5a16d353e0bcbe0ef8efa2e66502c54493c4b \\\n    --hash=sha256:9a904f2131cbfadab4df828088a9f66291ad33f49ff853872aed1f86848ef776 \\\n    --hash=sha256:a06d4772a8e13e772906ed736cc53ec6639e5e60554f8e5fa6ca165aabebc464 \\\n    --hash=sha256:a39c7a26538c41fda453f9a9692b5ff9b35a5437db1d94f3027f6f509c160eac \\\n    --hash=sha256:b6f9ad1ecc0eab52647298b379ee9426845f8903703e6127973f8f3d049a798b \\\n    --hash=sha256:b7d3e295624a28b3b1769228ce1345d94cf4d390dd31136766f76f2d20f718da \\\n    --hash=sha256:b7e6213620bbf97742e5f79832f9e9d769e6cf0f744c5b53dad80b76db633691 \\\n    --hash=sha256:c409e1c3fdebec7a3834465086dbda8bf7680eff79abf7fd2f10c6b59520a7a4 \\\n    --hash=sha256:d61a5abb6b42a0c0c311996c2ac4b83a94418a97182c83b055a2a4ae985e05aa \\\n    --hash=sha256:de6424b12887ad884f39a0ee446994ae3cd3b6a00a9cafe1bead85a031132af0 \\\n    --hash=sha256:e9d0e022c19a78552fb055d0414d47fecb4a649309b9968573daea160ba6869c \\\n    --hash=sha256:eb61804eb9dbe88c5a2a6c4da8dec1d80d2d0a6f18c999c524e32266cb1ebcd3 \\\n    --hash=sha256:ebc043cc5a4f0bf22e7680806dbba37ffb19e70f6953bbb44ed1a90aeb5c9bea \\\n    --hash=sha256:f13f12b3791a266de2d599cb8162925261622a037d87fc03132848343cf68f75 \\\n    --hash=sha256:fd10b5f994c210f4f6d6761cf686f82d748554adf486cb0979770c3252868c8f\n    # via\n    #   -r tools/rice-requirements.in\n    #   open-clip-torch\n    #   timm\ntqdm==4.70.1 \\\n    --hash=sha256:c293e525e6fef9c20e8728fd4612df02a0aa31bb5fe91ecd93e123b1b7bffa73 \\\n    --hash=sha256:cefd0eca11b2a37a3aee776544d4f4ae913f02688135b5556b8788dfa474afc4\n    # via\n    #   huggingface-hub\n    #   open-clip-torch\n    #   transformers\ntransformers==4.57.6 \\\n    --hash=sha256:4c9e9de11333ddfe5114bc872c9f370509198acf0b87a832a0ab9458e2bd0550 \\\n    --hash=sha256:55e44126ece9dc0a291521b7e5492b572e6ef2766338a610b9ab5afbb70689d3\n    # via -r tools/rice-requirements.in\ntriton==3.6.0 \\\n    --hash=sha256:0b3a97e8ed304dfa9bd23bb41ca04cdf6b2e617d5e782a8653d616037a5d537d \\\n    --hash=sha256:10c7f76c6e72d2ef08df639e3d0d30729112f47a56b0c81672edc05ee5116ac9 \\\n    --hash=sha256:1722e172d34e32abc3eb7711d0025bb69d7959ebea84e3b7f7a341cd7ed694d6 \\\n    --hash=sha256:374f52c11a711fd062b4bfbb201fd9ac0a5febd28a96fb41b4a0f51dde3157f4 \\\n    --hash=sha256:448e02fe6dc898e9e5aa89cf0ee5c371e99df5aa5e8ad976a80b93334f3494fd \\\n    --hash=sha256:46bd1c1af4b6704e554cad2eeb3b0a6513a980d470ccfa63189737340c7746a7 \\\n    --hash=sha256:49df5ef37379c0c2b5c0012286f80174fcf0e073e5ade1ca9a86c36814553651 \\\n    --hash=sha256:6c723cfb12f6842a0ae94ac307dba7e7a44741d720a40cf0e270ed4a4e3be781 \\\n    --hash=sha256:74caf5e34b66d9f3a429af689c1c7128daba1d8208df60e81106b115c00d6fca \\\n    --hash=sha256:a17a5d5985f0ac494ed8a8e54568f092f7057ef60e1b0fa09d3fd1512064e803 \\\n    --hash=sha256:a6550fae429e0667e397e5de64b332d1e5695b73650ee75a6146e2e902770bea \\\n    --hash=sha256:d002e07d7180fd65e622134fbd980c9a3d4211fb85224b56a0a0efbd422ab72f \\\n    --hash=sha256:e8e323d608e3a9bfcc2d9efcc90ceefb764a82b99dea12a86d643c72539ad5d3 \\\n    --hash=sha256:ef5523241e7d1abca00f1d240949eebdd7c673b005edbbce0aca95b8191f1d43\n    # via torch\ntyping-extensions==4.16.0 \\\n    --hash=sha256:481caa481374e813c1b176ada14e97f1f67a4539ce9cfeb3f350d78d6370c2e8 \\\n    --hash=sha256:dc983d19a509c94dba722ee6abd33940f7c05a89e243c47e907eb4db6f1a43e5\n    # via\n    #   huggingface-hub\n    #   torch\nurllib3==2.8.0 \\\n    --hash=sha256:0cf3cae568d36aa9576b28dfb35f11328f1cb974ca7647d9475ebb86c75ac6e3 \\\n    --hash=sha256:63bf2ead4c879426ebf22ef2a781eeb4aa3b4ae798a0435506f8687fd5bb9b63\n    # via requests\nwcwidth==0.9.1 \\\n    --hash=sha256:03cfca3dcbffa86564290fe3c9978a6191ba003e8ced7f7dbda315fcb3fbe725 \\\n    --hash=sha256:0665ee822ea04e25801e6e82e5407528be0863e88a17b0e7ca038843a4a3ac0c \\\n    --hash=sha256:0d68a30d504c68cfdff2a5f804675c1e7ab4c0bbe878024c8b680ec5579cde67 \\\n    --hash=sha256:10b00ba23482e352f874d2e8135e7ace9da838646c7dd800566246bbd46125ff \\\n    --hash=sha256:356376852357b8fca71fe5415808ec421679e04b4a98eb7c9cb6a7984b911a05 \\\n    --hash=sha256:40d936d72c9bdc10df43f93a8be502bc5024b487259139f66a328722c07f34a9 \\\n    --hash=sha256:5823209b0d43af322ce698c689380d7c15ca31fa8e6e3be8459f27031bef0af5 \\\n    --hash=sha256:61bd7aef9cafb6cb77a37a169998d7928ce82a51522146d11f60db9e7d1cb43a \\\n    --hash=sha256:69bb970cf5652b88cfdb9d3fdd1764fc15e5f8ad531643e7bb3e894cd969740e \\\n    --hash=sha256:6e1272b7986cefe79783737e38bdb9eaae0682b333c7bd132024441193dd5ce7 \\\n    --hash=sha256:708158c082364af442f9983de7b6ec9ac0d2e1b825ada25f0911b1f138d55405 \\\n    --hash=sha256:747fb724223f417a17541a95a17c1dac3a8ef9a0cf41684950f0eab191a35f65 \\\n    --hash=sha256:991d1c8834f548e9c1f16432075ee84638e122312556bbf1ed595ea8fffc4673 \\\n    --hash=sha256:9f1636c5075ffd5c2e835b4561874f6e5dd2bbeba3c6c2c99f067d0d16883af8 \\\n    --hash=sha256:b5da43d6967668982e44a52fb551967d293f86d26cd86036ac95bbdd34394ed9 \\\n    --hash=sha256:bcb9ed4a367cc025bf1092ac679a156759605182d60b7ed3c28f7221a42ddf55 \\\n    --hash=sha256:c4cead196551112cb8f43cdd1f80c235ef2456e34b1a9955c424537ec99961b2 \\\n    --hash=sha256:dc10e262c3ac0abbfd0a2a51e45a848b1b7f500b21ff512630277973ba25674d \\\n    --hash=sha256:e0c3a1c45c5b9550c6919a4449e95f5b177f6e165786376981db8f1addae9b21 \\\n    --hash=sha256:eab587e18e7cadf1a750b0098fc8bebfb62c125eb9306f2268f0443a282a3d78 \\\n    --hash=sha256:fe021c4d8de9d36c31a0cb41d0d2546dadd3b0708a301f1a0c66ce200851831f\n    # via ftfy\n', 'LICENSE.txt': '                                 Apache License\n                           Version 2.0, January 2004\n                        http://www.apache.org/licenses/\n\n   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION\n\n   1. Definitions.\n\n      "License" shall mean the terms and conditions for use, reproduction,\n      and distribution as defined by Sections 1 through 9 of this document.\n\n      "Licensor" shall mean the copyright owner or entity authorized by\n      the copyright owner that is granting the License.\n\n      "Legal Entity" shall mean the union of the acting entity and all\n      other entities that control, are controlled by, or are under common\n      control with that entity. For the purposes of this definition,\n      "control" means (i) the power, direct or indirect, to cause the\n      direction or management of such entity, whether by contract or\n      otherwise, or (ii) ownership of fifty percent (50%) or more of the\n      outstanding shares, or (iii) beneficial ownership of such entity.\n\n      "You" (or "Your") shall mean an individual or Legal Entity\n      exercising permissions granted by this License.\n\n      "Source" form shall mean the preferred form for making modifications,\n      including but not limited to software source code, documentation\n      source, and configuration files.\n\n      "Object" form shall mean any form resulting from mechanical\n      transformation or translation of a Source form, including but\n      not limited to compiled object code, generated documentation,\n      and conversions to other media types.\n\n      "Work" shall mean the work of authorship, whether in Source or\n      Object form, made available under the License, as indicated by a\n      copyright notice that is included in or attached to the work\n      (an example is provided in the Appendix below).\n\n      "Derivative Works" shall mean any work, whether in Source or Object\n      form, that is based on (or derived from) the Work and for which the\n      editorial revisions, annotations, elaborations, or other modifications\n      represent, as a whole, an original work of authorship. For the purposes\n      of this License, Derivative Works shall not include works that remain\n      separable from, or merely link (or bind by name) to the interfaces of,\n      the Work and Derivative Works thereof.\n\n      "Contribution" shall mean any work of authorship, including\n      the original version of the Work and any modifications or additions\n      to that Work or Derivative Works thereof, that is intentionally\n      submitted to Licensor for inclusion in the Work by the copyright owner\n      or by an individual or Legal Entity authorized to submit on behalf of\n      the copyright owner. For the purposes of this definition, "submitted"\n      means any form of electronic, verbal, or written communication sent\n      to the Licensor or its representatives, including but not limited to\n      communication on electronic mailing lists, source code control systems,\n      and issue tracking systems that are managed by, or on behalf of, the\n      Licensor for the purpose of discussing and improving the Work, but\n      excluding communication that is conspicuously marked or otherwise\n      designated in writing by the copyright owner as "Not a Contribution."\n\n      "Contributor" shall mean Licensor and any individual or Legal Entity\n      on behalf of whom a Contribution has been received by Licensor and\n      subsequently incorporated within the Work.\n\n   2. Grant of Copyright License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      copyright license to reproduce, prepare Derivative Works of,\n      publicly display, publicly perform, sublicense, and distribute the\n      Work and such Derivative Works in Source or Object form.\n\n   3. Grant of Patent License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      (except as stated in this section) patent license to make, have made,\n      use, offer to sell, sell, import, and otherwise transfer the Work,\n      where such license applies only to those patent claims licensable\n      by such Contributor that are necessarily infringed by their\n      Contribution(s) alone or by combination of their Contribution(s)\n      with the Work to which such Contribution(s) was submitted. If You\n      institute patent litigation against any entity (including a\n      cross-claim or counterclaim in a lawsuit) alleging that the Work\n      or a Contribution incorporated within the Work constitutes direct\n      or contributory patent infringement, then any patent licenses\n      granted to You under this License for that Work shall terminate\n      as of the date such litigation is filed.\n\n   4. Redistribution. You may reproduce and distribute copies of the\n      Work or Derivative Works thereof in any medium, with or without\n      modifications, and in Source or Object form, provided that You\n      meet the following conditions:\n\n      (a) You must give any other recipients of the Work or Derivative\n          Works a copy of this License; and\n\n      (b) You must cause any modified files to carry prominent notices\n          stating that You changed the files; and\n\n      (c) You must retain, in the Source form of any Derivative Works\n          that You distribute, all copyright, patent, trademark, and\n          attribution notices from the Source form of the Work,\n          excluding those notices that do not pertain to any part of\n          the Derivative Works; and\n\n      (d) If the Work includes a "NOTICE" text file as part of its\n          distribution, then any Derivative Works that You distribute must\n          include a readable copy of the attribution notices contained\n          within such NOTICE file, excluding those notices that do not\n          pertain to any part of the Derivative Works, in at least one\n          of the following places: within a NOTICE text file distributed\n          as part of the Derivative Works; within the Source form or\n          documentation, if provided along with the Derivative Works; or,\n          within a display generated by the Derivative Works, if and\n          wherever such third-party notices normally appear. The contents\n          of the NOTICE file are for informational purposes only and\n          do not modify the License. You may add Your own attribution\n          notices within Derivative Works that You distribute, alongside\n          or as an addendum to the NOTICE text from the Work, provided\n          that such additional attribution notices cannot be construed\n          as modifying the License.\n\n      You may add Your own copyright statement to Your modifications and\n      may provide additional or different license terms and conditions\n      for use, reproduction, or distribution of Your modifications, or\n      for any such Derivative Works as a whole, provided Your use,\n      reproduction, and distribution of the Work otherwise complies with\n      the conditions stated in this License.\n\n   5. Submission of Contributions. Unless You explicitly state otherwise,\n      any Contribution intentionally submitted for inclusion in the Work\n      by You to the Licensor shall be under the terms and conditions of\n      this License, without any additional terms or conditions.\n      Notwithstanding the above, nothing herein shall supersede or modify\n      the terms of any separate license agreement you may have executed\n      with Licensor regarding such Contributions.\n\n   6. Trademarks. This License does not grant permission to use the trade\n      names, trademarks, service marks, or product names of the Licensor,\n      except as required for reasonable and customary use in describing the\n      origin of the Work and reproducing the content of the NOTICE file.\n\n   7. Disclaimer of Warranty. Unless required by applicable law or\n      agreed to in writing, Licensor provides the Work (and each\n      Contributor provides its Contributions) on an "AS IS" BASIS,\n      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or\n      implied, including, without limitation, any warranties or conditions\n      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A\n      PARTICULAR PURPOSE. You are solely responsible for determining the\n      appropriateness of using or redistributing the Work and assume any\n      risks associated with Your exercise of permissions under this License.\n\n   8. Limitation of Liability. In no event and under no legal theory,\n      whether in tort (including negligence), contract, or otherwise,\n      unless required by applicable law (such as deliberate and grossly\n      negligent acts) or agreed to in writing, shall any Contributor be\n      liable to You for damages, including any direct, indirect, special,\n      incidental, or consequential damages of any character arising as a\n      result of this License or out of the use or inability to use the\n      Work (including but not limited to damages for loss of goodwill,\n      work stoppage, computer failure or malfunction, or any and all\n      other commercial damages or losses), even if such Contributor\n      has been advised of the possibility of such damages.\n\n   9. Accepting Warranty or Additional Liability. While redistributing\n      the Work or Derivative Works thereof, You may choose to offer,\n      and charge a fee for, acceptance of support, warranty, indemnity,\n      or other liability obligations and/or rights consistent with this\n      License. However, in accepting such obligations, You may act only\n      on Your own behalf and on Your sole responsibility, not on behalf\n      of any other Contributor, and only if You agree to indemnify,\n      defend, and hold each Contributor harmless for any liability\n      incurred by, or claims asserted against, such Contributor by reason\n      of your accepting any such warranty or additional liability.\n\n   END OF TERMS AND CONDITIONS\n\n   APPENDIX: How to apply the Apache License to your work.\n\n      To apply the Apache License to your work, attach the following\n      boilerplate notice, with the fields enclosed by brackets "[]"\n      replaced with your own identifying information. (Don\'t include\n      the brackets!)  The text should be enclosed in the appropriate\n      comment syntax for the file format. We also recommend that a\n      file or class name and description of purpose be included on the\n      same "printed page" as the copyright notice for easier\n      identification within third-party archives.\n\n   Copyright 2026 Kurt Valcorza\n\n   Licensed under the Apache License, Version 2.0 (the "License");\n   you may not use this file except in compliance with the License.\n   You may obtain a copy of the License at\n\n       http://www.apache.org/licenses/LICENSE-2.0\n\n   Unless required by applicable law or agreed to in writing, software\n   distributed under the License is distributed on an "AS IS" BASIS,\n   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\n   See the License for the specific language governing permissions and\n   limitations under the License.\n', 'source.json': '{\n  "repository": "kurtvalcorza/detr-detection-pipeline",\n  "generator": "build_rice_capstone.py/1",\n  "generator_sha256": "e14250d10c67787f522661965ae84e74bf386d08642bdcd3db6c326ed4bc9be7",\n  "files": {\n    "capstone.py": "a73cd2ffc7f0f537a940146571bff72abd493f2a2556775fd48000d3514830f7",\n    "rice_core.py": "d3d07f69cb6486681e62fa018bab2ba79ea4a51096a262e20d1c986c61b2b877",\n    "rice_assets.py": "473118d87d6dbd1e17ad4427173fbda94909fc891c56cabb5da7fcc96ec1da04",\n    "rice_figures.py": "09d4eace0c20e8b40b762337b4fbb8b45ab9080728c3c9c656a6577bc10586d8",\n    "data_manifest.json": "272ff5523de026d4e154d3e008ed39a046e434bb6c9e2974c2c222c71f8bf7a1",\n    "dataset_audit.json": "885c16ed3224d59525922f6c5d3cf7023c4f1a580f0d9b6525352f0f5d1f8be0",\n    "mirror.json": "3e8739ee0fcedb8c72c145fbe5aaf47a30c6c747a68ff0ec2b9bc4dea0a12b2f",\n    "model_manifest.json": "016ffe4eb895ec823f192780c028c9f6c8bc69f2f8d4cf13cc5ad2493379e91b",\n    "requirements.txt": "0df0680e45f99233827a7e4704c5d342d523779dd76fee19d8870b9dad69d13f",\n    "LICENSE.txt": "f9da6567cb5a1828ccec41c7b72b70997fe6d9d63811b3f3177a7b01da52d138"\n  }\n}'}
for name, source in FILES.items():
    target = ROOT / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(source.encode('utf-8'))
print('Carried files:', len(FILES))

started = time.perf_counter()
UV_URL = 'https://files.pythonhosted.org/packages/1e/fd/432451d732917c49152a291de3ef171aa6b0f1a22d39780fb2c1f085ca4c/uv-0.12.15-py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl'
with urllib.request.urlopen(UV_URL, timeout=90) as response:
    wheel = response.read(20081405)
if len(wheel) != 20081404 or hashlib.sha256(wheel).hexdigest() != 'aee9802f46bae436bd91751bb33ddeb379ef1596b5c19df193219d545d244b60':
    raise RuntimeError('Environment bootstrap size/hash mismatch')
with zipfile.ZipFile(io.BytesIO(wheel)) as archive:
    member = next(name for name in archive.namelist() if name.endswith('.data/scripts/uv'))
    UV = ROOT / 'uv'
    UV.write_bytes(archive.read(member))
UV.chmod(0o700)
ENV = dict(os.environ, HF_HUB_DISABLE_IMPLICIT_TOKEN='1', HF_HUB_DISABLE_TELEMETRY='1', DO_NOT_TRACK='1', MPLBACKEND='Agg')
ENV.pop('HF_TOKEN', None)
ENV.pop('HUGGING_FACE_HUB_TOKEN', None)

def checked(args, **kwargs):
    # Capture the child's stderr so a failure shows its real exception in the cell, not only an exit status.
    result = subprocess.run([str(a) for a in args], env=ENV, capture_output=True, text=True, **kwargs)
    if result.stdout.strip():
        print(result.stdout.rstrip(), flush=True)
    if result.returncode:
        print(result.stderr.rstrip(), flush=True)
        raise RuntimeError(f'{Path(str(args[0])).name} failed ({result.returncode}): ' + result.stderr.strip()[-2000:])
    return result

checked([UV, 'venv', '--managed-python', '--python', '3.12.12', ROOT / 'env'])
PYTHON = ROOT / 'env/bin/python'
checked([UV, 'pip', 'install', '--python', PYTHON, '--require-hashes', '--only-binary', ':all:', '-r', ROOT / 'requirements.txt'])
checked([PYTHON, '-c', 'import torch; assert torch.cuda.is_available(); print(torch.__version__)'])
(ROOT / 'bootstrap.json').write_text(json.dumps({'seconds': time.perf_counter()-started, 'gpu': GPU}), encoding='utf-8')

def run(stage):
    log = ROOT / (stage + '.log')
    print('Running', stage, '— retain this log if anything fails:', log, flush=True)
    with log.open('w', encoding='utf-8') as handle:
        process = subprocess.Popen([str(PYTHON), '-u', str(ROOT / 'capstone.py'), '--root', str(ROOT), '--stage', stage], env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            handle.write(line)
            handle.flush()
            if len(line) < 2000:
                print(line.rstrip(), flush=True)
        process.wait()
    if process.returncode:
        raise RuntimeError(f'{stage} failed ({process.returncode}); inspect {log}. Do not skip the failed stage.')

def record(name):
    value = json.loads((ROOT / 'outputs' / name).read_text(encoding='utf-8'))
    print(json.dumps(value, indent=2)[:18000])
    display(FileLink(str(ROOT / 'outputs' / name)))

def table(name, limit=16):
    path = ROOT / 'outputs' / name
    with path.open(encoding='utf-8', newline='') as handle:
        reader = csv.DictReader(handle)
        columns, rows = reader.fieldnames, list(reader)
    if not columns:
        raise RuntimeError('Missing table columns: ' + name)
    visible = [c for c in columns if c not in ('scores', 'bbox_xyxy')][:12]
    clean = lambda value: str(value).replace('|', '/').replace('\n', ' ')
    lines = ['| ' + ' | '.join(visible) + ' |', '| ' + ' | '.join('---' for _ in visible) + ' |']
    lines += ['| ' + ' | '.join(clean(row[c]) for c in visible) + ' |' for row in rows[:limit]]
    display(Markdown('\n'.join(lines)))
    print('Rows:', len(rows), '(preview limited to', limit, ')')
    display(FileLink(str(path)))

def figures(stage):
    checked([PYTHON, ROOT / 'rice_figures.py', '--root', ROOT, '--stage', stage])
    for path in sorted((ROOT / 'outputs' / 'figures').glob(stage + '*.png')):
        display(Image(filename=str(path)))
    if stage == 'evaluate':
        for path in sorted((ROOT / 'outputs' / 'local_figures').glob('*.png')):
            display(Image(filename=str(path)))

## 2. Audit the sample before using models
Only frozen ZIP members are downloaded. Their SHA-256, ZIP CRC, byte length and image dimensions are verified.

The 200 images come from a hash-pinned sample bundle published as a release asset of this repository (`mirror.json` records its URL, size and SHA-256). Every member is byte-identical to the Zenodo ZIP member named in the manifest and is re-checked against it. If the bundle cannot be fetched or fails a check, the cell prints one line and falls back to reading the frozen members directly from Zenodo, which remains the source of record. Source-family splits, annotations and limitations are checked before training.

The archive has 112,940 entries, mostly images and crops. We fetch a bounded sample rather than extracting it all. Per-member hashes establish the assets used; this run does not claim to hash the entire approximately 1 GB archive. Original creator attribution and licence references travel with the manifest.

**Notice:** image dimensions, objects per image, species support and source-family counts. The first table gives support per split: the species columns count reference boxes (insects), not images. The second lists the audit gates: *unresolved* gates are the reasons this is an exploratory benchmark. A missing annotation is not evidence that no pests are present.

In [ ]:
checked([PYTHON, '-c', "import json; from pathlib import Path; from rice_assets import prepare_images; root=Path('.'); load=lambda name: json.loads((root/name).read_text(encoding='utf-8')); print(prepare_images(root, load('data_manifest.json'), load('mirror.json')))"], cwd=ROOT)
run('prepare')
table('sample_summary.csv')
table('audit_gates.csv')
table('split_manifest.csv')
figures('prepare')

## 3. Reference crops and representation baselines
**Input:** crops reconstructed from published reference boxes. **Models:** majority class, RGB colour centroid, BioCLIP 2 and SigLIP 2. **Output:** validation classification metrics.

Crop evaluation assumes an object has already been located. It cannot measure missed insects. Common-name prompts and crop preprocessing are fixed before test inspection. RGB success can indicate background shortcuts; large confidence does not establish biological correctness.

The first figure shows what the classifier receives: training crops cut from reference boxes, labelled with their published species. Small crops carry little detail.

**Predict:** will the biodiversity representation outperform the general representation? Compare macro-F1 and class support rather than confidence alone.

**How to read the result:** for each method, `accuracy` and `macro_f1` range from 0 to 1; `confusion` rows are reference species and columns predicted species (rice black bug first). The majority baseline's macro-F1 shows what "no information" looks like.

In [ ]:
figures('crops')
run('features')
record('validation_crops.json')

## 4. Adapt only the species head
The BioCLIP image tower stays frozen. A small linear head starts from text embeddings and trains for at most 20 epochs. Epoch zero is eligible; minimum validation cross-entropy selects the checkpoint, with earlier ties.

**Notice:** whether validation loss improves with training loss. More epochs are not automatically better. The test split is not used to choose this head.

In [ ]:
run('classifier')
record('classifier_history.json')
figures('classifier')

## 5. Adapt the detector
DETR starts from COCO weights, but its new target-pest head is untrained. Epoch zero is an untrained-head baseline, not zero-shot pest detection. The ResNet backbone stays frozen while transformer and detection heads train. The bounded recipe uses gradient accumulation and validation AP50 checkpoint selection.

**Input:** complete image exports and their boxes. **Output:** ranked target-pest boxes. The detector cannot represent more than 100 objects per image. We do not truncate reference boxes to make a metric look better.

**Notice:** validation localisation quality and the chosen epoch. Poor adaptation remains a reportable result.

In [ ]:
run('detector')
record('detector_history.json')
figures('detector')

## 6. Lock the composed policy
Checkpoint selection is finished. Choose the detection threshold on validation data to minimise per-image/per-species count MAE, with higher-threshold ties. Separately choose a species-score-margin review policy.

Every retained detection contributes to the raw count. Review flags create unresolved work; they do not make insects disappear. A missed detection cannot be referred by the classifier. Scores are not calibrated probabilities or operational pest-alert thresholds.

The review policy first looks for the margin with the highest coverage that keeps at least 80% selective accuracy while accepting at least 50% of validation crops. If no margin reaches that target, it falls back to the most accurate margin, then the widest coverage. The output states which rule applied; a fallback can send many detections to review.

**Predict:** will the full pipeline match the reference-crop classifier? Name a failure the crop classifier cannot expose.

In [ ]:
run('policy')
record('selected_policy.json')
locked = json.loads((ROOT / 'outputs' / 'selected_policy.json').read_text(encoding='utf-8'))
print('Detector threshold:', locked['detector_threshold'])
print('Review margin:', locked['review_margin'], '|', locked['review_selection_rule'])
print('Validation coverage / selective accuracy:', locked['review_validation_coverage'], '/', locked['review_validation_accuracy'])

## 7. Reveal the held-out benchmark
The primary metric averages absolute count error over images and both species, including zero counts. Compare it with the training-mean and zero-count baselines. Oracle localisation uses published boxes; it is not a deployable system or a guaranteed mathematical upper bound because counting errors can cancel.

Inspect crop macro-F1, detection AP, misses, spurious/duplicate boxes and wrong-species matches. Bootstrap intervals resample source families; they do not establish independent capture events, complete annotations, pretraining independence or wider field performance. Two-class top-2 accuracy would be uninformative and is not a headline metric.

**Where does count error enter?** After the metrics, three tables trace it:
1. **Error summary**, per species: `reference = correct + wrong_out + missed` and `raw = correct + wrong_in + spurious`. `wrong_out` is a reference insect of this species labelled as the other; `wrong_in` is the reverse. `accepted` and `referred` split the raw count by the review policy.
2. **Matched-species confusion**: rows are reference species, columns predicted species; the last column counts missed references, and the last row counts spurious predictions.
3. **Example panels**: one held-out photograph per error category (misses, spurious, wrong species, success). Each title says which category it illustrates and gives its counts. Green boxes are correct matches, magenta boxes are wrong-species matches (the white dashed box is the reference), dashed blue boxes are missed references, and orange boxes are spurious predictions. Each error box carries a `#tag`; the table after the panels lists each tag's reference species, predicted species, **detector score**, **species score**, margin and review state. The last table reconciles each example's per-species counts.

If a category is absent from the held-out split, the inventory says so; no example is manufactured.

In [ ]:
run('evaluate')
table('metrics.csv')
record('test_crop_metrics.json')
record('detection_metrics.json')
record('bootstrap.json')
table('error_summary.csv')
table('matched_species_confusion.csv')
table('counts.csv')
figures('evaluate')
record('error_panel_inventory.json')
table('error_examples.csv', limit=40)
table('error_example_counts.csv')

## 8. Change one thing: the display threshold
Predict what a lower detector threshold will do to misses, spurious boxes and count MAE. The activity compares lower/canonical/higher settings with the same models. It is retrospective test analysis, not permission to replace the locked result with the best-looking test setting.

Canonical predictions and policy remain unchanged. Explain any error cancellation: a correct total can still contain missed and extra insects.

The second table follows one held-out photograph: the image whose raw count changes most across the three settings. For each setting and species it shows the reference count, the raw count and the stage errors, so you can see which errors a threshold change adds or removes.

In [ ]:
run('activity')
table('activity_thresholds.csv')
table('activity_paired_counts.csv')
figures('activity')

## 9. Export, reload and verify
Both adapters reload in a fresh process with pinned bases. Original held-out images are reprocessed; boxes, species, counts and review decisions must agree within declared tolerances. Hashes and class order must match before tensors are loaded.

The results archive includes derived evidence, attribution and the metric charts (learning curves, count comparison and threshold charts). Source photographs, photo previews and annotated panels are excluded by default. A passing archive check proves internal consistency, not independent biological validation.

In [ ]:
run('reload')
record('verification.json')
run('report')
record('run_summary.json')
display(FileLink(str(ROOT / 'outputs' / 'results.zip')))

## 10. Conclude with evidence
Complete this optional statement using the exported results:

> On [N] held-out image exports grouped into [G] source families, the composed pipeline achieved count MAE [value and interval], compared with [baseline]. Misses, extra detections and species errors contributed [evidence]. Changing only the display threshold showed [trade-off]. These results describe agreement with this dataset's published annotations; they do not establish independent-capture generalisation, field abundance, crop damage or intervention thresholds.

**Limitations to retain:** source-image grouping cannot resolve repeated specimens in different originals; exported images were stretched to 640×640; annotations and completeness were not independently verified; source capture locations are unavailable; the sample excludes over-capacity images; pretrained-model overlap is unresolved. Report absent error categories and zero detections honestly.

**Continue exploring:** compare the full [DETR](https://github.com/kurtvalcorza/detr-detection-pipeline), [BioCLIP 2](https://github.com/kurtvalcorza/bioclip2-biodiversity-pipeline) and [SigLIP 2](https://github.com/kurtvalcorza/siglip2-vision-language-pipeline) pipelines with the models available through [DIMER](https://training.dimer1.asti.dost.gov.ph/signin). A stronger follow-up would collect verified trap/session/specimen metadata and independently checked exhaustive annotations.

**Optional extension:** four-way species–sex classification requires separately verified sex labels and adequate independent support. It is not enabled by default. Completion records are not required submissions.

### Troubleshooting
If a source hash, model hash or receipt fails, preserve the log and stop; do not bypass the check. If T4 memory or time limits are exceeded, record actual use and revise the recipe before a new clean run. Do not silently lower sample size, epochs or image resolution. A model that fails to beat the count baseline is an informative result, not an execution error.

Sources: [dataset and citation](https://doi.org/10.5281/zenodo.20066074), [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/), [DIMER notebook standard](https://github.com/kurtvalcorza/ml-worker/blob/main/integrations/dimer/fleet-specs/NOTEBOOK_SPEC.md). Exact source/model identities and creator attribution are carried in the manifests.